In [ ]:
import os
import torch

print("=== Kaggle environment check ===")
print("Current working directory:", os.getcwd())

print("\n=== GPU check ===")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. We may need to enable GPU in notebook settings.")

print("\n=== Kaggle input datasets ===")
print(os.listdir("/kaggle/input"))

print("\n=== MU-Glioma-Post directory preview ===")
dataset_root = "/kaggle/input/mu-glioma-post"

if os.path.exists(dataset_root):
    print("Dataset root exists:", dataset_root)
    for root, dirs, files in os.walk(dataset_root):
        print("ROOT:", root)
        print("DIRS:", dirs[:20])
        print("FILES:", files[:20])
        break
else:
    print("Dataset root not found. Available folders:")
    print(os.listdir("/kaggle/input"))

In [ ]:
import os

root = "/kaggle/input"

print("=== Level 1 ===")
print(os.listdir(root))

print("\n=== Walk first 3 levels ===")
max_print = 80
count = 0

for dirpath, dirnames, filenames in os.walk(root):
    level = dirpath.replace(root, "").count(os.sep)
    if level <= 3:
        print("\nROOT:", dirpath)
        print("DIRS:", dirnames[:10])
        print("FILES:", filenames[:10])
        count += 1
        if count >= max_print:
            break

In [ ]:
import os
from pathlib import Path

root = Path("/kaggle/input")

print("=== All folders under /kaggle/input, first 5 levels ===")

for dirpath, dirnames, filenames in os.walk(root):
    level = str(dirpath).replace(str(root), "").count(os.sep)
    if level <= 5:
        print("ROOT:", repr(dirpath))
        print("DIRS:", [repr(d) for d in dirnames[:20]])
        print("FILES:", [repr(f) for f in filenames[:10]])
        print("-" * 80)

In [ ]:
import os
from pathlib import Path
from collections import Counter

root = Path("/kaggle/input")

candidate_dirs = []

for p in root.rglob("*"):
    if p.is_dir():
        name = p.name.lower()
        full = str(p).lower()
        if "glioma" in name or "glioma" in full or "mu" in name:
            candidate_dirs.append(p)

print("=== Candidate directories ===")
for i, p in enumerate(candidate_dirs[:50]):
    print(i, repr(str(p)))

print("\nTotal candidates:", len(candidate_dirs))

In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post")

print("DATA_ROOT exists:", DATA_ROOT.exists())
print("DATA_ROOT:", DATA_ROOT)

patients = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith("PatientID_")])

print("\nNumber of patient folders:", len(patients))
print("First 20 patients:")
for p in patients[:20]:
    print(p.name)

print("\n=== Inspect first 5 patient folders ===")
for patient_dir in patients[:5]:
    print("\n" + "=" * 80)
    print("PATIENT:", patient_dir.name)
    
    for dirpath, dirnames, filenames in os.walk(patient_dir):
        level = str(Path(dirpath).relative_to(patient_dir)).count(os.sep)
        if level <= 3:
            print("ROOT:", dirpath)
            print("DIRS:", dirnames[:20])
            print("FILES:", filenames[:20])
        if level > 3:
            break

In [ ]:
from pathlib import Path
from collections import Counter

DATA_ROOT = Path("/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post")

nii_files = []

for p in DATA_ROOT.rglob("*"):
    if p.is_file():
        name = p.name.lower()
        if name.endswith(".nii") or name.endswith(".nii.gz"):
            nii_files.append(p)

print("Total NIfTI files found:", len(nii_files))

print("\n=== First 100 NIfTI files ===")
for p in nii_files[:100]:
    print(str(p))

print("\n=== Filename keyword summary ===")
keywords = ["t1c", "t1ce", "t1gd", "t1", "t2", "flair", "seg", "mask", "tumor", "tumour"]
for k in keywords:
    count = sum(k in p.name.lower() for p in nii_files)
    print(k, count)

In [ ]:
import os
import re
import pandas as pd
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post")
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def timepoint_number(tp_name):
    m = re.search(r"Timepoint_(\d+)", tp_name)
    return int(m.group(1)) if m else None

def find_required_files(tp_dir):
    files = list(tp_dir.glob("*.nii"))
    t1c = [f for f in files if "_brain_t1c.nii" in f.name]
    mask = [f for f in files if "_tumorMask.nii" in f.name]
    return {
        "t1c": str(t1c[0]) if len(t1c) == 1 else None,
        "mask": str(mask[0]) if len(mask) == 1 else None,
    }

rows = []

patient_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith("PatientID_")])

for patient_dir in patient_dirs:
    tp_dirs = sorted(
        [p for p in patient_dir.iterdir() if p.is_dir() and p.name.startswith("Timepoint_")],
        key=lambda x: timepoint_number(x.name)
    )
    
    usable_tps = []
    for tp_dir in tp_dirs:
        files = find_required_files(tp_dir)
        tp_num = timepoint_number(tp_dir.name)
        if tp_num is not None and files["t1c"] is not None and files["mask"] is not None:
            usable_tps.append({
                "tp_num": tp_num,
                "tp_name": tp_dir.name,
                "t1c": files["t1c"],
                "mask": files["mask"],
            })
    
    # Use consecutive available timepoints only: T_i -> T_{i+1 in sorted available list}
    for i in range(len(usable_tps) - 1):
        cur = usable_tps[i]
        fut = usable_tps[i + 1]
        
        rows.append({
            "case_id": f"{patient_dir.name}_T{cur['tp_num']}_to_T{fut['tp_num']}_t1c",
            "patient_id": patient_dir.name,
            "current_timepoint": cur["tp_name"],
            "future_timepoint": fut["tp_name"],
            "current_tp_num": cur["tp_num"],
            "future_tp_num": fut["tp_num"],
            "current_t1c_path": cur["t1c"],
            "current_mask_path": cur["mask"],
            "future_t1c_path": fut["t1c"],
            "future_mask_path": fut["mask"],
        })

manifest_all = pd.DataFrame(rows)

print("Total longitudinal pairs found:", len(manifest_all))
print("Patients with at least one pair:", manifest_all["patient_id"].nunique())

display(manifest_all.head(20))

manifest_all_path = OUT_DIR / "manifest_all_pairs.csv"
manifest_all.to_csv(manifest_all_path, index=False)

# For the first experiment, select 40 pairs.
# Priority: use earliest available consecutive pair per patient, then first 40 patients.
manifest_40 = (
    manifest_all
    .sort_values(["patient_id", "current_tp_num", "future_tp_num"])
    .groupby("patient_id", as_index=False)
    .first()
    .sort_values(["patient_id"])
    .head(40)
)

manifest_40_path = OUT_DIR / "manifest_40.csv"
manifest_40.to_csv(manifest_40_path, index=False)

print("\nSaved:")
print(manifest_all_path)
print(manifest_40_path)

print("\nSelected 40-case manifest:")
display(manifest_40)

In [ ]:
!pip install -q nibabel

In [ ]:
import pandas as pd
import numpy as np
import nibabel as nib
from pathlib import Path
from tqdm import tqdm

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

manifest_all_path = OUT_DIR / "manifest_all_pairs.csv"
manifest_40_path = OUT_DIR / "manifest_40.csv"

manifest_all = pd.read_csv(manifest_all_path)
manifest_40 = pd.read_csv(manifest_40_path)

print("manifest_all:", manifest_all.shape)
print("manifest_40:", manifest_40.shape)

def load_mask(path):
    arr = nib.load(path).get_fdata()
    return arr > 0.5

def compute_pair_stats(row):
    cur_mask = load_mask(row["current_mask_path"])
    fut_mask = load_mask(row["future_mask_path"])
    
    if cur_mask.shape != fut_mask.shape:
        return {
            "case_id": row["case_id"],
            "patient_id": row["patient_id"],
            "shape_ok": False,
            "shape": str(cur_mask.shape),
            "current_voxels": np.nan,
            "future_voxels": np.nan,
            "growth_voxels": np.nan,
            "shrink_voxels": np.nan,
            "net_change_voxels": np.nan,
        }
    
    growth = np.logical_and(fut_mask, np.logical_not(cur_mask))
    shrink = np.logical_and(cur_mask, np.logical_not(fut_mask))
    
    current_voxels = int(cur_mask.sum())
    future_voxels = int(fut_mask.sum())
    growth_voxels = int(growth.sum())
    shrink_voxels = int(shrink.sum())
    
    return {
        "case_id": row["case_id"],
        "patient_id": row["patient_id"],
        "shape_ok": True,
        "shape": str(cur_mask.shape),
        "current_voxels": current_voxels,
        "future_voxels": future_voxels,
        "growth_voxels": growth_voxels,
        "shrink_voxels": shrink_voxels,
        "net_change_voxels": future_voxels - current_voxels,
        "current_timepoint": row["current_timepoint"],
        "future_timepoint": row["future_timepoint"],
        "current_tp_num": row["current_tp_num"],
        "future_tp_num": row["future_tp_num"],
        "current_t1c_path": row["current_t1c_path"],
        "current_mask_path": row["current_mask_path"],
        "future_t1c_path": row["future_t1c_path"],
        "future_mask_path": row["future_mask_path"],
    }

stats_rows = []

for _, row in tqdm(manifest_all.iterrows(), total=len(manifest_all)):
    try:
        stats_rows.append(compute_pair_stats(row))
    except Exception as e:
        stats_rows.append({
            "case_id": row["case_id"],
            "patient_id": row["patient_id"],
            "shape_ok": False,
            "error": str(e),
        })

stats_df = pd.DataFrame(stats_rows)

stats_path = OUT_DIR / "pair_growth_stats_all.csv"
stats_df.to_csv(stats_path, index=False)

print("Saved stats:", stats_path)

print("\n=== Overall summary ===")
print("Total pairs:", len(stats_df))
print("Shape OK:", int(stats_df["shape_ok"].sum()))
print("Shape failed:", int((~stats_df["shape_ok"]).sum()))

valid = stats_df[stats_df["shape_ok"] == True].copy()

print("\nGrowth voxel summary:")
display(valid["growth_voxels"].describe())

print("\nNumber of pairs by growth threshold:")
for th in [1, 10, 50, 100, 500, 1000, 2000]:
    print(f"growth_voxels >= {th}:", int((valid["growth_voxels"] >= th).sum()))

print("\nTop 20 growth pairs:")
display(valid.sort_values("growth_voxels", ascending=False).head(20))

print("\nBottom 20 growth pairs:")
display(valid.sort_values("growth_voxels", ascending=True).head(20))

In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

manifest_40_path = OUT_DIR / "manifest_40.csv"
stats_path = OUT_DIR / "pair_growth_stats_all.csv"

manifest_40 = pd.read_csv(manifest_40_path)
stats_all = pd.read_csv(stats_path)

locked_40 = manifest_40.merge(
    stats_all[
        [
            "case_id",
            "current_voxels",
            "future_voxels",
            "growth_voxels",
            "shrink_voxels",
            "net_change_voxels",
            "shape_ok",
            "shape"
        ]
    ],
    on="case_id",
    how="left"
)

locked_path = OUT_DIR / "manifest_locked_original_40.csv"
locked_40.to_csv(locked_path, index=False)

print("Saved:", locked_path)
print("Cases:", len(locked_40))
print("Patients:", locked_40["patient_id"].nunique())
print("Shape OK:", int(locked_40["shape_ok"].sum()))

print("\nGrowth voxel summary:")
print(locked_40["growth_voxels"].describe())

print("\nLocked original 40 cases:")
display(
    locked_40[
        [
            "case_id",
            "patient_id",
            "current_timepoint",
            "future_timepoint",
            "growth_voxels",
            "current_voxels",
            "future_voxels",
            "net_change_voxels"
        ]
    ]
)

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

locked_path = OUT_DIR / "manifest_locked_original_40.csv"
locked_40 = pd.read_csv(locked_path)

# Sort for deterministic order
locked_40 = locked_40.sort_values("patient_id").reset_index(drop=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_rows = []

for fold, (train_idx, test_idx) in enumerate(kf.split(locked_40), start=1):
    train_cases = locked_40.iloc[train_idx].copy()
    test_cases = locked_40.iloc[test_idx].copy()
    
    for _, row in train_cases.iterrows():
        r = row.to_dict()
        r["fold"] = fold
        r["split"] = "train"
        fold_rows.append(r)
    
    for _, row in test_cases.iterrows():
        r = row.to_dict()
        r["fold"] = fold
        r["split"] = "test"
        fold_rows.append(r)

fold_df = pd.DataFrame(fold_rows)

fold_path = OUT_DIR / "manifest_locked_original_40_5fold.csv"
fold_df.to_csv(fold_path, index=False)

print("Saved:", fold_path)

print("\nFold summary:")
summary = fold_df.groupby(["fold", "split"])["case_id"].count().unstack()
display(summary)

print("\nTest cases per fold:")
for fold in sorted(fold_df["fold"].unique()):
    print("\nFold", fold)
    display(
        fold_df[
            (fold_df["fold"] == fold) &
            (fold_df["split"] == "test")
        ][["case_id", "patient_id", "growth_voxels"]]
    )

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path
from tqdm import tqdm

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
PRE_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"
PRE_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = OUT_DIR / "manifest_locked_original_40.csv"
manifest = pd.read_csv(manifest_path)

print("Loaded manifest:", manifest.shape)
print("Output dir:", PRE_DIR)

def robust_normalize_t1c(img):
    """
    Normalize T1c MRI to approximately [0, 1] using nonzero brain voxels.
    This avoids extreme intensity outliers dominating the scale.
    """
    img = img.astype(np.float32)
    nonzero = img[img > 0]
    
    if nonzero.size < 100:
        return np.zeros_like(img, dtype=np.float32)
    
    p1, p99 = np.percentile(nonzero, [1, 99])
    if p99 <= p1:
        return np.zeros_like(img, dtype=np.float32)
    
    img = np.clip(img, p1, p99)
    img = (img - p1) / (p99 - p1 + 1e-8)
    img[img < 0] = 0
    img[img > 1] = 1
    return img.astype(np.float32)

case_summaries = []

for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
    case_id = row["case_id"]
    
    current_t1c = nib.load(row["current_t1c_path"]).get_fdata().astype(np.float32)
    current_mask = nib.load(row["current_mask_path"]).get_fdata() > 0.5
    future_mask = nib.load(row["future_mask_path"]).get_fdata() > 0.5
    
    assert current_t1c.shape == current_mask.shape == future_mask.shape, f"Shape mismatch: {case_id}"
    
    current_t1c_norm = robust_normalize_t1c(current_t1c)
    
    future_change = np.logical_and(future_mask, np.logical_not(current_mask))
    
    H, W, Z = current_t1c_norm.shape
    
    # X shape: [Z, 2, H, W]
    # channel 0 = current T1c
    # channel 1 = current tumour mask
    X = np.zeros((Z, 2, H, W), dtype=np.float16)
    Y = np.zeros((Z, 1, H, W), dtype=np.uint8)
    
    for z in range(Z):
        X[z, 0] = current_t1c_norm[:, :, z].astype(np.float16)
        X[z, 1] = current_mask[:, :, z].astype(np.float16)
        Y[z, 0] = future_change[:, :, z].astype(np.uint8)
    
    target_voxels_per_slice = Y.reshape(Z, -1).sum(axis=1)
    current_voxels_per_slice = X[:, 1].reshape(Z, -1).sum(axis=1)
    
    npz_path = PRE_DIR / f"{case_id}.npz"
    np.savez(
        npz_path,
        X=X,
        Y=Y,
        target_voxels_per_slice=target_voxels_per_slice,
        current_voxels_per_slice=current_voxels_per_slice,
        case_id=case_id,
        patient_id=row["patient_id"],
        current_timepoint=row["current_timepoint"],
        future_timepoint=row["future_timepoint"],
    )
    
    case_summaries.append({
        "case_id": case_id,
        "patient_id": row["patient_id"],
        "npz_path": str(npz_path),
        "shape": str(current_t1c.shape),
        "num_slices": Z,
        "current_voxels": int(current_mask.sum()),
        "future_voxels": int(future_mask.sum()),
        "growth_voxels": int(future_change.sum()),
        "positive_slices": int((target_voxels_per_slice > 0).sum()),
        "current_mask_slices": int((current_voxels_per_slice > 0).sum()),
    })

summary_df = pd.DataFrame(case_summaries)
summary_path = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nSaved preprocessing summary:")
print(summary_path)

print("\nSummary:")
print("Cases:", len(summary_df))
print("Total slices:", int(summary_df["num_slices"].sum()))
print("Total positive target slices:", int(summary_df["positive_slices"].sum()))
print("Mean positive slices per case:", float(summary_df["positive_slices"].mean()))
print("Min positive slices per case:", int(summary_df["positive_slices"].min()))
print("Max positive slices per case:", int(summary_df["positive_slices"].max()))

display(summary_df)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

files_to_check = [
    OUT_DIR / "manifest_locked_original_40.csv",
    OUT_DIR / "manifest_locked_original_40_5fold.csv",
    OUT_DIR / "preprocessed_locked40_2d_summary.csv",
    OUT_DIR / "preprocessed_locked40_2d_npz",
]

for p in files_to_check:
    print(p, "EXISTS =", p.exists())

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import KFold

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib

# =========================
# Paths
# =========================
DATA_ROOT = Path("/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post")
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
PRE_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PRE_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT exists:", DATA_ROOT.exists())
print("OUT_DIR:", OUT_DIR)

# =========================
# 1. Rebuild all-pair manifest
# =========================
def timepoint_number(tp_name):
    m = re.search(r"Timepoint_(\d+)", tp_name)
    return int(m.group(1)) if m else None

def find_required_files(tp_dir):
    files = list(tp_dir.glob("*.nii"))
    t1c = [f for f in files if "_brain_t1c.nii" in f.name]
    mask = [f for f in files if "_tumorMask.nii" in f.name]
    return {
        "t1c": str(t1c[0]) if len(t1c) == 1 else None,
        "mask": str(mask[0]) if len(mask) == 1 else None,
    }

rows = []
patient_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith("PatientID_")])

for patient_dir in patient_dirs:
    tp_dirs = sorted(
        [p for p in patient_dir.iterdir() if p.is_dir() and p.name.startswith("Timepoint_")],
        key=lambda x: timepoint_number(x.name)
    )
    
    usable_tps = []
    for tp_dir in tp_dirs:
        files = find_required_files(tp_dir)
        tp_num = timepoint_number(tp_dir.name)
        if tp_num is not None and files["t1c"] is not None and files["mask"] is not None:
            usable_tps.append({
                "tp_num": tp_num,
                "tp_name": tp_dir.name,
                "t1c": files["t1c"],
                "mask": files["mask"],
            })
    
    for i in range(len(usable_tps) - 1):
        cur = usable_tps[i]
        fut = usable_tps[i + 1]
        rows.append({
            "case_id": f"{patient_dir.name}_T{cur['tp_num']}_to_T{fut['tp_num']}_t1c",
            "patient_id": patient_dir.name,
            "current_timepoint": cur["tp_name"],
            "future_timepoint": fut["tp_name"],
            "current_tp_num": cur["tp_num"],
            "future_tp_num": fut["tp_num"],
            "current_t1c_path": cur["t1c"],
            "current_mask_path": cur["mask"],
            "future_t1c_path": fut["t1c"],
            "future_mask_path": fut["mask"],
        })

manifest_all = pd.DataFrame(rows)
manifest_all.to_csv(OUT_DIR / "manifest_all_pairs.csv", index=False)

manifest_40 = (
    manifest_all
    .sort_values(["patient_id", "current_tp_num", "future_tp_num"])
    .groupby("patient_id", as_index=False)
    .first()
    .sort_values("patient_id")
    .head(40)
    .reset_index(drop=True)
)

manifest_40.to_csv(OUT_DIR / "manifest_locked_original_40_base.csv", index=False)

print("Total pairs:", len(manifest_all))
print("Locked 40 cases:", len(manifest_40))

# =========================
# 2. Compute stats for locked 40 only
# =========================
def load_mask(path):
    return nib.load(path).get_fdata() > 0.5

stats_rows = []

for _, row in tqdm(manifest_40.iterrows(), total=len(manifest_40), desc="Computing locked-40 stats"):
    cur_mask = load_mask(row["current_mask_path"])
    fut_mask = load_mask(row["future_mask_path"])
    
    growth = np.logical_and(fut_mask, np.logical_not(cur_mask))
    shrink = np.logical_and(cur_mask, np.logical_not(fut_mask))
    
    r = row.to_dict()
    r.update({
        "shape_ok": cur_mask.shape == fut_mask.shape,
        "shape": str(cur_mask.shape),
        "current_voxels": int(cur_mask.sum()),
        "future_voxels": int(fut_mask.sum()),
        "growth_voxels": int(growth.sum()),
        "shrink_voxels": int(shrink.sum()),
        "net_change_voxels": int(fut_mask.sum()) - int(cur_mask.sum()),
    })
    stats_rows.append(r)

locked_40 = pd.DataFrame(stats_rows)
locked_path = OUT_DIR / "manifest_locked_original_40.csv"
locked_40.to_csv(locked_path, index=False)

print("\nSaved:", locked_path)
print("Cases:", len(locked_40))
print("Patients:", locked_40["patient_id"].nunique())
print("Shape OK:", int(locked_40["shape_ok"].sum()))
print("Growth voxel summary:")
print(locked_40["growth_voxels"].describe())

# =========================
# 3. Build 5-fold split
# =========================
locked_40 = locked_40.sort_values("patient_id").reset_index(drop=True)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_rows = []

for fold, (train_idx, test_idx) in enumerate(kf.split(locked_40), start=1):
    for _, row in locked_40.iloc[train_idx].iterrows():
        r = row.to_dict()
        r["fold"] = fold
        r["split"] = "train"
        fold_rows.append(r)
    
    for _, row in locked_40.iloc[test_idx].iterrows():
        r = row.to_dict()
        r["fold"] = fold
        r["split"] = "test"
        fold_rows.append(r)

fold_df = pd.DataFrame(fold_rows)
fold_path = OUT_DIR / "manifest_locked_original_40_5fold.csv"
fold_df.to_csv(fold_path, index=False)

print("\nSaved:", fold_path)
print("Fold summary:")
display(fold_df.groupby(["fold", "split"])["case_id"].count().unstack())

# =========================
# 4. Preprocess locked 40 into 2D npz
# =========================
def robust_normalize_t1c(img):
    img = img.astype(np.float32)
    nonzero = img[img > 0]
    if nonzero.size < 100:
        return np.zeros_like(img, dtype=np.float32)
    
    p1, p99 = np.percentile(nonzero, [1, 99])
    if p99 <= p1:
        return np.zeros_like(img, dtype=np.float32)
    
    img = np.clip(img, p1, p99)
    img = (img - p1) / (p99 - p1 + 1e-8)
    img = np.clip(img, 0, 1)
    return img.astype(np.float32)

case_summaries = []

for _, row in tqdm(locked_40.iterrows(), total=len(locked_40), desc="Preprocessing locked 40"):
    case_id = row["case_id"]
    
    current_t1c = nib.load(row["current_t1c_path"]).get_fdata().astype(np.float32)
    current_mask = nib.load(row["current_mask_path"]).get_fdata() > 0.5
    future_mask = nib.load(row["future_mask_path"]).get_fdata() > 0.5
    
    current_t1c_norm = robust_normalize_t1c(current_t1c)
    future_change = np.logical_and(future_mask, np.logical_not(current_mask))
    
    H, W, Z = current_t1c_norm.shape
    
    X = np.zeros((Z, 2, H, W), dtype=np.float16)
    Y = np.zeros((Z, 1, H, W), dtype=np.uint8)
    
    for z in range(Z):
        X[z, 0] = current_t1c_norm[:, :, z].astype(np.float16)
        X[z, 1] = current_mask[:, :, z].astype(np.float16)
        Y[z, 0] = future_change[:, :, z].astype(np.uint8)
    
    target_voxels_per_slice = Y.reshape(Z, -1).sum(axis=1)
    current_voxels_per_slice = X[:, 1].reshape(Z, -1).sum(axis=1)
    
    npz_path = PRE_DIR / f"{case_id}.npz"
    np.savez(
        npz_path,
        X=X,
        Y=Y,
        target_voxels_per_slice=target_voxels_per_slice,
        current_voxels_per_slice=current_voxels_per_slice,
        case_id=case_id,
        patient_id=row["patient_id"],
        current_timepoint=row["current_timepoint"],
        future_timepoint=row["future_timepoint"],
    )
    
    case_summaries.append({
        "case_id": case_id,
        "patient_id": row["patient_id"],
        "npz_path": str(npz_path),
        "shape": str(current_t1c.shape),
        "num_slices": Z,
        "current_voxels": int(current_mask.sum()),
        "future_voxels": int(future_mask.sum()),
        "growth_voxels": int(future_change.sum()),
        "positive_slices": int((target_voxels_per_slice > 0).sum()),
        "current_mask_slices": int((current_voxels_per_slice > 0).sum()),
    })

summary_df = pd.DataFrame(case_summaries)
summary_path = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nSaved:", summary_path)
print("Cases:", len(summary_df))
print("Total slices:", int(summary_df["num_slices"].sum()))
print("Total positive target slices:", int(summary_df["positive_slices"].sum()))
print("Mean positive slices per case:", float(summary_df["positive_slices"].mean()))
print("Min positive slices per case:", int(summary_df["positive_slices"].min()))
print("Max positive slices per case:", int(summary_df["positive_slices"].max()))

print("\nRecovery finished. Ready for smoke test.")

In [ ]:
from pathlib import Path

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

files_to_check = [
    OUT_DIR / "manifest_locked_original_40.csv",
    OUT_DIR / "manifest_locked_original_40_5fold.csv",
    OUT_DIR / "preprocessed_locked40_2d_summary.csv",
    OUT_DIR / "preprocessed_locked40_2d_npz",
]

for p in files_to_check:
    print(p, "EXISTS =", p.exists())

In [ ]:
import os, random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# =====================
# Config
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
PRE_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"
SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
FOLD_PATH = OUT_DIR / "manifest_locked_original_40_5fold.csv"

EXP_DIR = OUT_DIR / "independent_5fold_smoke"
EXP_DIR.mkdir(parents=True, exist_ok=True)

FOLD = 1
EPOCHS = 2
BATCH_SIZE = 8
LR = 1e-3
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

summary_df = pd.read_csv(SUMMARY_PATH)
fold_df = pd.read_csv(FOLD_PATH)
case_to_npz = dict(zip(summary_df["case_id"], summary_df["npz_path"]))

# =====================
# Dataset
# =====================
class SliceDataset(Dataset):
    def __init__(self, case_ids):
        xs, ys = [], []
        for case_id in case_ids:
            with np.load(case_to_npz[case_id], allow_pickle=True) as data:
                X = data["X"]
                Y = data["Y"]
                target_voxels = data["target_voxels_per_slice"]
                current_voxels = data["current_voxels_per_slice"]

                # Keep slices with either current tumour or future-change target.
                idx = np.where((target_voxels > 0) | (current_voxels > 0))[0]
                xs.append(X[idx])
                ys.append(Y[idx])

        self.X = np.concatenate(xs, axis=0)
        self.Y = np.concatenate(ys, axis=0)

        print("Loaded slices:", len(self.X))
        print("X:", self.X.shape, "Y:", self.Y.shape)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.X[idx]).float(),
            torch.from_numpy(self.Y[idx]).float()
        )

# =====================
# Small U-Net
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class SmallUNet(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)

        self.mid = ConvBlock(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        m = self.mid(self.pool(e3))

        d3 = self.up3(m)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)

# =====================
# Loss and metrics
# =====================
def dice_loss(logits, y, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p * y).sum(dim=(1, 2, 3))
    denom = p.sum(dim=(1, 2, 3)) + y.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()

def topk_metrics(pred, target, eps=1e-8):
    pred = pred.astype(np.float32)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {"dice": np.nan, "iou": np.nan, "target_focus": np.nan, "log10_ratio": np.nan}

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }

def predict_case(model, case_id):
    with np.load(case_to_npz[case_id], allow_pickle=True) as data:
        X = data["X"]
        Y = data["Y"][:, 0]

    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, len(X), BATCH_SIZE):
            xb = torch.from_numpy(X[i:i+BATCH_SIZE]).float().to(device)
            prob = torch.sigmoid(model(xb)).cpu().numpy()[:, 0]
            preds.append(prob)

    pred = np.concatenate(preds, axis=0)
    return pred, Y

# =====================
# Train Fold 1
# =====================
train_cases = fold_df[(fold_df["fold"] == FOLD) & (fold_df["split"] == "train")]["case_id"].tolist()
test_cases = fold_df[(fold_df["fold"] == FOLD) & (fold_df["split"] == "test")]["case_id"].tolist()

print("Train cases:", len(train_cases))
print("Test cases:", len(test_cases))

train_ds = SliceDataset(train_cases)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# Class imbalance handling
y_flat = train_ds.Y.reshape(-1)
pos = y_flat.sum()
neg = len(y_flat) - pos
pos_weight = min(float(neg / (pos + 1e-8)), 50.0)
print("pos:", int(pos), "neg:", int(neg), "pos_weight:", pos_weight)

model = SmallUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []

    for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)

        loss = 0.5 * bce(logits, yb) + 0.5 * dice_loss(logits, yb)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    print(f"Epoch {epoch} mean loss: {np.mean(losses):.6f}")

# =====================
# Evaluate Fold 1 test cases
# =====================
rows = []

for case_id in tqdm(test_cases, desc="Evaluating test cases"):
    pred, target = predict_case(model, case_id)
    m = topk_metrics(pred, target)

    pred_path = EXP_DIR / f"{case_id}_fold{FOLD}_smoke_pred.npy"
    np.save(pred_path, pred.astype(np.float16))

    rows.append({
        "fold": FOLD,
        "case_id": case_id,
        "pred_path": str(pred_path),
        **m
    })

result_df = pd.DataFrame(rows)
result_path = EXP_DIR / "fold1_smoke_metrics.csv"
result_df.to_csv(result_path, index=False)

print("\nSaved metrics:", result_path)
display(result_df)

print("\nMean metrics:")
display(result_df[["dice", "iou", "target_focus", "log10_ratio"]].mean())

In [ ]:
# =====================
# Formal 5-fold independent baseline
# =====================

FORMAL_EXP_DIR = OUT_DIR / "independent_5fold_formal_20epoch"
FORMAL_EXP_DIR.mkdir(parents=True, exist_ok=True)

FOLDS_TO_RUN = [1, 2, 3, 4, 5]
FORMAL_EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-3

all_rows = []
training_log_rows = []

for FOLD in FOLDS_TO_RUN:
    print("\n" + "=" * 100)
    print(f"FORMAL TRAINING: FOLD {FOLD}")
    print("=" * 100)

    fold_metric_path = FORMAL_EXP_DIR / f"fold{FOLD}_test_metrics.csv"
    fold_ckpt_path = FORMAL_EXP_DIR / f"fold{FOLD}_model.pt"

    if fold_metric_path.exists():
        print(f"Fold {FOLD} already completed. Loading existing metrics.")
        old_df = pd.read_csv(fold_metric_path)
        all_rows.extend(old_df.to_dict("records"))
        continue

    train_cases = fold_df[
        (fold_df["fold"] == FOLD) &
        (fold_df["split"] == "train")
    ]["case_id"].tolist()

    test_cases = fold_df[
        (fold_df["fold"] == FOLD) &
        (fold_df["split"] == "test")
    ]["case_id"].tolist()

    print("Train cases:", len(train_cases))
    print("Test cases:", len(test_cases))

    train_ds = SliceDataset(train_cases)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    # Class imbalance handling
    y_flat = train_ds.Y.reshape(-1)
    pos = y_flat.sum()
    neg = len(y_flat) - pos
    pos_weight = min(float(neg / (pos + 1e-8)), 50.0)

    print("pos:", int(pos), "neg:", int(neg), "pos_weight:", pos_weight)

    model = SmallUNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

    best_loss = float("inf")

    for epoch in range(1, FORMAL_EPOCHS + 1):
        model.train()
        losses = []

        for xb, yb in tqdm(train_loader, desc=f"Fold {FOLD} Epoch {epoch}/{FORMAL_EPOCHS}"):
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)

            loss = 0.5 * bce(logits, yb) + 0.5 * dice_loss(logits, yb)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        mean_loss = float(np.mean(losses))

        training_log_rows.append({
            "fold": FOLD,
            "epoch": epoch,
            "mean_loss": mean_loss,
            "pos_weight": pos_weight,
            "train_cases": len(train_cases),
            "train_slices": len(train_ds),
        })

        print(f"Fold {FOLD} Epoch {epoch}: mean loss = {mean_loss:.6f}")

        if mean_loss < best_loss:
            best_loss = mean_loss
            torch.save(model.state_dict(), fold_ckpt_path)

    print("Saved best checkpoint:", fold_ckpt_path)

    # Reload best model before evaluation
    model.load_state_dict(torch.load(fold_ckpt_path, map_location=device))
    model.eval()

    fold_rows = []

    for case_id in tqdm(test_cases, desc=f"Evaluating Fold {FOLD} test cases"):
        pred, target = predict_case(model, case_id)
        m = topk_metrics(pred, target)

        pred_path = FORMAL_EXP_DIR / f"{case_id}_fold{FOLD}_independent_pred.npy"
        np.save(pred_path, pred.astype(np.float16))

        row = {
            "fold": FOLD,
            "case_id": case_id,
            "pred_path": str(pred_path),
            **m
        }

        fold_rows.append(row)
        all_rows.append(row)

    fold_df_result = pd.DataFrame(fold_rows)
    fold_df_result.to_csv(fold_metric_path, index=False)

    print(f"\nSaved fold {FOLD} metrics:", fold_metric_path)
    display(fold_df_result)

# =====================
# Save all results
# =====================
all_result_df = pd.DataFrame(all_rows)

all_metric_path = FORMAL_EXP_DIR / "independent_5fold_40case_metrics.csv"
all_result_df.to_csv(all_metric_path, index=False)

training_log_df = pd.DataFrame(training_log_rows)
training_log_path = FORMAL_EXP_DIR / "training_log.csv"
training_log_df.to_csv(training_log_path, index=False)

print("\n" + "=" * 100)
print("FORMAL 5-FOLD TRAINING FINISHED")
print("=" * 100)

print("Saved all metrics:", all_metric_path)
print("Saved training log:", training_log_path)

print("\nCases evaluated:", len(all_result_df))
print("Unique cases:", all_result_df["case_id"].nunique())

print("\nOverall mean metrics:")
display(all_result_df[["dice", "iou", "target_focus", "log10_ratio"]].mean())

print("\nOverall median metrics:")
display(all_result_df[["dice", "iou", "target_focus", "log10_ratio"]].median())

print("\nPer-fold mean metrics:")
display(
    all_result_df
    .groupby("fold")[["dice", "iou", "target_focus", "log10_ratio"]]
    .mean()
)

In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
FORMAL_EXP_DIR = OUT_DIR / "independent_5fold_formal_20epoch"

ind_path = FORMAL_EXP_DIR / "independent_5fold_40case_metrics.csv"
ind = pd.read_csv(ind_path)

ind_mean = ind[["dice", "iou", "target_focus", "log10_ratio"]].mean()
ind_median = ind[["dice", "iou", "target_focus", "log10_ratio"]].median()

comparison = pd.DataFrame([
    {
        "Method": "Fixed baseline",
        "Used_as_PCC_starting_map": "Yes",
        "Dice_mean": 0.542112,
        "IoU_mean": 0.381554,
        "Target_focus_mean": 0.338537,
        "Log10_ratio_mean": 1.216147,
    },
    {
        "Method": "Naive self-tightening",
        "Used_as_PCC_starting_map": "Post-processing of baseline",
        "Dice_mean": 0.400270,
        "IoU_mean": 0.259192,
        "Target_focus_mean": 0.689656,
        "Log10_ratio_mean": 2.018870,
    },
    {
        "Method": "Independent 5-fold direct prediction baseline",
        "Used_as_PCC_starting_map": "No",
        "Dice_mean": float(ind_mean["dice"]),
        "IoU_mean": float(ind_mean["iou"]),
        "Target_focus_mean": float(ind_mean["target_focus"]),
        "Log10_ratio_mean": float(ind_mean["log10_ratio"]),
    },
    {
        "Method": "PCC correction",
        "Used_as_PCC_starting_map": "Final corrected method",
        "Dice_mean": 0.670115,
        "IoU_mean": 0.513793,
        "Target_focus_mean": 0.788182,
        "Log10_ratio_mean": 2.235115,
    },
])

comparison_path = FORMAL_EXP_DIR / "final_method_comparison_with_independent_baseline.csv"
comparison.to_csv(comparison_path, index=False)

print("Saved:", comparison_path)
display(comparison)

print("\nIndependent baseline median metrics:")
display(ind_median)

In [ ]:
# ============================================================
# Version A: Expanded T1c + mask independent baseline, 20 epochs
# Train: non-locked MU-Glioma-Post longitudinal pairs
# Test: locked original 40 cases
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import OrderedDict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib

# =====================
# Config
# =====================
DATA_ROOT = Path("/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post")
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
LOCKED_PRE_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"

EXP_DIR = OUT_DIR / "expanded_t1c_mask_baseline_20epoch"
TRAIN_PRE_DIR = EXP_DIR / "preprocessed_expanded_train_npz"
EXP_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PRE_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-3
SEED = 42
MIN_GROWTH_VOXELS = 1000   # remove almost-empty future-change targets
BASE_CHANNELS = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("DATA_ROOT exists:", DATA_ROOT.exists())
print("LOCKED_MANIFEST exists:", LOCKED_MANIFEST_PATH.exists())
print("LOCKED_PRE_SUMMARY exists:", LOCKED_PRE_SUMMARY_PATH.exists())
print("LOCKED_PRE_DIR exists:", LOCKED_PRE_DIR.exists())

assert DATA_ROOT.exists(), "DATA_ROOT not found."
assert LOCKED_MANIFEST_PATH.exists(), "Locked 40 manifest missing. Run recovery cell first."
assert LOCKED_PRE_SUMMARY_PATH.exists(), "Locked 40 preprocessing summary missing. Run recovery cell first."
assert LOCKED_PRE_DIR.exists(), "Locked 40 preprocessed npz folder missing. Run recovery cell first."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =====================
# Build all-pair manifest if missing
# =====================
def timepoint_number(tp_name):
    m = re.search(r"Timepoint_(\d+)", tp_name)
    return int(m.group(1)) if m else None

def find_required_files(tp_dir):
    files = list(tp_dir.glob("*.nii"))
    t1c = [f for f in files if "_brain_t1c.nii" in f.name]
    mask = [f for f in files if "_tumorMask.nii" in f.name]
    return {
        "t1c": str(t1c[0]) if len(t1c) == 1 else None,
        "mask": str(mask[0]) if len(mask) == 1 else None,
    }

manifest_all_path = OUT_DIR / "manifest_all_pairs.csv"

if manifest_all_path.exists():
    manifest_all = pd.read_csv(manifest_all_path)
    print("Loaded manifest_all:", manifest_all.shape)
else:
    rows = []
    patient_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith("PatientID_")])
    
    for patient_dir in patient_dirs:
        tp_dirs = sorted(
            [p for p in patient_dir.iterdir() if p.is_dir() and p.name.startswith("Timepoint_")],
            key=lambda x: timepoint_number(x.name)
        )
        
        usable_tps = []
        for tp_dir in tp_dirs:
            files = find_required_files(tp_dir)
            tp_num = timepoint_number(tp_dir.name)
            if tp_num is not None and files["t1c"] is not None and files["mask"] is not None:
                usable_tps.append({
                    "tp_num": tp_num,
                    "tp_name": tp_dir.name,
                    "t1c": files["t1c"],
                    "mask": files["mask"],
                })
        
        for i in range(len(usable_tps) - 1):
            cur = usable_tps[i]
            fut = usable_tps[i + 1]
            rows.append({
                "case_id": f"{patient_dir.name}_T{cur['tp_num']}_to_T{fut['tp_num']}_t1c",
                "patient_id": patient_dir.name,
                "current_timepoint": cur["tp_name"],
                "future_timepoint": fut["tp_name"],
                "current_tp_num": cur["tp_num"],
                "future_tp_num": fut["tp_num"],
                "current_t1c_path": cur["t1c"],
                "current_mask_path": cur["mask"],
                "future_t1c_path": fut["t1c"],
                "future_mask_path": fut["mask"],
            })
    
    manifest_all = pd.DataFrame(rows)
    manifest_all.to_csv(manifest_all_path, index=False)
    print("Created manifest_all:", manifest_all.shape)

locked_40 = pd.read_csv(LOCKED_MANIFEST_PATH)
locked_patients = set(locked_40["patient_id"].tolist())

# Exclude all locked patients from expanded training.
expanded_candidates = manifest_all[~manifest_all["patient_id"].isin(locked_patients)].copy()
expanded_candidates = expanded_candidates.reset_index(drop=True)

print("\nExpanded candidate pairs before filtering:", len(expanded_candidates))
print("Expanded candidate patients:", expanded_candidates["patient_id"].nunique())

# =====================
# Compute train-pair growth stats
# =====================
train_stats_path = EXP_DIR / "expanded_train_pair_stats.csv"

def load_mask(path):
    return nib.load(path).get_fdata() > 0.5

if train_stats_path.exists():
    train_stats = pd.read_csv(train_stats_path)
    print("Loaded existing train stats:", train_stats.shape)
else:
    stat_rows = []
    
    for _, row in tqdm(expanded_candidates.iterrows(), total=len(expanded_candidates), desc="Computing expanded train stats"):
        try:
            cur_mask = load_mask(row["current_mask_path"])
            fut_mask = load_mask(row["future_mask_path"])
            
            shape_ok = cur_mask.shape == fut_mask.shape
            if shape_ok:
                growth = np.logical_and(fut_mask, np.logical_not(cur_mask))
                shrink = np.logical_and(cur_mask, np.logical_not(fut_mask))
                current_voxels = int(cur_mask.sum())
                future_voxels = int(fut_mask.sum())
                growth_voxels = int(growth.sum())
                shrink_voxels = int(shrink.sum())
            else:
                current_voxels = future_voxels = growth_voxels = shrink_voxels = np.nan
            
            r = row.to_dict()
            r.update({
                "shape_ok": shape_ok,
                "shape": str(cur_mask.shape),
                "current_voxels": current_voxels,
                "future_voxels": future_voxels,
                "growth_voxels": growth_voxels,
                "shrink_voxels": shrink_voxels,
                "net_change_voxels": future_voxels - current_voxels if shape_ok else np.nan,
            })
            stat_rows.append(r)
        except Exception as e:
            r = row.to_dict()
            r.update({
                "shape_ok": False,
                "error": str(e),
            })
            stat_rows.append(r)
    
    train_stats = pd.DataFrame(stat_rows)
    train_stats.to_csv(train_stats_path, index=False)
    print("Saved train stats:", train_stats_path)

expanded_train = train_stats[
    (train_stats["shape_ok"] == True) &
    (train_stats["growth_voxels"] >= MIN_GROWTH_VOXELS)
].copy()

expanded_train = expanded_train.sort_values(["patient_id", "current_tp_num", "future_tp_num"]).reset_index(drop=True)

expanded_train_path = EXP_DIR / "expanded_train_manifest_nonlocked.csv"
expanded_train.to_csv(expanded_train_path, index=False)

print("\nExpanded train pairs after filtering:", len(expanded_train))
print("Expanded train patients:", expanded_train["patient_id"].nunique())
print("Growth voxel summary:")
print(expanded_train["growth_voxels"].describe())

# =====================
# Preprocess expanded train pairs into relevant 2D slices
# =====================
def robust_normalize_t1c(img):
    img = img.astype(np.float32)
    nonzero = img[img > 0]
    if nonzero.size < 100:
        return np.zeros_like(img, dtype=np.float32)
    
    p1, p99 = np.percentile(nonzero, [1, 99])
    if p99 <= p1:
        return np.zeros_like(img, dtype=np.float32)
    
    img = np.clip(img, p1, p99)
    img = (img - p1) / (p99 - p1 + 1e-8)
    img = np.clip(img, 0, 1)
    return img.astype(np.float32)

train_summary_path = EXP_DIR / "expanded_train_preprocessed_summary.csv"

if train_summary_path.exists():
    train_summary = pd.read_csv(train_summary_path)
    print("\nLoaded existing train preprocessing summary:", train_summary.shape)
else:
    summary_rows = []
    
    for _, row in tqdm(expanded_train.iterrows(), total=len(expanded_train), desc="Preprocessing expanded train pairs"):
        case_id = row["case_id"]
        npz_path = TRAIN_PRE_DIR / f"{case_id}.npz"
        
        if npz_path.exists():
            # Still record metadata if file exists.
            with np.load(npz_path, allow_pickle=True) as data:
                X = data["X"]
                Y = data["Y"]
                target_voxels_per_slice = data["target_voxels_per_slice"]
                current_voxels_per_slice = data["current_voxels_per_slice"]
            
            summary_rows.append({
                "case_id": case_id,
                "patient_id": row["patient_id"],
                "npz_path": str(npz_path),
                "num_slices": int(X.shape[0]),
                "growth_voxels": int(row["growth_voxels"]),
                "positive_slices": int((target_voxels_per_slice > 0).sum()),
                "current_mask_slices": int((current_voxels_per_slice > 0).sum()),
            })
            continue
        
        current_t1c = nib.load(row["current_t1c_path"]).get_fdata().astype(np.float32)
        current_mask = nib.load(row["current_mask_path"]).get_fdata() > 0.5
        future_mask = nib.load(row["future_mask_path"]).get_fdata() > 0.5
        
        current_t1c_norm = robust_normalize_t1c(current_t1c)
        future_change = np.logical_and(future_mask, np.logical_not(current_mask))
        
        H, W, Z = current_t1c_norm.shape
        
        target_voxels_full = np.array([future_change[:, :, z].sum() for z in range(Z)])
        current_voxels_full = np.array([current_mask[:, :, z].sum() for z in range(Z)])
        
        # Keep relevant slices only to reduce disk/RAM and focus training.
        idx = np.where((target_voxels_full > 0) | (current_voxels_full > 0))[0]
        
        X = np.zeros((len(idx), 2, H, W), dtype=np.float16)
        Y = np.zeros((len(idx), 1, H, W), dtype=np.uint8)
        
        for j, z in enumerate(idx):
            X[j, 0] = current_t1c_norm[:, :, z].astype(np.float16)
            X[j, 1] = current_mask[:, :, z].astype(np.float16)
            Y[j, 0] = future_change[:, :, z].astype(np.uint8)
        
        target_voxels_per_slice = Y.reshape(len(idx), -1).sum(axis=1)
        current_voxels_per_slice = X[:, 1].reshape(len(idx), -1).sum(axis=1)
        
        np.savez(
            npz_path,
            X=X,
            Y=Y,
            z_indices=idx,
            target_voxels_per_slice=target_voxels_per_slice,
            current_voxels_per_slice=current_voxels_per_slice,
            case_id=case_id,
            patient_id=row["patient_id"],
        )
        
        summary_rows.append({
            "case_id": case_id,
            "patient_id": row["patient_id"],
            "npz_path": str(npz_path),
            "num_slices": int(X.shape[0]),
            "growth_voxels": int(row["growth_voxels"]),
            "positive_slices": int((target_voxels_per_slice > 0).sum()),
            "current_mask_slices": int((current_voxels_per_slice > 0).sum()),
        })
    
    train_summary = pd.DataFrame(summary_rows)
    train_summary.to_csv(train_summary_path, index=False)
    print("Saved train preprocessing summary:", train_summary_path)

print("\nExpanded train preprocessing summary:")
print("Pairs:", len(train_summary))
print("Total relevant slices:", int(train_summary["num_slices"].sum()))
print("Total positive slices:", int(train_summary["positive_slices"].sum()))
print("Mean relevant slices per pair:", float(train_summary["num_slices"].mean()))

# =====================
# Dataset: load expanded training slices into RAM
# =====================
class ExpandedSliceDataset(Dataset):
    def __init__(self, summary_df):
        xs, ys = [], []
        for _, row in tqdm(summary_df.iterrows(), total=len(summary_df), desc="Loading expanded train npz into RAM"):
            with np.load(row["npz_path"], allow_pickle=True) as data:
                xs.append(data["X"])
                ys.append(data["Y"])
        
        self.X = np.concatenate(xs, axis=0)
        self.Y = np.concatenate(ys, axis=0)
        
        print("Training slices loaded:", len(self.X))
        print("X:", self.X.shape, "Y:", self.Y.shape)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.X[idx]).float(),
            torch.from_numpy(self.Y[idx]).float()
        )

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class SmallUNet(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m = self.mid(self.pool(e3))
        d3 = self.up3(m)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out(d1)

def dice_loss(logits, y, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p * y).sum(dim=(1, 2, 3))
    denom = p.sum(dim=(1, 2, 3)) + y.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()

# =====================
# Train one expanded model
# =====================
train_ds = ExpandedSliceDataset(train_summary)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

y_flat = train_ds.Y.reshape(-1)
pos = y_flat.sum()
neg = len(y_flat) - pos
pos_weight = min(float(neg / (pos + 1e-8)), 50.0)

print("Positive pixels:", int(pos))
print("Negative pixels:", int(neg))
print("pos_weight:", pos_weight)

model = SmallUNet(in_ch=2, out_ch=1, base=BASE_CHANNELS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

training_log = []
ckpt_path = EXP_DIR / "expanded_t1c_mask_model_best.pt"
last_ckpt_path = EXP_DIR / "expanded_t1c_mask_model_last.pt"

best_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    
    for xb, yb in tqdm(train_loader, desc=f"Expanded baseline epoch {epoch}/{EPOCHS}"):
        xb = xb.to(device)
        yb = yb.to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(xb)
            loss = 0.5 * bce(logits, yb) + 0.5 * dice_loss(logits, yb)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        losses.append(float(loss.item()))
    
    mean_loss = float(np.mean(losses))
    training_log.append({
        "epoch": epoch,
        "mean_loss": mean_loss,
        "pos_weight": pos_weight,
        "train_pairs": len(train_summary),
        "train_slices": len(train_ds),
    })
    
    print(f"Epoch {epoch}: mean loss = {mean_loss:.6f}")
    
    torch.save(model.state_dict(), last_ckpt_path)
    
    if mean_loss < best_loss:
        best_loss = mean_loss
        torch.save(model.state_dict(), ckpt_path)
        print("Saved best checkpoint:", ckpt_path)

training_log_df = pd.DataFrame(training_log)
training_log_path = EXP_DIR / "expanded_training_log.csv"
training_log_df.to_csv(training_log_path, index=False)

print("Training finished.")
print("Saved training log:", training_log_path)

# =====================
# Evaluate locked original 40 cases
# =====================
locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
locked_case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

def topk_metrics(pred, target, eps=1e-8):
    pred = pred.astype(np.float32)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {"dice": np.nan, "iou": np.nan, "target_focus": np.nan, "log10_ratio": np.nan}
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }

def predict_locked_case(model, case_id):
    with np.load(locked_case_to_npz[case_id], allow_pickle=True) as data:
        X = data["X"]
        Y = data["Y"][:, 0]
    
    model.eval()
    preds = []
    
    with torch.no_grad():
        for i in range(0, len(X), BATCH_SIZE):
            xb = torch.from_numpy(X[i:i+BATCH_SIZE]).float().to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                prob = torch.sigmoid(model(xb)).cpu().numpy()[:, 0]
            preds.append(prob)
    
    pred = np.concatenate(preds, axis=0)
    return pred, Y

# Load best checkpoint
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

rows = []
locked_cases = locked_40["case_id"].tolist()

for case_id in tqdm(locked_cases, desc="Evaluating expanded baseline on locked 40"):
    pred, target = predict_locked_case(model, case_id)
    m = topk_metrics(pred, target)
    
    pred_path = EXP_DIR / f"{case_id}_expanded_t1c_mask_pred.npy"
    np.save(pred_path, pred.astype(np.float16))
    
    rows.append({
        "case_id": case_id,
        "pred_path": str(pred_path),
        **m
    })

metrics_df = pd.DataFrame(rows)
metrics_path = EXP_DIR / "expanded_t1c_mask_locked40_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\nSaved locked-40 metrics:", metrics_path)
display(metrics_df)

print("\nExpanded T1c+mask baseline mean metrics:")
display(metrics_df[["dice", "iou", "target_focus", "log10_ratio"]].mean())

print("\nExpanded T1c+mask baseline median metrics:")
display(metrics_df[["dice", "iou", "target_focus", "log10_ratio"]].median())

# =====================
# Comparison table
# =====================
comparison = pd.DataFrame([
    {
        "Method": "Fixed baseline",
        "Used_as_PCC_starting_map": "Yes",
        "Dice_mean": 0.542112,
        "IoU_mean": 0.381554,
        "Target_focus_mean": 0.338537,
        "Log10_ratio_mean": 1.216147,
    },
    {
        "Method": "Naive self-tightening",
        "Used_as_PCC_starting_map": "Post-processing of baseline",
        "Dice_mean": 0.400270,
        "IoU_mean": 0.259192,
        "Target_focus_mean": 0.689656,
        "Log10_ratio_mean": 2.018870,
    },
    {
        "Method": "Strict locked-40 independent 5-fold baseline",
        "Used_as_PCC_starting_map": "No",
        "Dice_mean": 0.288115,
        "IoU_mean": 0.180704,
        "Target_focus_mean": 0.217837,
        "Log10_ratio_mean": 2.036753,
    },
    {
        "Method": "Expanded T1c+mask independent baseline",
        "Used_as_PCC_starting_map": "No",
        "Dice_mean": float(metrics_df["dice"].mean()),
        "IoU_mean": float(metrics_df["iou"].mean()),
        "Target_focus_mean": float(metrics_df["target_focus"].mean()),
        "Log10_ratio_mean": float(metrics_df["log10_ratio"].mean()),
    },
    {
        "Method": "PCC correction",
        "Used_as_PCC_starting_map": "Final corrected method",
        "Dice_mean": 0.670115,
        "IoU_mean": 0.513793,
        "Target_focus_mean": 0.788182,
        "Log10_ratio_mean": 2.235115,
    },
])

comparison_path = EXP_DIR / "comparison_with_expanded_t1c_mask_baseline.csv"
comparison.to_csv(comparison_path, index=False)

print("\nSaved comparison table:", comparison_path)
display(comparison)

In [ ]:
from pathlib import Path

search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

patterns = [
    "**/model_baseline_map_for_pcc.nii",
    "**/model_baseline_map_for_pcc.nii.gz",
    "**/*baseline*map*.nii",
    "**/*baseline*map*.nii.gz",
]

found = []

for root in search_roots:
    if root.exists():
        for pattern in patterns:
            found.extend(list(root.glob(pattern)))

found = sorted(set(found))

print("Found candidate baseline maps:", len(found))
for p in found[:50]:
    print(p)

if len(found) == 0:
    print("\nWARNING: 没找到原始 fixed baseline map。")
    print("EIA Linear 需要原 PCC 实验输出里的 model_baseline_map_for_pcc.nii。")
    print("如果 Kaggle 里没有这些文件，需要先上传原 PCC 输出文件夹或 zip。")

In [ ]:
# ============================================================
# Recovery cell: rebuild locked original 40 manifest + 2D preprocessing
# ============================================================

import re
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib

# ---------------------
# Find dataset root
# ---------------------
candidate_roots = [
    Path("/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post"),
    Path("/kaggle/input"),
]

DATA_ROOT = None

for p in candidate_roots:
    if p.exists() and p.name == "MU-Glioma-Post":
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    candidates = list(Path("/kaggle/input").glob("**/MU-Glioma-Post"))
    candidates = [p for p in candidates if p.is_dir()]
    if len(candidates) > 0:
        DATA_ROOT = candidates[0]

print("DATA_ROOT:", DATA_ROOT)
assert DATA_ROOT is not None and DATA_ROOT.exists(), "没有找到 MU-Glioma-Post 数据集路径。"

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
PRE_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PRE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------
# Locked original 40 case IDs
# ---------------------
LOCKED_CASE_IDS = [
    "PatientID_0003_T1_to_T2_t1c",
    "PatientID_0005_T3_to_T4_t1c",
    "PatientID_0006_T2_to_T4_t1c",
    "PatientID_0007_T2_to_T3_t1c",
    "PatientID_0008_T4_to_T6_t1c",
    "PatientID_0010_T1_to_T4_t1c",
    "PatientID_0011_T1_to_T2_t1c",
    "PatientID_0012_T2_to_T3_t1c",
    "PatientID_0013_T1_to_T2_t1c",
    "PatientID_0014_T1_to_T2_t1c",
    "PatientID_0018_T1_to_T2_t1c",
    "PatientID_0019_T4_to_T5_t1c",
    "PatientID_0020_T1_to_T2_t1c",
    "PatientID_0021_T2_to_T3_t1c",
    "PatientID_0022_T1_to_T2_t1c",
    "PatientID_0024_T2_to_T3_t1c",
    "PatientID_0025_T1_to_T2_t1c",
    "PatientID_0026_T1_to_T2_t1c",
    "PatientID_0029_T1_to_T3_t1c",
    "PatientID_0030_T1_to_T3_t1c",
    "PatientID_0031_T2_to_T3_t1c",
    "PatientID_0032_T1_to_T2_t1c",
    "PatientID_0033_T1_to_T2_t1c",
    "PatientID_0034_T1_to_T2_t1c",
    "PatientID_0035_T1_to_T2_t1c",
    "PatientID_0036_T1_to_T2_t1c",
    "PatientID_0037_T1_to_T2_t1c",
    "PatientID_0038_T1_to_T2_t1c",
    "PatientID_0039_T1_to_T2_t1c",
    "PatientID_0041_T1_to_T2_t1c",
    "PatientID_0044_T1_to_T2_t1c",
    "PatientID_0045_T1_to_T2_t1c",
    "PatientID_0051_T1_to_T3_t1c",
    "PatientID_0052_T1_to_T2_t1c",
    "PatientID_0053_T1_to_T3_t1c",
    "PatientID_0054_T1_to_T2_t1c",
    "PatientID_0055_T1_to_T4_t1c",
    "PatientID_0059_T1_to_T2_t1c",
    "PatientID_0060_T1_to_T2_t1c",
    "PatientID_0062_T1_to_T2_t1c",
]

# ---------------------
# Build all longitudinal pairs
# ---------------------
def timepoint_number(tp_name):
    m = re.search(r"Timepoint_(\d+)", tp_name)
    return int(m.group(1)) if m else None

def find_files(tp_dir):
    files = list(tp_dir.glob("*.nii"))
    t1c = [f for f in files if "_brain_t1c.nii" in f.name]
    mask = [f for f in files if "_tumorMask.nii" in f.name]
    return {
        "t1c": str(t1c[0]) if len(t1c) == 1 else None,
        "mask": str(mask[0]) if len(mask) == 1 else None,
    }

rows = []

patient_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith("PatientID_")])

for patient_dir in patient_dirs:
    tp_dirs = sorted(
        [p for p in patient_dir.iterdir() if p.is_dir() and p.name.startswith("Timepoint_")],
        key=lambda x: timepoint_number(x.name)
    )
    
    usable = []
    for tp_dir in tp_dirs:
        tp_num = timepoint_number(tp_dir.name)
        files = find_files(tp_dir)
        if tp_num is not None and files["t1c"] and files["mask"]:
            usable.append({
                "tp_num": tp_num,
                "tp_name": tp_dir.name,
                "t1c": files["t1c"],
                "mask": files["mask"],
            })
    
    for i in range(len(usable) - 1):
        cur = usable[i]
        fut = usable[i + 1]
        case_id = f"{patient_dir.name}_T{cur['tp_num']}_to_T{fut['tp_num']}_t1c"
        rows.append({
            "case_id": case_id,
            "patient_id": patient_dir.name,
            "current_timepoint": cur["tp_name"],
            "future_timepoint": fut["tp_name"],
            "current_tp_num": cur["tp_num"],
            "future_tp_num": fut["tp_num"],
            "current_t1c_path": cur["t1c"],
            "current_mask_path": cur["mask"],
            "future_t1c_path": fut["t1c"],
            "future_mask_path": fut["mask"],
        })

manifest_all = pd.DataFrame(rows)
manifest_all.to_csv(OUT_DIR / "manifest_all_pairs.csv", index=False)

locked = manifest_all[manifest_all["case_id"].isin(LOCKED_CASE_IDS)].copy()

# preserve locked order
locked["locked_order"] = locked["case_id"].apply(lambda x: LOCKED_CASE_IDS.index(x))
locked = locked.sort_values("locked_order").drop(columns=["locked_order"]).reset_index(drop=True)

print("All pairs:", len(manifest_all))
print("Locked matched:", len(locked), "/ 40")

missing = [c for c in LOCKED_CASE_IDS if c not in set(locked["case_id"])]
if missing:
    print("Missing locked cases:")
    for c in missing:
        print(c)

assert len(locked) == 40, "没有匹配到全部 locked 40 cases。"

locked_path = OUT_DIR / "manifest_locked_original_40.csv"
locked.to_csv(locked_path, index=False)
print("Saved:", locked_path)

# ---------------------
# Preprocess locked 40 into 2D npz
# ---------------------
def robust_normalize_t1c(img):
    img = img.astype(np.float32)
    nonzero = img[img > 0]
    if nonzero.size < 100:
        return np.zeros_like(img, dtype=np.float32)
    
    p1, p99 = np.percentile(nonzero, [1, 99])
    if p99 <= p1:
        return np.zeros_like(img, dtype=np.float32)
    
    img = np.clip(img, p1, p99)
    img = (img - p1) / (p99 - p1 + 1e-8)
    return np.clip(img, 0, 1).astype(np.float32)

summary_rows = []

for _, row in tqdm(locked.iterrows(), total=len(locked), desc="Preprocessing locked 40"):
    case_id = row["case_id"]
    npz_path = PRE_DIR / f"{case_id}.npz"
    
    current_t1c = nib.load(row["current_t1c_path"]).get_fdata().astype(np.float32)
    current_mask = nib.load(row["current_mask_path"]).get_fdata() > 0.5
    future_mask = nib.load(row["future_mask_path"]).get_fdata() > 0.5
    
    assert current_mask.shape == future_mask.shape, f"Mask shape mismatch: {case_id}"
    
    current_t1c_norm = robust_normalize_t1c(current_t1c)
    future_change = np.logical_and(future_mask, np.logical_not(current_mask))
    
    H, W, Z = current_t1c_norm.shape
    
    target_voxels_full = np.array([future_change[:, :, z].sum() for z in range(Z)])
    current_voxels_full = np.array([current_mask[:, :, z].sum() for z in range(Z)])
    
    # keep slices with current tumour or future-change signal
    idx = np.where((target_voxels_full > 0) | (current_voxels_full > 0))[0]
    
    X = np.zeros((len(idx), 2, H, W), dtype=np.float16)
    Y = np.zeros((len(idx), 1, H, W), dtype=np.uint8)
    
    for j, z in enumerate(idx):
        X[j, 0] = current_t1c_norm[:, :, z].astype(np.float16)
        X[j, 1] = current_mask[:, :, z].astype(np.float16)
        Y[j, 0] = future_change[:, :, z].astype(np.uint8)
    
    target_voxels_per_slice = Y.reshape(len(idx), -1).sum(axis=1)
    current_voxels_per_slice = X[:, 1].reshape(len(idx), -1).sum(axis=1)
    
    np.savez(
        npz_path,
        X=X,
        Y=Y,
        z_indices=idx,
        target_voxels_per_slice=target_voxels_per_slice,
        current_voxels_per_slice=current_voxels_per_slice,
        case_id=case_id,
        patient_id=row["patient_id"],
    )
    
    summary_rows.append({
        "case_id": case_id,
        "patient_id": row["patient_id"],
        "npz_path": str(npz_path),
        "num_slices": int(X.shape[0]),
        "growth_voxels": int(future_change.sum()),
        "positive_slices": int((target_voxels_per_slice > 0).sum()),
        "current_mask_slices": int((current_voxels_per_slice > 0).sum()),
    })

summary = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)
print("Cases:", len(summary))
print("Total slices:", int(summary["num_slices"].sum()))
print("Total positive slices:", int(summary["positive_slices"].sum()))
display(summary.head())

In [ ]:
# ============================================================
# Corrected preprocessing: keep ALL slices for locked original 40
# Expected total slices = 6200
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
PRE_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"
LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"

assert LOCKED_MANIFEST_PATH.exists(), "找不到 manifest_locked_original_40.csv，请先运行 recovery cell。"

PRE_DIR.mkdir(parents=True, exist_ok=True)

locked = pd.read_csv(LOCKED_MANIFEST_PATH)

def robust_normalize_t1c(img):
    img = img.astype(np.float32)
    nonzero = img[img > 0]
    if nonzero.size < 100:
        return np.zeros_like(img, dtype=np.float32)
    
    p1, p99 = np.percentile(nonzero, [1, 99])
    if p99 <= p1:
        return np.zeros_like(img, dtype=np.float32)
    
    img = np.clip(img, p1, p99)
    img = (img - p1) / (p99 - p1 + 1e-8)
    return np.clip(img, 0, 1).astype(np.float32)

summary_rows = []

for _, row in tqdm(locked.iterrows(), total=len(locked), desc="Corrected preprocessing locked 40"):
    case_id = row["case_id"]
    npz_path = PRE_DIR / f"{case_id}.npz"
    
    current_t1c = nib.load(row["current_t1c_path"]).get_fdata().astype(np.float32)
    current_mask = nib.load(row["current_mask_path"]).get_fdata() > 0.5
    future_mask = nib.load(row["future_mask_path"]).get_fdata() > 0.5
    
    assert current_mask.shape == future_mask.shape, f"Mask shape mismatch: {case_id}"
    
    current_t1c_norm = robust_normalize_t1c(current_t1c)
    future_change = np.logical_and(future_mask, np.logical_not(current_mask))
    
    H, W, Z = current_t1c_norm.shape
    
    # Important: keep ALL slices, not only relevant slices
    idx = np.arange(Z)
    
    X = np.zeros((Z, 2, H, W), dtype=np.float16)
    Y = np.zeros((Z, 1, H, W), dtype=np.uint8)
    
    for j, z in enumerate(idx):
        X[j, 0] = current_t1c_norm[:, :, z].astype(np.float16)
        X[j, 1] = current_mask[:, :, z].astype(np.float16)
        Y[j, 0] = future_change[:, :, z].astype(np.uint8)
    
    target_voxels_per_slice = Y.reshape(Z, -1).sum(axis=1)
    current_voxels_per_slice = X[:, 1].reshape(Z, -1).sum(axis=1)
    
    np.savez(
        npz_path,
        X=X,
        Y=Y,
        z_indices=idx,
        target_voxels_per_slice=target_voxels_per_slice,
        current_voxels_per_slice=current_voxels_per_slice,
        case_id=case_id,
        patient_id=row["patient_id"],
    )
    
    summary_rows.append({
        "case_id": case_id,
        "patient_id": row["patient_id"],
        "npz_path": str(npz_path),
        "num_slices": int(X.shape[0]),
        "growth_voxels": int(future_change.sum()),
        "positive_slices": int((target_voxels_per_slice > 0).sum()),
        "current_mask_slices": int((current_voxels_per_slice > 0).sum()),
    })

summary = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
summary.to_csv(summary_path, index=False)

print("Saved corrected summary:", summary_path)
print("Cases:", len(summary))
print("Total slices:", int(summary["num_slices"].sum()))
print("Total positive slices:", int(summary["positive_slices"].sum()))
display(summary.head())

In [ ]:
from pathlib import Path
import shutil

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
RECOMP_DIR = OUT_DIR / "recomputed_case_specific_fixed_baseline_20epoch"

if RECOMP_DIR.exists():
    shutil.rmtree(RECOMP_DIR)
    print("Deleted old recomputed baseline folder:", RECOMP_DIR)
else:
    print("No old recomputed baseline folder found.")

print("Ready to rerun recomputed fixed baseline.")

In [ ]:
# ============================================================
# Recompute case-specific fixed baseline maps for EIA Linear
# Each case trains its own small 2D U-Net
# ============================================================

import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# =====================
# Config
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
LOCKED_PRE_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"

RECOMP_DIR = OUT_DIR / "recomputed_case_specific_fixed_baseline_20epoch"
MAP_DIR = RECOMP_DIR / "recomputed_fixed_baseline_maps"
CKPT_DIR = RECOMP_DIR / "checkpoints"

RECOMP_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-3
BASE_CHANNELS = 16
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert LOCKED_MANIFEST_PATH.exists(), "缺少 manifest_locked_original_40.csv，请先运行 recovery cell。"
assert LOCKED_PRE_SUMMARY_PATH.exists(), "缺少 preprocessed_locked40_2d_summary.csv，请先运行 preprocessing/recovery cell。"
assert LOCKED_PRE_DIR.exists(), "缺少 preprocessed_locked40_2d_npz 文件夹，请先运行 preprocessing/recovery cell。"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

locked_40 = pd.read_csv(LOCKED_MANIFEST_PATH)
locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)

case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

print("Locked cases:", len(locked_40))
print("Preprocessed cases:", len(case_to_npz))


# =====================
# Dataset
# =====================
class SingleCaseSliceDataset(Dataset):
    def __init__(self, npz_path):
        with np.load(npz_path, allow_pickle=True) as data:
            self.X = data["X"]
            self.Y = data["Y"]
        
        self.X = self.X.astype(np.float32)
        self.Y = self.Y.astype(np.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.X[idx]).float(),
            torch.from_numpy(self.Y[idx]).float()
        )


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.net(x)


class SmallUNet(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m = self.mid(self.pool(e3))
        
        d3 = self.up3(m)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        return self.out(d1)


def dice_loss(logits, y, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p * y).sum(dim=(1, 2, 3))
    denom = p.sum(dim=(1, 2, 3)) + y.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()


def topk_metrics(pred, target, eps=1e-8):
    pred = pred.astype(np.float32)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def predict_case(model, npz_path, batch_size=8):
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"][:, 0].astype(np.uint8)
    
    model.eval()
    preds = []
    
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).float().to(device)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                prob = torch.sigmoid(model(xb)).cpu().numpy()[:, 0]
            preds.append(prob)
    
    pred = np.concatenate(preds, axis=0)
    return pred.astype(np.float32), Y


# =====================
# Train one model per case
# =====================
all_metrics = []
training_rows = []

for case_idx, row in locked_40.iterrows():
    case_id = row["case_id"]
    
    npz_path = case_to_npz[case_id]
    pred_path = MAP_DIR / f"{case_id}_model_baseline_map_for_pcc_recomputed.npy"
    ckpt_path = CKPT_DIR / f"{case_id}_recomputed_fixed_baseline_model.pt"
    
    print("\n============================================================")
    print(f"[{case_idx+1}/40] Case: {case_id}")
    print("NPZ:", npz_path)
    
    # Resume support
    if pred_path.exists():
        print("Prediction already exists, skipping training:", pred_path)
        with np.load(npz_path, allow_pickle=True) as data:
            target = data["Y"][:, 0].astype(np.uint8)
        pred = np.load(pred_path).astype(np.float32)
        metrics = topk_metrics(pred, target)
        all_metrics.append({
            "case_id": case_id,
            "pred_path": str(pred_path),
            "ckpt_path": str(ckpt_path),
            **metrics
        })
        continue
    
    ds = SingleCaseSliceDataset(npz_path)
    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )
    
    y_flat = ds.Y.reshape(-1)
    pos = y_flat.sum()
    neg = len(y_flat) - pos
    pos_weight = min(float(neg / (pos + 1e-8)), 50.0)
    
    print("Slices:", len(ds))
    print("Positive pixels:", int(pos))
    print("Negative pixels:", int(neg))
    print("pos_weight:", pos_weight)
    
    model = SmallUNet(in_ch=2, out_ch=1, base=BASE_CHANNELS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    
    best_loss = float("inf")
    best_state = None
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            
            optimizer.zero_grad()
            
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(xb)
                loss = 0.5 * bce(logits, yb) + 0.5 * dice_loss(logits, yb)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            losses.append(float(loss.item()))
        
        mean_loss = float(np.mean(losses))
        
        training_rows.append({
            "case_id": case_id,
            "epoch": epoch,
            "mean_loss": mean_loss,
            "pos_weight": pos_weight,
            "slices": len(ds),
        })
        
        if mean_loss < best_loss:
            best_loss = mean_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        print(f"Epoch {epoch:02d}/{EPOCHS} | loss={mean_loss:.6f}")
    
    # Save best checkpoint
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt_path)
    
    # Predict and save map
    pred, target = predict_case(model, npz_path, batch_size=BATCH_SIZE)
    np.save(pred_path, pred.astype(np.float16))
    
    metrics = topk_metrics(pred, target)
    
    print("Case metrics:", metrics)
    
    all_metrics.append({
        "case_id": case_id,
        "pred_path": str(pred_path),
        "ckpt_path": str(ckpt_path),
        **metrics
    })
    
    # Save progress after each case
    pd.DataFrame(all_metrics).to_csv(RECOMP_DIR / "recomputed_fixed_baseline_case_metrics_partial.csv", index=False)
    pd.DataFrame(training_rows).to_csv(RECOMP_DIR / "recomputed_fixed_baseline_training_log_partial.csv", index=False)


# =====================
# Save final results
# =====================
metrics_df = pd.DataFrame(all_metrics)
metrics_path = RECOMP_DIR / "recomputed_fixed_baseline_case_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

training_log_df = pd.DataFrame(training_rows)
training_log_path = RECOMP_DIR / "recomputed_fixed_baseline_training_log.csv"
training_log_df.to_csv(training_log_path, index=False)

summary = metrics_df[["dice", "iou", "target_focus", "log10_ratio"]].agg(["mean", "median", "min", "max"])
summary_path = RECOMP_DIR / "recomputed_fixed_baseline_summary.csv"
summary.to_csv(summary_path)

print("\n============================================================")
print("Recomputed fixed baseline finished.")
print("Saved metrics:", metrics_path)
print("Saved training log:", training_log_path)
print("Saved summary:", summary_path)

print("\nRecomputed fixed baseline summary:")
display(summary)

print("\nCase metrics:")
display(metrics_df)

In [ ]:
# ============================================================
# Recomputed PCC correction on recomputed case-specific baseline maps
# Starting map: recomputed fixed baseline B'
# Correction signal: future-change target = future mask AND NOT current mask
# PCC settings: rounds={5,10,15}, eta=0.30, dilation=26, sigma=2.0
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

try:
    from scipy.ndimage import gaussian_filter, distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import gaussian_filter, distance_transform_edt


# =====================
# Paths and config
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

RECOMP_DIR = OUT_DIR / "recomputed_case_specific_fixed_baseline_20epoch"
MAP_DIR = RECOMP_DIR / "recomputed_fixed_baseline_maps"

PCC_DIR = OUT_DIR / "recomputed_pcc_correction_from_recomputed_baseline"
PCC_MAP_DIR = PCC_DIR / "pcc_maps"
PCC_DIR.mkdir(parents=True, exist_ok=True)
PCC_MAP_DIR.mkdir(parents=True, exist_ok=True)

ROUNDS_LIST = [5, 10, 15]
ETA = 0.30
DILATION_RADIUS = 26
SIGMA = 2.0

assert LOCKED_MANIFEST_PATH.exists(), "Missing manifest_locked_original_40.csv"
assert LOCKED_PRE_SUMMARY_PATH.exists(), "Missing preprocessed_locked40_2d_summary.csv"
assert MAP_DIR.exists(), "Missing recomputed fixed baseline maps folder. Run recomputed baseline first."

locked_40 = pd.read_csv(LOCKED_MANIFEST_PATH)
locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)

case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

print("Locked cases:", len(locked_40))
print("Baseline map folder:", MAP_DIR)
print("PCC output folder:", PCC_DIR)


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    
    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)
    
    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)
    
    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    """
    Top-k Dice/IoU:
    k = number of true target pixels.
    Select top-k predicted pixels and compare with true future-change target.
    """
    pred = pred.astype(np.float32)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def make_slice_wise_dilation_region(current_mask, radius):
    """
    current_mask shape: [Z, H, W]
    Slice-wise 2D dilation region, consistent with a 2D baseline setting.
    """
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    R = np.zeros_like(current_mask, dtype=bool)
    
    for z in range(Z):
        m = current_mask[z]
        if m.sum() > 0:
            dist = distance_transform_edt(~m)
            R[z] = dist <= radius
        else:
            R[z] = False
    
    return R


def smooth_target_2d(target, sigma):
    """
    Smooth target slice-wise, not across z.
    target shape: [Z, H, W]
    """
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))
    
    if S.max() > 0:
        S = S / (S.max() + 1e-8)
    
    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_correction(B, current_mask, target, rounds, eta=0.30, dilation_radius=26, sigma=2.0):
    """
    PCC-style iterative target-comparison correction.
    
    B: recomputed fixed baseline probability map, shape [Z,H,W]
    current_mask: current tumour mask, shape [Z,H,W]
    target: future-change target, shape [Z,H,W]
    
    Correction:
    - Build correction region R from current tumour mask dilation.
    - Build smoothed target signal S.
    - Iteratively move B toward S inside R.
    - Outside R, preserve original baseline.
    """
    B = normalize_prob_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)
    
    R = make_slice_wise_dilation_region(current_mask, dilation_radius)
    S = smooth_target_2d(target, sigma)
    
    P = B.copy().astype(np.float32)
    
    for _ in range(rounds):
        P_new = P.copy()
        
        # Target-comparison correction inside the correction region.
        correction_signal = S - P
        
        P_new[R] = P[R] + eta * correction_signal[R]
        
        # Outside correction region, keep the original baseline unchanged.
        P_new[~R] = B[~R]
        
        P = np.clip(P_new, 0, 1).astype(np.float32)
    
    return P, R, S


# =====================
# Run recomputed PCC
# =====================
all_rows = []
pairwise_rows = []

for _, row in tqdm(locked_40.iterrows(), total=len(locked_40), desc="Running recomputed PCC"):
    case_id = row["case_id"]
    npz_path = case_to_npz[case_id]
    
    baseline_path = MAP_DIR / f"{case_id}_model_baseline_map_for_pcc_recomputed.npy"
    assert baseline_path.exists(), f"Missing recomputed baseline map for {case_id}: {baseline_path}"
    
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)        # [Z,2,H,W]
        Y = data["Y"].astype(np.uint8)          # [Z,1,H,W]
    
    current_mask = X[:, 1].astype(bool)         # [Z,H,W]
    target = Y[:, 0].astype(bool)               # [Z,H,W]
    
    B = np.load(baseline_path).astype(np.float32)
    B = normalize_prob_map(B)
    
    assert B.shape == target.shape, f"Shape mismatch for {case_id}: B={B.shape}, target={target.shape}"
    
    fixed_metrics = topk_metrics(B, target)
    
    all_rows.append({
        "case_id": case_id,
        "method": "Recomputed fixed baseline",
        "rounds": 0,
        "eta": np.nan,
        "dilation_radius": np.nan,
        "sigma": np.nan,
        "pred_path": str(baseline_path),
        **fixed_metrics,
    })
    
    for rounds in ROUNDS_LIST:
        P, R, S = run_pcc_correction(
            B=B,
            current_mask=current_mask,
            target=target,
            rounds=rounds,
            eta=ETA,
            dilation_radius=DILATION_RADIUS,
            sigma=SIGMA,
        )
        
        pcc_path = PCC_MAP_DIR / f"{case_id}_recomputed_PCC_rounds_{rounds}.npy"
        np.save(pcc_path, P.astype(np.float16))
        
        metrics = topk_metrics(P, target)
        
        all_rows.append({
            "case_id": case_id,
            "method": f"Recomputed PCC rounds={rounds}",
            "rounds": rounds,
            "eta": ETA,
            "dilation_radius": DILATION_RADIUS,
            "sigma": SIGMA,
            "pred_path": str(pcc_path),
            **metrics,
        })


metrics_df = pd.DataFrame(all_rows)
metrics_path = PCC_DIR / "recomputed_pcc_case_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\nSaved case metrics:", metrics_path)
display(metrics_df.head())


# =====================
# Summary table
# =====================
summary = (
    metrics_df
    .groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

summary_path = PCC_DIR / "recomputed_pcc_summary.csv"
summary.to_csv(summary_path)

print("\nRecomputed PCC summary:")
display(summary)
print("Saved summary:", summary_path)


# =====================
# Pairwise comparison: PCC vs recomputed fixed baseline
# =====================
fixed = metrics_df[metrics_df["method"] == "Recomputed fixed baseline"].set_index("case_id")

pairwise_results = []

for rounds in ROUNDS_LIST:
    method = f"Recomputed PCC rounds={rounds}"
    pcc = metrics_df[metrics_df["method"] == method].set_index("case_id")
    
    for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
        diff = pcc[metric] - fixed[metric]
        pairwise_results.append({
            "comparison": f"{method} vs Recomputed fixed baseline",
            "metric": metric,
            "mean_diff": float(diff.mean()),
            "median_diff": float(diff.median()),
            "wins": int((diff > 0).sum()),
            "total": int(diff.notna().sum()),
            "win_rate": float((diff > 0).mean()),
        })

pairwise_df = pd.DataFrame(pairwise_results)
pairwise_path = PCC_DIR / "recomputed_pcc_pairwise_vs_fixed.csv"
pairwise_df.to_csv(pairwise_path, index=False)

print("\nPairwise PCC vs fixed:")
display(pairwise_df)
print("Saved pairwise:", pairwise_path)


# =====================
# Paper-facing compact comparison
# =====================
mean_table = metrics_df.groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]].mean().reset_index()
mean_table = mean_table.rename(columns={
    "dice": "Dice_mean",
    "iou": "IoU_mean",
    "target_focus": "Target_focus_mean",
    "log10_ratio": "Log10_ratio_mean",
})

compact_path = PCC_DIR / "recomputed_pcc_compact_comparison.csv"
mean_table.to_csv(compact_path, index=False)

print("\nCompact comparison:")
display(mean_table)
print("Saved compact comparison:", compact_path)

In [ ]:
# ============================================================
# PCC-v2 repair scan:
# Target boost + non-target suppression
# Starting map: recomputed fixed baseline B'
# No model training. No EIA yet.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

try:
    from scipy.ndimage import gaussian_filter, distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import gaussian_filter, distance_transform_edt


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

RECOMP_DIR = OUT_DIR / "recomputed_case_specific_fixed_baseline_20epoch"
MAP_DIR = RECOMP_DIR / "recomputed_fixed_baseline_maps"

PCC_V2_DIR = OUT_DIR / "pcc_v2_target_boost_non_target_suppression_scan"
PCC_V2_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_MANIFEST_PATH.exists(), "Missing locked manifest."
assert LOCKED_PRE_SUMMARY_PATH.exists(), "Missing locked preprocessing summary."
assert MAP_DIR.exists(), "Missing recomputed fixed baseline maps."

locked_40 = pd.read_csv(LOCKED_MANIFEST_PATH)
locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

print("Locked cases:", len(locked_40))
print("Baseline map folder:", MAP_DIR)
print("PCC-v2 output folder:", PCC_V2_DIR)


# =====================
# Config grid
# =====================
DILATION_RADIUS = 26
SIGMA = 2.0
ETA_POS = 0.30

CONFIGS = []

for rounds in [3, 5, 10]:
    for eta_neg in [0.05, 0.10, 0.20]:
        CONFIGS.append({
            "method": f"PCC-v2 rounds={rounds}, eta_pos=0.30, eta_neg={eta_neg:.2f}",
            "rounds": rounds,
            "eta_pos": ETA_POS,
            "eta_neg": eta_neg,
            "dilation_radius": DILATION_RADIUS,
            "sigma": SIGMA,
        })

print("Configs:", len(CONFIGS))
for c in CONFIGS:
    print(c["method"])


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    
    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)
    
    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)
    
    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = pred.astype(np.float32)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def make_slice_wise_dilation_region(current_mask, radius):
    """
    current_mask shape: [Z, H, W]
    Dilation is slice-wise 2D, matching the 2D baseline setting.
    """
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    R = np.zeros_like(current_mask, dtype=bool)
    
    for z in range(Z):
        m = current_mask[z]
        if m.sum() > 0:
            dist = distance_transform_edt(~m)
            R[z] = dist <= radius
        else:
            R[z] = False
    
    return R


def smooth_target_2d(target, sigma):
    """
    Smooth target slice-wise, not across z.
    target shape: [Z, H, W]
    """
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))
    
    if S.max() > 0:
        S = S / (S.max() + 1e-8)
    
    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_v2(
    B,
    current_mask,
    target,
    rounds,
    eta_pos=0.30,
    eta_neg=0.10,
    dilation_radius=26,
    sigma=2.0
):
    """
    PCC-v2:
    - Start from recomputed fixed baseline B.
    - Use smoothed future-change target S as positive correction signal.
    - Boost target-like regions.
    - Suppress high-response non-target regions inside correction region.
    - Do NOT directly copy target.
    - Outside correction region, keep original baseline unchanged.
    """
    B = normalize_prob_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)
    
    R = make_slice_wise_dilation_region(current_mask, dilation_radius)
    S = smooth_target_2d(target, sigma)
    
    P = B.copy().astype(np.float32)
    
    for _ in range(rounds):
        P_new = P.copy()
        
        # Positive correction:
        # high S and low current P -> increase
        positive_signal = S * (1.0 - P)
        
        # Negative correction:
        # low S and high current P -> suppress
        # protect true target pixels from suppression
        negative_signal = (1.0 - S) * P
        negative_signal[target] = 0.0
        
        P_new[R] = (
            P[R]
            + eta_pos * positive_signal[R]
            - eta_neg * negative_signal[R]
        )
        
        # Outside the correction/search region, preserve original baseline
        P_new[~R] = B[~R]
        
        P = np.clip(P_new, 0, 1).astype(np.float32)
    
    return P, R, S


def target_coverage_in_region(target, R, eps=1e-8):
    target = target.astype(bool)
    if target.sum() == 0:
        return np.nan
    return float(np.logical_and(target, R).sum() / (target.sum() + eps))


# =====================
# Run PCC-v2 scan
# =====================
all_rows = []

for _, row in tqdm(locked_40.iterrows(), total=len(locked_40), desc="Running PCC-v2 scan"):
    case_id = row["case_id"]
    npz_path = case_to_npz[case_id]
    
    baseline_path = MAP_DIR / f"{case_id}_model_baseline_map_for_pcc_recomputed.npy"
    assert baseline_path.exists(), f"Missing baseline map: {baseline_path}"
    
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)        # [Z,2,H,W]
        Y = data["Y"].astype(np.uint8)          # [Z,1,H,W]
    
    current_mask = X[:, 1].astype(bool)
    target = Y[:, 0].astype(bool)
    
    B = np.load(baseline_path).astype(np.float32)
    B = normalize_prob_map(B)
    
    assert B.shape == target.shape, f"Shape mismatch for {case_id}: B={B.shape}, target={target.shape}"
    
    R0 = make_slice_wise_dilation_region(current_mask, DILATION_RADIUS)
    coverage = target_coverage_in_region(target, R0)
    
    fixed_metrics = topk_metrics(B, target)
    
    all_rows.append({
        "case_id": case_id,
        "method": "Recomputed fixed baseline",
        "rounds": 0,
        "eta_pos": np.nan,
        "eta_neg": np.nan,
        "dilation_radius": np.nan,
        "sigma": np.nan,
        "target_coverage_in_region": coverage,
        **fixed_metrics,
    })
    
    for cfg in CONFIGS:
        P, R, S = run_pcc_v2(
            B=B,
            current_mask=current_mask,
            target=target,
            rounds=cfg["rounds"],
            eta_pos=cfg["eta_pos"],
            eta_neg=cfg["eta_neg"],
            dilation_radius=cfg["dilation_radius"],
            sigma=cfg["sigma"],
        )
        
        metrics = topk_metrics(P, target)
        
        all_rows.append({
            "case_id": case_id,
            "method": cfg["method"],
            "rounds": cfg["rounds"],
            "eta_pos": cfg["eta_pos"],
            "eta_neg": cfg["eta_neg"],
            "dilation_radius": cfg["dilation_radius"],
            "sigma": cfg["sigma"],
            "target_coverage_in_region": coverage,
            **metrics,
        })


metrics_df = pd.DataFrame(all_rows)
case_metrics_path = PCC_V2_DIR / "pcc_v2_case_metrics.csv"
metrics_df.to_csv(case_metrics_path, index=False)

print("\nSaved case metrics:", case_metrics_path)
display(metrics_df.head())


# =====================
# Summary
# =====================
summary = (
    metrics_df
    .groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

summary_path = PCC_V2_DIR / "pcc_v2_summary.csv"
summary.to_csv(summary_path)

print("\nPCC-v2 summary:")
display(summary)
print("Saved summary:", summary_path)


# =====================
# Pairwise vs fixed
# =====================
fixed = metrics_df[metrics_df["method"] == "Recomputed fixed baseline"].set_index("case_id")

pairwise_rows = []

for method in sorted(metrics_df["method"].unique()):
    if method == "Recomputed fixed baseline":
        continue
    
    cur = metrics_df[metrics_df["method"] == method].set_index("case_id")
    
    for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
        diff = cur[metric] - fixed[metric]
        pairwise_rows.append({
            "comparison": f"{method} vs Recomputed fixed baseline",
            "method": method,
            "metric": metric,
            "mean_diff": float(diff.mean()),
            "median_diff": float(diff.median()),
            "wins": int((diff > 0).sum()),
            "total": int(diff.notna().sum()),
            "win_rate": float((diff > 0).mean()),
        })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = PCC_V2_DIR / "pcc_v2_pairwise_vs_fixed.csv"
pairwise_df.to_csv(pairwise_path, index=False)

print("\nPairwise PCC-v2 vs fixed:")
display(pairwise_df)
print("Saved pairwise:", pairwise_path)


# =====================
# Compact ranking table
# =====================
mean_table = metrics_df.groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]].mean().reset_index()

fixed_mean = mean_table[mean_table["method"] == "Recomputed fixed baseline"].iloc[0]

ranking = mean_table.copy()
ranking["dice_diff_vs_fixed"] = ranking["dice"] - float(fixed_mean["dice"])
ranking["iou_diff_vs_fixed"] = ranking["iou"] - float(fixed_mean["iou"])
ranking["target_focus_diff_vs_fixed"] = ranking["target_focus"] - float(fixed_mean["target_focus"])
ranking["log10_ratio_diff_vs_fixed"] = ranking["log10_ratio"] - float(fixed_mean["log10_ratio"])

# Composite score: prioritize Dice/IoU, but penalize target_focus/log-ratio drops
ranking["balanced_score"] = (
    ranking["dice_diff_vs_fixed"]
    + ranking["iou_diff_vs_fixed"]
    + 0.5 * ranking["target_focus_diff_vs_fixed"]
    + 0.1 * ranking["log10_ratio_diff_vs_fixed"]
)

ranking = ranking.sort_values("balanced_score", ascending=False)

ranking_path = PCC_V2_DIR / "pcc_v2_compact_ranking.csv"
ranking.to_csv(ranking_path, index=False)

print("\nCompact ranking:")
display(ranking)
print("Saved ranking:", ranking_path)


# =====================
# Region coverage diagnostic
# =====================
coverage_df = metrics_df[metrics_df["method"] == "Recomputed fixed baseline"][
    ["case_id", "target_coverage_in_region"]
].copy()

coverage_path = PCC_V2_DIR / "pcc_v2_target_coverage_diagnostic.csv"
coverage_df.to_csv(coverage_path, index=False)

print("\nTarget coverage in correction region:")
print(coverage_df["target_coverage_in_region"].describe())
display(coverage_df.sort_values("target_coverage_in_region").head(10))
print("Saved coverage diagnostic:", coverage_path)

In [ ]:
# ============================================================
# EIA Linear correction baseline
# Starting map: recomputed fixed baseline B'
# Equal information access: same future-change target, same smoothing, same correction region
# No model training.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

try:
    from scipy.ndimage import gaussian_filter, distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import gaussian_filter, distance_transform_edt


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

RECOMP_DIR = OUT_DIR / "recomputed_case_specific_fixed_baseline_20epoch"
MAP_DIR = RECOMP_DIR / "recomputed_fixed_baseline_maps"

PCC_V2_DIR = OUT_DIR / "pcc_v2_target_boost_non_target_suppression_scan"
PCC_V2_METRICS_PATH = PCC_V2_DIR / "pcc_v2_case_metrics.csv"

EIA_DIR = OUT_DIR / "eia_linear_from_recomputed_baseline"
EIA_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_MANIFEST_PATH.exists(), "Missing locked manifest."
assert LOCKED_PRE_SUMMARY_PATH.exists(), "Missing locked preprocessing summary."
assert MAP_DIR.exists(), "Missing recomputed fixed baseline maps."
assert PCC_V2_METRICS_PATH.exists(), "Missing PCC-v2 metrics. Run PCC-v2 scan first."

locked_40 = pd.read_csv(LOCKED_MANIFEST_PATH)
locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

print("Locked cases:", len(locked_40))
print("Baseline map folder:", MAP_DIR)
print("EIA output folder:", EIA_DIR)


# =====================
# Config
# =====================
DILATION_RADIUS = 26
SIGMA = 2.0

# 主结果看 0.30；其他 lambda 用于 sensitivity
LAMBDA_LIST = [0.10, 0.20, 0.30, 0.40, 0.50, 0.70, 0.90]

# 只保存主 lambda 的 map，避免磁盘过大
SAVE_MAP_LAMBDAS = [0.30]


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    
    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)
    
    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)
    
    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = pred.astype(np.float32)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def make_slice_wise_dilation_region(current_mask, radius):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    R = np.zeros_like(current_mask, dtype=bool)
    
    for z in range(Z):
        m = current_mask[z]
        if m.sum() > 0:
            dist = distance_transform_edt(~m)
            R[z] = dist <= radius
        else:
            R[z] = False
    
    return R


def smooth_target_2d(target, sigma):
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))
    
    if S.max() > 0:
        S = S / (S.max() + 1e-8)
    
    return np.clip(S, 0, 1).astype(np.float32)


def run_eia_linear(B, current_mask, target, lam, dilation_radius=26, sigma=2.0):
    """
    EIA Linear:
    - Same starting map as PCC-v2: B
    - Same target access: future-change target
    - Same correction region: dilated current tumour mask
    - Same smoothing sigma
    - No iterative PCC mechanism
    """
    B = normalize_prob_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)
    
    R = make_slice_wise_dilation_region(current_mask, dilation_radius)
    S = smooth_target_2d(target, sigma)
    
    E = B.copy().astype(np.float32)
    E[R] = (1.0 - lam) * B[R] + lam * S[R]
    E[~R] = B[~R]
    
    E = np.clip(E, 0, 1).astype(np.float32)
    return E, R, S


# =====================
# Run EIA Linear
# =====================
all_rows = []

for _, row in tqdm(locked_40.iterrows(), total=len(locked_40), desc="Running EIA Linear"):
    case_id = row["case_id"]
    npz_path = case_to_npz[case_id]
    
    baseline_path = MAP_DIR / f"{case_id}_model_baseline_map_for_pcc_recomputed.npy"
    assert baseline_path.exists(), f"Missing baseline map: {baseline_path}"
    
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"].astype(np.uint8)
    
    current_mask = X[:, 1].astype(bool)
    target = Y[:, 0].astype(bool)
    
    B = np.load(baseline_path).astype(np.float32)
    B = normalize_prob_map(B)
    
    assert B.shape == target.shape, f"Shape mismatch for {case_id}: B={B.shape}, target={target.shape}"
    
    fixed_metrics = topk_metrics(B, target)
    
    all_rows.append({
        "case_id": case_id,
        "method": "Recomputed fixed baseline",
        "lambda": np.nan,
        "dilation_radius": np.nan,
        "sigma": np.nan,
        "pred_path": str(baseline_path),
        **fixed_metrics,
    })
    
    for lam in LAMBDA_LIST:
        E, R, S = run_eia_linear(
            B=B,
            current_mask=current_mask,
            target=target,
            lam=lam,
            dilation_radius=DILATION_RADIUS,
            sigma=SIGMA,
        )
        
        pred_path = ""
        if lam in SAVE_MAP_LAMBDAS:
            pred_path = str(EIA_DIR / f"{case_id}_EIA_linear_lambda_{lam:.2f}.npy")
            np.save(pred_path, E.astype(np.float16))
        
        metrics = topk_metrics(E, target)
        
        all_rows.append({
            "case_id": case_id,
            "method": f"EIA Linear lambda={lam:.2f}",
            "lambda": lam,
            "dilation_radius": DILATION_RADIUS,
            "sigma": SIGMA,
            "pred_path": pred_path,
            **metrics,
        })


metrics_df = pd.DataFrame(all_rows)

case_metrics_path = EIA_DIR / "eia_linear_case_metrics.csv"
metrics_df.to_csv(case_metrics_path, index=False)

print("\nSaved EIA case metrics:", case_metrics_path)
display(metrics_df.head())


# =====================
# Summary
# =====================
summary = (
    metrics_df
    .groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

summary_path = EIA_DIR / "eia_linear_summary.csv"
summary.to_csv(summary_path)

print("\nEIA Linear summary:")
display(summary)
print("Saved summary:", summary_path)


# =====================
# Pairwise EIA vs fixed
# =====================
fixed = metrics_df[metrics_df["method"] == "Recomputed fixed baseline"].set_index("case_id")

pairwise_rows = []

for method in sorted(metrics_df["method"].unique()):
    if method == "Recomputed fixed baseline":
        continue
    
    cur = metrics_df[metrics_df["method"] == method].set_index("case_id")
    
    for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
        diff = cur[metric] - fixed[metric]
        pairwise_rows.append({
            "comparison": f"{method} vs Recomputed fixed baseline",
            "method": method,
            "metric": metric,
            "mean_diff": float(diff.mean()),
            "median_diff": float(diff.median()),
            "wins": int((diff > 0).sum()),
            "total": int(diff.notna().sum()),
            "win_rate": float((diff > 0).mean()),
        })

pairwise_df = pd.DataFrame(pairwise_rows)

pairwise_path = EIA_DIR / "eia_linear_pairwise_vs_fixed.csv"
pairwise_df.to_csv(pairwise_path, index=False)

print("\nPairwise EIA Linear vs fixed:")
display(pairwise_df)
print("Saved pairwise:", pairwise_path)


# =====================
# Compare EIA Linear with selected PCC-v2 settings
# =====================
pcc_v2_df = pd.read_csv(PCC_V2_METRICS_PATH)

selected_pcc_methods = [
    "Recomputed fixed baseline",
    "PCC-v2 rounds=10, eta_pos=0.30, eta_neg=0.10",
    "PCC-v2 rounds=10, eta_pos=0.30, eta_neg=0.20",
]

selected_eia_methods = [
    "EIA Linear lambda=0.10",
    "EIA Linear lambda=0.20",
    "EIA Linear lambda=0.30",
    "EIA Linear lambda=0.40",
    "EIA Linear lambda=0.50",
    "EIA Linear lambda=0.70",
    "EIA Linear lambda=0.90",
]

combined_rows = []

# Add fixed baseline from EIA result
fixed_mean = metrics_df[metrics_df["method"] == "Recomputed fixed baseline"][["dice", "iou", "target_focus", "log10_ratio"]].mean()
combined_rows.append({
    "Method": "Recomputed fixed baseline",
    "Type": "Starting map",
    "Same_target_access_as_PCC": "No",
    "Dice_mean": float(fixed_mean["dice"]),
    "IoU_mean": float(fixed_mean["iou"]),
    "Target_focus_mean": float(fixed_mean["target_focus"]),
    "Log10_ratio_mean": float(fixed_mean["log10_ratio"]),
})

# Add EIA methods
for method in selected_eia_methods:
    sub = metrics_df[metrics_df["method"] == method]
    if len(sub) == 0:
        continue
    m = sub[["dice", "iou", "target_focus", "log10_ratio"]].mean()
    combined_rows.append({
        "Method": method,
        "Type": "EIA Linear",
        "Same_target_access_as_PCC": "Yes",
        "Dice_mean": float(m["dice"]),
        "IoU_mean": float(m["iou"]),
        "Target_focus_mean": float(m["target_focus"]),
        "Log10_ratio_mean": float(m["log10_ratio"]),
    })

# Add selected PCC-v2 methods
for method in selected_pcc_methods:
    if method == "Recomputed fixed baseline":
        continue
    sub = pcc_v2_df[pcc_v2_df["method"] == method]
    if len(sub) == 0:
        print("Warning: selected PCC-v2 method not found:", method)
        continue
    m = sub[["dice", "iou", "target_focus", "log10_ratio"]].mean()
    combined_rows.append({
        "Method": method,
        "Type": "PCC-v2",
        "Same_target_access_as_PCC": "Yes",
        "Dice_mean": float(m["dice"]),
        "IoU_mean": float(m["iou"]),
        "Target_focus_mean": float(m["target_focus"]),
        "Log10_ratio_mean": float(m["log10_ratio"]),
    })

combined = pd.DataFrame(combined_rows)

combined_path = EIA_DIR / "eia_linear_vs_pcc_v2_comparison.csv"
combined.to_csv(combined_path, index=False)

print("\nEIA Linear vs PCC-v2 comparison:")
display(combined)
print("Saved comparison:", combined_path)


# =====================
# Pairwise EIA vs selected PCC-v2
# =====================
pairwise_eia_pcc_rows = []

for eia_method in selected_eia_methods:
    eia = metrics_df[metrics_df["method"] == eia_method].set_index("case_id")
    if len(eia) == 0:
        continue
    
    for pcc_method in selected_pcc_methods:
        if pcc_method == "Recomputed fixed baseline":
            continue
        
        pcc = pcc_v2_df[pcc_v2_df["method"] == pcc_method].set_index("case_id")
        if len(pcc) == 0:
            continue
        
        for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
            diff = pcc[metric] - eia[metric]
            pairwise_eia_pcc_rows.append({
                "comparison": f"{pcc_method} vs {eia_method}",
                "pcc_method": pcc_method,
                "eia_method": eia_method,
                "metric": metric,
                "mean_diff": float(diff.mean()),
                "median_diff": float(diff.median()),
                "pcc_wins": int((diff > 0).sum()),
                "total": int(diff.notna().sum()),
                "pcc_win_rate": float((diff > 0).mean()),
            })

pairwise_eia_pcc_df = pd.DataFrame(pairwise_eia_pcc_rows)

pairwise_eia_pcc_path = EIA_DIR / "pairwise_pcc_v2_vs_eia_linear.csv"
pairwise_eia_pcc_df.to_csv(pairwise_eia_pcc_path, index=False)

print("\nPairwise PCC-v2 vs EIA Linear:")
display(pairwise_eia_pcc_df)
print("Saved pairwise PCC-v2 vs EIA:", pairwise_eia_pcc_path)

In [ ]:
# ============================================================
# Version 2: EIA Budget-Matched Greedy Correction Baseline
# Starting map: recomputed fixed baseline B'
# Budget source: selected PCC-v2 correction map
# Purpose: strong equal-information-access stress-test
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

try:
    from scipy.ndimage import gaussian_filter, distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import gaussian_filter, distance_transform_edt


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

RECOMP_DIR = OUT_DIR / "recomputed_case_specific_fixed_baseline_20epoch"
MAP_DIR = RECOMP_DIR / "recomputed_fixed_baseline_maps"

PCC_V2_DIR = OUT_DIR / "pcc_v2_target_boost_non_target_suppression_scan"
PCC_V2_METRICS_PATH = PCC_V2_DIR / "pcc_v2_case_metrics.csv"

EIA_LINEAR_DIR = OUT_DIR / "eia_linear_from_recomputed_baseline"
EIA_LINEAR_METRICS_PATH = EIA_LINEAR_DIR / "eia_linear_case_metrics.csv"

GREEDY_DIR = OUT_DIR / "eia_budget_matched_greedy_from_recomputed_baseline"
GREEDY_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_MANIFEST_PATH.exists(), "Missing locked manifest."
assert LOCKED_PRE_SUMMARY_PATH.exists(), "Missing locked preprocessing summary."
assert MAP_DIR.exists(), "Missing recomputed fixed baseline maps."
assert PCC_V2_METRICS_PATH.exists(), "Missing PCC-v2 metrics."
assert EIA_LINEAR_METRICS_PATH.exists(), "Missing EIA Linear metrics."

locked_40 = pd.read_csv(LOCKED_MANIFEST_PATH)
locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

print("Locked cases:", len(locked_40))
print("Baseline map folder:", MAP_DIR)
print("Greedy EIA output folder:", GREEDY_DIR)


# =====================
# Config
# =====================
DILATION_RADIUS = 26
SIGMA = 2.0
ETA_POS = 0.30

# Budget-matched to these PCC-v2 settings
PCC_BUDGET_CONFIGS = [
    {
        "name": "Budget matched to PCC-v2 rounds=10 eta_neg=0.10",
        "rounds": 10,
        "eta_pos": 0.30,
        "eta_neg": 0.10,
    },
    {
        "name": "Budget matched to PCC-v2 rounds=10 eta_neg=0.20",
        "rounds": 10,
        "eta_pos": 0.30,
        "eta_neg": 0.20,
    },
]

# Save maps only for the main greedy setting to avoid too much disk use
SAVE_GREEDY_MAPS_FOR = [
    "Budget matched to PCC-v2 rounds=10 eta_neg=0.10"
]


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    
    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)
    
    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)
    
    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = pred.astype(np.float32)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def make_slice_wise_dilation_region(current_mask, radius):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    R = np.zeros_like(current_mask, dtype=bool)
    
    for z in range(Z):
        m = current_mask[z]
        if m.sum() > 0:
            dist = distance_transform_edt(~m)
            R[z] = dist <= radius
        else:
            R[z] = False
    
    return R


def smooth_target_2d(target, sigma):
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))
    
    if S.max() > 0:
        S = S / (S.max() + 1e-8)
    
    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_v2(
    B,
    current_mask,
    target,
    rounds,
    eta_pos=0.30,
    eta_neg=0.10,
    dilation_radius=26,
    sigma=2.0
):
    B = normalize_prob_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)
    
    R = make_slice_wise_dilation_region(current_mask, dilation_radius)
    S = smooth_target_2d(target, sigma)
    
    P = B.copy().astype(np.float32)
    
    for _ in range(rounds):
        P_new = P.copy()
        
        positive_signal = S * (1.0 - P)
        negative_signal = (1.0 - S) * P
        negative_signal[target] = 0.0
        
        P_new[R] = (
            P[R]
            + eta_pos * positive_signal[R]
            - eta_neg * negative_signal[R]
        )
        
        P_new[~R] = B[~R]
        P = np.clip(P_new, 0, 1).astype(np.float32)
    
    return P, R, S


def apply_raise_budget(flat, idx, budget):
    """
    Increase selected positions toward 1 using a fixed L1 budget.
    Greedy rule: raise positions with largest available deficit first.
    """
    if budget <= 0 or len(idx) == 0:
        return 0.0
    
    vals = flat[idx]
    capacity = 1.0 - vals
    valid = capacity > 1e-8
    
    if valid.sum() == 0:
        return 0.0
    
    idx = idx[valid]
    capacity = capacity[valid]
    
    order = np.argsort(-capacity)
    idx = idx[order]
    capacity = capacity[order]
    
    cum = np.cumsum(capacity)
    full_n = int((cum <= budget).sum())
    
    used = 0.0
    
    if full_n > 0:
        full_idx = idx[:full_n]
        used += float(capacity[:full_n].sum())
        flat[full_idx] = 1.0
    
    remaining = budget - used
    
    if remaining > 1e-8 and full_n < len(idx):
        partial_idx = idx[full_n]
        inc = min(float(remaining), float(1.0 - flat[partial_idx]))
        flat[partial_idx] += inc
        used += inc
    
    return used


def apply_suppress_budget(flat, idx, budget):
    """
    Decrease selected positions toward 0 using a fixed L1 budget.
    Greedy rule: suppress highest baseline responses first.
    """
    if budget <= 0 or len(idx) == 0:
        return 0.0
    
    vals = flat[idx]
    capacity = vals.copy()
    valid = capacity > 1e-8
    
    if valid.sum() == 0:
        return 0.0
    
    idx = idx[valid]
    capacity = capacity[valid]
    
    order = np.argsort(-capacity)
    idx = idx[order]
    capacity = capacity[order]
    
    cum = np.cumsum(capacity)
    full_n = int((cum <= budget).sum())
    
    used = 0.0
    
    if full_n > 0:
        full_idx = idx[:full_n]
        used += float(capacity[:full_n].sum())
        flat[full_idx] = 0.0
    
    remaining = budget - used
    
    if remaining > 1e-8 and full_n < len(idx):
        partial_idx = idx[full_n]
        dec = min(float(remaining), float(flat[partial_idx]))
        flat[partial_idx] -= dec
        used += dec
    
    return used


def run_budget_matched_greedy(B, target, R, pcc_map):
    """
    Budget-matched greedy EIA:
    - Same starting map B.
    - Same target access as PCC.
    - Same correction region R.
    - Same positive and negative L1 correction budgets as PCC.
    
    Positive budget:
      Sum of PCC increases relative to B inside R.
      Greedy spends it by raising target pixels inside R.
    
    Negative budget:
      Sum of PCC decreases relative to B inside R.
      Greedy spends it by suppressing non-target pixels inside R.
    """
    B = normalize_prob_map(B)
    target = target.astype(bool)
    R = R.astype(bool)
    pcc_map = normalize_prob_map(pcc_map)
    
    delta = pcc_map - B
    
    pos_budget = float(np.maximum(delta[R], 0).sum())
    neg_budget = float(np.maximum(-delta[R], 0).sum())
    total_budget = pos_budget + neg_budget
    
    G = B.copy().astype(np.float32)
    flat = G.reshape(-1)
    
    flat_target = target.reshape(-1)
    flat_R = R.reshape(-1)
    
    pos_idx = np.where(flat_R & flat_target)[0]
    neg_idx = np.where(flat_R & (~flat_target))[0]
    
    used_pos = apply_raise_budget(flat, pos_idx, pos_budget)
    used_neg = apply_suppress_budget(flat, neg_idx, neg_budget)
    
    G = flat.reshape(G.shape)
    G = np.clip(G, 0, 1).astype(np.float32)
    
    budget_info = {
        "pcc_pos_budget": pos_budget,
        "pcc_neg_budget": neg_budget,
        "pcc_total_budget": total_budget,
        "greedy_used_pos_budget": used_pos,
        "greedy_used_neg_budget": used_neg,
        "greedy_used_total_budget": used_pos + used_neg,
        "pos_budget_usage_ratio": used_pos / (pos_budget + 1e-8),
        "neg_budget_usage_ratio": used_neg / (neg_budget + 1e-8),
    }
    
    return G, budget_info


# =====================
# Run Budget-Matched Greedy
# =====================
all_rows = []
budget_rows = []

for _, row in tqdm(locked_40.iterrows(), total=len(locked_40), desc="Running Budget-Matched Greedy EIA"):
    case_id = row["case_id"]
    npz_path = case_to_npz[case_id]
    
    baseline_path = MAP_DIR / f"{case_id}_model_baseline_map_for_pcc_recomputed.npy"
    assert baseline_path.exists(), f"Missing baseline map: {baseline_path}"
    
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"].astype(np.uint8)
    
    current_mask = X[:, 1].astype(bool)
    target = Y[:, 0].astype(bool)
    
    B = np.load(baseline_path).astype(np.float32)
    B = normalize_prob_map(B)
    
    assert B.shape == target.shape, f"Shape mismatch for {case_id}: B={B.shape}, target={target.shape}"
    
    fixed_metrics = topk_metrics(B, target)
    
    all_rows.append({
        "case_id": case_id,
        "method": "Recomputed fixed baseline",
        "budget_source": "None",
        "rounds": 0,
        "eta_pos": np.nan,
        "eta_neg": np.nan,
        **fixed_metrics,
    })
    
    for cfg in PCC_BUDGET_CONFIGS:
        P, R, S = run_pcc_v2(
            B=B,
            current_mask=current_mask,
            target=target,
            rounds=cfg["rounds"],
            eta_pos=cfg["eta_pos"],
            eta_neg=cfg["eta_neg"],
            dilation_radius=DILATION_RADIUS,
            sigma=SIGMA,
        )
        
        pcc_metrics = topk_metrics(P, target)
        
        pcc_method_name = f"PCC-v2 rounds={cfg['rounds']}, eta_pos={cfg['eta_pos']:.2f}, eta_neg={cfg['eta_neg']:.2f}"
        
        all_rows.append({
            "case_id": case_id,
            "method": pcc_method_name,
            "budget_source": cfg["name"],
            "rounds": cfg["rounds"],
            "eta_pos": cfg["eta_pos"],
            "eta_neg": cfg["eta_neg"],
            **pcc_metrics,
        })
        
        G, budget_info = run_budget_matched_greedy(
            B=B,
            target=target,
            R=R,
            pcc_map=P,
        )
        
        greedy_method_name = f"EIA Greedy {cfg['name']}"
        greedy_metrics = topk_metrics(G, target)
        
        pred_path = ""
        if cfg["name"] in SAVE_GREEDY_MAPS_FOR:
            pred_path = str(GREEDY_DIR / f"{case_id}_EIA_budget_matched_greedy_eta_neg_{cfg['eta_neg']:.2f}.npy")
            np.save(pred_path, G.astype(np.float16))
        
        all_rows.append({
            "case_id": case_id,
            "method": greedy_method_name,
            "budget_source": cfg["name"],
            "rounds": cfg["rounds"],
            "eta_pos": cfg["eta_pos"],
            "eta_neg": cfg["eta_neg"],
            "pred_path": pred_path,
            **greedy_metrics,
        })
        
        budget_rows.append({
            "case_id": case_id,
            "budget_source": cfg["name"],
            "pcc_method": pcc_method_name,
            "greedy_method": greedy_method_name,
            **budget_info,
        })


metrics_df = pd.DataFrame(all_rows)
budget_df = pd.DataFrame(budget_rows)

metrics_path = GREEDY_DIR / "eia_budget_matched_greedy_case_metrics.csv"
budget_path = GREEDY_DIR / "eia_budget_matched_greedy_budget_diagnostics.csv"

metrics_df.to_csv(metrics_path, index=False)
budget_df.to_csv(budget_path, index=False)

print("\nSaved greedy case metrics:", metrics_path)
print("Saved budget diagnostics:", budget_path)
display(metrics_df.head())
display(budget_df.head())


# =====================
# Summary
# =====================
summary = (
    metrics_df
    .groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

summary_path = GREEDY_DIR / "eia_budget_matched_greedy_summary.csv"
summary.to_csv(summary_path)

print("\nBudget-Matched Greedy summary:")
display(summary)
print("Saved summary:", summary_path)


# =====================
# Compact comparison with EIA Linear and PCC-v2
# =====================
eia_linear_df = pd.read_csv(EIA_LINEAR_METRICS_PATH)
pcc_v2_df = pd.read_csv(PCC_V2_METRICS_PATH)

compact_rows = []

def add_method_from_df(df, method_name, method_type):
    sub = df[df["method"] == method_name]
    if len(sub) == 0:
        print("Warning: method not found:", method_name)
        return
    
    m = sub[["dice", "iou", "target_focus", "log10_ratio"]].mean()
    compact_rows.append({
        "Method": method_name,
        "Type": method_type,
        "Dice_mean": float(m["dice"]),
        "IoU_mean": float(m["iou"]),
        "Target_focus_mean": float(m["target_focus"]),
        "Log10_ratio_mean": float(m["log10_ratio"]),
    })

add_method_from_df(metrics_df, "Recomputed fixed baseline", "Starting map")
add_method_from_df(eia_linear_df, "EIA Linear lambda=0.30", "EIA Linear")
add_method_from_df(eia_linear_df, "EIA Linear lambda=0.50", "EIA Linear")
add_method_from_df(eia_linear_df, "EIA Linear lambda=0.90", "EIA Linear")

for cfg in PCC_BUDGET_CONFIGS:
    pcc_method_name = f"PCC-v2 rounds={cfg['rounds']}, eta_pos={cfg['eta_pos']:.2f}, eta_neg={cfg['eta_neg']:.2f}"
    greedy_method_name = f"EIA Greedy {cfg['name']}"
    
    add_method_from_df(metrics_df, pcc_method_name, "PCC-v2 recomputed in greedy run")
    add_method_from_df(metrics_df, greedy_method_name, "EIA Budget-Matched Greedy")

compact = pd.DataFrame(compact_rows)

compact_path = GREEDY_DIR / "eia_budget_matched_greedy_compact_comparison.csv"
compact.to_csv(compact_path, index=False)

print("\nCompact comparison:")
display(compact)
print("Saved compact comparison:", compact_path)


# =====================
# Pairwise: PCC-v2 vs Greedy
# =====================
pairwise_rows = []

for cfg in PCC_BUDGET_CONFIGS:
    pcc_method_name = f"PCC-v2 rounds={cfg['rounds']}, eta_pos={cfg['eta_pos']:.2f}, eta_neg={cfg['eta_neg']:.2f}"
    greedy_method_name = f"EIA Greedy {cfg['name']}"
    
    pcc = metrics_df[metrics_df["method"] == pcc_method_name].set_index("case_id")
    greedy = metrics_df[metrics_df["method"] == greedy_method_name].set_index("case_id")
    
    for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
        diff = pcc[metric] - greedy[metric]
        pairwise_rows.append({
            "comparison": f"{pcc_method_name} vs {greedy_method_name}",
            "pcc_method": pcc_method_name,
            "greedy_method": greedy_method_name,
            "metric": metric,
            "mean_diff": float(diff.mean()),
            "median_diff": float(diff.median()),
            "pcc_wins": int((diff > 0).sum()),
            "greedy_wins": int((diff < 0).sum()),
            "ties": int((diff == 0).sum()),
            "total": int(diff.notna().sum()),
            "pcc_win_rate": float((diff > 0).mean()),
        })

pairwise_df = pd.DataFrame(pairwise_rows)

pairwise_path = GREEDY_DIR / "pairwise_pcc_v2_vs_budget_matched_greedy.csv"
pairwise_df.to_csv(pairwise_path, index=False)

print("\nPairwise PCC-v2 vs Budget-Matched Greedy:")
display(pairwise_df)
print("Saved pairwise:", pairwise_path)


# =====================
# Budget diagnostics summary
# =====================
budget_summary = (
    budget_df
    .groupby("budget_source")[[
        "pcc_pos_budget",
        "pcc_neg_budget",
        "pcc_total_budget",
        "greedy_used_pos_budget",
        "greedy_used_neg_budget",
        "greedy_used_total_budget",
        "pos_budget_usage_ratio",
        "neg_budget_usage_ratio",
    ]]
    .agg(["mean", "median", "min", "max"])
)

budget_summary_path = GREEDY_DIR / "eia_budget_matched_greedy_budget_summary.csv"
budget_summary.to_csv(budget_summary_path)

print("\nBudget diagnostics summary:")
display(budget_summary)
print("Saved budget summary:", budget_summary_path)

In [ ]:
# ============================================================
# Step 2: Recover locked40 preprocessing npz files
# Output:
#   /kaggle/working/pcc_independent_baseline/manifest_locked_original_40.csv
#   /kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_summary.csv
#   /kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_npz/*.npz
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib


# =====================
# Output paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NPZ_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"
NPZ_DIR.mkdir(parents=True, exist_ok=True)

LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"


# =====================
# Locked 40 case list
# =====================
LOCKED_CASES = [
    "PatientID_0003_T1_to_T2_t1c",
    "PatientID_0005_T3_to_T4_t1c",
    "PatientID_0006_T2_to_T4_t1c",
    "PatientID_0007_T2_to_T3_t1c",
    "PatientID_0008_T4_to_T6_t1c",
    "PatientID_0010_T1_to_T4_t1c",
    "PatientID_0011_T1_to_T2_t1c",
    "PatientID_0012_T2_to_T3_t1c",
    "PatientID_0013_T1_to_T2_t1c",
    "PatientID_0014_T1_to_T2_t1c",
    "PatientID_0018_T1_to_T2_t1c",
    "PatientID_0019_T4_to_T5_t1c",
    "PatientID_0020_T1_to_T2_t1c",
    "PatientID_0021_T2_to_T3_t1c",
    "PatientID_0022_T1_to_T2_t1c",
    "PatientID_0024_T2_to_T3_t1c",
    "PatientID_0025_T1_to_T2_t1c",
    "PatientID_0026_T1_to_T2_t1c",
    "PatientID_0029_T1_to_T3_t1c",
    "PatientID_0030_T1_to_T3_t1c",
    "PatientID_0031_T2_to_T3_t1c",
    "PatientID_0032_T1_to_T2_t1c",
    "PatientID_0033_T1_to_T2_t1c",
    "PatientID_0034_T1_to_T2_t1c",
    "PatientID_0035_T1_to_T2_t1c",
    "PatientID_0036_T1_to_T2_t1c",
    "PatientID_0037_T1_to_T2_t1c",
    "PatientID_0038_T1_to_T2_t1c",
    "PatientID_0039_T1_to_T2_t1c",
    "PatientID_0041_T1_to_T2_t1c",
    "PatientID_0044_T1_to_T2_t1c",
    "PatientID_0045_T1_to_T2_t1c",
    "PatientID_0051_T1_to_T3_t1c",
    "PatientID_0052_T1_to_T2_t1c",
    "PatientID_0053_T1_to_T3_t1c",
    "PatientID_0054_T1_to_T2_t1c",
    "PatientID_0055_T1_to_T4_t1c",
    "PatientID_0059_T1_to_T2_t1c",
    "PatientID_0060_T1_to_T2_t1c",
    "PatientID_0062_T1_to_T2_t1c",
]


def parse_case_id(case_id):
    m = re.match(r"(PatientID_\d+)_T(\d+)_to_T(\d+)_t1c$", case_id)
    if not m:
        raise ValueError(f"Cannot parse case_id: {case_id}")
    patient_id = m.group(1)
    current_tp = int(m.group(2))
    future_tp = int(m.group(3))
    return patient_id, current_tp, future_tp


# =====================
# Locate MU-Glioma-Post root
# =====================
def locate_mu_root():
    input_root = Path("/kaggle/input")
    candidates = []

    for p in input_root.rglob("*"):
        if p.is_dir():
            patient_dirs = list(p.glob("PatientID_*"))
            if len(patient_dirs) > 10:
                candidates.append(p)

    if not candidates:
        print("Could not automatically find MU-Glioma-Post root.")
        print("Top-level /kaggle/input:")
        for p in input_root.iterdir():
            print(" ", p)
        raise FileNotFoundError("No folder containing PatientID_* found under /kaggle/input")

    candidates = sorted(candidates, key=lambda x: len(str(x)), reverse=True)
    return candidates[0]


DATA_ROOT = locate_mu_root()

print("Detected MU-Glioma-Post root:", DATA_ROOT)
print("Patient folders:", len(list(DATA_ROOT.glob("PatientID_*"))))


# =====================
# Find files
# =====================
def find_timepoint_file(patient_id, tp_num, kind):
    """
    kind:
      "t1c" or "mask"
    """
    patient_dir = DATA_ROOT / patient_id
    assert patient_dir.exists(), f"Missing patient folder: {patient_dir}"

    tp_patterns = [
        f"Timepoint_{tp_num}",
        f"Timepoint-{tp_num}",
        f"Timepoint {tp_num}",
        f"T{tp_num}",
        f"TP{tp_num}",
    ]

    candidate_files = []

    for f in patient_dir.rglob("*.nii*"):
        path_str = str(f)
        name = f.name.lower()

        if not any(tp_pat in path_str for tp_pat in tp_patterns):
            continue

        if kind == "t1c":
            if "t1c" in name and "brain" in name:
                candidate_files.append(f)

        elif kind == "mask":
            if "tumormask" in name or "tumourmask" in name:
                candidate_files.append(f)

        else:
            raise ValueError(kind)

    candidate_files = sorted(set(candidate_files))

    if len(candidate_files) == 0:
        raise FileNotFoundError(f"Missing {kind} for {patient_id} T{tp_num}")

    return candidate_files[0]


# =====================
# Loading and preprocessing
# =====================
def load_nii(path):
    arr = nib.load(str(path)).get_fdata()
    arr = np.asarray(arr, dtype=np.float32)
    return arr


def volume_to_z_hw(vol):
    """
    Convert 3D volume to [Z,H,W].
    MU-Glioma-Post is usually [H,W,Z].
    """
    if vol.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {vol.shape}")

    slice_axis = int(np.argmin(vol.shape))

    if slice_axis == 0:
        zhw = vol
    elif slice_axis == 1:
        zhw = np.transpose(vol, (1, 0, 2))
    else:
        zhw = np.transpose(vol, (2, 0, 1))

    return zhw.astype(np.float32)


def normalize_mri(vol):
    vol = vol.astype(np.float32)
    finite = np.isfinite(vol)
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)

    foreground = vol[(vol > 0) & finite]

    if foreground.size < 10:
        return np.zeros_like(vol, dtype=np.float32)

    lo, hi = np.percentile(foreground, [1, 99])

    if hi <= lo:
        return np.zeros_like(vol, dtype=np.float32)

    vol = np.clip(vol, lo, hi)
    vol = (vol - lo) / (hi - lo + 1e-8)
    vol = np.clip(vol, 0, 1)

    return vol.astype(np.float32)


# =====================
# Main preprocessing loop
# =====================
manifest_rows = []
summary_rows = []

for case_id in tqdm(LOCKED_CASES, desc="Preprocessing locked40"):
    patient_id, current_tp, future_tp = parse_case_id(case_id)

    cur_t1c_path = find_timepoint_file(patient_id, current_tp, "t1c")
    cur_mask_path = find_timepoint_file(patient_id, current_tp, "mask")
    fut_mask_path = find_timepoint_file(patient_id, future_tp, "mask")

    cur_t1c = volume_to_z_hw(load_nii(cur_t1c_path))
    cur_mask = volume_to_z_hw(load_nii(cur_mask_path)) > 0
    fut_mask = volume_to_z_hw(load_nii(fut_mask_path)) > 0

    assert cur_t1c.shape == cur_mask.shape == fut_mask.shape, (
        f"Shape mismatch {case_id}: "
        f"t1c={cur_t1c.shape}, cur_mask={cur_mask.shape}, fut_mask={fut_mask.shape}"
    )

    cur_t1c_norm = normalize_mri(cur_t1c)

    # Future-change target:
    # future tumour area not already included in current tumour mask
    target = np.logical_and(fut_mask, ~cur_mask)

    X = np.stack(
        [
            cur_t1c_norm,
            cur_mask.astype(np.float32),
        ],
        axis=1
    ).astype(np.float32)        # [Z,2,H,W]

    Y = target[:, None, :, :].astype(np.uint8)   # [Z,1,H,W]

    z_indices = np.arange(X.shape[0], dtype=np.int16)

    npz_path = NPZ_DIR / f"{case_id}.npz"

    np.savez_compressed(
        npz_path,
        X=X,
        Y=Y,
        z_indices=z_indices,
        case_id=np.array(case_id),
        patient_id=np.array(patient_id),
        current_tp=np.array(current_tp),
        future_tp=np.array(future_tp),
    )

    target_voxels = int(target.sum())
    current_voxels = int(cur_mask.sum())
    positive_slices = int((target.reshape(target.shape[0], -1).sum(axis=1) > 0).sum())

    manifest_rows.append({
        "case_id": case_id,
        "patient_id": patient_id,
        "current_tp": current_tp,
        "future_tp": future_tp,
        "current_t1c_path": str(cur_t1c_path),
        "current_mask_path": str(cur_mask_path),
        "future_mask_path": str(fut_mask_path),
        "growth_voxels": target_voxels,
    })

    summary_rows.append({
        "case_id": case_id,
        "patient_id": patient_id,
        "npz_path": str(npz_path),
        "num_slices": int(X.shape[0]),
        "height": int(X.shape[2]),
        "width": int(X.shape[3]),
        "positive_slices": positive_slices,
        "target_voxels": target_voxels,
        "current_voxels": current_voxels,
    })


manifest_df = pd.DataFrame(manifest_rows)
summary_df = pd.DataFrame(summary_rows)

manifest_df.to_csv(LOCKED_MANIFEST_PATH, index=False)
summary_df.to_csv(LOCKED_PRE_SUMMARY_PATH, index=False)

print("\nSaved locked manifest:", LOCKED_MANIFEST_PATH)
print("Saved preprocessing summary:", LOCKED_PRE_SUMMARY_PATH)
print("Saved npz folder:", NPZ_DIR)

print("\nSummary:")
print("Cases:", len(summary_df))
print("Total slices:", int(summary_df["num_slices"].sum()))
print("Total positive slices:", int(summary_df["positive_slices"].sum()))
print("Total target voxels:", int(summary_df["target_voxels"].sum()))

display(summary_df.head())
display(summary_df.describe(include="all"))

In [ ]:
# ============================================================
# Stage 2 / Model A:
# Matched 5-fold Direct-Target Student Baseline
#
# Purpose:
# Train a student model directly on true future-change targets.
# This will be the matched baseline for later PCC-teacher student.
#
# Test-time rule:
# The model only sees current T1c + current tumour mask.
# It does NOT use future target, PCC map, EIA map, or Greedy map at inference.
# ============================================================

import os
import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from sklearn.model_selection import KFold
except ImportError:
    !pip install -q scikit-learn
    from sklearn.model_selection import KFold


# =====================
# Basic config
# =====================
SEED = 42
N_SPLITS = 5
EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

# For direct hard future-change target
LOSS_BCE_WEIGHT = 0.5
LOSS_DICE_WEIGHT = 0.5

# Slice sampler: modestly oversample slices that contain future-change target
POS_SLICE_WEIGHT = 3.0
NEG_SLICE_WEIGHT = 1.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"
DIRECT_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
PRED_DIR = DIRECT_DIR / "direct_target_pred_maps"
CKPT_DIR = DIRECT_DIR / "checkpoints"

DIRECT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_MANIFEST_PATH.exists(), f"Missing: {LOCKED_MANIFEST_PATH}"
assert LOCKED_PRE_SUMMARY_PATH.exists(), f"Missing: {LOCKED_PRE_SUMMARY_PATH}"

locked_40 = pd.read_csv(LOCKED_MANIFEST_PATH)
locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)

case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

print("Locked cases:", len(locked_40))
print("Output folder:", DIRECT_DIR)


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    
    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)
    
    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)
    
    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = normalize_prob_map(pred)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case_npz(case_id):
    npz_path = case_to_npz[case_id]
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)       # [Z, 2, H, W]
        Y = data["Y"].astype(np.float32)       # [Z, 1, H, W]
    return X, Y


def compute_train_pos_weight(train_case_ids):
    pos = 0.0
    total = 0.0
    
    for case_id in train_case_ids:
        _, Y = load_case_npz(case_id)
        pos += float(Y.sum())
        total += float(Y.size)
    
    neg = total - pos
    pos_weight = neg / (pos + 1e-8)
    
    # Avoid unstable huge BCE weights
    pos_weight = float(np.clip(pos_weight, 1.0, 50.0))
    return pos_weight, pos, total


# =====================
# Dataset
# =====================
class CaseSliceDataset(Dataset):
    def __init__(self, case_ids, target_mode="direct_true_target"):
        self.case_ids = list(case_ids)
        self.target_mode = target_mode
        
        self.case_data = {}
        self.index = []
        self.slice_has_target = []
        
        for case_id in self.case_ids:
            X, Y = load_case_npz(case_id)
            self.case_data[case_id] = (X, Y)
            
            Z = X.shape[0]
            for z in range(Z):
                self.index.append((case_id, z))
                self.slice_has_target.append(float(Y[z].sum() > 0))
        
        self.slice_has_target = np.asarray(self.slice_has_target, dtype=np.float32)
    
    def __len__(self):
        return len(self.index)
    
    def __getitem__(self, idx):
        case_id, z = self.index[idx]
        X, Y = self.case_data[case_id]
        
        x = X[z]       # [2, H, W]
        y = Y[z]       # [1, H, W]
        
        return torch.from_numpy(x).float(), torch.from_numpy(y).float(), case_id, z


def make_weighted_sampler(dataset):
    weights = np.where(
        dataset.slice_has_target > 0,
        POS_SLICE_WEIGHT,
        NEG_SLICE_WEIGHT
    ).astype(np.float32)
    
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True
    )


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)
        
        self.pool = nn.MaxPool2d(2)
        
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        
        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        
        self.out = nn.Conv2d(base, out_ch, kernel_size=1)
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        
        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        return self.out(d1)


# =====================
# Loss
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
    
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def combined_loss(logits, targets, pos_weight_tensor):
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=pos_weight_tensor
    )
    dice = soft_dice_loss_from_logits(logits, targets)
    
    return LOSS_BCE_WEIGHT * bce + LOSS_DICE_WEIGHT * dice, bce.detach(), dice.detach()


# =====================
# Train / predict
# =====================
def train_one_fold(fold, train_case_ids, test_case_ids):
    print("\n" + "=" * 80)
    print(f"Fold {fold}")
    print("Train cases:", len(train_case_ids))
    print("Test cases:", len(test_case_ids))
    print("Test case IDs:")
    for c in test_case_ids:
        print("  ", c)
    
    pos_weight, train_pos, train_total = compute_train_pos_weight(train_case_ids)
    print(f"Train positive voxels: {train_pos:.0f} / {train_total:.0f}")
    print(f"BCE pos_weight clipped: {pos_weight:.4f}")
    
    train_ds = CaseSliceDataset(train_case_ids)
    train_sampler = make_weighted_sampler(train_ds)
    
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    
    model = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)
    
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    
    fold_loss_rows = []
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        running_bce = 0.0
        running_dice = 0.0
        n_batches = 0
        
        pbar = tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch}/{EPOCHS}", leave=False)
        
        for x, y, _, _ in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(x)
                loss, bce_part, dice_part = combined_loss(logits, y, pos_weight_tensor)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += float(loss.detach().cpu())
            running_bce += float(bce_part.detach().cpu())
            running_dice += float(dice_part.detach().cpu())
            n_batches += 1
            
            pbar.set_postfix({
                "loss": running_loss / max(n_batches, 1),
                "bce": running_bce / max(n_batches, 1),
                "dice_loss": running_dice / max(n_batches, 1),
            })
        
        scheduler.step()
        
        epoch_row = {
            "fold": fold,
            "epoch": epoch,
            "loss": running_loss / max(n_batches, 1),
            "bce_loss": running_bce / max(n_batches, 1),
            "dice_loss": running_dice / max(n_batches, 1),
            "lr": scheduler.get_last_lr()[0],
            "train_pos_weight": pos_weight,
        }
        fold_loss_rows.append(epoch_row)
        
        print(
            f"Fold {fold} Epoch {epoch:02d}: "
            f"loss={epoch_row['loss']:.5f}, "
            f"bce={epoch_row['bce_loss']:.5f}, "
            f"dice_loss={epoch_row['dice_loss']:.5f}"
        )
    
    ckpt_path = CKPT_DIR / f"direct_target_student_fold_{fold}.pt"
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "train_case_ids": list(train_case_ids),
        "test_case_ids": list(test_case_ids),
        "config": {
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "pos_slice_weight": POS_SLICE_WEIGHT,
            "neg_slice_weight": NEG_SLICE_WEIGHT,
            "loss_bce_weight": LOSS_BCE_WEIGHT,
            "loss_dice_weight": LOSS_DICE_WEIGHT,
            "train_pos_weight": pos_weight,
        }
    }, ckpt_path)
    
    print("Saved checkpoint:", ckpt_path)
    
    # Predict test cases
    model.eval()
    case_metric_rows = []
    
    with torch.no_grad():
        for case_id in tqdm(test_case_ids, desc=f"Fold {fold} prediction"):
            X, Y = load_case_npz(case_id)
            target = Y[:, 0].astype(bool)
            
            preds = []
            for start in range(0, X.shape[0], BATCH_SIZE):
                xb = torch.from_numpy(X[start:start+BATCH_SIZE]).float().to(DEVICE)
                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
                preds.append(prob.astype(np.float32))
            
            pred_map = np.concatenate(preds, axis=0)
            pred_map = normalize_prob_map(pred_map)
            
            pred_path = PRED_DIR / f"{case_id}_direct_target_student_fold_{fold}.npy"
            np.save(pred_path, pred_map.astype(np.float16))
            
            metrics = topk_metrics(pred_map, target)
            
            case_metric_rows.append({
                "case_id": case_id,
                "fold": fold,
                "method": "Direct-target student",
                "training_target": "true_future_change_target",
                "test_time_future_target_access": "No",
                "pred_path": str(pred_path),
                **metrics,
            })
    
    return fold_loss_rows, case_metric_rows


# =====================
# Build 5-fold split
# =====================
case_ids = locked_40["case_id"].tolist()

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

split_rows = []
fold_splits = []

for fold, (train_idx, test_idx) in enumerate(kf.split(case_ids), start=1):
    train_case_ids = [case_ids[i] for i in train_idx]
    test_case_ids = [case_ids[i] for i in test_idx]
    
    fold_splits.append((fold, train_case_ids, test_case_ids))
    
    for cid in train_case_ids:
        split_rows.append({
            "fold": fold,
            "case_id": cid,
            "split": "train"
        })
    for cid in test_case_ids:
        split_rows.append({
            "fold": fold,
            "case_id": cid,
            "split": "test"
        })

split_df = pd.DataFrame(split_rows)
split_path = DIRECT_DIR / "matched_5fold_splits_seed42.csv"
split_df.to_csv(split_path, index=False)

print("Saved split file:", split_path)

print("\nFold test cases:")
for fold, train_case_ids, test_case_ids in fold_splits:
    print(f"\nFold {fold}:")
    for cid in test_case_ids:
        print(" ", cid)


# =====================
# Run all folds
# =====================
all_loss_rows = []
all_case_metric_rows = []

start_time = time.time()

for fold, train_case_ids, test_case_ids in fold_splits:
    fold_loss_rows, case_metric_rows = train_one_fold(
        fold=fold,
        train_case_ids=train_case_ids,
        test_case_ids=test_case_ids
    )
    all_loss_rows.extend(fold_loss_rows)
    all_case_metric_rows.extend(case_metric_rows)
    
    # Save partial progress after each fold
    pd.DataFrame(all_loss_rows).to_csv(DIRECT_DIR / "direct_target_training_losses_partial.csv", index=False)
    pd.DataFrame(all_case_metric_rows).to_csv(DIRECT_DIR / "direct_target_case_metrics_partial.csv", index=False)


# =====================
# Save final results
# =====================
loss_df = pd.DataFrame(all_loss_rows)
case_metrics_df = pd.DataFrame(all_case_metric_rows)

loss_path = DIRECT_DIR / "direct_target_training_losses.csv"
case_metrics_path = DIRECT_DIR / "direct_target_case_metrics.csv"

loss_df.to_csv(loss_path, index=False)
case_metrics_df.to_csv(case_metrics_path, index=False)

summary = (
    case_metrics_df
    .groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

summary_path = DIRECT_DIR / "direct_target_summary.csv"
summary.to_csv(summary_path)

runtime_min = (time.time() - start_time) / 60.0

run_info = {
    "method": "Direct-target student",
    "stage": "Stage 2 / matched 5-fold student learning",
    "seed": SEED,
    "n_splits": N_SPLITS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "loss": {
        "bce_weight": LOSS_BCE_WEIGHT,
        "dice_weight": LOSS_DICE_WEIGHT,
        "bce_pos_weight": "computed per fold and clipped to [1, 50]"
    },
    "test_time_future_target_access": "No",
    "runtime_minutes": runtime_min,
    "outputs": {
        "split_path": str(split_path),
        "loss_path": str(loss_path),
        "case_metrics_path": str(case_metrics_path),
        "summary_path": str(summary_path),
        "prediction_folder": str(PRED_DIR),
        "checkpoint_folder": str(CKPT_DIR),
    }
}

with open(DIRECT_DIR / "direct_target_run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)

print("\n" + "=" * 80)
print("Direct-target 5-fold baseline finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved losses:", loss_path)
print("Saved case metrics:", case_metrics_path)
print("Saved summary:", summary_path)
print("Saved run info:", DIRECT_DIR / "direct_target_run_info.json")

print("\nDirect-target summary:")
display(summary)

print("\nCase metrics:")
display(case_metrics_df)

print("\nTraining losses tail:")
display(loss_df.tail(10))

In [ ]:
from pathlib import Path
import shutil
import os
from IPython.display import FileLink, display

# 1. Model A folder
src = Path("/kaggle/working/pcc_independent_baseline/stage2_pcc_guided_student_learning/model_A_direct_target_5fold")

# 2. Backup zip path
zip_base = "/kaggle/working/MODEL_A_BACKUP_FINAL"
zip_path = Path(zip_base + ".zip")

print("Source exists:", src.exists())
print("Source path:", src)

if not src.exists():
    print("\nSearching for Model A folder...")
    for p in Path("/kaggle/working").rglob("*model_A_direct_target_5fold*"):
        print(p)
    raise FileNotFoundError("Model A folder not found.")

# Remove old zip if needed
if zip_path.exists():
    print("Removing old zip:", zip_path)
    zip_path.unlink()

print("\nStart zipping Model A...")
zip_file = shutil.make_archive(zip_base, "zip", src)

print("\nDONE")
print("Saved:", zip_file)
print("Exists:", Path(zip_file).exists())
print("Size MB:", round(os.path.getsize(zip_file) / 1024 / 1024, 2))

print("\nFiles directly under /kaggle/working:")
for x in Path("/kaggle/working").iterdir():
    if x.is_file():
        print("FILE:", x.name, round(os.path.getsize(x) / 1024 / 1024, 2), "MB")
    else:
        print("DIR :", x.name)

print("\nDownload link:")
display(FileLink(zip_file))

In [ ]:
from pathlib import Path
import os

p = Path("/kaggle/working/MODEL_A_BACKUP.zip")

print("Exists:", p.exists())
if p.exists():
    print("Path:", p)
    print("Size MB:", round(os.path.getsize(p) / 1024 / 1024, 2))

print("\nFiles directly under /kaggle/working:")
for x in Path("/kaggle/working").iterdir():
    if x.is_file():
        print("FILE:", x.name, round(os.path.getsize(x) / 1024 / 1024, 2), "MB")
    else:
        print("DIR :", x.name)

In [ ]:
import shutil
from pathlib import Path

backup_items = [
    (
        Path("/kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_npz"),
        "/kaggle/working/preprocessed_locked40_2d_npz_backup"
    ),
    (
        Path("/kaggle/working/pcc_independent_baseline/stage2_pcc_guided_student_learning/model_A_direct_target_5fold"),
        "/kaggle/working/model_A_direct_target_5fold_backup"
    ),
]

for src, zip_path in backup_items:
    if src.exists():
        shutil.make_archive(zip_path, "zip", src)
        print("Saved:", zip_path + ".zip")
    else:
        print("Missing:", src)

In [ ]:
# ============================================================
# Stage 2 / Model B:
# Matched 5-fold EIA-Teacher Student
#
# Training target:
#   EIA Linear lambda=0.30 teacher map
#
# Test-time rule:
#   The model only sees current T1c + current tumour mask.
#   It does NOT use future target or EIA map at inference.
#
# Evaluation:
#   Student prediction is evaluated against the true future-change target.
# ============================================================

import os
import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


# =====================
# Basic config
# =====================
SEED = 42
EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

LOSS_BCE_WEIGHT = 0.5
LOSS_DICE_WEIGHT = 0.5

POS_SLICE_WEIGHT = 3.0
NEG_SLICE_WEIGHT = 1.0

TEACHER_LAMBDA = 0.30

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"

EIA_DIR = OUT_DIR / "eia_linear_from_recomputed_baseline"

MODEL_B_DIR = STAGE2_DIR / "model_B_eia_teacher_lambda030_5fold"
PRED_DIR = MODEL_B_DIR / "eia_teacher_student_pred_maps"
CKPT_DIR = MODEL_B_DIR / "checkpoints"

MODEL_B_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_PRE_SUMMARY_PATH.exists(), f"Missing: {LOCKED_PRE_SUMMARY_PATH}"
assert SPLIT_PATH.exists(), f"Missing Model A split file: {SPLIT_PATH}"
assert EIA_DIR.exists(), f"Missing EIA teacher folder: {EIA_DIR}"

locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

split_df = pd.read_csv(SPLIT_PATH)

print("Using split file:", SPLIT_PATH)
print("Output folder:", MODEL_B_DIR)


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    
    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)
    
    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)
    
    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = normalize_prob_map(pred)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case_npz(case_id):
    npz_path = case_to_npz[case_id]
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)       # [Z, 2, H, W]
        Y = data["Y"].astype(np.float32)       # [Z, 1, H, W], true future-change target
    return X, Y


def get_eia_teacher_path(case_id):
    return EIA_DIR / f"{case_id}_EIA_linear_lambda_{TEACHER_LAMBDA:.2f}.npy"


def load_eia_teacher(case_id):
    teacher_path = get_eia_teacher_path(case_id)
    assert teacher_path.exists(), f"Missing EIA teacher map: {teacher_path}"
    
    T = np.load(teacher_path).astype(np.float32)   # [Z, H, W]
    T = normalize_prob_map(T)
    T = T[:, None, :, :]                           # [Z, 1, H, W]
    return T


def compute_train_pos_weight_from_teacher(train_case_ids):
    pos = 0.0
    total = 0.0
    
    for case_id in train_case_ids:
        T = load_eia_teacher(case_id)
        pos += float(T.sum())
        total += float(T.size)
    
    neg = total - pos
    pos_weight = neg / (pos + 1e-8)
    pos_weight = float(np.clip(pos_weight, 1.0, 50.0))
    
    return pos_weight, pos, total


# =====================
# Dataset
# =====================
class EIASliceDataset(Dataset):
    def __init__(self, case_ids):
        self.case_ids = list(case_ids)
        
        self.case_data = {}
        self.index = []
        self.slice_has_teacher_signal = []
        
        for case_id in self.case_ids:
            X, Y_true = load_case_npz(case_id)
            T_teacher = load_eia_teacher(case_id)
            
            assert X.shape[0] == T_teacher.shape[0], f"Z mismatch: {case_id}"
            assert X.shape[-2:] == T_teacher.shape[-2:], f"HW mismatch: {case_id}"
            
            self.case_data[case_id] = (X, Y_true, T_teacher)
            
            Z = X.shape[0]
            for z in range(Z):
                self.index.append((case_id, z))
                self.slice_has_teacher_signal.append(float(T_teacher[z].sum() > 0))
        
        self.slice_has_teacher_signal = np.asarray(self.slice_has_teacher_signal, dtype=np.float32)
    
    def __len__(self):
        return len(self.index)
    
    def __getitem__(self, idx):
        case_id, z = self.index[idx]
        X, Y_true, T_teacher = self.case_data[case_id]
        
        x = X[z]             # [2, H, W]
        y = T_teacher[z]     # [1, H, W], soft teacher target
        
        return torch.from_numpy(x).float(), torch.from_numpy(y).float(), case_id, z


def make_weighted_sampler(dataset):
    weights = np.where(
        dataset.slice_has_teacher_signal > 0,
        POS_SLICE_WEIGHT,
        NEG_SLICE_WEIGHT
    ).astype(np.float32)
    
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True
    )


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)
        
        self.pool = nn.MaxPool2d(2)
        
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        
        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        
        self.out = nn.Conv2d(base, out_ch, kernel_size=1)
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        
        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        return self.out(d1)


# =====================
# Loss
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
    
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def combined_loss(logits, targets, pos_weight_tensor):
    # BCE supports soft targets in [0, 1]
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=pos_weight_tensor
    )
    dice = soft_dice_loss_from_logits(logits, targets)
    
    return LOSS_BCE_WEIGHT * bce + LOSS_DICE_WEIGHT * dice, bce.detach(), dice.detach()


# =====================
# Fold construction from Model A split
# =====================
fold_splits = []

for fold in sorted(split_df["fold"].unique()):
    train_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "train")]["case_id"].tolist()
    test_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")]["case_id"].tolist()
    fold_splits.append((int(fold), train_case_ids, test_case_ids))

print("\nFold test cases:")
for fold, train_case_ids, test_case_ids in fold_splits:
    print(f"\nFold {fold}:")
    for cid in test_case_ids:
        print(" ", cid)


# =====================
# Train / predict one fold
# =====================
def train_one_fold(fold, train_case_ids, test_case_ids):
    print("\n" + "=" * 80)
    print(f"Model B / EIA-teacher student / Fold {fold}")
    print("Train cases:", len(train_case_ids))
    print("Test cases:", len(test_case_ids))
    
    pos_weight, teacher_pos, teacher_total = compute_train_pos_weight_from_teacher(train_case_ids)
    print(f"Teacher positive mass: {teacher_pos:.2f} / {teacher_total:.0f}")
    print(f"BCE pos_weight clipped: {pos_weight:.4f}")
    
    train_ds = EIASliceDataset(train_case_ids)
    train_sampler = make_weighted_sampler(train_ds)
    
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    
    model = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)
    
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    
    fold_loss_rows = []
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        running_bce = 0.0
        running_dice = 0.0
        n_batches = 0
        
        pbar = tqdm(train_loader, desc=f"Model B Fold {fold} Epoch {epoch}/{EPOCHS}", leave=False)
        
        for x, y, _, _ in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(x)
                loss, bce_part, dice_part = combined_loss(logits, y, pos_weight_tensor)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += float(loss.detach().cpu())
            running_bce += float(bce_part.detach().cpu())
            running_dice += float(dice_part.detach().cpu())
            n_batches += 1
            
            pbar.set_postfix({
                "loss": running_loss / max(n_batches, 1),
                "bce": running_bce / max(n_batches, 1),
                "dice_loss": running_dice / max(n_batches, 1),
            })
        
        scheduler.step()
        
        epoch_row = {
            "fold": fold,
            "epoch": epoch,
            "loss": running_loss / max(n_batches, 1),
            "bce_loss": running_bce / max(n_batches, 1),
            "dice_loss": running_dice / max(n_batches, 1),
            "lr": scheduler.get_last_lr()[0],
            "teacher_pos_weight": pos_weight,
        }
        fold_loss_rows.append(epoch_row)
        
        print(
            f"Model B Fold {fold} Epoch {epoch:02d}: "
            f"loss={epoch_row['loss']:.5f}, "
            f"bce={epoch_row['bce_loss']:.5f}, "
            f"dice_loss={epoch_row['dice_loss']:.5f}"
        )
    
    ckpt_path = CKPT_DIR / f"eia_teacher_student_fold_{fold}.pt"
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "train_case_ids": list(train_case_ids),
        "test_case_ids": list(test_case_ids),
        "config": {
            "teacher": f"EIA Linear lambda={TEACHER_LAMBDA:.2f}",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "pos_slice_weight": POS_SLICE_WEIGHT,
            "neg_slice_weight": NEG_SLICE_WEIGHT,
            "loss_bce_weight": LOSS_BCE_WEIGHT,
            "loss_dice_weight": LOSS_DICE_WEIGHT,
            "teacher_pos_weight": pos_weight,
        }
    }, ckpt_path)
    
    print("Saved checkpoint:", ckpt_path)
    
    # Predict test cases
    model.eval()
    case_metric_rows = []
    
    with torch.no_grad():
        for case_id in tqdm(test_case_ids, desc=f"Model B Fold {fold} prediction"):
            X, Y_true = load_case_npz(case_id)
            target = Y_true[:, 0].astype(bool)
            
            preds = []
            for start in range(0, X.shape[0], BATCH_SIZE):
                xb = torch.from_numpy(X[start:start+BATCH_SIZE]).float().to(DEVICE)
                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
                preds.append(prob.astype(np.float32))
            
            pred_map = np.concatenate(preds, axis=0)
            pred_map = normalize_prob_map(pred_map)
            
            pred_path = PRED_DIR / f"{case_id}_eia_teacher_student_lambda030_fold_{fold}.npy"
            np.save(pred_path, pred_map.astype(np.float16))
            
            metrics = topk_metrics(pred_map, target)
            
            case_metric_rows.append({
                "case_id": case_id,
                "fold": fold,
                "method": "EIA-teacher student",
                "training_target": f"EIA Linear lambda={TEACHER_LAMBDA:.2f}",
                "test_time_future_target_access": "No",
                "pred_path": str(pred_path),
                **metrics,
            })
    
    return fold_loss_rows, case_metric_rows


# =====================
# Run all folds
# =====================
all_loss_rows = []
all_case_metric_rows = []

start_time = time.time()

for fold, train_case_ids, test_case_ids in fold_splits:
    fold_loss_rows, case_metric_rows = train_one_fold(
        fold=fold,
        train_case_ids=train_case_ids,
        test_case_ids=test_case_ids
    )
    
    all_loss_rows.extend(fold_loss_rows)
    all_case_metric_rows.extend(case_metric_rows)
    
    pd.DataFrame(all_loss_rows).to_csv(MODEL_B_DIR / "eia_teacher_training_losses_partial.csv", index=False)
    pd.DataFrame(all_case_metric_rows).to_csv(MODEL_B_DIR / "eia_teacher_case_metrics_partial.csv", index=False)


# =====================
# Save final results
# =====================
loss_df = pd.DataFrame(all_loss_rows)
case_metrics_df = pd.DataFrame(all_case_metric_rows)

loss_path = MODEL_B_DIR / "eia_teacher_training_losses.csv"
case_metrics_path = MODEL_B_DIR / "eia_teacher_case_metrics.csv"

loss_df.to_csv(loss_path, index=False)
case_metrics_df.to_csv(case_metrics_path, index=False)

summary = (
    case_metrics_df
    .groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

summary_path = MODEL_B_DIR / "eia_teacher_summary.csv"
summary.to_csv(summary_path)

runtime_min = (time.time() - start_time) / 60.0

run_info = {
    "method": "EIA-teacher student",
    "stage": "Stage 2 / matched 5-fold student learning",
    "teacher": f"EIA Linear lambda={TEACHER_LAMBDA:.2f}",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "test_time_future_target_access": "No",
    "split_file": str(SPLIT_PATH),
    "runtime_minutes": runtime_min,
    "outputs": {
        "loss_path": str(loss_path),
        "case_metrics_path": str(case_metrics_path),
        "summary_path": str(summary_path),
        "prediction_folder": str(PRED_DIR),
        "checkpoint_folder": str(CKPT_DIR),
    }
}

with open(MODEL_B_DIR / "eia_teacher_run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)

print("\n" + "=" * 80)
print("Model B / EIA-teacher student finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved losses:", loss_path)
print("Saved case metrics:", case_metrics_path)
print("Saved summary:", summary_path)
print("Saved run info:", MODEL_B_DIR / "eia_teacher_run_info.json")

print("\nEIA-teacher student summary:")
display(summary)

print("\nCase metrics:")
display(case_metrics_df)

print("\nTraining losses tail:")
display(loss_df.tail(10))

In [ ]:
# ============================================================
# Stage 2 / Model C:
# PCC-v2 Teacher Student
#
# Teacher:
#   PCC-v2 rounds=10, eta_pos=0.30, eta_neg=0.10
#
# Training target:
#   PCC-v2 corrected map
#
# Test-time rule:
#   The model only sees current T1c + current tumour mask.
#   It does NOT use future target or PCC map at inference.
#
# Evaluation:
#   Student prediction is evaluated against the true future-change target.
# ============================================================

import os
import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from scipy.ndimage import gaussian_filter, distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import gaussian_filter, distance_transform_edt


# =====================
# Basic config
# =====================
SEED = 42
EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

LOSS_BCE_WEIGHT = 0.5
LOSS_DICE_WEIGHT = 0.5

POS_SLICE_WEIGHT = 3.0
NEG_SLICE_WEIGHT = 1.0

# PCC-v2 teacher configuration
PCC_ROUNDS = 10
PCC_ETA_POS = 0.30
PCC_ETA_NEG = 0.10
PCC_DILATION_RADIUS = 26
PCC_SIGMA = 2.0

# Regenerate teacher maps even if they already exist?
FORCE_REGENERATE_TEACHER = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

RECOMP_DIR = OUT_DIR / "recomputed_case_specific_fixed_baseline_20epoch"
BASELINE_MAP_DIR = RECOMP_DIR / "recomputed_fixed_baseline_maps"

STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"

MODEL_C_DIR = STAGE2_DIR / "model_C_pcc_v2_teacher_eta_neg010_5fold"
PCC_TEACHER_DIR = STAGE2_DIR / "teacher_maps_pcc_v2_round10_eta030_neg010"

PRED_DIR = MODEL_C_DIR / "pcc_teacher_student_pred_maps"
CKPT_DIR = MODEL_C_DIR / "checkpoints"

MODEL_C_DIR.mkdir(parents=True, exist_ok=True)
PCC_TEACHER_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_PRE_SUMMARY_PATH.exists(), f"Missing: {LOCKED_PRE_SUMMARY_PATH}"
assert SPLIT_PATH.exists(), f"Missing Model A split file: {SPLIT_PATH}"
assert BASELINE_MAP_DIR.exists(), f"Missing recomputed baseline map folder: {BASELINE_MAP_DIR}"

locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

split_df = pd.read_csv(SPLIT_PATH)

all_case_ids = sorted(split_df["case_id"].unique().tolist())

print("Using split file:", SPLIT_PATH)
print("PCC teacher folder:", PCC_TEACHER_DIR)
print("Model C output folder:", MODEL_C_DIR)
print("Number of cases:", len(all_case_ids))


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    
    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)
    
    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)
    
    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = normalize_prob_map(pred)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case_npz(case_id):
    npz_path = case_to_npz[case_id]
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)       # [Z, 2, H, W]
        Y = data["Y"].astype(np.float32)       # [Z, 1, H, W]
    return X, Y


def get_recomputed_baseline_path(case_id):
    return BASELINE_MAP_DIR / f"{case_id}_model_baseline_map_for_pcc_recomputed.npy"


def get_pcc_teacher_path(case_id):
    return PCC_TEACHER_DIR / f"{case_id}_PCC_v2_round10_eta030_neg010_teacher.npy"


def make_slice_wise_dilation_region(current_mask, radius):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    R = np.zeros_like(current_mask, dtype=bool)
    
    for z in range(Z):
        m = current_mask[z]
        if m.sum() > 0:
            dist = distance_transform_edt(~m)
            R[z] = dist <= radius
        else:
            R[z] = False
    
    return R


def smooth_target_2d(target, sigma):
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))
    
    if S.max() > 0:
        S = S / (S.max() + 1e-8)
    
    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_v2_teacher(
    B,
    current_mask,
    target,
    rounds=PCC_ROUNDS,
    eta_pos=PCC_ETA_POS,
    eta_neg=PCC_ETA_NEG,
    dilation_radius=PCC_DILATION_RADIUS,
    sigma=PCC_SIGMA
):
    """
    PCC-v2 teacher:
    - Start from recomputed fixed baseline B.
    - Use future-change target only to generate teacher map.
    - This teacher is used only as training supervision.
    - At student test time, future target is not used.
    """
    B = normalize_prob_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)
    
    R = make_slice_wise_dilation_region(current_mask, dilation_radius)
    S = smooth_target_2d(target, sigma)
    
    P = B.copy().astype(np.float32)
    
    for _ in range(rounds):
        P_new = P.copy()
        
        positive_signal = S * (1.0 - P)
        negative_signal = (1.0 - S) * P
        negative_signal[target] = 0.0
        
        P_new[R] = (
            P[R]
            + eta_pos * positive_signal[R]
            - eta_neg * negative_signal[R]
        )
        
        P_new[~R] = B[~R]
        P = np.clip(P_new, 0, 1).astype(np.float32)
    
    return P


# =====================
# Step 1: Generate PCC-v2 teacher maps
# =====================
teacher_quality_rows = []

print("\nGenerating / checking PCC-v2 teacher maps...")

for case_id in tqdm(all_case_ids, desc="PCC-v2 teacher generation"):
    teacher_path = get_pcc_teacher_path(case_id)
    
    X, Y_true = load_case_npz(case_id)
    current_mask = X[:, 1].astype(bool)
    target = Y_true[:, 0].astype(bool)
    
    if teacher_path.exists() and not FORCE_REGENERATE_TEACHER:
        P_teacher = np.load(teacher_path).astype(np.float32)
        P_teacher = normalize_prob_map(P_teacher)
    else:
        baseline_path = get_recomputed_baseline_path(case_id)
        assert baseline_path.exists(), f"Missing recomputed baseline map: {baseline_path}"
        
        B = np.load(baseline_path).astype(np.float32)
        B = normalize_prob_map(B)
        
        assert B.shape == target.shape, f"Shape mismatch: B={B.shape}, target={target.shape}, case={case_id}"
        
        P_teacher = run_pcc_v2_teacher(
            B=B,
            current_mask=current_mask,
            target=target,
            rounds=PCC_ROUNDS,
            eta_pos=PCC_ETA_POS,
            eta_neg=PCC_ETA_NEG,
            dilation_radius=PCC_DILATION_RADIUS,
            sigma=PCC_SIGMA,
        )
        
        np.save(teacher_path, P_teacher.astype(np.float16))
    
    metrics = topk_metrics(P_teacher, target)
    teacher_quality_rows.append({
        "case_id": case_id,
        "teacher_method": "PCC-v2 teacher rounds=10 eta_pos=0.30 eta_neg=0.10",
        "teacher_path": str(teacher_path),
        **metrics,
    })

teacher_quality_df = pd.DataFrame(teacher_quality_rows)
teacher_quality_path = PCC_TEACHER_DIR / "pcc_v2_teacher_quality_case_metrics.csv"
teacher_quality_df.to_csv(teacher_quality_path, index=False)

teacher_quality_summary = (
    teacher_quality_df
    .groupby("teacher_method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

teacher_quality_summary_path = PCC_TEACHER_DIR / "pcc_v2_teacher_quality_summary.csv"
teacher_quality_summary.to_csv(teacher_quality_summary_path)

print("\nPCC-v2 teacher quality summary:")
display(teacher_quality_summary)
print("Saved teacher quality metrics:", teacher_quality_path)
print("Saved teacher quality summary:", teacher_quality_summary_path)


def load_pcc_teacher(case_id):
    teacher_path = get_pcc_teacher_path(case_id)
    assert teacher_path.exists(), f"Missing PCC teacher map: {teacher_path}"
    
    T = np.load(teacher_path).astype(np.float32)   # [Z, H, W]
    T = normalize_prob_map(T)
    T = T[:, None, :, :]                           # [Z, 1, H, W]
    return T


def compute_train_pos_weight_from_teacher(train_case_ids):
    pos = 0.0
    total = 0.0
    
    for case_id in train_case_ids:
        T = load_pcc_teacher(case_id)
        pos += float(T.sum())
        total += float(T.size)
    
    neg = total - pos
    pos_weight = neg / (pos + 1e-8)
    pos_weight = float(np.clip(pos_weight, 1.0, 50.0))
    
    return pos_weight, pos, total


# =====================
# Dataset
# =====================
class PCCSliceDataset(Dataset):
    def __init__(self, case_ids):
        self.case_ids = list(case_ids)
        
        self.case_data = {}
        self.index = []
        self.slice_has_true_target = []
        
        for case_id in self.case_ids:
            X, Y_true = load_case_npz(case_id)
            T_teacher = load_pcc_teacher(case_id)
            
            assert X.shape[0] == T_teacher.shape[0], f"Z mismatch: {case_id}"
            assert X.shape[-2:] == T_teacher.shape[-2:], f"HW mismatch: {case_id}"
            
            self.case_data[case_id] = (X, Y_true, T_teacher)
            
            Z = X.shape[0]
            for z in range(Z):
                self.index.append((case_id, z))
                # Use true future-change positive slices for sampler,
                # matching Model A protocol more closely.
                self.slice_has_true_target.append(float(Y_true[z].sum() > 0))
        
        self.slice_has_true_target = np.asarray(self.slice_has_true_target, dtype=np.float32)
    
    def __len__(self):
        return len(self.index)
    
    def __getitem__(self, idx):
        case_id, z = self.index[idx]
        X, Y_true, T_teacher = self.case_data[case_id]
        
        x = X[z]             # [2, H, W]
        y = T_teacher[z]     # [1, H, W], soft PCC teacher target
        
        return torch.from_numpy(x).float(), torch.from_numpy(y).float(), case_id, z


def make_weighted_sampler(dataset):
    weights = np.where(
        dataset.slice_has_true_target > 0,
        POS_SLICE_WEIGHT,
        NEG_SLICE_WEIGHT
    ).astype(np.float32)
    
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True
    )


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)
        
        self.pool = nn.MaxPool2d(2)
        
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        
        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        
        self.out = nn.Conv2d(base, out_ch, kernel_size=1)
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        
        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        return self.out(d1)


# =====================
# Loss
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
    
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def combined_loss(logits, targets, pos_weight_tensor):
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=pos_weight_tensor
    )
    dice = soft_dice_loss_from_logits(logits, targets)
    
    return LOSS_BCE_WEIGHT * bce + LOSS_DICE_WEIGHT * dice, bce.detach(), dice.detach()


# =====================
# Fold construction from Model A split
# =====================
fold_splits = []

for fold in sorted(split_df["fold"].unique()):
    train_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "train")]["case_id"].tolist()
    test_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")]["case_id"].tolist()
    fold_splits.append((int(fold), train_case_ids, test_case_ids))

print("\nFold test cases:")
for fold, train_case_ids, test_case_ids in fold_splits:
    print(f"\nFold {fold}:")
    for cid in test_case_ids:
        print(" ", cid)


# =====================
# Train / predict one fold
# =====================
def train_one_fold(fold, train_case_ids, test_case_ids):
    print("\n" + "=" * 80)
    print(f"Model C / PCC-v2 teacher student / Fold {fold}")
    print("Train cases:", len(train_case_ids))
    print("Test cases:", len(test_case_ids))
    
    pos_weight, teacher_pos, teacher_total = compute_train_pos_weight_from_teacher(train_case_ids)
    print(f"PCC teacher positive mass: {teacher_pos:.2f} / {teacher_total:.0f}")
    print(f"BCE pos_weight clipped: {pos_weight:.4f}")
    
    train_ds = PCCSliceDataset(train_case_ids)
    train_sampler = make_weighted_sampler(train_ds)
    
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    
    model = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)
    
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    
    fold_loss_rows = []
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        running_bce = 0.0
        running_dice = 0.0
        n_batches = 0
        
        pbar = tqdm(train_loader, desc=f"Model C Fold {fold} Epoch {epoch}/{EPOCHS}", leave=False)
        
        for x, y, _, _ in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(x)
                loss, bce_part, dice_part = combined_loss(logits, y, pos_weight_tensor)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += float(loss.detach().cpu())
            running_bce += float(bce_part.detach().cpu())
            running_dice += float(dice_part.detach().cpu())
            n_batches += 1
            
            pbar.set_postfix({
                "loss": running_loss / max(n_batches, 1),
                "bce": running_bce / max(n_batches, 1),
                "dice_loss": running_dice / max(n_batches, 1),
            })
        
        scheduler.step()
        
        epoch_row = {
            "fold": fold,
            "epoch": epoch,
            "loss": running_loss / max(n_batches, 1),
            "bce_loss": running_bce / max(n_batches, 1),
            "dice_loss": running_dice / max(n_batches, 1),
            "lr": scheduler.get_last_lr()[0],
            "teacher_pos_weight": pos_weight,
        }
        fold_loss_rows.append(epoch_row)
        
        print(
            f"Model C Fold {fold} Epoch {epoch:02d}: "
            f"loss={epoch_row['loss']:.5f}, "
            f"bce={epoch_row['bce_loss']:.5f}, "
            f"dice_loss={epoch_row['dice_loss']:.5f}"
        )
    
    ckpt_path = CKPT_DIR / f"pcc_teacher_student_fold_{fold}.pt"
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "train_case_ids": list(train_case_ids),
        "test_case_ids": list(test_case_ids),
        "config": {
            "teacher": "PCC-v2 rounds=10 eta_pos=0.30 eta_neg=0.10",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "pos_slice_weight": POS_SLICE_WEIGHT,
            "neg_slice_weight": NEG_SLICE_WEIGHT,
            "loss_bce_weight": LOSS_BCE_WEIGHT,
            "loss_dice_weight": LOSS_DICE_WEIGHT,
            "teacher_pos_weight": pos_weight,
            "sampler": "true_future_change_positive_slices",
        }
    }, ckpt_path)
    
    print("Saved checkpoint:", ckpt_path)
    
    # Predict test cases
    model.eval()
    case_metric_rows = []
    
    with torch.no_grad():
        for case_id in tqdm(test_case_ids, desc=f"Model C Fold {fold} prediction"):
            X, Y_true = load_case_npz(case_id)
            target = Y_true[:, 0].astype(bool)
            
            preds = []
            for start in range(0, X.shape[0], BATCH_SIZE):
                xb = torch.from_numpy(X[start:start+BATCH_SIZE]).float().to(DEVICE)
                logits = model(xb)
                prob = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
                preds.append(prob.astype(np.float32))
            
            pred_map = np.concatenate(preds, axis=0)
            pred_map = normalize_prob_map(pred_map)
            
            pred_path = PRED_DIR / f"{case_id}_pcc_teacher_student_eta_neg010_fold_{fold}.npy"
            np.save(pred_path, pred_map.astype(np.float16))
            
            metrics = topk_metrics(pred_map, target)
            
            case_metric_rows.append({
                "case_id": case_id,
                "fold": fold,
                "method": "PCC-v2-teacher student",
                "training_target": "PCC-v2 rounds=10 eta_pos=0.30 eta_neg=0.10",
                "test_time_future_target_access": "No",
                "pred_path": str(pred_path),
                **metrics,
            })
    
    return fold_loss_rows, case_metric_rows


# =====================
# Run all folds
# =====================
all_loss_rows = []
all_case_metric_rows = []

start_time = time.time()

for fold, train_case_ids, test_case_ids in fold_splits:
    fold_loss_rows, case_metric_rows = train_one_fold(
        fold=fold,
        train_case_ids=train_case_ids,
        test_case_ids=test_case_ids
    )
    
    all_loss_rows.extend(fold_loss_rows)
    all_case_metric_rows.extend(case_metric_rows)
    
    pd.DataFrame(all_loss_rows).to_csv(MODEL_C_DIR / "pcc_teacher_training_losses_partial.csv", index=False)
    pd.DataFrame(all_case_metric_rows).to_csv(MODEL_C_DIR / "pcc_teacher_case_metrics_partial.csv", index=False)


# =====================
# Save final results
# =====================
loss_df = pd.DataFrame(all_loss_rows)
case_metrics_df = pd.DataFrame(all_case_metric_rows)

loss_path = MODEL_C_DIR / "pcc_teacher_training_losses.csv"
case_metrics_path = MODEL_C_DIR / "pcc_teacher_case_metrics.csv"

loss_df.to_csv(loss_path, index=False)
case_metrics_df.to_csv(case_metrics_path, index=False)

summary = (
    case_metrics_df
    .groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

summary_path = MODEL_C_DIR / "pcc_teacher_summary.csv"
summary.to_csv(summary_path)

runtime_min = (time.time() - start_time) / 60.0

run_info = {
    "method": "PCC-v2-teacher student",
    "stage": "Stage 2 / matched 5-fold student learning",
    "teacher": "PCC-v2 rounds=10 eta_pos=0.30 eta_neg=0.10",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "test_time_future_target_access": "No",
    "split_file": str(SPLIT_PATH),
    "sampler": "true_future_change_positive_slices",
    "runtime_minutes": runtime_min,
    "outputs": {
        "teacher_quality_path": str(teacher_quality_path),
        "teacher_quality_summary_path": str(teacher_quality_summary_path),
        "loss_path": str(loss_path),
        "case_metrics_path": str(case_metrics_path),
        "summary_path": str(summary_path),
        "prediction_folder": str(PRED_DIR),
        "checkpoint_folder": str(CKPT_DIR),
    }
}

with open(MODEL_C_DIR / "pcc_teacher_run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)

print("\n" + "=" * 80)
print("Model C / PCC-v2-teacher student finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved losses:", loss_path)
print("Saved case metrics:", case_metrics_path)
print("Saved summary:", summary_path)
print("Saved run info:", MODEL_C_DIR / "pcc_teacher_run_info.json")

print("\nPCC-v2-teacher student summary:")
display(summary)

print("\nCase metrics:")
display(case_metrics_df)

print("\nTraining losses tail:")
display(loss_df.tail(10))


# =====================
# Optional compact comparison with Model A and current Model B
# =====================
comparison_rows = []

def add_summary_from_case_metrics(path, model_name, training_target):
    if not Path(path).exists():
        print("Comparison file not found:", path)
        return
    
    df = pd.read_csv(path)
    m = df[["dice", "iou", "target_focus", "log10_ratio"]].mean()
    comparison_rows.append({
        "Model": model_name,
        "Training target": training_target,
        "Dice_mean": float(m["dice"]),
        "IoU_mean": float(m["iou"]),
        "Target_focus_mean": float(m["target_focus"]),
        "Log10_ratio_mean": float(m["log10_ratio"]),
    })

add_summary_from_case_metrics(
    STAGE2_DIR / "model_A_direct_target_5fold" / "direct_target_case_metrics.csv",
    "Model A",
    "true future-change target"
)

add_summary_from_case_metrics(
    STAGE2_DIR / "model_B_eia_teacher_lambda030_5fold" / "eia_teacher_case_metrics.csv",
    "Model B",
    "EIA Linear lambda=0.30"
)

add_summary_from_case_metrics(
    case_metrics_path,
    "Model C",
    "PCC-v2 teacher eta_neg=0.10"
)

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = MODEL_C_DIR / "model_A_B_C_compact_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nCompact comparison:")
display(comparison_df)
print("Saved compact comparison:", comparison_path)

In [ ]:
# ============================================================
# Stage 2 / Model C2:
# PCC-Residual Correction Student
#
# Core idea:
#   Input:
#       current T1c + current mask + Model A direct prediction B
#
#   PCC-derived supervision:
#       delta_teacher = PCC_teacher - B
#
#   Student learns:
#       delta_hat
#
#   Final prediction:
#       P_final = B + delta_hat
#
# Test-time rule:
#   The model only uses:
#       current T1c + current mask + Model A direct prediction B
#   It does NOT use future target or PCC teacher at inference.
#
# Evaluation:
#   P_final is evaluated against true future-change target.
# ============================================================

import os
import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from scipy.ndimage import gaussian_filter, distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import gaussian_filter, distance_transform_edt


# =====================
# Basic config
# =====================
SEED = 42
EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

POS_SLICE_WEIGHT = 3.0
NEG_SLICE_WEIGHT = 1.0

# C2 loss weights
TRUE_LOSS_WEIGHT = 0.70
PCC_RESIDUAL_LOSS_WEIGHT = 0.30

# Residual loss gives larger weight to locations where PCC changed the baseline more
RESIDUAL_IMPORTANCE_WEIGHT = 4.0

# PCC teacher config
PCC_ROUNDS = 10
PCC_ETA_POS = 0.30
PCC_ETA_NEG = 0.10
PCC_DILATION_RADIUS = 26
PCC_SIGMA = 2.0

FORCE_REGENERATE_TEACHER = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

RECOMP_DIR = OUT_DIR / "recomputed_case_specific_fixed_baseline_20epoch"
BASELINE_MAP_DIR = RECOMP_DIR / "recomputed_fixed_baseline_maps"

STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"
MODEL_A_PRED_DIR = MODEL_A_DIR / "direct_target_pred_maps"

# Reuse / create PCC teacher maps
PCC_TEACHER_DIR = STAGE2_DIR / "teacher_maps_pcc_v2_round10_eta030_neg010"

MODEL_C2_DIR = STAGE2_DIR / "model_C2_pcc_residual_correction_student_5fold"
PRED_DIR = MODEL_C2_DIR / "pcc_residual_final_pred_maps"
RESIDUAL_PRED_DIR = MODEL_C2_DIR / "pcc_residual_delta_pred_maps"
CKPT_DIR = MODEL_C2_DIR / "checkpoints"

MODEL_C2_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
RESIDUAL_PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PCC_TEACHER_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_PRE_SUMMARY_PATH.exists(), f"Missing: {LOCKED_PRE_SUMMARY_PATH}"
assert SPLIT_PATH.exists(), f"Missing split file: {SPLIT_PATH}"
assert MODEL_A_PRED_DIR.exists(), f"Missing Model A prediction folder: {MODEL_A_PRED_DIR}"
assert BASELINE_MAP_DIR.exists(), f"Missing recomputed baseline folder: {BASELINE_MAP_DIR}"

locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

split_df = pd.read_csv(SPLIT_PATH)
all_case_ids = sorted(split_df["case_id"].unique().tolist())

# Each case has a Model A out-of-fold prediction from the fold where it was test
test_split_df = split_df[split_df["split"] == "test"].copy()
case_to_modelA_fold = dict(zip(test_split_df["case_id"], test_split_df["fold"]))

print("Using split file:", SPLIT_PATH)
print("Model A prediction folder:", MODEL_A_PRED_DIR)
print("PCC teacher folder:", PCC_TEACHER_DIR)
print("Model C2 output folder:", MODEL_C2_DIR)
print("Number of cases:", len(all_case_ids))


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    
    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)
    
    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)
    
    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = normalize_prob_map(pred)
    target = target.astype(bool)
    
    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }
    
    fp = pred.reshape(-1)
    ft = target.reshape(-1)
    
    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True
    
    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()
    
    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)
    
    target_focus = fp[ft].sum() / (fp.sum() + eps)
    
    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))
    
    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case_npz(case_id):
    npz_path = case_to_npz[case_id]
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)       # [Z, 2, H, W]
        Y = data["Y"].astype(np.float32)       # [Z, 1, H, W]
    return X, Y


def get_modelA_pred_path(case_id):
    fold = int(case_to_modelA_fold[case_id])
    return MODEL_A_PRED_DIR / f"{case_id}_direct_target_student_fold_{fold}.npy"


def load_modelA_pred(case_id):
    path = get_modelA_pred_path(case_id)
    assert path.exists(), f"Missing Model A OOF prediction: {path}"
    B = np.load(path).astype(np.float32)        # [Z, H, W]
    B = normalize_prob_map(B)
    return B


def get_recomputed_baseline_path(case_id):
    return BASELINE_MAP_DIR / f"{case_id}_model_baseline_map_for_pcc_recomputed.npy"


def get_pcc_teacher_path(case_id):
    return PCC_TEACHER_DIR / f"{case_id}_PCC_v2_round10_eta030_neg010_teacher.npy"


def make_slice_wise_dilation_region(current_mask, radius):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    R = np.zeros_like(current_mask, dtype=bool)
    
    for z in range(Z):
        m = current_mask[z]
        if m.sum() > 0:
            dist = distance_transform_edt(~m)
            R[z] = dist <= radius
        else:
            R[z] = False
    
    return R


def smooth_target_2d(target, sigma):
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))
    
    if S.max() > 0:
        S = S / (S.max() + 1e-8)
    
    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_v2_teacher(
    B,
    current_mask,
    target,
    rounds=PCC_ROUNDS,
    eta_pos=PCC_ETA_POS,
    eta_neg=PCC_ETA_NEG,
    dilation_radius=PCC_DILATION_RADIUS,
    sigma=PCC_SIGMA
):
    B = normalize_prob_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)
    
    R = make_slice_wise_dilation_region(current_mask, dilation_radius)
    S = smooth_target_2d(target, sigma)
    
    P = B.copy().astype(np.float32)
    
    for _ in range(rounds):
        P_new = P.copy()
        
        positive_signal = S * (1.0 - P)
        negative_signal = (1.0 - S) * P
        negative_signal[target] = 0.0
        
        P_new[R] = (
            P[R]
            + eta_pos * positive_signal[R]
            - eta_neg * negative_signal[R]
        )
        
        P_new[~R] = B[~R]
        P = np.clip(P_new, 0, 1).astype(np.float32)
    
    return P


# =====================
# Step 1: Generate / check PCC-v2 teacher maps
# =====================
teacher_quality_rows = []

print("\nGenerating / checking PCC-v2 teacher maps...")

for case_id in tqdm(all_case_ids, desc="PCC-v2 teacher generation"):
    teacher_path = get_pcc_teacher_path(case_id)
    
    X, Y_true = load_case_npz(case_id)
    current_mask = X[:, 1].astype(bool)
    target = Y_true[:, 0].astype(bool)
    
    if teacher_path.exists() and not FORCE_REGENERATE_TEACHER:
        P_teacher = np.load(teacher_path).astype(np.float32)
        P_teacher = normalize_prob_map(P_teacher)
    else:
        baseline_path = get_recomputed_baseline_path(case_id)
        assert baseline_path.exists(), f"Missing recomputed baseline map: {baseline_path}"
        
        B_recomputed = np.load(baseline_path).astype(np.float32)
        B_recomputed = normalize_prob_map(B_recomputed)
        
        P_teacher = run_pcc_v2_teacher(
            B=B_recomputed,
            current_mask=current_mask,
            target=target,
            rounds=PCC_ROUNDS,
            eta_pos=PCC_ETA_POS,
            eta_neg=PCC_ETA_NEG,
            dilation_radius=PCC_DILATION_RADIUS,
            sigma=PCC_SIGMA,
        )
        
        np.save(teacher_path, P_teacher.astype(np.float16))
    
    metrics = topk_metrics(P_teacher, target)
    teacher_quality_rows.append({
        "case_id": case_id,
        "teacher_method": "PCC-v2 teacher rounds=10 eta_pos=0.30 eta_neg=0.10",
        "teacher_path": str(teacher_path),
        **metrics,
    })

teacher_quality_df = pd.DataFrame(teacher_quality_rows)
teacher_quality_path = PCC_TEACHER_DIR / "pcc_v2_teacher_quality_case_metrics_for_C2.csv"
teacher_quality_df.to_csv(teacher_quality_path, index=False)

teacher_quality_summary = (
    teacher_quality_df
    .groupby("teacher_method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

teacher_quality_summary_path = PCC_TEACHER_DIR / "pcc_v2_teacher_quality_summary_for_C2.csv"
teacher_quality_summary.to_csv(teacher_quality_summary_path)

print("\nPCC-v2 teacher quality summary:")
display(teacher_quality_summary)
print("Saved teacher quality metrics:", teacher_quality_path)


def load_pcc_teacher(case_id):
    path = get_pcc_teacher_path(case_id)
    assert path.exists(), f"Missing PCC teacher: {path}"
    T = np.load(path).astype(np.float32)        # [Z, H, W]
    T = normalize_prob_map(T)
    return T


def compute_train_pos_weight_from_true_target(train_case_ids):
    pos = 0.0
    total = 0.0
    
    for case_id in train_case_ids:
        _, Y = load_case_npz(case_id)
        pos += float(Y.sum())
        total += float(Y.size)
    
    neg = total - pos
    pos_weight = neg / (pos + 1e-8)
    pos_weight = float(np.clip(pos_weight, 1.0, 50.0))
    
    return pos_weight, pos, total


# =====================
# Dataset
# =====================
class PCCResidualDataset(Dataset):
    def __init__(self, case_ids):
        self.case_ids = list(case_ids)
        
        self.case_data = {}
        self.index = []
        self.slice_has_true_target = []
        
        for case_id in self.case_ids:
            X, Y_true = load_case_npz(case_id)       # X [Z,2,H,W], Y [Z,1,H,W]
            B_direct = load_modelA_pred(case_id)     # [Z,H,W]
            P_teacher = load_pcc_teacher(case_id)    # [Z,H,W]
            
            assert X.shape[0] == B_direct.shape[0] == P_teacher.shape[0], f"Z mismatch: {case_id}"
            assert X.shape[-2:] == B_direct.shape[-2:] == P_teacher.shape[-2:], f"HW mismatch: {case_id}"
            
            delta_teacher = np.clip(P_teacher - B_direct, -1.0, 1.0).astype(np.float32)
            
            self.case_data[case_id] = (X, Y_true, B_direct, P_teacher, delta_teacher)
            
            Z = X.shape[0]
            for z in range(Z):
                self.index.append((case_id, z))
                self.slice_has_true_target.append(float(Y_true[z].sum() > 0))
        
        self.slice_has_true_target = np.asarray(self.slice_has_true_target, dtype=np.float32)
    
    def __len__(self):
        return len(self.index)
    
    def __getitem__(self, idx):
        case_id, z = self.index[idx]
        X, Y_true, B_direct, P_teacher, delta_teacher = self.case_data[case_id]
        
        # Input channels: T1c, current mask, direct baseline prediction B
        x3 = np.concatenate([
            X[z],                         # [2,H,W]
            B_direct[z][None, :, :]       # [1,H,W]
        ], axis=0).astype(np.float32)     # [3,H,W]
        
        y_true = Y_true[z].astype(np.float32)                         # [1,H,W]
        b = B_direct[z][None, :, :].astype(np.float32)                # [1,H,W]
        delta = delta_teacher[z][None, :, :].astype(np.float32)       # [1,H,W]
        
        return (
            torch.from_numpy(x3).float(),
            torch.from_numpy(y_true).float(),
            torch.from_numpy(b).float(),
            torch.from_numpy(delta).float(),
            case_id,
            z
        )


def make_weighted_sampler(dataset):
    weights = np.where(
        dataset.slice_has_true_target > 0,
        POS_SLICE_WEIGHT,
        NEG_SLICE_WEIGHT
    ).astype(np.float32)
    
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True
    )


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.net(x)


class SmallUNet2DResidual(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=16):
        super().__init__()
        
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)
        
        self.pool = nn.MaxPool2d(2)
        
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        
        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        
        self.out = nn.Conv2d(base, out_ch, kernel_size=1)
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        
        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        # Raw residual; tanh will convert it into [-1, 1]
        return self.out(d1)


# =====================
# Loss functions
# =====================
def weighted_bce_prob(pred_prob, target, pos_weight_tensor, eps=1e-6):
    pred_prob = torch.clamp(pred_prob, eps, 1.0 - eps)
    loss_pos = -pos_weight_tensor * target * torch.log(pred_prob)
    loss_neg = -(1.0 - target) * torch.log(1.0 - pred_prob)
    return (loss_pos + loss_neg).mean()


def soft_dice_loss_prob(pred_prob, target, eps=1e-6):
    pred_prob = torch.clamp(pred_prob, 0.0, 1.0)
    dims = (1, 2, 3)
    inter = torch.sum(pred_prob * target, dim=dims)
    denom = torch.sum(pred_prob, dim=dims) + torch.sum(target, dim=dims)
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def residual_smooth_l1_loss(delta_pred, delta_teacher):
    # Higher weight where PCC made larger correction
    w = 1.0 + RESIDUAL_IMPORTANCE_WEIGHT * torch.abs(delta_teacher)
    raw = F.smooth_l1_loss(delta_pred, delta_teacher, reduction="none")
    return (w * raw).mean()


def combined_c2_loss(delta_raw, b_direct, y_true, delta_teacher, pos_weight_tensor):
    delta_pred = torch.tanh(delta_raw)  # [-1, 1]
    final_prob = torch.clamp(b_direct + delta_pred, 0.0, 1.0)
    
    true_bce = weighted_bce_prob(final_prob, y_true, pos_weight_tensor)
    true_dice = soft_dice_loss_prob(final_prob, y_true)
    true_loss = 0.5 * true_bce + 0.5 * true_dice
    
    pcc_residual_loss = residual_smooth_l1_loss(delta_pred, delta_teacher)
    
    total = TRUE_LOSS_WEIGHT * true_loss + PCC_RESIDUAL_LOSS_WEIGHT * pcc_residual_loss
    
    return total, true_loss.detach(), true_bce.detach(), true_dice.detach(), pcc_residual_loss.detach(), final_prob.detach(), delta_pred.detach()


# =====================
# Fold construction
# =====================
fold_splits = []

for fold in sorted(split_df["fold"].unique()):
    train_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "train")]["case_id"].tolist()
    test_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")]["case_id"].tolist()
    fold_splits.append((int(fold), train_case_ids, test_case_ids))

print("\nFold test cases:")
for fold, train_case_ids, test_case_ids in fold_splits:
    print(f"\nFold {fold}:")
    for cid in test_case_ids:
        print(" ", cid)


# =====================
# Train / predict one fold
# =====================
def train_one_fold(fold, train_case_ids, test_case_ids):
    print("\n" + "=" * 80)
    print(f"Model C2 / PCC-residual correction student / Fold {fold}")
    print("Train cases:", len(train_case_ids))
    print("Test cases:", len(test_case_ids))
    
    pos_weight, true_pos, true_total = compute_train_pos_weight_from_true_target(train_case_ids)
    print(f"True target positive voxels: {true_pos:.0f} / {true_total:.0f}")
    print(f"BCE pos_weight clipped: {pos_weight:.4f}")
    
    train_ds = PCCResidualDataset(train_case_ids)
    train_sampler = make_weighted_sampler(train_ds)
    
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    
    model = SmallUNet2DResidual(in_ch=3, out_ch=1, base=16).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)
    
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    
    fold_loss_rows = []
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_total = 0.0
        running_true = 0.0
        running_bce = 0.0
        running_dice = 0.0
        running_res = 0.0
        n_batches = 0
        
        pbar = tqdm(train_loader, desc=f"Model C2 Fold {fold} Epoch {epoch}/{EPOCHS}", leave=False)
        
        for x3, y_true, b_direct, delta_teacher, _, _ in pbar:
            x3 = x3.to(DEVICE, non_blocking=True)
            y_true = y_true.to(DEVICE, non_blocking=True)
            b_direct = b_direct.to(DEVICE, non_blocking=True)
            delta_teacher = delta_teacher.to(DEVICE, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                delta_raw = model(x3)
                loss, true_loss, true_bce, true_dice, res_loss, _, _ = combined_c2_loss(
                    delta_raw=delta_raw,
                    b_direct=b_direct,
                    y_true=y_true,
                    delta_teacher=delta_teacher,
                    pos_weight_tensor=pos_weight_tensor
                )
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_total += float(loss.detach().cpu())
            running_true += float(true_loss.detach().cpu())
            running_bce += float(true_bce.detach().cpu())
            running_dice += float(true_dice.detach().cpu())
            running_res += float(res_loss.detach().cpu())
            n_batches += 1
            
            pbar.set_postfix({
                "loss": running_total / max(n_batches, 1),
                "true": running_true / max(n_batches, 1),
                "res": running_res / max(n_batches, 1),
            })
        
        scheduler.step()
        
        epoch_row = {
            "fold": fold,
            "epoch": epoch,
            "loss": running_total / max(n_batches, 1),
            "true_loss": running_true / max(n_batches, 1),
            "true_bce": running_bce / max(n_batches, 1),
            "true_dice_loss": running_dice / max(n_batches, 1),
            "pcc_residual_loss": running_res / max(n_batches, 1),
            "lr": scheduler.get_last_lr()[0],
            "true_pos_weight": pos_weight,
            "true_loss_weight": TRUE_LOSS_WEIGHT,
            "pcc_residual_loss_weight": PCC_RESIDUAL_LOSS_WEIGHT,
        }
        fold_loss_rows.append(epoch_row)
        
        print(
            f"Model C2 Fold {fold} Epoch {epoch:02d}: "
            f"loss={epoch_row['loss']:.5f}, "
            f"true={epoch_row['true_loss']:.5f}, "
            f"res={epoch_row['pcc_residual_loss']:.5f}, "
            f"dice_loss={epoch_row['true_dice_loss']:.5f}"
        )
    
    ckpt_path = CKPT_DIR / f"pcc_residual_student_fold_{fold}.pt"
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "train_case_ids": list(train_case_ids),
        "test_case_ids": list(test_case_ids),
        "config": {
            "method": "PCC-residual correction student",
            "input_channels": "current T1c + current mask + Model A direct prediction",
            "final_prediction": "Model A direct prediction + learned residual",
            "teacher": "PCC-v2 rounds=10 eta_pos=0.30 eta_neg=0.10",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "pos_slice_weight": POS_SLICE_WEIGHT,
            "neg_slice_weight": NEG_SLICE_WEIGHT,
            "true_loss_weight": TRUE_LOSS_WEIGHT,
            "pcc_residual_loss_weight": PCC_RESIDUAL_LOSS_WEIGHT,
            "residual_importance_weight": RESIDUAL_IMPORTANCE_WEIGHT,
            "true_pos_weight": pos_weight,
        }
    }, ckpt_path)
    
    print("Saved checkpoint:", ckpt_path)
    
    # Predict test cases
    model.eval()
    case_metric_rows = []
    
    with torch.no_grad():
        for case_id in tqdm(test_case_ids, desc=f"Model C2 Fold {fold} prediction"):
            X, Y_true = load_case_npz(case_id)
            target = Y_true[:, 0].astype(bool)
            B_direct = load_modelA_pred(case_id)  # [Z,H,W]
            
            preds_final = []
            preds_delta = []
            
            for start in range(0, X.shape[0], BATCH_SIZE):
                x_part = X[start:start+BATCH_SIZE]                  # [b,2,H,W]
                b_part = B_direct[start:start+BATCH_SIZE]           # [b,H,W]
                
                x3 = np.concatenate([
                    x_part,
                    b_part[:, None, :, :]
                ], axis=1).astype(np.float32)
                
                x3_t = torch.from_numpy(x3).float().to(DEVICE)
                b_t = torch.from_numpy(b_part[:, None, :, :]).float().to(DEVICE)
                
                delta_raw = model(x3_t)
                delta_pred = torch.tanh(delta_raw)
                final_prob = torch.clamp(b_t + delta_pred, 0.0, 1.0)
                
                preds_final.append(final_prob.detach().cpu().numpy()[:, 0].astype(np.float32))
                preds_delta.append(delta_pred.detach().cpu().numpy()[:, 0].astype(np.float32))
            
            final_map = np.concatenate(preds_final, axis=0)
            delta_map = np.concatenate(preds_delta, axis=0)
            
            final_map = normalize_prob_map(final_map)
            delta_map = np.clip(delta_map, -1, 1).astype(np.float32)
            
            final_path = PRED_DIR / f"{case_id}_pcc_residual_final_fold_{fold}.npy"
            delta_path = RESIDUAL_PRED_DIR / f"{case_id}_pcc_residual_delta_fold_{fold}.npy"
            
            np.save(final_path, final_map.astype(np.float16))
            np.save(delta_path, delta_map.astype(np.float16))
            
            final_metrics = topk_metrics(final_map, target)
            base_metrics = topk_metrics(B_direct, target)
            
            case_metric_rows.append({
                "case_id": case_id,
                "fold": fold,
                "method": "PCC-residual correction student",
                "training_target": "true target + PCC residual teacher",
                "test_time_future_target_access": "No",
                "input_uses_model_A_prediction": "Yes",
                "final_pred_path": str(final_path),
                "delta_pred_path": str(delta_path),
                "model_A_direct_dice": base_metrics["dice"],
                "model_A_direct_iou": base_metrics["iou"],
                "model_A_direct_target_focus": base_metrics["target_focus"],
                "model_A_direct_log10_ratio": base_metrics["log10_ratio"],
                **final_metrics,
            })
    
    return fold_loss_rows, case_metric_rows


# =====================
# Run all folds
# =====================
all_loss_rows = []
all_case_metric_rows = []

start_time = time.time()

for fold, train_case_ids, test_case_ids in fold_splits:
    fold_loss_rows, case_metric_rows = train_one_fold(
        fold=fold,
        train_case_ids=train_case_ids,
        test_case_ids=test_case_ids
    )
    
    all_loss_rows.extend(fold_loss_rows)
    all_case_metric_rows.extend(case_metric_rows)
    
    pd.DataFrame(all_loss_rows).to_csv(MODEL_C2_DIR / "pcc_residual_training_losses_partial.csv", index=False)
    pd.DataFrame(all_case_metric_rows).to_csv(MODEL_C2_DIR / "pcc_residual_case_metrics_partial.csv", index=False)


# =====================
# Save final results
# =====================
loss_df = pd.DataFrame(all_loss_rows)
case_metrics_df = pd.DataFrame(all_case_metric_rows)

loss_path = MODEL_C2_DIR / "pcc_residual_training_losses.csv"
case_metrics_path = MODEL_C2_DIR / "pcc_residual_case_metrics.csv"

loss_df.to_csv(loss_path, index=False)
case_metrics_df.to_csv(case_metrics_path, index=False)

summary = (
    case_metrics_df
    .groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
)

summary_path = MODEL_C2_DIR / "pcc_residual_summary.csv"
summary.to_csv(summary_path)

# Pairwise vs Model A direct prediction input B
pairwise_rows = []
for metric, base_col in [
    ("dice", "model_A_direct_dice"),
    ("iou", "model_A_direct_iou"),
    ("target_focus", "model_A_direct_target_focus"),
    ("log10_ratio", "model_A_direct_log10_ratio"),
]:
    diff = case_metrics_df[metric] - case_metrics_df[base_col]
    pairwise_rows.append({
        "comparison": "C2 final prediction vs Model A direct prediction input",
        "metric": metric,
        "mean_diff": float(diff.mean()),
        "median_diff": float(diff.median()),
        "c2_wins": int((diff > 0).sum()),
        "model_A_wins": int((diff < 0).sum()),
        "ties": int((diff == 0).sum()),
        "total": int(diff.notna().sum()),
        "c2_win_rate": float((diff > 0).mean()),
    })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = MODEL_C2_DIR / "pairwise_C2_vs_model_A_input.csv"
pairwise_df.to_csv(pairwise_path, index=False)

runtime_min = (time.time() - start_time) / 60.0

run_info = {
    "method": "PCC-residual correction student",
    "stage": "Stage 2 / matched 5-fold residual correction learning",
    "teacher": "PCC-v2 rounds=10 eta_pos=0.30 eta_neg=0.10",
    "input": "current T1c + current tumour mask + Model A direct prediction",
    "output": "learned residual delta; final prediction = Model A direct prediction + delta",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "test_time_future_target_access": "No",
    "split_file": str(SPLIT_PATH),
    "runtime_minutes": runtime_min,
    "outputs": {
        "teacher_quality_path": str(teacher_quality_path),
        "teacher_quality_summary_path": str(teacher_quality_summary_path),
        "loss_path": str(loss_path),
        "case_metrics_path": str(case_metrics_path),
        "summary_path": str(summary_path),
        "pairwise_path": str(pairwise_path),
        "prediction_folder": str(PRED_DIR),
        "residual_prediction_folder": str(RESIDUAL_PRED_DIR),
        "checkpoint_folder": str(CKPT_DIR),
    }
}

with open(MODEL_C2_DIR / "pcc_residual_run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)

print("\n" + "=" * 80)
print("Model C2 / PCC-residual correction student finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved losses:", loss_path)
print("Saved case metrics:", case_metrics_path)
print("Saved summary:", summary_path)
print("Saved pairwise:", pairwise_path)
print("Saved run info:", MODEL_C2_DIR / "pcc_residual_run_info.json")

print("\nPCC-v2 teacher quality summary:")
display(teacher_quality_summary)

print("\nModel C2 summary:")
display(summary)

print("\nPairwise C2 vs Model A input:")
display(pairwise_df)

print("\nCase metrics:")
display(case_metrics_df)

print("\nTraining losses tail:")
display(loss_df.tail(10))


# =====================
# Compact comparison with Model A/B/C/C2
# =====================
comparison_rows = []

def add_summary_from_case_metrics(path, model_name, training_target, note):
    if not Path(path).exists():
        print("Comparison file not found:", path)
        return
    
    df = pd.read_csv(path)
    m = df[["dice", "iou", "target_focus", "log10_ratio"]].mean()
    comparison_rows.append({
        "Model": model_name,
        "Training target / mechanism": training_target,
        "Note": note,
        "Dice_mean": float(m["dice"]),
        "IoU_mean": float(m["iou"]),
        "Target_focus_mean": float(m["target_focus"]),
        "Log10_ratio_mean": float(m["log10_ratio"]),
    })

add_summary_from_case_metrics(
    STAGE2_DIR / "model_A_direct_target_5fold" / "direct_target_case_metrics.csv",
    "Model A",
    "true future-change target",
    "direct prediction"
)

add_summary_from_case_metrics(
    STAGE2_DIR / "model_B_eia_teacher_lambda030_5fold" / "eia_teacher_case_metrics.csv",
    "Model B",
    "EIA Linear lambda=0.30",
    "teacher final map replacement"
)

add_summary_from_case_metrics(
    STAGE2_DIR / "model_C_pcc_v2_teacher_eta_neg010_5fold" / "pcc_teacher_case_metrics.csv",
    "Model C",
    "PCC-v2 final teacher map",
    "teacher final map replacement"
)

add_summary_from_case_metrics(
    case_metrics_path,
    "Model C2",
    "true target + PCC residual correction",
    "learns PCC correction residual over Model A prediction"
)

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = MODEL_C2_DIR / "model_A_B_C_C2_compact_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nCompact comparison:")
display(comparison_df)
print("Saved compact comparison:", comparison_path)

In [ ]:
# ============================================================
# Model C2-fixed:
# PCC-on-Model-A Logit-Residual Correction Student
#
# Core idea:
#   B_A = Model A out-of-fold direct prediction
#   PCC_A = PCC-v2(B_A, true future-change target)
#   delta_teacher = logit(PCC_A) - logit(B_A)
#
# Student input:
#   current T1c + current mask + B_A
#
# Student output:
#   delta_hat in logit space
#
# Final prediction:
#   sigmoid(logit(B_A) + delta_hat)
#
# Test-time:
#   Uses only current T1c + current mask + Model A prediction.
#   Does NOT use future target or PCC teacher at inference.
# ============================================================

import os
import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from scipy.ndimage import gaussian_filter, distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import gaussian_filter, distance_transform_edt


# =====================
# Config
# =====================
SEED = 42
EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

POS_SLICE_WEIGHT = 3.0
NEG_SLICE_WEIGHT = 1.0

# Important: true target keeps the final prediction aligned with the actual evaluation target.
# PCC residual is the correction mechanism signal.
TRUE_LOSS_WEIGHT = 0.85
PCC_RESIDUAL_LOSS_WEIGHT = 0.15

RESIDUAL_IMPORTANCE_WEIGHT = 2.0
MAX_DELTA_LOGIT = 8.0
LOGIT_EPS = 1e-4

PCC_ROUNDS = 10
PCC_ETA_POS = 0.30
PCC_ETA_NEG = 0.10
PCC_DILATION_RADIUS = 26
PCC_SIGMA = 2.0

FORCE_REGENERATE_TEACHER = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"
MODEL_A_PRED_DIR = MODEL_A_DIR / "direct_target_pred_maps"

LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_C2_DIR = STAGE2_DIR / "model_C2_fixed_pcc_on_modelA_logit_residual"
PCC_ON_A_TEACHER_DIR = STAGE2_DIR / "teacher_maps_pcc_on_modelA_round10_eta030_neg010"

PRED_DIR = MODEL_C2_DIR / "final_pred_maps"
DELTA_DIR = MODEL_C2_DIR / "delta_logit_pred_maps"
CKPT_DIR = MODEL_C2_DIR / "checkpoints"

MODEL_C2_DIR.mkdir(parents=True, exist_ok=True)
PCC_ON_A_TEACHER_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
DELTA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert SPLIT_PATH.exists(), f"Missing split file: {SPLIT_PATH}"
assert MODEL_A_PRED_DIR.exists(), f"Missing Model A prediction folder: {MODEL_A_PRED_DIR}"

split_df = pd.read_csv(SPLIT_PATH)
all_case_ids = sorted(split_df["case_id"].unique().tolist())

test_split_df = split_df[split_df["split"] == "test"].copy()
case_to_modelA_fold = dict(zip(test_split_df["case_id"], test_split_df["fold"]))

print("Using split file:", SPLIT_PATH)
print("Model A prediction folder:", MODEL_A_PRED_DIR)
print("C2 output folder:", MODEL_C2_DIR)
print("Cases:", len(all_case_ids))


# =====================
# Robust preprocessing summary recovery
# =====================
def find_or_rebuild_case_to_npz():
    if LOCKED_PRE_SUMMARY_PATH.exists():
        print("Found preprocessing summary:", LOCKED_PRE_SUMMARY_PATH)
        summary_df = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
        return dict(zip(summary_df["case_id"], summary_df["npz_path"]))

    print("Missing preprocessing summary. Trying to rebuild it by scanning .npz files...")
    candidates = list(OUT_DIR.rglob("*.npz"))

    mapping = {}

    for p in candidates:
        try:
            with np.load(p, allow_pickle=True) as data:
                keys = set(data.files)
                if not {"X", "Y"}.issubset(keys):
                    continue

                cid = None
                if "case_id" in keys:
                    raw = data["case_id"]
                    try:
                        cid = str(raw.item())
                    except Exception:
                        cid = str(raw)

                if cid in all_case_ids:
                    mapping[cid] = str(p)
                    continue

                for case_id in all_case_ids:
                    if case_id in str(p):
                        mapping[case_id] = str(p)
                        break

        except Exception:
            continue

    missing = [c for c in all_case_ids if c not in mapping]

    if missing:
        raise FileNotFoundError(
            "Cannot rebuild preprocessing summary. Missing npz for cases:\n"
            + "\n".join(missing[:20])
            + "\n\nYou need to rerun the locked40 preprocessing step or restore the Kaggle working outputs."
        )

    summary_df = pd.DataFrame([
        {"case_id": cid, "npz_path": mapping[cid]}
        for cid in all_case_ids
    ])
    summary_df.to_csv(LOCKED_PRE_SUMMARY_PATH, index=False)

    print("Rebuilt preprocessing summary:", LOCKED_PRE_SUMMARY_PATH)
    return mapping


case_to_npz = find_or_rebuild_case_to_npz()


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)

    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)

    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def prob_to_logit_np(p, eps=LOGIT_EPS):
    p = np.clip(p.astype(np.float32), eps, 1.0 - eps)
    return np.log(p / (1.0 - p)).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = normalize_prob_map(pred)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)

    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case_npz(case_id):
    npz_path = case_to_npz[case_id]
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"].astype(np.float32)
    return X, Y


def get_modelA_pred_path(case_id):
    fold = int(case_to_modelA_fold[case_id])
    return MODEL_A_PRED_DIR / f"{case_id}_direct_target_student_fold_{fold}.npy"


def load_modelA_pred(case_id):
    path = get_modelA_pred_path(case_id)
    assert path.exists(), f"Missing Model A prediction: {path}"
    B = np.load(path).astype(np.float32)
    B = normalize_prob_map(B)
    return B


def get_pcc_on_A_teacher_path(case_id):
    return PCC_ON_A_TEACHER_DIR / f"{case_id}_PCC_on_ModelA_round10_eta030_neg010.npy"


def make_slice_wise_dilation_region(current_mask, radius):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    R = np.zeros_like(current_mask, dtype=bool)

    for z in range(Z):
        m = current_mask[z]
        if m.sum() > 0:
            dist = distance_transform_edt(~m)
            R[z] = dist <= radius
        else:
            R[z] = False

    return R


def smooth_target_2d(target, sigma):
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))

    if S.max() > 0:
        S = S / (S.max() + 1e-8)

    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_v2_on_modelA(
    B_A,
    current_mask,
    target,
    rounds=PCC_ROUNDS,
    eta_pos=PCC_ETA_POS,
    eta_neg=PCC_ETA_NEG,
    dilation_radius=PCC_DILATION_RADIUS,
    sigma=PCC_SIGMA
):
    B_A = normalize_prob_map(B_A)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)

    R = make_slice_wise_dilation_region(current_mask, dilation_radius)
    S = smooth_target_2d(target, sigma)

    P = B_A.copy().astype(np.float32)

    for _ in range(rounds):
        P_new = P.copy()

        positive_signal = S * (1.0 - P)
        negative_signal = (1.0 - S) * P
        negative_signal[target] = 0.0

        P_new[R] = (
            P[R]
            + eta_pos * positive_signal[R]
            - eta_neg * negative_signal[R]
        )

        P_new[~R] = B_A[~R]
        P = np.clip(P_new, 0, 1).astype(np.float32)

    return P


# =====================
# Step 1: Generate PCC-on-Model-A teacher maps
# =====================
teacher_rows = []

print("\nGenerating PCC-on-Model-A teacher maps...")

for case_id in tqdm(all_case_ids, desc="PCC-on-A teacher"):
    teacher_path = get_pcc_on_A_teacher_path(case_id)

    X, Y = load_case_npz(case_id)
    current_mask = X[:, 1].astype(bool)
    target = Y[:, 0].astype(bool)

    B_A = load_modelA_pred(case_id)

    if teacher_path.exists() and not FORCE_REGENERATE_TEACHER:
        P_teacher = np.load(teacher_path).astype(np.float32)
        P_teacher = normalize_prob_map(P_teacher)
    else:
        P_teacher = run_pcc_v2_on_modelA(
            B_A=B_A,
            current_mask=current_mask,
            target=target,
            rounds=PCC_ROUNDS,
            eta_pos=PCC_ETA_POS,
            eta_neg=PCC_ETA_NEG,
            dilation_radius=PCC_DILATION_RADIUS,
            sigma=PCC_SIGMA,
        )
        np.save(teacher_path, P_teacher.astype(np.float16))

    base_metrics = topk_metrics(B_A, target)
    teacher_metrics = topk_metrics(P_teacher, target)

    row = {
        "case_id": case_id,
        "model_A_dice": base_metrics["dice"],
        "model_A_iou": base_metrics["iou"],
        "model_A_target_focus": base_metrics["target_focus"],
        "model_A_log10_ratio": base_metrics["log10_ratio"],
        "teacher_dice": teacher_metrics["dice"],
        "teacher_iou": teacher_metrics["iou"],
        "teacher_target_focus": teacher_metrics["target_focus"],
        "teacher_log10_ratio": teacher_metrics["log10_ratio"],
        "teacher_path": str(teacher_path),
    }
    teacher_rows.append(row)

teacher_df = pd.DataFrame(teacher_rows)
teacher_path_csv = PCC_ON_A_TEACHER_DIR / "pcc_on_modelA_teacher_quality_case_metrics.csv"
teacher_df.to_csv(teacher_path_csv, index=False)

teacher_summary = pd.DataFrame([{
    "model_A_dice_mean": teacher_df["model_A_dice"].mean(),
    "model_A_iou_mean": teacher_df["model_A_iou"].mean(),
    "teacher_dice_mean": teacher_df["teacher_dice"].mean(),
    "teacher_iou_mean": teacher_df["teacher_iou"].mean(),
    "teacher_target_focus_mean": teacher_df["teacher_target_focus"].mean(),
    "teacher_log10_ratio_mean": teacher_df["teacher_log10_ratio"].mean(),
}])

teacher_summary_path = PCC_ON_A_TEACHER_DIR / "pcc_on_modelA_teacher_quality_summary.csv"
teacher_summary.to_csv(teacher_summary_path, index=False)

print("\nPCC-on-Model-A teacher quality summary:")
display(teacher_summary)
print("Saved teacher quality:", teacher_path_csv)


def load_pcc_on_A_teacher(case_id):
    path = get_pcc_on_A_teacher_path(case_id)
    assert path.exists(), f"Missing PCC-on-A teacher: {path}"
    P = np.load(path).astype(np.float32)
    return normalize_prob_map(P)


def compute_train_pos_weight_from_true_target(train_case_ids):
    pos = 0.0
    total = 0.0

    for case_id in train_case_ids:
        _, Y = load_case_npz(case_id)
        pos += float(Y.sum())
        total += float(Y.size)

    neg = total - pos
    pos_weight = neg / (pos + 1e-8)
    pos_weight = float(np.clip(pos_weight, 1.0, 50.0))
    return pos_weight, pos, total


# =====================
# Dataset
# =====================
class PCCOnModelAResidualDataset(Dataset):
    def __init__(self, case_ids):
        self.case_ids = list(case_ids)

        self.case_data = {}
        self.index = []
        self.slice_has_true_target = []

        for case_id in self.case_ids:
            X, Y_true = load_case_npz(case_id)
            B_A = load_modelA_pred(case_id)
            P_teacher = load_pcc_on_A_teacher(case_id)

            assert X.shape[0] == B_A.shape[0] == P_teacher.shape[0], f"Z mismatch: {case_id}"
            assert X.shape[-2:] == B_A.shape[-2:] == P_teacher.shape[-2:], f"HW mismatch: {case_id}"

            B_logit = prob_to_logit_np(B_A)
            T_logit = prob_to_logit_np(P_teacher)
            delta_logit_teacher = np.clip(T_logit - B_logit, -MAX_DELTA_LOGIT, MAX_DELTA_LOGIT).astype(np.float32)

            self.case_data[case_id] = (X, Y_true, B_A, B_logit, delta_logit_teacher)

            Z = X.shape[0]
            for z in range(Z):
                self.index.append((case_id, z))
                self.slice_has_true_target.append(float(Y_true[z].sum() > 0))

        self.slice_has_true_target = np.asarray(self.slice_has_true_target, dtype=np.float32)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        case_id, z = self.index[idx]
        X, Y_true, B_A, B_logit, delta_logit_teacher = self.case_data[case_id]

        x3 = np.concatenate([
            X[z],
            B_A[z][None, :, :]
        ], axis=0).astype(np.float32)

        y_true = Y_true[z].astype(np.float32)
        b_logit = B_logit[z][None, :, :].astype(np.float32)
        delta_teacher = delta_logit_teacher[z][None, :, :].astype(np.float32)

        return (
            torch.from_numpy(x3).float(),
            torch.from_numpy(y_true).float(),
            torch.from_numpy(b_logit).float(),
            torch.from_numpy(delta_teacher).float(),
            case_id,
            z
        )


def make_weighted_sampler(dataset):
    weights = np.where(
        dataset.slice_has_true_target > 0,
        POS_SLICE_WEIGHT,
        NEG_SLICE_WEIGHT
    ).astype(np.float32)

    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True
    )


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2DResidual(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


# =====================
# Loss
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def residual_loss(delta_pred, delta_teacher):
    w = 1.0 + RESIDUAL_IMPORTANCE_WEIGHT * torch.clamp(torch.abs(delta_teacher) / MAX_DELTA_LOGIT, 0.0, 1.0)
    raw = F.smooth_l1_loss(delta_pred, delta_teacher, reduction="none")
    return (w * raw).mean()


def combined_c2_loss(delta_raw, b_logit, y_true, delta_teacher, pos_weight_tensor):
    delta_pred = MAX_DELTA_LOGIT * torch.tanh(delta_raw)
    final_logit = b_logit + delta_pred

    true_bce = F.binary_cross_entropy_with_logits(
        final_logit,
        y_true,
        pos_weight=pos_weight_tensor
    )
    true_dice = soft_dice_loss_from_logits(final_logit, y_true)
    true_loss = 0.5 * true_bce + 0.5 * true_dice

    res_loss = residual_loss(delta_pred, delta_teacher)

    total = TRUE_LOSS_WEIGHT * true_loss + PCC_RESIDUAL_LOSS_WEIGHT * res_loss

    return total, true_loss.detach(), true_bce.detach(), true_dice.detach(), res_loss.detach(), final_logit.detach(), delta_pred.detach()


# =====================
# Folds
# =====================
fold_splits = []

for fold in sorted(split_df["fold"].unique()):
    train_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "train")]["case_id"].tolist()
    test_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")]["case_id"].tolist()
    fold_splits.append((int(fold), train_case_ids, test_case_ids))

print("\nFold test cases:")
for fold, _, test_case_ids in fold_splits:
    print(f"\nFold {fold}:")
    for cid in test_case_ids:
        print(" ", cid)


# =====================
# Train / predict
# =====================
def train_one_fold(fold, train_case_ids, test_case_ids):
    print("\n" + "=" * 80)
    print(f"Model C2-fixed / PCC-on-Model-A residual / Fold {fold}")
    print("Train cases:", len(train_case_ids))
    print("Test cases:", len(test_case_ids))

    pos_weight, true_pos, true_total = compute_train_pos_weight_from_true_target(train_case_ids)
    print(f"True target positive voxels: {true_pos:.0f} / {true_total:.0f}")
    print(f"BCE pos_weight clipped: {pos_weight:.4f}")

    train_ds = PCCOnModelAResidualDataset(train_case_ids)
    train_sampler = make_weighted_sampler(train_ds)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    model = SmallUNet2DResidual(in_ch=3, out_ch=1, base=16).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    fold_loss_rows = []

    for epoch in range(1, EPOCHS + 1):
        model.train()

        running_total = 0.0
        running_true = 0.0
        running_bce = 0.0
        running_dice = 0.0
        running_res = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"C2 Fold {fold} Epoch {epoch}/{EPOCHS}", leave=False)

        for x3, y_true, b_logit, delta_teacher, _, _ in pbar:
            x3 = x3.to(DEVICE, non_blocking=True)
            y_true = y_true.to(DEVICE, non_blocking=True)
            b_logit = b_logit.to(DEVICE, non_blocking=True)
            delta_teacher = delta_teacher.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                delta_raw = model(x3)
                loss, true_loss, true_bce, true_dice, res_loss, _, _ = combined_c2_loss(
                    delta_raw=delta_raw,
                    b_logit=b_logit,
                    y_true=y_true,
                    delta_teacher=delta_teacher,
                    pos_weight_tensor=pos_weight_tensor
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_total += float(loss.detach().cpu())
            running_true += float(true_loss.detach().cpu())
            running_bce += float(true_bce.detach().cpu())
            running_dice += float(true_dice.detach().cpu())
            running_res += float(res_loss.detach().cpu())
            n_batches += 1

            pbar.set_postfix({
                "loss": running_total / max(n_batches, 1),
                "true": running_true / max(n_batches, 1),
                "res": running_res / max(n_batches, 1),
            })

        scheduler.step()

        epoch_row = {
            "fold": fold,
            "epoch": epoch,
            "loss": running_total / max(n_batches, 1),
            "true_loss": running_true / max(n_batches, 1),
            "true_bce": running_bce / max(n_batches, 1),
            "true_dice_loss": running_dice / max(n_batches, 1),
            "pcc_residual_loss": running_res / max(n_batches, 1),
            "lr": scheduler.get_last_lr()[0],
            "true_pos_weight": pos_weight,
        }
        fold_loss_rows.append(epoch_row)

        print(
            f"C2 Fold {fold} Epoch {epoch:02d}: "
            f"loss={epoch_row['loss']:.5f}, "
            f"true={epoch_row['true_loss']:.5f}, "
            f"res={epoch_row['pcc_residual_loss']:.5f}, "
            f"dice_loss={epoch_row['true_dice_loss']:.5f}"
        )

    ckpt_path = CKPT_DIR / f"c2_fixed_pcc_on_modelA_residual_fold_{fold}.pt"
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "train_case_ids": list(train_case_ids),
        "test_case_ids": list(test_case_ids),
        "config": {
            "method": "PCC-on-Model-A logit residual correction",
            "input": "current T1c + current mask + Model A prediction",
            "teacher": "PCC-v2 applied directly to Model A prediction",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "true_loss_weight": TRUE_LOSS_WEIGHT,
            "pcc_residual_loss_weight": PCC_RESIDUAL_LOSS_WEIGHT,
            "max_delta_logit": MAX_DELTA_LOGIT,
        }
    }, ckpt_path)

    print("Saved checkpoint:", ckpt_path)

    model.eval()
    metric_rows = []

    with torch.no_grad():
        for case_id in tqdm(test_case_ids, desc=f"C2 Fold {fold} prediction"):
            X, Y = load_case_npz(case_id)
            target = Y[:, 0].astype(bool)
            B_A = load_modelA_pred(case_id)
            B_logit = prob_to_logit_np(B_A)

            final_parts = []
            delta_parts = []

            for start in range(0, X.shape[0], BATCH_SIZE):
                x_part = X[start:start+BATCH_SIZE]
                b_part = B_A[start:start+BATCH_SIZE]
                b_logit_part = B_logit[start:start+BATCH_SIZE]

                x3 = np.concatenate([
                    x_part,
                    b_part[:, None, :, :]
                ], axis=1).astype(np.float32)

                x3_t = torch.from_numpy(x3).float().to(DEVICE)
                b_logit_t = torch.from_numpy(b_logit_part[:, None, :, :]).float().to(DEVICE)

                delta_raw = model(x3_t)
                delta_pred = MAX_DELTA_LOGIT * torch.tanh(delta_raw)
                final_logit = b_logit_t + delta_pred
                final_prob = torch.sigmoid(final_logit)

                final_parts.append(final_prob.detach().cpu().numpy()[:, 0].astype(np.float32))
                delta_parts.append(delta_pred.detach().cpu().numpy()[:, 0].astype(np.float32))

            final_map = np.concatenate(final_parts, axis=0)
            delta_map = np.concatenate(delta_parts, axis=0)

            final_map = normalize_prob_map(final_map)
            delta_map = np.clip(delta_map, -MAX_DELTA_LOGIT, MAX_DELTA_LOGIT).astype(np.float32)

            final_path = PRED_DIR / f"{case_id}_c2_fixed_final_fold_{fold}.npy"
            delta_path = DELTA_DIR / f"{case_id}_c2_fixed_delta_logit_fold_{fold}.npy"

            np.save(final_path, final_map.astype(np.float16))
            np.save(delta_path, delta_map.astype(np.float16))

            c2_metrics = topk_metrics(final_map, target)
            base_metrics = topk_metrics(B_A, target)

            metric_rows.append({
                "case_id": case_id,
                "fold": fold,
                "method": "C2-fixed PCC-on-Model-A residual student",
                "test_time_future_target_access": "No",
                "input_uses_model_A_prediction": "Yes",
                "training_signal": "true target + PCC-on-Model-A logit residual",
                "final_pred_path": str(final_path),
                "delta_logit_pred_path": str(delta_path),
                "model_A_direct_dice": base_metrics["dice"],
                "model_A_direct_iou": base_metrics["iou"],
                "model_A_direct_target_focus": base_metrics["target_focus"],
                "model_A_direct_log10_ratio": base_metrics["log10_ratio"],
                **c2_metrics,
            })

    return fold_loss_rows, metric_rows


# =====================
# Run all folds
# =====================
all_loss_rows = []
all_metric_rows = []

start_time = time.time()

for fold, train_case_ids, test_case_ids in fold_splits:
    fold_loss_rows, metric_rows = train_one_fold(fold, train_case_ids, test_case_ids)

    all_loss_rows.extend(fold_loss_rows)
    all_metric_rows.extend(metric_rows)

    pd.DataFrame(all_loss_rows).to_csv(MODEL_C2_DIR / "c2_fixed_training_losses_partial.csv", index=False)
    pd.DataFrame(all_metric_rows).to_csv(MODEL_C2_DIR / "c2_fixed_case_metrics_partial.csv", index=False)


loss_df = pd.DataFrame(all_loss_rows)
metrics_df = pd.DataFrame(all_metric_rows)

loss_path = MODEL_C2_DIR / "c2_fixed_training_losses.csv"
metrics_path = MODEL_C2_DIR / "c2_fixed_case_metrics.csv"

loss_df.to_csv(loss_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

summary = metrics_df.groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]].agg(["mean", "median", "min", "max"])
summary_path = MODEL_C2_DIR / "c2_fixed_summary.csv"
summary.to_csv(summary_path)

pairwise_rows = []
for metric, base_col in [
    ("dice", "model_A_direct_dice"),
    ("iou", "model_A_direct_iou"),
    ("target_focus", "model_A_direct_target_focus"),
    ("log10_ratio", "model_A_direct_log10_ratio"),
]:
    diff = metrics_df[metric] - metrics_df[base_col]
    pairwise_rows.append({
        "comparison": "C2-fixed final prediction vs Model A direct prediction",
        "metric": metric,
        "mean_diff": float(diff.mean()),
        "median_diff": float(diff.median()),
        "c2_wins": int((diff > 0).sum()),
        "model_A_wins": int((diff < 0).sum()),
        "ties": int((diff == 0).sum()),
        "total": int(diff.notna().sum()),
        "c2_win_rate": float((diff > 0).mean()),
    })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = MODEL_C2_DIR / "pairwise_C2_fixed_vs_Model_A.csv"
pairwise_df.to_csv(pairwise_path, index=False)

runtime_min = (time.time() - start_time) / 60.0

run_info = {
    "method": "C2-fixed PCC-on-Model-A logit residual student",
    "runtime_minutes": runtime_min,
    "test_time_future_target_access": "No",
    "input": "current T1c + current mask + Model A prediction",
    "teacher": "PCC-v2 applied directly to Model A prediction",
    "outputs": {
        "loss_path": str(loss_path),
        "metrics_path": str(metrics_path),
        "summary_path": str(summary_path),
        "pairwise_path": str(pairwise_path),
        "teacher_quality_path": str(teacher_path_csv),
        "teacher_quality_summary_path": str(teacher_summary_path),
    }
}

with open(MODEL_C2_DIR / "c2_fixed_run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)

print("\n" + "=" * 80)
print("C2-fixed finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved losses:", loss_path)
print("Saved metrics:", metrics_path)
print("Saved summary:", summary_path)
print("Saved pairwise:", pairwise_path)

print("\nPCC-on-Model-A teacher quality summary:")
display(teacher_summary)

print("\nC2-fixed summary:")
display(summary)

print("\nPairwise C2-fixed vs Model A:")
display(pairwise_df)

print("\nTraining losses tail:")
display(loss_df.tail(10))

print("\nCase metrics:")
display(metrics_df)


# =====================
# Compact comparison
# =====================
comparison_rows = []

def add_model(path, model_name, mechanism):
    if not Path(path).exists():
        print("Missing comparison file:", path)
        return
    df = pd.read_csv(path)
    m = df[["dice", "iou", "target_focus", "log10_ratio"]].mean()
    comparison_rows.append({
        "Model": model_name,
        "Mechanism": mechanism,
        "Dice_mean": float(m["dice"]),
        "IoU_mean": float(m["iou"]),
        "Target_focus_mean": float(m["target_focus"]),
        "Log10_ratio_mean": float(m["log10_ratio"]),
    })

add_model(
    STAGE2_DIR / "model_A_direct_target_5fold" / "direct_target_case_metrics.csv",
    "Model A",
    "direct future-change prediction"
)

add_model(
    STAGE2_DIR / "model_B_eia_teacher_lambda030_5fold" / "eia_teacher_case_metrics.csv",
    "Model B",
    "EIA teacher final-map replacement"
)

add_model(
    STAGE2_DIR / "model_C_pcc_v2_teacher_eta_neg010_5fold" / "pcc_teacher_case_metrics.csv",
    "Model C",
    "PCC teacher final-map replacement"
)

add_model(
    metrics_path,
    "Model C2-fixed",
    "PCC-on-Model-A logit residual correction"
)

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = MODEL_C2_DIR / "model_A_B_C_C2_fixed_compact_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nCompact comparison:")
display(comparison_df)
print("Saved compact comparison:", comparison_path)

In [ ]:
from pathlib import Path
import pandas as pd
import re

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"
MODEL_A_PRED_DIR = MODEL_A_DIR / "direct_target_pred_maps"

print("OUT_DIR exists:", OUT_DIR.exists(), OUT_DIR)
print("STAGE2_DIR exists:", STAGE2_DIR.exists(), STAGE2_DIR)
print("MODEL_A_DIR exists:", MODEL_A_DIR.exists(), MODEL_A_DIR)
print("SPLIT_PATH exists:", SPLIT_PATH.exists(), SPLIT_PATH)
print("MODEL_A_PRED_DIR exists:", MODEL_A_PRED_DIR.exists(), MODEL_A_PRED_DIR)

print("\nSearching for Model A prediction maps...")
pred_candidates = list(Path("/kaggle/working").rglob("*_direct_target_student_fold_*.npy"))

print("Found prediction maps:", len(pred_candidates))
for p in pred_candidates[:10]:
    print(" ", p)

if len(pred_candidates) == 40:
    print("\nFound 40 Model A out-of-fold prediction maps. Rebuilding split file...")

    MODEL_A_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_A_PRED_DIR.mkdir(parents=True, exist_ok=True)

    case_to_fold = {}

    for p in pred_candidates:
        name = p.name
        # filename example:
        # PatientID_0008_T4_to_T6_t1c_direct_target_student_fold_1.npy
        m = re.match(r"(.+)_direct_target_student_fold_(\d+)\.npy$", name)
        if not m:
            print("Cannot parse:", name)
            continue
        case_id = m.group(1)
        fold = int(m.group(2))
        case_to_fold[case_id] = fold

        # If found somewhere else, copy path is not necessary unless folder differs.
        # But C2 expects them in MODEL_A_PRED_DIR.
        target_path = MODEL_A_PRED_DIR / p.name
        if p.resolve() != target_path.resolve():
            import shutil
            shutil.copy2(p, target_path)

    all_case_ids = sorted(case_to_fold.keys())

    split_rows = []
    for fold in sorted(set(case_to_fold.values())):
        test_cases = [cid for cid, f in case_to_fold.items() if f == fold]
        train_cases = [cid for cid in all_case_ids if cid not in test_cases]

        for cid in train_cases:
            split_rows.append({"fold": fold, "case_id": cid, "split": "train"})
        for cid in test_cases:
            split_rows.append({"fold": fold, "case_id": cid, "split": "test"})

    split_df = pd.DataFrame(split_rows)
    split_df.to_csv(SPLIT_PATH, index=False)

    print("Rebuilt split file:", SPLIT_PATH)
    print(split_df.groupby(["fold", "split"]).size())

elif len(pred_candidates) == 0:
    print("\nNo Model A prediction maps found.")
    print("This means Kaggle working outputs were probably reset.")
    print("You need to rerun preprocessing if needed, then rerun Model A before C2.")
else:
    print("\nFound some Model A prediction maps, but not 40.")
    print("This is incomplete. Safer choice: rerun Model A.")

In [ ]:
from pathlib import Path

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

print("OUT_DIR exists:", OUT_DIR.exists())

npz_files = list(OUT_DIR.rglob("*.npz"))
print("Found npz files:", len(npz_files))

for p in npz_files[:10]:
    print(p)

In [ ]:
# ============================================================
# Recover locked40 preprocessing npz files
# Output:
#   /kaggle/working/pcc_independent_baseline/manifest_locked_original_40.csv
#   /kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_summary.csv
#   /kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_npz/*.npz
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib


OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NPZ_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"
NPZ_DIR.mkdir(parents=True, exist_ok=True)

LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"


# ---------------------
# Locked 40 case list
# ---------------------
LOCKED_CASES = [
    "PatientID_0003_T1_to_T2_t1c",
    "PatientID_0005_T3_to_T4_t1c",
    "PatientID_0006_T2_to_T4_t1c",
    "PatientID_0007_T2_to_T3_t1c",
    "PatientID_0008_T4_to_T6_t1c",
    "PatientID_0010_T1_to_T4_t1c",
    "PatientID_0011_T1_to_T2_t1c",
    "PatientID_0012_T2_to_T3_t1c",
    "PatientID_0013_T1_to_T2_t1c",
    "PatientID_0014_T1_to_T2_t1c",
    "PatientID_0018_T1_to_T2_t1c",
    "PatientID_0019_T4_to_T5_t1c",
    "PatientID_0020_T1_to_T2_t1c",
    "PatientID_0021_T2_to_T3_t1c",
    "PatientID_0022_T1_to_T2_t1c",
    "PatientID_0024_T2_to_T3_t1c",
    "PatientID_0025_T1_to_T2_t1c",
    "PatientID_0026_T1_to_T2_t1c",
    "PatientID_0029_T1_to_T3_t1c",
    "PatientID_0030_T1_to_T3_t1c",
    "PatientID_0031_T2_to_T3_t1c",
    "PatientID_0032_T1_to_T2_t1c",
    "PatientID_0033_T1_to_T2_t1c",
    "PatientID_0034_T1_to_T2_t1c",
    "PatientID_0035_T1_to_T2_t1c",
    "PatientID_0036_T1_to_T2_t1c",
    "PatientID_0037_T1_to_T2_t1c",
    "PatientID_0038_T1_to_T2_t1c",
    "PatientID_0039_T1_to_T2_t1c",
    "PatientID_0041_T1_to_T2_t1c",
    "PatientID_0044_T1_to_T2_t1c",
    "PatientID_0045_T1_to_T2_t1c",
    "PatientID_0051_T1_to_T3_t1c",
    "PatientID_0052_T1_to_T2_t1c",
    "PatientID_0053_T1_to_T3_t1c",
    "PatientID_0054_T1_to_T2_t1c",
    "PatientID_0055_T1_to_T4_t1c",
    "PatientID_0059_T1_to_T2_t1c",
    "PatientID_0060_T1_to_T2_t1c",
    "PatientID_0062_T1_to_T2_t1c",
]


def parse_case_id(case_id):
    m = re.match(r"(PatientID_\d+)_T(\d+)_to_T(\d+)_t1c$", case_id)
    if not m:
        raise ValueError(f"Cannot parse case_id: {case_id}")
    patient_id = m.group(1)
    current_tp = int(m.group(2))
    future_tp = int(m.group(3))
    return patient_id, current_tp, future_tp


# ---------------------
# Locate MU-Glioma-Post root
# ---------------------
def locate_mu_root():
    input_root = Path("/kaggle/input")
    candidates = []

    for p in input_root.rglob("*"):
        if p.is_dir():
            # A valid root should contain PatientID_* folders
            patient_dirs = list(p.glob("PatientID_*"))
            if len(patient_dirs) > 10:
                candidates.append(p)

    if not candidates:
        print("Could not automatically find MU-Glioma-Post root.")
        print("Top-level /kaggle/input:")
        for p in input_root.iterdir():
            print(" ", p)
        raise FileNotFoundError("No folder containing PatientID_* found under /kaggle/input")

    # Prefer the deepest folder with many PatientID folders
    candidates = sorted(candidates, key=lambda x: len(str(x)), reverse=True)
    root = candidates[0]
    return root


DATA_ROOT = locate_mu_root()
print("Detected MU-Glioma-Post root:", DATA_ROOT)
print("Patient folders:", len(list(DATA_ROOT.glob("PatientID_*"))))


# ---------------------
# File finding
# ---------------------
def find_timepoint_file(patient_id, tp_num, kind):
    """
    kind:
      "t1c" or "mask"
    """
    patient_dir = DATA_ROOT / patient_id
    assert patient_dir.exists(), f"Missing patient folder: {patient_dir}"

    tp_patterns = [
        f"Timepoint_{tp_num}",
        f"Timepoint-{tp_num}",
        f"Timepoint {tp_num}",
        f"T{tp_num}",
        f"TP{tp_num}",
    ]

    if kind == "t1c":
        suffix_patterns = ["*brain_t1c.nii*", "*t1c.nii*"]
    elif kind == "mask":
        suffix_patterns = ["*tumorMask.nii*", "*tumourMask.nii*", "*mask.nii*"]
    else:
        raise ValueError(kind)

    candidate_files = []

    for f in patient_dir.rglob("*.nii*"):
        path_str = str(f)
        if not any(tp_pat in path_str for tp_pat in tp_patterns):
            continue

        name = f.name.lower()
        if kind == "t1c":
            if "t1c" in name and "brain" in name:
                candidate_files.append(f)
        else:
            if "tumormask" in name.lower() or "tumourmask" in name.lower():
                candidate_files.append(f)

    if not candidate_files:
        # fallback with suffix patterns
        for tp_pat in tp_patterns:
            for tp_dir in patient_dir.rglob(f"*{tp_pat}*"):
                if tp_dir.is_dir():
                    for pat in suffix_patterns:
                        candidate_files.extend(list(tp_dir.rglob(pat)))

    candidate_files = sorted(set(candidate_files))

    if len(candidate_files) == 0:
        raise FileNotFoundError(f"Missing {kind} for {patient_id} T{tp_num}")

    # Prefer exact expected names
    return candidate_files[0]


# ---------------------
# Loading and preprocessing
# ---------------------
def load_nii(path):
    arr = nib.load(str(path)).get_fdata()
    arr = np.asarray(arr, dtype=np.float32)
    return arr


def volume_to_z_hw(vol):
    """
    Convert 3D volume to [Z,H,W].
    Usually MU-Glioma-Post is [H,W,Z], but this handles either.
    """
    if vol.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {vol.shape}")

    # Usually the slice axis is the smallest dimension, e.g., 155 in 240x240x155
    slice_axis = int(np.argmin(vol.shape))
    if slice_axis == 0:
        zhw = vol
    elif slice_axis == 1:
        zhw = np.transpose(vol, (1, 0, 2))
    else:
        zhw = np.transpose(vol, (2, 0, 1))

    return zhw.astype(np.float32)


def normalize_mri(vol):
    vol = vol.astype(np.float32)
    finite = np.isfinite(vol)
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)

    foreground = vol[(vol > 0) & finite]
    if foreground.size < 10:
        return np.zeros_like(vol, dtype=np.float32)

    lo, hi = np.percentile(foreground, [1, 99])
    if hi <= lo:
        return np.zeros_like(vol, dtype=np.float32)

    vol = np.clip(vol, lo, hi)
    vol = (vol - lo) / (hi - lo + 1e-8)
    vol = np.clip(vol, 0, 1)
    return vol.astype(np.float32)


manifest_rows = []
summary_rows = []

for case_id in tqdm(LOCKED_CASES, desc="Preprocessing locked40"):
    patient_id, current_tp, future_tp = parse_case_id(case_id)

    cur_t1c_path = find_timepoint_file(patient_id, current_tp, "t1c")
    cur_mask_path = find_timepoint_file(patient_id, current_tp, "mask")
    fut_mask_path = find_timepoint_file(patient_id, future_tp, "mask")

    cur_t1c = volume_to_z_hw(load_nii(cur_t1c_path))
    cur_mask = volume_to_z_hw(load_nii(cur_mask_path)) > 0
    fut_mask = volume_to_z_hw(load_nii(fut_mask_path)) > 0

    assert cur_t1c.shape == cur_mask.shape == fut_mask.shape, (
        f"Shape mismatch {case_id}: "
        f"t1c={cur_t1c.shape}, cur_mask={cur_mask.shape}, fut_mask={fut_mask.shape}"
    )

    cur_t1c_norm = normalize_mri(cur_t1c)

    # Growth target: future tumour area not already current tumour
    target = np.logical_and(fut_mask, ~cur_mask)

    X = np.stack([
        cur_t1c_norm,
        cur_mask.astype(np.float32),
    ], axis=1).astype(np.float32)       # [Z,2,H,W]

    Y = target[:, None, :, :].astype(np.uint8)   # [Z,1,H,W]

    z_indices = np.arange(X.shape[0], dtype=np.int16)

    npz_path = NPZ_DIR / f"{case_id}.npz"

    np.savez_compressed(
        npz_path,
        X=X,
        Y=Y,
        z_indices=z_indices,
        case_id=np.array(case_id),
        patient_id=np.array(patient_id),
        current_tp=np.array(current_tp),
        future_tp=np.array(future_tp),
    )

    target_voxels = int(target.sum())
    current_voxels = int(cur_mask.sum())
    positive_slices = int((target.reshape(target.shape[0], -1).sum(axis=1) > 0).sum())

    manifest_rows.append({
        "case_id": case_id,
        "patient_id": patient_id,
        "current_tp": current_tp,
        "future_tp": future_tp,
        "current_t1c_path": str(cur_t1c_path),
        "current_mask_path": str(cur_mask_path),
        "future_mask_path": str(fut_mask_path),
        "growth_voxels": target_voxels,
    })

    summary_rows.append({
        "case_id": case_id,
        "patient_id": patient_id,
        "npz_path": str(npz_path),
        "num_slices": int(X.shape[0]),
        "height": int(X.shape[2]),
        "width": int(X.shape[3]),
        "positive_slices": positive_slices,
        "target_voxels": target_voxels,
        "current_voxels": current_voxels,
    })


manifest_df = pd.DataFrame(manifest_rows)
summary_df = pd.DataFrame(summary_rows)

manifest_df.to_csv(LOCKED_MANIFEST_PATH, index=False)
summary_df.to_csv(LOCKED_PRE_SUMMARY_PATH, index=False)

print("\nSaved locked manifest:", LOCKED_MANIFEST_PATH)
print("Saved preprocessing summary:", LOCKED_PRE_SUMMARY_PATH)
print("Saved npz folder:", NPZ_DIR)

print("\nSummary:")
print("Cases:", len(summary_df))
print("Total slices:", int(summary_df["num_slices"].sum()))
print("Total positive slices:", int(summary_df["positive_slices"].sum()))
print("Total target voxels:", int(summary_df["target_voxels"].sum()))

display(summary_df.head())
display(summary_df.describe(include="all"))

In [ ]:
import shutil
from pathlib import Path

backup_items = [
    (
        Path("/kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_npz"),
        "/kaggle/working/preprocessed_locked40_2d_npz_backup"
    ),
    (
        Path("/kaggle/working/pcc_independent_baseline/stage2_pcc_guided_student_learning/model_A_direct_target_5fold"),
        "/kaggle/working/model_A_direct_target_5fold_backup"
    ),
]

for src, zip_path in backup_items:
    if src.exists():
        shutil.make_archive(zip_path, "zip", src)
        print("Saved:", zip_path + ".zip")
    else:
        print("Missing:", src)

In [ ]:
# ============================================================
# Model C2-fixed STRICT:
# PCC-on-Model-A Fold-Specific Logit-Residual Correction Student
#
# For each fold:
#   1. Load the corresponding Model A fold checkpoint.
#   2. Generate Model A base prediction B_A for train + test cases using that fold model.
#   3. Generate PCC_on_A teacher by applying PCC directly to B_A.
#   4. Train residual student:
#          input = current T1c + current mask + B_A
#          target = delta_logit = logit(PCC_on_A) - logit(B_A)
#          final prediction = sigmoid(logit(B_A) + delta_hat)
#   5. Evaluate final prediction against true future-change target.
#
# Test-time:
#   Uses only current T1c + current mask + Model A prediction.
#   Does NOT use future target or PCC teacher at inference.
# ============================================================

import os
import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from scipy.ndimage import gaussian_filter, distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import gaussian_filter, distance_transform_edt


# =====================
# Config
# =====================
SEED = 42
EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

POS_SLICE_WEIGHT = 3.0
NEG_SLICE_WEIGHT = 1.0

# Main final-target supervision + PCC residual correction supervision
TRUE_LOSS_WEIGHT = 0.85
PCC_RESIDUAL_LOSS_WEIGHT = 0.15

RESIDUAL_IMPORTANCE_WEIGHT = 2.0
MAX_DELTA_LOGIT = 8.0
LOGIT_EPS = 1e-4

# PCC-on-Model-A teacher parameters
PCC_ROUNDS = 10
PCC_ETA_POS = 0.30
PCC_ETA_NEG = 0.10
PCC_DILATION_RADIUS = 26
PCC_SIGMA = 2.0

FORCE_REGENERATE_BASE_AND_TEACHER = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
MODEL_A_CKPT_DIR = MODEL_A_DIR / "checkpoints"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"

MODEL_C2_DIR = STAGE2_DIR / "model_C2_fixed_strict_pcc_on_modelA_logit_residual"
BASE_MAP_DIR = MODEL_C2_DIR / "fold_specific_modelA_base_maps"
PCC_TEACHER_DIR = MODEL_C2_DIR / "fold_specific_pcc_on_modelA_teacher_maps"
FINAL_PRED_DIR = MODEL_C2_DIR / "final_pred_maps"
DELTA_PRED_DIR = MODEL_C2_DIR / "delta_logit_pred_maps"
CKPT_DIR = MODEL_C2_DIR / "checkpoints"

MODEL_C2_DIR.mkdir(parents=True, exist_ok=True)
BASE_MAP_DIR.mkdir(parents=True, exist_ok=True)
PCC_TEACHER_DIR.mkdir(parents=True, exist_ok=True)
FINAL_PRED_DIR.mkdir(parents=True, exist_ok=True)
DELTA_PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_PRE_SUMMARY_PATH.exists(), f"Missing preprocessing summary: {LOCKED_PRE_SUMMARY_PATH}"
assert SPLIT_PATH.exists(), f"Missing split file: {SPLIT_PATH}"
assert MODEL_A_CKPT_DIR.exists(), f"Missing Model A checkpoint folder: {MODEL_A_CKPT_DIR}"

locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

split_df = pd.read_csv(SPLIT_PATH)

print("Using preprocessing summary:", LOCKED_PRE_SUMMARY_PATH)
print("Using split file:", SPLIT_PATH)
print("Using Model A checkpoints:", MODEL_A_CKPT_DIR)
print("C2 output folder:", MODEL_C2_DIR)


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)

    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)

    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def prob_to_logit_np(p, eps=LOGIT_EPS):
    p = np.clip(p.astype(np.float32), eps, 1.0 - eps)
    return np.log(p / (1.0 - p)).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = normalize_prob_map(pred)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)

    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case_npz(case_id):
    npz_path = case_to_npz[case_id]
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)       # [Z,2,H,W]
        Y = data["Y"].astype(np.float32)       # [Z,1,H,W]
    return X, Y


def make_slice_wise_dilation_region(current_mask, radius):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    R = np.zeros_like(current_mask, dtype=bool)

    for z in range(Z):
        m = current_mask[z]
        if m.sum() > 0:
            dist = distance_transform_edt(~m)
            R[z] = dist <= radius
        else:
            R[z] = False

    return R


def smooth_target_2d(target, sigma):
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))

    if S.max() > 0:
        S = S / (S.max() + 1e-8)

    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_v2_on_base_map(
    B,
    current_mask,
    target,
    rounds=PCC_ROUNDS,
    eta_pos=PCC_ETA_POS,
    eta_neg=PCC_ETA_NEG,
    dilation_radius=PCC_DILATION_RADIUS,
    sigma=PCC_SIGMA,
):
    B = normalize_prob_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)

    R = make_slice_wise_dilation_region(current_mask, dilation_radius)
    S = smooth_target_2d(target, sigma)

    P = B.copy().astype(np.float32)

    for _ in range(rounds):
        P_new = P.copy()

        positive_signal = S * (1.0 - P)
        negative_signal = (1.0 - S) * P
        negative_signal[target] = 0.0

        P_new[R] = (
            P[R]
            + eta_pos * positive_signal[R]
            - eta_neg * negative_signal[R]
        )

        P_new[~R] = B[~R]
        P = np.clip(P_new, 0, 1).astype(np.float32)

    return P


def compute_train_pos_weight_from_true_target(train_case_ids):
    pos = 0.0
    total = 0.0

    for case_id in train_case_ids:
        _, Y = load_case_npz(case_id)
        pos += float(Y.sum())
        total += float(Y.size)

    neg = total - pos
    pos_weight = neg / (pos + 1e-8)
    pos_weight = float(np.clip(pos_weight, 1.0, 50.0))
    return pos_weight, pos, total


# =====================
# Model definitions
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    """
    Same architecture as Model A.
    """
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


class SmallUNet2DResidual(nn.Module):
    """
    Residual correction model:
    input channels = current T1c + current mask + Model A base prediction
    output = delta logit
    """
    def __init__(self, in_ch=3, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


def load_model_A_checkpoint(fold):
    ckpt_path = MODEL_A_CKPT_DIR / f"direct_target_student_fold_{fold}.pt"
    assert ckpt_path.exists(), f"Missing Model A checkpoint: {ckpt_path}"

    model = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model


# =====================
# Fold-specific Model A base predictions
# =====================
def get_base_map_path(fold, case_id):
    fold_dir = BASE_MAP_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    return fold_dir / f"{case_id}_modelA_fold_{fold}_base.npy"


def get_pcc_teacher_path(fold, case_id):
    fold_dir = PCC_TEACHER_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    return fold_dir / f"{case_id}_pcc_on_modelA_fold_{fold}_teacher.npy"


def generate_modelA_base_map_for_case(model_A, fold, case_id):
    path = get_base_map_path(fold, case_id)

    if path.exists() and not FORCE_REGENERATE_BASE_AND_TEACHER:
        B = np.load(path).astype(np.float32)
        return normalize_prob_map(B)

    X, _ = load_case_npz(case_id)

    preds = []
    with torch.no_grad():
        for start in range(0, X.shape[0], BATCH_SIZE):
            xb = torch.from_numpy(X[start:start+BATCH_SIZE]).float().to(DEVICE)
            logits = model_A(xb)
            prob = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
            preds.append(prob.astype(np.float32))

    B = np.concatenate(preds, axis=0)
    B = normalize_prob_map(B)

    np.save(path, B.astype(np.float16))
    return B


def generate_pcc_teacher_for_case(fold, case_id, B_A):
    path = get_pcc_teacher_path(fold, case_id)

    if path.exists() and not FORCE_REGENERATE_BASE_AND_TEACHER:
        P = np.load(path).astype(np.float32)
        return normalize_prob_map(P)

    X, Y = load_case_npz(case_id)
    current_mask = X[:, 1].astype(bool)
    target = Y[:, 0].astype(bool)

    P = run_pcc_v2_on_base_map(
        B=B_A,
        current_mask=current_mask,
        target=target,
        rounds=PCC_ROUNDS,
        eta_pos=PCC_ETA_POS,
        eta_neg=PCC_ETA_NEG,
        dilation_radius=PCC_DILATION_RADIUS,
        sigma=PCC_SIGMA,
    )

    np.save(path, P.astype(np.float16))
    return P


# =====================
# Dataset
# =====================
class C2ResidualDataset(Dataset):
    def __init__(self, fold, case_ids):
        self.fold = fold
        self.case_ids = list(case_ids)

        self.case_data = {}
        self.index = []
        self.slice_has_true_target = []

        for case_id in self.case_ids:
            X, Y_true = load_case_npz(case_id)
            B_A = np.load(get_base_map_path(fold, case_id)).astype(np.float32)
            B_A = normalize_prob_map(B_A)

            P_teacher = np.load(get_pcc_teacher_path(fold, case_id)).astype(np.float32)
            P_teacher = normalize_prob_map(P_teacher)

            assert X.shape[0] == B_A.shape[0] == P_teacher.shape[0], f"Z mismatch: {case_id}"
            assert X.shape[-2:] == B_A.shape[-2:] == P_teacher.shape[-2:], f"HW mismatch: {case_id}"

            B_logit = prob_to_logit_np(B_A)
            T_logit = prob_to_logit_np(P_teacher)

            delta_logit_teacher = np.clip(
                T_logit - B_logit,
                -MAX_DELTA_LOGIT,
                MAX_DELTA_LOGIT
            ).astype(np.float32)

            self.case_data[case_id] = (X, Y_true, B_A, B_logit, delta_logit_teacher)

            Z = X.shape[0]
            for z in range(Z):
                self.index.append((case_id, z))
                self.slice_has_true_target.append(float(Y_true[z].sum() > 0))

        self.slice_has_true_target = np.asarray(self.slice_has_true_target, dtype=np.float32)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        case_id, z = self.index[idx]
        X, Y_true, B_A, B_logit, delta_teacher = self.case_data[case_id]

        x3 = np.concatenate([
            X[z],
            B_A[z][None, :, :]
        ], axis=0).astype(np.float32)

        y_true = Y_true[z].astype(np.float32)
        b_logit = B_logit[z][None, :, :].astype(np.float32)
        delta_teacher = delta_teacher[z][None, :, :].astype(np.float32)

        return (
            torch.from_numpy(x3).float(),
            torch.from_numpy(y_true).float(),
            torch.from_numpy(b_logit).float(),
            torch.from_numpy(delta_teacher).float(),
            case_id,
            z
        )


def make_weighted_sampler(dataset):
    weights = np.where(
        dataset.slice_has_true_target > 0,
        POS_SLICE_WEIGHT,
        NEG_SLICE_WEIGHT
    ).astype(np.float32)

    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True
    )


# =====================
# Loss
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)

    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def residual_loss(delta_pred, delta_teacher):
    # Larger weight where PCC made larger correction
    w = 1.0 + RESIDUAL_IMPORTANCE_WEIGHT * torch.clamp(
        torch.abs(delta_teacher) / MAX_DELTA_LOGIT,
        0.0,
        1.0
    )
    raw = F.smooth_l1_loss(delta_pred, delta_teacher, reduction="none")
    return (w * raw).mean()


def combined_c2_loss(delta_raw, b_logit, y_true, delta_teacher, pos_weight_tensor):
    delta_pred = MAX_DELTA_LOGIT * torch.tanh(delta_raw)
    final_logit = b_logit + delta_pred

    true_bce = F.binary_cross_entropy_with_logits(
        final_logit,
        y_true,
        pos_weight=pos_weight_tensor
    )
    true_dice = soft_dice_loss_from_logits(final_logit, y_true)
    true_loss = 0.5 * true_bce + 0.5 * true_dice

    res_loss = residual_loss(delta_pred, delta_teacher)

    total = TRUE_LOSS_WEIGHT * true_loss + PCC_RESIDUAL_LOSS_WEIGHT * res_loss

    return (
        total,
        true_loss.detach(),
        true_bce.detach(),
        true_dice.detach(),
        res_loss.detach(),
        final_logit.detach(),
        delta_pred.detach()
    )


# =====================
# Fold construction
# =====================
fold_splits = []

for fold in sorted(split_df["fold"].unique()):
    train_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "train")]["case_id"].tolist()
    test_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")]["case_id"].tolist()
    fold_splits.append((int(fold), train_case_ids, test_case_ids))

print("\nFold test cases:")
for fold, _, test_case_ids in fold_splits:
    print(f"\nFold {fold}:")
    for cid in test_case_ids:
        print(" ", cid)


# =====================
# Train / predict one fold
# =====================
def train_one_fold(fold, train_case_ids, test_case_ids):
    print("\n" + "=" * 80)
    print(f"C2-fixed strict / Fold {fold}")
    print("Train cases:", len(train_case_ids))
    print("Test cases:", len(test_case_ids))

    # Load fold-specific Model A
    model_A = load_model_A_checkpoint(fold)

    # Generate fold-specific base maps and PCC teachers for train + test cases
    teacher_quality_rows = []

    for case_id in tqdm(train_case_ids + test_case_ids, desc=f"Fold {fold}: base + PCC teacher"):
        X, Y = load_case_npz(case_id)
        target = Y[:, 0].astype(bool)

        B_A = generate_modelA_base_map_for_case(model_A, fold, case_id)
        P_teacher = generate_pcc_teacher_for_case(fold, case_id, B_A)

        base_metrics = topk_metrics(B_A, target)
        teacher_metrics = topk_metrics(P_teacher, target)

        split_type = "train" if case_id in train_case_ids else "test_diagnostic_only"

        teacher_quality_rows.append({
            "fold": fold,
            "case_id": case_id,
            "split": split_type,
            "model_A_dice": base_metrics["dice"],
            "model_A_iou": base_metrics["iou"],
            "model_A_target_focus": base_metrics["target_focus"],
            "model_A_log10_ratio": base_metrics["log10_ratio"],
            "pcc_on_modelA_teacher_dice": teacher_metrics["dice"],
            "pcc_on_modelA_teacher_iou": teacher_metrics["iou"],
            "pcc_on_modelA_teacher_target_focus": teacher_metrics["target_focus"],
            "pcc_on_modelA_teacher_log10_ratio": teacher_metrics["log10_ratio"],
            "base_path": str(get_base_map_path(fold, case_id)),
            "teacher_path": str(get_pcc_teacher_path(fold, case_id)),
        })

    teacher_quality_df = pd.DataFrame(teacher_quality_rows)
    teacher_quality_df.to_csv(
        MODEL_C2_DIR / f"fold_{fold}_pcc_on_modelA_teacher_quality.csv",
        index=False
    )

    pos_weight, true_pos, true_total = compute_train_pos_weight_from_true_target(train_case_ids)
    print(f"True target positive voxels: {true_pos:.0f} / {true_total:.0f}")
    print(f"BCE pos_weight clipped: {pos_weight:.4f}")

    train_ds = C2ResidualDataset(fold, train_case_ids)
    train_sampler = make_weighted_sampler(train_ds)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    model = SmallUNet2DResidual(in_ch=3, out_ch=1, base=16).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    fold_loss_rows = []

    for epoch in range(1, EPOCHS + 1):
        model.train()

        running_total = 0.0
        running_true = 0.0
        running_bce = 0.0
        running_dice = 0.0
        running_res = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"C2 Fold {fold} Epoch {epoch}/{EPOCHS}", leave=False)

        for x3, y_true, b_logit, delta_teacher, _, _ in pbar:
            x3 = x3.to(DEVICE, non_blocking=True)
            y_true = y_true.to(DEVICE, non_blocking=True)
            b_logit = b_logit.to(DEVICE, non_blocking=True)
            delta_teacher = delta_teacher.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                delta_raw = model(x3)
                loss, true_loss, true_bce, true_dice, res_loss, _, _ = combined_c2_loss(
                    delta_raw=delta_raw,
                    b_logit=b_logit,
                    y_true=y_true,
                    delta_teacher=delta_teacher,
                    pos_weight_tensor=pos_weight_tensor
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_total += float(loss.detach().cpu())
            running_true += float(true_loss.detach().cpu())
            running_bce += float(true_bce.detach().cpu())
            running_dice += float(true_dice.detach().cpu())
            running_res += float(res_loss.detach().cpu())
            n_batches += 1

            pbar.set_postfix({
                "loss": running_total / max(n_batches, 1),
                "true": running_true / max(n_batches, 1),
                "res": running_res / max(n_batches, 1),
            })

        scheduler.step()

        epoch_row = {
            "fold": fold,
            "epoch": epoch,
            "loss": running_total / max(n_batches, 1),
            "true_loss": running_true / max(n_batches, 1),
            "true_bce": running_bce / max(n_batches, 1),
            "true_dice_loss": running_dice / max(n_batches, 1),
            "pcc_residual_loss": running_res / max(n_batches, 1),
            "lr": scheduler.get_last_lr()[0],
            "true_pos_weight": pos_weight,
            "true_loss_weight": TRUE_LOSS_WEIGHT,
            "pcc_residual_loss_weight": PCC_RESIDUAL_LOSS_WEIGHT,
        }
        fold_loss_rows.append(epoch_row)

        print(
            f"C2 Fold {fold} Epoch {epoch:02d}: "
            f"loss={epoch_row['loss']:.5f}, "
            f"true={epoch_row['true_loss']:.5f}, "
            f"res={epoch_row['pcc_residual_loss']:.5f}, "
            f"dice_loss={epoch_row['true_dice_loss']:.5f}"
        )

    ckpt_path = CKPT_DIR / f"c2_fixed_strict_fold_{fold}.pt"
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "train_case_ids": list(train_case_ids),
        "test_case_ids": list(test_case_ids),
        "config": {
            "method": "C2-fixed strict PCC-on-Model-A logit residual student",
            "input": "current T1c + current mask + fold-specific Model A prediction",
            "teacher": "PCC-v2 applied directly to fold-specific Model A prediction",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "true_loss_weight": TRUE_LOSS_WEIGHT,
            "pcc_residual_loss_weight": PCC_RESIDUAL_LOSS_WEIGHT,
            "max_delta_logit": MAX_DELTA_LOGIT,
        }
    }, ckpt_path)

    print("Saved checkpoint:", ckpt_path)

    # Predict test cases
    model.eval()
    metric_rows = []

    with torch.no_grad():
        for case_id in tqdm(test_case_ids, desc=f"C2 Fold {fold} prediction"):
            X, Y = load_case_npz(case_id)
            target = Y[:, 0].astype(bool)

            B_A = np.load(get_base_map_path(fold, case_id)).astype(np.float32)
            B_A = normalize_prob_map(B_A)
            B_logit = prob_to_logit_np(B_A)

            final_parts = []
            delta_parts = []

            for start in range(0, X.shape[0], BATCH_SIZE):
                x_part = X[start:start+BATCH_SIZE]
                b_part = B_A[start:start+BATCH_SIZE]
                b_logit_part = B_logit[start:start+BATCH_SIZE]

                x3 = np.concatenate([
                    x_part,
                    b_part[:, None, :, :]
                ], axis=1).astype(np.float32)

                x3_t = torch.from_numpy(x3).float().to(DEVICE)
                b_logit_t = torch.from_numpy(b_logit_part[:, None, :, :]).float().to(DEVICE)

                delta_raw = model(x3_t)
                delta_pred = MAX_DELTA_LOGIT * torch.tanh(delta_raw)
                final_logit = b_logit_t + delta_pred
                final_prob = torch.sigmoid(final_logit)

                final_parts.append(final_prob.detach().cpu().numpy()[:, 0].astype(np.float32))
                delta_parts.append(delta_pred.detach().cpu().numpy()[:, 0].astype(np.float32))

            final_map = np.concatenate(final_parts, axis=0)
            delta_map = np.concatenate(delta_parts, axis=0)

            final_map = normalize_prob_map(final_map)
            delta_map = np.clip(delta_map, -MAX_DELTA_LOGIT, MAX_DELTA_LOGIT).astype(np.float32)

            final_path = FINAL_PRED_DIR / f"{case_id}_c2_fixed_strict_final_fold_{fold}.npy"
            delta_path = DELTA_PRED_DIR / f"{case_id}_c2_fixed_strict_delta_logit_fold_{fold}.npy"

            np.save(final_path, final_map.astype(np.float16))
            np.save(delta_path, delta_map.astype(np.float16))

            c2_metrics = topk_metrics(final_map, target)
            base_metrics = topk_metrics(B_A, target)

            metric_rows.append({
                "case_id": case_id,
                "fold": fold,
                "method": "C2-fixed strict PCC-on-Model-A residual student",
                "test_time_future_target_access": "No",
                "input_uses_fold_specific_model_A_prediction": "Yes",
                "training_signal": "true target + PCC-on-Model-A logit residual",
                "final_pred_path": str(final_path),
                "delta_logit_pred_path": str(delta_path),
                "model_A_direct_dice": base_metrics["dice"],
                "model_A_direct_iou": base_metrics["iou"],
                "model_A_direct_target_focus": base_metrics["target_focus"],
                "model_A_direct_log10_ratio": base_metrics["log10_ratio"],
                **c2_metrics,
            })

    return fold_loss_rows, metric_rows, teacher_quality_rows


# =====================
# Run all folds
# =====================
all_loss_rows = []
all_metric_rows = []
all_teacher_quality_rows = []

start_time = time.time()

for fold, train_case_ids, test_case_ids in fold_splits:
    fold_loss_rows, metric_rows, teacher_quality_rows = train_one_fold(
        fold,
        train_case_ids,
        test_case_ids
    )

    all_loss_rows.extend(fold_loss_rows)
    all_metric_rows.extend(metric_rows)
    all_teacher_quality_rows.extend(teacher_quality_rows)

    pd.DataFrame(all_loss_rows).to_csv(MODEL_C2_DIR / "c2_fixed_strict_training_losses_partial.csv", index=False)
    pd.DataFrame(all_metric_rows).to_csv(MODEL_C2_DIR / "c2_fixed_strict_case_metrics_partial.csv", index=False)
    pd.DataFrame(all_teacher_quality_rows).to_csv(MODEL_C2_DIR / "c2_fixed_strict_teacher_quality_partial.csv", index=False)


loss_df = pd.DataFrame(all_loss_rows)
metrics_df = pd.DataFrame(all_metric_rows)
teacher_quality_df = pd.DataFrame(all_teacher_quality_rows)

loss_path = MODEL_C2_DIR / "c2_fixed_strict_training_losses.csv"
metrics_path = MODEL_C2_DIR / "c2_fixed_strict_case_metrics.csv"
teacher_quality_path = MODEL_C2_DIR / "c2_fixed_strict_teacher_quality.csv"

loss_df.to_csv(loss_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
teacher_quality_df.to_csv(teacher_quality_path, index=False)

summary = metrics_df.groupby("method")[["dice", "iou", "target_focus", "log10_ratio"]].agg(["mean", "median", "min", "max"])
summary_path = MODEL_C2_DIR / "c2_fixed_strict_summary.csv"
summary.to_csv(summary_path)

# Pairwise C2 vs fold-specific Model A base on the same test cases
pairwise_rows = []

for metric, base_col in [
    ("dice", "model_A_direct_dice"),
    ("iou", "model_A_direct_iou"),
    ("target_focus", "model_A_direct_target_focus"),
    ("log10_ratio", "model_A_direct_log10_ratio"),
]:
    diff = metrics_df[metric] - metrics_df[base_col]

    pairwise_rows.append({
        "comparison": "C2-fixed strict final prediction vs fold-specific Model A base prediction",
        "metric": metric,
        "mean_diff": float(diff.mean()),
        "median_diff": float(diff.median()),
        "c2_wins": int((diff > 0).sum()),
        "model_A_wins": int((diff < 0).sum()),
        "ties": int((diff == 0).sum()),
        "total": int(diff.notna().sum()),
        "c2_win_rate": float((diff > 0).mean()),
    })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = MODEL_C2_DIR / "pairwise_C2_fixed_strict_vs_Model_A.csv"
pairwise_df.to_csv(pairwise_path, index=False)

# Teacher diagnostic summary
teacher_test = teacher_quality_df[teacher_quality_df["split"] == "test_diagnostic_only"].copy()

teacher_summary = pd.DataFrame([{
    "diagnostic_set": "test cases only",
    "model_A_dice_mean": teacher_test["model_A_dice"].mean(),
    "model_A_iou_mean": teacher_test["model_A_iou"].mean(),
    "pcc_on_modelA_teacher_dice_mean": teacher_test["pcc_on_modelA_teacher_dice"].mean(),
    "pcc_on_modelA_teacher_iou_mean": teacher_test["pcc_on_modelA_teacher_iou"].mean(),
    "pcc_on_modelA_teacher_target_focus_mean": teacher_test["pcc_on_modelA_teacher_target_focus"].mean(),
    "pcc_on_modelA_teacher_log10_ratio_mean": teacher_test["pcc_on_modelA_teacher_log10_ratio"].mean(),
}])

teacher_summary_path = MODEL_C2_DIR / "c2_fixed_strict_teacher_diagnostic_summary.csv"
teacher_summary.to_csv(teacher_summary_path, index=False)

runtime_min = (time.time() - start_time) / 60.0

run_info = {
    "method": "C2-fixed strict PCC-on-Model-A logit residual student",
    "runtime_minutes": runtime_min,
    "test_time_future_target_access": "No",
    "input": "current T1c + current mask + fold-specific Model A prediction",
    "teacher": "PCC-v2 applied directly to fold-specific Model A prediction",
    "outputs": {
        "loss_path": str(loss_path),
        "metrics_path": str(metrics_path),
        "summary_path": str(summary_path),
        "pairwise_path": str(pairwise_path),
        "teacher_quality_path": str(teacher_quality_path),
        "teacher_summary_path": str(teacher_summary_path),
    }
}

with open(MODEL_C2_DIR / "c2_fixed_strict_run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)

print("\n" + "=" * 80)
print("C2-fixed strict finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved losses:", loss_path)
print("Saved metrics:", metrics_path)
print("Saved summary:", summary_path)
print("Saved pairwise:", pairwise_path)

print("\nPCC-on-Model-A teacher diagnostic summary:")
display(teacher_summary)

print("\nC2-fixed strict summary:")
display(summary)

print("\nPairwise C2-fixed strict vs Model A:")
display(pairwise_df)

print("\nTraining losses tail:")
display(loss_df.tail(10))

print("\nCase metrics:")
display(metrics_df)


# =====================
# Compact comparison with Model A/B/C/C2 if available
# =====================
comparison_rows = []

def add_model(path, model_name, mechanism):
    if not Path(path).exists():
        print("Missing comparison file:", path)
        return

    df = pd.read_csv(path)
    m = df[["dice", "iou", "target_focus", "log10_ratio"]].mean()

    comparison_rows.append({
        "Model": model_name,
        "Mechanism": mechanism,
        "Dice_mean": float(m["dice"]),
        "IoU_mean": float(m["iou"]),
        "Target_focus_mean": float(m["target_focus"]),
        "Log10_ratio_mean": float(m["log10_ratio"]),
    })

add_model(
    STAGE2_DIR / "model_A_direct_target_5fold" / "direct_target_case_metrics.csv",
    "Model A",
    "direct future-change prediction"
)

add_model(
    STAGE2_DIR / "model_B_eia_teacher_lambda030_5fold" / "eia_teacher_case_metrics.csv",
    "Model B",
    "EIA teacher final-map replacement"
)

add_model(
    STAGE2_DIR / "model_C_pcc_v2_teacher_eta_neg010_5fold" / "pcc_teacher_case_metrics.csv",
    "Model C",
    "PCC teacher final-map replacement"
)

add_model(
    metrics_path,
    "Model C2-fixed strict",
    "PCC-on-Model-A logit residual correction"
)

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = MODEL_C2_DIR / "model_A_B_C_C2_fixed_strict_compact_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nCompact comparison:")
display(comparison_df)
print("Saved compact comparison:", comparison_path)

In [ ]:
import shutil
from pathlib import Path

src = Path("/kaggle/working/pcc_independent_baseline/stage2_pcc_guided_student_learning/model_C2_fixed_strict_pcc_on_modelA_logit_residual")
zip_path = "/kaggle/working/model_C2_fixed_strict_pcc_on_modelA_logit_residual_backup"

shutil.make_archive(zip_path, "zip", src)

print("Saved:", zip_path + ".zip")

In [ ]:
# ============================================================
# Full backup before shutdown
# This saves:
# 1. whole pcc_independent_baseline project
# 2. preprocessing only
# 3. stage2 experiments only
# 4. Model A only
# 5. C2-fixed strict only
# ============================================================

import shutil
from pathlib import Path
import os
import json
import pandas as pd
from datetime import datetime

ROOT = Path("/kaggle/working/pcc_independent_baseline")
WORKING = Path("/kaggle/working")

assert ROOT.exists(), f"Missing root folder: {ROOT}"

snapshot = {
    "created_at": datetime.now().isoformat(),
    "root": str(ROOT),
    "important_paths": {
        "preprocessing_summary": str(ROOT / "preprocessed_locked40_2d_summary.csv"),
        "npz_dir": str(ROOT / "preprocessed_locked40_2d_npz"),
        "model_A_dir": str(ROOT / "stage2_pcc_guided_student_learning/model_A_direct_target_5fold"),
        "c2_dir": str(ROOT / "stage2_pcc_guided_student_learning/model_C2_fixed_strict_pcc_on_modelA_logit_residual"),
    }
}

with open(ROOT / "BACKUP_SNAPSHOT_INFO.json", "w") as f:
    json.dump(snapshot, f, indent=2)

backup_items = [
    (
        ROOT,
        WORKING / "FULL_pcc_independent_baseline_backup"
    ),
    (
        ROOT / "preprocessed_locked40_2d_npz",
        WORKING / "preprocessed_locked40_2d_npz_backup"
    ),
    (
        ROOT / "stage2_pcc_guided_student_learning",
        WORKING / "stage2_pcc_guided_student_learning_backup"
    ),
    (
        ROOT / "stage2_pcc_guided_student_learning/model_A_direct_target_5fold",
        WORKING / "model_A_direct_target_5fold_backup"
    ),
    (
        ROOT / "stage2_pcc_guided_student_learning/model_C2_fixed_strict_pcc_on_modelA_logit_residual",
        WORKING / "model_C2_fixed_strict_backup"
    ),
]

created = []

for src, zip_base in backup_items:
    if src.exists():
        print(f"Zipping: {src}")
        zip_file = shutil.make_archive(str(zip_base), "zip", src)
        size_mb = os.path.getsize(zip_file) / (1024 * 1024)
        created.append({"zip_file": zip_file, "size_mb": round(size_mb, 2)})
        print(f"Saved: {zip_file} | {size_mb:.2f} MB\n")
    else:
        print(f"Missing, skipped: {src}\n")

backup_df = pd.DataFrame(created)
backup_df.to_csv(WORKING / "backup_file_list.csv", index=False)

print("=" * 80)
print("Backup files created:")
display(backup_df)
print("Saved backup list:", WORKING / "backup_file_list.csv")

In [ ]:
from pathlib import Path
import shutil
import os

src = Path("/kaggle/working/pcc_independent_baseline")
zip_base = "/kaggle/working/PCC_FULL_BACKUP_BEFORE_SHUTDOWN"

print("Source exists:", src.exists())
print("Start zipping...")

zip_file = shutil.make_archive(zip_base, "zip", src)

print("DONE")
print("Saved:", zip_file)
print("Size MB:", round(os.path.getsize(zip_file) / 1024 / 1024, 2))

In [ ]:
from pathlib import Path
import os

print("Files in /kaggle/working:")
for p in Path("/kaggle/working").iterdir():
    if p.is_file():
        print("FILE:", p.name, round(os.path.getsize(p) / 1024 / 1024, 2), "MB")
    else:
        print("DIR :", p.name)

print("\nSearching zip files:")
for p in Path("/kaggle/working").rglob("*.zip"):
    print(p, round(os.path.getsize(p) / 1024 / 1024, 2), "MB")

print("\nSearching pcc folder:")
for p in Path("/kaggle/working").rglob("*pcc*"):
    print(p)

In [ ]:
from pathlib import Path
import os

print("Files in /kaggle/working:")
for p in Path("/kaggle/working").iterdir():
    if p.is_file():
        print("FILE:", p.name, round(os.path.getsize(p) / 1024 / 1024, 2), "MB")
    else:
        print("DIR :", p.name)

print("\nInput datasets:")
for p in Path("/kaggle/input").iterdir():
    print(p)

In [ ]:
# ============================================================
# Step 2: Recover locked40 preprocessing npz files
# Output:
#   /kaggle/working/pcc_independent_baseline/manifest_locked_original_40.csv
#   /kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_summary.csv
#   /kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_npz/*.npz
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib


# =====================
# Output paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NPZ_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"
NPZ_DIR.mkdir(parents=True, exist_ok=True)

LOCKED_MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"
LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"


# =====================
# Locked 40 case list
# =====================
LOCKED_CASES = [
    "PatientID_0003_T1_to_T2_t1c",
    "PatientID_0005_T3_to_T4_t1c",
    "PatientID_0006_T2_to_T4_t1c",
    "PatientID_0007_T2_to_T3_t1c",
    "PatientID_0008_T4_to_T6_t1c",
    "PatientID_0010_T1_to_T4_t1c",
    "PatientID_0011_T1_to_T2_t1c",
    "PatientID_0012_T2_to_T3_t1c",
    "PatientID_0013_T1_to_T2_t1c",
    "PatientID_0014_T1_to_T2_t1c",
    "PatientID_0018_T1_to_T2_t1c",
    "PatientID_0019_T4_to_T5_t1c",
    "PatientID_0020_T1_to_T2_t1c",
    "PatientID_0021_T2_to_T3_t1c",
    "PatientID_0022_T1_to_T2_t1c",
    "PatientID_0024_T2_to_T3_t1c",
    "PatientID_0025_T1_to_T2_t1c",
    "PatientID_0026_T1_to_T2_t1c",
    "PatientID_0029_T1_to_T3_t1c",
    "PatientID_0030_T1_to_T3_t1c",
    "PatientID_0031_T2_to_T3_t1c",
    "PatientID_0032_T1_to_T2_t1c",
    "PatientID_0033_T1_to_T2_t1c",
    "PatientID_0034_T1_to_T2_t1c",
    "PatientID_0035_T1_to_T2_t1c",
    "PatientID_0036_T1_to_T2_t1c",
    "PatientID_0037_T1_to_T2_t1c",
    "PatientID_0038_T1_to_T2_t1c",
    "PatientID_0039_T1_to_T2_t1c",
    "PatientID_0041_T1_to_T2_t1c",
    "PatientID_0044_T1_to_T2_t1c",
    "PatientID_0045_T1_to_T2_t1c",
    "PatientID_0051_T1_to_T3_t1c",
    "PatientID_0052_T1_to_T2_t1c",
    "PatientID_0053_T1_to_T3_t1c",
    "PatientID_0054_T1_to_T2_t1c",
    "PatientID_0055_T1_to_T4_t1c",
    "PatientID_0059_T1_to_T2_t1c",
    "PatientID_0060_T1_to_T2_t1c",
    "PatientID_0062_T1_to_T2_t1c",
]


def parse_case_id(case_id):
    m = re.match(r"(PatientID_\d+)_T(\d+)_to_T(\d+)_t1c$", case_id)
    if not m:
        raise ValueError(f"Cannot parse case_id: {case_id}")
    patient_id = m.group(1)
    current_tp = int(m.group(2))
    future_tp = int(m.group(3))
    return patient_id, current_tp, future_tp


# =====================
# Locate MU-Glioma-Post root
# =====================
def locate_mu_root():
    input_root = Path("/kaggle/input")
    candidates = []

    for p in input_root.rglob("*"):
        if p.is_dir():
            patient_dirs = list(p.glob("PatientID_*"))
            if len(patient_dirs) > 10:
                candidates.append(p)

    if not candidates:
        print("Could not automatically find MU-Glioma-Post root.")
        print("Top-level /kaggle/input:")
        for p in input_root.iterdir():
            print(" ", p)
        raise FileNotFoundError("No folder containing PatientID_* found under /kaggle/input")

    candidates = sorted(candidates, key=lambda x: len(str(x)), reverse=True)
    return candidates[0]


DATA_ROOT = locate_mu_root()

print("Detected MU-Glioma-Post root:", DATA_ROOT)
print("Patient folders:", len(list(DATA_ROOT.glob("PatientID_*"))))


# =====================
# Find files
# =====================
def find_timepoint_file(patient_id, tp_num, kind):
    """
    kind:
      "t1c" or "mask"
    """
    patient_dir = DATA_ROOT / patient_id
    assert patient_dir.exists(), f"Missing patient folder: {patient_dir}"

    tp_patterns = [
        f"Timepoint_{tp_num}",
        f"Timepoint-{tp_num}",
        f"Timepoint {tp_num}",
        f"T{tp_num}",
        f"TP{tp_num}",
    ]

    candidate_files = []

    for f in patient_dir.rglob("*.nii*"):
        path_str = str(f)
        name = f.name.lower()

        if not any(tp_pat in path_str for tp_pat in tp_patterns):
            continue

        if kind == "t1c":
            if "t1c" in name and "brain" in name:
                candidate_files.append(f)

        elif kind == "mask":
            if "tumormask" in name or "tumourmask" in name:
                candidate_files.append(f)

        else:
            raise ValueError(kind)

    candidate_files = sorted(set(candidate_files))

    if len(candidate_files) == 0:
        raise FileNotFoundError(f"Missing {kind} for {patient_id} T{tp_num}")

    return candidate_files[0]


# =====================
# Loading and preprocessing
# =====================
def load_nii(path):
    arr = nib.load(str(path)).get_fdata()
    arr = np.asarray(arr, dtype=np.float32)
    return arr


def volume_to_z_hw(vol):
    """
    Convert 3D volume to [Z,H,W].
    MU-Glioma-Post is usually [H,W,Z].
    """
    if vol.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {vol.shape}")

    slice_axis = int(np.argmin(vol.shape))

    if slice_axis == 0:
        zhw = vol
    elif slice_axis == 1:
        zhw = np.transpose(vol, (1, 0, 2))
    else:
        zhw = np.transpose(vol, (2, 0, 1))

    return zhw.astype(np.float32)


def normalize_mri(vol):
    vol = vol.astype(np.float32)
    finite = np.isfinite(vol)
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)

    foreground = vol[(vol > 0) & finite]

    if foreground.size < 10:
        return np.zeros_like(vol, dtype=np.float32)

    lo, hi = np.percentile(foreground, [1, 99])

    if hi <= lo:
        return np.zeros_like(vol, dtype=np.float32)

    vol = np.clip(vol, lo, hi)
    vol = (vol - lo) / (hi - lo + 1e-8)
    vol = np.clip(vol, 0, 1)

    return vol.astype(np.float32)


# =====================
# Main preprocessing loop
# =====================
manifest_rows = []
summary_rows = []

for case_id in tqdm(LOCKED_CASES, desc="Preprocessing locked40"):
    patient_id, current_tp, future_tp = parse_case_id(case_id)

    cur_t1c_path = find_timepoint_file(patient_id, current_tp, "t1c")
    cur_mask_path = find_timepoint_file(patient_id, current_tp, "mask")
    fut_mask_path = find_timepoint_file(patient_id, future_tp, "mask")

    cur_t1c = volume_to_z_hw(load_nii(cur_t1c_path))
    cur_mask = volume_to_z_hw(load_nii(cur_mask_path)) > 0
    fut_mask = volume_to_z_hw(load_nii(fut_mask_path)) > 0

    assert cur_t1c.shape == cur_mask.shape == fut_mask.shape, (
        f"Shape mismatch {case_id}: "
        f"t1c={cur_t1c.shape}, cur_mask={cur_mask.shape}, fut_mask={fut_mask.shape}"
    )

    cur_t1c_norm = normalize_mri(cur_t1c)

    # Future-change target:
    # future tumour area not already included in current tumour mask
    target = np.logical_and(fut_mask, ~cur_mask)

    X = np.stack(
        [
            cur_t1c_norm,
            cur_mask.astype(np.float32),
        ],
        axis=1
    ).astype(np.float32)        # [Z,2,H,W]

    Y = target[:, None, :, :].astype(np.uint8)   # [Z,1,H,W]

    z_indices = np.arange(X.shape[0], dtype=np.int16)

    npz_path = NPZ_DIR / f"{case_id}.npz"

    np.savez_compressed(
        npz_path,
        X=X,
        Y=Y,
        z_indices=z_indices,
        case_id=np.array(case_id),
        patient_id=np.array(patient_id),
        current_tp=np.array(current_tp),
        future_tp=np.array(future_tp),
    )

    target_voxels = int(target.sum())
    current_voxels = int(cur_mask.sum())
    positive_slices = int((target.reshape(target.shape[0], -1).sum(axis=1) > 0).sum())

    manifest_rows.append({
        "case_id": case_id,
        "patient_id": patient_id,
        "current_tp": current_tp,
        "future_tp": future_tp,
        "current_t1c_path": str(cur_t1c_path),
        "current_mask_path": str(cur_mask_path),
        "future_mask_path": str(fut_mask_path),
        "growth_voxels": target_voxels,
    })

    summary_rows.append({
        "case_id": case_id,
        "patient_id": patient_id,
        "npz_path": str(npz_path),
        "num_slices": int(X.shape[0]),
        "height": int(X.shape[2]),
        "width": int(X.shape[3]),
        "positive_slices": positive_slices,
        "target_voxels": target_voxels,
        "current_voxels": current_voxels,
    })


manifest_df = pd.DataFrame(manifest_rows)
summary_df = pd.DataFrame(summary_rows)

manifest_df.to_csv(LOCKED_MANIFEST_PATH, index=False)
summary_df.to_csv(LOCKED_PRE_SUMMARY_PATH, index=False)

print("\nSaved locked manifest:", LOCKED_MANIFEST_PATH)
print("Saved preprocessing summary:", LOCKED_PRE_SUMMARY_PATH)
print("Saved npz folder:", NPZ_DIR)

print("\nSummary:")
print("Cases:", len(summary_df))
print("Total slices:", int(summary_df["num_slices"].sum()))
print("Total positive slices:", int(summary_df["positive_slices"].sum()))
print("Total target voxels:", int(summary_df["target_voxels"].sum()))

display(summary_df.head())
display(summary_df.describe(include="all"))

In [ ]:
# ============================================================
# C2-v2 Conservative OOF-Gated Residual + Alpha Sweep
#
# Goal:
#   Learn a conservative PCC-derived correction over Model A's
#   out-of-fold prediction maps.
#
# Key changes vs previous C2:
#   1. Uses Model A OOF prediction maps for all cases.
#   2. Conservative residual strength.
#   3. Current-mask dilation gate.
#   4. Residual magnitude penalty.
#   5. Alpha sweep: 0, 0.10, 0.25, 0.50, 0.75, 1.00.
#
# Test-time:
#   Uses current MRI + current mask + Model A OOF prediction.
#   Does NOT use future target or PCC teacher at inference.
# ============================================================

import os
import re
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from scipy.ndimage import gaussian_filter, distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import gaussian_filter, distance_transform_edt


# =====================
# Config
# =====================
SEED = 42

# First run this quick version.
# If results are promising, later change to 20.
EPOCHS = 5

BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

POS_SLICE_WEIGHT = 3.0
NEG_SLICE_WEIGHT = 1.0

# Conservative correction
MAX_DELTA_LOGIT = 3.0
TRUE_LOSS_WEIGHT = 0.95
PCC_RESIDUAL_LOSS_WEIGHT = 0.05
MAGNITUDE_PENALTY_WEIGHT = 0.005
RESIDUAL_IMPORTANCE_WEIGHT = 2.0
LOGIT_EPS = 1e-4

# PCC teacher parameters
PCC_ROUNDS = 10
PCC_ETA_POS = 0.30
PCC_ETA_NEG = 0.10
PCC_DILATION_RADIUS = 26
PCC_SIGMA = 2.0

# Inference alpha sweep
ALPHAS = [0.0, 0.10, 0.25, 0.50, 0.75, 1.00]

FORCE_REGENERATE_TEACHERS = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
MODEL_A_METRICS_PATH = MODEL_A_DIR / "direct_target_case_metrics.csv"
MODEL_A_PRED_DIR = MODEL_A_DIR / "direct_target_pred_maps"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"

C2_DIR = STAGE2_DIR / "model_C2_v2_conservative_oof_gated_alpha_sweep_5epoch"
TEACHER_DIR = C2_DIR / "pcc_on_modelA_oof_teacher_maps"
DELTA_PRED_DIR = C2_DIR / "delta_logit_pred_maps"
CKPT_DIR = C2_DIR / "checkpoints"

C2_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_DIR.mkdir(parents=True, exist_ok=True)
DELTA_PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_PRE_SUMMARY_PATH.exists(), f"Missing preprocessing summary: {LOCKED_PRE_SUMMARY_PATH}"
assert MODEL_A_METRICS_PATH.exists(), f"Missing Model A case metrics: {MODEL_A_METRICS_PATH}"
assert MODEL_A_PRED_DIR.exists(), f"Missing Model A prediction folder: {MODEL_A_PRED_DIR}"
assert SPLIT_PATH.exists(), f"Missing split file: {SPLIT_PATH}"

locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))

modelA_metrics_df = pd.read_csv(MODEL_A_METRICS_PATH)
split_df = pd.read_csv(SPLIT_PATH)

print("Using preprocessing summary:", LOCKED_PRE_SUMMARY_PATH)
print("Using Model A metrics:", MODEL_A_METRICS_PATH)
print("Using Model A OOF prediction folder:", MODEL_A_PRED_DIR)
print("Using split file:", SPLIT_PATH)
print("C2-v2 output folder:", C2_DIR)


# =====================
# Utility functions
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)

    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)

    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def prob_to_logit_np(p, eps=LOGIT_EPS):
    p = np.clip(p.astype(np.float32), eps, 1.0 - eps)
    return np.log(p / (1.0 - p)).astype(np.float32)


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def topk_metrics(pred, target, eps=1e-8):
    pred = normalize_prob_map(pred)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)

    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case_npz(case_id):
    npz_path = case_to_npz[case_id]
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)       # [Z,2,H,W]
        Y = data["Y"].astype(np.float32)       # [Z,1,H,W]
    return X, Y


def load_modelA_oof_map(case_id):
    """
    Load Model A out-of-fold prediction map for a case.
    First use direct_target_case_metrics.csv pred_path.
    Fall back to glob search if needed.
    """
    row = modelA_metrics_df[modelA_metrics_df["case_id"] == case_id]

    if len(row) == 1:
        p = Path(row.iloc[0]["pred_path"])
        if p.exists():
            return normalize_prob_map(np.load(p).astype(np.float32))

    candidates = sorted(MODEL_A_PRED_DIR.glob(f"{case_id}_direct_target_student_fold_*.npy"))
    if len(candidates) != 1:
        print("Candidates found:", candidates)
        raise FileNotFoundError(f"Could not uniquely find Model A OOF map for {case_id}")

    return normalize_prob_map(np.load(candidates[0]).astype(np.float32))


def make_slice_wise_dilation_region(current_mask, radius):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    R = np.zeros_like(current_mask, dtype=bool)

    for z in range(Z):
        m = current_mask[z]
        if m.sum() > 0:
            dist = distance_transform_edt(~m)
            R[z] = dist <= radius
        else:
            R[z] = False

    return R


def smooth_target_2d(target, sigma):
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))

    if S.max() > 0:
        S = S / (S.max() + 1e-8)

    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_v2_on_base_map(
    B,
    current_mask,
    target,
    rounds=PCC_ROUNDS,
    eta_pos=PCC_ETA_POS,
    eta_neg=PCC_ETA_NEG,
    dilation_radius=PCC_DILATION_RADIUS,
    sigma=PCC_SIGMA,
):
    B = normalize_prob_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)

    R = make_slice_wise_dilation_region(current_mask, dilation_radius)
    S = smooth_target_2d(target, sigma)

    P = B.copy().astype(np.float32)

    for _ in range(rounds):
        P_new = P.copy()

        positive_signal = S * (1.0 - P)
        negative_signal = (1.0 - S) * P
        negative_signal[target] = 0.0

        P_new[R] = (
            P[R]
            + eta_pos * positive_signal[R]
            - eta_neg * negative_signal[R]
        )

        P_new[~R] = B[~R]
        P = np.clip(P_new, 0, 1).astype(np.float32)

    return P.astype(np.float32)


def compute_train_pos_weight_from_true_target(train_case_ids):
    pos = 0.0
    total = 0.0

    for case_id in train_case_ids:
        _, Y = load_case_npz(case_id)
        pos += float(Y.sum())
        total += float(Y.size)

    neg = total - pos
    pos_weight = neg / (pos + 1e-8)
    pos_weight = float(np.clip(pos_weight, 1.0, 50.0))
    return pos_weight, pos, total


# =====================
# Model definitions
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2DResidual(nn.Module):
    """
    Residual correction model:
    input channels = current T1c + current mask + Model A OOF prediction
    output = delta logit
    """
    def __init__(self, in_ch=3, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


# =====================
# Generate OOF PCC teachers
# =====================
def get_teacher_path(case_id):
    return TEACHER_DIR / f"{case_id}_pcc_on_modelA_oof_teacher.npy"


def generate_teacher_for_case(case_id):
    teacher_path = get_teacher_path(case_id)

    if teacher_path.exists() and not FORCE_REGENERATE_TEACHERS:
        return normalize_prob_map(np.load(teacher_path).astype(np.float32))

    X, Y = load_case_npz(case_id)
    B = load_modelA_oof_map(case_id)

    current_mask = X[:, 1].astype(bool)
    target = Y[:, 0].astype(bool)

    P_teacher = run_pcc_v2_on_base_map(
        B=B,
        current_mask=current_mask,
        target=target,
        rounds=PCC_ROUNDS,
        eta_pos=PCC_ETA_POS,
        eta_neg=PCC_ETA_NEG,
        dilation_radius=PCC_DILATION_RADIUS,
        sigma=PCC_SIGMA,
    )

    np.save(teacher_path, P_teacher.astype(np.float16))
    return P_teacher


all_case_ids = sorted(case_to_npz.keys())

teacher_quality_rows = []

print("\nGenerating / loading PCC-on-Model-A OOF teachers...")
for case_id in tqdm(all_case_ids):
    X, Y = load_case_npz(case_id)
    target = Y[:, 0].astype(bool)

    B = load_modelA_oof_map(case_id)
    P_teacher = generate_teacher_for_case(case_id)

    base_m = topk_metrics(B, target)
    teacher_m = topk_metrics(P_teacher, target)

    teacher_quality_rows.append({
        "case_id": case_id,
        "model_A_dice": base_m["dice"],
        "model_A_iou": base_m["iou"],
        "model_A_target_focus": base_m["target_focus"],
        "model_A_log10_ratio": base_m["log10_ratio"],
        "pcc_teacher_dice": teacher_m["dice"],
        "pcc_teacher_iou": teacher_m["iou"],
        "pcc_teacher_target_focus": teacher_m["target_focus"],
        "pcc_teacher_log10_ratio": teacher_m["log10_ratio"],
        "teacher_path": str(get_teacher_path(case_id)),
    })

teacher_quality_df = pd.DataFrame(teacher_quality_rows)
teacher_quality_path = C2_DIR / "c2_v2_teacher_quality.csv"
teacher_quality_df.to_csv(teacher_quality_path, index=False)

teacher_summary = pd.DataFrame([{
    "model_A_dice_mean": teacher_quality_df["model_A_dice"].mean(),
    "model_A_iou_mean": teacher_quality_df["model_A_iou"].mean(),
    "model_A_target_focus_mean": teacher_quality_df["model_A_target_focus"].mean(),
    "model_A_log10_ratio_mean": teacher_quality_df["model_A_log10_ratio"].mean(),
    "pcc_teacher_dice_mean": teacher_quality_df["pcc_teacher_dice"].mean(),
    "pcc_teacher_iou_mean": teacher_quality_df["pcc_teacher_iou"].mean(),
    "pcc_teacher_target_focus_mean": teacher_quality_df["pcc_teacher_target_focus"].mean(),
    "pcc_teacher_log10_ratio_mean": teacher_quality_df["pcc_teacher_log10_ratio"].mean(),
}])

teacher_summary_path = C2_DIR / "c2_v2_teacher_quality_summary.csv"
teacher_summary.to_csv(teacher_summary_path, index=False)

print("\nTeacher quality summary:")
display(teacher_summary)


# =====================
# Dataset
# =====================
class C2V2Dataset(Dataset):
    def __init__(self, case_ids):
        self.case_ids = list(case_ids)
        self.case_data = {}
        self.index = []
        self.slice_has_target = []

        for case_id in self.case_ids:
            X, Y_true = load_case_npz(case_id)
            B = load_modelA_oof_map(case_id)
            P_teacher = normalize_prob_map(np.load(get_teacher_path(case_id)).astype(np.float32))

            current_mask = X[:, 1].astype(bool)
            gate = make_slice_wise_dilation_region(current_mask, PCC_DILATION_RADIUS).astype(np.float32)

            B_logit = prob_to_logit_np(B)
            T_logit = prob_to_logit_np(P_teacher)

            delta_teacher = np.clip(
                T_logit - B_logit,
                -MAX_DELTA_LOGIT,
                MAX_DELTA_LOGIT
            ).astype(np.float32)

            # Enforce teacher residual only inside gate.
            delta_teacher = delta_teacher * gate

            self.case_data[case_id] = {
                "X": X,
                "Y": Y_true,
                "B": B,
                "B_logit": B_logit,
                "gate": gate,
                "delta_teacher": delta_teacher,
            }

            Z = X.shape[0]
            for z in range(Z):
                self.index.append((case_id, z))
                self.slice_has_target.append(float(Y_true[z].sum() > 0))

        self.slice_has_target = np.asarray(self.slice_has_target, dtype=np.float32)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        case_id, z = self.index[idx]
        d = self.case_data[case_id]

        X = d["X"]
        Y = d["Y"]
        B = d["B"]
        B_logit = d["B_logit"]
        gate = d["gate"]
        delta_teacher = d["delta_teacher"]

        x3 = np.concatenate([
            X[z],
            B[z][None, :, :]
        ], axis=0).astype(np.float32)

        return (
            torch.from_numpy(x3).float(),
            torch.from_numpy(Y[z]).float(),
            torch.from_numpy(B_logit[z][None, :, :]).float(),
            torch.from_numpy(gate[z][None, :, :]).float(),
            torch.from_numpy(delta_teacher[z][None, :, :]).float(),
            case_id,
            z
        )


def make_weighted_sampler(dataset):
    weights = np.where(
        dataset.slice_has_target > 0,
        POS_SLICE_WEIGHT,
        NEG_SLICE_WEIGHT
    ).astype(np.float32)

    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True
    )


# =====================
# Loss
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)

    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def gated_residual_loss(delta_gated, delta_teacher, gate, eps=1e-6):
    # Weight residual learning where PCC teacher made larger correction.
    w = 1.0 + RESIDUAL_IMPORTANCE_WEIGHT * torch.clamp(
        torch.abs(delta_teacher) / MAX_DELTA_LOGIT,
        0.0,
        1.0
    )

    raw = F.smooth_l1_loss(delta_gated, delta_teacher, reduction="none")
    mask = (gate > 0.5).float()

    numerator = torch.sum(raw * w * mask)
    denominator = torch.sum(mask) + eps

    return numerator / denominator


def magnitude_penalty(delta_gated, gate, eps=1e-6):
    mask = (gate > 0.5).float()
    numerator = torch.sum(torch.abs(delta_gated) * mask)
    denominator = torch.sum(mask) + eps
    return numerator / denominator


def combined_loss(delta_raw, b_logit, gate, y_true, delta_teacher, pos_weight_tensor):
    delta_pred = MAX_DELTA_LOGIT * torch.tanh(delta_raw)
    delta_gated = delta_pred * gate

    final_logit = b_logit + delta_gated

    true_bce = F.binary_cross_entropy_with_logits(
        final_logit,
        y_true,
        pos_weight=pos_weight_tensor
    )
    true_dice = soft_dice_loss_from_logits(final_logit, y_true)
    true_loss = 0.5 * true_bce + 0.5 * true_dice

    res_loss = gated_residual_loss(delta_gated, delta_teacher, gate)
    mag_loss = magnitude_penalty(delta_gated, gate)

    total = (
        TRUE_LOSS_WEIGHT * true_loss
        + PCC_RESIDUAL_LOSS_WEIGHT * res_loss
        + MAGNITUDE_PENALTY_WEIGHT * mag_loss
    )

    return (
        total,
        true_loss.detach(),
        true_bce.detach(),
        true_dice.detach(),
        res_loss.detach(),
        mag_loss.detach(),
        delta_gated.detach(),
        final_logit.detach()
    )


# =====================
# Fold splits
# =====================
fold_splits = []

for fold in sorted(split_df["fold"].unique()):
    train_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "train")]["case_id"].tolist()
    test_case_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")]["case_id"].tolist()
    fold_splits.append((int(fold), train_case_ids, test_case_ids))

print("\nFold test cases:")
for fold, _, test_case_ids in fold_splits:
    print(f"\nFold {fold}:")
    for cid in test_case_ids:
        print(" ", cid)


# =====================
# Train / evaluate one fold
# =====================
def train_one_fold(fold, train_case_ids, test_case_ids):
    print("\n" + "=" * 80)
    print(f"C2-v2 conservative OOF gated / Fold {fold}")
    print("Train cases:", len(train_case_ids))
    print("Test cases:", len(test_case_ids))

    pos_weight, true_pos, true_total = compute_train_pos_weight_from_true_target(train_case_ids)

    print(f"True target positive voxels: {true_pos:.0f} / {true_total:.0f}")
    print(f"BCE pos_weight clipped: {pos_weight:.4f}")

    train_ds = C2V2Dataset(train_case_ids)
    train_sampler = make_weighted_sampler(train_ds)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    model = SmallUNet2DResidual(in_ch=3, out_ch=1, base=16).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    fold_loss_rows = []

    for epoch in range(1, EPOCHS + 1):
        model.train()

        running_total = 0.0
        running_true = 0.0
        running_bce = 0.0
        running_dice = 0.0
        running_res = 0.0
        running_mag = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"C2-v2 Fold {fold} Epoch {epoch}/{EPOCHS}", leave=False)

        for x3, y_true, b_logit, gate, delta_teacher, _, _ in pbar:
            x3 = x3.to(DEVICE, non_blocking=True)
            y_true = y_true.to(DEVICE, non_blocking=True)
            b_logit = b_logit.to(DEVICE, non_blocking=True)
            gate = gate.to(DEVICE, non_blocking=True)
            delta_teacher = delta_teacher.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                delta_raw = model(x3)
                loss, true_loss, true_bce, true_dice, res_loss, mag_loss, _, _ = combined_loss(
                    delta_raw=delta_raw,
                    b_logit=b_logit,
                    gate=gate,
                    y_true=y_true,
                    delta_teacher=delta_teacher,
                    pos_weight_tensor=pos_weight_tensor
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_total += float(loss.detach().cpu())
            running_true += float(true_loss.detach().cpu())
            running_bce += float(true_bce.detach().cpu())
            running_dice += float(true_dice.detach().cpu())
            running_res += float(res_loss.detach().cpu())
            running_mag += float(mag_loss.detach().cpu())
            n_batches += 1

            pbar.set_postfix({
                "loss": running_total / max(n_batches, 1),
                "true": running_true / max(n_batches, 1),
                "res": running_res / max(n_batches, 1),
                "mag": running_mag / max(n_batches, 1),
            })

        scheduler.step()

        row = {
            "fold": fold,
            "epoch": epoch,
            "loss": running_total / max(n_batches, 1),
            "true_loss": running_true / max(n_batches, 1),
            "true_bce": running_bce / max(n_batches, 1),
            "true_dice_loss": running_dice / max(n_batches, 1),
            "pcc_residual_loss": running_res / max(n_batches, 1),
            "magnitude_penalty": running_mag / max(n_batches, 1),
            "lr": scheduler.get_last_lr()[0],
            "true_pos_weight": pos_weight,
            "max_delta_logit": MAX_DELTA_LOGIT,
            "true_loss_weight": TRUE_LOSS_WEIGHT,
            "pcc_residual_loss_weight": PCC_RESIDUAL_LOSS_WEIGHT,
            "magnitude_penalty_weight": MAGNITUDE_PENALTY_WEIGHT,
        }

        fold_loss_rows.append(row)

        print(
            f"C2-v2 Fold {fold} Epoch {epoch:02d}: "
            f"loss={row['loss']:.5f}, "
            f"true={row['true_loss']:.5f}, "
            f"res={row['pcc_residual_loss']:.5f}, "
            f"mag={row['magnitude_penalty']:.5f}, "
            f"dice_loss={row['true_dice_loss']:.5f}"
        )

    ckpt_path = CKPT_DIR / f"c2_v2_conservative_oof_gated_fold_{fold}.pt"
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "train_case_ids": list(train_case_ids),
        "test_case_ids": list(test_case_ids),
        "config": {
            "method": "C2-v2 conservative OOF-gated residual alpha sweep",
            "epochs": EPOCHS,
            "max_delta_logit": MAX_DELTA_LOGIT,
            "true_loss_weight": TRUE_LOSS_WEIGHT,
            "pcc_residual_loss_weight": PCC_RESIDUAL_LOSS_WEIGHT,
            "magnitude_penalty_weight": MAGNITUDE_PENALTY_WEIGHT,
            "alphas": ALPHAS,
        }
    }, ckpt_path)

    print("Saved checkpoint:", ckpt_path)

    # Evaluate test cases under alpha sweep
    model.eval()

    metric_rows = []
    delta_rows = []

    with torch.no_grad():
        for case_id in tqdm(test_case_ids, desc=f"C2-v2 Fold {fold} prediction"):
            X, Y = load_case_npz(case_id)
            target = Y[:, 0].astype(bool)

            B = load_modelA_oof_map(case_id)
            B_logit = prob_to_logit_np(B)

            current_mask = X[:, 1].astype(bool)
            gate = make_slice_wise_dilation_region(current_mask, PCC_DILATION_RADIUS).astype(np.float32)

            delta_parts = []

            for start in range(0, X.shape[0], BATCH_SIZE):
                x_part = X[start:start+BATCH_SIZE]
                b_part = B[start:start+BATCH_SIZE]
                gate_part = gate[start:start+BATCH_SIZE]

                x3 = np.concatenate([
                    x_part,
                    b_part[:, None, :, :]
                ], axis=1).astype(np.float32)

                x3_t = torch.from_numpy(x3).float().to(DEVICE)
                gate_t = torch.from_numpy(gate_part[:, None, :, :]).float().to(DEVICE)

                delta_raw = model(x3_t)
                delta_pred = MAX_DELTA_LOGIT * torch.tanh(delta_raw)
                delta_gated = delta_pred * gate_t

                delta_parts.append(delta_gated.detach().cpu().numpy()[:, 0].astype(np.float32))

            delta_map = np.concatenate(delta_parts, axis=0)
            delta_map = np.clip(delta_map, -MAX_DELTA_LOGIT, MAX_DELTA_LOGIT).astype(np.float32)

            delta_path = DELTA_PRED_DIR / f"{case_id}_c2_v2_delta_logit_fold_{fold}.npy"
            np.save(delta_path, delta_map.astype(np.float16))

            base_metrics = topk_metrics(B, target)

            for alpha in ALPHAS:
                final_logit = B_logit + alpha * delta_map
                final_prob = sigmoid_np(final_logit).astype(np.float32)
                final_prob = normalize_prob_map(final_prob)

                m = topk_metrics(final_prob, target)

                metric_rows.append({
                    "case_id": case_id,
                    "fold": fold,
                    "alpha": alpha,
                    "method": f"C2-v2 conservative OOF-gated alpha={alpha}",
                    "test_time_future_target_access": "No",
                    "input_uses_model_A_oof_prediction": "Yes",
                    "training_signal": "true target + conservative PCC-on-OOF residual",
                    "delta_logit_pred_path": str(delta_path),

                    "model_A_direct_dice": base_metrics["dice"],
                    "model_A_direct_iou": base_metrics["iou"],
                    "model_A_direct_target_focus": base_metrics["target_focus"],
                    "model_A_direct_log10_ratio": base_metrics["log10_ratio"],

                    "dice": m["dice"],
                    "iou": m["iou"],
                    "target_focus": m["target_focus"],
                    "log10_ratio": m["log10_ratio"],
                })

            delta_rows.append({
                "case_id": case_id,
                "fold": fold,
                "delta_path": str(delta_path),
                "delta_abs_mean": float(np.mean(np.abs(delta_map))),
                "delta_abs_max": float(np.max(np.abs(delta_map))),
                "delta_positive_fraction": float((delta_map > 0).mean()),
                "delta_negative_fraction": float((delta_map < 0).mean()),
            })

    return fold_loss_rows, metric_rows, delta_rows


# =====================
# Run all folds
# =====================
start_time = time.time()

all_loss_rows = []
all_metric_rows = []
all_delta_rows = []

for fold, train_case_ids, test_case_ids in fold_splits:
    loss_rows, metric_rows, delta_rows = train_one_fold(
        fold,
        train_case_ids,
        test_case_ids
    )

    all_loss_rows.extend(loss_rows)
    all_metric_rows.extend(metric_rows)
    all_delta_rows.extend(delta_rows)

    pd.DataFrame(all_loss_rows).to_csv(C2_DIR / "c2_v2_training_losses_partial.csv", index=False)
    pd.DataFrame(all_metric_rows).to_csv(C2_DIR / "c2_v2_alpha_case_metrics_partial.csv", index=False)
    pd.DataFrame(all_delta_rows).to_csv(C2_DIR / "c2_v2_delta_diagnostics_partial.csv", index=False)


loss_df = pd.DataFrame(all_loss_rows)
metrics_df = pd.DataFrame(all_metric_rows)
delta_df = pd.DataFrame(all_delta_rows)

loss_path = C2_DIR / "c2_v2_training_losses.csv"
metrics_path = C2_DIR / "c2_v2_alpha_case_metrics.csv"
delta_path = C2_DIR / "c2_v2_delta_diagnostics.csv"

loss_df.to_csv(loss_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
delta_df.to_csv(delta_path, index=False)


# =====================
# Summaries
# =====================
summary = metrics_df.groupby("alpha")[["dice", "iou", "target_focus", "log10_ratio"]].agg(["mean", "median", "min", "max"])
summary_path = C2_DIR / "c2_v2_alpha_summary.csv"
summary.to_csv(summary_path)

pairwise_rows = []

for alpha in ALPHAS:
    sub = metrics_df[metrics_df["alpha"] == alpha].copy()

    for metric, base_col in [
        ("dice", "model_A_direct_dice"),
        ("iou", "model_A_direct_iou"),
        ("target_focus", "model_A_direct_target_focus"),
        ("log10_ratio", "model_A_direct_log10_ratio"),
    ]:
        diff = sub[metric] - sub[base_col]

        pairwise_rows.append({
            "alpha": alpha,
            "comparison": "C2-v2 alpha prediction vs Model A OOF base",
            "metric": metric,
            "mean_diff": float(diff.mean()),
            "median_diff": float(diff.median()),
            "c2_wins": int((diff > 0).sum()),
            "model_A_wins": int((diff < 0).sum()),
            "ties": int((diff == 0).sum()),
            "total": int(diff.notna().sum()),
            "c2_win_rate": float((diff > 0).mean()),
        })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = C2_DIR / "pairwise_C2_v2_alpha_vs_Model_A.csv"
pairwise_df.to_csv(pairwise_path, index=False)


# Compact comparison
modelA_baseline = modelA_metrics_df[["dice", "iou", "target_focus", "log10_ratio"]].mean()

comparison_rows = [{
    "Model": "Model A",
    "alpha": 0.0,
    "Mechanism": "direct future-change prediction",
    "Dice_mean": float(modelA_baseline["dice"]),
    "IoU_mean": float(modelA_baseline["iou"]),
    "Target_focus_mean": float(modelA_baseline["target_focus"]),
    "Log10_ratio_mean": float(modelA_baseline["log10_ratio"]),
}]

for alpha in ALPHAS:
    sub = metrics_df[metrics_df["alpha"] == alpha]
    means = sub[["dice", "iou", "target_focus", "log10_ratio"]].mean()

    comparison_rows.append({
        "Model": "C2-v2",
        "alpha": alpha,
        "Mechanism": "conservative OOF-gated PCC residual",
        "Dice_mean": float(means["dice"]),
        "IoU_mean": float(means["iou"]),
        "Target_focus_mean": float(means["target_focus"]),
        "Log10_ratio_mean": float(means["log10_ratio"]),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = C2_DIR / "model_A_vs_C2_v2_alpha_compact_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)


# Best alpha by mean Dice, excluding alpha=0 if wanted
nonzero_comparison = comparison_df[(comparison_df["Model"] == "C2-v2") & (comparison_df["alpha"] > 0)].copy()
best_by_dice = nonzero_comparison.sort_values("Dice_mean", ascending=False).head(1)

runtime_min = (time.time() - start_time) / 60.0

run_info = {
    "method": "C2-v2 conservative OOF-gated residual alpha sweep",
    "runtime_minutes": runtime_min,
    "epochs": EPOCHS,
    "test_time_future_target_access": "No",
    "input": "current T1c + current mask + Model A OOF prediction",
    "teacher": "PCC-v2 applied to Model A OOF prediction",
    "config": {
        "max_delta_logit": MAX_DELTA_LOGIT,
        "true_loss_weight": TRUE_LOSS_WEIGHT,
        "pcc_residual_loss_weight": PCC_RESIDUAL_LOSS_WEIGHT,
        "magnitude_penalty_weight": MAGNITUDE_PENALTY_WEIGHT,
        "alphas": ALPHAS,
    },
    "outputs": {
        "teacher_quality": str(teacher_quality_path),
        "teacher_summary": str(teacher_summary_path),
        "losses": str(loss_path),
        "metrics": str(metrics_path),
        "summary": str(summary_path),
        "pairwise": str(pairwise_path),
        "comparison": str(comparison_path),
        "delta_diagnostics": str(delta_path),
    }
}

with open(C2_DIR / "c2_v2_run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)


print("\n" + "=" * 80)
print("C2-v2 conservative OOF-gated alpha sweep finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved losses:", loss_path)
print("Saved metrics:", metrics_path)
print("Saved alpha summary:", summary_path)
print("Saved pairwise:", pairwise_path)
print("Saved comparison:", comparison_path)

print("\nTeacher quality summary:")
display(teacher_summary)

print("\nAlpha summary:")
display(summary)

print("\nPairwise C2-v2 alpha vs Model A:")
display(pairwise_df)

print("\nCompact comparison:")
display(comparison_df)

print("\nBest nonzero alpha by Dice:")
display(best_by_dice)

print("\nTraining losses tail:")
display(loss_df.tail(10))

print("\nDelta diagnostics:")
display(delta_df.describe(include="all"))

In [ ]:
from pathlib import Path
import shutil
import os

src = Path("/kaggle/working/pcc_independent_baseline/stage2_pcc_guided_student_learning/model_C2_v2_conservative_oof_gated_alpha_sweep_5epoch")
zip_base = "/kaggle/working/C2_V2_5EPOCH_BACKUP"

print("Source exists:", src.exists())

zip_file = shutil.make_archive(zip_base, "zip", src)

print("Saved:", zip_file)
print("Size MB:", round(os.path.getsize(zip_file) / 1024 / 1024, 2))

In [ ]:
# ============================================================
# Geometry / Ring-Prior Diagnostic Sweep
#
# Purpose:
#   Diagnose whether current tumour geometry can strongly predict
#   future-change target.
#
# It tests:
#   1. Model A raw prediction
#   2. Model A with current-mask suppression
#   3. Ring prior only
#   4. Model A + ring prior
#   5. Model A suppressed + ring prior
#
# No training. Very fast compared with C2 training.
# ============================================================

from pathlib import Path
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    from scipy.ndimage import distance_transform_edt, gaussian_filter
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import distance_transform_edt, gaussian_filter


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
MODEL_A_METRICS_PATH = MODEL_A_DIR / "direct_target_case_metrics.csv"
MODEL_A_PRED_DIR = MODEL_A_DIR / "direct_target_pred_maps"

GEOM_DIR = STAGE2_DIR / "geometry_ring_prior_diagnostic"
GEOM_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_PRE_SUMMARY_PATH.exists(), f"Missing preprocessing summary: {LOCKED_PRE_SUMMARY_PATH}"
assert MODEL_A_METRICS_PATH.exists(), f"Missing Model A metrics: {MODEL_A_METRICS_PATH}"
assert MODEL_A_PRED_DIR.exists(), f"Missing Model A prediction folder: {MODEL_A_PRED_DIR}"

locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
modelA_metrics_df = pd.read_csv(MODEL_A_METRICS_PATH)

case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))
all_case_ids = sorted(case_to_npz.keys())

print("Cases:", len(all_case_ids))
print("Output:", GEOM_DIR)


# =====================
# Config
# =====================
# Ring prior width: distance from current tumour boundary
SIGMAS = [2, 3, 4, 6, 8, 12, 16, 24, 32]

# Additive blend:
# score = B + beta * ring
BETAS = [0.10, 0.25, 0.50, 1.00, 2.00, 4.00, 8.00]

# Convex blend:
# score = (1-lambda) * B + lambda * ring
LAMBDAS = [0.10, 0.25, 0.50, 0.75]

# Suppress current tumour mask:
# score = B * outside + inside_weight * B * inside
INSIDE_WEIGHTS = [0.0, 0.05, 0.10, 0.25]

EPS = 1e-8


# =====================
# Utilities
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    if x.min() >= 0 and x.max() <= 1:
        return np.clip(x, 0, 1).astype(np.float32)

    x_min, x_max = float(x.min()), float(x.max())
    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)

    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics(pred, target, eps=1e-8):
    pred = normalize_prob_map(pred)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    top_idx = np.argsort(fp)[-k:]
    pb = np.zeros_like(ft, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)

    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case_npz(case_id):
    npz_path = case_to_npz[case_id]
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)       # [Z,2,H,W]
        Y = data["Y"].astype(np.float32)       # [Z,1,H,W]
    return X, Y


def load_modelA_oof_map(case_id):
    row = modelA_metrics_df[modelA_metrics_df["case_id"] == case_id]

    if len(row) == 1:
        p = Path(row.iloc[0]["pred_path"])
        if p.exists():
            return normalize_prob_map(np.load(p).astype(np.float32))

    candidates = sorted(MODEL_A_PRED_DIR.glob(f"{case_id}_direct_target_student_fold_*.npy"))

    if len(candidates) != 1:
        print("Candidates:", candidates)
        raise FileNotFoundError(f"Cannot uniquely find Model A prediction for {case_id}")

    return normalize_prob_map(np.load(candidates[0]).astype(np.float32))


def compute_outside_distance(current_mask):
    """
    For each slice, compute distance outside the current tumour mask.
    Inside current mask is set to 0.
    """
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    dist = np.zeros((Z, H, W), dtype=np.float32)

    for z in range(Z):
        m = current_mask[z]

        if m.sum() == 0:
            dist[z] = 0.0
            continue

        # distance_transform_edt(~m):
        # outside pixels get distance to nearest tumour pixel;
        # inside pixels are 0.
        d = distance_transform_edt(~m).astype(np.float32)
        d[m] = 0.0
        dist[z] = d

    return dist


def make_ring_prior_from_distance(dist_outside, current_mask, sigma):
    """
    Gaussian ring outside current tumour:
      ring = exp(-d^2 / (2*sigma^2))
    inside current mask is forced to 0.
    """
    ring = np.exp(-(dist_outside ** 2) / (2.0 * sigma * sigma)).astype(np.float32)
    ring[current_mask.astype(bool)] = 0.0

    # If a slice has no mask, ring will be close to 1 everywhere due dist=0,
    # so suppress slices with no current tumour.
    slice_has_mask = current_mask.reshape(current_mask.shape[0], -1).sum(axis=1) > 0
    for z, ok in enumerate(slice_has_mask):
        if not ok:
            ring[z] = 0.0

    return normalize_prob_map(ring)


def suppress_inside_current_mask(B, current_mask, inside_weight):
    current_mask = current_mask.astype(bool)

    out = B.copy().astype(np.float32)
    out[current_mask] = out[current_mask] * float(inside_weight)

    return normalize_prob_map(out)


def add_metrics(rows, case_id, method, score_map, target, **params):
    m = topk_metrics(score_map, target)

    row = {
        "case_id": case_id,
        "method": method,
        "dice": m["dice"],
        "iou": m["iou"],
        "target_focus": m["target_focus"],
        "log10_ratio": m["log10_ratio"],
    }

    row.update(params)
    rows.append(row)


# =====================
# Main diagnostic sweep
# =====================
start = time.time()
rows = []

for case_id in tqdm(all_case_ids, desc="Geometry diagnostic"):
    X, Y = load_case_npz(case_id)

    current_mask = X[:, 1].astype(bool)
    target = Y[:, 0].astype(bool)

    B = load_modelA_oof_map(case_id)
    B = normalize_prob_map(B)

    # Raw Model A
    add_metrics(
        rows,
        case_id,
        "ModelA_raw",
        B,
        target,
        sigma=np.nan,
        beta=np.nan,
        lam=np.nan,
        inside_weight=np.nan,
    )

    # Current-mask suppression only
    for inside_weight in INSIDE_WEIGHTS:
        B_supp = suppress_inside_current_mask(B, current_mask, inside_weight)

        add_metrics(
            rows,
            case_id,
            "ModelA_current_mask_suppressed",
            B_supp,
            target,
            sigma=np.nan,
            beta=np.nan,
            lam=np.nan,
            inside_weight=inside_weight,
        )

    # Distance and ring priors
    dist_outside = compute_outside_distance(current_mask)

    for sigma in SIGMAS:
        ring = make_ring_prior_from_distance(dist_outside, current_mask, sigma)

        # Ring only
        add_metrics(
            rows,
            case_id,
            "Ring_prior_only",
            ring,
            target,
            sigma=sigma,
            beta=np.nan,
            lam=np.nan,
            inside_weight=np.nan,
        )

        # Model A + ring additive
        for beta in BETAS:
            score = normalize_prob_map(B + beta * ring)

            add_metrics(
                rows,
                case_id,
                "ModelA_plus_ring_additive",
                score,
                target,
                sigma=sigma,
                beta=beta,
                lam=np.nan,
                inside_weight=np.nan,
            )

        # Model A suppressed + ring additive
        for inside_weight in INSIDE_WEIGHTS:
            B_supp = suppress_inside_current_mask(B, current_mask, inside_weight)

            for beta in BETAS:
                score = normalize_prob_map(B_supp + beta * ring)

                add_metrics(
                    rows,
                    case_id,
                    "ModelA_suppressed_plus_ring_additive",
                    score,
                    target,
                    sigma=sigma,
                    beta=beta,
                    lam=np.nan,
                    inside_weight=inside_weight,
                )

        # Model A + ring convex blend
        for lam in LAMBDAS:
            score = normalize_prob_map((1.0 - lam) * B + lam * ring)

            add_metrics(
                rows,
                case_id,
                "ModelA_ring_convex_blend",
                score,
                target,
                sigma=sigma,
                beta=np.nan,
                lam=lam,
                inside_weight=np.nan,
            )

        # Model A suppressed + ring convex blend
        for inside_weight in INSIDE_WEIGHTS:
            B_supp = suppress_inside_current_mask(B, current_mask, inside_weight)

            for lam in LAMBDAS:
                score = normalize_prob_map((1.0 - lam) * B_supp + lam * ring)

                add_metrics(
                    rows,
                    case_id,
                    "ModelA_suppressed_ring_convex_blend",
                    score,
                    target,
                    sigma=sigma,
                    beta=np.nan,
                    lam=lam,
                    inside_weight=inside_weight,
                )


case_metrics_df = pd.DataFrame(rows)

case_metrics_path = GEOM_DIR / "geometry_ring_prior_case_metrics.csv"
case_metrics_df.to_csv(case_metrics_path, index=False)

# Summary by configuration
summary_cols = ["method", "sigma", "beta", "lam", "inside_weight"]

summary_df = (
    case_metrics_df
    .groupby(summary_cols, dropna=False)[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
    .reset_index()
)

summary_path = GEOM_DIR / "geometry_ring_prior_summary.csv"
summary_df.to_csv(summary_path, index=False)

# Flatten columns for easier sorting/display
flat_summary = summary_df.copy()
flat_summary.columns = [
    "_".join([str(c) for c in col if str(c) != ""]).strip("_")
    if isinstance(col, tuple) else col
    for col in flat_summary.columns
]

flat_summary_path = GEOM_DIR / "geometry_ring_prior_summary_flat.csv"
flat_summary.to_csv(flat_summary_path, index=False)

# Best configs
best_by_dice = flat_summary.sort_values("dice_mean", ascending=False).head(20)
best_by_iou = flat_summary.sort_values("iou_mean", ascending=False).head(20)
best_by_focus = flat_summary.sort_values("target_focus_mean", ascending=False).head(20)

best_by_dice_path = GEOM_DIR / "geometry_best_by_dice_top20.csv"
best_by_iou_path = GEOM_DIR / "geometry_best_by_iou_top20.csv"
best_by_focus_path = GEOM_DIR / "geometry_best_by_focus_top20.csv"

best_by_dice.to_csv(best_by_dice_path, index=False)
best_by_iou.to_csv(best_by_iou_path, index=False)
best_by_focus.to_csv(best_by_focus_path, index=False)

# Pairwise best configs vs Model A raw
base = case_metrics_df[case_metrics_df["method"] == "ModelA_raw"][
    ["case_id", "dice", "iou", "target_focus", "log10_ratio"]
].rename(columns={
    "dice": "base_dice",
    "iou": "base_iou",
    "target_focus": "base_target_focus",
    "log10_ratio": "base_log10_ratio",
})

pairwise_rows = []

# Only top 50 dice configs for pairwise table to keep it compact
top_configs = flat_summary.sort_values("dice_mean", ascending=False).head(50)

for _, cfg in top_configs.iterrows():
    method = cfg["method"]
    sigma = cfg["sigma"]
    beta = cfg["beta"]
    lam = cfg["lam"]
    inside_weight = cfg["inside_weight"]

    sub = case_metrics_df[
        (case_metrics_df["method"] == method)
        & (case_metrics_df["sigma"].fillna(-9999) == (sigma if pd.notna(sigma) else -9999))
        & (case_metrics_df["beta"].fillna(-9999) == (beta if pd.notna(beta) else -9999))
        & (case_metrics_df["lam"].fillna(-9999) == (lam if pd.notna(lam) else -9999))
        & (case_metrics_df["inside_weight"].fillna(-9999) == (inside_weight if pd.notna(inside_weight) else -9999))
    ].merge(base, on="case_id", how="left")

    for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
        diff = sub[metric] - sub[f"base_{metric}"]

        pairwise_rows.append({
            "method": method,
            "sigma": sigma,
            "beta": beta,
            "lam": lam,
            "inside_weight": inside_weight,
            "metric": metric,
            "mean_diff": float(diff.mean()),
            "median_diff": float(diff.median()),
            "wins": int((diff > 0).sum()),
            "losses": int((diff < 0).sum()),
            "ties": int((diff == 0).sum()),
            "total": int(diff.notna().sum()),
            "win_rate": float((diff > 0).mean()),
        })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = GEOM_DIR / "geometry_pairwise_top50_vs_ModelA.csv"
pairwise_df.to_csv(pairwise_path, index=False)

runtime_min = (time.time() - start) / 60.0

print("\n" + "=" * 80)
print("Geometry / ring-prior diagnostic finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved case metrics:", case_metrics_path)
print("Saved summary:", summary_path)
print("Saved flat summary:", flat_summary_path)
print("Saved best by dice:", best_by_dice_path)
print("Saved pairwise:", pairwise_path)

print("\nBaseline Model A raw:")
display(flat_summary[flat_summary["method"] == "ModelA_raw"])

print("\nTop 20 configs by Dice:")
display(best_by_dice)

print("\nTop 20 configs by IoU:")
display(best_by_iou)

print("\nTop 20 configs by Target Focus:")
display(best_by_focus)

print("\nPairwise vs Model A for top Dice configs:")
display(pairwise_df.head(40))

In [ ]:
# ============================================================
# FAST Geometry / Ring-Prior Diagnostic Sweep
#
# Optimized:
#   - uses np.argpartition instead of full np.argsort
#   - fewer configs
#   - saves partial results after each case
# ============================================================

from pathlib import Path
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    from scipy.ndimage import distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import distance_transform_edt


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

LOCKED_PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
MODEL_A_METRICS_PATH = MODEL_A_DIR / "direct_target_case_metrics.csv"
MODEL_A_PRED_DIR = MODEL_A_DIR / "direct_target_pred_maps"

GEOM_DIR = STAGE2_DIR / "geometry_ring_prior_diagnostic_FAST"
GEOM_DIR.mkdir(parents=True, exist_ok=True)

assert LOCKED_PRE_SUMMARY_PATH.exists(), f"Missing preprocessing summary: {LOCKED_PRE_SUMMARY_PATH}"
assert MODEL_A_METRICS_PATH.exists(), f"Missing Model A metrics: {MODEL_A_METRICS_PATH}"
assert MODEL_A_PRED_DIR.exists(), f"Missing Model A prediction folder: {MODEL_A_PRED_DIR}"

locked_summary = pd.read_csv(LOCKED_PRE_SUMMARY_PATH)
modelA_metrics_df = pd.read_csv(MODEL_A_METRICS_PATH)

case_to_npz = dict(zip(locked_summary["case_id"], locked_summary["npz_path"]))
all_case_ids = sorted(case_to_npz.keys())

print("Cases:", len(all_case_ids))
print("Output:", GEOM_DIR)


# =====================
# Reduced config
# =====================
SIGMAS = [4, 8, 12, 16, 24, 32]
BETAS = [0.25, 0.50, 1.00, 2.00, 4.00]
INSIDE_WEIGHTS = [0.0, 0.10]

EPS = 1e-8


# =====================
# Utilities
# =====================
def normalize_prob_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    x_min = float(x.min())
    x_max = float(x.max())

    if x_max <= x_min:
        return np.zeros_like(x, dtype=np.float32)

    # For ranking metrics, min-max normalization does not change order.
    x = (x - x_min) / (x_max - x_min + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def topk_metrics_fast(pred, target, eps=1e-8):
    """
    Much faster top-k metric:
    uses argpartition, not full argsort.
    """
    pred = normalize_prob_map(pred)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    n = fp.size
    if k >= n:
        top_idx = np.arange(n)
    else:
        top_idx = np.argpartition(fp, n - k)[n - k:]

    pb = np.zeros(n, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)

    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case_npz(case_id):
    npz_path = case_to_npz[case_id]
    with np.load(npz_path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"].astype(np.float32)
    return X, Y


def load_modelA_oof_map(case_id):
    row = modelA_metrics_df[modelA_metrics_df["case_id"] == case_id]

    if len(row) == 1:
        p = Path(row.iloc[0]["pred_path"])
        if p.exists():
            return normalize_prob_map(np.load(p).astype(np.float32))

    candidates = sorted(MODEL_A_PRED_DIR.glob(f"{case_id}_direct_target_student_fold_*.npy"))

    if len(candidates) != 1:
        print("Candidates:", candidates)
        raise FileNotFoundError(f"Cannot uniquely find Model A prediction for {case_id}")

    return normalize_prob_map(np.load(candidates[0]).astype(np.float32))


def compute_outside_distance(current_mask):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape

    dist = np.zeros((Z, H, W), dtype=np.float32)

    for z in range(Z):
        m = current_mask[z]

        if m.sum() == 0:
            dist[z] = 0.0
            continue

        d = distance_transform_edt(~m).astype(np.float32)
        d[m] = 0.0
        dist[z] = d

    return dist


def make_ring_prior(dist_outside, current_mask, sigma):
    current_mask = current_mask.astype(bool)

    ring = np.exp(-(dist_outside ** 2) / (2.0 * sigma * sigma)).astype(np.float32)
    ring[current_mask] = 0.0

    slice_has_mask = current_mask.reshape(current_mask.shape[0], -1).sum(axis=1) > 0
    for z, ok in enumerate(slice_has_mask):
        if not ok:
            ring[z] = 0.0

    return normalize_prob_map(ring)


def suppress_inside_current_mask(B, current_mask, inside_weight):
    out = B.copy().astype(np.float32)
    out[current_mask.astype(bool)] *= float(inside_weight)
    return normalize_prob_map(out)


def add_row(rows, case_id, method, score, target, sigma=np.nan, beta=np.nan, inside_weight=np.nan):
    m = topk_metrics_fast(score, target)

    rows.append({
        "case_id": case_id,
        "method": method,
        "sigma": sigma,
        "beta": beta,
        "inside_weight": inside_weight,
        "dice": m["dice"],
        "iou": m["iou"],
        "target_focus": m["target_focus"],
        "log10_ratio": m["log10_ratio"],
    })


# =====================
# Run diagnostic
# =====================
start = time.time()
rows = []

partial_path = GEOM_DIR / "geometry_fast_case_metrics_PARTIAL.csv"

for idx, case_id in enumerate(all_case_ids, start=1):
    print(f"\n[{idx}/{len(all_case_ids)}] Processing {case_id}", flush=True)

    X, Y = load_case_npz(case_id)

    current_mask = X[:, 1].astype(bool)
    target = Y[:, 0].astype(bool)

    B = load_modelA_oof_map(case_id)

    # Baseline Model A
    add_row(
        rows,
        case_id,
        "ModelA_raw",
        B,
        target,
    )

    # Model A suppression only
    for inside_weight in INSIDE_WEIGHTS:
        B_supp = suppress_inside_current_mask(B, current_mask, inside_weight)

        add_row(
            rows,
            case_id,
            "ModelA_current_mask_suppressed",
            B_supp,
            target,
            inside_weight=inside_weight,
        )

    # Ring prior
    dist_outside = compute_outside_distance(current_mask)

    for sigma in SIGMAS:
        ring = make_ring_prior(dist_outside, current_mask, sigma)

        # Ring only
        add_row(
            rows,
            case_id,
            "Ring_prior_only",
            ring,
            target,
            sigma=sigma,
        )

        # Model A + ring
        for beta in BETAS:
            score = B + beta * ring

            add_row(
                rows,
                case_id,
                "ModelA_plus_ring_additive",
                score,
                target,
                sigma=sigma,
                beta=beta,
            )

        # Model A suppressed + ring
        for inside_weight in INSIDE_WEIGHTS:
            B_supp = suppress_inside_current_mask(B, current_mask, inside_weight)

            for beta in BETAS:
                score = B_supp + beta * ring

                add_row(
                    rows,
                    case_id,
                    "ModelA_suppressed_plus_ring_additive",
                    score,
                    target,
                    sigma=sigma,
                    beta=beta,
                    inside_weight=inside_weight,
                )

    # save partial every case
    pd.DataFrame(rows).to_csv(partial_path, index=False)
    print("Saved partial:", partial_path, "rows:", len(rows), flush=True)


case_metrics_df = pd.DataFrame(rows)
case_metrics_path = GEOM_DIR / "geometry_fast_case_metrics.csv"
case_metrics_df.to_csv(case_metrics_path, index=False)

# Summary
summary_cols = ["method", "sigma", "beta", "inside_weight"]

summary_df = (
    case_metrics_df
    .groupby(summary_cols, dropna=False)[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
    .reset_index()
)

summary_df.columns = [
    "_".join([str(c) for c in col if str(c) != ""]).strip("_")
    if isinstance(col, tuple) else col
    for col in summary_df.columns
]

summary_path = GEOM_DIR / "geometry_fast_summary.csv"
summary_df.to_csv(summary_path, index=False)

best_by_dice = summary_df.sort_values("dice_mean", ascending=False).head(20)
best_by_iou = summary_df.sort_values("iou_mean", ascending=False).head(20)
best_by_focus = summary_df.sort_values("target_focus_mean", ascending=False).head(20)

best_by_dice.to_csv(GEOM_DIR / "geometry_fast_best_by_dice_top20.csv", index=False)
best_by_iou.to_csv(GEOM_DIR / "geometry_fast_best_by_iou_top20.csv", index=False)
best_by_focus.to_csv(GEOM_DIR / "geometry_fast_best_by_focus_top20.csv", index=False)

# Pairwise vs Model A raw for top dice configs
base = case_metrics_df[case_metrics_df["method"] == "ModelA_raw"][
    ["case_id", "dice", "iou", "target_focus", "log10_ratio"]
].rename(columns={
    "dice": "base_dice",
    "iou": "base_iou",
    "target_focus": "base_target_focus",
    "log10_ratio": "base_log10_ratio",
})

pairwise_rows = []

for _, cfg in best_by_dice.iterrows():
    method = cfg["method"]
    sigma = cfg["sigma"]
    beta = cfg["beta"]
    inside_weight = cfg["inside_weight"]

    sub = case_metrics_df[
        (case_metrics_df["method"] == method)
        & (case_metrics_df["sigma"].fillna(-9999) == (sigma if pd.notna(sigma) else -9999))
        & (case_metrics_df["beta"].fillna(-9999) == (beta if pd.notna(beta) else -9999))
        & (case_metrics_df["inside_weight"].fillna(-9999) == (inside_weight if pd.notna(inside_weight) else -9999))
    ].merge(base, on="case_id", how="left")

    for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
        diff = sub[metric] - sub[f"base_{metric}"]

        pairwise_rows.append({
            "method": method,
            "sigma": sigma,
            "beta": beta,
            "inside_weight": inside_weight,
            "metric": metric,
            "mean_diff": float(diff.mean()),
            "median_diff": float(diff.median()),
            "wins": int((diff > 0).sum()),
            "losses": int((diff < 0).sum()),
            "ties": int((diff == 0).sum()),
            "total": int(diff.notna().sum()),
            "win_rate": float((diff > 0).mean()),
        })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = GEOM_DIR / "geometry_fast_pairwise_top20_vs_ModelA.csv"
pairwise_df.to_csv(pairwise_path, index=False)

runtime_min = (time.time() - start) / 60.0

print("\n" + "=" * 80)
print("FAST geometry / ring-prior diagnostic finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved case metrics:", case_metrics_path)
print("Saved summary:", summary_path)
print("Saved pairwise:", pairwise_path)

print("\nBaseline Model A raw:")
display(summary_df[summary_df["method"] == "ModelA_raw"])

print("\nTop 20 configs by Dice:")
display(best_by_dice)

print("\nTop 20 configs by IoU:")
display(best_by_iou)

print("\nTop 20 configs by Target Focus:")
display(best_by_focus)

print("\nPairwise vs Model A for top Dice configs:")
display(pairwise_df)

In [ ]:
from pathlib import Path
import zipfile
import shutil
import os

WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"
TARGET_MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
TMP_RESTORE_DIR = WORKING / "_restore_model_A_tmp"

print("Listing /kaggle/input:")
for p in INPUT.iterdir():
    print(" ", p)

# ------------------------------------------------------------
# Step 1: First try to find already-extracted Model A files
# ------------------------------------------------------------
metrics_candidates = list(INPUT.rglob("direct_target_case_metrics.csv"))

print("\nFound direct_target_case_metrics.csv candidates in /kaggle/input:")
for p in metrics_candidates:
    print(" ", p)

source_model_a_dir = None

if len(metrics_candidates) > 0:
    source_model_a_dir = metrics_candidates[0].parent
    print("\nUsing extracted Model A folder:", source_model_a_dir)

else:
    # ------------------------------------------------------------
    # Step 2: If not extracted, search zip and extract it
    # ------------------------------------------------------------
    zip_candidates = list(INPUT.rglob("*.zip"))

    print("\nNo extracted Model A folder found.")
    print("Zip candidates:")
    for p in zip_candidates:
        print(" ", p, round(os.path.getsize(p) / 1024 / 1024, 2), "MB")

    if len(zip_candidates) == 0:
        print("\nFiles under /kaggle/input/model-a-backup-final if exists:")
        possible = INPUT / "model-a-backup-final"
        if possible.exists():
            for p in possible.rglob("*"):
                print(" ", p)
        raise FileNotFoundError("Cannot find Model A zip or extracted Model A files under /kaggle/input.")

    model_a_zip = sorted(zip_candidates, key=lambda p: os.path.getsize(p), reverse=True)[0]
    print("\nUsing zip:", model_a_zip)

    if TMP_RESTORE_DIR.exists():
        shutil.rmtree(TMP_RESTORE_DIR)
    TMP_RESTORE_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(model_a_zip, "r") as zf:
        zf.extractall(TMP_RESTORE_DIR)

    metrics_candidates = list(TMP_RESTORE_DIR.rglob("direct_target_case_metrics.csv"))

    print("\nFound metrics candidates after extraction:")
    for p in metrics_candidates:
        print(" ", p)

    if len(metrics_candidates) == 0:
        raise FileNotFoundError("Zip does not contain direct_target_case_metrics.csv.")

    source_model_a_dir = metrics_candidates[0].parent
    print("\nUsing extracted Model A folder:", source_model_a_dir)

# ------------------------------------------------------------
# Step 3: Restore to expected working path
# ------------------------------------------------------------
TARGET_MODEL_A_DIR.parent.mkdir(parents=True, exist_ok=True)

if TARGET_MODEL_A_DIR.exists():
    print("\nRemoving existing target:", TARGET_MODEL_A_DIR)
    shutil.rmtree(TARGET_MODEL_A_DIR)

shutil.copytree(source_model_a_dir, TARGET_MODEL_A_DIR)

print("\nRestored Model A to:", TARGET_MODEL_A_DIR)

# ------------------------------------------------------------
# Step 4: Verify
# ------------------------------------------------------------
required = [
    TARGET_MODEL_A_DIR / "direct_target_case_metrics.csv",
    TARGET_MODEL_A_DIR / "direct_target_summary.csv",
    TARGET_MODEL_A_DIR / "direct_target_training_losses.csv",
    TARGET_MODEL_A_DIR / "matched_5fold_splits_seed42.csv",
    TARGET_MODEL_A_DIR / "direct_target_pred_maps",
]

print("\nVerification:")
for p in required:
    print(p, "exists:", p.exists())

pred_maps = list((TARGET_MODEL_A_DIR / "direct_target_pred_maps").glob("*.npy"))
print("\nPrediction maps:", len(pred_maps))

ckpt_dir = TARGET_MODEL_A_DIR / "checkpoints"
ckpts = list(ckpt_dir.glob("*.pt")) if ckpt_dir.exists() else []
print("Checkpoints:", len(ckpts))

assert len(pred_maps) == 40, f"Expected 40 prediction maps, got {len(pred_maps)}"
assert (TARGET_MODEL_A_DIR / "direct_target_case_metrics.csv").exists()
assert (TARGET_MODEL_A_DIR / "matched_5fold_splits_seed42.csv").exists()

print("\nMODEL A RESTORE SUCCESSFUL")

In [ ]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")

pre_summary = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
model_a_dir = OUT_DIR / "stage2_pcc_guided_student_learning/model_A_direct_target_5fold"
model_a_metrics = model_a_dir / "direct_target_case_metrics.csv"
pred_dir = model_a_dir / "direct_target_pred_maps"

print("Preprocessing summary exists:", pre_summary.exists())
print("Model A metrics exists:", model_a_metrics.exists())
print("Prediction folder exists:", pred_dir.exists())
print("Prediction maps:", len(list(pred_dir.glob('*.npy'))) if pred_dir.exists() else 0)

if pre_summary.exists():
    df = pd.read_csv(pre_summary)
    print("\nPreprocessed cases:", len(df))
    print("Total slices:", int(df["num_slices"].sum()))
    print("Total positive slices:", int(df["positive_slices"].sum()))
    print("Total target voxels:", int(df["target_voxels"].sum()))

if model_a_metrics.exists():
    mdf = pd.read_csv(model_a_metrics)
    print("\nModel A summary:")
    print(mdf[["dice", "iou", "target_focus", "log10_ratio"]].mean())

In [ ]:
# ============================================================
# FAST Geometry / Ring-Prior Diagnostic
# No training. Tests whether current-mask geometry can improve Dice.
# ============================================================

from pathlib import Path
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    from scipy.ndimage import distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import distance_transform_edt


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
MODEL_A_METRICS_PATH = MODEL_A_DIR / "direct_target_case_metrics.csv"
MODEL_A_PRED_DIR = MODEL_A_DIR / "direct_target_pred_maps"

GEOM_DIR = STAGE2_DIR / "geometry_ring_prior_diagnostic_FAST_v2"
GEOM_DIR.mkdir(parents=True, exist_ok=True)

assert PRE_SUMMARY_PATH.exists(), f"Missing: {PRE_SUMMARY_PATH}"
assert MODEL_A_METRICS_PATH.exists(), f"Missing: {MODEL_A_METRICS_PATH}"
assert MODEL_A_PRED_DIR.exists(), f"Missing: {MODEL_A_PRED_DIR}"

summary_df = pd.read_csv(PRE_SUMMARY_PATH)
modelA_df = pd.read_csv(MODEL_A_METRICS_PATH)

case_to_npz = dict(zip(summary_df["case_id"], summary_df["npz_path"]))
case_ids = sorted(case_to_npz.keys())

print("Cases:", len(case_ids))
print("Output:", GEOM_DIR)


# =====================
# Config: intentionally small and fast
# =====================
SIGMAS = [4, 8, 12, 16, 24, 32]
BETAS = [0.5, 1.0, 2.0]
INSIDE_WEIGHTS = [0.0, 0.10]

EPS = 1e-8


# =====================
# Utility
# =====================
def normalize_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    mn, mx = float(x.min()), float(x.max())
    if mx <= mn:
        return np.zeros_like(x, dtype=np.float32)

    return ((x - mn) / (mx - mn + 1e-8)).astype(np.float32)


def topk_metrics_fast(pred, target, eps=1e-8):
    pred = normalize_map(pred)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    n = fp.size
    if k >= n:
        top_idx = np.arange(n)
    else:
        top_idx = np.argpartition(fp, n - k)[n - k:]

    pb = np.zeros(n, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)

    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case(case_id):
    with np.load(case_to_npz[case_id], allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"].astype(np.float32)
    return X, Y


def load_modelA_map(case_id):
    row = modelA_df[modelA_df["case_id"] == case_id]

    if len(row) == 1:
        p = Path(row.iloc[0]["pred_path"])
        if p.exists():
            return normalize_map(np.load(p).astype(np.float32))

    candidates = sorted(MODEL_A_PRED_DIR.glob(f"{case_id}_direct_target_student_fold_*.npy"))
    if len(candidates) != 1:
        raise FileNotFoundError(f"Cannot find Model A map for {case_id}, found {len(candidates)}")

    return normalize_map(np.load(candidates[0]).astype(np.float32))


def outside_distance(current_mask):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    dist = np.zeros((Z, H, W), dtype=np.float32)

    for z in range(Z):
        m = current_mask[z]
        if m.sum() == 0:
            continue

        d = distance_transform_edt(~m).astype(np.float32)
        d[m] = 0.0
        dist[z] = d

    return dist


def ring_prior(dist, current_mask, sigma):
    current_mask = current_mask.astype(bool)

    ring = np.exp(-(dist ** 2) / (2.0 * sigma * sigma)).astype(np.float32)
    ring[current_mask] = 0.0

    # suppress slices without current tumour
    has_mask = current_mask.reshape(current_mask.shape[0], -1).sum(axis=1) > 0
    for z, ok in enumerate(has_mask):
        if not ok:
            ring[z] = 0.0

    return normalize_map(ring)


def suppress_inside(B, current_mask, inside_weight):
    out = B.copy().astype(np.float32)
    out[current_mask.astype(bool)] *= float(inside_weight)
    return normalize_map(out)


def add_result(rows, case_id, method, score, target, sigma=np.nan, beta=np.nan, inside_weight=np.nan):
    m = topk_metrics_fast(score, target)

    rows.append({
        "case_id": case_id,
        "method": method,
        "sigma": sigma,
        "beta": beta,
        "inside_weight": inside_weight,
        "dice": m["dice"],
        "iou": m["iou"],
        "target_focus": m["target_focus"],
        "log10_ratio": m["log10_ratio"],
    })


# =====================
# Run
# =====================
rows = []
start = time.time()
partial_path = GEOM_DIR / "geometry_fast_partial.csv"

for i, case_id in enumerate(case_ids, start=1):
    print(f"\n[{i}/{len(case_ids)}] {case_id}", flush=True)

    X, Y = load_case(case_id)
    current_mask = X[:, 1].astype(bool)
    target = Y[:, 0].astype(bool)

    B = load_modelA_map(case_id)

    # Model A raw
    add_result(rows, case_id, "ModelA_raw", B, target)

    # Model A with current tumour suppressed
    for iw in INSIDE_WEIGHTS:
        B_supp = suppress_inside(B, current_mask, iw)
        add_result(rows, case_id, "ModelA_current_mask_suppressed", B_supp, target, inside_weight=iw)

    # Ring priors
    dist = outside_distance(current_mask)

    for sigma in SIGMAS:
        R = ring_prior(dist, current_mask, sigma)

        # Ring only
        add_result(rows, case_id, "Ring_prior_only", R, target, sigma=sigma)

        # Model A + ring
        for beta in BETAS:
            score = B + beta * R
            add_result(rows, case_id, "ModelA_plus_ring", score, target, sigma=sigma, beta=beta)

        # Model A suppressed + ring
        for iw in INSIDE_WEIGHTS:
            B_supp = suppress_inside(B, current_mask, iw)
            for beta in BETAS:
                score = B_supp + beta * R
                add_result(
                    rows,
                    case_id,
                    "ModelA_suppressed_plus_ring",
                    score,
                    target,
                    sigma=sigma,
                    beta=beta,
                    inside_weight=iw,
                )

    pd.DataFrame(rows).to_csv(partial_path, index=False)
    print("Saved partial rows:", len(rows), flush=True)


case_df = pd.DataFrame(rows)
case_path = GEOM_DIR / "geometry_fast_case_metrics.csv"
case_df.to_csv(case_path, index=False)

summary = (
    case_df
    .groupby(["method", "sigma", "beta", "inside_weight"], dropna=False)[["dice", "iou", "target_focus", "log10_ratio"]]
    .agg(["mean", "median", "min", "max"])
    .reset_index()
)

summary.columns = [
    "_".join([str(c) for c in col if str(c) != ""]).strip("_")
    if isinstance(col, tuple) else col
    for col in summary.columns
]

summary_path = GEOM_DIR / "geometry_fast_summary.csv"
summary.to_csv(summary_path, index=False)

best_dice = summary.sort_values("dice_mean", ascending=False).head(20)
best_iou = summary.sort_values("iou_mean", ascending=False).head(20)
best_focus = summary.sort_values("target_focus_mean", ascending=False).head(20)

best_dice.to_csv(GEOM_DIR / "best_by_dice_top20.csv", index=False)
best_iou.to_csv(GEOM_DIR / "best_by_iou_top20.csv", index=False)
best_focus.to_csv(GEOM_DIR / "best_by_focus_top20.csv", index=False)

# Pairwise vs Model A raw for top dice configs
base = case_df[case_df["method"] == "ModelA_raw"][
    ["case_id", "dice", "iou", "target_focus", "log10_ratio"]
].rename(columns={
    "dice": "base_dice",
    "iou": "base_iou",
    "target_focus": "base_target_focus",
    "log10_ratio": "base_log10_ratio",
})

pairwise_rows = []

for _, cfg in best_dice.iterrows():
    method = cfg["method"]
    sigma = cfg["sigma"]
    beta = cfg["beta"]
    iw = cfg["inside_weight"]

    sub = case_df[
        (case_df["method"] == method)
        & (case_df["sigma"].fillna(-9999) == (sigma if pd.notna(sigma) else -9999))
        & (case_df["beta"].fillna(-9999) == (beta if pd.notna(beta) else -9999))
        & (case_df["inside_weight"].fillna(-9999) == (iw if pd.notna(iw) else -9999))
    ].merge(base, on="case_id", how="left")

    for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
        diff = sub[metric] - sub[f"base_{metric}"]

        pairwise_rows.append({
            "method": method,
            "sigma": sigma,
            "beta": beta,
            "inside_weight": iw,
            "metric": metric,
            "mean_diff": float(diff.mean()),
            "median_diff": float(diff.median()),
            "wins": int((diff > 0).sum()),
            "losses": int((diff < 0).sum()),
            "ties": int((diff == 0).sum()),
            "win_rate": float((diff > 0).mean()),
        })

pairwise = pd.DataFrame(pairwise_rows)
pairwise_path = GEOM_DIR / "pairwise_top20_vs_ModelA.csv"
pairwise.to_csv(pairwise_path, index=False)

runtime = (time.time() - start) / 60.0

print("\n" + "=" * 80)
print("FAST geometry diagnostic finished.")
print(f"Runtime: {runtime:.2f} minutes")
print("Saved:", GEOM_DIR)

print("\nBaseline Model A raw:")
display(summary[summary["method"] == "ModelA_raw"])

print("\nTop 20 by Dice:")
display(best_dice)

print("\nTop 20 by IoU:")
display(best_iou)

print("\nTop 20 by Target Focus:")
display(best_focus)

print("\nPairwise vs Model A for top Dice configs:")
display(pairwise)

In [ ]:
# ============================================================
# C2-v3 Geometry-Aware Quick
#
# Goal:
#   Start from a stronger geometry-enhanced base map:
#       Geometry base = Model A OOF + ring prior
#   Then learn PCC-derived residual correction over this stronger base.
#
# This is a quick 5-epoch diagnostic before full training.
#
# Key idea:
#   Model A Dice ≈ 0.277
#   Geometry base Dice ≈ 0.321
#   C2-v3 tries to improve beyond geometry base.
#
# Test-time:
#   Uses current MRI + current mask + Model A OOF + ring priors.
#   Does NOT use future target or PCC teacher at inference.
# ============================================================

import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from scipy.ndimage import distance_transform_edt, gaussian_filter
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import distance_transform_edt, gaussian_filter


# =====================
# Config
# =====================
SEED = 42

# Quick version first.
EPOCHS = 5

BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

POS_SLICE_WEIGHT = 3.0
NEG_SLICE_WEIGHT = 1.0

# Geometry base chosen from FAST diagnostic:
# best Dice config was sigma=8, beta=2, inside_weight=0.1
GEOM_MAIN_SIGMA = 8
GEOM_MAIN_BETA = 2.0
GEOM_INSIDE_WEIGHT = 0.10

# Extra ring channel for balance/focus
GEOM_AUX_SIGMA = 4

# PCC teacher
PCC_ROUNDS = 10
PCC_ETA_POS = 0.30
PCC_ETA_NEG = 0.10
PCC_DILATION_RADIUS = 26
PCC_SIGMA = 2.0

# Residual learning
MAX_DELTA_LOGIT = 4.0
TRUE_LOSS_WEIGHT = 0.90
PCC_RESIDUAL_LOSS_WEIGHT = 0.07
RANKING_LOSS_WEIGHT = 0.05
MAGNITUDE_PENALTY_WEIGHT = 0.002

RESIDUAL_IMPORTANCE_WEIGHT = 2.0
LOGIT_EPS = 1e-4

# Alpha sweep at inference
ALPHAS = [0.0, 0.10, 0.25, 0.50, 0.75, 1.00, 1.25]

FORCE_REGENERATE_TEACHERS = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
MODEL_A_METRICS_PATH = MODEL_A_DIR / "direct_target_case_metrics.csv"
MODEL_A_PRED_DIR = MODEL_A_DIR / "direct_target_pred_maps"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"

C2_DIR = STAGE2_DIR / "model_C2_v3_geometry_aware_quick_5epoch"
TEACHER_DIR = C2_DIR / "pcc_on_geometry_base_teacher_maps"
DELTA_DIR = C2_DIR / "delta_logit_pred_maps"
CKPT_DIR = C2_DIR / "checkpoints"

C2_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_DIR.mkdir(parents=True, exist_ok=True)
DELTA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert PRE_SUMMARY_PATH.exists(), f"Missing preprocessing summary: {PRE_SUMMARY_PATH}"
assert MODEL_A_METRICS_PATH.exists(), f"Missing Model A metrics: {MODEL_A_METRICS_PATH}"
assert MODEL_A_PRED_DIR.exists(), f"Missing Model A pred folder: {MODEL_A_PRED_DIR}"
assert SPLIT_PATH.exists(), f"Missing split file: {SPLIT_PATH}"

pre_df = pd.read_csv(PRE_SUMMARY_PATH)
modelA_df = pd.read_csv(MODEL_A_METRICS_PATH)
split_df = pd.read_csv(SPLIT_PATH)

case_to_npz = dict(zip(pre_df["case_id"], pre_df["npz_path"]))
case_ids = sorted(case_to_npz.keys())

print("Cases:", len(case_ids))
print("C2-v3 output:", C2_DIR)


# =====================
# Utilities
# =====================
def normalize_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    mn, mx = float(x.min()), float(x.max())
    if mx <= mn:
        return np.zeros_like(x, dtype=np.float32)

    x = (x - mn) / (mx - mn + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def prob_to_logit_np(p, eps=LOGIT_EPS):
    p = np.clip(p.astype(np.float32), eps, 1.0 - eps)
    return np.log(p / (1.0 - p)).astype(np.float32)


def sigmoid_np(x):
    x = np.clip(x, -30, 30)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)


def topk_metrics_fast(pred, target, eps=1e-8):
    pred = normalize_map(pred)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    n = fp.size
    if k >= n:
        top_idx = np.arange(n)
    else:
        top_idx = np.argpartition(fp, n - k)[n - k:]

    pb = np.zeros(n, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)

    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case(case_id):
    with np.load(case_to_npz[case_id], allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"].astype(np.float32)
    return X, Y


def load_modelA_oof(case_id):
    row = modelA_df[modelA_df["case_id"] == case_id]

    if len(row) == 1:
        p = Path(row.iloc[0]["pred_path"])
        if p.exists():
            return normalize_map(np.load(p).astype(np.float32))

    candidates = sorted(MODEL_A_PRED_DIR.glob(f"{case_id}_direct_target_student_fold_*.npy"))
    if len(candidates) != 1:
        raise FileNotFoundError(f"Cannot find Model A OOF map for {case_id}, found {len(candidates)}")

    return normalize_map(np.load(candidates[0]).astype(np.float32))


def outside_distance(current_mask):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    dist = np.zeros((Z, H, W), dtype=np.float32)

    for z in range(Z):
        m = current_mask[z]
        if m.sum() == 0:
            continue

        d = distance_transform_edt(~m).astype(np.float32)
        d[m] = 0.0
        dist[z] = d

    return dist


def make_ring_from_dist(dist, current_mask, sigma):
    current_mask = current_mask.astype(bool)

    ring = np.exp(-(dist ** 2) / (2.0 * sigma * sigma)).astype(np.float32)
    ring[current_mask] = 0.0

    has_mask = current_mask.reshape(current_mask.shape[0], -1).sum(axis=1) > 0
    for z, ok in enumerate(has_mask):
        if not ok:
            ring[z] = 0.0

    return normalize_map(ring)


def suppress_inside(B, current_mask, inside_weight):
    out = B.copy().astype(np.float32)
    out[current_mask.astype(bool)] *= float(inside_weight)
    return normalize_map(out)


def make_gate_from_dist(dist, current_mask, radius):
    current_mask = current_mask.astype(bool)

    gate = (dist <= radius).astype(np.float32)

    has_mask = current_mask.reshape(current_mask.shape[0], -1).sum(axis=1) > 0
    for z, ok in enumerate(has_mask):
        if not ok:
            gate[z] = 0.0

    return gate.astype(np.float32)


def make_geometry_features(case_id):
    X, Y = load_case(case_id)

    current_mask = X[:, 1].astype(bool)
    B = load_modelA_oof(case_id)

    dist = outside_distance(current_mask)

    ring_main = make_ring_from_dist(dist, current_mask, GEOM_MAIN_SIGMA)  # sigma=8
    ring_aux = make_ring_from_dist(dist, current_mask, GEOM_AUX_SIGMA)    # sigma=4

    B_supp = suppress_inside(B, current_mask, GEOM_INSIDE_WEIGHT)

    geometry_base = normalize_map(B_supp + GEOM_MAIN_BETA * ring_main)

    gate = make_gate_from_dist(dist, current_mask, PCC_DILATION_RADIUS)

    return {
        "X": X,
        "Y": Y,
        "B": B,
        "ring_main": ring_main,
        "ring_aux": ring_aux,
        "geometry_base": geometry_base,
        "gate": gate,
        "current_mask": current_mask,
    }


def smooth_target_2d(target, sigma):
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))

    if S.max() > 0:
        S = S / (S.max() + 1e-8)

    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_v2_on_base_map(B, current_mask, target):
    B = normalize_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)

    dist = outside_distance(current_mask)
    R = make_gate_from_dist(dist, current_mask, PCC_DILATION_RADIUS).astype(bool)
    S = smooth_target_2d(target, PCC_SIGMA)

    P = B.copy().astype(np.float32)

    for _ in range(PCC_ROUNDS):
        P_new = P.copy()

        positive_signal = S * (1.0 - P)
        negative_signal = (1.0 - S) * P
        negative_signal[target] = 0.0

        P_new[R] = (
            P[R]
            + PCC_ETA_POS * positive_signal[R]
            - PCC_ETA_NEG * negative_signal[R]
        )

        P_new[~R] = B[~R]
        P = np.clip(P_new, 0, 1).astype(np.float32)

    return P.astype(np.float32)


def teacher_path(case_id):
    return TEACHER_DIR / f"{case_id}_pcc_on_geometry_base_teacher.npy"


def generate_or_load_teacher(case_id):
    p = teacher_path(case_id)

    if p.exists() and not FORCE_REGENERATE_TEACHERS:
        return normalize_map(np.load(p).astype(np.float32))

    feats = make_geometry_features(case_id)
    target = feats["Y"][:, 0].astype(bool)

    P_teacher = run_pcc_v2_on_base_map(
        B=feats["geometry_base"],
        current_mask=feats["current_mask"],
        target=target,
    )

    np.save(p, P_teacher.astype(np.float16))
    return P_teacher


def compute_train_pos_weight(train_case_ids):
    pos = 0.0
    total = 0.0

    for cid in train_case_ids:
        _, Y = load_case(cid)
        pos += float(Y.sum())
        total += float(Y.size)

    neg = total - pos
    pos_weight = neg / (pos + 1e-8)
    pos_weight = float(np.clip(pos_weight, 1.0, 50.0))

    return pos_weight, pos, total


# =====================
# Teacher generation / diagnostic
# =====================
teacher_rows = []

print("\nGenerating / loading PCC-on-geometry-base teachers...")
for cid in tqdm(case_ids):
    feats = make_geometry_features(cid)

    target = feats["Y"][:, 0].astype(bool)
    B = feats["B"]
    G = feats["geometry_base"]
    P_teacher = generate_or_load_teacher(cid)

    mA = topk_metrics_fast(B, target)
    mG = topk_metrics_fast(G, target)
    mT = topk_metrics_fast(P_teacher, target)

    teacher_rows.append({
        "case_id": cid,

        "model_A_dice": mA["dice"],
        "model_A_iou": mA["iou"],
        "model_A_target_focus": mA["target_focus"],
        "model_A_log10_ratio": mA["log10_ratio"],

        "geometry_base_dice": mG["dice"],
        "geometry_base_iou": mG["iou"],
        "geometry_base_target_focus": mG["target_focus"],
        "geometry_base_log10_ratio": mG["log10_ratio"],

        "pcc_teacher_dice": mT["dice"],
        "pcc_teacher_iou": mT["iou"],
        "pcc_teacher_target_focus": mT["target_focus"],
        "pcc_teacher_log10_ratio": mT["log10_ratio"],

        "teacher_path": str(teacher_path(cid)),
    })

teacher_df = pd.DataFrame(teacher_rows)
teacher_path_csv = C2_DIR / "c2_v3_teacher_quality.csv"
teacher_df.to_csv(teacher_path_csv, index=False)

teacher_summary = pd.DataFrame([{
    "model_A_dice_mean": teacher_df["model_A_dice"].mean(),
    "model_A_iou_mean": teacher_df["model_A_iou"].mean(),
    "model_A_target_focus_mean": teacher_df["model_A_target_focus"].mean(),
    "model_A_log10_ratio_mean": teacher_df["model_A_log10_ratio"].mean(),

    "geometry_base_dice_mean": teacher_df["geometry_base_dice"].mean(),
    "geometry_base_iou_mean": teacher_df["geometry_base_iou"].mean(),
    "geometry_base_target_focus_mean": teacher_df["geometry_base_target_focus"].mean(),
    "geometry_base_log10_ratio_mean": teacher_df["geometry_base_log10_ratio"].mean(),

    "pcc_teacher_dice_mean": teacher_df["pcc_teacher_dice"].mean(),
    "pcc_teacher_iou_mean": teacher_df["pcc_teacher_iou"].mean(),
    "pcc_teacher_target_focus_mean": teacher_df["pcc_teacher_target_focus"].mean(),
    "pcc_teacher_log10_ratio_mean": teacher_df["pcc_teacher_log10_ratio"].mean(),
}])

teacher_summary_path = C2_DIR / "c2_v3_teacher_quality_summary.csv"
teacher_summary.to_csv(teacher_summary_path, index=False)

print("\nTeacher quality summary:")
display(teacher_summary)


# =====================
# Dataset
# =====================
class C2V3Dataset(Dataset):
    def __init__(self, case_ids):
        self.case_ids = list(case_ids)
        self.case_data = {}
        self.index = []
        self.slice_has_target = []

        for cid in self.case_ids:
            feats = make_geometry_features(cid)

            X = feats["X"]
            Y = feats["Y"]
            B = feats["B"]
            ring_main = feats["ring_main"]
            ring_aux = feats["ring_aux"]
            G = feats["geometry_base"]
            gate = feats["gate"]

            P_teacher = normalize_map(np.load(teacher_path(cid)).astype(np.float32))

            G_logit = prob_to_logit_np(G)
            T_logit = prob_to_logit_np(P_teacher)

            delta_teacher = np.clip(
                T_logit - G_logit,
                -MAX_DELTA_LOGIT,
                MAX_DELTA_LOGIT
            ).astype(np.float32)

            delta_teacher = delta_teacher * gate

            self.case_data[cid] = {
                "X": X,
                "Y": Y,
                "B": B,
                "ring_main": ring_main,
                "ring_aux": ring_aux,
                "G": G,
                "G_logit": G_logit,
                "gate": gate,
                "delta_teacher": delta_teacher,
            }

            Z = X.shape[0]
            for z in range(Z):
                self.index.append((cid, z))
                self.slice_has_target.append(float(Y[z].sum() > 0))

        self.slice_has_target = np.asarray(self.slice_has_target, dtype=np.float32)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        cid, z = self.index[idx]
        d = self.case_data[cid]

        X = d["X"]
        Y = d["Y"]
        B = d["B"]
        ring_main = d["ring_main"]
        ring_aux = d["ring_aux"]
        G = d["G"]
        G_logit = d["G_logit"]
        gate = d["gate"]
        delta_teacher = d["delta_teacher"]

        # 6-channel input:
        # current T1c, current mask, Model A OOF, ring8, ring4, geometry base
        x6 = np.concatenate([
            X[z],                            # 2 channels
            B[z][None, :, :],                # 1
            ring_main[z][None, :, :],        # 1
            ring_aux[z][None, :, :],         # 1
            G[z][None, :, :],                # 1
        ], axis=0).astype(np.float32)

        return (
            torch.from_numpy(x6).float(),
            torch.from_numpy(Y[z]).float(),
            torch.from_numpy(G_logit[z][None, :, :]).float(),
            torch.from_numpy(gate[z][None, :, :]).float(),
            torch.from_numpy(delta_teacher[z][None, :, :]).float(),
            cid,
            z,
        )


def make_sampler(ds):
    weights = np.where(
        ds.slice_has_target > 0,
        POS_SLICE_WEIGHT,
        NEG_SLICE_WEIGHT
    ).astype(np.float32)

    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True,
    )


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2DResidual(nn.Module):
    def __init__(self, in_ch=6, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


# =====================
# Losses
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)

    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def residual_loss(delta_gated, delta_teacher, gate, eps=1e-6):
    w = 1.0 + RESIDUAL_IMPORTANCE_WEIGHT * torch.clamp(
        torch.abs(delta_teacher) / MAX_DELTA_LOGIT,
        0.0,
        1.0
    )

    raw = F.smooth_l1_loss(delta_gated, delta_teacher, reduction="none")
    mask = (gate > 0.5).float()

    return torch.sum(raw * w * mask) / (torch.sum(mask) + eps)


def magnitude_penalty(delta_gated, gate, eps=1e-6):
    mask = (gate > 0.5).float()
    return torch.sum(torch.abs(delta_gated) * mask) / (torch.sum(mask) + eps)


def sampled_ranking_loss(logits, targets, max_pairs=512, hard_neg_pool=4096, margin=0.20):
    """
    Sampled ranking loss:
    encourages target pixels to rank above hard non-target pixels.
    This directly supports top-k Dice.
    """
    B = logits.shape[0]
    losses = []

    for i in range(B):
        li = logits[i, 0].reshape(-1)
        ti = targets[i, 0].reshape(-1) > 0.5

        pos = li[ti]
        neg = li[~ti]

        if pos.numel() == 0 or neg.numel() == 0:
            continue

        n = min(max_pairs, pos.numel(), neg.numel())

        pos_idx = torch.randint(0, pos.numel(), (n,), device=logits.device)
        pos_s = pos[pos_idx]

        pool = min(hard_neg_pool, neg.numel())
        hard_neg = torch.topk(neg, k=pool).values
        neg_idx = torch.randint(0, pool, (n,), device=logits.device)
        neg_s = hard_neg[neg_idx]

        losses.append(F.softplus(margin - (pos_s - neg_s)).mean())

    if len(losses) == 0:
        return logits.sum() * 0.0

    return torch.stack(losses).mean()


def combined_loss(delta_raw, base_logit, gate, y_true, delta_teacher, pos_weight_tensor):
    delta_pred = MAX_DELTA_LOGIT * torch.tanh(delta_raw)
    delta_gated = delta_pred * gate

    final_logit = base_logit + delta_gated

    true_bce = F.binary_cross_entropy_with_logits(
        final_logit,
        y_true,
        pos_weight=pos_weight_tensor,
    )
    true_dice = soft_dice_loss_from_logits(final_logit, y_true)
    true_loss = 0.5 * true_bce + 0.5 * true_dice

    res = residual_loss(delta_gated, delta_teacher, gate)
    mag = magnitude_penalty(delta_gated, gate)
    rank = sampled_ranking_loss(final_logit, y_true)

    total = (
        TRUE_LOSS_WEIGHT * true_loss
        + PCC_RESIDUAL_LOSS_WEIGHT * res
        + RANKING_LOSS_WEIGHT * rank
        + MAGNITUDE_PENALTY_WEIGHT * mag
    )

    return (
        total,
        true_loss.detach(),
        true_bce.detach(),
        true_dice.detach(),
        res.detach(),
        mag.detach(),
        rank.detach(),
        delta_gated.detach(),
        final_logit.detach(),
    )


# =====================
# Fold splits
# =====================
fold_splits = []

for fold in sorted(split_df["fold"].unique()):
    train_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "train")]["case_id"].tolist()
    test_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")]["case_id"].tolist()
    fold_splits.append((int(fold), train_ids, test_ids))

print("\nFold test cases:")
for fold, _, test_ids in fold_splits:
    print(f"\nFold {fold}:")
    for cid in test_ids:
        print(" ", cid)


# =====================
# Train one fold
# =====================
def train_one_fold(fold, train_ids, test_ids):
    print("\n" + "=" * 80)
    print(f"C2-v3 Geometry-Aware Quick / Fold {fold}")
    print("Train cases:", len(train_ids))
    print("Test cases:", len(test_ids))

    pos_weight, true_pos, total_vox = compute_train_pos_weight(train_ids)
    print(f"True target positive voxels: {true_pos:.0f} / {total_vox:.0f}")
    print(f"BCE pos_weight clipped: {pos_weight:.4f}")

    train_ds = C2V3Dataset(train_ids)
    sampler = make_sampler(train_ds)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model = SmallUNet2DResidual(in_ch=6, out_ch=1, base=16).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    loss_rows = []

    for epoch in range(1, EPOCHS + 1):
        model.train()

        totals = {
            "loss": 0.0,
            "true": 0.0,
            "bce": 0.0,
            "dice": 0.0,
            "res": 0.0,
            "mag": 0.0,
            "rank": 0.0,
        }
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"C2-v3 Fold {fold} Epoch {epoch}/{EPOCHS}", leave=False)

        for x6, y_true, base_logit, gate, delta_teacher, _, _ in pbar:
            x6 = x6.to(DEVICE, non_blocking=True)
            y_true = y_true.to(DEVICE, non_blocking=True)
            base_logit = base_logit.to(DEVICE, non_blocking=True)
            gate = gate.to(DEVICE, non_blocking=True)
            delta_teacher = delta_teacher.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                delta_raw = model(x6)
                (
                    loss,
                    true_loss,
                    true_bce,
                    true_dice,
                    res,
                    mag,
                    rank,
                    _,
                    _,
                ) = combined_loss(
                    delta_raw=delta_raw,
                    base_logit=base_logit,
                    gate=gate,
                    y_true=y_true,
                    delta_teacher=delta_teacher,
                    pos_weight_tensor=pos_weight_tensor,
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            totals["loss"] += float(loss.detach().cpu())
            totals["true"] += float(true_loss.detach().cpu())
            totals["bce"] += float(true_bce.detach().cpu())
            totals["dice"] += float(true_dice.detach().cpu())
            totals["res"] += float(res.detach().cpu())
            totals["mag"] += float(mag.detach().cpu())
            totals["rank"] += float(rank.detach().cpu())
            n_batches += 1

            pbar.set_postfix({
                "loss": totals["loss"] / max(n_batches, 1),
                "true": totals["true"] / max(n_batches, 1),
                "res": totals["res"] / max(n_batches, 1),
                "rank": totals["rank"] / max(n_batches, 1),
            })

        scheduler.step()

        row = {
            "fold": fold,
            "epoch": epoch,
            "loss": totals["loss"] / max(n_batches, 1),
            "true_loss": totals["true"] / max(n_batches, 1),
            "true_bce": totals["bce"] / max(n_batches, 1),
            "true_dice_loss": totals["dice"] / max(n_batches, 1),
            "pcc_residual_loss": totals["res"] / max(n_batches, 1),
            "magnitude_penalty": totals["mag"] / max(n_batches, 1),
            "ranking_loss": totals["rank"] / max(n_batches, 1),
            "lr": scheduler.get_last_lr()[0],
            "pos_weight": pos_weight,
        }

        loss_rows.append(row)

        print(
            f"C2-v3 Fold {fold} Epoch {epoch:02d}: "
            f"loss={row['loss']:.5f}, "
            f"true={row['true_loss']:.5f}, "
            f"res={row['pcc_residual_loss']:.5f}, "
            f"rank={row['ranking_loss']:.5f}, "
            f"dice_loss={row['true_dice_loss']:.5f}"
        )

    ckpt_path = CKPT_DIR / f"c2_v3_geometry_aware_fold_{fold}.pt"
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "train_ids": train_ids,
        "test_ids": test_ids,
        "config": {
            "method": "C2-v3 geometry-aware quick",
            "epochs": EPOCHS,
            "geom_main_sigma": GEOM_MAIN_SIGMA,
            "geom_main_beta": GEOM_MAIN_BETA,
            "geom_inside_weight": GEOM_INSIDE_WEIGHT,
            "geom_aux_sigma": GEOM_AUX_SIGMA,
            "max_delta_logit": MAX_DELTA_LOGIT,
            "alphas": ALPHAS,
        }
    }, ckpt_path)

    print("Saved checkpoint:", ckpt_path)

    # Evaluate test cases
    model.eval()

    metric_rows = []
    delta_rows = []

    with torch.no_grad():
        for cid in tqdm(test_ids, desc=f"C2-v3 Fold {fold} prediction"):
            feats = make_geometry_features(cid)

            X = feats["X"]
            Y = feats["Y"]
            B = feats["B"]
            ring_main = feats["ring_main"]
            ring_aux = feats["ring_aux"]
            G = feats["geometry_base"]
            gate = feats["gate"]

            target = Y[:, 0].astype(bool)
            G_logit = prob_to_logit_np(G)

            delta_parts = []

            for start in range(0, X.shape[0], BATCH_SIZE):
                x_part = X[start:start+BATCH_SIZE]
                b_part = B[start:start+BATCH_SIZE]
                ring_main_part = ring_main[start:start+BATCH_SIZE]
                ring_aux_part = ring_aux[start:start+BATCH_SIZE]
                g_part = G[start:start+BATCH_SIZE]
                gate_part = gate[start:start+BATCH_SIZE]

                x6 = np.concatenate([
                    x_part,
                    b_part[:, None, :, :],
                    ring_main_part[:, None, :, :],
                    ring_aux_part[:, None, :, :],
                    g_part[:, None, :, :],
                ], axis=1).astype(np.float32)

                x6_t = torch.from_numpy(x6).float().to(DEVICE)
                gate_t = torch.from_numpy(gate_part[:, None, :, :]).float().to(DEVICE)

                delta_raw = model(x6_t)
                delta_pred = MAX_DELTA_LOGIT * torch.tanh(delta_raw)
                delta_gated = delta_pred * gate_t

                delta_parts.append(delta_gated.detach().cpu().numpy()[:, 0].astype(np.float32))

            delta_map = np.concatenate(delta_parts, axis=0)
            delta_map = np.clip(delta_map, -MAX_DELTA_LOGIT, MAX_DELTA_LOGIT).astype(np.float32)

            dpath = DELTA_DIR / f"{cid}_c2_v3_delta_logit_fold_{fold}.npy"
            np.save(dpath, delta_map.astype(np.float16))

            mA = topk_metrics_fast(B, target)
            mG = topk_metrics_fast(G, target)

            for alpha in ALPHAS:
                final_logit = G_logit + alpha * delta_map
                final_prob = sigmoid_np(final_logit)
                final_prob = normalize_map(final_prob)

                m = topk_metrics_fast(final_prob, target)

                metric_rows.append({
                    "case_id": cid,
                    "fold": fold,
                    "alpha": alpha,
                    "method": f"C2-v3 geometry-aware alpha={alpha}",
                    "test_time_future_target_access": "No",
                    "input_uses_model_A_oof": "Yes",
                    "input_uses_ring_prior": "Yes",
                    "base_map": "geometry_base",

                    "model_A_dice": mA["dice"],
                    "model_A_iou": mA["iou"],
                    "model_A_target_focus": mA["target_focus"],
                    "model_A_log10_ratio": mA["log10_ratio"],

                    "geometry_base_dice": mG["dice"],
                    "geometry_base_iou": mG["iou"],
                    "geometry_base_target_focus": mG["target_focus"],
                    "geometry_base_log10_ratio": mG["log10_ratio"],

                    "dice": m["dice"],
                    "iou": m["iou"],
                    "target_focus": m["target_focus"],
                    "log10_ratio": m["log10_ratio"],

                    "delta_path": str(dpath),
                })

            delta_rows.append({
                "case_id": cid,
                "fold": fold,
                "delta_abs_mean": float(np.mean(np.abs(delta_map))),
                "delta_abs_max": float(np.max(np.abs(delta_map))),
                "delta_positive_fraction": float((delta_map > 0).mean()),
                "delta_negative_fraction": float((delta_map < 0).mean()),
                "delta_path": str(dpath),
            })

    return loss_rows, metric_rows, delta_rows


# =====================
# Run all folds
# =====================
start_time = time.time()

all_loss_rows = []
all_metric_rows = []
all_delta_rows = []

for fold, train_ids, test_ids in fold_splits:
    loss_rows, metric_rows, delta_rows = train_one_fold(fold, train_ids, test_ids)

    all_loss_rows.extend(loss_rows)
    all_metric_rows.extend(metric_rows)
    all_delta_rows.extend(delta_rows)

    pd.DataFrame(all_loss_rows).to_csv(C2_DIR / "c2_v3_training_losses_partial.csv", index=False)
    pd.DataFrame(all_metric_rows).to_csv(C2_DIR / "c2_v3_alpha_case_metrics_partial.csv", index=False)
    pd.DataFrame(all_delta_rows).to_csv(C2_DIR / "c2_v3_delta_diagnostics_partial.csv", index=False)


loss_df = pd.DataFrame(all_loss_rows)
metrics_df = pd.DataFrame(all_metric_rows)
delta_df = pd.DataFrame(all_delta_rows)

loss_path = C2_DIR / "c2_v3_training_losses.csv"
metrics_path = C2_DIR / "c2_v3_alpha_case_metrics.csv"
delta_path = C2_DIR / "c2_v3_delta_diagnostics.csv"

loss_df.to_csv(loss_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
delta_df.to_csv(delta_path, index=False)


# =====================
# Summaries
# =====================
alpha_summary = metrics_df.groupby("alpha")[["dice", "iou", "target_focus", "log10_ratio"]].agg(["mean", "median", "min", "max"])
alpha_summary_path = C2_DIR / "c2_v3_alpha_summary.csv"
alpha_summary.to_csv(alpha_summary_path)

pairwise_rows = []

for alpha in ALPHAS:
    sub = metrics_df[metrics_df["alpha"] == alpha].copy()

    for baseline_name, prefix in [
        ("Model A", "model_A"),
        ("Geometry base", "geometry_base"),
    ]:
        for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
            base_col = f"{prefix}_{metric}"
            diff = sub[metric] - sub[base_col]

            pairwise_rows.append({
                "alpha": alpha,
                "baseline": baseline_name,
                "metric": metric,
                "mean_diff": float(diff.mean()),
                "median_diff": float(diff.median()),
                "wins": int((diff > 0).sum()),
                "losses": int((diff < 0).sum()),
                "ties": int((diff == 0).sum()),
                "total": int(diff.notna().sum()),
                "win_rate": float((diff > 0).mean()),
            })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = C2_DIR / "pairwise_C2_v3_alpha_vs_baselines.csv"
pairwise_df.to_csv(pairwise_path, index=False)


# Compact comparison
modelA_mean = modelA_df[["dice", "iou", "target_focus", "log10_ratio"]].mean()

# alpha=0 row is geometry base under this pipeline
geom_base = metrics_df[metrics_df["alpha"] == 0.0][[
    "geometry_base_dice",
    "geometry_base_iou",
    "geometry_base_target_focus",
    "geometry_base_log10_ratio"
]].mean()

comparison_rows = [
    {
        "Model": "Model A",
        "alpha": np.nan,
        "Mechanism": "direct future-change prediction",
        "Dice_mean": float(modelA_mean["dice"]),
        "IoU_mean": float(modelA_mean["iou"]),
        "Target_focus_mean": float(modelA_mean["target_focus"]),
        "Log10_ratio_mean": float(modelA_mean["log10_ratio"]),
    },
    {
        "Model": "Geometry base",
        "alpha": 0.0,
        "Mechanism": "Model A OOF + ring prior sigma=8 beta=2",
        "Dice_mean": float(geom_base["geometry_base_dice"]),
        "IoU_mean": float(geom_base["geometry_base_iou"]),
        "Target_focus_mean": float(geom_base["geometry_base_target_focus"]),
        "Log10_ratio_mean": float(geom_base["geometry_base_log10_ratio"]),
    }
]

for alpha in ALPHAS:
    sub = metrics_df[metrics_df["alpha"] == alpha]
    means = sub[["dice", "iou", "target_focus", "log10_ratio"]].mean()

    comparison_rows.append({
        "Model": "C2-v3",
        "alpha": alpha,
        "Mechanism": "geometry-aware PCC residual",
        "Dice_mean": float(means["dice"]),
        "IoU_mean": float(means["iou"]),
        "Target_focus_mean": float(means["target_focus"]),
        "Log10_ratio_mean": float(means["log10_ratio"]),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = C2_DIR / "model_A_geometry_C2_v3_compact_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

best_nonzero = comparison_df[
    (comparison_df["Model"] == "C2-v3") & (comparison_df["alpha"] > 0)
].sort_values("Dice_mean", ascending=False).head(1)

runtime_min = (time.time() - start_time) / 60.0

run_info = {
    "method": "C2-v3 geometry-aware quick",
    "runtime_minutes": runtime_min,
    "epochs": EPOCHS,
    "base": "geometry_base = suppressed Model A OOF + beta * ring sigma=8",
    "test_time_future_target_access": "No",
    "outputs": {
        "teacher_quality": str(teacher_path_csv),
        "teacher_summary": str(teacher_summary_path),
        "losses": str(loss_path),
        "metrics": str(metrics_path),
        "alpha_summary": str(alpha_summary_path),
        "pairwise": str(pairwise_path),
        "comparison": str(comparison_path),
        "delta": str(delta_path),
    }
}

with open(C2_DIR / "c2_v3_run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)


print("\n" + "=" * 80)
print("C2-v3 geometry-aware quick finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved:", C2_DIR)

print("\nTeacher quality summary:")
display(teacher_summary)

print("\nAlpha summary:")
display(alpha_summary)

print("\nPairwise C2-v3 alpha vs baselines:")
display(pairwise_df)

print("\nCompact comparison:")
display(comparison_df)

print("\nBest nonzero alpha by Dice:")
display(best_nonzero)

print("\nTraining losses tail:")
display(loss_df.tail(10))

print("\nDelta diagnostics:")
display(delta_df.describe(include="all"))

In [ ]:
# ============================================================
# V4-clean Quick: Fold-clean Direct PCC-Teacher Distillation
#
# Goal:
#   Make PCC have a stronger effect on model learning without cheating.
#
# Clean rule:
#   For each fold:
#       - PCC teacher is generated ONLY for train cases.
#       - Test cases do NOT have PCC teacher generated/loaded.
#       - Test-time input uses ONLY current-time information:
#           current T1c, current tumour mask, Model A OOF prediction,
#           ring priors, geometry base.
#       - Future target is used ONLY for final evaluation on test cases.
#
# Difference from C2-v3:
#   C2-v3 learned a small residual over geometry base.
#   V4-clean directly learns the final future-change probability map,
#   supervised by both true target and train-only PCC teacher soft map.
# ============================================================

import os
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from scipy.ndimage import distance_transform_edt, gaussian_filter
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import distance_transform_edt, gaussian_filter


# =====================
# Config
# =====================
SEED = 42

# Quick diagnostic first
EPOCHS = 5

BATCH_SIZE = 12
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

POS_SLICE_WEIGHT = 3.0
NEG_SLICE_WEIGHT = 1.0

# Geometry base from previous diagnostic
GEOM_MAIN_SIGMA = 8
GEOM_MAIN_BETA = 2.0
GEOM_INSIDE_WEIGHT = 0.10
GEOM_AUX_SIGMA = 4

# PCC teacher construction
PCC_ROUNDS = 10
PCC_ETA_POS = 0.30
PCC_ETA_NEG = 0.10
PCC_DILATION_RADIUS = 26
PCC_SIGMA = 2.0

# Direct distillation losses
HARD_TARGET_LOSS_WEIGHT = 0.50
PCC_TEACHER_LOSS_WEIGHT = 0.38
RANKING_LOSS_WEIGHT = 0.08
GEOMETRY_CONSISTENCY_WEIGHT = 0.04

# Evaluation blend:
# alpha = 0.0 means pure geometry base
# alpha = 1.0 means pure V4 student output
BLEND_ALPHAS = [0.0, 0.10, 0.25, 0.50, 0.75, 1.00]

LOGIT_EPS = 1e-4
FORCE_REGENERATE_FOLD_TEACHERS = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
OUT_DIR = Path("/kaggle/working/pcc_independent_baseline")
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
MODEL_A_METRICS_PATH = MODEL_A_DIR / "direct_target_case_metrics.csv"
MODEL_A_PRED_DIR = MODEL_A_DIR / "direct_target_pred_maps"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"

V4_DIR = STAGE2_DIR / "model_V4_clean_direct_pcc_teacher_distillation_quick_5epoch"
TEACHER_ROOT = V4_DIR / "fold_clean_train_only_pcc_teachers"
PRED_DIR = V4_DIR / "v4_student_pred_maps"
CKPT_DIR = V4_DIR / "checkpoints"

V4_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_ROOT.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert PRE_SUMMARY_PATH.exists(), f"Missing preprocessing summary: {PRE_SUMMARY_PATH}"
assert MODEL_A_METRICS_PATH.exists(), f"Missing Model A metrics: {MODEL_A_METRICS_PATH}"
assert MODEL_A_PRED_DIR.exists(), f"Missing Model A pred folder: {MODEL_A_PRED_DIR}"
assert SPLIT_PATH.exists(), f"Missing split file: {SPLIT_PATH}"

pre_df = pd.read_csv(PRE_SUMMARY_PATH)
modelA_df = pd.read_csv(MODEL_A_METRICS_PATH)
split_df = pd.read_csv(SPLIT_PATH)

case_to_npz = dict(zip(pre_df["case_id"], pre_df["npz_path"]))
case_ids = sorted(case_to_npz.keys())

print("Cases:", len(case_ids))
print("V4-clean output:", V4_DIR)


# =====================
# Utilities
# =====================
def normalize_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    mn, mx = float(x.min()), float(x.max())
    if mx <= mn:
        return np.zeros_like(x, dtype=np.float32)

    x = (x - mn) / (mx - mn + 1e-8)
    return np.clip(x, 0, 1).astype(np.float32)


def prob_to_logit_np(p, eps=LOGIT_EPS):
    p = np.clip(p.astype(np.float32), eps, 1.0 - eps)
    return np.log(p / (1.0 - p)).astype(np.float32)


def sigmoid_np(x):
    x = np.clip(x, -30, 30)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)


def topk_metrics_fast(pred, target, eps=1e-8):
    pred = normalize_map(pred)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    n = fp.size
    if k >= n:
        top_idx = np.arange(n)
    else:
        top_idx = np.argpartition(fp, n - k)[n - k:]

    pb = np.zeros(n, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)

    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def load_case(case_id):
    with np.load(case_to_npz[case_id], allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"].astype(np.float32)
    return X, Y


def load_modelA_oof(case_id):
    row = modelA_df[modelA_df["case_id"] == case_id]

    if len(row) == 1:
        p = Path(row.iloc[0]["pred_path"])
        if p.exists():
            return normalize_map(np.load(p).astype(np.float32))

    candidates = sorted(MODEL_A_PRED_DIR.glob(f"{case_id}_direct_target_student_fold_*.npy"))
    if len(candidates) != 1:
        raise FileNotFoundError(f"Cannot find Model A OOF map for {case_id}, found {len(candidates)}")

    return normalize_map(np.load(candidates[0]).astype(np.float32))


def outside_distance(current_mask):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    dist = np.zeros((Z, H, W), dtype=np.float32)

    for z in range(Z):
        m = current_mask[z]
        if m.sum() == 0:
            continue

        d = distance_transform_edt(~m).astype(np.float32)
        d[m] = 0.0
        dist[z] = d

    return dist


def make_ring_from_dist(dist, current_mask, sigma):
    current_mask = current_mask.astype(bool)

    ring = np.exp(-(dist ** 2) / (2.0 * sigma * sigma)).astype(np.float32)
    ring[current_mask] = 0.0

    has_mask = current_mask.reshape(current_mask.shape[0], -1).sum(axis=1) > 0
    for z, ok in enumerate(has_mask):
        if not ok:
            ring[z] = 0.0

    return normalize_map(ring)


def suppress_inside(B, current_mask, inside_weight):
    out = B.copy().astype(np.float32)
    out[current_mask.astype(bool)] *= float(inside_weight)
    return normalize_map(out)


def make_gate_from_dist(dist, current_mask, radius):
    current_mask = current_mask.astype(bool)

    gate = (dist <= radius).astype(np.float32)

    has_mask = current_mask.reshape(current_mask.shape[0], -1).sum(axis=1) > 0
    for z, ok in enumerate(has_mask):
        if not ok:
            gate[z] = 0.0

    return gate.astype(np.float32)


def make_geometry_features(case_id):
    X, Y = load_case(case_id)

    current_mask = X[:, 1].astype(bool)
    B = load_modelA_oof(case_id)

    dist = outside_distance(current_mask)

    ring_main = make_ring_from_dist(dist, current_mask, GEOM_MAIN_SIGMA)
    ring_aux = make_ring_from_dist(dist, current_mask, GEOM_AUX_SIGMA)

    B_supp = suppress_inside(B, current_mask, GEOM_INSIDE_WEIGHT)
    geometry_base = normalize_map(B_supp + GEOM_MAIN_BETA * ring_main)

    gate = make_gate_from_dist(dist, current_mask, PCC_DILATION_RADIUS)

    return {
        "X": X,
        "Y": Y,
        "B": B,
        "ring_main": ring_main,
        "ring_aux": ring_aux,
        "geometry_base": geometry_base,
        "gate": gate,
        "current_mask": current_mask,
    }


def smooth_target_2d(target, sigma):
    target = target.astype(np.float32)
    S = gaussian_filter(target, sigma=(0, sigma, sigma))

    if S.max() > 0:
        S = S / (S.max() + 1e-8)

    return np.clip(S, 0, 1).astype(np.float32)


def run_pcc_v2_on_base_map(B, current_mask, target):
    B = normalize_map(B)
    current_mask = current_mask.astype(bool)
    target = target.astype(bool)

    dist = outside_distance(current_mask)
    R = make_gate_from_dist(dist, current_mask, PCC_DILATION_RADIUS).astype(bool)
    S = smooth_target_2d(target, PCC_SIGMA)

    P = B.copy().astype(np.float32)

    for _ in range(PCC_ROUNDS):
        P_new = P.copy()

        positive_signal = S * (1.0 - P)
        negative_signal = (1.0 - S) * P
        negative_signal[target] = 0.0

        P_new[R] = (
            P[R]
            + PCC_ETA_POS * positive_signal[R]
            - PCC_ETA_NEG * negative_signal[R]
        )

        P_new[~R] = B[~R]
        P = np.clip(P_new, 0, 1).astype(np.float32)

    return P.astype(np.float32)


def fold_teacher_dir(fold):
    return TEACHER_ROOT / f"fold_{fold}_train_only"


def teacher_path(fold, case_id):
    return fold_teacher_dir(fold) / f"{case_id}_train_only_pcc_teacher.npy"


def generate_train_teacher_for_fold(fold, case_id):
    """
    This function is called ONLY for training cases in a fold.
    It must never be called for test cases.
    """
    tdir = fold_teacher_dir(fold)
    tdir.mkdir(parents=True, exist_ok=True)

    p = teacher_path(fold, case_id)
    if p.exists() and not FORCE_REGENERATE_FOLD_TEACHERS:
        return normalize_map(np.load(p).astype(np.float32))

    feats = make_geometry_features(case_id)
    target = feats["Y"][:, 0].astype(bool)

    P_teacher = run_pcc_v2_on_base_map(
        B=feats["geometry_base"],
        current_mask=feats["current_mask"],
        target=target,
    )

    np.save(p, P_teacher.astype(np.float16))
    return P_teacher


def compute_train_pos_weight(train_case_ids):
    pos = 0.0
    total = 0.0

    for cid in train_case_ids:
        _, Y = load_case(cid)
        pos += float(Y.sum())
        total += float(Y.size)

    neg = total - pos
    pos_weight = neg / (pos + 1e-8)
    pos_weight = float(np.clip(pos_weight, 1.0, 50.0))

    return pos_weight, pos, total


# =====================
# Dataset
# =====================
class V4CleanDataset(Dataset):
    def __init__(self, fold, case_ids):
        self.fold = int(fold)
        self.case_ids = list(case_ids)
        self.case_data = {}
        self.index = []
        self.slice_has_target = []

        for cid in self.case_ids:
            feats = make_geometry_features(cid)

            X = feats["X"]
            Y = feats["Y"]
            B = feats["B"]
            ring_main = feats["ring_main"]
            ring_aux = feats["ring_aux"]
            G = feats["geometry_base"]

            # Teacher must already be generated for train cases only.
            tp = teacher_path(self.fold, cid)
            if not tp.exists():
                raise FileNotFoundError(f"Missing train-only teacher for {cid}: {tp}")

            T = normalize_map(np.load(tp).astype(np.float32))

            # Store compactly to reduce memory
            self.case_data[cid] = {
                "X": X.astype(np.float16),
                "Y": Y.astype(np.uint8),
                "B": B.astype(np.float16),
                "ring_main": ring_main.astype(np.float16),
                "ring_aux": ring_aux.astype(np.float16),
                "G": G.astype(np.float16),
                "T": T.astype(np.float16),
            }

            Z = X.shape[0]
            for z in range(Z):
                self.index.append((cid, z))
                self.slice_has_target.append(float(Y[z].sum() > 0))

        self.slice_has_target = np.asarray(self.slice_has_target, dtype=np.float32)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        cid, z = self.index[idx]
        d = self.case_data[cid]

        X = d["X"][z].astype(np.float32)
        Y = d["Y"][z].astype(np.float32)
        B = d["B"][z].astype(np.float32)
        ring_main = d["ring_main"][z].astype(np.float32)
        ring_aux = d["ring_aux"][z].astype(np.float32)
        G = d["G"][z].astype(np.float32)
        T = d["T"][z].astype(np.float32)

        # 6-channel input:
        # current T1c, current mask, Model A OOF, ring8, ring4, geometry base
        x6 = np.concatenate([
            X,                              # 2 channels
            B[None, :, :],                  # 1
            ring_main[None, :, :],          # 1
            ring_aux[None, :, :],           # 1
            G[None, :, :],                  # 1
        ], axis=0).astype(np.float32)

        return (
            torch.from_numpy(x6).float(),
            torch.from_numpy(Y).float(),
            torch.from_numpy(T[None, :, :]).float(),
            torch.from_numpy(G[None, :, :]).float(),
            cid,
            z,
        )


def make_sampler(ds):
    weights = np.where(
        ds.slice_has_target > 0,
        POS_SLICE_WEIGHT,
        NEG_SLICE_WEIGHT
    ).astype(np.float32)

    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True,
    )


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2DDirect(nn.Module):
    def __init__(self, in_ch=6, out_ch=1, base=24):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


# =====================
# Losses
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)

    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def soft_dice_loss_from_probs(probs, targets, eps=1e-6):
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)

    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def weighted_hard_target_loss(logits, y_true, teacher_prob, geom_prob, pos_weight_tensor):
    # More weight on true positive areas and PCC teacher-supported areas.
    with torch.no_grad():
        w = 1.0 + 2.0 * y_true + 1.0 * torch.clamp(teacher_prob, 0, 1) + 0.5 * torch.clamp(geom_prob, 0, 1)

    bce_raw = F.binary_cross_entropy_with_logits(
        logits,
        y_true,
        pos_weight=pos_weight_tensor,
        reduction="none",
    )

    bce = torch.sum(bce_raw * w) / (torch.sum(w) + 1e-6)
    dice = soft_dice_loss_from_logits(logits, y_true)

    return 0.5 * bce + 0.5 * dice, bce.detach(), dice.detach()


def pcc_teacher_distillation_loss(logits, teacher_prob):
    # Soft-label BCE + soft Dice to directly imitate PCC teacher map.
    bce = F.binary_cross_entropy_with_logits(logits, teacher_prob)
    probs = torch.sigmoid(logits)
    dice = soft_dice_loss_from_probs(probs, teacher_prob)

    return 0.5 * bce + 0.5 * dice, bce.detach(), dice.detach()


def geometry_consistency_loss(logits, geom_prob):
    # Small regularizer: do not let direct student destroy the geometry base too aggressively.
    probs = torch.sigmoid(logits)
    return F.smooth_l1_loss(probs, geom_prob)


def sampled_ranking_loss(logits, targets, max_pairs=512, hard_neg_pool=4096, margin=0.20):
    """
    Encourages true future-change pixels to rank above hard non-target pixels.
    Supports top-k Dice evaluation.
    """
    B = logits.shape[0]
    losses = []

    for i in range(B):
        li = logits[i, 0].reshape(-1)
        ti = targets[i, 0].reshape(-1) > 0.5

        pos = li[ti]
        neg = li[~ti]

        if pos.numel() == 0 or neg.numel() == 0:
            continue

        n = min(max_pairs, pos.numel(), neg.numel())

        pos_idx = torch.randint(0, pos.numel(), (n,), device=logits.device)
        pos_s = pos[pos_idx]

        pool = min(hard_neg_pool, neg.numel())
        hard_neg = torch.topk(neg, k=pool).values
        neg_idx = torch.randint(0, pool, (n,), device=logits.device)
        neg_s = hard_neg[neg_idx]

        losses.append(F.softplus(margin - (pos_s - neg_s)).mean())

    if len(losses) == 0:
        return logits.sum() * 0.0

    return torch.stack(losses).mean()


def combined_v4_loss(logits, y_true, teacher_prob, geom_prob, pos_weight_tensor):
    hard_loss, hard_bce, hard_dice = weighted_hard_target_loss(
        logits=logits,
        y_true=y_true,
        teacher_prob=teacher_prob,
        geom_prob=geom_prob,
        pos_weight_tensor=pos_weight_tensor,
    )

    teacher_loss, teacher_bce, teacher_dice = pcc_teacher_distillation_loss(
        logits=logits,
        teacher_prob=teacher_prob,
    )

    rank = sampled_ranking_loss(logits, y_true)
    geom = geometry_consistency_loss(logits, geom_prob)

    total = (
        HARD_TARGET_LOSS_WEIGHT * hard_loss
        + PCC_TEACHER_LOSS_WEIGHT * teacher_loss
        + RANKING_LOSS_WEIGHT * rank
        + GEOMETRY_CONSISTENCY_WEIGHT * geom
    )

    return (
        total,
        hard_loss.detach(),
        hard_bce,
        hard_dice,
        teacher_loss.detach(),
        teacher_bce,
        teacher_dice,
        rank.detach(),
        geom.detach(),
    )


# =====================
# Fold splits
# =====================
fold_splits = []

for fold in sorted(split_df["fold"].unique()):
    train_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "train")]["case_id"].tolist()
    test_ids = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")]["case_id"].tolist()
    fold_splits.append((int(fold), train_ids, test_ids))

print("\nFold test cases:")
for fold, _, test_ids in fold_splits:
    print(f"\nFold {fold}:")
    for cid in test_ids:
        print(" ", cid)


# =====================
# Train one fold
# =====================
def train_one_fold(fold, train_ids, test_ids):
    print("\n" + "=" * 90)
    print(f"V4-clean Direct PCC-Teacher Distillation / Fold {fold}")
    print("Train cases:", len(train_ids))
    print("Test cases:", len(test_ids))

    # Strict clean teacher folder for this fold.
    # Remove any stale files first, then generate teachers only for train cases.
    ftdir = fold_teacher_dir(fold)
    if ftdir.exists() and FORCE_REGENERATE_FOLD_TEACHERS:
        shutil.rmtree(ftdir)
    ftdir.mkdir(parents=True, exist_ok=True)

    print("\nGenerating PCC teachers for TRAIN cases only...")
    teacher_quality_rows = []

    for cid in tqdm(train_ids, desc=f"Fold {fold} train-only PCC teachers"):
        P_teacher = generate_train_teacher_for_fold(fold, cid)

        feats = make_geometry_features(cid)
        target = feats["Y"][:, 0].astype(bool)
        B = feats["B"]
        G = feats["geometry_base"]

        mA = topk_metrics_fast(B, target)
        mG = topk_metrics_fast(G, target)
        mT = topk_metrics_fast(P_teacher, target)

        teacher_quality_rows.append({
            "fold": fold,
            "case_id": cid,
            "split": "train",
            "model_A_dice": mA["dice"],
            "model_A_iou": mA["iou"],
            "geometry_base_dice": mG["dice"],
            "geometry_base_iou": mG["iou"],
            "pcc_teacher_dice": mT["dice"],
            "pcc_teacher_iou": mT["iou"],
            "pcc_teacher_target_focus": mT["target_focus"],
            "pcc_teacher_log10_ratio": mT["log10_ratio"],
            "teacher_path": str(teacher_path(fold, cid)),
        })

    # Anti-leakage check: there must be no teacher files for test cases in this fold.
    leaked_test_teachers = [cid for cid in test_ids if teacher_path(fold, cid).exists()]
    assert len(leaked_test_teachers) == 0, f"Leakage risk: teacher files exist for test cases: {leaked_test_teachers}"

    print("Clean check passed: no PCC teacher files for fold test cases.")

    pos_weight, true_pos, total_vox = compute_train_pos_weight(train_ids)
    print(f"True target positive voxels: {true_pos:.0f} / {total_vox:.0f}")
    print(f"BCE pos_weight clipped: {pos_weight:.4f}")

    train_ds = V4CleanDataset(fold=fold, case_ids=train_ids)
    sampler = make_sampler(train_ds)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model = SmallUNet2DDirect(in_ch=6, out_ch=1, base=24).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE).view(1, 1, 1, 1)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    loss_rows = []

    for epoch in range(1, EPOCHS + 1):
        model.train()

        totals = {
            "loss": 0.0,
            "hard": 0.0,
            "hard_bce": 0.0,
            "hard_dice": 0.0,
            "teacher": 0.0,
            "teacher_bce": 0.0,
            "teacher_dice": 0.0,
            "rank": 0.0,
            "geom": 0.0,
        }

        n_batches = 0

        pbar = tqdm(train_loader, desc=f"V4 Fold {fold} Epoch {epoch}/{EPOCHS}", leave=False)

        for x6, y_true, teacher_prob, geom_prob, _, _ in pbar:
            x6 = x6.to(DEVICE, non_blocking=True)
            y_true = y_true.to(DEVICE, non_blocking=True)
            teacher_prob = teacher_prob.to(DEVICE, non_blocking=True)
            geom_prob = geom_prob.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(x6)

                (
                    loss,
                    hard_loss,
                    hard_bce,
                    hard_dice,
                    teacher_loss,
                    teacher_bce,
                    teacher_dice,
                    rank,
                    geom,
                ) = combined_v4_loss(
                    logits=logits,
                    y_true=y_true,
                    teacher_prob=teacher_prob,
                    geom_prob=geom_prob,
                    pos_weight_tensor=pos_weight_tensor,
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            totals["loss"] += float(loss.detach().cpu())
            totals["hard"] += float(hard_loss.detach().cpu())
            totals["hard_bce"] += float(hard_bce.detach().cpu())
            totals["hard_dice"] += float(hard_dice.detach().cpu())
            totals["teacher"] += float(teacher_loss.detach().cpu())
            totals["teacher_bce"] += float(teacher_bce.detach().cpu())
            totals["teacher_dice"] += float(teacher_dice.detach().cpu())
            totals["rank"] += float(rank.detach().cpu())
            totals["geom"] += float(geom.detach().cpu())

            n_batches += 1

            pbar.set_postfix({
                "loss": totals["loss"] / max(n_batches, 1),
                "hard": totals["hard"] / max(n_batches, 1),
                "teacher": totals["teacher"] / max(n_batches, 1),
                "rank": totals["rank"] / max(n_batches, 1),
            })

        scheduler.step()

        row = {
            "fold": fold,
            "epoch": epoch,
            "loss": totals["loss"] / max(n_batches, 1),
            "hard_target_loss": totals["hard"] / max(n_batches, 1),
            "hard_bce": totals["hard_bce"] / max(n_batches, 1),
            "hard_dice_loss": totals["hard_dice"] / max(n_batches, 1),
            "pcc_teacher_loss": totals["teacher"] / max(n_batches, 1),
            "teacher_bce": totals["teacher_bce"] / max(n_batches, 1),
            "teacher_dice_loss": totals["teacher_dice"] / max(n_batches, 1),
            "ranking_loss": totals["rank"] / max(n_batches, 1),
            "geometry_consistency_loss": totals["geom"] / max(n_batches, 1),
            "lr": scheduler.get_last_lr()[0],
            "pos_weight": pos_weight,
        }

        loss_rows.append(row)

        print(
            f"V4 Fold {fold} Epoch {epoch:02d}: "
            f"loss={row['loss']:.5f}, "
            f"hard={row['hard_target_loss']:.5f}, "
            f"teacher={row['pcc_teacher_loss']:.5f}, "
            f"rank={row['ranking_loss']:.5f}, "
            f"geom={row['geometry_consistency_loss']:.5f}"
        )

    ckpt_path = CKPT_DIR / f"v4_clean_direct_pcc_distillation_fold_{fold}.pt"
    torch.save({
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "train_ids": train_ids,
        "test_ids": test_ids,
        "config": {
            "method": "V4-clean direct PCC-teacher distillation",
            "epochs": EPOCHS,
            "clean_rule": "PCC teachers generated only for train cases in each fold",
            "test_time_future_access": "No",
            "input_channels": [
                "current_t1c",
                "current_tumour_mask",
                "model_A_oof_prediction",
                "ring_sigma8",
                "ring_sigma4",
                "geometry_base",
            ],
            "blend_alphas": BLEND_ALPHAS,
        }
    }, ckpt_path)

    print("Saved checkpoint:", ckpt_path)

    # Evaluate test cases.
    # No teacher generation, no teacher loading, no future input.
    model.eval()

    metric_rows = []
    pred_diag_rows = []

    with torch.no_grad():
        for cid in tqdm(test_ids, desc=f"V4 Fold {fold} prediction"):
            # Safety check again: do not allow test teacher to exist for this fold.
            assert not teacher_path(fold, cid).exists(), f"Leakage risk: test teacher exists for {cid}"

            feats = make_geometry_features(cid)

            X = feats["X"]
            Y = feats["Y"]
            B = feats["B"]
            ring_main = feats["ring_main"]
            ring_aux = feats["ring_aux"]
            G = feats["geometry_base"]

            target = Y[:, 0].astype(bool)

            pred_parts = []

            for start in range(0, X.shape[0], BATCH_SIZE):
                x_part = X[start:start+BATCH_SIZE].astype(np.float32)
                b_part = B[start:start+BATCH_SIZE].astype(np.float32)
                r8_part = ring_main[start:start+BATCH_SIZE].astype(np.float32)
                r4_part = ring_aux[start:start+BATCH_SIZE].astype(np.float32)
                g_part = G[start:start+BATCH_SIZE].astype(np.float32)

                x6 = np.concatenate([
                    x_part,
                    b_part[:, None, :, :],
                    r8_part[:, None, :, :],
                    r4_part[:, None, :, :],
                    g_part[:, None, :, :],
                ], axis=1).astype(np.float32)

                x6_t = torch.from_numpy(x6).float().to(DEVICE)
                logits = model(x6_t)
                probs = torch.sigmoid(logits)

                pred_parts.append(probs.detach().cpu().numpy()[:, 0].astype(np.float32))

            student_prob = np.concatenate(pred_parts, axis=0)
            student_prob = normalize_map(student_prob)

            pred_path = PRED_DIR / f"{cid}_v4_clean_student_fold_{fold}.npy"
            np.save(pred_path, student_prob.astype(np.float16))

            mA = topk_metrics_fast(B, target)
            mG = topk_metrics_fast(G, target)
            mS = topk_metrics_fast(student_prob, target)

            for alpha in BLEND_ALPHAS:
                blended = normalize_map((1.0 - alpha) * G + alpha * student_prob)
                m = topk_metrics_fast(blended, target)

                metric_rows.append({
                    "case_id": cid,
                    "fold": fold,
                    "alpha": alpha,
                    "method": f"V4-clean blend alpha={alpha}",
                    "test_time_future_target_access": "No",
                    "test_teacher_used": "No",
                    "pcc_teacher_generated_for_test_case": "No",
                    "input_uses_model_A_oof": "Yes",
                    "input_uses_ring_prior": "Yes",

                    "model_A_dice": mA["dice"],
                    "model_A_iou": mA["iou"],
                    "model_A_target_focus": mA["target_focus"],
                    "model_A_log10_ratio": mA["log10_ratio"],

                    "geometry_base_dice": mG["dice"],
                    "geometry_base_iou": mG["iou"],
                    "geometry_base_target_focus": mG["target_focus"],
                    "geometry_base_log10_ratio": mG["log10_ratio"],

                    "student_direct_dice": mS["dice"],
                    "student_direct_iou": mS["iou"],
                    "student_direct_target_focus": mS["target_focus"],
                    "student_direct_log10_ratio": mS["log10_ratio"],

                    "dice": m["dice"],
                    "iou": m["iou"],
                    "target_focus": m["target_focus"],
                    "log10_ratio": m["log10_ratio"],

                    "pred_path": str(pred_path),
                })

            pred_diag_rows.append({
                "case_id": cid,
                "fold": fold,
                "student_prob_mean": float(student_prob.mean()),
                "student_prob_max": float(student_prob.max()),
                "student_prob_min": float(student_prob.min()),
                "student_prob_sum": float(student_prob.sum()),
                "pred_path": str(pred_path),
            })

    return loss_rows, metric_rows, pred_diag_rows, teacher_quality_rows


# =====================
# Run all folds
# =====================
start_time = time.time()

all_loss_rows = []
all_metric_rows = []
all_pred_diag_rows = []
all_teacher_quality_rows = []

for fold, train_ids, test_ids in fold_splits:
    loss_rows, metric_rows, pred_diag_rows, teacher_quality_rows = train_one_fold(fold, train_ids, test_ids)

    all_loss_rows.extend(loss_rows)
    all_metric_rows.extend(metric_rows)
    all_pred_diag_rows.extend(pred_diag_rows)
    all_teacher_quality_rows.extend(teacher_quality_rows)

    pd.DataFrame(all_loss_rows).to_csv(V4_DIR / "v4_clean_training_losses_partial.csv", index=False)
    pd.DataFrame(all_metric_rows).to_csv(V4_DIR / "v4_clean_alpha_case_metrics_partial.csv", index=False)
    pd.DataFrame(all_pred_diag_rows).to_csv(V4_DIR / "v4_clean_prediction_diagnostics_partial.csv", index=False)
    pd.DataFrame(all_teacher_quality_rows).to_csv(V4_DIR / "v4_clean_train_only_teacher_quality_partial.csv", index=False)


loss_df = pd.DataFrame(all_loss_rows)
metrics_df = pd.DataFrame(all_metric_rows)
pred_diag_df = pd.DataFrame(all_pred_diag_rows)
teacher_quality_df = pd.DataFrame(all_teacher_quality_rows)

loss_path = V4_DIR / "v4_clean_training_losses.csv"
metrics_path = V4_DIR / "v4_clean_alpha_case_metrics.csv"
pred_diag_path = V4_DIR / "v4_clean_prediction_diagnostics.csv"
teacher_quality_path = V4_DIR / "v4_clean_train_only_teacher_quality.csv"

loss_df.to_csv(loss_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
pred_diag_df.to_csv(pred_diag_path, index=False)
teacher_quality_df.to_csv(teacher_quality_path, index=False)


# =====================
# Summaries
# =====================
alpha_summary = metrics_df.groupby("alpha")[["dice", "iou", "target_focus", "log10_ratio"]].agg(["mean", "median", "min", "max"])
alpha_summary_path = V4_DIR / "v4_clean_alpha_summary.csv"
alpha_summary.to_csv(alpha_summary_path)

pairwise_rows = []

for alpha in BLEND_ALPHAS:
    sub = metrics_df[metrics_df["alpha"] == alpha].copy()

    for baseline_name, prefix in [
        ("Model A", "model_A"),
        ("Geometry base", "geometry_base"),
    ]:
        for metric in ["dice", "iou", "target_focus", "log10_ratio"]:
            base_col = f"{prefix}_{metric}"
            diff = sub[metric] - sub[base_col]

            pairwise_rows.append({
                "alpha": alpha,
                "baseline": baseline_name,
                "metric": metric,
                "mean_diff": float(diff.mean()),
                "median_diff": float(diff.median()),
                "wins": int((diff > 0).sum()),
                "losses": int((diff < 0).sum()),
                "ties": int((diff == 0).sum()),
                "total": int(diff.notna().sum()),
                "win_rate": float((diff > 0).mean()),
            })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = V4_DIR / "pairwise_V4_clean_alpha_vs_baselines.csv"
pairwise_df.to_csv(pairwise_path, index=False)

modelA_mean = modelA_df[["dice", "iou", "target_focus", "log10_ratio"]].mean()

geom_base = metrics_df[metrics_df["alpha"] == 0.0][[
    "geometry_base_dice",
    "geometry_base_iou",
    "geometry_base_target_focus",
    "geometry_base_log10_ratio"
]].mean()

comparison_rows = [
    {
        "Model": "Model A",
        "alpha": np.nan,
        "Mechanism": "direct future-change prediction",
        "Dice_mean": float(modelA_mean["dice"]),
        "IoU_mean": float(modelA_mean["iou"]),
        "Target_focus_mean": float(modelA_mean["target_focus"]),
        "Log10_ratio_mean": float(modelA_mean["log10_ratio"]),
    },
    {
        "Model": "Geometry base",
        "alpha": 0.0,
        "Mechanism": "Model A OOF + ring prior sigma=8 beta=2",
        "Dice_mean": float(geom_base["geometry_base_dice"]),
        "IoU_mean": float(geom_base["geometry_base_iou"]),
        "Target_focus_mean": float(geom_base["geometry_base_target_focus"]),
        "Log10_ratio_mean": float(geom_base["geometry_base_log10_ratio"]),
    }
]

for alpha in BLEND_ALPHAS:
    sub = metrics_df[metrics_df["alpha"] == alpha]
    means = sub[["dice", "iou", "target_focus", "log10_ratio"]].mean()

    comparison_rows.append({
        "Model": "V4-clean",
        "alpha": alpha,
        "Mechanism": "fold-clean direct PCC-teacher distillation",
        "Dice_mean": float(means["dice"]),
        "IoU_mean": float(means["iou"]),
        "Target_focus_mean": float(means["target_focus"]),
        "Log10_ratio_mean": float(means["log10_ratio"]),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = V4_DIR / "model_A_geometry_V4_clean_compact_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

best_nonzero = comparison_df[
    (comparison_df["Model"] == "V4-clean") & (comparison_df["alpha"] > 0)
].sort_values("Dice_mean", ascending=False).head(1)

teacher_summary = pd.DataFrame([{
    "train_only_teacher_rows": len(teacher_quality_df),
    "train_only_teacher_model_A_dice_mean": teacher_quality_df["model_A_dice"].mean(),
    "train_only_teacher_geometry_base_dice_mean": teacher_quality_df["geometry_base_dice"].mean(),
    "train_only_teacher_pcc_teacher_dice_mean": teacher_quality_df["pcc_teacher_dice"].mean(),
    "train_only_teacher_pcc_teacher_iou_mean": teacher_quality_df["pcc_teacher_iou"].mean(),
}])

teacher_summary_path = V4_DIR / "v4_clean_train_only_teacher_quality_summary.csv"
teacher_summary.to_csv(teacher_summary_path, index=False)

runtime_min = (time.time() - start_time) / 60.0

no_leakage_protocol = {
    "method": "V4-clean fold-clean direct PCC-teacher distillation",
    "runtime_minutes": runtime_min,
    "epochs": EPOCHS,
    "teacher_generation_rule": "For each fold, PCC teachers are generated only for train cases.",
    "test_teacher_rule": "No PCC teacher is generated or loaded for test cases during evaluation.",
    "test_time_future_target_access": "No",
    "test_time_inputs": [
        "current T1c MRI",
        "current tumour mask",
        "Model A out-of-fold prediction",
        "ring prior sigma=8",
        "ring prior sigma=4",
        "geometry base",
    ],
    "future_target_usage": "Only train cases use future target for teacher generation and supervised training; test future target is used only for final metric evaluation.",
    "outputs": {
        "losses": str(loss_path),
        "metrics": str(metrics_path),
        "alpha_summary": str(alpha_summary_path),
        "pairwise": str(pairwise_path),
        "comparison": str(comparison_path),
        "teacher_quality": str(teacher_quality_path),
        "teacher_summary": str(teacher_summary_path),
        "prediction_diagnostics": str(pred_diag_path),
    }
}

with open(V4_DIR / "NO_LEAKAGE_PROTOCOL.json", "w") as f:
    json.dump(no_leakage_protocol, f, indent=2)


print("\n" + "=" * 90)
print("V4-clean quick finished.")
print(f"Runtime: {runtime_min:.2f} minutes")
print("Saved:", V4_DIR)

print("\nNO-LEAKAGE PROTOCOL:")
print(json.dumps(no_leakage_protocol, indent=2))

print("\nTrain-only PCC teacher quality summary:")
display(teacher_summary)

print("\nAlpha summary:")
display(alpha_summary)

print("\nPairwise V4-clean alpha vs baselines:")
display(pairwise_df)

print("\nCompact comparison:")
display(comparison_df)

print("\nBest nonzero alpha by Dice:")
display(best_nonzero)

print("\nTraining losses tail:")
display(loss_df.tail(10))

print("\nPrediction diagnostics:")
display(pred_diag_df.describe(include="all"))

In [ ]:
from pathlib import Path
import zipfile

WORKING = Path("/kaggle/working")

GEOM_DIR = WORKING / "pcc_independent_baseline/stage2_pcc_guided_student_learning/geometry_ring_prior_diagnostic_FAST_v2"
C2_DIR = WORKING / "pcc_independent_baseline/stage2_pcc_guided_student_learning/model_C2_v3_geometry_aware_quick_5epoch"
V4_DIR = WORKING / "pcc_independent_baseline/stage2_pcc_guided_student_learning/model_V4_clean_direct_pcc_teacher_distillation_quick_5epoch"

backup_zip = WORKING / "PCC_RESULTS_TABLES_ONLY_BACKUP.zip"

target_dirs = [GEOM_DIR, C2_DIR, V4_DIR]
include_suffixes = {".csv", ".json", ".txt"}

if backup_zip.exists():
    backup_zip.unlink()

with zipfile.ZipFile(backup_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for d in target_dirs:
        if not d.exists():
            print("Missing:", d)
            continue

        for p in d.rglob("*"):
            if p.is_file() and p.suffix.lower() in include_suffixes:
                zf.write(p, p.relative_to(WORKING))

print("Created:", backup_zip)
print("Size MB:", round(backup_zip.stat().st_size / 1024 / 1024, 3))

with zipfile.ZipFile(backup_zip, "r") as zf:
    files = zf.namelist()
    print("Files in zip:", len(files))
    print("First 20 files:")
    for f in files[:20]:
        print(" ", f)

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display

zip_path = Path("/kaggle/working/PCC_RESULTS_TABLES_ONLY_BACKUP.zip")

print("Exists:", zip_path.exists())

if zip_path.exists():
    print("Size MB:", round(zip_path.stat().st_size / 1024 / 1024, 3))
    display(FileLink(str(zip_path)))
else:
    print("Zip file not found. You need to rerun the backup cell.")

In [ ]:
from pathlib import Path
import os

print("=== /kaggle/input ===")
for p in Path("/kaggle/input").iterdir():
    print(p)

print("\n=== search csv/json/pt/zip under input ===")
for suffix in ["*.zip", "*.csv", "*.json", "*.pt"]:
    print(f"\n--- {suffix} ---")
    for p in Path("/kaggle/input").rglob(suffix):
        print(p, round(os.path.getsize(p) / 1024 / 1024, 3), "MB")

In [ ]:
from pathlib import Path
import os
import shutil
import json
import re
import zipfile
import numpy as np
import pandas as pd

import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available. If you need training later, enable GPU first.")

WORKING = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input/datasets")

OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

OUT_DIR.mkdir(parents=True, exist_ok=True)
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

print("WORKING:", WORKING)
print("INPUT_ROOT exists:", INPUT_ROOT.exists())

print("\nDatasets under input:")
for p in INPUT_ROOT.rglob("*"):
    if p.is_dir() and p.parent == INPUT_ROOT:
        print(" ", p)

In [ ]:
from pathlib import Path
import shutil
import os

WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input/datasets/jeechangxin")

OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

MODEL_A_SRC = INPUT / "model-a-backup-final"
C2V2_SRC = INPUT / "c2-v2-5epoch-backup"
TABLES_SRC = INPUT / "pcc-results-tables-only-backup" / "pcc_independent_baseline" / "stage2_pcc_guided_student_learning"

MODEL_A_DST = STAGE2_DIR / "model_A_direct_target_5fold"
C2V2_DST = STAGE2_DIR / "model_C2_v2_conservative_oof_gated_quick_5epoch"
TABLES_DST = STAGE2_DIR

print("Model A src exists:", MODEL_A_SRC.exists())
print("C2-v2 src exists:", C2V2_SRC.exists())
print("Tables src exists:", TABLES_SRC.exists())

assert MODEL_A_SRC.exists(), MODEL_A_SRC
assert C2V2_SRC.exists(), C2V2_SRC

STAGE2_DIR.mkdir(parents=True, exist_ok=True)

# Restore Model A and C2-v2
for src, dst in [
    (MODEL_A_SRC, MODEL_A_DST),
    (C2V2_SRC, C2V2_DST),
]:
    if dst.exists():
        print("Removing old:", dst)
        shutil.rmtree(dst)

    print("\nCopying:")
    print("  from:", src)
    print("  to:  ", dst)
    shutil.copytree(src, dst)

# Restore previous result tables if available
if TABLES_SRC.exists():
    print("\nRestoring previous result tables...")
    for item in TABLES_SRC.iterdir():
        dst_item = TABLES_DST / item.name
        if dst_item.exists():
            print("Removing old table folder:", dst_item)
            shutil.rmtree(dst_item)
        shutil.copytree(item, dst_item)
    print("Tables restored.")

print("\nModel A checkpoints:")
for p in sorted((MODEL_A_DST / "checkpoints").glob("*.pt")):
    print(" ", p.name, round(os.path.getsize(p) / 1024 / 1024, 3), "MB")

print("\nC2-v2 checkpoints:")
for p in sorted((C2V2_DST / "checkpoints").glob("*.pt")):
    print(" ", p.name, round(os.path.getsize(p) / 1024 / 1024, 3), "MB")

print("\nVerification files:")
required = [
    MODEL_A_DST / "direct_target_case_metrics.csv",
    MODEL_A_DST / "matched_5fold_splits_seed42.csv",
    C2V2_DST / "c2_v2_alpha_case_metrics.csv",
    C2V2_DST / "c2_v2_run_info.json",
]

for p in required:
    print(p, "exists:", p.exists())

print("\nRESTORE MODEL A + C2-V2 COMPLETE")

In [ ]:
# ============================================================
# Regenerate preprocessing for locked 40 cases
# Uses case IDs from restored Model A metrics.
# Output:
#   preprocessed_locked40_2d_summary.csv
#   preprocessed_locked40_2d_npz/*.npz
# ============================================================

from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib


WORKING = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input/datasets")

OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
MODEL_A_METRICS = MODEL_A_DIR / "direct_target_case_metrics.csv"

NPZ_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"
SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
MANIFEST_PATH = OUT_DIR / "manifest_locked_original_40.csv"

NPZ_DIR.mkdir(parents=True, exist_ok=True)

assert MODEL_A_METRICS.exists(), f"Missing Model A metrics: {MODEL_A_METRICS}"


# ------------------------------------------------------------
# 1. Locate MU-Glioma-Post root
# ------------------------------------------------------------
def find_mu_root(input_root):
    patient_dirs = list(input_root.rglob("PatientID_*"))
    patient_dirs = [p for p in patient_dirs if p.is_dir()]

    print("PatientID dirs found:", len(patient_dirs))

    if len(patient_dirs) == 0:
        raise FileNotFoundError("Cannot find PatientID_* folders under /kaggle/input/datasets")

    # choose parent that contains most PatientID folders
    parent_counts = {}
    for p in patient_dirs:
        parent_counts[p.parent] = parent_counts.get(p.parent, 0) + 1

    best_parent = sorted(parent_counts.items(), key=lambda x: x[1], reverse=True)[0][0]
    print("Detected MU root:", best_parent)
    print("Patient folders under root:", parent_counts[best_parent])

    return best_parent


MU_ROOT = find_mu_root(INPUT_ROOT)


# ------------------------------------------------------------
# 2. Read locked case IDs from Model A metrics
# ------------------------------------------------------------
modelA_df = pd.read_csv(MODEL_A_METRICS)
case_ids = sorted(modelA_df["case_id"].unique().tolist())

print("\nLocked cases:", len(case_ids))
print("First 5:")
for cid in case_ids[:5]:
    print(" ", cid)


# ------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------
def parse_case_id(case_id):
    """
    Expected format:
      PatientID_0003_T1_to_T2_t1c
      PatientID_0008_T4_to_T6_t1c
    """
    m = re.match(r"(PatientID_\d+)_T(\d+)_to_T(\d+)_t1c", case_id)
    if not m:
        raise ValueError(f"Cannot parse case_id: {case_id}")

    patient_id = m.group(1)
    cur_t = int(m.group(2))
    fut_t = int(m.group(3))
    return patient_id, cur_t, fut_t


def find_patient_dir(patient_id):
    candidates = [p for p in MU_ROOT.rglob(patient_id) if p.is_dir()]

    if len(candidates) == 0:
        raise FileNotFoundError(f"Cannot find patient dir for {patient_id}")

    # Prefer exact child under MU_ROOT
    candidates = sorted(candidates, key=lambda p: len(str(p)))
    return candidates[0]


def find_timepoint_dir(patient_dir, t):
    """
    Flexible matching:
      Timepoint_1, Timepoint1, T1, TP1, etc.
    """
    dirs = [d for d in patient_dir.rglob("*") if d.is_dir()]
    dirs.append(patient_dir)

    patterns = [
        rf"^timepoint[_\-\s]*0*{t}$",
        rf"^tp[_\-\s]*0*{t}$",
        rf"^t[_\-\s]*0*{t}$",
        rf"timepoint[_\-\s]*0*{t}",
        rf"\bt[_\-\s]*0*{t}\b",
    ]

    scored = []
    for d in dirs:
        name = d.name.lower()
        full = str(d).lower()

        score = 0
        for pat in patterns:
            if re.search(pat, name):
                score += 10
            if re.search(pat, full):
                score += 2

        if score > 0:
            scored.append((score, d))

    if len(scored) == 0:
        print("\nAvailable dirs for", patient_dir)
        for d in dirs[:50]:
            print(" ", d)
        raise FileNotFoundError(f"Cannot find timepoint T{t} under {patient_dir}")

    scored = sorted(scored, key=lambda x: (-x[0], len(str(x[1]))))
    return scored[0][1]


def nii_files_under(d):
    files = []
    for p in d.rglob("*"):
        if p.is_file():
            name = p.name.lower()
            if name.endswith(".nii") or name.endswith(".nii.gz"):
                files.append(p)
    return files


def score_t1c_file(p):
    name = p.name.lower()
    s = 0

    if "t1c" in name:
        s += 100
    if "t1ce" in name:
        s += 80
    if "t1gd" in name:
        s += 80
    if "contrast" in name:
        s += 40

    # avoid masks/labels
    bad = ["mask", "seg", "label", "tumor", "tumour"]
    if any(b in name for b in bad):
        s -= 100

    return s


def score_mask_file(p):
    name = p.name.lower()
    s = 0

    if "tumormask" in name:
        s += 120
    if "tumourmask" in name:
        s += 120
    if "tumor_mask" in name:
        s += 120
    if "tumour_mask" in name:
        s += 120
    if "seg" in name:
        s += 90
    if "mask" in name:
        s += 80
    if "label" in name:
        s += 50

    # avoid modality images
    bad = ["t1c", "t1ce", "t1gd", "t1n", "t2", "flair"]
    if any(b in name for b in bad):
        s -= 60

    return s


def choose_best_file(files, scorer, desc):
    scored = [(scorer(p), p) for p in files]
    scored = [x for x in scored if x[0] > 0]

    if len(scored) == 0:
        print("\nAvailable NIfTI files:")
        for p in files:
            print(" ", p.name)
        raise FileNotFoundError(f"Cannot find {desc}")

    scored = sorted(scored, key=lambda x: (-x[0], len(x[1].name)))
    return scored[0][1]


def load_nii_as_zyx(path):
    arr = np.asarray(nib.load(str(path)).get_fdata())

    arr = np.squeeze(arr)

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D NIfTI, got shape {arr.shape} for {path}")

    # Convert to [Z, H, W]
    shape = arr.shape

    # Common MRI storage: [H, W, Z], e.g. 240 x 240 x 155
    if shape[-1] <= shape[0] and shape[-1] <= shape[1]:
        arr = np.moveaxis(arr, -1, 0)
    # Already [Z, H, W], e.g. 155 x 240 x 240
    elif shape[0] <= shape[1] and shape[0] <= shape[2]:
        arr = arr
    else:
        # fallback
        arr = np.moveaxis(arr, -1, 0)

    return arr.astype(np.float32)


def crop_to_common_shape(*arrays):
    z = min(a.shape[0] for a in arrays)
    h = min(a.shape[1] for a in arrays)
    w = min(a.shape[2] for a in arrays)
    return [a[:z, :h, :w] for a in arrays]


def normalize_t1c(img):
    img = img.astype(np.float32)

    foreground = img[np.isfinite(img) & (img > 0)]
    if foreground.size < 100:
        foreground = img[np.isfinite(img)]

    if foreground.size == 0:
        return np.zeros_like(img, dtype=np.float32)

    lo, hi = np.percentile(foreground, [1, 99])

    if hi <= lo:
        return np.zeros_like(img, dtype=np.float32)

    out = (img - lo) / (hi - lo + 1e-8)
    out = np.clip(out, 0, 1)
    return out.astype(np.float32)


def binarize_mask(mask):
    return (mask > 0).astype(np.uint8)


# ------------------------------------------------------------
# 4. Process cases
# ------------------------------------------------------------
rows = []

for case_id in tqdm(case_ids, desc="Preprocessing locked 40"):
    patient_id, cur_t, fut_t = parse_case_id(case_id)

    patient_dir = find_patient_dir(patient_id)
    cur_dir = find_timepoint_dir(patient_dir, cur_t)
    fut_dir = find_timepoint_dir(patient_dir, fut_t)

    cur_files = nii_files_under(cur_dir)
    fut_files = nii_files_under(fut_dir)

    cur_t1c_path = choose_best_file(cur_files, score_t1c_file, f"current T1c for {case_id}")
    cur_mask_path = choose_best_file(cur_files, score_mask_file, f"current mask for {case_id}")
    fut_mask_path = choose_best_file(fut_files, score_mask_file, f"future mask for {case_id}")

    cur_t1c = load_nii_as_zyx(cur_t1c_path)
    cur_mask = load_nii_as_zyx(cur_mask_path)
    fut_mask = load_nii_as_zyx(fut_mask_path)

    cur_t1c, cur_mask, fut_mask = crop_to_common_shape(cur_t1c, cur_mask, fut_mask)

    cur_t1c_norm = normalize_t1c(cur_t1c)
    cur_mask_bin = binarize_mask(cur_mask)
    fut_mask_bin = binarize_mask(fut_mask)

    # Future-change target: future tumour area not already tumour at current time
    target = ((fut_mask_bin > 0) & ~(cur_mask_bin > 0)).astype(np.uint8)

    X = np.stack([
        cur_t1c_norm,
        cur_mask_bin.astype(np.float32),
    ], axis=1).astype(np.float32)   # [Z, 2, H, W]

    Y = target[:, None, :, :].astype(np.uint8)  # [Z, 1, H, W]

    npz_path = NPZ_DIR / f"{case_id}.npz"
    np.savez_compressed(
        npz_path,
        X=X.astype(np.float16),
        Y=Y,
        case_id=case_id,
        patient_id=patient_id,
        current_timepoint=cur_t,
        future_timepoint=fut_t,
        current_t1c_path=str(cur_t1c_path),
        current_mask_path=str(cur_mask_path),
        future_mask_path=str(fut_mask_path),
    )

    rows.append({
        "case_id": case_id,
        "patient_id": patient_id,
        "current_timepoint": cur_t,
        "future_timepoint": fut_t,
        "num_slices": int(X.shape[0]),
        "height": int(X.shape[2]),
        "width": int(X.shape[3]),
        "positive_slices": int((Y[:, 0].reshape(Y.shape[0], -1).sum(axis=1) > 0).sum()),
        "target_voxels": int(Y.sum()),
        "npz_path": str(npz_path),
        "current_t1c_path": str(cur_t1c_path),
        "current_mask_path": str(cur_mask_path),
        "future_mask_path": str(fut_mask_path),
    })


summary_df = pd.DataFrame(rows)
summary_df.to_csv(SUMMARY_PATH, index=False)
summary_df.to_csv(MANIFEST_PATH, index=False)

print("\nSaved summary:", SUMMARY_PATH)
print("Saved manifest:", MANIFEST_PATH)

print("\nPreprocessing summary:")
print("Cases:", len(summary_df))
print("Total slices:", int(summary_df["num_slices"].sum()))
print("Total positive slices:", int(summary_df["positive_slices"].sum()))
print("Total target voxels:", int(summary_df["target_voxels"].sum()))

display(summary_df.head())

In [ ]:
from pathlib import Path
import pandas as pd
import os

WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

pre_summary = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
C2V2_DIR = STAGE2_DIR / "model_C2_v2_conservative_oof_gated_quick_5epoch"

model_a_metrics = MODEL_A_DIR / "direct_target_case_metrics.csv"
model_a_splits = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"
model_a_ckpts = list((MODEL_A_DIR / "checkpoints").glob("*.pt"))

c2_metrics = C2V2_DIR / "c2_v2_alpha_case_metrics.csv"
c2_ckpts = list((C2V2_DIR / "checkpoints").glob("*.pt"))

print("=== Core files ===")
print("Preprocessed summary exists:", pre_summary.exists())
print("Model A metrics exists:", model_a_metrics.exists())
print("Model A split exists:", model_a_splits.exists())
print("Model A checkpoints:", len(model_a_ckpts))
print("C2-v2 metrics exists:", c2_metrics.exists())
print("C2-v2 checkpoints:", len(c2_ckpts))

if pre_summary.exists():
    df = pd.read_csv(pre_summary)
    print("\n=== Preprocessed data ===")
    print("Cases:", len(df))
    print("Total slices:", int(df["num_slices"].sum()))
    print("Total positive slices:", int(df["positive_slices"].sum()))
    print("Total target voxels:", int(df["target_voxels"].sum()))

if model_a_metrics.exists():
    m = pd.read_csv(model_a_metrics)
    print("\n=== Model A mean metrics ===")
    print(m[["dice", "iou", "target_focus", "log10_ratio"]].mean())

if c2_metrics.exists():
    c2 = pd.read_csv(c2_metrics)
    print("\n=== C2-v2 alpha summary ===")
    print(c2.groupby("alpha")[["dice", "iou", "target_focus", "log10_ratio"]].mean())

print("\n=== Checkpoints ===")
print("Model A:")
for p in sorted(model_a_ckpts):
    print(" ", p.name, round(os.path.getsize(p) / 1024 / 1024, 3), "MB")

print("C2-v2:")
for p in sorted(c2_ckpts):
    print(" ", p.name, round(os.path.getsize(p) / 1024 / 1024, 3), "MB")

assert pre_summary.exists(), "Missing preprocessing summary"
assert len(pd.read_csv(pre_summary)) == 40, "Expected 40 preprocessed cases"
assert len(model_a_ckpts) == 5, "Expected 5 Model A checkpoints"
assert len(c2_ckpts) == 5, "Expected 5 C2-v2 checkpoints"

print("\nRECOVERY READY")

In [ ]:
from pathlib import Path
import torch
import json

WORKING = Path("/kaggle/working")
STAGE2_DIR = WORKING / "pcc_independent_baseline/stage2_pcc_guided_student_learning"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
C2V2_DIR = STAGE2_DIR / "model_C2_v2_conservative_oof_gated_quick_5epoch"

model_a_ckpt = MODEL_A_DIR / "checkpoints/direct_target_student_fold_1.pt"
c2_ckpt = C2V2_DIR / "checkpoints/c2_v2_conservative_oof_gated_fold_1.pt"
c2_info = C2V2_DIR / "c2_v2_run_info.json"

print("Model A checkpoint exists:", model_a_ckpt.exists())
print("C2-v2 checkpoint exists:", c2_ckpt.exists())
print("C2-v2 run info exists:", c2_info.exists())

def inspect_checkpoint(path, name):
    ckpt = torch.load(path, map_location="cpu")
    print("\n" + "=" * 80)
    print(name)
    print("Top-level keys:", ckpt.keys())

    if "model_state_dict" in ckpt:
        sd = ckpt["model_state_dict"]
    elif "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    else:
        sd = ckpt

    print("State dict keys sample:")
    for i, k in enumerate(sd.keys()):
        print(" ", k, tuple(sd[k].shape) if hasattr(sd[k], "shape") else type(sd[k]))
        if i >= 15:
            break

    first_conv = None
    for k, v in sd.items():
        if "weight" in k and len(v.shape) == 4:
            first_conv = (k, v.shape)
            break

    print("First conv:", first_conv)

    if first_conv is not None:
        k, shape = first_conv
        print("Detected input channels:", int(shape[1]))
        print("Detected base channels:", int(shape[0]))

    if isinstance(ckpt, dict):
        for key in ["fold", "config", "train_ids", "test_ids"]:
            if key in ckpt:
                print(f"\n{key}:")
                print(ckpt[key])

inspect_checkpoint(model_a_ckpt, "MODEL A")
inspect_checkpoint(c2_ckpt, "C2-V2")

if c2_info.exists():
    print("\n" + "=" * 80)
    print("C2-v2 run_info.json:")
    with open(c2_info, "r") as f:
        info = json.load(f)
    print(json.dumps(info, indent=2))

In [ ]:
# ============================================================
# PCC Pathology Evidence Audit Pilot
#
# Compare:
#   Model A baseline
#   C2-v2 PCC-trained student, alpha=0.5
#
# Question:
#   Does PCC-trained model rely more on true pathology-relevant regions?
#
# Audit:
#   For each held-out case:
#       1. Original prediction
#       2. Perturb true future-change target region
#       3. Perturb matched control region
#       4. Measure output/performance change
#
# Main evidence:
#   ES = Δtarget - Δcontrol
#   PCC Gain = ES_C2v2 - ES_ModelA
#
# No cheating:
#   Future target is used only to define audit/evaluation regions.
#   It is NOT used as model input.
# ============================================================

from pathlib import Path
import os
import re
import json
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from scipy.ndimage import distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import distance_transform_edt


# =====================
# Config
# =====================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 24

# C2-v2 best Dice alpha from previous results
C2_ALPHA = 0.50
C2_MAX_DELTA_LOGIT = 3.0
C2_GATE_RADIUS = 26

# Dose response
DOSES = [0.10, 0.20, 0.30, 0.40, 0.50]

# Minimum target voxels required for audit
MIN_TARGET_VOXELS = 50

# Perturbation mode:
# replace selected T1c voxels with case-level background/normal-tissue median
PERTURB_MODE = "median_replace"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
C2V2_DIR = STAGE2_DIR / "model_C2_v2_conservative_oof_gated_quick_5epoch"

SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"

AUDIT_DIR = STAGE2_DIR / "pcc_pathology_evidence_audit_pilot_modelA_vs_C2v2"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

assert PRE_SUMMARY_PATH.exists(), PRE_SUMMARY_PATH
assert SPLIT_PATH.exists(), SPLIT_PATH
assert MODEL_A_DIR.exists(), MODEL_A_DIR
assert C2V2_DIR.exists(), C2V2_DIR

pre_df = pd.read_csv(PRE_SUMMARY_PATH)
split_df = pd.read_csv(SPLIT_PATH)

case_to_npz = dict(zip(pre_df["case_id"], pre_df["npz_path"]))
case_ids = sorted(case_to_npz.keys())

print("Cases:", len(case_ids))
print("Audit output:", AUDIT_DIR)


# =====================
# Utility functions
# =====================
def normalize_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    mn, mx = float(x.min()), float(x.max())
    if mx <= mn:
        return np.zeros_like(x, dtype=np.float32)

    out = (x - mn) / (mx - mn + 1e-8)
    return np.clip(out, 0, 1).astype(np.float32)


def prob_to_logit_np(p, eps=1e-4):
    p = np.clip(p.astype(np.float32), eps, 1.0 - eps)
    return np.log(p / (1.0 - p)).astype(np.float32)


def sigmoid_np(x):
    x = np.clip(x, -30, 30)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)


def load_case(case_id):
    with np.load(case_to_npz[case_id], allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"].astype(np.float32)

    # X: [Z, 2, H, W]
    # Y: [Z, 1, H, W]
    return X, Y


def topk_metrics_fast(pred, target, eps=1e-8):
    pred = normalize_map(pred)
    target = target.astype(bool)

    k = int(target.sum())
    if k <= 0:
        return {
            "dice": np.nan,
            "iou": np.nan,
            "target_focus": np.nan,
            "log10_ratio": np.nan,
        }

    fp = pred.reshape(-1)
    ft = target.reshape(-1)

    n = fp.size
    if k >= n:
        top_idx = np.arange(n)
    else:
        top_idx = np.argpartition(fp, n - k)[n - k:]

    pb = np.zeros(n, dtype=bool)
    pb[top_idx] = True

    inter = np.logical_and(pb, ft).sum()
    union = np.logical_or(pb, ft).sum()

    dice = (2 * inter + eps) / (pb.sum() + ft.sum() + eps)
    iou = (inter + eps) / (union + eps)

    target_focus = fp[ft].sum() / (fp.sum() + eps)

    mean_t = fp[ft].mean()
    mean_n = fp[~ft].mean()
    log10_ratio = np.log10((mean_t + eps) / (mean_n + eps))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "target_focus": float(target_focus),
        "log10_ratio": float(log10_ratio),
    }


def score_in_region(pred, region):
    pred = normalize_map(pred)
    region = region.astype(bool)

    if region.sum() == 0:
        return np.nan

    return float(pred[region].mean())


def sum_in_region(pred, region):
    pred = normalize_map(pred)
    region = region.astype(bool)

    if region.sum() == 0:
        return np.nan

    return float(pred[region].sum())


def outside_distance(current_mask):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    dist = np.zeros((Z, H, W), dtype=np.float32)

    for z in range(Z):
        m = current_mask[z]
        if m.sum() == 0:
            continue

        d = distance_transform_edt(~m).astype(np.float32)
        d[m] = 0.0
        dist[z] = d

    return dist


def make_gate_from_dist(dist, current_mask, radius):
    current_mask = current_mask.astype(bool)

    gate = (dist <= radius).astype(np.float32)

    has_mask = current_mask.reshape(current_mask.shape[0], -1).sum(axis=1) > 0
    for z, ok in enumerate(has_mask):
        if not ok:
            gate[z] = 0.0

    return gate.astype(np.float32)


def case_fill_value(X):
    """
    Replacement value for perturbation:
    median T1c intensity outside current tumour mask.
    """
    t1c = X[:, 0]
    cur_mask = X[:, 1].astype(bool)

    brain = t1c > 0.01
    normal = brain & (~cur_mask)

    vals = t1c[normal]
    if vals.size < 100:
        vals = t1c[brain]
    if vals.size < 100:
        vals = t1c.reshape(-1)

    return float(np.median(vals))


def perturb_X_t1c(X, region_mask, fill_value):
    """
    Perturb only T1c channel.
    Keep tumour mask channel unchanged.
    """
    Xp = X.copy().astype(np.float32)
    region_mask = region_mask.astype(bool)

    if PERTURB_MODE == "median_replace":
        Xp[:, 0][region_mask] = fill_value
    else:
        raise ValueError(f"Unknown perturb mode: {PERTURB_MODE}")

    return Xp


def random_subset_mask(mask, fraction, rng):
    mask = mask.astype(bool)
    coords = np.flatnonzero(mask.reshape(-1))

    n_total = len(coords)
    if n_total == 0:
        return np.zeros_like(mask, dtype=bool)

    n_select = max(1, int(round(n_total * fraction)))
    n_select = min(n_select, n_total)

    selected = rng.choice(coords, size=n_select, replace=False)

    out = np.zeros(mask.size, dtype=bool)
    out[selected] = True
    return out.reshape(mask.shape)


def matched_control_mask(target_mask, current_mask, dist, fraction, rng):
    """
    Build a matched control region:
    - same voxel count as selected target region
    - inside boundary candidate band
    - outside true target
    - outside current tumour mask
    - roughly distance-matched to selected target voxels
    """
    target_mask = target_mask.astype(bool)
    current_mask = current_mask.astype(bool)

    # Select target subset first
    target_selected = random_subset_mask(target_mask, fraction, rng)
    n_select = int(target_selected.sum())

    if n_select <= 0:
        return target_selected, np.zeros_like(target_selected, dtype=bool)

    candidate = (dist <= C2_GATE_RADIUS) & (~target_mask) & (~current_mask)

    # If too few candidates, relax to non-target/non-current full image
    if candidate.sum() < n_select:
        candidate = (~target_mask) & (~current_mask)

    target_dist_vals = dist[target_selected]
    cand_flat = np.flatnonzero(candidate.reshape(-1))

    if len(cand_flat) == 0:
        return target_selected, np.zeros_like(target_selected, dtype=bool)

    # Distance-bin matching
    control = np.zeros(target_selected.size, dtype=bool)

    bins = [0, 2, 4, 8, 12, 16, 24, 32, 64, 1e9]

    for b0, b1 in zip(bins[:-1], bins[1:]):
        n_bin = int(((target_dist_vals >= b0) & (target_dist_vals < b1)).sum())
        if n_bin <= 0:
            continue

        cand_bin_mask = candidate & (dist >= b0) & (dist < b1)
        cand_bin = np.flatnonzero(cand_bin_mask.reshape(-1))
        cand_bin = np.array([x for x in cand_bin if not control[x]], dtype=np.int64)

        if len(cand_bin) == 0:
            continue

        choose_n = min(n_bin, len(cand_bin))
        chosen = rng.choice(cand_bin, size=choose_n, replace=False)
        control[chosen] = True

    # Fill shortage from remaining candidates
    current_n = int(control.sum())
    if current_n < n_select:
        remaining = np.array([x for x in cand_flat if not control[x]], dtype=np.int64)
        if len(remaining) > 0:
            add_n = min(n_select - current_n, len(remaining))
            chosen = rng.choice(remaining, size=add_n, replace=False)
            control[chosen] = True

    control = control.reshape(target_selected.shape)

    # If control too large/small, force exact-ish count
    if control.sum() > n_select:
        coords = np.flatnonzero(control.reshape(-1))
        keep = rng.choice(coords, size=n_select, replace=False)
        out = np.zeros(control.size, dtype=bool)
        out[keep] = True
        control = out.reshape(control.shape)

    return target_selected, control.astype(bool)


# =====================
# Models
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


def load_model_A_for_fold(fold):
    ckpt_path = MODEL_A_DIR / "checkpoints" / f"direct_target_student_fold_{fold}.pt"
    assert ckpt_path.exists(), ckpt_path

    ckpt = torch.load(ckpt_path, map_location="cpu")
    model = SmallUNet2D(in_ch=2, out_ch=1, base=16)

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    return model


def load_C2v2_for_fold(fold):
    ckpt_path = C2V2_DIR / "checkpoints" / f"c2_v2_conservative_oof_gated_fold_{fold}.pt"
    assert ckpt_path.exists(), ckpt_path

    ckpt = torch.load(ckpt_path, map_location="cpu")
    model = SmallUNet2D(in_ch=3, out_ch=1, base=16)

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    return model


@torch.no_grad()
def predict_model_A(modelA, X):
    """
    X: [Z, 2, H, W]
    returns prob map [Z, H, W]
    """
    parts = []

    for start in range(0, X.shape[0], BATCH_SIZE):
        xb = torch.from_numpy(X[start:start+BATCH_SIZE].astype(np.float32)).float().to(DEVICE)
        logits = modelA(xb)
        probs = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
        parts.append(probs.astype(np.float32))

    P = np.concatenate(parts, axis=0)
    return normalize_map(P)


@torch.no_grad()
def predict_C2v2(modelA, modelC2, X, alpha=C2_ALPHA):
    """
    C2-v2 input:
      current T1c + current mask + Model A prediction

    Final:
      base_logit + alpha * gated_delta
    """
    B = predict_model_A(modelA, X)
    base_logit = prob_to_logit_np(B)

    current_mask = X[:, 1].astype(bool)
    dist = outside_distance(current_mask)
    gate = make_gate_from_dist(dist, current_mask, C2_GATE_RADIUS)

    delta_parts = []

    for start in range(0, X.shape[0], BATCH_SIZE):
        x_part = X[start:start+BATCH_SIZE].astype(np.float32)
        b_part = B[start:start+BATCH_SIZE].astype(np.float32)
        gate_part = gate[start:start+BATCH_SIZE].astype(np.float32)

        x3 = np.concatenate([
            x_part,
            b_part[:, None, :, :],
        ], axis=1).astype(np.float32)

        xb = torch.from_numpy(x3).float().to(DEVICE)
        gt = torch.from_numpy(gate_part[:, None, :, :]).float().to(DEVICE)

        delta_raw = modelC2(xb)
        delta = C2_MAX_DELTA_LOGIT * torch.tanh(delta_raw)
        delta = delta * gt

        delta_parts.append(delta.detach().cpu().numpy()[:, 0].astype(np.float32))

    delta_map = np.concatenate(delta_parts, axis=0)

    final_prob = sigmoid_np(base_logit + alpha * delta_map)
    return normalize_map(final_prob)


# =====================
# Fold mapping
# =====================
case_to_fold = {}

for _, row in split_df.iterrows():
    if row["split"] == "test":
        case_to_fold[row["case_id"]] = int(row["fold"])

print("Test case fold mapping:", len(case_to_fold))
assert len(case_to_fold) == 40


# =====================
# Optional reproduction check
# =====================
print("\nRunning reproduction sanity check for original predictions...")

repro_rows = []

loaded_model_A = {}
loaded_model_C2 = {}

for case_id in tqdm(case_ids, desc="Reproduction check"):
    fold = case_to_fold[case_id]

    if fold not in loaded_model_A:
        loaded_model_A[fold] = load_model_A_for_fold(fold)
        loaded_model_C2[fold] = load_C2v2_for_fold(fold)

    modelA = loaded_model_A[fold]
    modelC2 = loaded_model_C2[fold]

    X, Y = load_case(case_id)
    target = Y[:, 0].astype(bool)

    PA = predict_model_A(modelA, X)
    PC2 = predict_C2v2(modelA, modelC2, X, alpha=C2_ALPHA)

    mA = topk_metrics_fast(PA, target)
    mC2 = topk_metrics_fast(PC2, target)

    repro_rows.append({
        "case_id": case_id,
        "fold": fold,
        "model_A_dice": mA["dice"],
        "model_A_iou": mA["iou"],
        "model_A_target_focus": mA["target_focus"],
        "model_A_log10_ratio": mA["log10_ratio"],
        "c2v2_dice": mC2["dice"],
        "c2v2_iou": mC2["iou"],
        "c2v2_target_focus": mC2["target_focus"],
        "c2v2_log10_ratio": mC2["log10_ratio"],
    })

repro_df = pd.DataFrame(repro_rows)
repro_path = AUDIT_DIR / "audit_reproduction_check.csv"
repro_df.to_csv(repro_path, index=False)

print("\nReproduction means:")
display(repro_df[[
    "model_A_dice", "model_A_iou", "model_A_target_focus", "model_A_log10_ratio",
    "c2v2_dice", "c2v2_iou", "c2v2_target_focus", "c2v2_log10_ratio"
]].mean())


# =====================
# Audit loop
# =====================
audit_rows = []

print("\nRunning target-vs-control perturbation audit...")

for case_id in tqdm(case_ids, desc="Evidence audit cases"):
    fold = case_to_fold[case_id]

    modelA = loaded_model_A[fold]
    modelC2 = loaded_model_C2[fold]

    X, Y = load_case(case_id)
    target_full = Y[:, 0].astype(bool)
    current_mask = X[:, 1].astype(bool)

    if target_full.sum() < MIN_TARGET_VOXELS:
        print("Skipping small target:", case_id, "target voxels:", int(target_full.sum()))
        continue

    dist = outside_distance(current_mask)

    # Focus audit on future-change voxels within candidate boundary band.
    candidate_band = (dist <= C2_GATE_RADIUS) & (~current_mask)
    target_region = target_full & candidate_band

    # fallback if too small
    if target_region.sum() < MIN_TARGET_VOXELS:
        target_region = target_full.copy()

    fill_value = case_fill_value(X)

    # Original predictions
    PA0 = predict_model_A(modelA, X)
    PC20 = predict_C2v2(modelA, modelC2, X, alpha=C2_ALPHA)

    original = {
        "Model_A": PA0,
        "C2_v2_PCC": PC20,
    }

    original_metrics = {
        name: topk_metrics_fast(pred, target_full)
        for name, pred in original.items()
    }

    rng = np.random.default_rng(abs(hash(case_id)) % (2**32))

    for dose in DOSES:
        target_selected, control_selected = matched_control_mask(
            target_mask=target_region,
            current_mask=current_mask,
            dist=dist,
            fraction=dose,
            rng=rng,
        )

        if target_selected.sum() < 1 or control_selected.sum() < 1:
            continue

        X_target = perturb_X_t1c(X, target_selected, fill_value=fill_value)
        X_control = perturb_X_t1c(X, control_selected, fill_value=fill_value)

        # Perturbed predictions
        PA_target = predict_model_A(modelA, X_target)
        PA_control = predict_model_A(modelA, X_control)

        PC2_target = predict_C2v2(modelA, modelC2, X_target, alpha=C2_ALPHA)
        PC2_control = predict_C2v2(modelA, modelC2, X_control, alpha=C2_ALPHA)

        perturbed = {
            "Model_A": {
                "target": PA_target,
                "control": PA_control,
            },
            "C2_v2_PCC": {
                "target": PC2_target,
                "control": PC2_control,
            },
        }

        for model_name in ["Model_A", "C2_v2_PCC"]:
            P0 = original[model_name]
            P_t = perturbed[model_name]["target"]
            P_c = perturbed[model_name]["control"]

            m0 = original_metrics[model_name]
            mt = topk_metrics_fast(P_t, target_full)
            mc = topk_metrics_fast(P_c, target_full)

            # Performance drop
            dice_drop_target = m0["dice"] - mt["dice"]
            dice_drop_control = m0["dice"] - mc["dice"]

            iou_drop_target = m0["iou"] - mt["iou"]
            iou_drop_control = m0["iou"] - mc["iou"]

            focus_drop_target = m0["target_focus"] - mt["target_focus"]
            focus_drop_control = m0["target_focus"] - mc["target_focus"]

            # Output sensitivity
            global_l1_target = float(np.mean(np.abs(P0 - P_t)))
            global_l1_control = float(np.mean(np.abs(P0 - P_c)))

            target_score0 = score_in_region(P0, target_full)
            target_score_t = score_in_region(P_t, target_full)
            target_score_c = score_in_region(P_c, target_full)

            target_score_drop_target = target_score0 - target_score_t
            target_score_drop_control = target_score0 - target_score_c

            selected_region_score0 = score_in_region(P0, target_selected)
            selected_region_score_t = score_in_region(P_t, target_selected)

            control_region_score0 = score_in_region(P0, control_selected)
            control_region_score_c = score_in_region(P_c, control_selected)

            # Evidential scores
            es_dice = dice_drop_target - dice_drop_control
            es_iou = iou_drop_target - iou_drop_control
            es_focus = focus_drop_target - focus_drop_control
            es_global_l1 = global_l1_target - global_l1_control
            es_target_score = target_score_drop_target - target_score_drop_control

            norm_es_global_l1 = es_global_l1 / (global_l1_target + global_l1_control + 1e-8)
            norm_es_target_score = es_target_score / (
                abs(target_score_drop_target) + abs(target_score_drop_control) + 1e-8
            )

            audit_rows.append({
                "case_id": case_id,
                "fold": fold,
                "model": model_name,
                "dose": dose,
                "alpha": C2_ALPHA if model_name == "C2_v2_PCC" else np.nan,

                "target_region_voxels_full": int(target_region.sum()),
                "target_selected_voxels": int(target_selected.sum()),
                "control_selected_voxels": int(control_selected.sum()),
                "fill_value": fill_value,

                "orig_dice": m0["dice"],
                "orig_iou": m0["iou"],
                "orig_target_focus": m0["target_focus"],
                "orig_log10_ratio": m0["log10_ratio"],

                "target_pert_dice": mt["dice"],
                "control_pert_dice": mc["dice"],
                "dice_drop_target": dice_drop_target,
                "dice_drop_control": dice_drop_control,
                "ES_dice_drop": es_dice,

                "target_pert_iou": mt["iou"],
                "control_pert_iou": mc["iou"],
                "iou_drop_target": iou_drop_target,
                "iou_drop_control": iou_drop_control,
                "ES_iou_drop": es_iou,

                "target_pert_focus": mt["target_focus"],
                "control_pert_focus": mc["target_focus"],
                "focus_drop_target": focus_drop_target,
                "focus_drop_control": focus_drop_control,
                "ES_focus_drop": es_focus,

                "global_l1_target": global_l1_target,
                "global_l1_control": global_l1_control,
                "ES_global_l1": es_global_l1,
                "Norm_ES_global_l1": norm_es_global_l1,

                "target_score0": target_score0,
                "target_score_after_target_pert": target_score_t,
                "target_score_after_control_pert": target_score_c,
                "target_score_drop_target": target_score_drop_target,
                "target_score_drop_control": target_score_drop_control,
                "ES_target_score_drop": es_target_score,
                "Norm_ES_target_score_drop": norm_es_target_score,

                "selected_region_score0": selected_region_score0,
                "selected_region_score_after_target_pert": selected_region_score_t,
                "control_region_score0": control_region_score0,
                "control_region_score_after_control_pert": control_region_score_c,
            })

audit_df = pd.DataFrame(audit_rows)
audit_path = AUDIT_DIR / "pcc_pathology_evidence_audit_case_metrics.csv"
audit_df.to_csv(audit_path, index=False)

print("\nSaved audit metrics:", audit_path)
print("Rows:", len(audit_df))


# =====================
# Summaries
# =====================
summary = (
    audit_df
    .groupby(["model", "dose"])[[
        "ES_dice_drop",
        "ES_iou_drop",
        "ES_focus_drop",
        "ES_global_l1",
        "Norm_ES_global_l1",
        "ES_target_score_drop",
        "Norm_ES_target_score_drop",
        "global_l1_target",
        "global_l1_control",
        "dice_drop_target",
        "dice_drop_control",
        "target_score_drop_target",
        "target_score_drop_control",
    ]]
    .agg(["mean", "median", "std", "min", "max"])
    .reset_index()
)

summary.columns = [
    "_".join([str(c) for c in col if str(c) != ""]).strip("_")
    if isinstance(col, tuple) else col
    for col in summary.columns
]

summary_path = AUDIT_DIR / "pcc_pathology_evidence_audit_summary_by_model_dose.csv"
summary.to_csv(summary_path, index=False)


# PCC gain: C2-v2 ES minus Model A ES, paired by case+dose
base = audit_df[audit_df["model"] == "Model_A"].copy()
pcc = audit_df[audit_df["model"] == "C2_v2_PCC"].copy()

merge_cols = ["case_id", "fold", "dose"]

paired = pcc.merge(
    base,
    on=merge_cols,
    suffixes=("_pcc", "_base"),
)

gain_rows = []

for metric in [
    "ES_dice_drop",
    "ES_iou_drop",
    "ES_focus_drop",
    "ES_global_l1",
    "Norm_ES_global_l1",
    "ES_target_score_drop",
    "Norm_ES_target_score_drop",
]:
    diff = paired[f"{metric}_pcc"] - paired[f"{metric}_base"]

    for dose in DOSES:
        sub = paired[paired["dose"] == dose]
        diff_d = sub[f"{metric}_pcc"] - sub[f"{metric}_base"]

        gain_rows.append({
            "dose": dose,
            "metric": metric,
            "mean_gain": float(diff_d.mean()),
            "median_gain": float(diff_d.median()),
            "std_gain": float(diff_d.std()),
            "wins": int((diff_d > 0).sum()),
            "losses": int((diff_d < 0).sum()),
            "ties": int((diff_d == 0).sum()),
            "total": int(diff_d.notna().sum()),
            "win_rate": float((diff_d > 0).mean()),
        })

    gain_rows.append({
        "dose": "all",
        "metric": metric,
        "mean_gain": float(diff.mean()),
        "median_gain": float(diff.median()),
        "std_gain": float(diff.std()),
        "wins": int((diff > 0).sum()),
        "losses": int((diff < 0).sum()),
        "ties": int((diff == 0).sum()),
        "total": int(diff.notna().sum()),
        "win_rate": float((diff > 0).mean()),
    })

gain_df = pd.DataFrame(gain_rows)
gain_path = AUDIT_DIR / "pcc_gain_C2v2_minus_ModelA.csv"
gain_df.to_csv(gain_path, index=False)


# Overall compact table
compact_rows = []

for model_name in ["Model_A", "C2_v2_PCC"]:
    sub = audit_df[audit_df["model"] == model_name]

    compact_rows.append({
        "model": model_name,
        "mean_ES_dice_drop": sub["ES_dice_drop"].mean(),
        "mean_ES_iou_drop": sub["ES_iou_drop"].mean(),
        "mean_ES_global_l1": sub["ES_global_l1"].mean(),
        "mean_Norm_ES_global_l1": sub["Norm_ES_global_l1"].mean(),
        "mean_ES_target_score_drop": sub["ES_target_score_drop"].mean(),
        "mean_Norm_ES_target_score_drop": sub["Norm_ES_target_score_drop"].mean(),
        "mean_global_l1_target": sub["global_l1_target"].mean(),
        "mean_global_l1_control": sub["global_l1_control"].mean(),
        "mean_dice_drop_target": sub["dice_drop_target"].mean(),
        "mean_dice_drop_control": sub["dice_drop_control"].mean(),
    })

compact_df = pd.DataFrame(compact_rows)
compact_path = AUDIT_DIR / "pcc_pathology_evidence_audit_compact_comparison.csv"
compact_df.to_csv(compact_path, index=False)

print("\n" + "=" * 90)
print("PCC Pathology Evidence Audit Pilot Finished")
print("Saved to:", AUDIT_DIR)

print("\nReproduction means:")
display(repro_df[[
    "model_A_dice", "model_A_iou", "model_A_target_focus", "model_A_log10_ratio",
    "c2v2_dice", "c2v2_iou", "c2v2_target_focus", "c2v2_log10_ratio"
]].mean())

print("\nAudit compact comparison:")
display(compact_df)

print("\nAudit summary by model and dose:")
display(summary)

print("\nPCC gain: C2-v2 minus Model A")
display(gain_df)

print("\nKey output files:")
print("Case metrics:", audit_path)
print("Summary:", summary_path)
print("Gain:", gain_path)
print("Compact:", compact_path)

In [ ]:
# ============================================================
# ULTRA-FAST Boundary-Anchor Evidence Audit
# Target runtime: ~5-7 minutes on T4
#
# Compare:
#   Model A baseline
#   C2-v2 PCC-trained student
#
# Core question:
#   Does PCC-trained C2-v2 rely more on current visible tumour-boundary
#   structures aligned with true future-change direction?
#
# Fast settings:
#   - 15 test cases only: 3 per fold
#   - one dose: 0.50
#   - one perturbation mode: t1c_plus_mask
#
# No cheating:
#   Future target is used only to define audit regions, not model input.
# ============================================================

from pathlib import Path
import os
import re
import json
import hashlib
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from scipy.ndimage import distance_transform_edt
except ImportError:
    !pip install -q scipy
    from scipy.ndimage import distance_transform_edt


# =====================
# Config
# =====================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32

# Ultra-fast setting
CASES_PER_FOLD = 3
ANCHOR_DOSES = [0.50]
PERTURB_MODES = ["t1c_plus_mask"]

# C2-v2 setting
C2_ALPHA = 0.50
C2_MAX_DELTA_LOGIT = 3.0
C2_GATE_RADIUS = 26

BOUNDARY_THICKNESS = 3
MIN_TARGET_VOXELS = 50
MIN_ANCHOR_VOXELS = 30

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
C2V2_DIR = STAGE2_DIR / "model_C2_v2_conservative_oof_gated_quick_5epoch"
SPLIT_PATH = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"

AUDIT_DIR = STAGE2_DIR / "ULTRA_FAST_boundary_anchor_audit_modelA_vs_C2v2"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

assert PRE_SUMMARY_PATH.exists(), PRE_SUMMARY_PATH
assert MODEL_A_DIR.exists(), MODEL_A_DIR
assert C2V2_DIR.exists(), C2V2_DIR
assert SPLIT_PATH.exists(), SPLIT_PATH

pre_df = pd.read_csv(PRE_SUMMARY_PATH)
split_df = pd.read_csv(SPLIT_PATH)

case_to_npz = dict(zip(pre_df["case_id"], pre_df["npz_path"]))

print("Output:", AUDIT_DIR)


# =====================
# Choose 15 fast pilot cases: 3 per fold
# =====================
quick_case_rows = []

for fold in sorted(split_df["fold"].unique()):
    sub = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")].copy()
    sub = sub.sort_values("case_id").head(CASES_PER_FOLD)
    quick_case_rows.append(sub)

quick_df = pd.concat(quick_case_rows, ignore_index=True)
quick_case_ids = quick_df["case_id"].tolist()

case_to_fold = dict(zip(quick_df["case_id"], quick_df["fold"].astype(int)))

print("Quick cases:", len(quick_case_ids))
print(quick_df[["fold", "case_id"]])


# =====================
# Utilities
# =====================
def stable_seed_from_string(s):
    h = hashlib.md5(s.encode("utf-8")).hexdigest()
    return int(h[:8], 16)


def normalize_map(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    mn, mx = float(x.min()), float(x.max())
    if mx <= mn:
        return np.zeros_like(x, dtype=np.float32)

    out = (x - mn) / (mx - mn + 1e-8)
    return np.clip(out, 0, 1).astype(np.float32)


def prob_to_logit_np(p, eps=1e-4):
    p = np.clip(p.astype(np.float32), eps, 1.0 - eps)
    return np.log(p / (1.0 - p)).astype(np.float32)


def sigmoid_np(x):
    x = np.clip(x, -30, 30)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)


def load_case(case_id):
    with np.load(case_to_npz[case_id], allow_pickle=True) as data:
        X = data["X"].astype(np.float32)
        Y = data["Y"].astype(np.float32)
    return X, Y


def score_in_region(pred, region):
    pred = normalize_map(pred)
    region = region.astype(bool)

    if region.sum() == 0:
        return np.nan

    return float(pred[region].mean())


def outside_distance(current_mask):
    current_mask = current_mask.astype(bool)
    Z, H, W = current_mask.shape
    dist = np.zeros((Z, H, W), dtype=np.float32)

    for z in range(Z):
        m = current_mask[z]
        if m.sum() == 0:
            continue
        d = distance_transform_edt(~m).astype(np.float32)
        d[m] = 0.0
        dist[z] = d

    return dist


def make_gate_from_dist(dist, current_mask, radius):
    current_mask = current_mask.astype(bool)
    gate = (dist <= radius).astype(np.float32)

    has_mask = current_mask.reshape(current_mask.shape[0], -1).sum(axis=1) > 0
    for z, ok in enumerate(has_mask):
        if not ok:
            gate[z] = 0.0

    return gate.astype(np.float32)


def case_fill_value(X):
    t1c = X[:, 0]
    cur_mask = X[:, 1].astype(bool)

    brain = t1c > 0.01
    normal = brain & (~cur_mask)

    vals = t1c[normal]
    if vals.size < 100:
        vals = t1c[brain]
    if vals.size < 100:
        vals = t1c.reshape(-1)

    return float(np.median(vals))


def current_boundary_inside(current_mask, thickness=3):
    current_mask = current_mask.astype(bool)
    inside_dist = np.zeros_like(current_mask, dtype=np.float32)

    for z in range(current_mask.shape[0]):
        m = current_mask[z]
        if m.sum() == 0:
            continue
        inside_dist[z] = distance_transform_edt(m).astype(np.float32)

    boundary = current_mask & (inside_dist > 0) & (inside_dist <= thickness)
    return boundary, inside_dist


def select_boundary_anchor_and_control(current_mask, future_target, dose):
    """
    Target anchor:
      current tumour boundary voxels closest to true future-change target.

    Control anchor:
      current tumour boundary voxels farthest from true future-change target.

    Both are visible current-time structures.
    """
    boundary, _ = current_boundary_inside(current_mask, BOUNDARY_THICKNESS)

    if boundary.sum() < MIN_ANCHOR_VOXELS:
        return None, None

    future_target = future_target.astype(bool)

    if future_target.sum() == 0:
        return None, None

    dist_to_future = distance_transform_edt(~future_target).astype(np.float32)

    boundary_flat = np.flatnonzero(boundary.reshape(-1))
    if len(boundary_flat) < MIN_ANCHOR_VOXELS:
        return None, None

    # Smaller distance to future target = more pathology-direction aligned.
    dist_vals = dist_to_future.reshape(-1)[boundary_flat]
    order_near = np.argsort(dist_vals)        # near future-change
    order_far = np.argsort(dist_vals)[::-1]   # far from future-change

    n_select = int(round(len(boundary_flat) * dose))
    n_select = max(MIN_ANCHOR_VOXELS, n_select)
    n_select = min(n_select, len(boundary_flat) // 2)

    if n_select < MIN_ANCHOR_VOXELS:
        return None, None

    target_idx = boundary_flat[order_near[:n_select]]

    target_set = set(target_idx.tolist())
    far_candidates = [int(boundary_flat[i]) for i in order_far if int(boundary_flat[i]) not in target_set]

    if len(far_candidates) < n_select:
        return None, None

    control_idx = np.array(far_candidates[:n_select], dtype=np.int64)

    target_mask = np.zeros(current_mask.size, dtype=bool)
    control_mask = np.zeros(current_mask.size, dtype=bool)

    target_mask[target_idx] = True
    control_mask[control_idx] = True

    return target_mask.reshape(current_mask.shape), control_mask.reshape(current_mask.shape)


def perturb_X_boundary(X, region_mask, fill_value, mode="t1c_plus_mask"):
    Xp = X.copy().astype(np.float32)
    region_mask = region_mask.astype(bool)

    # perturb T1c
    Xp[:, 0][region_mask] = fill_value

    # remove current-mask signal in selected boundary region
    if mode == "t1c_plus_mask":
        Xp[:, 1][region_mask] = 0.0
    elif mode == "t1c_only":
        pass
    else:
        raise ValueError(mode)

    return Xp


def map_change_stats(P0, Pp, future_target, anchor_mask):
    P0 = normalize_map(P0)
    Pp = normalize_map(Pp)

    future_target = future_target.astype(bool)
    anchor_mask = anchor_mask.astype(bool)

    global_l1 = float(np.mean(np.abs(P0 - Pp)))

    target_score0 = score_in_region(P0, future_target)
    target_scorep = score_in_region(Pp, future_target)
    target_score_drop = target_score0 - target_scorep

    anchor_score0 = score_in_region(P0, anchor_mask)
    anchor_scorep = score_in_region(Pp, anchor_mask)
    anchor_score_drop = anchor_score0 - anchor_scorep

    return {
        "global_l1": global_l1,
        "target_score0": target_score0,
        "target_score_after": target_scorep,
        "target_score_drop": target_score_drop,
        "anchor_score0": anchor_score0,
        "anchor_score_after": anchor_scorep,
        "anchor_score_drop": anchor_score_drop,
    }


# =====================
# Model definitions
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


def load_model_A_for_fold(fold):
    ckpt_path = MODEL_A_DIR / "checkpoints" / f"direct_target_student_fold_{fold}.pt"
    assert ckpt_path.exists(), ckpt_path

    ckpt = torch.load(ckpt_path, map_location="cpu")
    model = SmallUNet2D(in_ch=2, out_ch=1, base=16)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()
    return model


def load_C2v2_for_fold(fold):
    ckpt_path = C2V2_DIR / "checkpoints" / f"c2_v2_conservative_oof_gated_fold_{fold}.pt"
    assert ckpt_path.exists(), ckpt_path

    ckpt = torch.load(ckpt_path, map_location="cpu")
    model = SmallUNet2D(in_ch=3, out_ch=1, base=16)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()
    return model


@torch.no_grad()
def predict_model_A(modelA, X):
    parts = []

    for start in range(0, X.shape[0], BATCH_SIZE):
        xb = torch.from_numpy(X[start:start+BATCH_SIZE].astype(np.float32)).float().to(DEVICE)
        logits = modelA(xb)
        probs = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
        parts.append(probs.astype(np.float32))

    return normalize_map(np.concatenate(parts, axis=0))


@torch.no_grad()
def predict_C2v2_from_base(modelC2, X, base_pred, alpha=C2_ALPHA):
    """
    Use already-computed Model A base_pred to avoid duplicate inference.
    """
    B = normalize_map(base_pred)
    base_logit = prob_to_logit_np(B)

    current_mask = X[:, 1].astype(bool)
    dist = outside_distance(current_mask)
    gate = make_gate_from_dist(dist, current_mask, C2_GATE_RADIUS)

    delta_parts = []

    for start in range(0, X.shape[0], BATCH_SIZE):
        x_part = X[start:start+BATCH_SIZE].astype(np.float32)
        b_part = B[start:start+BATCH_SIZE].astype(np.float32)
        gate_part = gate[start:start+BATCH_SIZE].astype(np.float32)

        x3 = np.concatenate([
            x_part,
            b_part[:, None, :, :],
        ], axis=1).astype(np.float32)

        xb = torch.from_numpy(x3).float().to(DEVICE)
        gt = torch.from_numpy(gate_part[:, None, :, :]).float().to(DEVICE)

        delta_raw = modelC2(xb)
        delta = C2_MAX_DELTA_LOGIT * torch.tanh(delta_raw)
        delta = delta * gt

        delta_parts.append(delta.detach().cpu().numpy()[:, 0].astype(np.float32))

    delta_map = np.concatenate(delta_parts, axis=0)

    final_prob = sigmoid_np(base_logit + alpha * delta_map)
    return normalize_map(final_prob)


# =====================
# Load models per fold
# =====================
loaded_model_A = {}
loaded_model_C2 = {}

for fold in sorted(quick_df["fold"].unique()):
    loaded_model_A[int(fold)] = load_model_A_for_fold(int(fold))
    loaded_model_C2[int(fold)] = load_C2v2_for_fold(int(fold))

print("Loaded folds:", sorted(loaded_model_A.keys()))


# =====================
# Run ultra-fast audit
# =====================
rows = []

print("\nRunning ULTRA-FAST boundary-anchor audit...")

for case_id in tqdm(quick_case_ids, desc="Cases"):
    fold = case_to_fold[case_id]

    modelA = loaded_model_A[fold]
    modelC2 = loaded_model_C2[fold]

    X, Y = load_case(case_id)

    future_target = Y[:, 0].astype(bool)
    current_mask = X[:, 1].astype(bool)

    if future_target.sum() < MIN_TARGET_VOXELS:
        continue

    fill_value = case_fill_value(X)

    # Original predictions
    PA0 = predict_model_A(modelA, X)
    PC20 = predict_C2v2_from_base(modelC2, X, PA0, alpha=C2_ALPHA)

    for dose in ANCHOR_DOSES:
        target_anchor, control_anchor = select_boundary_anchor_and_control(
            current_mask=current_mask,
            future_target=future_target,
            dose=dose,
        )

        if target_anchor is None:
            continue

        for mode in PERTURB_MODES:
            X_target = perturb_X_boundary(X, target_anchor, fill_value, mode)
            X_control = perturb_X_boundary(X, control_anchor, fill_value, mode)

            # Model A perturbed predictions
            PA_t = predict_model_A(modelA, X_target)
            PA_c = predict_model_A(modelA, X_control)

            # C2-v2 perturbed predictions, reusing perturbed Model A bases
            PC2_t = predict_C2v2_from_base(modelC2, X_target, PA_t, alpha=C2_ALPHA)
            PC2_c = predict_C2v2_from_base(modelC2, X_control, PA_c, alpha=C2_ALPHA)

            preds = {
                "Model_A": {
                    "original": PA0,
                    "target": PA_t,
                    "control": PA_c,
                },
                "C2_v2_PCC": {
                    "original": PC20,
                    "target": PC2_t,
                    "control": PC2_c,
                }
            }

            for model_name, dct in preds.items():
                P0 = dct["original"]
                Pt = dct["target"]
                Pc = dct["control"]

                st = map_change_stats(P0, Pt, future_target, target_anchor)
                sc = map_change_stats(P0, Pc, future_target, control_anchor)

                ES_global_l1 = st["global_l1"] - sc["global_l1"]
                ES_target_score_drop = st["target_score_drop"] - sc["target_score_drop"]
                ES_anchor_score_drop = st["anchor_score_drop"] - sc["anchor_score_drop"]

                Norm_ES_global_l1 = ES_global_l1 / (st["global_l1"] + sc["global_l1"] + 1e-8)
                Norm_ES_target_score_drop = ES_target_score_drop / (
                    abs(st["target_score_drop"]) + abs(sc["target_score_drop"]) + 1e-8
                )

                rows.append({
                    "case_id": case_id,
                    "fold": fold,
                    "model": model_name,
                    "dose": dose,
                    "perturb_mode": mode,
                    "target_anchor_voxels": int(target_anchor.sum()),
                    "control_anchor_voxels": int(control_anchor.sum()),
                    "future_target_voxels": int(future_target.sum()),

                    "global_l1_target_anchor": st["global_l1"],
                    "global_l1_control_anchor": sc["global_l1"],
                    "ES_global_l1": ES_global_l1,
                    "Norm_ES_global_l1": Norm_ES_global_l1,

                    "target_score_drop_target_anchor": st["target_score_drop"],
                    "target_score_drop_control_anchor": sc["target_score_drop"],
                    "ES_target_score_drop": ES_target_score_drop,
                    "Norm_ES_target_score_drop": Norm_ES_target_score_drop,

                    "anchor_score_drop_target_anchor": st["anchor_score_drop"],
                    "anchor_score_drop_control_anchor": sc["anchor_score_drop"],
                    "ES_anchor_score_drop": ES_anchor_score_drop,
                })


df = pd.DataFrame(rows)

case_path = AUDIT_DIR / "ultra_fast_boundary_anchor_case_metrics.csv"
df.to_csv(case_path, index=False)

print("\nRows:", len(df))
print("Saved case metrics:", case_path)


# =====================
# Summaries
# =====================
compact_rows = []

for model_name in ["Model_A", "C2_v2_PCC"]:
    sub = df[df["model"] == model_name]

    compact_rows.append({
        "model": model_name,
        "n_rows": len(sub),
        "mean_ES_global_l1": sub["ES_global_l1"].mean(),
        "median_ES_global_l1": sub["ES_global_l1"].median(),
        "mean_Norm_ES_global_l1": sub["Norm_ES_global_l1"].mean(),
        "median_Norm_ES_global_l1": sub["Norm_ES_global_l1"].median(),
        "mean_ES_target_score_drop": sub["ES_target_score_drop"].mean(),
        "median_ES_target_score_drop": sub["ES_target_score_drop"].median(),
        "mean_Norm_ES_target_score_drop": sub["Norm_ES_target_score_drop"].mean(),
        "median_Norm_ES_target_score_drop": sub["Norm_ES_target_score_drop"].median(),
        "mean_ES_anchor_score_drop": sub["ES_anchor_score_drop"].mean(),
        "mean_global_l1_target_anchor": sub["global_l1_target_anchor"].mean(),
        "mean_global_l1_control_anchor": sub["global_l1_control_anchor"].mean(),
    })

compact_df = pd.DataFrame(compact_rows)
compact_path = AUDIT_DIR / "ultra_fast_boundary_anchor_compact_comparison.csv"
compact_df.to_csv(compact_path, index=False)


# Paired PCC gain
base = df[df["model"] == "Model_A"]
pcc = df[df["model"] == "C2_v2_PCC"]

paired = pcc.merge(
    base,
    on=["case_id", "fold", "dose", "perturb_mode"],
    suffixes=("_pcc", "_base")
)

gain_rows = []

for metric in [
    "ES_global_l1",
    "Norm_ES_global_l1",
    "ES_target_score_drop",
    "Norm_ES_target_score_drop",
    "ES_anchor_score_drop",
]:
    diff = paired[f"{metric}_pcc"] - paired[f"{metric}_base"]

    gain_rows.append({
        "metric": metric,
        "mean_gain": float(diff.mean()),
        "median_gain": float(diff.median()),
        "std_gain": float(diff.std()),
        "wins": int((diff > 0).sum()),
        "losses": int((diff < 0).sum()),
        "ties": int((diff == 0).sum()),
        "total": int(diff.notna().sum()),
        "win_rate": float((diff > 0).mean()),
    })

gain_df = pd.DataFrame(gain_rows)
gain_path = AUDIT_DIR / "ultra_fast_boundary_anchor_pcc_gain_C2v2_minus_ModelA.csv"
gain_df.to_csv(gain_path, index=False)


print("\n" + "=" * 90)
print("ULTRA-FAST Boundary-Anchor Audit Finished")
print("Saved to:", AUDIT_DIR)

print("\nCompact comparison:")
display(compact_df)

print("\nPCC gain: C2-v2 minus Model A")
display(gain_df)

print("\nFiles:")
print("Case metrics:", case_path)
print("Compact:", compact_path)
print("Gain:", gain_path)

In [ ]:
# ============================================================
# 40-case FAST Boundary-Anchor Evidence Audit
#
# Same setting as ultra-fast pilot:
#   - all 40 held-out cases
#   - dose = 0.50
#   - perturb_mode = t1c_plus_mask
#
# Purpose:
#   Confirm whether the 15-case positive target-local signal is stable.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

# =====================
# Output path
# =====================
AUDIT40_DIR = STAGE2_DIR / "FAST40_boundary_anchor_audit_modelA_vs_C2v2"
AUDIT40_DIR.mkdir(parents=True, exist_ok=True)

print("Output:", AUDIT40_DIR)

# =====================
# Use all 40 test cases from the original 5-fold split
# =====================
all_test_df = split_df[split_df["split"] == "test"].copy()
all_test_df = all_test_df.sort_values(["fold", "case_id"]).reset_index(drop=True)

all_case_ids = all_test_df["case_id"].tolist()
case_to_fold_40 = dict(zip(all_test_df["case_id"], all_test_df["fold"].astype(int)))

print("40-case test set:", len(all_case_ids))
display(all_test_df[["fold", "case_id"]])

# =====================
# Make sure all fold models are loaded
# =====================
if "loaded_model_A" not in globals():
    loaded_model_A = {}

if "loaded_model_C2" not in globals():
    loaded_model_C2 = {}

for fold in sorted(all_test_df["fold"].unique()):
    fold = int(fold)
    if fold not in loaded_model_A:
        loaded_model_A[fold] = load_model_A_for_fold(fold)
    if fold not in loaded_model_C2:
        loaded_model_C2[fold] = load_C2v2_for_fold(fold)

print("Loaded folds:", sorted(loaded_model_A.keys()))

# =====================
# Fixed audit setting
# =====================
DOSE = 0.50
PERTURB_MODE = "t1c_plus_mask"

rows = []

print("\nRunning 40-case FAST boundary-anchor audit...")

for case_id in tqdm(all_case_ids, desc="40-case audit"):
    fold = case_to_fold_40[case_id]

    modelA = loaded_model_A[fold]
    modelC2 = loaded_model_C2[fold]

    X, Y = load_case(case_id)

    future_target = Y[:, 0].astype(bool)
    current_mask = X[:, 1].astype(bool)

    if future_target.sum() < MIN_TARGET_VOXELS:
        print("Skipping small future target:", case_id, int(future_target.sum()))
        continue

    fill_value = case_fill_value(X)

    # Original predictions
    PA0 = predict_model_A(modelA, X)
    PC20 = predict_C2v2_from_base(modelC2, X, PA0, alpha=C2_ALPHA)

    # Select boundary anchor and matched control
    target_anchor, control_anchor = select_boundary_anchor_and_control(
        current_mask=current_mask,
        future_target=future_target,
        dose=DOSE,
    )

    if target_anchor is None:
        print("Skipping no boundary anchor:", case_id)
        continue

    # Perturb target anchor and control anchor
    X_target = perturb_X_boundary(X, target_anchor, fill_value, PERTURB_MODE)
    X_control = perturb_X_boundary(X, control_anchor, fill_value, PERTURB_MODE)

    # Model A perturbed predictions
    PA_t = predict_model_A(modelA, X_target)
    PA_c = predict_model_A(modelA, X_control)

    # C2-v2 perturbed predictions
    PC2_t = predict_C2v2_from_base(modelC2, X_target, PA_t, alpha=C2_ALPHA)
    PC2_c = predict_C2v2_from_base(modelC2, X_control, PA_c, alpha=C2_ALPHA)

    preds = {
        "Model_A": {
            "original": PA0,
            "target": PA_t,
            "control": PA_c,
        },
        "C2_v2_PCC": {
            "original": PC20,
            "target": PC2_t,
            "control": PC2_c,
        }
    }

    for model_name, dct in preds.items():
        P0 = dct["original"]
        Pt = dct["target"]
        Pc = dct["control"]

        st = map_change_stats(P0, Pt, future_target, target_anchor)
        sc = map_change_stats(P0, Pc, future_target, control_anchor)

        ES_global_l1 = st["global_l1"] - sc["global_l1"]
        ES_target_score_drop = st["target_score_drop"] - sc["target_score_drop"]
        ES_anchor_score_drop = st["anchor_score_drop"] - sc["anchor_score_drop"]

        Norm_ES_global_l1 = ES_global_l1 / (
            st["global_l1"] + sc["global_l1"] + 1e-8
        )

        Norm_ES_target_score_drop = ES_target_score_drop / (
            abs(st["target_score_drop"]) + abs(sc["target_score_drop"]) + 1e-8
        )

        rows.append({
            "case_id": case_id,
            "fold": fold,
            "model": model_name,
            "dose": DOSE,
            "perturb_mode": PERTURB_MODE,

            "target_anchor_voxels": int(target_anchor.sum()),
            "control_anchor_voxels": int(control_anchor.sum()),
            "future_target_voxels": int(future_target.sum()),

            "global_l1_target_anchor": st["global_l1"],
            "global_l1_control_anchor": sc["global_l1"],
            "ES_global_l1": ES_global_l1,
            "Norm_ES_global_l1": Norm_ES_global_l1,

            "target_score_drop_target_anchor": st["target_score_drop"],
            "target_score_drop_control_anchor": sc["target_score_drop"],
            "ES_target_score_drop": ES_target_score_drop,
            "Norm_ES_target_score_drop": Norm_ES_target_score_drop,

            "anchor_score_drop_target_anchor": st["anchor_score_drop"],
            "anchor_score_drop_control_anchor": sc["anchor_score_drop"],
            "ES_anchor_score_drop": ES_anchor_score_drop,
        })

df40 = pd.DataFrame(rows)

case_path = AUDIT40_DIR / "fast40_boundary_anchor_case_metrics.csv"
df40.to_csv(case_path, index=False)

print("\nRows:", len(df40))
print("Saved case metrics:", case_path)

# =====================
# Compact comparison
# =====================
compact_rows = []

for model_name in ["Model_A", "C2_v2_PCC"]:
    sub = df40[df40["model"] == model_name]

    compact_rows.append({
        "model": model_name,
        "n_rows": len(sub),

        "mean_ES_global_l1": sub["ES_global_l1"].mean(),
        "median_ES_global_l1": sub["ES_global_l1"].median(),

        "mean_Norm_ES_global_l1": sub["Norm_ES_global_l1"].mean(),
        "median_Norm_ES_global_l1": sub["Norm_ES_global_l1"].median(),

        "mean_ES_target_score_drop": sub["ES_target_score_drop"].mean(),
        "median_ES_target_score_drop": sub["ES_target_score_drop"].median(),

        "mean_Norm_ES_target_score_drop": sub["Norm_ES_target_score_drop"].mean(),
        "median_Norm_ES_target_score_drop": sub["Norm_ES_target_score_drop"].median(),

        "mean_ES_anchor_score_drop": sub["ES_anchor_score_drop"].mean(),
        "median_ES_anchor_score_drop": sub["ES_anchor_score_drop"].median(),

        "mean_global_l1_target_anchor": sub["global_l1_target_anchor"].mean(),
        "mean_global_l1_control_anchor": sub["global_l1_control_anchor"].mean(),

        "mean_target_score_drop_target_anchor": sub["target_score_drop_target_anchor"].mean(),
        "mean_target_score_drop_control_anchor": sub["target_score_drop_control_anchor"].mean(),
    })

compact40_df = pd.DataFrame(compact_rows)

compact_path = AUDIT40_DIR / "fast40_boundary_anchor_compact_comparison.csv"
compact40_df.to_csv(compact_path, index=False)

# =====================
# Paired PCC gain: C2-v2 minus Model A
# =====================
base = df40[df40["model"] == "Model_A"]
pcc = df40[df40["model"] == "C2_v2_PCC"]

paired = pcc.merge(
    base,
    on=["case_id", "fold", "dose", "perturb_mode"],
    suffixes=("_pcc", "_base")
)

gain_rows = []

for metric in [
    "ES_global_l1",
    "Norm_ES_global_l1",
    "ES_target_score_drop",
    "Norm_ES_target_score_drop",
    "ES_anchor_score_drop",
]:
    diff = paired[f"{metric}_pcc"] - paired[f"{metric}_base"]

    gain_rows.append({
        "metric": metric,
        "mean_gain": float(diff.mean()),
        "median_gain": float(diff.median()),
        "std_gain": float(diff.std()),
        "wins": int((diff > 0).sum()),
        "losses": int((diff < 0).sum()),
        "ties": int((diff == 0).sum()),
        "total": int(diff.notna().sum()),
        "win_rate": float((diff > 0).mean()),
    })

gain40_df = pd.DataFrame(gain_rows)

gain_path = AUDIT40_DIR / "fast40_boundary_anchor_pcc_gain_C2v2_minus_ModelA.csv"
gain40_df.to_csv(gain_path, index=False)

# =====================
# Optional simple bootstrap CI for primary metrics
# =====================
boot_rows = []
rng = np.random.default_rng(42)

for metric in [
    "ES_target_score_drop",
    "Norm_ES_target_score_drop",
    "ES_anchor_score_drop",
]:
    diff = paired[f"{metric}_pcc"].values - paired[f"{metric}_base"].values
    diff = diff[np.isfinite(diff)]

    boots = []
    for _ in range(2000):
        sample = rng.choice(diff, size=len(diff), replace=True)
        boots.append(sample.mean())

    boot_rows.append({
        "metric": metric,
        "mean_gain": float(diff.mean()),
        "ci95_low": float(np.percentile(boots, 2.5)),
        "ci95_high": float(np.percentile(boots, 97.5)),
        "n": int(len(diff)),
    })

boot40_df = pd.DataFrame(boot_rows)

boot_path = AUDIT40_DIR / "fast40_boundary_anchor_bootstrap_CI_primary_metrics.csv"
boot40_df.to_csv(boot_path, index=False)

# =====================
# Final display
# =====================
print("\n" + "=" * 90)
print("FAST40 Boundary-Anchor Audit Finished")
print("Saved to:", AUDIT40_DIR)

print("\nCompact comparison:")
display(compact40_df)

print("\nPCC gain: C2-v2 minus Model A")
display(gain40_df)

print("\nBootstrap 95% CI for primary target-local metrics:")
display(boot40_df)

print("\nFiles:")
print("Case metrics:", case_path)
print("Compact:", compact_path)
print("Gain:", gain_path)
print("Bootstrap CI:", boot_path)

In [ ]:
from pathlib import Path
import zipfile
import json
import pandas as pd
from datetime import datetime

STAGE2_DIR = Path("/kaggle/working/pcc_independent_baseline/stage2_pcc_guided_student_learning")

folders_to_save = [
    STAGE2_DIR / "FAST40_boundary_anchor_audit_modelA_vs_C2v2",
    STAGE2_DIR / "ULTRA_FAST_boundary_anchor_audit_modelA_vs_C2v2",
    STAGE2_DIR / "pcc_pathology_evidence_audit_pilot_modelA_vs_C2v2",
]

backup_root = Path("/kaggle/working")
zip_path = backup_root / "PCC_third_layer_audit_results_BACKUP.zip"

manifest = {
    "created_at": datetime.now().isoformat(),
    "purpose": "Backup of PCC third-layer perturbation/evidence-reliance audit results",
    "main_result": "FAST40_boundary_anchor_audit_modelA_vs_C2v2",
    "interpretation": {
        "status": "Third-layer audit feasibility gate passed",
        "important_note": "These results support continuing the third-layer perturbation audit, but do not prove PCC as a final method.",
        "primary_metrics": [
            "ES_target_score_drop",
            "Norm_ES_target_score_drop",
            "ES_anchor_score_drop"
        ],
        "secondary_metric": "ES_global_l1"
    },
    "folders": [str(p) for p in folders_to_save],
}

manifest_path = backup_root / "PCC_third_layer_audit_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(manifest_path, manifest_path.name)

    for folder in folders_to_save:
        if folder.exists():
            for file in folder.rglob("*"):
                if file.is_file():
                    arcname = file.relative_to(backup_root)
                    z.write(file, arcname)
        else:
            print("Missing folder:", folder)

print("Backup created:")
print(zip_path)
print("Size MB:", zip_path.stat().st_size / 1024 / 1024)

print("\nIncluded files:")
with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print(name)

In [ ]:
# ============================================================
# Layer 1 Ultra-Fast Smoke Test
#
# Goal:
#   Test whether PCC-style correction can improve CURRENT tumour segmentation.
#
# Task:
#   Input  = current T1c only
#   Target = current tumour mask
#
# Compare:
#   1. Baseline-Seg:
#        T1c -> current mask
#
#   2. PCC-Seg:
#        T1c + Baseline prediction -> corrected current mask
#
# This is only a 5-10 min smoke test:
#   - fold 1 only
#   - 3 epochs baseline
#   - 3 epochs PCC corrector
#   - evaluate on fold 1 test cases
# ============================================================

from pathlib import Path
import json
import time
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


# =====================
# Config
# =====================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FOLD = 1

BASELINE_EPOCHS = 3
PCC_EPOCHS = 3

BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-5

MAX_TRAIN_SLICES = 3500

PCC_MAX_DELTA_LOGIT = 3.0

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
SPLIT_PATH = STAGE2_DIR / "model_A_direct_target_5fold" / "matched_5fold_splits_seed42.csv"

LAYER1_DIR = STAGE2_DIR / "layer1_current_segmentation_smoke_fold1"
LAYER1_DIR.mkdir(parents=True, exist_ok=True)

assert PRE_SUMMARY_PATH.exists(), PRE_SUMMARY_PATH
assert SPLIT_PATH.exists(), SPLIT_PATH

pre_df = pd.read_csv(PRE_SUMMARY_PATH)
split_df = pd.read_csv(SPLIT_PATH)

case_to_npz = dict(zip(pre_df["case_id"], pre_df["npz_path"]))

train_cases = split_df[(split_df["fold"] == FOLD) & (split_df["split"] == "train")]["case_id"].tolist()
test_cases = split_df[(split_df["fold"] == FOLD) & (split_df["split"] == "test")]["case_id"].tolist()

print("Train cases:", len(train_cases))
print("Test cases:", len(test_cases))
print("Output:", LAYER1_DIR)


# =====================
# Data loading
# =====================
def load_case_seg(case_id):
    """
    Return:
      X_t1c: [Z, 1, H, W]
      Y_cur: [Z, 1, H, W]
    """
    path = case_to_npz[case_id]
    with np.load(path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)

    # X[:,0] = current T1c
    # X[:,1] = current tumour mask
    X_t1c = X[:, 0:1].astype(np.float32)
    Y_cur = X[:, 1:2].astype(np.float32)

    return X_t1c, Y_cur


def build_train_arrays(case_ids, max_slices=3500):
    xs, ys = [], []

    for cid in case_ids:
        x, y = load_case_seg(cid)
        xs.append(x)
        ys.append(y)

    X = np.concatenate(xs, axis=0)
    Y = np.concatenate(ys, axis=0)

    # Balanced-ish sampling for speed
    voxel_sum = Y.reshape(Y.shape[0], -1).sum(axis=1)
    pos_idx = np.where(voxel_sum > 0)[0]
    neg_idx = np.where(voxel_sum == 0)[0]

    rng = np.random.default_rng(SEED)

    if max_slices is not None and len(X) > max_slices:
        n_pos = min(len(pos_idx), max_slices // 2)
        n_neg = max_slices - n_pos

        chosen_pos = rng.choice(pos_idx, size=n_pos, replace=False) if len(pos_idx) > n_pos else pos_idx
        chosen_neg = rng.choice(neg_idx, size=min(len(neg_idx), n_neg), replace=False) if len(neg_idx) > 0 else np.array([], dtype=int)

        chosen = np.concatenate([chosen_pos, chosen_neg])
        rng.shuffle(chosen)

        X = X[chosen]
        Y = Y[chosen]

    return X.astype(np.float32), Y.astype(np.float32)


X_train, Y_train = build_train_arrays(train_cases, max_slices=MAX_TRAIN_SLICES)

print("Train slices used:", X_train.shape[0])
print("Input shape:", X_train.shape)
print("Target shape:", Y_train.shape)
print("Positive target voxels:", int(Y_train.sum()))


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


# =====================
# Loss and metrics
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)

    dice = (2 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def make_loss_fn(Y_train):
    pos = float(Y_train.sum())
    total = float(Y_train.size)
    neg = total - pos

    pos_weight_value = neg / max(pos, 1.0)
    pos_weight_value = min(pos_weight_value, 50.0)

    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

    print("BCE pos_weight:", pos_weight_value)

    def loss_fn(logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)
        dice = soft_dice_loss_from_logits(logits, targets)
        return 0.5 * bce + 0.5 * dice

    return loss_fn


def dice_iou_np(prob, target, threshold=0.5, eps=1e-8):
    pred = prob >= threshold
    target = target.astype(bool)

    inter = np.logical_and(pred, target).sum()
    pred_sum = pred.sum()
    target_sum = target.sum()
    union = np.logical_or(pred, target).sum()

    dice = (2 * inter + eps) / (pred_sum + target_sum + eps)
    iou = (inter + eps) / (union + eps)

    return float(dice), float(iou)


def prob_to_logit_np(p, eps=1e-4):
    p = np.clip(p.astype(np.float32), eps, 1.0 - eps)
    return np.log(p / (1.0 - p)).astype(np.float32)


def sigmoid_np(x):
    x = np.clip(x, -30, 30)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)


# =====================
# Training helpers
# =====================
def train_baseline_model(X_train, Y_train):
    model = SmallUNet2D(in_ch=1, out_ch=1, base=16).to(DEVICE)

    ds = TensorDataset(
        torch.from_numpy(X_train).float(),
        torch.from_numpy(Y_train).float()
    )

    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = make_loss_fn(Y_train)

    losses = []

    for epoch in range(1, BASELINE_EPOCHS + 1):
        model.train()
        running = 0.0

        for xb, yb in tqdm(dl, desc=f"Baseline epoch {epoch}/{BASELINE_EPOCHS}"):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            running += loss.item() * xb.size(0)

        epoch_loss = running / len(ds)
        losses.append({"model": "baseline", "epoch": epoch, "loss": epoch_loss})
        print(f"Baseline epoch {epoch}: loss={epoch_loss:.5f}")

    return model, losses


@torch.no_grad()
def predict_prob(model, X, batch_size=BATCH_SIZE):
    model.eval()
    parts = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X[start:start+batch_size]).float().to(DEVICE)
        logits = model(xb)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        parts.append(probs.astype(np.float32))

    return np.concatenate(parts, axis=0)


def train_pcc_corrector(X_train, Y_train, base_prob_train):
    """
    PCC-Seg:
      input  = T1c + baseline probability
      output = correction residual
      final_logit = logit(base_prob) + max_delta * tanh(residual)
    """
    pcc_model = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)

    X_pcc = np.concatenate([X_train, base_prob_train.astype(np.float32)], axis=1).astype(np.float32)

    ds = TensorDataset(
        torch.from_numpy(X_pcc).float(),
        torch.from_numpy(Y_train).float(),
        torch.from_numpy(base_prob_train).float(),
    )

    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    opt = torch.optim.AdamW(pcc_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = make_loss_fn(Y_train)

    losses = []

    for epoch in range(1, PCC_EPOCHS + 1):
        pcc_model.train()
        running = 0.0

        for xb, yb, basep in tqdm(dl, desc=f"PCC epoch {epoch}/{PCC_EPOCHS}"):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            basep = basep.to(DEVICE, non_blocking=True)

            base_logit = torch.logit(torch.clamp(basep, 1e-4, 1.0 - 1e-4))

            opt.zero_grad(set_to_none=True)

            residual_raw = pcc_model(xb)
            corrected_logits = base_logit + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)

            loss = loss_fn(corrected_logits, yb)

            # Small regularization: avoid wild correction
            loss = loss + 0.002 * torch.mean(torch.abs(torch.tanh(residual_raw)))

            loss.backward()
            opt.step()

            running += loss.item() * xb.size(0)

        epoch_loss = running / len(ds)
        losses.append({"model": "pcc_corrector", "epoch": epoch, "loss": epoch_loss})
        print(f"PCC epoch {epoch}: loss={epoch_loss:.5f}")

    return pcc_model, losses


@torch.no_grad()
def predict_pcc_corrected(pcc_model, X, base_prob, batch_size=BATCH_SIZE):
    pcc_model.eval()

    X_pcc = np.concatenate([X, base_prob.astype(np.float32)], axis=1).astype(np.float32)
    parts = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X_pcc[start:start+batch_size]).float().to(DEVICE)
        bp = torch.from_numpy(base_prob[start:start+batch_size]).float().to(DEVICE)

        base_logit = torch.logit(torch.clamp(bp, 1e-4, 1.0 - 1e-4))
        residual_raw = pcc_model(xb)
        corrected_logits = base_logit + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)
        corrected_prob = torch.sigmoid(corrected_logits).detach().cpu().numpy()

        parts.append(corrected_prob.astype(np.float32))

    return np.concatenate(parts, axis=0)


def find_best_threshold(prob, target):
    thresholds = np.arange(0.10, 0.91, 0.05)

    best_t = 0.5
    best_dice = -1

    for t in thresholds:
        d, _ = dice_iou_np(prob, target, threshold=t)
        if d > best_dice:
            best_dice = d
            best_t = float(t)

    return best_t, float(best_dice)


# =====================
# Run experiment
# =====================
start_time = time.time()

print("\n" + "=" * 80)
print("Training Baseline-Seg...")
baseline_model, baseline_losses = train_baseline_model(X_train, Y_train)

print("\nGenerating baseline train predictions...")
base_prob_train = predict_prob(baseline_model, X_train)

baseline_threshold, baseline_train_dice = find_best_threshold(base_prob_train, Y_train)
print("Baseline selected threshold:", baseline_threshold)
print("Baseline train Dice at threshold:", baseline_train_dice)

print("\n" + "=" * 80)
print("Training PCC-Seg corrector...")
pcc_model, pcc_losses = train_pcc_corrector(X_train, Y_train, base_prob_train)

print("\nGenerating PCC train predictions...")
pcc_prob_train = predict_pcc_corrected(pcc_model, X_train, base_prob_train)

pcc_threshold, pcc_train_dice = find_best_threshold(pcc_prob_train, Y_train)
print("PCC selected threshold:", pcc_threshold)
print("PCC train Dice at threshold:", pcc_train_dice)


# =====================
# Evaluate on fold-1 test cases
# =====================
case_rows = []

print("\nEvaluating on fold-1 test cases...")

for cid in tqdm(test_cases, desc="Test cases"):
    X_test, Y_test = load_case_seg(cid)

    base_prob = predict_prob(baseline_model, X_test)
    pcc_prob = predict_pcc_corrected(pcc_model, X_test, base_prob)

    base_dice, base_iou = dice_iou_np(base_prob, Y_test, threshold=baseline_threshold)
    pcc_dice, pcc_iou = dice_iou_np(pcc_prob, Y_test, threshold=pcc_threshold)

    # Also report fixed 0.5 for sanity
    base_dice_05, base_iou_05 = dice_iou_np(base_prob, Y_test, threshold=0.5)
    pcc_dice_05, pcc_iou_05 = dice_iou_np(pcc_prob, Y_test, threshold=0.5)

    case_rows.append({
        "case_id": cid,
        "fold": FOLD,

        "baseline_threshold": baseline_threshold,
        "pcc_threshold": pcc_threshold,

        "baseline_dice": base_dice,
        "pcc_dice": pcc_dice,
        "dice_gain": pcc_dice - base_dice,

        "baseline_iou": base_iou,
        "pcc_iou": pcc_iou,
        "iou_gain": pcc_iou - base_iou,

        "baseline_dice_fixed05": base_dice_05,
        "pcc_dice_fixed05": pcc_dice_05,
        "dice_gain_fixed05": pcc_dice_05 - base_dice_05,

        "baseline_iou_fixed05": base_iou_05,
        "pcc_iou_fixed05": pcc_iou_05,
        "iou_gain_fixed05": pcc_iou_05 - base_iou_05,

        "target_voxels": int(Y_test.sum()),
    })

case_df = pd.DataFrame(case_rows)

summary = {
    "fold": FOLD,
    "train_cases": len(train_cases),
    "test_cases": len(test_cases),
    "train_slices_used": int(X_train.shape[0]),
    "baseline_epochs": BASELINE_EPOCHS,
    "pcc_epochs": PCC_EPOCHS,
    "baseline_threshold": baseline_threshold,
    "pcc_threshold": pcc_threshold,

    "mean_baseline_dice": float(case_df["baseline_dice"].mean()),
    "mean_pcc_dice": float(case_df["pcc_dice"].mean()),
    "mean_dice_gain": float(case_df["dice_gain"].mean()),
    "median_dice_gain": float(case_df["dice_gain"].median()),
    "dice_wins": int((case_df["dice_gain"] > 0).sum()),
    "dice_losses": int((case_df["dice_gain"] < 0).sum()),
    "dice_win_rate": float((case_df["dice_gain"] > 0).mean()),

    "mean_baseline_iou": float(case_df["baseline_iou"].mean()),
    "mean_pcc_iou": float(case_df["pcc_iou"].mean()),
    "mean_iou_gain": float(case_df["iou_gain"].mean()),
    "median_iou_gain": float(case_df["iou_gain"].median()),
    "iou_wins": int((case_df["iou_gain"] > 0).sum()),
    "iou_losses": int((case_df["iou_gain"] < 0).sum()),
    "iou_win_rate": float((case_df["iou_gain"] > 0).mean()),

    "mean_baseline_dice_fixed05": float(case_df["baseline_dice_fixed05"].mean()),
    "mean_pcc_dice_fixed05": float(case_df["pcc_dice_fixed05"].mean()),
    "mean_dice_gain_fixed05": float(case_df["dice_gain_fixed05"].mean()),

    "runtime_minutes": float((time.time() - start_time) / 60.0),
}

summary_df = pd.DataFrame([summary])

# Save outputs
case_path = LAYER1_DIR / "layer1_smoke_fold1_case_metrics.csv"
summary_path = LAYER1_DIR / "layer1_smoke_fold1_summary.csv"
loss_path = LAYER1_DIR / "layer1_smoke_training_losses.csv"
run_info_path = LAYER1_DIR / "layer1_smoke_run_info.json"

case_df.to_csv(case_path, index=False)
summary_df.to_csv(summary_path, index=False)
pd.DataFrame(baseline_losses + pcc_losses).to_csv(loss_path, index=False)

with open(run_info_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

torch.save(
    {
        "fold": FOLD,
        "model_state_dict": baseline_model.state_dict(),
        "threshold": baseline_threshold,
        "config": {
            "input": "current T1c only",
            "target": "current tumour mask",
            "epochs": BASELINE_EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
        }
    },
    LAYER1_DIR / "baseline_seg_fold1_smoke.pt"
)

torch.save(
    {
        "fold": FOLD,
        "model_state_dict": pcc_model.state_dict(),
        "threshold": pcc_threshold,
        "config": {
            "input": "current T1c + baseline prediction",
            "target": "current tumour mask",
            "method": "PCC-style residual correction",
            "epochs": PCC_EPOCHS,
            "max_delta_logit": PCC_MAX_DELTA_LOGIT,
            "batch_size": BATCH_SIZE,
            "lr": LR,
        }
    },
    LAYER1_DIR / "pcc_seg_corrector_fold1_smoke.pt"
)

print("\n" + "=" * 80)
print("Layer 1 smoke test finished.")
print("Runtime minutes:", summary["runtime_minutes"])

print("\nCase metrics:")
display(case_df)

print("\nSummary:")
display(summary_df)

print("\nSaved files:")
print(case_path)
print(summary_path)
print(loss_path)
print(run_info_path)
print(LAYER1_DIR / "baseline_seg_fold1_smoke.pt")
print(LAYER1_DIR / "pcc_seg_corrector_fold1_smoke.pt")

In [ ]:
from pathlib import Path
import os

print("=== Kaggle input datasets ===")
for p in Path("/kaggle/input").iterdir():
    print(p)

print("\n=== Working directory check ===")

WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

paths = {
    "OUT_DIR": OUT_DIR,
    "STAGE2_DIR": STAGE2_DIR,
    "preprocessed summary": OUT_DIR / "preprocessed_locked40_2d_summary.csv",
    "preprocessed npz dir": OUT_DIR / "preprocessed_locked40_2d_npz",
    "split file": STAGE2_DIR / "model_A_direct_target_5fold" / "matched_5fold_splits_seed42.csv",

    "Model A dir": STAGE2_DIR / "model_A_direct_target_5fold",
    "C2-v2 dir": STAGE2_DIR / "model_C2_v2_conservative_oof_gated_quick_5epoch",

    "FAST40 third-layer audit": STAGE2_DIR / "FAST40_boundary_anchor_audit_modelA_vs_C2v2",
    "Layer1 smoke fold1": STAGE2_DIR / "layer1_current_segmentation_smoke_fold1",
}

for name, p in paths.items():
    print(f"{name:35s}: {p.exists()}  -> {p}")

print("\n=== Important file counts ===")

npz_dir = OUT_DIR / "preprocessed_locked40_2d_npz"
if npz_dir.exists():
    print("npz files:", len(list(npz_dir.glob("*.npz"))))
else:
    print("npz files: missing")

model_a_ckpt_dir = STAGE2_DIR / "model_A_direct_target_5fold" / "checkpoints"
if model_a_ckpt_dir.exists():
    print("Model A checkpoints:", len(list(model_a_ckpt_dir.glob("*.pt"))))
else:
    print("Model A checkpoints: missing")

c2_ckpt_dir = STAGE2_DIR / "model_C2_v2_conservative_oof_gated_quick_5epoch" / "checkpoints"
if c2_ckpt_dir.exists():
    print("C2-v2 checkpoints:", len(list(c2_ckpt_dir.glob("*.pt"))))
else:
    print("C2-v2 checkpoints: missing")

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("=== /kaggle/input top-level ===")
for p in INPUT_ROOT.iterdir():
    print(p)

print("\n=== Search important files/directories ===")

patterns = [
    "*.nii",
    "*.nii.gz",
    "*.npz",
    "*.csv",
    "*.json",
    "*.zip",
    "*.pt",
]

for pattern in patterns:
    files = list(INPUT_ROOT.rglob(pattern))
    print(f"\nPattern {pattern}: {len(files)} found")
    for f in files[:30]:
        print(" ", f)
    if len(files) > 30:
        print("  ...")

print("\n=== Directory tree, limited depth ===")

def print_tree(root, max_depth=4, current_depth=0, max_items=80):
    if current_depth > max_depth:
        return
    
    try:
        items = sorted(list(root.iterdir()))
    except Exception as e:
        print("Cannot read:", root, e)
        return
    
    for i, item in enumerate(items[:max_items]):
        indent = "  " * current_depth
        print(f"{indent}{item.name}/" if item.is_dir() else f"{indent}{item.name}")
        if item.is_dir():
            print_tree(item, max_depth=max_depth, current_depth=current_depth+1, max_items=max_items)

print_tree(INPUT_ROOT, max_depth=4)

In [ ]:
# ============================================================
# Restore Layer-1 working data after Kaggle /working reset
#
# Goal:
#   Rebuild the minimal data needed for Layer 1:
#     Input  = current T1c
#     Target = current tumour mask
#
# It uses:
#   - raw MU-Glioma NIfTI files from /kaggle/input
#   - matched_5fold_splits_seed42.csv from model-a-backup-final
# ============================================================

from pathlib import Path
import re
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib


# =====================
# Paths
# =====================
RAW_ROOT = Path("/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post")

SPLIT_SRC = Path("/kaggle/input/datasets/jeechangxin/model-a-backup-final/matched_5fold_splits_seed42.csv")

WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
NPZ_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"

OUT_DIR.mkdir(parents=True, exist_ok=True)
STAGE2_DIR.mkdir(parents=True, exist_ok=True)
MODEL_A_DIR.mkdir(parents=True, exist_ok=True)
NPZ_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DST = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"
PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

assert RAW_ROOT.exists(), RAW_ROOT
assert SPLIT_SRC.exists(), SPLIT_SRC

shutil.copy2(SPLIT_SRC, SPLIT_DST)

print("Raw root:", RAW_ROOT)
print("Split copied to:", SPLIT_DST)
print("NPZ output:", NPZ_DIR)


# =====================
# Helpers
# =====================
def parse_case_id(case_id):
    """
    Example:
      PatientID_0008_T4_to_T6_t1c
    """
    m = re.match(r"(PatientID_\d+)_T(\d+)_to_T(\d+)_t1c", case_id)
    if m is None:
        raise ValueError(f"Cannot parse case_id: {case_id}")
    
    patient_id = m.group(1)
    current_tp = int(m.group(2))
    future_tp = int(m.group(3))
    return patient_id, current_tp, future_tp


def load_nii_as_zhw(path):
    """
    Load NIfTI and return [Z, H, W].
    MU-Glioma files are usually [H, W, Z].
    """
    img = nib.load(str(path))
    arr = img.get_fdata().astype(np.float32)

    if arr.ndim == 4:
        arr = arr[..., 0]

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    # Assume final axis is slice axis
    arr = np.moveaxis(arr, -1, 0)  # [Z, H, W]
    return arr.astype(np.float32)


def normalize_volume(vol):
    """
    Robust per-volume normalization to [0, 1].
    """
    vol = vol.astype(np.float32)
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)

    nonzero = vol[vol > 0]

    if nonzero.size < 100:
        mn, mx = float(vol.min()), float(vol.max())
    else:
        mn, mx = np.percentile(nonzero, [1, 99])

    if mx <= mn:
        return np.zeros_like(vol, dtype=np.float32)

    out = (vol - mn) / (mx - mn + 1e-8)
    out = np.clip(out, 0.0, 1.0)
    return out.astype(np.float32)


def get_paths_for_case(case_id):
    patient_id, current_tp, future_tp = parse_case_id(case_id)

    tp_dir = RAW_ROOT / patient_id / f"Timepoint_{current_tp}"

    t1c_path = tp_dir / f"{patient_id}_Timepoint_{current_tp}_brain_t1c.nii"
    mask_path = tp_dir / f"{patient_id}_Timepoint_{current_tp}_tumorMask.nii"

    if not t1c_path.exists():
        raise FileNotFoundError(t1c_path)
    if not mask_path.exists():
        raise FileNotFoundError(mask_path)

    return patient_id, current_tp, future_tp, t1c_path, mask_path


# =====================
# Build locked 40 case list from split file
# =====================
split_df = pd.read_csv(SPLIT_DST)
case_ids = sorted(split_df["case_id"].unique())

print("Unique locked cases:", len(case_ids))
print(split_df.head())


# =====================
# Preprocess
# =====================
rows = []

for case_id in tqdm(case_ids, desc="Preprocessing locked 40 current segmentation data"):
    patient_id, current_tp, future_tp, t1c_path, mask_path = get_paths_for_case(case_id)

    t1c = load_nii_as_zhw(t1c_path)
    mask = load_nii_as_zhw(mask_path)

    t1c = normalize_volume(t1c)
    mask = (mask > 0).astype(np.float32)

    assert t1c.shape == mask.shape, (case_id, t1c.shape, mask.shape)

    # X format remains compatible with previous code:
    # X[:, 0] = current T1c
    # X[:, 1] = current tumour mask
    X = np.stack([t1c, mask], axis=1).astype(np.float32)  # [Z, 2, H, W]

    out_path = NPZ_DIR / f"{case_id}.npz"
    np.savez_compressed(
        out_path,
        X=X,
        case_id=case_id,
        patient_id=patient_id,
        current_timepoint=current_tp,
        future_timepoint=future_tp,
    )

    rows.append({
        "case_id": case_id,
        "patient_id": patient_id,
        "current_timepoint": current_tp,
        "future_timepoint": future_tp,
        "npz_path": str(out_path),
        "z_slices": int(X.shape[0]),
        "height": int(X.shape[2]),
        "width": int(X.shape[3]),
        "current_mask_voxels": int(mask.sum()),
        "positive_slices": int((mask.reshape(mask.shape[0], -1).sum(axis=1) > 0).sum()),
        "t1c_path": str(t1c_path),
        "mask_path": str(mask_path),
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(PRE_SUMMARY_PATH, index=False)

print("\nDone.")
print("Saved summary:", PRE_SUMMARY_PATH)
print("Saved npz dir:", NPZ_DIR)
print("NPZ files:", len(list(NPZ_DIR.glob('*.npz'))))

print("\nSummary:")
display(summary_df.head())
display(summary_df[["z_slices", "current_mask_voxels", "positive_slices"]].describe())

print("\nCheck paths:")
print("preprocessed summary exists:", PRE_SUMMARY_PATH.exists())
print("split file exists:", SPLIT_DST.exists())
print("npz count:", len(list(NPZ_DIR.glob('*.npz'))))

In [ ]:
# ============================================================
# Layer 1 Quick 5-Fold Current Tumour Segmentation Experiment
#
# Goal:
#   Test whether PCC-style correction improves CURRENT tumour segmentation.
#
# Task:
#   Input  = current T1c only
#   Target = current tumour mask
#
# Compare:
#   1. Baseline-Seg:
#        T1c -> current tumour mask
#
#   2. PCC-Seg:
#        T1c + baseline prediction -> corrected current tumour mask
#
# Setting:
#   - 5 folds
#   - 3 epochs baseline per fold
#   - 3 epochs PCC corrector per fold
#   - max 3500 train slices per fold
#   - report both train-calibrated threshold and fixed 0.5 threshold
# ============================================================

from pathlib import Path
import json
import time
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


# =====================
# Config
# =====================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FOLDS = [1, 2, 3, 4, 5]

BASELINE_EPOCHS = 3
PCC_EPOCHS = 3

BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-5

MAX_TRAIN_SLICES = 3500
PCC_MAX_DELTA_LOGIT = 3.0

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
SPLIT_PATH = STAGE2_DIR / "model_A_direct_target_5fold" / "matched_5fold_splits_seed42.csv"

LAYER1_DIR = STAGE2_DIR / "layer1_current_segmentation_quick5fold"
CKPT_DIR = LAYER1_DIR / "checkpoints"

LAYER1_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert PRE_SUMMARY_PATH.exists(), PRE_SUMMARY_PATH
assert SPLIT_PATH.exists(), SPLIT_PATH

pre_df = pd.read_csv(PRE_SUMMARY_PATH)
split_df = pd.read_csv(SPLIT_PATH)

case_to_npz = dict(zip(pre_df["case_id"], pre_df["npz_path"]))

print("Output:", LAYER1_DIR)
print("Total cases:", len(case_to_npz))


# =====================
# Data loading
# =====================
def load_case_seg(case_id):
    """
    Return:
      X_t1c: [Z, 1, H, W]
      Y_cur: [Z, 1, H, W]
    """
    path = case_to_npz[case_id]
    with np.load(path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)

    X_t1c = X[:, 0:1].astype(np.float32)
    Y_cur = X[:, 1:2].astype(np.float32)

    return X_t1c, Y_cur


def build_train_arrays(case_ids, fold_seed, max_slices=3500):
    xs, ys = [], []

    for cid in case_ids:
        x, y = load_case_seg(cid)
        xs.append(x)
        ys.append(y)

    X = np.concatenate(xs, axis=0)
    Y = np.concatenate(ys, axis=0)

    voxel_sum = Y.reshape(Y.shape[0], -1).sum(axis=1)
    pos_idx = np.where(voxel_sum > 0)[0]
    neg_idx = np.where(voxel_sum == 0)[0]

    rng = np.random.default_rng(fold_seed)

    if max_slices is not None and len(X) > max_slices:
        n_pos = min(len(pos_idx), max_slices // 2)
        n_neg = max_slices - n_pos

        chosen_pos = rng.choice(pos_idx, size=n_pos, replace=False) if len(pos_idx) > n_pos else pos_idx

        if len(neg_idx) > 0:
            chosen_neg = rng.choice(neg_idx, size=min(len(neg_idx), n_neg), replace=False)
        else:
            chosen_neg = np.array([], dtype=int)

        chosen = np.concatenate([chosen_pos, chosen_neg])
        rng.shuffle(chosen)

        X = X[chosen]
        Y = Y[chosen]

    return X.astype(np.float32), Y.astype(np.float32)


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


# =====================
# Loss / metrics
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)

    dice = (2 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def make_loss_fn(Y_train):
    pos = float(Y_train.sum())
    total = float(Y_train.size)
    neg = total - pos

    pos_weight_value = neg / max(pos, 1.0)
    pos_weight_value = min(pos_weight_value, 50.0)

    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

    print("BCE pos_weight:", pos_weight_value)

    def loss_fn(logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)
        dice = soft_dice_loss_from_logits(logits, targets)
        return 0.5 * bce + 0.5 * dice

    return loss_fn


def dice_iou_np(prob, target, threshold=0.5, eps=1e-8):
    pred = prob >= threshold
    target = target.astype(bool)

    inter = np.logical_and(pred, target).sum()
    pred_sum = pred.sum()
    target_sum = target.sum()
    union = np.logical_or(pred, target).sum()

    dice = (2 * inter + eps) / (pred_sum + target_sum + eps)
    iou = (inter + eps) / (union + eps)

    return float(dice), float(iou)


def find_best_threshold(prob, target):
    thresholds = np.arange(0.10, 0.91, 0.05)

    best_t = 0.5
    best_dice = -1.0

    for t in thresholds:
        d, _ = dice_iou_np(prob, target, threshold=t)
        if d > best_dice:
            best_dice = d
            best_t = float(t)

    return best_t, float(best_dice)


# =====================
# Training / prediction helpers
# =====================
def train_baseline_model(X_train, Y_train, fold):
    model = SmallUNet2D(in_ch=1, out_ch=1, base=16).to(DEVICE)

    ds = TensorDataset(
        torch.from_numpy(X_train).float(),
        torch.from_numpy(Y_train).float()
    )

    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = make_loss_fn(Y_train)

    losses = []

    for epoch in range(1, BASELINE_EPOCHS + 1):
        model.train()
        running = 0.0

        for xb, yb in tqdm(dl, desc=f"Fold {fold} Baseline epoch {epoch}/{BASELINE_EPOCHS}"):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            running += loss.item() * xb.size(0)

        epoch_loss = running / len(ds)
        losses.append({
            "fold": fold,
            "model": "baseline",
            "epoch": epoch,
            "loss": epoch_loss,
        })
        print(f"Fold {fold} Baseline epoch {epoch}: loss={epoch_loss:.5f}")

    return model, losses


@torch.no_grad()
def predict_prob(model, X, batch_size=BATCH_SIZE):
    model.eval()
    parts = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X[start:start + batch_size]).float().to(DEVICE)
        logits = model(xb)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        parts.append(probs.astype(np.float32))

    return np.concatenate(parts, axis=0)


def train_pcc_corrector(X_train, Y_train, base_prob_train, fold):
    """
    PCC-Seg:
      input  = current T1c + baseline probability
      output = residual correction
      final_logit = logit(base_prob) + max_delta * tanh(residual)
    """
    pcc_model = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)

    X_pcc = np.concatenate(
        [X_train, base_prob_train.astype(np.float32)],
        axis=1
    ).astype(np.float32)

    ds = TensorDataset(
        torch.from_numpy(X_pcc).float(),
        torch.from_numpy(Y_train).float(),
        torch.from_numpy(base_prob_train).float(),
    )

    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    opt = torch.optim.AdamW(pcc_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = make_loss_fn(Y_train)

    losses = []

    for epoch in range(1, PCC_EPOCHS + 1):
        pcc_model.train()
        running = 0.0

        for xb, yb, basep in tqdm(dl, desc=f"Fold {fold} PCC epoch {epoch}/{PCC_EPOCHS}"):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            basep = basep.to(DEVICE, non_blocking=True)

            base_logit = torch.logit(torch.clamp(basep, 1e-4, 1.0 - 1e-4))

            opt.zero_grad(set_to_none=True)

            residual_raw = pcc_model(xb)
            corrected_logits = base_logit + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)

            loss = loss_fn(corrected_logits, yb)

            # Conservative correction regularization
            loss = loss + 0.002 * torch.mean(torch.abs(torch.tanh(residual_raw)))

            loss.backward()
            opt.step()

            running += loss.item() * xb.size(0)

        epoch_loss = running / len(ds)
        losses.append({
            "fold": fold,
            "model": "pcc_corrector",
            "epoch": epoch,
            "loss": epoch_loss,
        })
        print(f"Fold {fold} PCC epoch {epoch}: loss={epoch_loss:.5f}")

    return pcc_model, losses


@torch.no_grad()
def predict_pcc_corrected(pcc_model, X, base_prob, batch_size=BATCH_SIZE):
    pcc_model.eval()

    X_pcc = np.concatenate(
        [X, base_prob.astype(np.float32)],
        axis=1
    ).astype(np.float32)

    parts = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X_pcc[start:start + batch_size]).float().to(DEVICE)
        bp = torch.from_numpy(base_prob[start:start + batch_size]).float().to(DEVICE)

        base_logit = torch.logit(torch.clamp(bp, 1e-4, 1.0 - 1e-4))
        residual_raw = pcc_model(xb)
        corrected_logits = base_logit + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)
        corrected_prob = torch.sigmoid(corrected_logits).detach().cpu().numpy()

        parts.append(corrected_prob.astype(np.float32))

    return np.concatenate(parts, axis=0)


# =====================
# Main 5-fold experiment
# =====================
start_time = time.time()

all_case_rows = []
all_fold_rows = []
all_losses = []

for fold in FOLDS:
    print("\n" + "=" * 100)
    print(f"START FOLD {fold}")

    train_cases = split_df[(split_df["fold"] == fold) & (split_df["split"] == "train")]["case_id"].tolist()
    test_cases = split_df[(split_df["fold"] == fold) & (split_df["split"] == "test")]["case_id"].tolist()

    print("Train cases:", len(train_cases))
    print("Test cases:", len(test_cases))

    X_train, Y_train = build_train_arrays(
        train_cases,
        fold_seed=SEED + fold,
        max_slices=MAX_TRAIN_SLICES
    )

    print("Train slices used:", X_train.shape[0])
    print("Input shape:", X_train.shape)
    print("Target shape:", Y_train.shape)
    print("Positive target voxels:", int(Y_train.sum()))

    # ---- Baseline ----
    print("\nTraining Baseline-Seg...")
    baseline_model, baseline_losses = train_baseline_model(X_train, Y_train, fold)
    all_losses.extend(baseline_losses)

    print("\nGenerating baseline train predictions...")
    base_prob_train = predict_prob(baseline_model, X_train)

    baseline_threshold, baseline_train_dice = find_best_threshold(base_prob_train, Y_train)

    print("Baseline selected threshold:", baseline_threshold)
    print("Baseline train Dice:", baseline_train_dice)

    # ---- PCC corrector ----
    print("\nTraining PCC-Seg corrector...")
    pcc_model, pcc_losses = train_pcc_corrector(X_train, Y_train, base_prob_train, fold)
    all_losses.extend(pcc_losses)

    print("\nGenerating PCC train predictions...")
    pcc_prob_train = predict_pcc_corrected(pcc_model, X_train, base_prob_train)

    pcc_threshold, pcc_train_dice = find_best_threshold(pcc_prob_train, Y_train)

    print("PCC selected threshold:", pcc_threshold)
    print("PCC train Dice:", pcc_train_dice)

    # ---- Evaluate ----
    fold_case_rows = []

    print("\nEvaluating fold test cases...")

    for cid in tqdm(test_cases, desc=f"Fold {fold} test cases"):
        X_test, Y_test = load_case_seg(cid)

        base_prob = predict_prob(baseline_model, X_test)
        pcc_prob = predict_pcc_corrected(pcc_model, X_test, base_prob)

        # Train-calibrated thresholds
        base_dice, base_iou = dice_iou_np(base_prob, Y_test, threshold=baseline_threshold)
        pcc_dice, pcc_iou = dice_iou_np(pcc_prob, Y_test, threshold=pcc_threshold)

        # Fixed 0.5 threshold
        base_dice_05, base_iou_05 = dice_iou_np(base_prob, Y_test, threshold=0.5)
        pcc_dice_05, pcc_iou_05 = dice_iou_np(pcc_prob, Y_test, threshold=0.5)

        row = {
            "case_id": cid,
            "fold": fold,

            "baseline_threshold": baseline_threshold,
            "pcc_threshold": pcc_threshold,
            "baseline_train_dice": baseline_train_dice,
            "pcc_train_dice": pcc_train_dice,

            "baseline_dice": base_dice,
            "pcc_dice": pcc_dice,
            "dice_gain": pcc_dice - base_dice,

            "baseline_iou": base_iou,
            "pcc_iou": pcc_iou,
            "iou_gain": pcc_iou - base_iou,

            "baseline_dice_fixed05": base_dice_05,
            "pcc_dice_fixed05": pcc_dice_05,
            "dice_gain_fixed05": pcc_dice_05 - base_dice_05,

            "baseline_iou_fixed05": base_iou_05,
            "pcc_iou_fixed05": pcc_iou_05,
            "iou_gain_fixed05": pcc_iou_05 - base_iou_05,

            "target_voxels": int(Y_test.sum()),
        }

        fold_case_rows.append(row)
        all_case_rows.append(row)

    fold_df = pd.DataFrame(fold_case_rows)

    fold_summary = {
        "fold": fold,
        "train_cases": len(train_cases),
        "test_cases": len(test_cases),
        "train_slices_used": int(X_train.shape[0]),

        "baseline_threshold": baseline_threshold,
        "pcc_threshold": pcc_threshold,
        "baseline_train_dice": baseline_train_dice,
        "pcc_train_dice": pcc_train_dice,

        "mean_baseline_dice": float(fold_df["baseline_dice"].mean()),
        "mean_pcc_dice": float(fold_df["pcc_dice"].mean()),
        "mean_dice_gain": float(fold_df["dice_gain"].mean()),
        "median_dice_gain": float(fold_df["dice_gain"].median()),
        "dice_wins": int((fold_df["dice_gain"] > 0).sum()),
        "dice_losses": int((fold_df["dice_gain"] < 0).sum()),
        "dice_win_rate": float((fold_df["dice_gain"] > 0).mean()),

        "mean_baseline_iou": float(fold_df["baseline_iou"].mean()),
        "mean_pcc_iou": float(fold_df["pcc_iou"].mean()),
        "mean_iou_gain": float(fold_df["iou_gain"].mean()),
        "median_iou_gain": float(fold_df["iou_gain"].median()),
        "iou_wins": int((fold_df["iou_gain"] > 0).sum()),
        "iou_losses": int((fold_df["iou_gain"] < 0).sum()),
        "iou_win_rate": float((fold_df["iou_gain"] > 0).mean()),

        "mean_baseline_dice_fixed05": float(fold_df["baseline_dice_fixed05"].mean()),
        "mean_pcc_dice_fixed05": float(fold_df["pcc_dice_fixed05"].mean()),
        "mean_dice_gain_fixed05": float(fold_df["dice_gain_fixed05"].mean()),

        "mean_baseline_iou_fixed05": float(fold_df["baseline_iou_fixed05"].mean()),
        "mean_pcc_iou_fixed05": float(fold_df["pcc_iou_fixed05"].mean()),
        "mean_iou_gain_fixed05": float(fold_df["iou_gain_fixed05"].mean()),
    }

    all_fold_rows.append(fold_summary)

    # ---- Save checkpoints ----
    torch.save(
        {
            "fold": fold,
            "model_state_dict": baseline_model.state_dict(),
            "threshold": baseline_threshold,
            "train_dice": baseline_train_dice,
            "config": {
                "input": "current T1c only",
                "target": "current tumour mask",
                "epochs": BASELINE_EPOCHS,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "max_train_slices": MAX_TRAIN_SLICES,
            }
        },
        CKPT_DIR / f"baseline_seg_fold_{fold}_quick.pt"
    )

    torch.save(
        {
            "fold": fold,
            "model_state_dict": pcc_model.state_dict(),
            "threshold": pcc_threshold,
            "train_dice": pcc_train_dice,
            "config": {
                "input": "current T1c + baseline prediction",
                "target": "current tumour mask",
                "method": "PCC-style residual correction",
                "epochs": PCC_EPOCHS,
                "max_delta_logit": PCC_MAX_DELTA_LOGIT,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "max_train_slices": MAX_TRAIN_SLICES,
            }
        },
        CKPT_DIR / f"pcc_seg_corrector_fold_{fold}_quick.pt"
    )

    # Free memory
    del X_train, Y_train, base_prob_train, pcc_prob_train
    del baseline_model, pcc_model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


# =====================
# Save final results
# =====================
case_df = pd.DataFrame(all_case_rows)
fold_summary_df = pd.DataFrame(all_fold_rows)
loss_df = pd.DataFrame(all_losses)

runtime_minutes = (time.time() - start_time) / 60.0

overall_summary = {
    "experiment": "Layer 1 quick 5-fold current tumour segmentation",
    "baseline": "T1c -> current tumour mask",
    "pcc": "T1c + baseline prediction -> corrected current tumour mask",
    "folds": len(FOLDS),
    "total_test_cases": int(len(case_df)),
    "baseline_epochs": BASELINE_EPOCHS,
    "pcc_epochs": PCC_EPOCHS,
    "max_train_slices": MAX_TRAIN_SLICES,
    "runtime_minutes": float(runtime_minutes),

    "mean_baseline_dice": float(case_df["baseline_dice"].mean()),
    "mean_pcc_dice": float(case_df["pcc_dice"].mean()),
    "mean_dice_gain": float(case_df["dice_gain"].mean()),
    "median_dice_gain": float(case_df["dice_gain"].median()),
    "dice_wins": int((case_df["dice_gain"] > 0).sum()),
    "dice_losses": int((case_df["dice_gain"] < 0).sum()),
    "dice_win_rate": float((case_df["dice_gain"] > 0).mean()),

    "mean_baseline_iou": float(case_df["baseline_iou"].mean()),
    "mean_pcc_iou": float(case_df["pcc_iou"].mean()),
    "mean_iou_gain": float(case_df["iou_gain"].mean()),
    "median_iou_gain": float(case_df["iou_gain"].median()),
    "iou_wins": int((case_df["iou_gain"] > 0).sum()),
    "iou_losses": int((case_df["iou_gain"] < 0).sum()),
    "iou_win_rate": float((case_df["iou_gain"] > 0).mean()),

    "mean_baseline_dice_fixed05": float(case_df["baseline_dice_fixed05"].mean()),
    "mean_pcc_dice_fixed05": float(case_df["pcc_dice_fixed05"].mean()),
    "mean_dice_gain_fixed05": float(case_df["dice_gain_fixed05"].mean()),

    "mean_baseline_iou_fixed05": float(case_df["baseline_iou_fixed05"].mean()),
    "mean_pcc_iou_fixed05": float(case_df["pcc_iou_fixed05"].mean()),
    "mean_iou_gain_fixed05": float(case_df["iou_gain_fixed05"].mean()),
}

# Bootstrap CI for main calibrated metrics
rng = np.random.default_rng(42)
boot_rows = []

for metric in ["dice_gain", "iou_gain", "dice_gain_fixed05", "iou_gain_fixed05"]:
    vals = case_df[metric].values
    vals = vals[np.isfinite(vals)]

    boots = []
    for _ in range(2000):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots.append(sample.mean())

    boot_rows.append({
        "metric": metric,
        "mean": float(vals.mean()),
        "ci95_low": float(np.percentile(boots, 2.5)),
        "ci95_high": float(np.percentile(boots, 97.5)),
        "n": int(len(vals)),
    })

boot_df = pd.DataFrame(boot_rows)

case_path = LAYER1_DIR / "layer1_quick5fold_case_metrics.csv"
fold_summary_path = LAYER1_DIR / "layer1_quick5fold_summary_by_fold.csv"
overall_summary_path = LAYER1_DIR / "layer1_quick5fold_overall_summary.csv"
loss_path = LAYER1_DIR / "layer1_quick5fold_training_losses.csv"
boot_path = LAYER1_DIR / "layer1_quick5fold_bootstrap_CI.csv"
run_info_path = LAYER1_DIR / "layer1_quick5fold_run_info.json"

case_df.to_csv(case_path, index=False)
fold_summary_df.to_csv(fold_summary_path, index=False)
pd.DataFrame([overall_summary]).to_csv(overall_summary_path, index=False)
loss_df.to_csv(loss_path, index=False)
boot_df.to_csv(boot_path, index=False)

with open(run_info_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)

print("\n" + "=" * 100)
print("Layer 1 quick 5-fold finished.")
print("Runtime minutes:", runtime_minutes)

print("\nOverall summary:")
display(pd.DataFrame([overall_summary]))

print("\nSummary by fold:")
display(fold_summary_df)

print("\nBootstrap 95% CI:")
display(boot_df)

print("\nCase metrics:")
display(case_df)

print("\nSaved files:")
print(case_path)
print(fold_summary_path)
print(overall_summary_path)
print(loss_path)
print(boot_path)
print(run_info_path)
print("Checkpoints:", CKPT_DIR)

In [ ]:
# ============================================================
# Layer 1 Formal v1: 5-Fold Current Tumour Segmentation
#
# Goal:
#   Formally test whether PCC-style correction improves
#   CURRENT tumour segmentation.
#
# Task:
#   Input  = current T1c only
#   Target = current tumour mask
#
# Compare:
#   Baseline-Seg:
#       T1c -> current tumour mask
#
#   PCC-Seg:
#       T1c + baseline prediction -> corrected current tumour mask
#
# Formal v1 setting:
#   - 5 folds
#   - 10 epochs baseline per fold
#   - 10 epochs PCC corrector per fold
#   - use all training slices per fold
#   - save checkpoints after every fold
#   - save partial results after every fold
# ============================================================

from pathlib import Path
import json
import time
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None


# =====================
# Formal config
# =====================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FOLDS = [1, 2, 3, 4, 5]

# Formal v1: longer than quick version
BASELINE_EPOCHS = 10
PCC_EPOCHS = 10

BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-5

# Use all train slices. Each fold has about 32 * 155 = 4960 slices.
MAX_TRAIN_SLICES = 999999

PCC_MAX_DELTA_LOGIT = 3.0

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
SPLIT_PATH = STAGE2_DIR / "model_A_direct_target_5fold" / "matched_5fold_splits_seed42.csv"

FORMAL_DIR = STAGE2_DIR / "layer1_current_segmentation_FORMAL_v1_5fold"
CKPT_DIR = FORMAL_DIR / "checkpoints"
PARTIAL_DIR = FORMAL_DIR / "partial_results"

FORMAL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

assert PRE_SUMMARY_PATH.exists(), PRE_SUMMARY_PATH
assert SPLIT_PATH.exists(), SPLIT_PATH

pre_df = pd.read_csv(PRE_SUMMARY_PATH)
split_df = pd.read_csv(SPLIT_PATH)
case_to_npz = dict(zip(pre_df["case_id"], pre_df["npz_path"]))

print("Output:", FORMAL_DIR)
print("Total cases:", len(case_to_npz))
print("Baseline epochs:", BASELINE_EPOCHS)
print("PCC epochs:", PCC_EPOCHS)
print("Max train slices:", MAX_TRAIN_SLICES)


# =====================
# Main formal experiment
# =====================
start_time = time.time()

all_case_rows = []
all_fold_rows = []
all_losses = []

for fold in FOLDS:
    print("\n" + "=" * 100)
    print(f"FORMAL LAYER 1 - START FOLD {fold}")

    fold_start = time.time()

    train_cases = split_df[
        (split_df["fold"] == fold) & (split_df["split"] == "train")
    ]["case_id"].tolist()

    test_cases = split_df[
        (split_df["fold"] == fold) & (split_df["split"] == "test")
    ]["case_id"].tolist()

    print("Train cases:", len(train_cases))
    print("Test cases:", len(test_cases))

    X_train, Y_train = build_train_arrays(
        train_cases,
        fold_seed=SEED + fold,
        max_slices=MAX_TRAIN_SLICES
    )

    print("Train slices used:", X_train.shape[0])
    print("Input shape:", X_train.shape)
    print("Target shape:", Y_train.shape)
    print("Positive target voxels:", int(Y_train.sum()))

    # --------------------------------------------------------
    # 1. Train Baseline-Seg
    # --------------------------------------------------------
    print("\nTraining Baseline-Seg formal v1...")
    baseline_model, baseline_losses = train_baseline_model(X_train, Y_train, fold)
    all_losses.extend(baseline_losses)

    print("\nGenerating baseline train predictions...")
    base_prob_train = predict_prob(baseline_model, X_train)

    baseline_threshold, baseline_train_dice = find_best_threshold(
        base_prob_train,
        Y_train
    )

    print("Baseline selected threshold:", baseline_threshold)
    print("Baseline train Dice:", baseline_train_dice)

    # --------------------------------------------------------
    # 2. Train PCC-Seg corrector
    # --------------------------------------------------------
    print("\nTraining PCC-Seg corrector formal v1...")
    pcc_model, pcc_losses = train_pcc_corrector(
        X_train,
        Y_train,
        base_prob_train,
        fold
    )
    all_losses.extend(pcc_losses)

    print("\nGenerating PCC train predictions...")
    pcc_prob_train = predict_pcc_corrected(
        pcc_model,
        X_train,
        base_prob_train
    )

    pcc_threshold, pcc_train_dice = find_best_threshold(
        pcc_prob_train,
        Y_train
    )

    print("PCC selected threshold:", pcc_threshold)
    print("PCC train Dice:", pcc_train_dice)

    # --------------------------------------------------------
    # 3. Evaluate on fold test cases
    # --------------------------------------------------------
    fold_case_rows = []

    print("\nEvaluating fold test cases...")

    for cid in tqdm(test_cases, desc=f"Fold {fold} test cases"):
        X_test, Y_test = load_case_seg(cid)

        base_prob = predict_prob(baseline_model, X_test)
        pcc_prob = predict_pcc_corrected(pcc_model, X_test, base_prob)

        # Train-calibrated thresholds
        base_dice, base_iou = dice_iou_np(
            base_prob,
            Y_test,
            threshold=baseline_threshold
        )

        pcc_dice, pcc_iou = dice_iou_np(
            pcc_prob,
            Y_test,
            threshold=pcc_threshold
        )

        # Fixed 0.5 threshold
        base_dice_05, base_iou_05 = dice_iou_np(
            base_prob,
            Y_test,
            threshold=0.5
        )

        pcc_dice_05, pcc_iou_05 = dice_iou_np(
            pcc_prob,
            Y_test,
            threshold=0.5
        )

        row = {
            "case_id": cid,
            "fold": fold,

            "baseline_threshold": baseline_threshold,
            "pcc_threshold": pcc_threshold,

            "baseline_train_dice": baseline_train_dice,
            "pcc_train_dice": pcc_train_dice,

            "baseline_dice": base_dice,
            "pcc_dice": pcc_dice,
            "dice_gain": pcc_dice - base_dice,

            "baseline_iou": base_iou,
            "pcc_iou": pcc_iou,
            "iou_gain": pcc_iou - base_iou,

            "baseline_dice_fixed05": base_dice_05,
            "pcc_dice_fixed05": pcc_dice_05,
            "dice_gain_fixed05": pcc_dice_05 - base_dice_05,

            "baseline_iou_fixed05": base_iou_05,
            "pcc_iou_fixed05": pcc_iou_05,
            "iou_gain_fixed05": pcc_iou_05 - base_iou_05,

            "target_voxels": int(Y_test.sum()),
        }

        fold_case_rows.append(row)
        all_case_rows.append(row)

    fold_df = pd.DataFrame(fold_case_rows)

    fold_runtime = (time.time() - fold_start) / 60.0

    fold_summary = {
        "fold": fold,
        "train_cases": len(train_cases),
        "test_cases": len(test_cases),
        "train_slices_used": int(X_train.shape[0]),
        "fold_runtime_minutes": float(fold_runtime),

        "baseline_threshold": baseline_threshold,
        "pcc_threshold": pcc_threshold,

        "baseline_train_dice": baseline_train_dice,
        "pcc_train_dice": pcc_train_dice,

        "mean_baseline_dice": float(fold_df["baseline_dice"].mean()),
        "mean_pcc_dice": float(fold_df["pcc_dice"].mean()),
        "mean_dice_gain": float(fold_df["dice_gain"].mean()),
        "median_dice_gain": float(fold_df["dice_gain"].median()),
        "dice_wins": int((fold_df["dice_gain"] > 0).sum()),
        "dice_losses": int((fold_df["dice_gain"] < 0).sum()),
        "dice_win_rate": float((fold_df["dice_gain"] > 0).mean()),

        "mean_baseline_iou": float(fold_df["baseline_iou"].mean()),
        "mean_pcc_iou": float(fold_df["pcc_iou"].mean()),
        "mean_iou_gain": float(fold_df["iou_gain"].mean()),
        "median_iou_gain": float(fold_df["iou_gain"].median()),
        "iou_wins": int((fold_df["iou_gain"] > 0).sum()),
        "iou_losses": int((fold_df["iou_gain"] < 0).sum()),
        "iou_win_rate": float((fold_df["iou_gain"] > 0).mean()),

        "mean_baseline_dice_fixed05": float(fold_df["baseline_dice_fixed05"].mean()),
        "mean_pcc_dice_fixed05": float(fold_df["pcc_dice_fixed05"].mean()),
        "mean_dice_gain_fixed05": float(fold_df["dice_gain_fixed05"].mean()),

        "mean_baseline_iou_fixed05": float(fold_df["baseline_iou_fixed05"].mean()),
        "mean_pcc_iou_fixed05": float(fold_df["pcc_iou_fixed05"].mean()),
        "mean_iou_gain_fixed05": float(fold_df["iou_gain_fixed05"].mean()),
    }

    all_fold_rows.append(fold_summary)

    # --------------------------------------------------------
    # 4. Save checkpoints after every fold
    # --------------------------------------------------------
    torch.save(
        {
            "fold": fold,
            "model_state_dict": baseline_model.state_dict(),
            "threshold": baseline_threshold,
            "train_dice": baseline_train_dice,
            "config": {
                "experiment": "Layer 1 Formal v1",
                "input": "current T1c only",
                "target": "current tumour mask",
                "epochs": BASELINE_EPOCHS,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "max_train_slices": MAX_TRAIN_SLICES,
            }
        },
        CKPT_DIR / f"baseline_seg_fold_{fold}_FORMAL_v1.pt"
    )

    torch.save(
        {
            "fold": fold,
            "model_state_dict": pcc_model.state_dict(),
            "threshold": pcc_threshold,
            "train_dice": pcc_train_dice,
            "config": {
                "experiment": "Layer 1 Formal v1",
                "input": "current T1c + baseline prediction",
                "target": "current tumour mask",
                "method": "PCC-style residual correction",
                "epochs": PCC_EPOCHS,
                "max_delta_logit": PCC_MAX_DELTA_LOGIT,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "max_train_slices": MAX_TRAIN_SLICES,
            }
        },
        CKPT_DIR / f"pcc_seg_corrector_fold_{fold}_FORMAL_v1.pt"
    )

    # --------------------------------------------------------
    # 5. Save partial results after every fold
    # --------------------------------------------------------
    pd.DataFrame(all_case_rows).to_csv(
        PARTIAL_DIR / "partial_case_metrics_so_far.csv",
        index=False
    )

    pd.DataFrame(all_fold_rows).to_csv(
        PARTIAL_DIR / "partial_summary_by_fold_so_far.csv",
        index=False
    )

    pd.DataFrame(all_losses).to_csv(
        PARTIAL_DIR / "partial_training_losses_so_far.csv",
        index=False
    )

    with open(PARTIAL_DIR / "last_completed_fold.txt", "w") as f:
        f.write(str(fold))

    print("\nFold completed:", fold)
    print("Fold runtime minutes:", fold_runtime)
    print("Fold summary:")
    display(pd.DataFrame([fold_summary]))

    # Free memory
    del X_train, Y_train, base_prob_train, pcc_prob_train
    del baseline_model, pcc_model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


# =====================
# Final aggregation
# =====================
case_df = pd.DataFrame(all_case_rows)
fold_summary_df = pd.DataFrame(all_fold_rows)
loss_df = pd.DataFrame(all_losses)

runtime_minutes = (time.time() - start_time) / 60.0


# Bootstrap CI
rng = np.random.default_rng(42)
boot_rows = []

for metric in ["dice_gain", "iou_gain", "dice_gain_fixed05", "iou_gain_fixed05"]:
    vals = case_df[metric].values
    vals = vals[np.isfinite(vals)]

    boots = []
    for _ in range(5000):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots.append(sample.mean())

    boot_rows.append({
        "metric": metric,
        "mean": float(vals.mean()),
        "median": float(np.median(vals)),
        "ci95_low": float(np.percentile(boots, 2.5)),
        "ci95_high": float(np.percentile(boots, 97.5)),
        "n": int(len(vals)),
    })

boot_df = pd.DataFrame(boot_rows)


# Paired Wilcoxon test if scipy is available
stat_rows = []

for metric in ["dice_gain", "iou_gain", "dice_gain_fixed05", "iou_gain_fixed05"]:
    vals = case_df[metric].values
    vals = vals[np.isfinite(vals)]

    if wilcoxon is not None:
        try:
            w = wilcoxon(vals, alternative="greater")
            p_value = float(w.pvalue)
            statistic = float(w.statistic)
        except Exception as e:
            p_value = np.nan
            statistic = np.nan
    else:
        p_value = np.nan
        statistic = np.nan

    stat_rows.append({
        "metric": metric,
        "mean": float(vals.mean()),
        "median": float(np.median(vals)),
        "wins": int((vals > 0).sum()),
        "losses": int((vals < 0).sum()),
        "ties": int((vals == 0).sum()),
        "win_rate": float((vals > 0).mean()),
        "wilcoxon_greater_statistic": statistic,
        "wilcoxon_greater_p_value": p_value,
    })

stat_df = pd.DataFrame(stat_rows)


overall_summary = {
    "experiment": "Layer 1 Formal v1 5-fold current tumour segmentation",
    "baseline": "T1c -> current tumour mask",
    "pcc": "T1c + baseline prediction -> corrected current tumour mask",
    "folds": len(FOLDS),
    "total_test_cases": int(len(case_df)),
    "baseline_epochs": BASELINE_EPOCHS,
    "pcc_epochs": PCC_EPOCHS,
    "max_train_slices": MAX_TRAIN_SLICES,
    "runtime_minutes": float(runtime_minutes),

    "mean_baseline_dice": float(case_df["baseline_dice"].mean()),
    "mean_pcc_dice": float(case_df["pcc_dice"].mean()),
    "mean_dice_gain": float(case_df["dice_gain"].mean()),
    "median_dice_gain": float(case_df["dice_gain"].median()),
    "dice_wins": int((case_df["dice_gain"] > 0).sum()),
    "dice_losses": int((case_df["dice_gain"] < 0).sum()),
    "dice_win_rate": float((case_df["dice_gain"] > 0).mean()),

    "mean_baseline_iou": float(case_df["baseline_iou"].mean()),
    "mean_pcc_iou": float(case_df["pcc_iou"].mean()),
    "mean_iou_gain": float(case_df["iou_gain"].mean()),
    "median_iou_gain": float(case_df["iou_gain"].median()),
    "iou_wins": int((case_df["iou_gain"] > 0).sum()),
    "iou_losses": int((case_df["iou_gain"] < 0).sum()),
    "iou_win_rate": float((case_df["iou_gain"] > 0).mean()),

    "mean_baseline_dice_fixed05": float(case_df["baseline_dice_fixed05"].mean()),
    "mean_pcc_dice_fixed05": float(case_df["pcc_dice_fixed05"].mean()),
    "mean_dice_gain_fixed05": float(case_df["dice_gain_fixed05"].mean()),

    "mean_baseline_iou_fixed05": float(case_df["baseline_iou_fixed05"].mean()),
    "mean_pcc_iou_fixed05": float(case_df["pcc_iou_fixed05"].mean()),
    "mean_iou_gain_fixed05": float(case_df["iou_gain_fixed05"].mean()),
}


# =====================
# Save final outputs
# =====================
case_path = FORMAL_DIR / "layer1_FORMAL_v1_case_metrics.csv"
fold_summary_path = FORMAL_DIR / "layer1_FORMAL_v1_summary_by_fold.csv"
overall_summary_path = FORMAL_DIR / "layer1_FORMAL_v1_overall_summary.csv"
loss_path = FORMAL_DIR / "layer1_FORMAL_v1_training_losses.csv"
boot_path = FORMAL_DIR / "layer1_FORMAL_v1_bootstrap_CI.csv"
stat_path = FORMAL_DIR / "layer1_FORMAL_v1_paired_stats.csv"
run_info_path = FORMAL_DIR / "layer1_FORMAL_v1_run_info.json"

case_df.to_csv(case_path, index=False)
fold_summary_df.to_csv(fold_summary_path, index=False)
pd.DataFrame([overall_summary]).to_csv(overall_summary_path, index=False)
loss_df.to_csv(loss_path, index=False)
boot_df.to_csv(boot_path, index=False)
stat_df.to_csv(stat_path, index=False)

with open(run_info_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)


# =====================
# Display final results
# =====================
print("\n" + "=" * 100)
print("Layer 1 Formal v1 finished.")
print("Runtime minutes:", runtime_minutes)
print("Saved to:", FORMAL_DIR)

print("\nOverall summary:")
display(pd.DataFrame([overall_summary]))

print("\nSummary by fold:")
display(fold_summary_df)

print("\nBootstrap 95% CI:")
display(boot_df)

print("\nPaired statistics:")
display(stat_df)

print("\nSaved files:")
print(case_path)
print(fold_summary_path)
print(overall_summary_path)
print(loss_path)
print(boot_path)
print(stat_path)
print(run_info_path)
print("Checkpoints:", CKPT_DIR)

In [ ]:
from pathlib import Path
import zipfile
import json
from datetime import datetime

FORMAL_DIR = Path("/kaggle/working/pcc_independent_baseline/stage2_pcc_guided_student_learning/layer1_current_segmentation_FORMAL_v1_5fold")

assert FORMAL_DIR.exists(), f"Missing folder: {FORMAL_DIR}"

zip_path = Path("/kaggle/working/Layer1_FORMAL_v1_5fold_RESULTS_BACKUP.zip")

manifest = {
    "created_at": datetime.now().isoformat(),
    "experiment": "Layer 1 Formal v1 5-fold current tumour segmentation",
    "task": "current T1c -> current tumour mask",
    "comparison": "Baseline-Seg vs PCC-Seg correction",
    "important_results": {
        "mean_baseline_dice": 0.415621,
        "mean_dice_gain": 0.086323,
        "dice_wins": "36/40",
        "dice_win_rate": 0.90,
        "dice_gain_95ci": [0.060166, 0.113784],
        "mean_iou_gain": 0.077183,
        "iou_wins": "36/40",
        "iou_win_rate": 0.90,
        "iou_gain_95ci": [0.054869, 0.100161],
        "fixed05_dice_gain": 0.038662,
        "fixed05_iou_gain": 0.030834
    },
    "interpretation": "Layer 1 Formal v1 passed. PCC-style correction shows stable positive improvement for current tumour segmentation.",
    "folder": str(FORMAL_DIR)
}

manifest_path = Path("/kaggle/working/Layer1_FORMAL_v1_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(manifest_path, manifest_path.name)

    for file in FORMAL_DIR.rglob("*"):
        if file.is_file():
            arcname = file.relative_to(Path("/kaggle/working"))
            z.write(file, arcname)

print("Backup created:")
print(zip_path)
print("Size MB:", zip_path.stat().st_size / 1024 / 1024)

print("\nIncluded files:")
with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print(name)

In [ ]:
# ============================================================
# Restore Layer-1 working data after Kaggle /working reset
#
# Goal:
#   Rebuild the minimal data needed for Layer 1:
#     Input  = current T1c
#     Target = current tumour mask
#
# Output:
#   /kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_npz
#   /kaggle/working/pcc_independent_baseline/preprocessed_locked40_2d_summary.csv
#   /kaggle/working/pcc_independent_baseline/stage2_pcc_guided_student_learning/model_A_direct_target_5fold/matched_5fold_splits_seed42.csv
# ============================================================

from pathlib import Path
import re
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import nibabel as nib
except ImportError:
    !pip install -q nibabel
    import nibabel as nib


# =====================
# Robust path discovery
# =====================
INPUT_ROOT = Path("/kaggle/input")

print("=== Searching input files ===")

# Find split file
split_candidates = list(INPUT_ROOT.rglob("matched_5fold_splits_seed42.csv"))
assert len(split_candidates) > 0, "Cannot find matched_5fold_splits_seed42.csv in /kaggle/input"

# Prefer model-a-backup-final if available
split_candidates_sorted = sorted(
    split_candidates,
    key=lambda p: (0 if "model-a-backup-final" in str(p).lower() else 1, len(str(p)))
)
SPLIT_SRC = split_candidates_sorted[0]

print("Using split file:", SPLIT_SRC)


# Find raw MU-Glioma-Post root that contains PatientID_xxxx folders
raw_candidates = []

for p in INPUT_ROOT.rglob("*"):
    if p.is_dir() and p.name == "MU-Glioma-Post":
        patient_dirs = list(p.glob("PatientID_*"))
        if len(patient_dirs) > 0:
            raw_candidates.append(p)

assert len(raw_candidates) > 0, "Cannot find raw MU-Glioma-Post folder containing PatientID_* directories"

# Choose the deepest/most specific candidate
RAW_ROOT = sorted(raw_candidates, key=lambda p: len(str(p)), reverse=True)[0]

print("Using raw root:", RAW_ROOT)


# =====================
# Working paths
# =====================
WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

MODEL_A_DIR = STAGE2_DIR / "model_A_direct_target_5fold"
NPZ_DIR = OUT_DIR / "preprocessed_locked40_2d_npz"

OUT_DIR.mkdir(parents=True, exist_ok=True)
STAGE2_DIR.mkdir(parents=True, exist_ok=True)
MODEL_A_DIR.mkdir(parents=True, exist_ok=True)
NPZ_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DST = MODEL_A_DIR / "matched_5fold_splits_seed42.csv"
PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"

shutil.copy2(SPLIT_SRC, SPLIT_DST)

print("\n=== Output paths ===")
print("Split copied to:", SPLIT_DST)
print("NPZ output:", NPZ_DIR)
print("Summary output:", PRE_SUMMARY_PATH)


# =====================
# Helper functions
# =====================
def parse_case_id(case_id):
    """
    Example:
      PatientID_0008_T4_to_T6_t1c
    """
    m = re.match(r"(PatientID_\d+)_T(\d+)_to_T(\d+)_t1c", case_id)
    if m is None:
        raise ValueError(f"Cannot parse case_id: {case_id}")
    
    patient_id = m.group(1)
    current_tp = int(m.group(2))
    future_tp = int(m.group(3))
    return patient_id, current_tp, future_tp


def load_nii_as_zhw(path):
    """
    Load NIfTI and return [Z, H, W].
    MU-Glioma files are usually [H, W, Z].
    """
    img = nib.load(str(path))
    arr = img.get_fdata().astype(np.float32)

    if arr.ndim == 4:
        arr = arr[..., 0]

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    # Assume final axis is slice axis
    arr = np.moveaxis(arr, -1, 0)  # [Z, H, W]
    return arr.astype(np.float32)


def normalize_volume(vol):
    """
    Robust per-volume normalization to [0, 1].
    """
    vol = vol.astype(np.float32)
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)

    nonzero = vol[vol > 0]

    if nonzero.size < 100:
        mn, mx = float(vol.min()), float(vol.max())
    else:
        mn, mx = np.percentile(nonzero, [1, 99])

    if mx <= mn:
        return np.zeros_like(vol, dtype=np.float32)

    out = (vol - mn) / (mx - mn + 1e-8)
    out = np.clip(out, 0.0, 1.0)
    return out.astype(np.float32)


def get_paths_for_case(case_id):
    patient_id, current_tp, future_tp = parse_case_id(case_id)

    tp_dir = RAW_ROOT / patient_id / f"Timepoint_{current_tp}"

    t1c_path = tp_dir / f"{patient_id}_Timepoint_{current_tp}_brain_t1c.nii"
    mask_path = tp_dir / f"{patient_id}_Timepoint_{current_tp}_tumorMask.nii"

    if not t1c_path.exists():
        raise FileNotFoundError(t1c_path)
    if not mask_path.exists():
        raise FileNotFoundError(mask_path)

    return patient_id, current_tp, future_tp, t1c_path, mask_path


# =====================
# Build locked 40 case list from split file
# =====================
split_df = pd.read_csv(SPLIT_DST)
assert "case_id" in split_df.columns, split_df.columns
assert "fold" in split_df.columns, split_df.columns
assert "split" in split_df.columns, split_df.columns

case_ids = sorted(split_df["case_id"].unique())

print("\n=== Split check ===")
print("Unique locked cases:", len(case_ids))
display(split_df.head())

assert len(case_ids) == 40, f"Expected 40 locked cases, got {len(case_ids)}"


# =====================
# Preprocess locked 40 cases
# =====================
rows = []

for case_id in tqdm(case_ids, desc="Preprocessing locked 40 current segmentation data"):
    patient_id, current_tp, future_tp, t1c_path, mask_path = get_paths_for_case(case_id)

    t1c = load_nii_as_zhw(t1c_path)
    mask = load_nii_as_zhw(mask_path)

    t1c = normalize_volume(t1c)
    mask = (mask > 0).astype(np.float32)

    assert t1c.shape == mask.shape, (case_id, t1c.shape, mask.shape)

    # X format:
    # X[:, 0] = current T1c
    # X[:, 1] = current tumour mask
    X = np.stack([t1c, mask], axis=1).astype(np.float32)  # [Z, 2, H, W]

    out_path = NPZ_DIR / f"{case_id}.npz"

    np.savez_compressed(
        out_path,
        X=X,
        case_id=case_id,
        patient_id=patient_id,
        current_timepoint=current_tp,
        future_timepoint=future_tp,
    )

    rows.append({
        "case_id": case_id,
        "patient_id": patient_id,
        "current_timepoint": current_tp,
        "future_timepoint": future_tp,
        "npz_path": str(out_path),
        "z_slices": int(X.shape[0]),
        "height": int(X.shape[2]),
        "width": int(X.shape[3]),
        "current_mask_voxels": int(mask.sum()),
        "positive_slices": int((mask.reshape(mask.shape[0], -1).sum(axis=1) > 0).sum()),
        "t1c_path": str(t1c_path),
        "mask_path": str(mask_path),
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(PRE_SUMMARY_PATH, index=False)


# =====================
# Final check
# =====================
print("\nDone.")
print("Saved summary:", PRE_SUMMARY_PATH)
print("Saved npz dir:", NPZ_DIR)
print("NPZ files:", len(list(NPZ_DIR.glob('*.npz'))))

print("\nSummary preview:")
display(summary_df.head())

print("\nVolume summary:")
display(summary_df[["z_slices", "current_mask_voxels", "positive_slices"]].describe())

print("\nCheck paths:")
print("preprocessed summary exists:", PRE_SUMMARY_PATH.exists())
print("split file exists:", SPLIT_DST.exists())
print("npz count:", len(list(NPZ_DIR.glob('*.npz'))))

In [ ]:
# ============================================================
# Layer 1 Formal v1.1: 15-Epoch Sensitivity Experiment
#
# Goal:
#   Test whether PCC improvement remains stable under longer training.
#
# Task:
#   Input  = current T1c only
#   Target = current tumour mask
#
# Compare:
#   Baseline-Seg:
#       T1c -> current tumour mask
#
#   PCC-Seg:
#       T1c + baseline prediction -> corrected current tumour mask
#
# Setting:
#   - 5 folds
#   - 15 epochs baseline per fold
#   - 15 epochs PCC corrector per fold
#   - all train slices per fold
#   - save checkpoints and partial results after every fold
# ============================================================

from pathlib import Path
import json
import time
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None


# =====================
# Config
# =====================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FOLDS = [1, 2, 3, 4, 5]

BASELINE_EPOCHS = 15
PCC_EPOCHS = 15

BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-5

# Use all training slices. Each fold has about 4960 train slices.
MAX_TRAIN_SLICES = 999999

PCC_MAX_DELTA_LOGIT = 3.0

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =====================
# Paths
# =====================
WORKING = Path("/kaggle/working")
OUT_DIR = WORKING / "pcc_independent_baseline"
STAGE2_DIR = OUT_DIR / "stage2_pcc_guided_student_learning"

PRE_SUMMARY_PATH = OUT_DIR / "preprocessed_locked40_2d_summary.csv"
SPLIT_PATH = STAGE2_DIR / "model_A_direct_target_5fold" / "matched_5fold_splits_seed42.csv"

FORMAL_DIR = STAGE2_DIR / "layer1_current_segmentation_FORMAL_v1_1_15epoch_sensitivity"
CKPT_DIR = FORMAL_DIR / "checkpoints"
PARTIAL_DIR = FORMAL_DIR / "partial_results"

FORMAL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

assert PRE_SUMMARY_PATH.exists(), PRE_SUMMARY_PATH
assert SPLIT_PATH.exists(), SPLIT_PATH

pre_df = pd.read_csv(PRE_SUMMARY_PATH)
split_df = pd.read_csv(SPLIT_PATH)

case_to_npz = dict(zip(pre_df["case_id"], pre_df["npz_path"]))

print("Output:", FORMAL_DIR)
print("Total cases:", len(case_to_npz))
print("Baseline epochs:", BASELINE_EPOCHS)
print("PCC epochs:", PCC_EPOCHS)


# =====================
# Data loading
# =====================
def load_case_seg(case_id):
    """
    Return:
      X_t1c: [Z, 1, H, W]
      Y_cur: [Z, 1, H, W]
    """
    path = case_to_npz[case_id]
    with np.load(path, allow_pickle=True) as data:
        X = data["X"].astype(np.float32)

    X_t1c = X[:, 0:1].astype(np.float32)
    Y_cur = X[:, 1:2].astype(np.float32)

    return X_t1c, Y_cur


def build_train_arrays(case_ids, fold_seed, max_slices=999999):
    xs, ys = [], []

    for cid in case_ids:
        x, y = load_case_seg(cid)
        xs.append(x)
        ys.append(y)

    X = np.concatenate(xs, axis=0)
    Y = np.concatenate(ys, axis=0)

    if max_slices is not None and len(X) > max_slices:
        voxel_sum = Y.reshape(Y.shape[0], -1).sum(axis=1)
        pos_idx = np.where(voxel_sum > 0)[0]
        neg_idx = np.where(voxel_sum == 0)[0]

        rng = np.random.default_rng(fold_seed)

        n_pos = min(len(pos_idx), max_slices // 2)
        n_neg = max_slices - n_pos

        chosen_pos = rng.choice(pos_idx, size=n_pos, replace=False) if len(pos_idx) > n_pos else pos_idx

        if len(neg_idx) > 0:
            chosen_neg = rng.choice(neg_idx, size=min(len(neg_idx), n_neg), replace=False)
        else:
            chosen_neg = np.array([], dtype=int)

        chosen = np.concatenate([chosen_pos, chosen_neg])
        rng.shuffle(chosen)

        X = X[chosen]
        Y = Y[chosen]

    return X.astype(np.float32), Y.astype(np.float32)


# =====================
# Model
# =====================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


# =====================
# Loss / metrics
# =====================
def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)

    dice = (2 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def make_loss_fn(Y_train):
    pos = float(Y_train.sum())
    total = float(Y_train.size)
    neg = total - pos

    pos_weight_value = neg / max(pos, 1.0)
    pos_weight_value = min(pos_weight_value, 50.0)

    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

    print("BCE pos_weight:", pos_weight_value)

    def loss_fn(logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)
        dice = soft_dice_loss_from_logits(logits, targets)
        return 0.5 * bce + 0.5 * dice

    return loss_fn


def dice_iou_np(prob, target, threshold=0.5, eps=1e-8):
    pred = prob >= threshold
    target = target.astype(bool)

    inter = np.logical_and(pred, target).sum()
    pred_sum = pred.sum()
    target_sum = target.sum()
    union = np.logical_or(pred, target).sum()

    dice = (2 * inter + eps) / (pred_sum + target_sum + eps)
    iou = (inter + eps) / (union + eps)

    return float(dice), float(iou)


def find_best_threshold(prob, target):
    thresholds = np.arange(0.10, 0.91, 0.05)

    best_t = 0.5
    best_dice = -1.0

    for t in thresholds:
        d, _ = dice_iou_np(prob, target, threshold=t)
        if d > best_dice:
            best_dice = d
            best_t = float(t)

    return best_t, float(best_dice)


# =====================
# Training / prediction
# =====================
def train_baseline_model(X_train, Y_train, fold):
    model = SmallUNet2D(in_ch=1, out_ch=1, base=16).to(DEVICE)

    ds = TensorDataset(
        torch.from_numpy(X_train).float(),
        torch.from_numpy(Y_train).float()
    )

    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = make_loss_fn(Y_train)

    losses = []

    for epoch in range(1, BASELINE_EPOCHS + 1):
        model.train()
        running = 0.0

        for xb, yb in tqdm(dl, desc=f"Fold {fold} Baseline epoch {epoch}/{BASELINE_EPOCHS}"):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            running += loss.item() * xb.size(0)

        epoch_loss = running / len(ds)
        losses.append({
            "fold": fold,
            "model": "baseline",
            "epoch": epoch,
            "loss": epoch_loss,
        })
        print(f"Fold {fold} Baseline epoch {epoch}: loss={epoch_loss:.5f}")

    return model, losses


@torch.no_grad()
def predict_prob(model, X, batch_size=BATCH_SIZE):
    model.eval()
    parts = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X[start:start + batch_size]).float().to(DEVICE)
        logits = model(xb)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        parts.append(probs.astype(np.float32))

    return np.concatenate(parts, axis=0)


def train_pcc_corrector(X_train, Y_train, base_prob_train, fold):
    """
    PCC-Seg:
      input  = current T1c + baseline probability
      output = residual correction
      final_logit = logit(base_prob) + max_delta * tanh(residual)
    """
    pcc_model = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)

    X_pcc = np.concatenate(
        [X_train, base_prob_train.astype(np.float32)],
        axis=1
    ).astype(np.float32)

    ds = TensorDataset(
        torch.from_numpy(X_pcc).float(),
        torch.from_numpy(Y_train).float(),
        torch.from_numpy(base_prob_train).float(),
    )

    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    opt = torch.optim.AdamW(pcc_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = make_loss_fn(Y_train)

    losses = []

    for epoch in range(1, PCC_EPOCHS + 1):
        pcc_model.train()
        running = 0.0

        for xb, yb, basep in tqdm(dl, desc=f"Fold {fold} PCC epoch {epoch}/{PCC_EPOCHS}"):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            basep = basep.to(DEVICE, non_blocking=True)

            base_logit = torch.logit(torch.clamp(basep, 1e-4, 1.0 - 1e-4))

            opt.zero_grad(set_to_none=True)

            residual_raw = pcc_model(xb)
            corrected_logits = base_logit + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)

            loss = loss_fn(corrected_logits, yb)

            # Conservative correction regularization
            loss = loss + 0.002 * torch.mean(torch.abs(torch.tanh(residual_raw)))

            loss.backward()
            opt.step()

            running += loss.item() * xb.size(0)

        epoch_loss = running / len(ds)
        losses.append({
            "fold": fold,
            "model": "pcc_corrector",
            "epoch": epoch,
            "loss": epoch_loss,
        })
        print(f"Fold {fold} PCC epoch {epoch}: loss={epoch_loss:.5f}")

    return pcc_model, losses


@torch.no_grad()
def predict_pcc_corrected(pcc_model, X, base_prob, batch_size=BATCH_SIZE):
    pcc_model.eval()

    X_pcc = np.concatenate(
        [X, base_prob.astype(np.float32)],
        axis=1
    ).astype(np.float32)

    parts = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X_pcc[start:start + batch_size]).float().to(DEVICE)
        bp = torch.from_numpy(base_prob[start:start + batch_size]).float().to(DEVICE)

        base_logit = torch.logit(torch.clamp(bp, 1e-4, 1.0 - 1e-4))
        residual_raw = pcc_model(xb)
        corrected_logits = base_logit + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)
        corrected_prob = torch.sigmoid(corrected_logits).detach().cpu().numpy()

        parts.append(corrected_prob.astype(np.float32))

    return np.concatenate(parts, axis=0)


# =====================
# Main experiment
# =====================
start_time = time.time()

all_case_rows = []
all_fold_rows = []
all_losses = []

for fold in FOLDS:
    print("\n" + "=" * 100)
    print(f"Layer 1 Formal v1.1 15-epoch sensitivity - START FOLD {fold}")

    fold_start = time.time()

    train_cases = split_df[
        (split_df["fold"] == fold) & (split_df["split"] == "train")
    ]["case_id"].tolist()

    test_cases = split_df[
        (split_df["fold"] == fold) & (split_df["split"] == "test")
    ]["case_id"].tolist()

    print("Train cases:", len(train_cases))
    print("Test cases:", len(test_cases))

    X_train, Y_train = build_train_arrays(
        train_cases,
        fold_seed=SEED + fold,
        max_slices=MAX_TRAIN_SLICES
    )

    print("Train slices used:", X_train.shape[0])
    print("Input shape:", X_train.shape)
    print("Target shape:", Y_train.shape)
    print("Positive target voxels:", int(Y_train.sum()))

    # --------------------
    # Baseline
    # --------------------
    print("\nTraining Baseline-Seg 15 epochs...")
    baseline_model, baseline_losses = train_baseline_model(X_train, Y_train, fold)
    all_losses.extend(baseline_losses)

    print("\nGenerating baseline train predictions...")
    base_prob_train = predict_prob(baseline_model, X_train)

    baseline_threshold, baseline_train_dice = find_best_threshold(
        base_prob_train,
        Y_train
    )

    print("Baseline selected threshold:", baseline_threshold)
    print("Baseline train Dice:", baseline_train_dice)

    # --------------------
    # PCC corrector
    # --------------------
    print("\nTraining PCC-Seg corrector 15 epochs...")
    pcc_model, pcc_losses = train_pcc_corrector(
        X_train,
        Y_train,
        base_prob_train,
        fold
    )
    all_losses.extend(pcc_losses)

    print("\nGenerating PCC train predictions...")
    pcc_prob_train = predict_pcc_corrected(
        pcc_model,
        X_train,
        base_prob_train
    )

    pcc_threshold, pcc_train_dice = find_best_threshold(
        pcc_prob_train,
        Y_train
    )

    print("PCC selected threshold:", pcc_threshold)
    print("PCC train Dice:", pcc_train_dice)

    # --------------------
    # Evaluate
    # --------------------
    fold_case_rows = []

    print("\nEvaluating fold test cases...")

    for cid in tqdm(test_cases, desc=f"Fold {fold} test cases"):
        X_test, Y_test = load_case_seg(cid)

        base_prob = predict_prob(baseline_model, X_test)
        pcc_prob = predict_pcc_corrected(pcc_model, X_test, base_prob)

        # Train-calibrated thresholds
        base_dice, base_iou = dice_iou_np(
            base_prob,
            Y_test,
            threshold=baseline_threshold
        )

        pcc_dice, pcc_iou = dice_iou_np(
            pcc_prob,
            Y_test,
            threshold=pcc_threshold
        )

        # Fixed 0.5 threshold
        base_dice_05, base_iou_05 = dice_iou_np(
            base_prob,
            Y_test,
            threshold=0.5
        )

        pcc_dice_05, pcc_iou_05 = dice_iou_np(
            pcc_prob,
            Y_test,
            threshold=0.5
        )

        row = {
            "case_id": cid,
            "fold": fold,

            "baseline_threshold": baseline_threshold,
            "pcc_threshold": pcc_threshold,

            "baseline_train_dice": baseline_train_dice,
            "pcc_train_dice": pcc_train_dice,

            "baseline_dice": base_dice,
            "pcc_dice": pcc_dice,
            "dice_gain": pcc_dice - base_dice,

            "baseline_iou": base_iou,
            "pcc_iou": pcc_iou,
            "iou_gain": pcc_iou - base_iou,

            "baseline_dice_fixed05": base_dice_05,
            "pcc_dice_fixed05": pcc_dice_05,
            "dice_gain_fixed05": pcc_dice_05 - base_dice_05,

            "baseline_iou_fixed05": base_iou_05,
            "pcc_iou_fixed05": pcc_iou_05,
            "iou_gain_fixed05": pcc_iou_05 - base_iou_05,

            "target_voxels": int(Y_test.sum()),
        }

        fold_case_rows.append(row)
        all_case_rows.append(row)

    fold_df = pd.DataFrame(fold_case_rows)

    fold_runtime = (time.time() - fold_start) / 60.0

    fold_summary = {
        "fold": fold,
        "train_cases": len(train_cases),
        "test_cases": len(test_cases),
        "train_slices_used": int(X_train.shape[0]),
        "fold_runtime_minutes": float(fold_runtime),

        "baseline_threshold": baseline_threshold,
        "pcc_threshold": pcc_threshold,

        "baseline_train_dice": baseline_train_dice,
        "pcc_train_dice": pcc_train_dice,

        "mean_baseline_dice": float(fold_df["baseline_dice"].mean()),
        "mean_pcc_dice": float(fold_df["pcc_dice"].mean()),
        "mean_dice_gain": float(fold_df["dice_gain"].mean()),
        "median_dice_gain": float(fold_df["dice_gain"].median()),
        "dice_wins": int((fold_df["dice_gain"] > 0).sum()),
        "dice_losses": int((fold_df["dice_gain"] < 0).sum()),
        "dice_win_rate": float((fold_df["dice_gain"] > 0).mean()),

        "mean_baseline_iou": float(fold_df["baseline_iou"].mean()),
        "mean_pcc_iou": float(fold_df["pcc_iou"].mean()),
        "mean_iou_gain": float(fold_df["iou_gain"].mean()),
        "median_iou_gain": float(fold_df["iou_gain"].median()),
        "iou_wins": int((fold_df["iou_gain"] > 0).sum()),
        "iou_losses": int((fold_df["iou_gain"] < 0).sum()),
        "iou_win_rate": float((fold_df["iou_gain"] > 0).mean()),

        "mean_baseline_dice_fixed05": float(fold_df["baseline_dice_fixed05"].mean()),
        "mean_pcc_dice_fixed05": float(fold_df["pcc_dice_fixed05"].mean()),
        "mean_dice_gain_fixed05": float(fold_df["dice_gain_fixed05"].mean()),

        "mean_baseline_iou_fixed05": float(fold_df["baseline_iou_fixed05"].mean()),
        "mean_pcc_iou_fixed05": float(fold_df["pcc_iou_fixed05"].mean()),
        "mean_iou_gain_fixed05": float(fold_df["iou_gain_fixed05"].mean()),
    }

    all_fold_rows.append(fold_summary)

    # --------------------
    # Save checkpoints
    # --------------------
    torch.save(
        {
            "fold": fold,
            "model_state_dict": baseline_model.state_dict(),
            "threshold": baseline_threshold,
            "train_dice": baseline_train_dice,
            "config": {
                "experiment": "Layer 1 Formal v1.1 15-epoch sensitivity",
                "input": "current T1c only",
                "target": "current tumour mask",
                "epochs": BASELINE_EPOCHS,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "max_train_slices": MAX_TRAIN_SLICES,
            }
        },
        CKPT_DIR / f"baseline_seg_fold_{fold}_FORMAL_v1_1_15epoch.pt"
    )

    torch.save(
        {
            "fold": fold,
            "model_state_dict": pcc_model.state_dict(),
            "threshold": pcc_threshold,
            "train_dice": pcc_train_dice,
            "config": {
                "experiment": "Layer 1 Formal v1.1 15-epoch sensitivity",
                "input": "current T1c + baseline prediction",
                "target": "current tumour mask",
                "method": "PCC-style residual correction",
                "epochs": PCC_EPOCHS,
                "max_delta_logit": PCC_MAX_DELTA_LOGIT,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "max_train_slices": MAX_TRAIN_SLICES,
            }
        },
        CKPT_DIR / f"pcc_seg_corrector_fold_{fold}_FORMAL_v1_1_15epoch.pt"
    )

    # --------------------
    # Save partial results after every fold
    # --------------------
    pd.DataFrame(all_case_rows).to_csv(
        PARTIAL_DIR / "partial_case_metrics_so_far.csv",
        index=False
    )

    pd.DataFrame(all_fold_rows).to_csv(
        PARTIAL_DIR / "partial_summary_by_fold_so_far.csv",
        index=False
    )

    pd.DataFrame(all_losses).to_csv(
        PARTIAL_DIR / "partial_training_losses_so_far.csv",
        index=False
    )

    with open(PARTIAL_DIR / "last_completed_fold.txt", "w") as f:
        f.write(str(fold))

    print("\nFold completed:", fold)
    print("Fold runtime minutes:", fold_runtime)
    print("Fold summary:")
    display(pd.DataFrame([fold_summary]))

    # Free memory
    del X_train, Y_train, base_prob_train, pcc_prob_train
    del baseline_model, pcc_model
    gc.collect()

    if DEVICE == "cuda":
        torch.cuda.empty_cache()


# =====================
# Final aggregation
# =====================
case_df = pd.DataFrame(all_case_rows)
fold_summary_df = pd.DataFrame(all_fold_rows)
loss_df = pd.DataFrame(all_losses)

runtime_minutes = (time.time() - start_time) / 60.0


# Bootstrap CI
rng = np.random.default_rng(42)
boot_rows = []

for metric in ["dice_gain", "iou_gain", "dice_gain_fixed05", "iou_gain_fixed05"]:
    vals = case_df[metric].values
    vals = vals[np.isfinite(vals)]

    boots = []
    for _ in range(5000):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots.append(sample.mean())

    boot_rows.append({
        "metric": metric,
        "mean": float(vals.mean()),
        "median": float(np.median(vals)),
        "ci95_low": float(np.percentile(boots, 2.5)),
        "ci95_high": float(np.percentile(boots, 97.5)),
        "n": int(len(vals)),
    })

boot_df = pd.DataFrame(boot_rows)


# Paired Wilcoxon test
stat_rows = []

for metric in ["dice_gain", "iou_gain", "dice_gain_fixed05", "iou_gain_fixed05"]:
    vals = case_df[metric].values
    vals = vals[np.isfinite(vals)]

    if wilcoxon is not None:
        try:
            w = wilcoxon(vals, alternative="greater")
            p_value = float(w.pvalue)
            statistic = float(w.statistic)
        except Exception:
            p_value = np.nan
            statistic = np.nan
    else:
        p_value = np.nan
        statistic = np.nan

    stat_rows.append({
        "metric": metric,
        "mean": float(vals.mean()),
        "median": float(np.median(vals)),
        "wins": int((vals > 0).sum()),
        "losses": int((vals < 0).sum()),
        "ties": int((vals == 0).sum()),
        "win_rate": float((vals > 0).mean()),
        "wilcoxon_greater_statistic": statistic,
        "wilcoxon_greater_p_value": p_value,
    })

stat_df = pd.DataFrame(stat_rows)


overall_summary = {
    "experiment": "Layer 1 Formal v1.1 15-epoch sensitivity",
    "baseline": "T1c -> current tumour mask",
    "pcc": "T1c + baseline prediction -> corrected current tumour mask",
    "folds": len(FOLDS),
    "total_test_cases": int(len(case_df)),
    "baseline_epochs": BASELINE_EPOCHS,
    "pcc_epochs": PCC_EPOCHS,
    "max_train_slices": MAX_TRAIN_SLICES,
    "runtime_minutes": float(runtime_minutes),

    "mean_baseline_dice": float(case_df["baseline_dice"].mean()),
    "mean_pcc_dice": float(case_df["pcc_dice"].mean()),
    "mean_dice_gain": float(case_df["dice_gain"].mean()),
    "median_dice_gain": float(case_df["dice_gain"].median()),
    "dice_wins": int((case_df["dice_gain"] > 0).sum()),
    "dice_losses": int((case_df["dice_gain"] < 0).sum()),
    "dice_win_rate": float((case_df["dice_gain"] > 0).mean()),

    "mean_baseline_iou": float(case_df["baseline_iou"].mean()),
    "mean_pcc_iou": float(case_df["pcc_iou"].mean()),
    "mean_iou_gain": float(case_df["iou_gain"].mean()),
    "median_iou_gain": float(case_df["iou_gain"].median()),
    "iou_wins": int((case_df["iou_gain"] > 0).sum()),
    "iou_losses": int((case_df["iou_gain"] < 0).sum()),
    "iou_win_rate": float((case_df["iou_gain"] > 0).mean()),

    "mean_baseline_dice_fixed05": float(case_df["baseline_dice_fixed05"].mean()),
    "mean_pcc_dice_fixed05": float(case_df["pcc_dice_fixed05"].mean()),
    "mean_dice_gain_fixed05": float(case_df["dice_gain_fixed05"].mean()),

    "mean_baseline_iou_fixed05": float(case_df["baseline_iou_fixed05"].mean()),
    "mean_pcc_iou_fixed05": float(case_df["pcc_iou_fixed05"].mean()),
    "mean_iou_gain_fixed05": float(case_df["iou_gain_fixed05"].mean()),
}


# =====================
# Save final outputs
# =====================
case_path = FORMAL_DIR / "layer1_FORMAL_v1_1_15epoch_case_metrics.csv"
fold_summary_path = FORMAL_DIR / "layer1_FORMAL_v1_1_15epoch_summary_by_fold.csv"
overall_summary_path = FORMAL_DIR / "layer1_FORMAL_v1_1_15epoch_overall_summary.csv"
loss_path = FORMAL_DIR / "layer1_FORMAL_v1_1_15epoch_training_losses.csv"
boot_path = FORMAL_DIR / "layer1_FORMAL_v1_1_15epoch_bootstrap_CI.csv"
stat_path = FORMAL_DIR / "layer1_FORMAL_v1_1_15epoch_paired_stats.csv"
run_info_path = FORMAL_DIR / "layer1_FORMAL_v1_1_15epoch_run_info.json"

case_df.to_csv(case_path, index=False)
fold_summary_df.to_csv(fold_summary_path, index=False)
pd.DataFrame([overall_summary]).to_csv(overall_summary_path, index=False)
loss_df.to_csv(loss_path, index=False)
boot_df.to_csv(boot_path, index=False)
stat_df.to_csv(stat_path, index=False)

with open(run_info_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)


# =====================
# Display final results
# =====================
print("\n" + "=" * 100)
print("Layer 1 Formal v1.1 15-epoch sensitivity finished.")
print("Runtime minutes:", runtime_minutes)
print("Saved to:", FORMAL_DIR)

print("\nOverall summary:")
display(pd.DataFrame([overall_summary]))

print("\nSummary by fold:")
display(fold_summary_df)

print("\nBootstrap 95% CI:")
display(boot_df)

print("\nPaired statistics:")
display(stat_df)

print("\nSaved files:")
print(case_path)
print(fold_summary_path)
print(overall_summary_path)
print(loss_path)
print(boot_path)
print(stat_path)
print(run_info_path)
print("Checkpoints:", CKPT_DIR)

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

print("=== Kaggle input datasets ===")
for p in Path("/kaggle/input").iterdir():
    print(p)

print("\n=== Searching Layer1 results ===")

# 1. Search existing extracted Layer1 result files
hits = list(Path("/kaggle/input").rglob("layer1_FORMAL_v1_1_15epoch_case_metrics.csv"))
hits += list(Path("/kaggle/working").rglob("layer1_FORMAL_v1_1_15epoch_case_metrics.csv"))

print("Found case_metrics files:", len(hits))
for h in hits[:10]:
    print(" -", h)

# 2. Search Layer1 backup zip
zip_hits = list(Path("/kaggle/input").rglob("*.zip"))
print("\nFound zip files:", len(zip_hits))
for z in zip_hits:
    print(" -", z)

# 3. Extract Layer1 backup if needed
extract_root = Path("/kaggle/working/layer1_backup_unzipped")
extract_root.mkdir(parents=True, exist_ok=True)

layer1_zip = None
for z in zip_hits:
    if "Layer1" in z.name or "layer1" in z.name:
        layer1_zip = z
        break

if layer1_zip is not None:
    print("\nExtracting:", layer1_zip)
    with zipfile.ZipFile(layer1_zip, "r") as zip_ref:
        zip_ref.extractall(extract_root)
else:
    print("\nNo Layer1 zip found by name. If case_metrics already exists, this is okay.")

# 4. Search again after extraction
hits = list(Path("/kaggle/working").rglob("layer1_FORMAL_v1_1_15epoch_case_metrics.csv"))
hits += list(Path("/kaggle/input").rglob("layer1_FORMAL_v1_1_15epoch_case_metrics.csv"))

print("\nAfter extraction, found case_metrics files:", len(hits))
for h in hits[:10]:
    print(" -", h)

assert len(hits) > 0, "Layer1 case_metrics not found."

case_metrics_path = hits[0]
layer1_dir = case_metrics_path.parent
ckpt_dir = layer1_dir / "checkpoints"

print("\nSelected Layer1 dir:", layer1_dir)
print("Checkpoint dir:", ckpt_dir)
print("Checkpoint dir exists:", ckpt_dir.exists())

ckpts = list(ckpt_dir.glob("*.pt")) if ckpt_dir.exists() else []
print("Checkpoint files:", len(ckpts))
for c in ckpts[:20]:
    print(" -", c.name)

df = pd.read_csv(case_metrics_path)
print("\nCase metrics shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

print("\n=== Searching MU-Glioma-Post root ===")

candidate_roots = []
for p in Path("/kaggle/input").rglob("MU-Glioma-Post"):
    if p.is_dir():
        patient_dirs = list(p.glob("PatientID_*"))
        if len(patient_dirs) > 0:
            candidate_roots.append(p)

print("Candidate dataset roots:", len(candidate_roots))
for r in candidate_roots[:10]:
    print(" -", r, "| patients:", len(list(r.glob("PatientID_*"))))

assert len(candidate_roots) > 0, "MU-Glioma-Post root not found."

print("\nPRE-FLIGHT CHECK PASSED.")
print("You can now run Layer3 quick v0.")

In [ ]:
from pathlib import Path

print("=== /kaggle/input ===")
!find /kaggle/input -maxdepth 4 -type d | head -200

print("\n=== CSV files ===")
!find /kaggle/input -maxdepth 8 -type f -name "*.csv" | head -200

print("\n=== PT checkpoint files ===")
!find /kaggle/input -maxdepth 8 -type f -name "*.pt" | head -200

print("\n=== ZIP files ===")
!find /kaggle/input -maxdepth 8 -type f -name "*.zip" | head -100

In [ ]:
# ============================================================
# Layer3 quick v0: Pathology-Reliance Occlusion Audit
# FULL FIXED VERSION
#
# Fixed issue:
#   MU-Glioma-Post files are named like:
#     PatientID_0030_Timepoint_1_brain_t1c.nii
#     PatientID_0030_Timepoint_1_tumorMask.nii
#
# Goal:
#   Test whether PCC is more functionally dependent on real
#   pathological structures than the baseline.
#
# Uses:
#   Layer1 FORMAL_v1 5fold checkpoints
#
# No training. Inference + occlusion audit only.
#
# Main metric:
#   PRI = pathology_occlusion_drop - control_occlusion_drop
#
# If:
#   PCC_PRI > Baseline_PRI
# then:
#   PCC shows stronger pathology-specific reliance.
# ============================================================

from pathlib import Path
import re, json, time, gc, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import nibabel as nib
import scipy.ndimage as ndi

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None


# =========================
# Config
# =========================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

QUICK_CASES = 12
N_CONTROL_REGIONS = 3

BATCH_SIZE = 32
BOUNDARY_RADIUS = 5
AVOID_RADIUS = 8

PCC_MAX_DELTA_LOGIT = 3.0

OUT_DIR = Path("/kaggle/working/Layer3_quick_v0_pathology_reliance_occlusion_FORMAL_v1_FIXED")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUT_DIR)


# =========================
# Fixed Kaggle paths
# =========================
LAYER1_DIR = Path(
    "/kaggle/input/datasets/jeechangxin/layer1-formal-v1-5fold-results-backup/"
    "pcc_independent_baseline/stage2_pcc_guided_student_learning/"
    "layer1_current_segmentation_FORMAL_v1_5fold"
)

CKPT_DIR = LAYER1_DIR / "checkpoints"
CASE_METRICS_PATH = LAYER1_DIR / "layer1_FORMAL_v1_case_metrics.csv"
OVERALL_SUMMARY_PATH = LAYER1_DIR / "layer1_FORMAL_v1_overall_summary.csv"

DATA_ROOT = Path(
    "/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/"
    "PKG - MU-Glioma-Post/MU-Glioma-Post"
)

assert LAYER1_DIR.exists(), f"Layer1 dir not found: {LAYER1_DIR}"
assert CKPT_DIR.exists(), f"Checkpoint dir not found: {CKPT_DIR}"
assert CASE_METRICS_PATH.exists(), f"Case metrics not found: {CASE_METRICS_PATH}"
assert DATA_ROOT.exists(), f"Dataset root not found: {DATA_ROOT}"

case_metrics_all = pd.read_csv(CASE_METRICS_PATH)

print("Layer1 dir:", LAYER1_DIR)
print("Checkpoint dir:", CKPT_DIR)
print("Dataset root:", DATA_ROOT)
print("Patient folders:", len(list(DATA_ROOT.glob("PatientID_*"))))

print("\nCheckpoint files:")
for p in sorted(CKPT_DIR.glob("*.pt")):
    print(" -", p.name)

print("\nCase metrics:")
print(case_metrics_all.shape)
display(case_metrics_all.head())


# =========================
# Robust column handling
# =========================
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns found: {candidates}. Existing columns: {list(df.columns)}")


CASE_COL = pick_col(case_metrics_all, ["case_id", "sample_id"])
FOLD_COL = pick_col(case_metrics_all, ["fold", "fold_id"])

BASE_DICE_COL = None
PCC_DICE_COL = None
DICE_GAIN_COL = None

for c in ["baseline_dice", "base_dice"]:
    if c in case_metrics_all.columns:
        BASE_DICE_COL = c
        break

for c in ["pcc_dice", "corrected_dice"]:
    if c in case_metrics_all.columns:
        PCC_DICE_COL = c
        break

for c in ["dice_gain", "pcc_minus_baseline_dice", "dice_diff"]:
    if c in case_metrics_all.columns:
        DICE_GAIN_COL = c
        break

print("CASE_COL:", CASE_COL)
print("FOLD_COL:", FOLD_COL)
print("BASE_DICE_COL:", BASE_DICE_COL)
print("PCC_DICE_COL:", PCC_DICE_COL)
print("DICE_GAIN_COL:", DICE_GAIN_COL)


# =========================
# Model architecture
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


def get_threshold_from_ckpt(ckpt, default=0.5):
    for k in ["threshold", "best_threshold", "selected_threshold", "thr"]:
        if k in ckpt:
            return float(ckpt[k])
    return float(default)


def load_fold_models(fold):
    base_path = CKPT_DIR / f"baseline_seg_fold_{fold}_FORMAL_v1.pt"
    pcc_path = CKPT_DIR / f"pcc_seg_corrector_fold_{fold}_FORMAL_v1.pt"

    if not base_path.exists():
        raise FileNotFoundError(base_path)
    if not pcc_path.exists():
        raise FileNotFoundError(pcc_path)

    base_ckpt = torch.load(base_path, map_location=DEVICE)
    pcc_ckpt = torch.load(pcc_path, map_location=DEVICE)

    baseline = SmallUNet2D(in_ch=1, out_ch=1, base=16).to(DEVICE)
    pcc = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)

    baseline.load_state_dict(base_ckpt["model_state_dict"])
    pcc.load_state_dict(pcc_ckpt["model_state_dict"])

    baseline.eval()
    pcc.eval()

    base_thr = get_threshold_from_ckpt(base_ckpt, default=0.5)
    pcc_thr = get_threshold_from_ckpt(pcc_ckpt, default=0.5)

    print(f"Loaded fold {fold}: base_thr={base_thr}, pcc_thr={pcc_thr}")

    return baseline, pcc, base_thr, pcc_thr


# =========================
# Data loading helpers
# =========================
def parse_case_id(case_id):
    """
    Example:
      PatientID_0030_T1_to_T3_t1c
      PatientID_0020_T1_to_T2_t1c

    For Layer1 current segmentation:
      use the first T as current timepoint.
    """
    case_id = str(case_id)

    m_patient = re.search(r"(PatientID_\d+)", case_id)
    m_t = re.search(r"_T(\d+)", case_id)

    if m_patient is None or m_t is None:
        raise ValueError(f"Cannot parse case_id: {case_id}")

    patient_id = m_patient.group(1)
    current_tp = int(m_t.group(1))

    return patient_id, current_tp


def find_patient_dir(patient_id):
    direct = DATA_ROOT / patient_id
    if direct.exists():
        return direct

    hits = list(DATA_ROOT.glob(f"**/{patient_id}"))
    hits = [h for h in hits if h.is_dir()]

    if len(hits) == 0:
        raise FileNotFoundError(f"Patient folder not found: {patient_id}")

    return hits[0]


def find_timepoint_file(patient_dir, tp, kind):
    """
    Robust finder for MU-Glioma-Post.

    Actual filenames usually look like:
      PatientID_0030_Timepoint_1_brain_t1c.nii
      PatientID_0030_Timepoint_1_tumorMask.nii

    kind:
      "brain_t1c"
      "tumorMask"
    """
    patient_dir = Path(patient_dir)

    if kind == "brain_t1c":
        patterns = ["*brain_t1c.nii", "*brain_t1c.nii.gz"]
    elif kind == "tumorMask":
        patterns = ["*tumorMask.nii", "*tumorMask.nii.gz"]
    else:
        patterns = [f"*{kind}.nii", f"*{kind}.nii.gz"]

    candidates = []
    for pat in patterns:
        candidates.extend(list(patient_dir.rglob(pat)))

    candidates = list(dict.fromkeys(candidates))

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No {kind} file found in {patient_dir}. Tried patterns={patterns}"
        )

    tp_tokens = [
        f"Timepoint_{tp}",
        f"Timepoint-{tp}",
        f"Timepoint {tp}",
        f"_T{tp}_",
        f"/T{tp}/",
        f"TP{tp}",
    ]

    filtered = []
    for p in candidates:
        s = str(p)
        name = p.name
        if any(tok in s for tok in tp_tokens) or f"_Timepoint_{tp}_" in name:
            filtered.append(p)

    if len(filtered) > 0:
        return filtered[0]

    raise FileNotFoundError(
        f"Found {len(candidates)} {kind} candidates in {patient_dir}, "
        f"but none matched Timepoint {tp}. First candidates: "
        f"{[str(c) for c in candidates[:8]]}"
    )


def to_z_hw(arr):
    arr = np.asarray(arr)

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {arr.shape}")

    # Common NIfTI shape: H,W,Z -> Z,H,W
    if arr.shape[0] >= 128 and arr.shape[1] >= 128 and arr.shape[2] < 200:
        arr = np.moveaxis(arr, -1, 0)

    return arr.astype(np.float32)


def normalize_t1c(vol):
    vol = np.nan_to_num(vol.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    brain = np.abs(vol) > 1e-6
    if brain.sum() < 100:
        brain = np.ones_like(vol, dtype=bool)

    vals = vol[brain]
    mean = float(vals.mean())
    std = float(vals.std() + 1e-6)

    norm = (vol - mean) / std
    norm = np.clip(norm, -5.0, 5.0).astype(np.float32)

    return norm, brain.astype(bool)


def load_layer1_case(case_id):
    patient_id, current_tp = parse_case_id(case_id)
    pdir = find_patient_dir(patient_id)

    t1c_path = find_timepoint_file(pdir, current_tp, "brain_t1c")
    mask_path = find_timepoint_file(pdir, current_tp, "tumorMask")

    img_raw = to_z_hw(nib.load(str(t1c_path)).get_fdata())
    mask = to_z_hw(nib.load(str(mask_path)).get_fdata())

    img_norm, brain_mask = normalize_t1c(img_raw)
    mask_bool = mask > 0.5

    X = img_norm[:, None, :, :].astype(np.float32)
    Y = mask_bool[:, None, :, :].astype(np.float32)

    return X, Y, brain_mask.astype(bool), {
        "patient_id": patient_id,
        "current_tp": current_tp,
        "t1c_path": str(t1c_path),
        "mask_path": str(mask_path),
    }


# =========================
# Prediction and metrics
# =========================
@torch.no_grad()
def predict_baseline_and_pcc(baseline_model, pcc_model, X, batch_size=32):
    baseline_model.eval()
    pcc_model.eval()

    base_probs = []
    pcc_probs = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X[start:start + batch_size]).float().to(DEVICE)

        base_logits = baseline_model(xb)
        base_prob = torch.sigmoid(base_logits)

        pcc_input = torch.cat([xb, base_prob], dim=1)
        residual_raw = pcc_model(pcc_input)

        corrected_logits = base_logits + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)
        pcc_prob = torch.sigmoid(corrected_logits)

        base_probs.append(base_prob.detach().cpu().numpy().astype(np.float32))
        pcc_probs.append(pcc_prob.detach().cpu().numpy().astype(np.float32))

    return np.concatenate(base_probs, axis=0), np.concatenate(pcc_probs, axis=0)


def dice_iou_prob(prob, target, threshold=0.5, eps=1e-8):
    pred = prob >= threshold
    target = target.astype(bool)

    inter = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()
    pred_sum = pred.sum()
    target_sum = target.sum()

    dice = (2 * inter + eps) / (pred_sum + target_sum + eps)
    iou = (inter + eps) / (union + eps)

    return float(dice), float(iou)


def occlude_volume(X, occ_mask_zyx, replacement_value=0.0):
    """
    X: [Z,1,H,W]
    occ_mask_zyx: [Z,H,W] bool

    Since X is z-score normalized inside brain,
    replacement_value=0 roughly means brain-mean replacement.
    """
    X_occ = X.copy()
    X_occ[:, 0, :, :][occ_mask_zyx] = replacement_value
    return X_occ


# =========================
# Occlusion region helpers
# =========================
def make_regions(target_zyx, brain_zyx):
    target = target_zyx.astype(bool)
    brain = brain_zyx.astype(bool)

    core = np.logical_and(target, brain)

    if core.sum() > 0:
        dil = ndi.binary_dilation(core, iterations=BOUNDARY_RADIUS)
        ero = ndi.binary_erosion(core, iterations=BOUNDARY_RADIUS)
        boundary = np.logical_and(dil, np.logical_not(ero))
        boundary = np.logical_and(boundary, brain)
    else:
        boundary = np.zeros_like(core, dtype=bool)

    avoid = ndi.binary_dilation(core, iterations=AVOID_RADIUS)
    avoid = np.logical_or(avoid, np.logical_not(brain))

    return core, boundary, avoid


def make_shifted_control(region_mask, avoid_mask, brain_mask, rng, tries=80):
    """
    Matched non-pathological control region.
    Prefer shifted copy of the target region to preserve approximate shape.
    Fallback to random matched voxels inside brain and outside avoid.
    """
    region = region_mask.astype(bool)
    avoid = avoid_mask.astype(bool)
    brain = brain_mask.astype(bool)

    n = int(region.sum())
    if n <= 0:
        return np.zeros_like(region, dtype=bool)

    zdim, h, w = region.shape

    best = None
    best_count = -1

    for _ in range(tries):
        dz = rng.integers(-max(2, zdim // 3), max(3, zdim // 3))
        dy = rng.integers(-max(8, h // 3), max(9, h // 3))
        dx = rng.integers(-max(8, w // 3), max(9, w // 3))

        shifted = np.roll(region, shift=(dz, dy, dx), axis=(0, 1, 2))
        shifted = np.logical_and(shifted, brain)
        shifted = np.logical_and(shifted, np.logical_not(avoid))

        c = int(shifted.sum())

        if c > best_count:
            best = shifted
            best_count = c

        if c >= int(0.80 * n):
            return shifted

    candidates = np.logical_and(brain, np.logical_not(avoid))
    idx = np.argwhere(candidates)

    if len(idx) == 0:
        return np.zeros_like(region, dtype=bool)

    take = min(n, len(idx))
    chosen = idx[rng.choice(len(idx), size=take, replace=False)]

    control = np.zeros_like(region, dtype=bool)
    control[chosen[:, 0], chosen[:, 1], chosen[:, 2]] = True

    return control


# =========================
# Bootstrap/stat helpers
# =========================
def bootstrap_ci(vals, n_boot=3000, seed=42):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = []

    for _ in range(n_boot):
        s = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.mean(s))

    return float(np.mean(vals)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def summarize_metric(case_df, metric):
    vals = case_df[metric].values.astype(float)
    mean, low, high = bootstrap_ci(vals)

    if wilcoxon is not None:
        try:
            w = wilcoxon(vals, alternative="greater")
            p = float(w.pvalue)
            stat = float(w.statistic)
        except Exception:
            p = np.nan
            stat = np.nan
    else:
        p = np.nan
        stat = np.nan

    return {
        "metric": metric,
        "mean": mean,
        "median": float(np.nanmedian(vals)),
        "ci95_low": low,
        "ci95_high": high,
        "positive_cases": int(np.sum(vals > 0)),
        "negative_cases": int(np.sum(vals < 0)),
        "positive_rate": float(np.mean(vals > 0)),
        "wilcoxon_greater_statistic": stat,
        "wilcoxon_greater_p_value": p,
    }


# =========================
# Figure helper
# =========================
def save_case_figure(
    case_id, X, Y, core, boundary, control,
    base_orig, pcc_orig, base_occ_boundary, pcc_occ_boundary,
    base_thr, pcc_thr, out_path
):
    target = Y[:, 0] > 0.5

    score = boundary.reshape(boundary.shape[0], -1).sum(axis=1)
    if score.max() <= 0:
        score = target.reshape(target.shape[0], -1).sum(axis=1)

    z = int(np.argmax(score))

    img = X[z, 0]
    true_z = target[z]
    core_z = core[z]
    boundary_z = boundary[z]
    control_z = control[z]

    base_bin = base_orig[z, 0] >= base_thr
    pcc_bin = pcc_orig[z, 0] >= pcc_thr
    base_occ_bin = base_occ_boundary[z, 0] >= base_thr
    pcc_occ_bin = pcc_occ_boundary[z, 0] >= pcc_thr

    pcc_change = np.abs(pcc_orig[z, 0] - pcc_occ_boundary[z, 0])

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    axes[0, 0].imshow(img, cmap="gray")
    axes[0, 0].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 0].set_title("Original MRI + true tumour")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(img, cmap="gray")
    axes[0, 1].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 1].contour(base_bin, levels=[0.5], colors="blue", linewidths=1)
    axes[0, 1].contour(pcc_bin, levels=[0.5], colors="red", linewidths=1)
    axes[0, 1].set_title("Original prediction: blue=baseline, red=PCC")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(img, cmap="gray")
    axes[0, 2].imshow(boundary_z, cmap="Reds", alpha=0.35)
    axes[0, 2].imshow(control_z, cmap="Blues", alpha=0.30)
    axes[0, 2].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 2].set_title("Occlusion masks: red=boundary, blue=control")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(img, cmap="gray")
    axes[1, 0].imshow(core_z, cmap="Reds", alpha=0.35)
    axes[1, 0].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 0].set_title("Tumour core occlusion region")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(img, cmap="gray")
    axes[1, 1].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 1].contour(base_occ_bin, levels=[0.5], colors="blue", linewidths=1)
    axes[1, 1].contour(pcc_occ_bin, levels=[0.5], colors="red", linewidths=1)
    axes[1, 1].set_title("After boundary occlusion")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(img, cmap="gray")
    axes[1, 2].imshow(
        pcc_change,
        cmap="hot",
        alpha=np.clip(pcc_change / (pcc_change.max() + 1e-8), 0, 0.85)
    )
    axes[1, 2].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 2].set_title("|PCC original - PCC boundary-occluded|")
    axes[1, 2].axis("off")

    fig.suptitle(f"{case_id} | slice {z}", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close(fig)


# =========================
# Select quick cases
# =========================
case_metrics_all = case_metrics_all.copy()
case_metrics_all = case_metrics_all.dropna(subset=[CASE_COL, FOLD_COL]).reset_index(drop=True)

quick_df = case_metrics_all.sample(
    n=min(QUICK_CASES, len(case_metrics_all)),
    random_state=SEED
).sort_values([FOLD_COL, CASE_COL]).reset_index(drop=True)

print("Quick cases:", len(quick_df))

show_cols = [CASE_COL, FOLD_COL]
for c in [BASE_DICE_COL, PCC_DICE_COL, DICE_GAIN_COL]:
    if c is not None:
        show_cols.append(c)

display(quick_df[show_cols])


# =========================
# Main Layer3 quick experiment
# =========================
start_time = time.time()

rng = np.random.default_rng(SEED)
fold_model_cache = {}

rows = []
figure_cache = {}
failed_cases = []

for _, row in tqdm(quick_df.iterrows(), total=len(quick_df), desc="Layer3 quick occlusion audit"):
    case_id = row[CASE_COL]
    fold = int(row[FOLD_COL])

    if fold not in fold_model_cache:
        fold_model_cache[fold] = load_fold_models(fold)

    baseline_model, pcc_model, base_thr, pcc_thr = fold_model_cache[fold]

    try:
        X, Y, brain_mask, meta = load_layer1_case(case_id)
    except Exception as e:
        print("Skipping failed case:", case_id, "reason:", repr(e))
        failed_cases.append({"case_id": str(case_id), "reason": repr(e)})
        continue

    target_zyx = Y[:, 0] > 0.5
    core, boundary, avoid = make_regions(target_zyx, brain_mask)

    if core.sum() <= 0 or boundary.sum() <= 0:
        print("Skipping empty tumour/boundary case:", case_id)
        failed_cases.append({"case_id": str(case_id), "reason": "empty core or boundary"})
        continue

    # Original predictions
    base_orig, pcc_orig = predict_baseline_and_pcc(
        baseline_model, pcc_model, X, batch_size=BATCH_SIZE
    )

    base_orig_dice, base_orig_iou = dice_iou_prob(base_orig, Y, threshold=base_thr)
    pcc_orig_dice, pcc_orig_iou = dice_iou_prob(pcc_orig, Y, threshold=pcc_thr)

    # Core occlusion
    X_core = occlude_volume(X, core, replacement_value=0.0)
    base_core, pcc_core = predict_baseline_and_pcc(
        baseline_model, pcc_model, X_core, batch_size=BATCH_SIZE
    )

    base_core_dice, base_core_iou = dice_iou_prob(base_core, Y, threshold=base_thr)
    pcc_core_dice, pcc_core_iou = dice_iou_prob(pcc_core, Y, threshold=pcc_thr)

    base_core_drop = base_orig_dice - base_core_dice
    pcc_core_drop = pcc_orig_dice - pcc_core_dice

    # Boundary occlusion
    X_boundary = occlude_volume(X, boundary, replacement_value=0.0)
    base_boundary, pcc_boundary = predict_baseline_and_pcc(
        baseline_model, pcc_model, X_boundary, batch_size=BATCH_SIZE
    )

    base_boundary_dice, base_boundary_iou = dice_iou_prob(base_boundary, Y, threshold=base_thr)
    pcc_boundary_dice, pcc_boundary_iou = dice_iou_prob(pcc_boundary, Y, threshold=pcc_thr)

    base_boundary_drop = base_orig_dice - base_boundary_dice
    pcc_boundary_drop = pcc_orig_dice - pcc_boundary_dice

    # Control occlusions
    base_control_drops = []
    pcc_control_drops = []
    control_masks = []

    control_source = boundary if boundary.sum() > 0 else core

    for k in range(N_CONTROL_REGIONS):
        control = make_shifted_control(control_source, avoid, brain_mask, rng)
        control_masks.append(control)

        X_control = occlude_volume(X, control, replacement_value=0.0)

        base_control, pcc_control = predict_baseline_and_pcc(
            baseline_model, pcc_model, X_control, batch_size=BATCH_SIZE
        )

        base_control_dice, _ = dice_iou_prob(base_control, Y, threshold=base_thr)
        pcc_control_dice, _ = dice_iou_prob(pcc_control, Y, threshold=pcc_thr)

        base_control_drops.append(base_orig_dice - base_control_dice)
        pcc_control_drops.append(pcc_orig_dice - pcc_control_dice)

        del X_control, base_control, pcc_control
        gc.collect()

    base_control_drop_mean = float(np.mean(base_control_drops))
    pcc_control_drop_mean = float(np.mean(pcc_control_drops))

    # PRI
    base_core_PRI = base_core_drop - base_control_drop_mean
    pcc_core_PRI = pcc_core_drop - pcc_control_drop_mean

    base_boundary_PRI = base_boundary_drop - base_control_drop_mean
    pcc_boundary_PRI = pcc_boundary_drop - pcc_control_drop_mean

    eps = 1e-8

    out = {
        "case_id": str(case_id),
        "fold": fold,
        "patient_id": meta["patient_id"],
        "current_tp": meta["current_tp"],
        "target_voxels": int(target_zyx.sum()),
        "core_voxels": int(core.sum()),
        "boundary_voxels": int(boundary.sum()),
        "brain_voxels": int(brain_mask.sum()),
        "t1c_path": meta["t1c_path"],
        "mask_path": meta["mask_path"],

        "baseline_threshold": base_thr,
        "pcc_threshold": pcc_thr,

        "baseline_original_dice": base_orig_dice,
        "pcc_original_dice": pcc_orig_dice,
        "original_dice_gain": pcc_orig_dice - base_orig_dice,

        "baseline_original_iou": base_orig_iou,
        "pcc_original_iou": pcc_orig_iou,

        "baseline_core_occluded_dice": base_core_dice,
        "pcc_core_occluded_dice": pcc_core_dice,
        "baseline_core_drop": base_core_drop,
        "pcc_core_drop": pcc_core_drop,
        "pcc_minus_baseline_core_drop": pcc_core_drop - base_core_drop,

        "baseline_boundary_occluded_dice": base_boundary_dice,
        "pcc_boundary_occluded_dice": pcc_boundary_dice,
        "baseline_boundary_drop": base_boundary_drop,
        "pcc_boundary_drop": pcc_boundary_drop,
        "pcc_minus_baseline_boundary_drop": pcc_boundary_drop - base_boundary_drop,

        "baseline_control_drop_mean": base_control_drop_mean,
        "pcc_control_drop_mean": pcc_control_drop_mean,
        "pcc_minus_baseline_control_drop": pcc_control_drop_mean - base_control_drop_mean,

        "baseline_core_PRI": base_core_PRI,
        "pcc_core_PRI": pcc_core_PRI,
        "pcc_minus_baseline_core_PRI": pcc_core_PRI - base_core_PRI,

        "baseline_boundary_PRI": base_boundary_PRI,
        "pcc_boundary_PRI": pcc_boundary_PRI,
        "pcc_minus_baseline_boundary_PRI": pcc_boundary_PRI - base_boundary_PRI,

        "baseline_core_relative_PRI": base_core_PRI / (base_orig_dice + eps),
        "pcc_core_relative_PRI": pcc_core_PRI / (pcc_orig_dice + eps),
        "pcc_minus_baseline_core_relative_PRI": (pcc_core_PRI / (pcc_orig_dice + eps)) - (base_core_PRI / (base_orig_dice + eps)),

        "baseline_boundary_relative_PRI": base_boundary_PRI / (base_orig_dice + eps),
        "pcc_boundary_relative_PRI": pcc_boundary_PRI / (pcc_orig_dice + eps),
        "pcc_minus_baseline_boundary_relative_PRI": (pcc_boundary_PRI / (pcc_orig_dice + eps)) - (base_boundary_PRI / (base_orig_dice + eps)),

        "baseline_boundary_sensitivity_ratio": base_boundary_drop / (base_control_drop_mean + eps),
        "pcc_boundary_sensitivity_ratio": pcc_boundary_drop / (pcc_control_drop_mean + eps),
        "pcc_minus_baseline_boundary_sensitivity_ratio": (pcc_boundary_drop / (pcc_control_drop_mean + eps)) - (base_boundary_drop / (base_control_drop_mean + eps)),
    }

    rows.append(out)

    figure_cache[str(case_id)] = {
        "X": X,
        "Y": Y,
        "core": core,
        "boundary": boundary,
        "control": control_masks[0] if len(control_masks) > 0 else np.zeros_like(core),
        "base_orig": base_orig,
        "pcc_orig": pcc_orig,
        "base_boundary": base_boundary,
        "pcc_boundary": pcc_boundary,
        "base_thr": base_thr,
        "pcc_thr": pcc_thr,
        "boundary_PRI_gain": out["pcc_minus_baseline_boundary_PRI"],
    }

    del X, Y, X_core, X_boundary
    del base_orig, pcc_orig, base_core, pcc_core, base_boundary, pcc_boundary
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

case_df = pd.DataFrame(rows)
failed_df = pd.DataFrame(failed_cases)

if len(case_df) == 0:
    print("Failed cases:")
    display(failed_df)
    raise RuntimeError("No cases were successfully evaluated. Check file structure or model compatibility.")


# =========================
# Summaries
# =========================
summary_metrics = [
    "original_dice_gain",

    "pcc_minus_baseline_core_drop",
    "pcc_minus_baseline_boundary_drop",
    "pcc_minus_baseline_control_drop",

    "pcc_minus_baseline_core_PRI",
    "pcc_minus_baseline_boundary_PRI",

    "pcc_minus_baseline_core_relative_PRI",
    "pcc_minus_baseline_boundary_relative_PRI",

    "pcc_minus_baseline_boundary_sensitivity_ratio",
]

stat_df = pd.DataFrame([summarize_metric(case_df, m) for m in summary_metrics])

overall_summary = {
    "experiment": "Layer3 quick v0 pathology-reliance occlusion audit based on Layer1 FORMAL_v1 5fold FIXED",
    "quick_cases_requested": QUICK_CASES,
    "cases_evaluated": int(len(case_df)),
    "cases_failed": int(len(failed_df)),
    "control_regions_per_case": N_CONTROL_REGIONS,
    "boundary_radius": BOUNDARY_RADIUS,
    "avoid_radius": AVOID_RADIUS,
    "runtime_minutes": float((time.time() - start_time) / 60.0),

    "mean_baseline_original_dice": float(case_df["baseline_original_dice"].mean()),
    "mean_pcc_original_dice": float(case_df["pcc_original_dice"].mean()),
    "mean_original_dice_gain": float(case_df["original_dice_gain"].mean()),
    "original_dice_win_rate": float((case_df["original_dice_gain"] > 0).mean()),

    "mean_baseline_core_drop": float(case_df["baseline_core_drop"].mean()),
    "mean_pcc_core_drop": float(case_df["pcc_core_drop"].mean()),
    "mean_pcc_minus_baseline_core_drop": float(case_df["pcc_minus_baseline_core_drop"].mean()),

    "mean_baseline_boundary_drop": float(case_df["baseline_boundary_drop"].mean()),
    "mean_pcc_boundary_drop": float(case_df["pcc_boundary_drop"].mean()),
    "mean_pcc_minus_baseline_boundary_drop": float(case_df["pcc_minus_baseline_boundary_drop"].mean()),

    "mean_baseline_control_drop": float(case_df["baseline_control_drop_mean"].mean()),
    "mean_pcc_control_drop": float(case_df["pcc_control_drop_mean"].mean()),
    "mean_pcc_minus_baseline_control_drop": float(case_df["pcc_minus_baseline_control_drop"].mean()),

    "mean_baseline_core_PRI": float(case_df["baseline_core_PRI"].mean()),
    "mean_pcc_core_PRI": float(case_df["pcc_core_PRI"].mean()),
    "mean_pcc_minus_baseline_core_PRI": float(case_df["pcc_minus_baseline_core_PRI"].mean()),
    "core_PRI_win_rate": float((case_df["pcc_minus_baseline_core_PRI"] > 0).mean()),

    "mean_baseline_boundary_PRI": float(case_df["baseline_boundary_PRI"].mean()),
    "mean_pcc_boundary_PRI": float(case_df["pcc_boundary_PRI"].mean()),
    "mean_pcc_minus_baseline_boundary_PRI": float(case_df["pcc_minus_baseline_boundary_PRI"].mean()),
    "boundary_PRI_win_rate": float((case_df["pcc_minus_baseline_boundary_PRI"] > 0).mean()),

    "mean_pcc_minus_baseline_boundary_relative_PRI": float(case_df["pcc_minus_baseline_boundary_relative_PRI"].mean()),
    "boundary_relative_PRI_win_rate": float((case_df["pcc_minus_baseline_boundary_relative_PRI"] > 0).mean()),
}

summary_df = pd.DataFrame([overall_summary])


# =========================
# Save figures
# =========================
top_cases = case_df.sort_values("pcc_minus_baseline_boundary_PRI", ascending=False)["case_id"].head(2).tolist()
worst_cases = case_df.sort_values("pcc_minus_baseline_boundary_PRI", ascending=True)["case_id"].head(1).tolist()
fig_cases = list(dict.fromkeys(top_cases + worst_cases))

for case_id in fig_cases:
    item = figure_cache[case_id]
    fig_path = FIG_DIR / f"{case_id}_layer3_occlusion_audit.png"

    save_case_figure(
        case_id=case_id,
        X=item["X"],
        Y=item["Y"],
        core=item["core"],
        boundary=item["boundary"],
        control=item["control"],
        base_orig=item["base_orig"],
        pcc_orig=item["pcc_orig"],
        base_occ_boundary=item["base_boundary"],
        pcc_occ_boundary=item["pcc_boundary"],
        base_thr=item["base_thr"],
        pcc_thr=item["pcc_thr"],
        out_path=fig_path,
    )

print("Saved figures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(p)


# =========================
# Save outputs
# =========================
case_path = OUT_DIR / "Layer3_quick_v0_case_metrics.csv"
summary_path = OUT_DIR / "Layer3_quick_v0_overall_summary.csv"
stats_path = OUT_DIR / "Layer3_quick_v0_paired_stats_bootstrap.csv"
failed_path = OUT_DIR / "Layer3_quick_v0_failed_cases.csv"
run_info_path = OUT_DIR / "Layer3_quick_v0_run_info.json"

case_df.to_csv(case_path, index=False)
summary_df.to_csv(summary_path, index=False)
stat_df.to_csv(stats_path, index=False)
failed_df.to_csv(failed_path, index=False)

with open(run_info_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)

print("\n" + "=" * 100)
print("Layer3 quick v0 FIXED finished.")
print("Runtime minutes:", overall_summary["runtime_minutes"])
print("Saved to:", OUT_DIR)

print("\nOverall summary:")
display(summary_df)

print("\nPaired stats / bootstrap CI:")
display(stat_df)

print("\nCase metrics:")
display(case_df)

if len(failed_df) > 0:
    print("\nFailed cases:")
    display(failed_df)

print("\nSaved files:")
print(case_path)
print(summary_path)
print(stats_path)
print(failed_path)
print(run_info_path)
print("Figures:", FIG_DIR)


# =========================
# Quick interpretation
# =========================
boundary_gain = overall_summary["mean_pcc_minus_baseline_boundary_PRI"]
boundary_win = overall_summary["boundary_PRI_win_rate"]
control_extra = overall_summary["mean_pcc_minus_baseline_control_drop"]
rel_boundary_gain = overall_summary["mean_pcc_minus_baseline_boundary_relative_PRI"]

print("\n" + "=" * 100)
print("Quick interpretation:")

print("mean_pcc_minus_baseline_boundary_PRI:", boundary_gain)
print("boundary_PRI_win_rate:", boundary_win)
print("mean_pcc_minus_baseline_control_drop:", control_extra)
print("mean_pcc_minus_baseline_boundary_relative_PRI:", rel_boundary_gain)

if boundary_gain > 0 and boundary_win >= 0.65 and control_extra <= 0.02 and rel_boundary_gain > 0:
    print("\nPASS / strong quick signal:")
    print("PCC shows higher boundary pathology-reliance than baseline, without excessive control sensitivity.")
elif boundary_gain > 0 and boundary_win >= 0.55 and rel_boundary_gain > 0:
    print("\nPARTIAL PASS / moderate quick signal:")
    print("PCC shows a positive pathology-reliance trend. Formal 40-case audit is recommended.")
else:
    print("\nNOT PASSED in quick version:")
    print("The current quick audit does not yet show clear PCC pathology-reliance advantage.")

In [ ]:
# ============================================================
# Layer3 Preflight Diagnostic:
# Reproduce Layer1 original Dice before occlusion
#
# Goal:
#   Find the preprocessing / normalization mode that reproduces
#   Layer1 FORMAL_v1 case_metrics Dice using saved checkpoints.
#
# If reproduction fails, Layer3 occlusion audit is not interpretable.
# ============================================================

from pathlib import Path
import re, json, time, random, gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F

import nibabel as nib


# =========================
# Config
# =========================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DIAG_CASES = 12
BATCH_SIZE = 32

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# =========================
# Fixed paths
# =========================
LAYER1_DIR = Path(
    "/kaggle/input/datasets/jeechangxin/layer1-formal-v1-5fold-results-backup/"
    "pcc_independent_baseline/stage2_pcc_guided_student_learning/"
    "layer1_current_segmentation_FORMAL_v1_5fold"
)

CKPT_DIR = LAYER1_DIR / "checkpoints"
CASE_METRICS_PATH = LAYER1_DIR / "layer1_FORMAL_v1_case_metrics.csv"

DATA_ROOT = Path(
    "/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/"
    "PKG - MU-Glioma-Post/MU-Glioma-Post"
)

OUT_DIR = Path("/kaggle/working/Layer3_reproduction_diagnostic_FORMAL_v1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert LAYER1_DIR.exists(), LAYER1_DIR
assert CKPT_DIR.exists(), CKPT_DIR
assert CASE_METRICS_PATH.exists(), CASE_METRICS_PATH
assert DATA_ROOT.exists(), DATA_ROOT

case_df_all = pd.read_csv(CASE_METRICS_PATH)

print("Layer1 dir:", LAYER1_DIR)
print("Checkpoint dir:", CKPT_DIR)
print("Dataset root:", DATA_ROOT)
print("case_metrics:", case_df_all.shape)
display(case_df_all.head())


# =========================
# Model architecture
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


def get_threshold_from_ckpt(ckpt, default=0.5):
    for k in ["threshold", "best_threshold", "selected_threshold", "thr"]:
        if k in ckpt:
            return float(ckpt[k])
    return float(default)


def load_fold_models(fold):
    base_path = CKPT_DIR / f"baseline_seg_fold_{fold}_FORMAL_v1.pt"
    pcc_path = CKPT_DIR / f"pcc_seg_corrector_fold_{fold}_FORMAL_v1.pt"

    assert base_path.exists(), base_path
    assert pcc_path.exists(), pcc_path

    base_ckpt = torch.load(base_path, map_location=DEVICE)
    pcc_ckpt = torch.load(pcc_path, map_location=DEVICE)

    baseline = SmallUNet2D(in_ch=1, out_ch=1, base=16).to(DEVICE)
    pcc = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)

    baseline.load_state_dict(base_ckpt["model_state_dict"])
    pcc.load_state_dict(pcc_ckpt["model_state_dict"])

    baseline.eval()
    pcc.eval()

    base_thr = get_threshold_from_ckpt(base_ckpt, default=0.5)
    pcc_thr = get_threshold_from_ckpt(pcc_ckpt, default=0.5)

    return baseline, pcc, base_thr, pcc_thr


# =========================
# File loading helpers
# =========================
def parse_case_id(case_id):
    case_id = str(case_id)
    m_patient = re.search(r"(PatientID_\d+)", case_id)
    m_t = re.search(r"_T(\d+)", case_id)

    if m_patient is None or m_t is None:
        raise ValueError(f"Cannot parse case_id: {case_id}")

    return m_patient.group(1), int(m_t.group(1))


def find_patient_dir(patient_id):
    direct = DATA_ROOT / patient_id
    if direct.exists():
        return direct

    hits = list(DATA_ROOT.glob(f"**/{patient_id}"))
    hits = [h for h in hits if h.is_dir()]

    if not hits:
        raise FileNotFoundError(f"Patient folder not found: {patient_id}")

    return hits[0]


def find_timepoint_file(patient_dir, tp, kind):
    patient_dir = Path(patient_dir)

    if kind == "brain_t1c":
        patterns = ["*brain_t1c.nii", "*brain_t1c.nii.gz"]
    elif kind == "tumorMask":
        patterns = ["*tumorMask.nii", "*tumorMask.nii.gz"]
    else:
        patterns = [f"*{kind}.nii", f"*{kind}.nii.gz"]

    candidates = []
    for pat in patterns:
        candidates.extend(list(patient_dir.rglob(pat)))

    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        raise FileNotFoundError(f"No {kind} found in {patient_dir}")

    tp_tokens = [
        f"Timepoint_{tp}",
        f"Timepoint-{tp}",
        f"Timepoint {tp}",
        f"_T{tp}_",
        f"/T{tp}/",
        f"TP{tp}",
    ]

    filtered = []
    for p in candidates:
        s = str(p)
        if any(tok in s for tok in tp_tokens) or f"_Timepoint_{tp}_" in p.name:
            filtered.append(p)

    if filtered:
        return filtered[0]

    raise FileNotFoundError(
        f"Found {len(candidates)} {kind} candidates but none matched Timepoint {tp}. "
        f"First candidates: {[str(c) for c in candidates[:5]]}"
    )


def to_z_hw_default(arr):
    arr = np.asarray(arr)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume, got {arr.shape}")

    # Common NIfTI: H,W,Z -> Z,H,W
    if arr.shape[0] >= 128 and arr.shape[1] >= 128 and arr.shape[2] < 200:
        arr = np.moveaxis(arr, -1, 0)

    return arr.astype(np.float32)


def load_raw_case(case_id):
    patient_id, current_tp = parse_case_id(case_id)
    pdir = find_patient_dir(patient_id)

    t1c_path = find_timepoint_file(pdir, current_tp, "brain_t1c")
    mask_path = find_timepoint_file(pdir, current_tp, "tumorMask")

    img_raw = to_z_hw_default(nib.load(str(t1c_path)).get_fdata()).astype(np.float32)
    mask_raw = to_z_hw_default(nib.load(str(mask_path)).get_fdata()).astype(np.float32)

    mask = (mask_raw > 0.5).astype(np.float32)
    brain = np.abs(img_raw) > 1e-6

    if brain.sum() < 100:
        brain = np.ones_like(img_raw, dtype=bool)

    return img_raw, mask, brain, {
        "patient_id": patient_id,
        "current_tp": current_tp,
        "t1c_path": str(t1c_path),
        "mask_path": str(mask_path),
    }


# =========================
# Normalization candidates
# =========================
def normalize_candidate(vol, brain, mode):
    vol = np.nan_to_num(vol.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    brain = brain.astype(bool)

    vals = vol[brain]
    if vals.size < 100:
        vals = vol.reshape(-1)

    if mode == "zscore_brain_clip5":
        mean = float(vals.mean())
        std = float(vals.std() + 1e-6)
        x = (vol - mean) / std
        x = np.clip(x, -5, 5)

    elif mode == "zscore_brain_noclip":
        mean = float(vals.mean())
        std = float(vals.std() + 1e-6)
        x = (vol - mean) / std

    elif mode == "minmax_brain":
        lo = float(vals.min())
        hi = float(vals.max())
        x = (vol - lo) / (hi - lo + 1e-6)
        x = np.clip(x, 0, 1)

    elif mode == "p01_p99_brain":
        lo, hi = np.percentile(vals, [1, 99])
        x = (vol - lo) / (hi - lo + 1e-6)
        x = np.clip(x, 0, 1)

    elif mode == "p05_p995_brain":
        lo, hi = np.percentile(vals, [0.5, 99.5])
        x = (vol - lo) / (hi - lo + 1e-6)
        x = np.clip(x, 0, 1)

    elif mode == "p02_p98_brain":
        lo, hi = np.percentile(vals, [2, 98])
        x = (vol - lo) / (hi - lo + 1e-6)
        x = np.clip(x, 0, 1)

    elif mode == "divide_max_brain":
        hi = float(vals.max())
        x = vol / (hi + 1e-6)
        x = np.clip(x, 0, 1)

    elif mode == "raw_float":
        x = vol.astype(np.float32)

    else:
        raise ValueError(mode)

    return x.astype(np.float32)


NORM_MODES = [
    "zscore_brain_clip5",
    "zscore_brain_noclip",
    "minmax_brain",
    "p01_p99_brain",
    "p05_p995_brain",
    "p02_p98_brain",
    "divide_max_brain",
    "raw_float",
]


# =========================
# Prediction helpers
# =========================
@torch.no_grad()
def predict_baseline_prob(model, X, batch_size=32):
    model.eval()
    outs = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X[start:start+batch_size]).float().to(DEVICE)
        logits = model(xb)
        prob = torch.sigmoid(logits).detach().cpu().numpy().astype(np.float32)
        outs.append(prob)

    return np.concatenate(outs, axis=0)


@torch.no_grad()
def predict_pcc_prob(pcc_model, X, base_prob, mode, batch_size=32):
    pcc_model.eval()
    outs = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X[start:start+batch_size]).float().to(DEVICE)
        bp = torch.from_numpy(base_prob[start:start+batch_size]).float().to(DEVICE)

        pcc_input = torch.cat([xb, bp], dim=1)
        raw = pcc_model(pcc_input)

        base_logit = torch.logit(torch.clamp(bp, 1e-4, 1.0 - 1e-4))

        if mode == "residual_tanh3":
            logits = base_logit + 3.0 * torch.tanh(raw)
        elif mode == "residual_tanh2":
            logits = base_logit + 2.0 * torch.tanh(raw)
        elif mode == "residual_tanh1":
            logits = base_logit + 1.0 * torch.tanh(raw)
        elif mode == "residual_raw1":
            logits = base_logit + raw
        elif mode == "direct_logits":
            logits = raw
        else:
            raise ValueError(mode)

        prob = torch.sigmoid(logits).detach().cpu().numpy().astype(np.float32)
        outs.append(prob)

    return np.concatenate(outs, axis=0)


PCC_MODES = [
    "residual_tanh3",
    "residual_tanh2",
    "residual_tanh1",
    "residual_raw1",
    "direct_logits",
]


def dice_prob(prob, target, threshold=0.5, eps=1e-8):
    pred = prob >= threshold
    target = target.astype(bool)

    inter = np.logical_and(pred, target).sum()
    denom = pred.sum() + target.sum()

    return float((2 * inter + eps) / (denom + eps))


# =========================
# Select diagnostic cases
# =========================
diag_df = case_df_all.sample(
    n=min(DIAG_CASES, len(case_df_all)),
    random_state=SEED
).sort_values(["fold", "case_id"]).reset_index(drop=True)

print("\nDiagnostic cases:")
display(diag_df[["case_id", "fold", "baseline_dice", "pcc_dice", "dice_gain", "target_voxels"]])


# =========================
# Cache raw cases and models
# =========================
print("\nLoading raw cases...")
raw_cache = {}

for _, row in tqdm(diag_df.iterrows(), total=len(diag_df)):
    cid = row["case_id"]
    raw_cache[cid] = load_raw_case(cid)

fold_model_cache = {}
for fold in sorted(diag_df["fold"].unique()):
    fold_model_cache[int(fold)] = load_fold_models(int(fold))


# =========================
# Step 1: diagnose baseline normalization
# =========================
print("\nStep 1: Testing normalization modes for baseline reproduction...")

norm_rows = []

for norm_mode in NORM_MODES:
    case_rows = []

    for _, row in tqdm(diag_df.iterrows(), total=len(diag_df), desc=f"Norm {norm_mode}"):
        cid = row["case_id"]
        fold = int(row["fold"])
        target_baseline_dice = float(row["baseline_dice"])

        img_raw, mask, brain, meta = raw_cache[cid]
        x = normalize_candidate(img_raw, brain, norm_mode)

        X = x[:, None, :, :].astype(np.float32)
        Y = mask[:, None, :, :].astype(np.float32)

        baseline_model, pcc_model, base_thr, pcc_thr = fold_model_cache[fold]

        base_prob = predict_baseline_prob(baseline_model, X, batch_size=BATCH_SIZE)
        reproduced_dice = dice_prob(base_prob, Y, threshold=base_thr)

        case_rows.append({
            "case_id": cid,
            "fold": fold,
            "norm_mode": norm_mode,
            "csv_baseline_dice": target_baseline_dice,
            "reproduced_baseline_dice": reproduced_dice,
            "abs_error": abs(reproduced_dice - target_baseline_dice),
            "signed_error": reproduced_dice - target_baseline_dice,
            "base_threshold": base_thr,
        })

        del X, Y, base_prob
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    tmp = pd.DataFrame(case_rows)

    norm_rows.append({
        "norm_mode": norm_mode,
        "mean_csv_baseline_dice": float(tmp["csv_baseline_dice"].mean()),
        "mean_reproduced_baseline_dice": float(tmp["reproduced_baseline_dice"].mean()),
        "mean_abs_error": float(tmp["abs_error"].mean()),
        "median_abs_error": float(tmp["abs_error"].median()),
        "max_abs_error": float(tmp["abs_error"].max()),
        "mean_signed_error": float(tmp["signed_error"].mean()),
        "case_level_correlation": float(tmp[["csv_baseline_dice", "reproduced_baseline_dice"]].corr().iloc[0,1]),
    })

norm_summary_df = pd.DataFrame(norm_rows).sort_values("mean_abs_error").reset_index(drop=True)

print("\nBaseline normalization diagnostic summary:")
display(norm_summary_df)

best_norm = norm_summary_df.iloc[0]["norm_mode"]
print("\nBEST NORM MODE:", best_norm)


# =========================
# Step 2: using best norm, diagnose PCC formula
# =========================
print("\nStep 2: Testing PCC inference formula modes using best norm...")

pcc_rows = []
pcc_case_all_rows = []

for pcc_mode in PCC_MODES:
    case_rows = []

    for _, row in tqdm(diag_df.iterrows(), total=len(diag_df), desc=f"PCC mode {pcc_mode}"):
        cid = row["case_id"]
        fold = int(row["fold"])

        target_base_dice = float(row["baseline_dice"])
        target_pcc_dice = float(row["pcc_dice"])

        img_raw, mask, brain, meta = raw_cache[cid]
        x = normalize_candidate(img_raw, brain, best_norm)

        X = x[:, None, :, :].astype(np.float32)
        Y = mask[:, None, :, :].astype(np.float32)

        baseline_model, pcc_model, base_thr, pcc_thr = fold_model_cache[fold]

        base_prob = predict_baseline_prob(baseline_model, X, batch_size=BATCH_SIZE)
        pcc_prob = predict_pcc_prob(pcc_model, X, base_prob, pcc_mode, batch_size=BATCH_SIZE)

        base_dice = dice_prob(base_prob, Y, threshold=base_thr)
        pcc_dice = dice_prob(pcc_prob, Y, threshold=pcc_thr)

        case_rows.append({
            "case_id": cid,
            "fold": fold,
            "norm_mode": best_norm,
            "pcc_mode": pcc_mode,

            "csv_baseline_dice": target_base_dice,
            "reproduced_baseline_dice": base_dice,
            "baseline_abs_error": abs(base_dice - target_base_dice),

            "csv_pcc_dice": target_pcc_dice,
            "reproduced_pcc_dice": pcc_dice,
            "pcc_abs_error": abs(pcc_dice - target_pcc_dice),
            "pcc_signed_error": pcc_dice - target_pcc_dice,

            "csv_dice_gain": target_pcc_dice - target_base_dice,
            "reproduced_dice_gain": pcc_dice - base_dice,
            "gain_abs_error": abs((pcc_dice - base_dice) - (target_pcc_dice - target_base_dice)),

            "base_threshold": base_thr,
            "pcc_threshold": pcc_thr,
        })

        del X, Y, base_prob, pcc_prob
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    tmp = pd.DataFrame(case_rows)
    pcc_case_all_rows.extend(case_rows)

    pcc_rows.append({
        "norm_mode": best_norm,
        "pcc_mode": pcc_mode,

        "mean_csv_pcc_dice": float(tmp["csv_pcc_dice"].mean()),
        "mean_reproduced_pcc_dice": float(tmp["reproduced_pcc_dice"].mean()),
        "mean_pcc_abs_error": float(tmp["pcc_abs_error"].mean()),
        "median_pcc_abs_error": float(tmp["pcc_abs_error"].median()),
        "max_pcc_abs_error": float(tmp["pcc_abs_error"].max()),
        "mean_pcc_signed_error": float(tmp["pcc_signed_error"].mean()),

        "mean_csv_gain": float(tmp["csv_dice_gain"].mean()),
        "mean_reproduced_gain": float(tmp["reproduced_dice_gain"].mean()),
        "mean_gain_abs_error": float(tmp["gain_abs_error"].mean()),

        "pcc_case_level_correlation": float(tmp[["csv_pcc_dice", "reproduced_pcc_dice"]].corr().iloc[0,1]),
    })

pcc_summary_df = pd.DataFrame(pcc_rows).sort_values("mean_pcc_abs_error").reset_index(drop=True)
pcc_case_df = pd.DataFrame(pcc_case_all_rows)

print("\nPCC formula diagnostic summary:")
display(pcc_summary_df)

best_pcc_mode = pcc_summary_df.iloc[0]["pcc_mode"]
print("\nBEST PCC MODE:", best_pcc_mode)


# =========================
# Save outputs
# =========================
norm_summary_path = OUT_DIR / "Layer3_reproduction_norm_diagnostic_summary.csv"
pcc_summary_path = OUT_DIR / "Layer3_reproduction_pcc_formula_summary.csv"
pcc_case_path = OUT_DIR / "Layer3_reproduction_pcc_formula_case_metrics.csv"
best_config_path = OUT_DIR / "Layer3_reproduction_best_config.json"

norm_summary_df.to_csv(norm_summary_path, index=False)
pcc_summary_df.to_csv(pcc_summary_path, index=False)
pcc_case_df.to_csv(pcc_case_path, index=False)

best_config = {
    "best_norm_mode": str(best_norm),
    "best_pcc_mode": str(best_pcc_mode),
    "baseline_mean_abs_error": float(norm_summary_df.iloc[0]["mean_abs_error"]),
    "pcc_mean_abs_error": float(pcc_summary_df.iloc[0]["mean_pcc_abs_error"]),
    "gain_mean_abs_error": float(pcc_summary_df.iloc[0]["mean_gain_abs_error"]),
    "diagnostic_cases": int(len(diag_df)),
}

with open(best_config_path, "w", encoding="utf-8") as f:
    json.dump(best_config, f, indent=2)

print("\nSaved files:")
print(norm_summary_path)
print(pcc_summary_path)
print(pcc_case_path)
print(best_config_path)

print("\nBest config:")
print(json.dumps(best_config, indent=2))


# =========================
# Interpretation gate
# =========================
print("\n" + "=" * 100)
print("REPRODUCTION GATE INTERPRETATION")

base_mae = best_config["baseline_mean_abs_error"]
pcc_mae = best_config["pcc_mean_abs_error"]

print("Best baseline MAE:", base_mae)
print("Best PCC MAE:", pcc_mae)

if base_mae <= 0.03 and pcc_mae <= 0.03:
    print("PASS: reproduction is good enough for Layer3 occlusion audit.")
    print("Use this best_norm_mode and best_pcc_mode in the Layer3 quick code.")
elif base_mae <= 0.05 and pcc_mae <= 0.05:
    print("PARTIAL PASS: reproduction is close but not perfect.")
    print("Layer3 can be used cautiously, but formal results should verify preprocessing.")
else:
    print("FAIL: reproduction is not close enough.")
    print("Do NOT interpret occlusion results yet.")
    print("Next step: inspect original Layer1 preprocessing or use saved preprocessed arrays/prediction maps if available.")

In [ ]:
# ============================================================
# Layer3 quick v1: Pathology-Reliance Occlusion Audit
# REPRODUCTION-PASSED VERSION
#
# Confirmed by diagnostic:
#   best_norm_mode = p01_p99_brain
#   best_pcc_mode  = residual_tanh3
#
# Goal:
#   Test whether PCC is more functionally dependent on true
#   pathological structures than the baseline.
#
# Main metric:
#   PRI = pathology_occlusion_drop - control_occlusion_drop
#
# Success signal:
#   PCC_boundary_PRI > Baseline_boundary_PRI
# ============================================================

from pathlib import Path
import re, json, time, gc, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import nibabel as nib
import scipy.ndimage as ndi

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None


# =========================
# Config
# =========================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

QUICK_CASES = 12
N_CONTROL_REGIONS = 5

BATCH_SIZE = 32
BOUNDARY_RADIUS = 5
AVOID_RADIUS = 8

PCC_MAX_DELTA_LOGIT = 3.0

OUT_DIR = Path("/kaggle/working/Layer3_quick_v1_reproduction_passed_occlusion_FORMAL_v1")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUT_DIR)


# =========================
# Fixed Kaggle paths
# =========================
LAYER1_DIR = Path(
    "/kaggle/input/datasets/jeechangxin/layer1-formal-v1-5fold-results-backup/"
    "pcc_independent_baseline/stage2_pcc_guided_student_learning/"
    "layer1_current_segmentation_FORMAL_v1_5fold"
)

CKPT_DIR = LAYER1_DIR / "checkpoints"
CASE_METRICS_PATH = LAYER1_DIR / "layer1_FORMAL_v1_case_metrics.csv"

DATA_ROOT = Path(
    "/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/"
    "PKG - MU-Glioma-Post/MU-Glioma-Post"
)

assert LAYER1_DIR.exists(), LAYER1_DIR
assert CKPT_DIR.exists(), CKPT_DIR
assert CASE_METRICS_PATH.exists(), CASE_METRICS_PATH
assert DATA_ROOT.exists(), DATA_ROOT

case_metrics_all = pd.read_csv(CASE_METRICS_PATH)

print("Layer1 dir:", LAYER1_DIR)
print("Checkpoint dir:", CKPT_DIR)
print("Dataset root:", DATA_ROOT)
print("case_metrics:", case_metrics_all.shape)
display(case_metrics_all.head())


# =========================
# Model architecture
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


def get_threshold_from_ckpt(ckpt, default=0.5):
    for k in ["threshold", "best_threshold", "selected_threshold", "thr"]:
        if k in ckpt:
            return float(ckpt[k])
    return float(default)


def load_fold_models(fold):
    base_path = CKPT_DIR / f"baseline_seg_fold_{fold}_FORMAL_v1.pt"
    pcc_path = CKPT_DIR / f"pcc_seg_corrector_fold_{fold}_FORMAL_v1.pt"

    assert base_path.exists(), base_path
    assert pcc_path.exists(), pcc_path

    base_ckpt = torch.load(base_path, map_location=DEVICE)
    pcc_ckpt = torch.load(pcc_path, map_location=DEVICE)

    baseline = SmallUNet2D(in_ch=1, out_ch=1, base=16).to(DEVICE)
    pcc = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)

    baseline.load_state_dict(base_ckpt["model_state_dict"])
    pcc.load_state_dict(pcc_ckpt["model_state_dict"])

    baseline.eval()
    pcc.eval()

    base_thr = get_threshold_from_ckpt(base_ckpt, 0.5)
    pcc_thr = get_threshold_from_ckpt(pcc_ckpt, 0.5)

    print(f"Loaded fold {fold}: base_thr={base_thr}, pcc_thr={pcc_thr}")

    return baseline, pcc, base_thr, pcc_thr


# =========================
# File loading helpers
# =========================
def parse_case_id(case_id):
    case_id = str(case_id)
    m_patient = re.search(r"(PatientID_\d+)", case_id)
    m_t = re.search(r"_T(\d+)", case_id)

    if m_patient is None or m_t is None:
        raise ValueError(f"Cannot parse case_id: {case_id}")

    return m_patient.group(1), int(m_t.group(1))


def find_patient_dir(patient_id):
    direct = DATA_ROOT / patient_id
    if direct.exists():
        return direct

    hits = list(DATA_ROOT.glob(f"**/{patient_id}"))
    hits = [h for h in hits if h.is_dir()]

    if not hits:
        raise FileNotFoundError(f"Patient folder not found: {patient_id}")

    return hits[0]


def find_timepoint_file(patient_dir, tp, kind):
    patient_dir = Path(patient_dir)

    if kind == "brain_t1c":
        patterns = ["*brain_t1c.nii", "*brain_t1c.nii.gz"]
    elif kind == "tumorMask":
        patterns = ["*tumorMask.nii", "*tumorMask.nii.gz"]
    else:
        patterns = [f"*{kind}.nii", f"*{kind}.nii.gz"]

    candidates = []
    for pat in patterns:
        candidates.extend(list(patient_dir.rglob(pat)))

    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        raise FileNotFoundError(f"No {kind} found in {patient_dir}")

    tp_tokens = [
        f"Timepoint_{tp}",
        f"Timepoint-{tp}",
        f"Timepoint {tp}",
        f"_T{tp}_",
        f"/T{tp}/",
        f"TP{tp}",
    ]

    filtered = []
    for p in candidates:
        s = str(p)
        if any(tok in s for tok in tp_tokens) or f"_Timepoint_{tp}_" in p.name:
            filtered.append(p)

    if filtered:
        return filtered[0]

    raise FileNotFoundError(
        f"Found {len(candidates)} {kind} candidates but none matched Timepoint {tp}. "
        f"First candidates: {[str(c) for c in candidates[:5]]}"
    )


def to_z_hw(arr):
    arr = np.asarray(arr)

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {arr.shape}")

    if arr.shape[0] >= 128 and arr.shape[1] >= 128 and arr.shape[2] < 200:
        arr = np.moveaxis(arr, -1, 0)

    return arr.astype(np.float32)


def normalize_p01_p99_brain(vol, brain):
    vol = np.nan_to_num(vol.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    brain = brain.astype(bool)

    vals = vol[brain]
    if vals.size < 100:
        vals = vol.reshape(-1)

    lo, hi = np.percentile(vals, [1, 99])
    x = (vol - lo) / (hi - lo + 1e-6)
    x = np.clip(x, 0.0, 1.0).astype(np.float32)

    brain_mean_norm = float(x[brain].mean()) if brain.sum() > 0 else float(x.mean())

    return x, brain_mean_norm


def load_layer1_case(case_id):
    patient_id, current_tp = parse_case_id(case_id)
    pdir = find_patient_dir(patient_id)

    t1c_path = find_timepoint_file(pdir, current_tp, "brain_t1c")
    mask_path = find_timepoint_file(pdir, current_tp, "tumorMask")

    img_raw = to_z_hw(nib.load(str(t1c_path)).get_fdata()).astype(np.float32)
    mask_raw = to_z_hw(nib.load(str(mask_path)).get_fdata()).astype(np.float32)

    mask = mask_raw > 0.5
    brain = np.abs(img_raw) > 1e-6
    if brain.sum() < 100:
        brain = np.ones_like(img_raw, dtype=bool)

    img_norm, brain_mean_norm = normalize_p01_p99_brain(img_raw, brain)

    X = img_norm[:, None, :, :].astype(np.float32)
    Y = mask[:, None, :, :].astype(np.float32)

    return X, Y, brain.astype(bool), brain_mean_norm, {
        "patient_id": patient_id,
        "current_tp": current_tp,
        "t1c_path": str(t1c_path),
        "mask_path": str(mask_path),
    }


# =========================
# Prediction and metrics
# =========================
@torch.no_grad()
def predict_baseline_and_pcc(baseline_model, pcc_model, X, batch_size=32):
    baseline_model.eval()
    pcc_model.eval()

    base_probs = []
    pcc_probs = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X[start:start + batch_size]).float().to(DEVICE)

        base_logits = baseline_model(xb)
        base_prob = torch.sigmoid(base_logits)

        pcc_input = torch.cat([xb, base_prob], dim=1)
        residual_raw = pcc_model(pcc_input)

        corrected_logits = base_logits + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)
        pcc_prob = torch.sigmoid(corrected_logits)

        base_probs.append(base_prob.detach().cpu().numpy().astype(np.float32))
        pcc_probs.append(pcc_prob.detach().cpu().numpy().astype(np.float32))

    return np.concatenate(base_probs, axis=0), np.concatenate(pcc_probs, axis=0)


def dice_iou_prob(prob, target, threshold=0.5, eps=1e-8):
    pred = prob >= threshold
    target = target.astype(bool)

    inter = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()
    pred_sum = pred.sum()
    target_sum = target.sum()

    dice = (2 * inter + eps) / (pred_sum + target_sum + eps)
    iou = (inter + eps) / (union + eps)

    return float(dice), float(iou)


def occlude_volume(X, occ_mask_zyx, replacement_value):
    X_occ = X.copy()
    X_occ[:, 0, :, :][occ_mask_zyx] = float(replacement_value)
    return X_occ


# =========================
# Region helpers
# =========================
def make_regions(target_zyx, brain_zyx):
    target = target_zyx.astype(bool)
    brain = brain_zyx.astype(bool)

    core = np.logical_and(target, brain)

    dil = ndi.binary_dilation(core, iterations=BOUNDARY_RADIUS)
    ero = ndi.binary_erosion(core, iterations=BOUNDARY_RADIUS)
    boundary = np.logical_and(dil, np.logical_not(ero))
    boundary = np.logical_and(boundary, brain)

    peritumour = ndi.binary_dilation(core, iterations=10)
    peritumour = np.logical_and(peritumour, np.logical_not(core))
    peritumour = np.logical_and(peritumour, brain)

    avoid = ndi.binary_dilation(core, iterations=AVOID_RADIUS)
    avoid = np.logical_or(avoid, np.logical_not(brain))

    return core, boundary, peritumour, avoid


def make_shifted_control(region_mask, avoid_mask, brain_mask, rng, tries=100):
    region = region_mask.astype(bool)
    avoid = avoid_mask.astype(bool)
    brain = brain_mask.astype(bool)

    n = int(region.sum())
    if n <= 0:
        return np.zeros_like(region, dtype=bool)

    zdim, h, w = region.shape
    best = None
    best_count = -1

    for _ in range(tries):
        dz = rng.integers(-max(2, zdim // 3), max(3, zdim // 3))
        dy = rng.integers(-max(8, h // 3), max(9, h // 3))
        dx = rng.integers(-max(8, w // 3), max(9, w // 3))

        shifted = np.roll(region, shift=(dz, dy, dx), axis=(0, 1, 2))
        shifted = np.logical_and(shifted, brain)
        shifted = np.logical_and(shifted, np.logical_not(avoid))

        c = int(shifted.sum())

        if c > best_count:
            best = shifted
            best_count = c

        if c >= int(0.80 * n):
            return shifted

    candidates = np.logical_and(brain, np.logical_not(avoid))
    idx = np.argwhere(candidates)

    take = min(n, len(idx))
    chosen = idx[rng.choice(len(idx), size=take, replace=False)]

    control = np.zeros_like(region, dtype=bool)
    control[chosen[:, 0], chosen[:, 1], chosen[:, 2]] = True

    return control


# =========================
# Stats helpers
# =========================
def bootstrap_ci(vals, n_boot=3000, seed=42):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = []

    for _ in range(n_boot):
        s = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.mean(s))

    return float(np.mean(vals)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def summarize_metric(case_df, metric):
    vals = case_df[metric].values.astype(float)
    mean, low, high = bootstrap_ci(vals)

    if wilcoxon is not None:
        try:
            w = wilcoxon(vals, alternative="greater")
            p = float(w.pvalue)
            stat = float(w.statistic)
        except Exception:
            p = np.nan
            stat = np.nan
    else:
        p = np.nan
        stat = np.nan

    return {
        "metric": metric,
        "mean": mean,
        "median": float(np.nanmedian(vals)),
        "ci95_low": low,
        "ci95_high": high,
        "positive_cases": int(np.sum(vals > 0)),
        "negative_cases": int(np.sum(vals < 0)),
        "positive_rate": float(np.mean(vals > 0)),
        "wilcoxon_greater_statistic": stat,
        "wilcoxon_greater_p_value": p,
    }


# =========================
# Figure helper
# =========================
def save_case_figure(
    case_id, X, Y, boundary, control,
    base_orig, pcc_orig, base_occ_boundary, pcc_occ_boundary,
    base_thr, pcc_thr, out_path
):
    target = Y[:, 0] > 0.5

    score = boundary.reshape(boundary.shape[0], -1).sum(axis=1)
    if score.max() <= 0:
        score = target.reshape(target.shape[0], -1).sum(axis=1)

    z = int(np.argmax(score))

    img = X[z, 0]
    true_z = target[z]
    boundary_z = boundary[z]
    control_z = control[z]

    base_bin = base_orig[z, 0] >= base_thr
    pcc_bin = pcc_orig[z, 0] >= pcc_thr
    base_occ_bin = base_occ_boundary[z, 0] >= base_thr
    pcc_occ_bin = pcc_occ_boundary[z, 0] >= pcc_thr

    pcc_change = np.abs(pcc_orig[z, 0] - pcc_occ_boundary[z, 0])

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    axes[0, 0].imshow(img, cmap="gray")
    axes[0, 0].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 0].set_title("Original MRI + true mask")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(img, cmap="gray")
    axes[0, 1].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 1].contour(base_bin, levels=[0.5], colors="blue", linewidths=1)
    axes[0, 1].contour(pcc_bin, levels=[0.5], colors="red", linewidths=1)
    axes[0, 1].set_title("Original: blue=baseline, red=PCC")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(img, cmap="gray")
    axes[0, 2].imshow(boundary_z, cmap="Reds", alpha=0.35)
    axes[0, 2].imshow(control_z, cmap="Blues", alpha=0.30)
    axes[0, 2].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 2].set_title("Occlusion masks: red=boundary, blue=control")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(img, cmap="gray")
    axes[1, 0].imshow(boundary_z, cmap="Reds", alpha=0.35)
    axes[1, 0].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 0].set_title("Boundary occlusion region")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(img, cmap="gray")
    axes[1, 1].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 1].contour(base_occ_bin, levels=[0.5], colors="blue", linewidths=1)
    axes[1, 1].contour(pcc_occ_bin, levels=[0.5], colors="red", linewidths=1)
    axes[1, 1].set_title("After boundary occlusion")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(img, cmap="gray")
    axes[1, 2].imshow(
        pcc_change,
        cmap="hot",
        alpha=np.clip(pcc_change / (pcc_change.max() + 1e-8), 0, 0.85)
    )
    axes[1, 2].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 2].set_title("|PCC original - PCC boundary occluded|")
    axes[1, 2].axis("off")

    fig.suptitle(f"{case_id} | slice {z}", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close(fig)


# =========================
# Select cases
# =========================
quick_df = case_metrics_all.sample(
    n=min(QUICK_CASES, len(case_metrics_all)),
    random_state=SEED
).sort_values(["fold", "case_id"]).reset_index(drop=True)

print("Quick cases:")
display(quick_df[["case_id", "fold", "baseline_dice", "pcc_dice", "dice_gain", "target_voxels"]])


# =========================
# Main experiment
# =========================
start_time = time.time()
rng = np.random.default_rng(SEED)
fold_model_cache = {}

rows = []
figure_cache = []

for _, row in tqdm(quick_df.iterrows(), total=len(quick_df), desc="Layer3 quick v1"):
    case_id = row["case_id"]
    fold = int(row["fold"])

    if fold not in fold_model_cache:
        fold_model_cache[fold] = load_fold_models(fold)

    baseline_model, pcc_model, base_thr, pcc_thr = fold_model_cache[fold]

    X, Y, brain_mask, brain_mean_norm, meta = load_layer1_case(case_id)

    target_zyx = Y[:, 0] > 0.5
    core, boundary, peritumour, avoid = make_regions(target_zyx, brain_mask)

    if core.sum() <= 0 or boundary.sum() <= 0:
        print("Skipping empty tumour/boundary:", case_id)
        continue

    # Original prediction
    base_orig, pcc_orig = predict_baseline_and_pcc(baseline_model, pcc_model, X, BATCH_SIZE)

    base_orig_dice, base_orig_iou = dice_iou_prob(base_orig, Y, base_thr)
    pcc_orig_dice, pcc_orig_iou = dice_iou_prob(pcc_orig, Y, pcc_thr)

    # Reproduction check against CSV
    csv_base = float(row["baseline_dice"])
    csv_pcc = float(row["pcc_dice"])
    base_repro_error = abs(base_orig_dice - csv_base)
    pcc_repro_error = abs(pcc_orig_dice - csv_pcc)

    # Core occlusion
    X_core = occlude_volume(X, core, brain_mean_norm)
    base_core, pcc_core = predict_baseline_and_pcc(baseline_model, pcc_model, X_core, BATCH_SIZE)
    base_core_dice, _ = dice_iou_prob(base_core, Y, base_thr)
    pcc_core_dice, _ = dice_iou_prob(pcc_core, Y, pcc_thr)

    # Boundary occlusion
    X_boundary = occlude_volume(X, boundary, brain_mean_norm)
    base_boundary, pcc_boundary = predict_baseline_and_pcc(baseline_model, pcc_model, X_boundary, BATCH_SIZE)
    base_boundary_dice, _ = dice_iou_prob(base_boundary, Y, base_thr)
    pcc_boundary_dice, _ = dice_iou_prob(pcc_boundary, Y, pcc_thr)

    # Peritumour occlusion
    X_peri = occlude_volume(X, peritumour, brain_mean_norm)
    base_peri, pcc_peri = predict_baseline_and_pcc(baseline_model, pcc_model, X_peri, BATCH_SIZE)
    base_peri_dice, _ = dice_iou_prob(base_peri, Y, base_thr)
    pcc_peri_dice, _ = dice_iou_prob(pcc_peri, Y, pcc_thr)

    # Control occlusion: boundary-shaped controls
    base_control_drops = []
    pcc_control_drops = []
    control_masks = []

    for k in range(N_CONTROL_REGIONS):
        control = make_shifted_control(boundary, avoid, brain_mask, rng)
        control_masks.append(control)

        X_control = occlude_volume(X, control, brain_mean_norm)
        base_control, pcc_control = predict_baseline_and_pcc(baseline_model, pcc_model, X_control, BATCH_SIZE)

        base_control_dice, _ = dice_iou_prob(base_control, Y, base_thr)
        pcc_control_dice, _ = dice_iou_prob(pcc_control, Y, pcc_thr)

        base_control_drops.append(base_orig_dice - base_control_dice)
        pcc_control_drops.append(pcc_orig_dice - pcc_control_dice)

        del X_control, base_control, pcc_control
        gc.collect()

    base_control_drop_mean = float(np.mean(base_control_drops))
    pcc_control_drop_mean = float(np.mean(pcc_control_drops))

    # Drops
    base_core_drop = base_orig_dice - base_core_dice
    pcc_core_drop = pcc_orig_dice - pcc_core_dice

    base_boundary_drop = base_orig_dice - base_boundary_dice
    pcc_boundary_drop = pcc_orig_dice - pcc_boundary_dice

    base_peri_drop = base_orig_dice - base_peri_dice
    pcc_peri_drop = pcc_orig_dice - pcc_peri_dice

    # PRI
    base_core_PRI = base_core_drop - base_control_drop_mean
    pcc_core_PRI = pcc_core_drop - pcc_control_drop_mean

    base_boundary_PRI = base_boundary_drop - base_control_drop_mean
    pcc_boundary_PRI = pcc_boundary_drop - pcc_control_drop_mean

    base_peri_PRI = base_peri_drop - base_control_drop_mean
    pcc_peri_PRI = pcc_peri_drop - pcc_control_drop_mean

    eps = 1e-8

    rows.append({
        "case_id": case_id,
        "fold": fold,
        "patient_id": meta["patient_id"],
        "current_tp": meta["current_tp"],

        "target_voxels": int(target_zyx.sum()),
        "core_voxels": int(core.sum()),
        "boundary_voxels": int(boundary.sum()),
        "peritumour_voxels": int(peritumour.sum()),
        "brain_voxels": int(brain_mask.sum()),

        "brain_mean_norm": brain_mean_norm,

        "csv_baseline_dice": csv_base,
        "csv_pcc_dice": csv_pcc,
        "baseline_original_dice": base_orig_dice,
        "pcc_original_dice": pcc_orig_dice,
        "original_dice_gain": pcc_orig_dice - base_orig_dice,
        "baseline_repro_abs_error": base_repro_error,
        "pcc_repro_abs_error": pcc_repro_error,

        "baseline_original_iou": base_orig_iou,
        "pcc_original_iou": pcc_orig_iou,

        "baseline_core_drop": base_core_drop,
        "pcc_core_drop": pcc_core_drop,
        "pcc_minus_baseline_core_drop": pcc_core_drop - base_core_drop,

        "baseline_boundary_drop": base_boundary_drop,
        "pcc_boundary_drop": pcc_boundary_drop,
        "pcc_minus_baseline_boundary_drop": pcc_boundary_drop - base_boundary_drop,

        "baseline_peritumour_drop": base_peri_drop,
        "pcc_peritumour_drop": pcc_peri_drop,
        "pcc_minus_baseline_peritumour_drop": pcc_peri_drop - base_peri_drop,

        "baseline_control_drop_mean": base_control_drop_mean,
        "pcc_control_drop_mean": pcc_control_drop_mean,
        "pcc_minus_baseline_control_drop": pcc_control_drop_mean - base_control_drop_mean,

        "baseline_core_PRI": base_core_PRI,
        "pcc_core_PRI": pcc_core_PRI,
        "pcc_minus_baseline_core_PRI": pcc_core_PRI - base_core_PRI,

        "baseline_boundary_PRI": base_boundary_PRI,
        "pcc_boundary_PRI": pcc_boundary_PRI,
        "pcc_minus_baseline_boundary_PRI": pcc_boundary_PRI - base_boundary_PRI,

        "baseline_peritumour_PRI": base_peri_PRI,
        "pcc_peritumour_PRI": pcc_peri_PRI,
        "pcc_minus_baseline_peritumour_PRI": pcc_peri_PRI - base_peri_PRI,

        "baseline_core_relative_PRI": base_core_PRI / (base_orig_dice + eps),
        "pcc_core_relative_PRI": pcc_core_PRI / (pcc_orig_dice + eps),
        "pcc_minus_baseline_core_relative_PRI": (pcc_core_PRI / (pcc_orig_dice + eps)) - (base_core_PRI / (base_orig_dice + eps)),

        "baseline_boundary_relative_PRI": base_boundary_PRI / (base_orig_dice + eps),
        "pcc_boundary_relative_PRI": pcc_boundary_PRI / (pcc_orig_dice + eps),
        "pcc_minus_baseline_boundary_relative_PRI": (pcc_boundary_PRI / (pcc_orig_dice + eps)) - (base_boundary_PRI / (base_orig_dice + eps)),

        "baseline_peritumour_relative_PRI": base_peri_PRI / (base_orig_dice + eps),
        "pcc_peritumour_relative_PRI": pcc_peri_PRI / (pcc_orig_dice + eps),
        "pcc_minus_baseline_peritumour_relative_PRI": (pcc_peri_PRI / (pcc_orig_dice + eps)) - (base_peri_PRI / (base_orig_dice + eps)),
    })

    figure_cache.append({
        "case_id": case_id,
        "score": pcc_boundary_PRI - base_boundary_PRI,
        "X": X,
        "Y": Y,
        "boundary": boundary,
        "control": control_masks[0],
        "base_orig": base_orig,
        "pcc_orig": pcc_orig,
        "base_boundary": base_boundary,
        "pcc_boundary": pcc_boundary,
        "base_thr": base_thr,
        "pcc_thr": pcc_thr,
    })

    del X, Y, X_core, X_boundary, X_peri
    del base_orig, pcc_orig, base_core, pcc_core, base_boundary, pcc_boundary, base_peri, pcc_peri
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


case_df = pd.DataFrame(rows)
assert len(case_df) > 0, "No cases evaluated."


# =========================
# Summary
# =========================
summary_metrics = [
    "original_dice_gain",

    "baseline_repro_abs_error",
    "pcc_repro_abs_error",

    "pcc_minus_baseline_core_drop",
    "pcc_minus_baseline_boundary_drop",
    "pcc_minus_baseline_peritumour_drop",
    "pcc_minus_baseline_control_drop",

    "pcc_minus_baseline_core_PRI",
    "pcc_minus_baseline_boundary_PRI",
    "pcc_minus_baseline_peritumour_PRI",

    "pcc_minus_baseline_core_relative_PRI",
    "pcc_minus_baseline_boundary_relative_PRI",
    "pcc_minus_baseline_peritumour_relative_PRI",
]

stat_df = pd.DataFrame([summarize_metric(case_df, m) for m in summary_metrics])

overall_summary = {
    "experiment": "Layer3 quick v1 reproduction-passed pathology-reliance occlusion audit",
    "cases_evaluated": int(len(case_df)),
    "control_regions_per_case": int(N_CONTROL_REGIONS),
    "normalization": "p01_p99_brain",
    "pcc_mode": "residual_tanh3",
    "occlusion_replacement": "normalized_brain_mean",
    "runtime_minutes": float((time.time() - start_time) / 60.0),

    "mean_csv_baseline_dice": float(case_df["csv_baseline_dice"].mean()),
    "mean_csv_pcc_dice": float(case_df["csv_pcc_dice"].mean()),
    "mean_baseline_original_dice": float(case_df["baseline_original_dice"].mean()),
    "mean_pcc_original_dice": float(case_df["pcc_original_dice"].mean()),
    "mean_original_dice_gain": float(case_df["original_dice_gain"].mean()),
    "original_dice_win_rate": float((case_df["original_dice_gain"] > 0).mean()),

    "mean_baseline_repro_abs_error": float(case_df["baseline_repro_abs_error"].mean()),
    "mean_pcc_repro_abs_error": float(case_df["pcc_repro_abs_error"].mean()),

    "mean_baseline_core_drop": float(case_df["baseline_core_drop"].mean()),
    "mean_pcc_core_drop": float(case_df["pcc_core_drop"].mean()),
    "mean_pcc_minus_baseline_core_drop": float(case_df["pcc_minus_baseline_core_drop"].mean()),

    "mean_baseline_boundary_drop": float(case_df["baseline_boundary_drop"].mean()),
    "mean_pcc_boundary_drop": float(case_df["pcc_boundary_drop"].mean()),
    "mean_pcc_minus_baseline_boundary_drop": float(case_df["pcc_minus_baseline_boundary_drop"].mean()),

    "mean_baseline_control_drop": float(case_df["baseline_control_drop_mean"].mean()),
    "mean_pcc_control_drop": float(case_df["pcc_control_drop_mean"].mean()),
    "mean_pcc_minus_baseline_control_drop": float(case_df["pcc_minus_baseline_control_drop"].mean()),

    "mean_baseline_core_PRI": float(case_df["baseline_core_PRI"].mean()),
    "mean_pcc_core_PRI": float(case_df["pcc_core_PRI"].mean()),
    "mean_pcc_minus_baseline_core_PRI": float(case_df["pcc_minus_baseline_core_PRI"].mean()),
    "core_PRI_win_rate": float((case_df["pcc_minus_baseline_core_PRI"] > 0).mean()),

    "mean_baseline_boundary_PRI": float(case_df["baseline_boundary_PRI"].mean()),
    "mean_pcc_boundary_PRI": float(case_df["pcc_boundary_PRI"].mean()),
    "mean_pcc_minus_baseline_boundary_PRI": float(case_df["pcc_minus_baseline_boundary_PRI"].mean()),
    "boundary_PRI_win_rate": float((case_df["pcc_minus_baseline_boundary_PRI"] > 0).mean()),

    "mean_pcc_minus_baseline_boundary_relative_PRI": float(case_df["pcc_minus_baseline_boundary_relative_PRI"].mean()),
    "boundary_relative_PRI_win_rate": float((case_df["pcc_minus_baseline_boundary_relative_PRI"] > 0).mean()),
}

summary_df = pd.DataFrame([overall_summary])


# =========================
# Save figures
# =========================
figure_cache_sorted = sorted(figure_cache, key=lambda x: x["score"], reverse=True)
selected_figs = figure_cache_sorted[:2] + figure_cache_sorted[-1:]

for item in selected_figs:
    fig_path = FIG_DIR / f"{item['case_id']}_layer3_quick_v1_occlusion_audit.png"
    save_case_figure(
        case_id=item["case_id"],
        X=item["X"],
        Y=item["Y"],
        boundary=item["boundary"],
        control=item["control"],
        base_orig=item["base_orig"],
        pcc_orig=item["pcc_orig"],
        base_occ_boundary=item["base_boundary"],
        pcc_occ_boundary=item["pcc_boundary"],
        base_thr=item["base_thr"],
        pcc_thr=item["pcc_thr"],
        out_path=fig_path,
    )

print("Saved figures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(p)


# =========================
# Save outputs
# =========================
case_path = OUT_DIR / "Layer3_quick_v1_case_metrics.csv"
summary_path = OUT_DIR / "Layer3_quick_v1_overall_summary.csv"
stats_path = OUT_DIR / "Layer3_quick_v1_paired_stats_bootstrap.csv"
run_info_path = OUT_DIR / "Layer3_quick_v1_run_info.json"

case_df.to_csv(case_path, index=False)
summary_df.to_csv(summary_path, index=False)
stat_df.to_csv(stats_path, index=False)

with open(run_info_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)

print("\n" + "=" * 100)
print("Layer3 quick v1 finished.")
print("Saved to:", OUT_DIR)

print("\nOverall summary:")
display(summary_df)

print("\nPaired stats / bootstrap CI:")
display(stat_df)

print("\nCase metrics:")
display(case_df)

print("\nSaved files:")
print(case_path)
print(summary_path)
print(stats_path)
print(run_info_path)
print("Figures:", FIG_DIR)


# =========================
# Interpretation
# =========================
boundary_gain = overall_summary["mean_pcc_minus_baseline_boundary_PRI"]
boundary_win = overall_summary["boundary_PRI_win_rate"]
control_extra = overall_summary["mean_pcc_minus_baseline_control_drop"]
rel_boundary_gain = overall_summary["mean_pcc_minus_baseline_boundary_relative_PRI"]
repro_base = overall_summary["mean_baseline_repro_abs_error"]
repro_pcc = overall_summary["mean_pcc_repro_abs_error"]

print("\n" + "=" * 100)
print("Quick interpretation:")

print("mean_baseline_repro_abs_error:", repro_base)
print("mean_pcc_repro_abs_error:", repro_pcc)
print("mean_pcc_minus_baseline_boundary_PRI:", boundary_gain)
print("boundary_PRI_win_rate:", boundary_win)
print("mean_pcc_minus_baseline_control_drop:", control_extra)
print("mean_pcc_minus_baseline_boundary_relative_PRI:", rel_boundary_gain)

if repro_base > 0.01 or repro_pcc > 0.01:
    print("\nREPRODUCTION WARNING:")
    print("Original reproduction is not close enough. Do not interpret occlusion.")
elif boundary_gain > 0 and boundary_win >= 0.65 and rel_boundary_gain > 0:
    print("\nPASS / strong quick signal:")
    print("PCC shows stronger boundary pathology-reliance than baseline.")
elif boundary_gain > 0 and boundary_win >= 0.55:
    print("\nPARTIAL PASS / moderate quick signal:")
    print("PCC shows positive pathology-reliance trend. Formal 40-case audit is recommended.")
else:
    print("\nNOT PASSED in quick version:")
    print("This quick sample does not show clear PCC pathology-reliance advantage.")

In [ ]:
# ============================================================
# Layer3 FORMAL v1: 40-case Pathology-Reliance Occlusion Audit
#
# Based on reproduction-passed quick v1:
#   normalization = p01_p99_brain
#   PCC mode      = residual_tanh3
#
# Uses:
#   Layer1 FORMAL_v1 5-fold checkpoints
#   all 40 held-out cases from layer1_FORMAL_v1_case_metrics.csv
#
# Main scientific question:
#   Does PCC show stronger functional dependence on true
#   pathological structures than the baseline?
#
# Main metric:
#   PRI = pathology_occlusion_drop - matched_control_occlusion_drop
#
# Main success signal:
#   PCC boundary PRI > Baseline boundary PRI
# ============================================================

from pathlib import Path
import re, json, time, gc, random, zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import nibabel as nib
import scipy.ndimage as ndi

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None


# =========================
# Config
# =========================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

N_CONTROL_REGIONS = 10
BATCH_SIZE = 32

BOUNDARY_RADIUS = 5
AVOID_RADIUS = 8
PERITUMOUR_RADIUS = 10

PCC_MAX_DELTA_LOGIT = 3.0

OUT_DIR = Path("/kaggle/working/Layer3_FORMAL_v1_40case_pathology_reliance_occlusion")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUT_DIR)


# =========================
# Fixed Kaggle paths
# =========================
LAYER1_DIR = Path(
    "/kaggle/input/datasets/jeechangxin/layer1-formal-v1-5fold-results-backup/"
    "pcc_independent_baseline/stage2_pcc_guided_student_learning/"
    "layer1_current_segmentation_FORMAL_v1_5fold"
)

CKPT_DIR = LAYER1_DIR / "checkpoints"
CASE_METRICS_PATH = LAYER1_DIR / "layer1_FORMAL_v1_case_metrics.csv"

DATA_ROOT = Path(
    "/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/"
    "PKG - MU-Glioma-Post/MU-Glioma-Post"
)

assert LAYER1_DIR.exists(), LAYER1_DIR
assert CKPT_DIR.exists(), CKPT_DIR
assert CASE_METRICS_PATH.exists(), CASE_METRICS_PATH
assert DATA_ROOT.exists(), DATA_ROOT

case_metrics_all = pd.read_csv(CASE_METRICS_PATH)
case_metrics_all = case_metrics_all.sort_values(["fold", "case_id"]).reset_index(drop=True)

print("Layer1 dir:", LAYER1_DIR)
print("Checkpoint dir:", CKPT_DIR)
print("Dataset root:", DATA_ROOT)
print("case_metrics:", case_metrics_all.shape)
display(case_metrics_all.head())


# =========================
# Model architecture
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


def get_threshold_from_ckpt(ckpt, default=0.5):
    for k in ["threshold", "best_threshold", "selected_threshold", "thr"]:
        if k in ckpt:
            return float(ckpt[k])
    return float(default)


def load_fold_models(fold):
    base_path = CKPT_DIR / f"baseline_seg_fold_{fold}_FORMAL_v1.pt"
    pcc_path = CKPT_DIR / f"pcc_seg_corrector_fold_{fold}_FORMAL_v1.pt"

    assert base_path.exists(), base_path
    assert pcc_path.exists(), pcc_path

    base_ckpt = torch.load(base_path, map_location=DEVICE)
    pcc_ckpt = torch.load(pcc_path, map_location=DEVICE)

    baseline = SmallUNet2D(in_ch=1, out_ch=1, base=16).to(DEVICE)
    pcc = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)

    baseline.load_state_dict(base_ckpt["model_state_dict"])
    pcc.load_state_dict(pcc_ckpt["model_state_dict"])

    baseline.eval()
    pcc.eval()

    base_thr = get_threshold_from_ckpt(base_ckpt, 0.5)
    pcc_thr = get_threshold_from_ckpt(pcc_ckpt, 0.5)

    print(f"Loaded fold {fold}: base_thr={base_thr}, pcc_thr={pcc_thr}")

    return baseline, pcc, base_thr, pcc_thr


# =========================
# File loading helpers
# =========================
def parse_case_id(case_id):
    case_id = str(case_id)
    m_patient = re.search(r"(PatientID_\d+)", case_id)
    m_t = re.search(r"_T(\d+)", case_id)

    if m_patient is None or m_t is None:
        raise ValueError(f"Cannot parse case_id: {case_id}")

    return m_patient.group(1), int(m_t.group(1))


def find_patient_dir(patient_id):
    direct = DATA_ROOT / patient_id
    if direct.exists():
        return direct

    hits = list(DATA_ROOT.glob(f"**/{patient_id}"))
    hits = [h for h in hits if h.is_dir()]

    if not hits:
        raise FileNotFoundError(f"Patient folder not found: {patient_id}")

    return hits[0]


def find_timepoint_file(patient_dir, tp, kind):
    patient_dir = Path(patient_dir)

    if kind == "brain_t1c":
        patterns = ["*brain_t1c.nii", "*brain_t1c.nii.gz"]
    elif kind == "tumorMask":
        patterns = ["*tumorMask.nii", "*tumorMask.nii.gz"]
    else:
        patterns = [f"*{kind}.nii", f"*{kind}.nii.gz"]

    candidates = []
    for pat in patterns:
        candidates.extend(list(patient_dir.rglob(pat)))

    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        raise FileNotFoundError(f"No {kind} found in {patient_dir}")

    tp_tokens = [
        f"Timepoint_{tp}",
        f"Timepoint-{tp}",
        f"Timepoint {tp}",
        f"_T{tp}_",
        f"/T{tp}/",
        f"TP{tp}",
    ]

    filtered = []
    for p in candidates:
        s = str(p)
        if any(tok in s for tok in tp_tokens) or f"_Timepoint_{tp}_" in p.name:
            filtered.append(p)

    if filtered:
        return filtered[0]

    raise FileNotFoundError(
        f"Found {len(candidates)} {kind} candidates but none matched Timepoint {tp}. "
        f"First candidates: {[str(c) for c in candidates[:5]]}"
    )


def to_z_hw(arr):
    arr = np.asarray(arr)

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {arr.shape}")

    # Common NIfTI shape: H,W,Z -> Z,H,W
    if arr.shape[0] >= 128 and arr.shape[1] >= 128 and arr.shape[2] < 200:
        arr = np.moveaxis(arr, -1, 0)

    return arr.astype(np.float32)


def normalize_p01_p99_brain(vol, brain):
    vol = np.nan_to_num(vol.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    brain = brain.astype(bool)

    vals = vol[brain]
    if vals.size < 100:
        vals = vol.reshape(-1)

    lo, hi = np.percentile(vals, [1, 99])
    x = (vol - lo) / (hi - lo + 1e-6)
    x = np.clip(x, 0.0, 1.0).astype(np.float32)

    brain_mean_norm = float(x[brain].mean()) if brain.sum() > 0 else float(x.mean())

    return x, brain_mean_norm


def load_layer1_case(case_id):
    patient_id, current_tp = parse_case_id(case_id)
    pdir = find_patient_dir(patient_id)

    t1c_path = find_timepoint_file(pdir, current_tp, "brain_t1c")
    mask_path = find_timepoint_file(pdir, current_tp, "tumorMask")

    img_raw = to_z_hw(nib.load(str(t1c_path)).get_fdata()).astype(np.float32)
    mask_raw = to_z_hw(nib.load(str(mask_path)).get_fdata()).astype(np.float32)

    mask = mask_raw > 0.5
    brain = np.abs(img_raw) > 1e-6
    if brain.sum() < 100:
        brain = np.ones_like(img_raw, dtype=bool)

    img_norm, brain_mean_norm = normalize_p01_p99_brain(img_raw, brain)

    X = img_norm[:, None, :, :].astype(np.float32)
    Y = mask[:, None, :, :].astype(np.float32)

    return X, Y, brain.astype(bool), brain_mean_norm, {
        "patient_id": patient_id,
        "current_tp": current_tp,
        "t1c_path": str(t1c_path),
        "mask_path": str(mask_path),
    }


# =========================
# Prediction and metrics
# =========================
@torch.no_grad()
def predict_baseline_and_pcc(baseline_model, pcc_model, X, batch_size=32):
    baseline_model.eval()
    pcc_model.eval()

    base_probs = []
    pcc_probs = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X[start:start + batch_size]).float().to(DEVICE)

        base_logits = baseline_model(xb)
        base_prob = torch.sigmoid(base_logits)

        pcc_input = torch.cat([xb, base_prob], dim=1)
        residual_raw = pcc_model(pcc_input)

        corrected_logits = base_logits + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)
        pcc_prob = torch.sigmoid(corrected_logits)

        base_probs.append(base_prob.detach().cpu().numpy().astype(np.float32))
        pcc_probs.append(pcc_prob.detach().cpu().numpy().astype(np.float32))

    return np.concatenate(base_probs, axis=0), np.concatenate(pcc_probs, axis=0)


def dice_iou_prob(prob, target, threshold=0.5, eps=1e-8):
    pred = prob >= threshold
    target = target.astype(bool)

    inter = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()
    pred_sum = pred.sum()
    target_sum = target.sum()

    dice = (2 * inter + eps) / (pred_sum + target_sum + eps)
    iou = (inter + eps) / (union + eps)

    return float(dice), float(iou)


def occlude_volume(X, occ_mask_zyx, replacement_value):
    X_occ = X.copy()
    X_occ[:, 0, :, :][occ_mask_zyx] = float(replacement_value)
    return X_occ


# =========================
# Region helpers
# =========================
def make_regions(target_zyx, brain_zyx):
    target = target_zyx.astype(bool)
    brain = brain_zyx.astype(bool)

    core = np.logical_and(target, brain)

    dil = ndi.binary_dilation(core, iterations=BOUNDARY_RADIUS)
    ero = ndi.binary_erosion(core, iterations=BOUNDARY_RADIUS)
    boundary = np.logical_and(dil, np.logical_not(ero))
    boundary = np.logical_and(boundary, brain)

    peritumour = ndi.binary_dilation(core, iterations=PERITUMOUR_RADIUS)
    peritumour = np.logical_and(peritumour, np.logical_not(core))
    peritumour = np.logical_and(peritumour, brain)

    avoid = ndi.binary_dilation(core, iterations=AVOID_RADIUS)
    avoid = np.logical_or(avoid, np.logical_not(brain))

    return core, boundary, peritumour, avoid


def make_shifted_control(region_mask, avoid_mask, brain_mask, rng, tries=100):
    region = region_mask.astype(bool)
    avoid = avoid_mask.astype(bool)
    brain = brain_mask.astype(bool)

    n = int(region.sum())
    if n <= 0:
        return np.zeros_like(region, dtype=bool)

    zdim, h, w = region.shape
    best = None
    best_count = -1

    for _ in range(tries):
        dz = rng.integers(-max(2, zdim // 3), max(3, zdim // 3))
        dy = rng.integers(-max(8, h // 3), max(9, h // 3))
        dx = rng.integers(-max(8, w // 3), max(9, w // 3))

        shifted = np.roll(region, shift=(dz, dy, dx), axis=(0, 1, 2))
        shifted = np.logical_and(shifted, brain)
        shifted = np.logical_and(shifted, np.logical_not(avoid))

        c = int(shifted.sum())

        if c > best_count:
            best = shifted
            best_count = c

        if c >= int(0.80 * n):
            return shifted

    candidates = np.logical_and(brain, np.logical_not(avoid))
    idx = np.argwhere(candidates)

    if len(idx) == 0:
        return np.zeros_like(region, dtype=bool)

    take = min(n, len(idx))
    chosen = idx[rng.choice(len(idx), size=take, replace=False)]

    control = np.zeros_like(region, dtype=bool)
    control[chosen[:, 0], chosen[:, 1], chosen[:, 2]] = True

    return control


# =========================
# Stats helpers
# =========================
def bootstrap_ci(vals, n_boot=5000, seed=42):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = []

    for _ in range(n_boot):
        s = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.mean(s))

    return float(np.mean(vals)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def summarize_metric(case_df, metric):
    vals = case_df[metric].values.astype(float)
    mean, low, high = bootstrap_ci(vals)

    if wilcoxon is not None:
        try:
            w_greater = wilcoxon(vals, alternative="greater")
            p_greater = float(w_greater.pvalue)
            stat_greater = float(w_greater.statistic)

            w_two = wilcoxon(vals, alternative="two-sided")
            p_two = float(w_two.pvalue)
            stat_two = float(w_two.statistic)
        except Exception:
            p_greater = np.nan
            stat_greater = np.nan
            p_two = np.nan
            stat_two = np.nan
    else:
        p_greater = np.nan
        stat_greater = np.nan
        p_two = np.nan
        stat_two = np.nan

    return {
        "metric": metric,
        "mean": mean,
        "median": float(np.nanmedian(vals)),
        "std": float(np.nanstd(vals)),
        "ci95_low": low,
        "ci95_high": high,
        "positive_cases": int(np.sum(vals > 0)),
        "negative_cases": int(np.sum(vals < 0)),
        "zero_cases": int(np.sum(vals == 0)),
        "positive_rate": float(np.mean(vals > 0)),
        "wilcoxon_greater_statistic": stat_greater,
        "wilcoxon_greater_p_value": p_greater,
        "wilcoxon_two_sided_statistic": stat_two,
        "wilcoxon_two_sided_p_value": p_two,
    }


def summarize_by_fold(case_df, metric):
    rows = []
    for fold, g in case_df.groupby("fold"):
        vals = g[metric].values.astype(float)
        rows.append({
            "fold": int(fold),
            "metric": metric,
            "n_cases": int(len(g)),
            "mean": float(np.mean(vals)),
            "median": float(np.median(vals)),
            "positive_cases": int(np.sum(vals > 0)),
            "positive_rate": float(np.mean(vals > 0)),
        })
    return rows


# =========================
# Figure helper
# =========================
def save_case_figure(
    case_id, X, Y, boundary, control,
    base_orig, pcc_orig, base_occ_boundary, pcc_occ_boundary,
    base_thr, pcc_thr, out_path
):
    target = Y[:, 0] > 0.5

    score = boundary.reshape(boundary.shape[0], -1).sum(axis=1)
    if score.max() <= 0:
        score = target.reshape(target.shape[0], -1).sum(axis=1)

    z = int(np.argmax(score))

    img = X[z, 0]
    true_z = target[z]
    boundary_z = boundary[z]
    control_z = control[z]

    base_bin = base_orig[z, 0] >= base_thr
    pcc_bin = pcc_orig[z, 0] >= pcc_thr
    base_occ_bin = base_occ_boundary[z, 0] >= base_thr
    pcc_occ_bin = pcc_occ_boundary[z, 0] >= pcc_thr

    pcc_change = np.abs(pcc_orig[z, 0] - pcc_occ_boundary[z, 0])
    base_change = np.abs(base_orig[z, 0] - base_occ_boundary[z, 0])

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    axes[0, 0].imshow(img, cmap="gray")
    axes[0, 0].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 0].set_title("Original MRI + true mask")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(img, cmap="gray")
    axes[0, 1].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 1].contour(base_bin, levels=[0.5], colors="blue", linewidths=1)
    axes[0, 1].set_title("Baseline original")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(img, cmap="gray")
    axes[0, 2].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 2].contour(pcc_bin, levels=[0.5], colors="red", linewidths=1)
    axes[0, 2].set_title("PCC original")
    axes[0, 2].axis("off")

    axes[0, 3].imshow(img, cmap="gray")
    axes[0, 3].imshow(boundary_z, cmap="Reds", alpha=0.35)
    axes[0, 3].imshow(control_z, cmap="Blues", alpha=0.30)
    axes[0, 3].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 3].set_title("Occlusion masks: red=boundary, blue=control")
    axes[0, 3].axis("off")

    axes[1, 0].imshow(img, cmap="gray")
    axes[1, 0].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 0].contour(base_occ_bin, levels=[0.5], colors="blue", linewidths=1)
    axes[1, 0].set_title("Baseline after boundary occlusion")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(img, cmap="gray")
    axes[1, 1].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 1].contour(pcc_occ_bin, levels=[0.5], colors="red", linewidths=1)
    axes[1, 1].set_title("PCC after boundary occlusion")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(img, cmap="gray")
    axes[1, 2].imshow(
        base_change,
        cmap="hot",
        alpha=np.clip(base_change / (base_change.max() + 1e-8), 0, 0.85)
    )
    axes[1, 2].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 2].set_title("|Baseline original - occluded|")
    axes[1, 2].axis("off")

    axes[1, 3].imshow(img, cmap="gray")
    axes[1, 3].imshow(
        pcc_change,
        cmap="hot",
        alpha=np.clip(pcc_change / (pcc_change.max() + 1e-8), 0, 0.85)
    )
    axes[1, 3].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 3].set_title("|PCC original - occluded|")
    axes[1, 3].axis("off")

    fig.suptitle(f"{case_id} | slice {z}", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close(fig)


# =========================
# Main formal experiment
# =========================
print("\nFormal cases:")
display(case_metrics_all[["case_id", "fold", "baseline_dice", "pcc_dice", "dice_gain", "target_voxels"]])

start_time = time.time()
rng = np.random.default_rng(SEED)
fold_model_cache = {}

rows = []
failed_cases = []
figure_cache = []

for _, row in tqdm(case_metrics_all.iterrows(), total=len(case_metrics_all), desc="Layer3 FORMAL 40-case"):
    case_id = row["case_id"]
    fold = int(row["fold"])

    if fold not in fold_model_cache:
        fold_model_cache[fold] = load_fold_models(fold)

    baseline_model, pcc_model, base_thr, pcc_thr = fold_model_cache[fold]

    try:
        X, Y, brain_mask, brain_mean_norm, meta = load_layer1_case(case_id)

        target_zyx = Y[:, 0] > 0.5
        core, boundary, peritumour, avoid = make_regions(target_zyx, brain_mask)

        if core.sum() <= 0 or boundary.sum() <= 0:
            raise ValueError("Empty tumour core or boundary")

        # Original prediction
        base_orig, pcc_orig = predict_baseline_and_pcc(baseline_model, pcc_model, X, BATCH_SIZE)

        base_orig_dice, base_orig_iou = dice_iou_prob(base_orig, Y, base_thr)
        pcc_orig_dice, pcc_orig_iou = dice_iou_prob(pcc_orig, Y, pcc_thr)

        csv_base = float(row["baseline_dice"])
        csv_pcc = float(row["pcc_dice"])

        base_repro_error = abs(base_orig_dice - csv_base)
        pcc_repro_error = abs(pcc_orig_dice - csv_pcc)

        # Core occlusion
        X_core = occlude_volume(X, core, brain_mean_norm)
        base_core, pcc_core = predict_baseline_and_pcc(baseline_model, pcc_model, X_core, BATCH_SIZE)
        base_core_dice, _ = dice_iou_prob(base_core, Y, base_thr)
        pcc_core_dice, _ = dice_iou_prob(pcc_core, Y, pcc_thr)

        # Boundary occlusion
        X_boundary = occlude_volume(X, boundary, brain_mean_norm)
        base_boundary, pcc_boundary = predict_baseline_and_pcc(baseline_model, pcc_model, X_boundary, BATCH_SIZE)
        base_boundary_dice, _ = dice_iou_prob(base_boundary, Y, base_thr)
        pcc_boundary_dice, _ = dice_iou_prob(pcc_boundary, Y, pcc_thr)

        # Peritumour occlusion
        X_peri = occlude_volume(X, peritumour, brain_mean_norm)
        base_peri, pcc_peri = predict_baseline_and_pcc(baseline_model, pcc_model, X_peri, BATCH_SIZE)
        base_peri_dice, _ = dice_iou_prob(base_peri, Y, base_thr)
        pcc_peri_dice, _ = dice_iou_prob(pcc_peri, Y, pcc_thr)

        # Matched control occlusions
        base_control_drops = []
        pcc_control_drops = []
        control_voxels = []
        first_control = None

        for k in range(N_CONTROL_REGIONS):
            control = make_shifted_control(boundary, avoid, brain_mask, rng)
            if first_control is None:
                first_control = control.copy()

            X_control = occlude_volume(X, control, brain_mean_norm)
            base_control, pcc_control = predict_baseline_and_pcc(baseline_model, pcc_model, X_control, BATCH_SIZE)

            base_control_dice, _ = dice_iou_prob(base_control, Y, base_thr)
            pcc_control_dice, _ = dice_iou_prob(pcc_control, Y, pcc_thr)

            base_control_drops.append(base_orig_dice - base_control_dice)
            pcc_control_drops.append(pcc_orig_dice - pcc_control_dice)
            control_voxels.append(int(control.sum()))

            del X_control, base_control, pcc_control
            gc.collect()

        base_control_drop_mean = float(np.mean(base_control_drops))
        pcc_control_drop_mean = float(np.mean(pcc_control_drops))
        base_control_drop_std = float(np.std(base_control_drops))
        pcc_control_drop_std = float(np.std(pcc_control_drops))

        # Drops
        base_core_drop = base_orig_dice - base_core_dice
        pcc_core_drop = pcc_orig_dice - pcc_core_dice

        base_boundary_drop = base_orig_dice - base_boundary_dice
        pcc_boundary_drop = pcc_orig_dice - pcc_boundary_dice

        base_peri_drop = base_orig_dice - base_peri_dice
        pcc_peri_drop = pcc_orig_dice - pcc_peri_dice

        # PRI
        base_core_PRI = base_core_drop - base_control_drop_mean
        pcc_core_PRI = pcc_core_drop - pcc_control_drop_mean

        base_boundary_PRI = base_boundary_drop - base_control_drop_mean
        pcc_boundary_PRI = pcc_boundary_drop - pcc_control_drop_mean

        base_peri_PRI = base_peri_drop - base_control_drop_mean
        pcc_peri_PRI = pcc_peri_drop - pcc_control_drop_mean

        eps = 1e-8

        case_out = {
            "case_id": case_id,
            "fold": fold,
            "patient_id": meta["patient_id"],
            "current_tp": meta["current_tp"],

            "target_voxels": int(target_zyx.sum()),
            "core_voxels": int(core.sum()),
            "boundary_voxels": int(boundary.sum()),
            "peritumour_voxels": int(peritumour.sum()),
            "mean_control_voxels": float(np.mean(control_voxels)),
            "brain_voxels": int(brain_mask.sum()),

            "brain_mean_norm": brain_mean_norm,

            "csv_baseline_dice": csv_base,
            "csv_pcc_dice": csv_pcc,
            "csv_dice_gain": csv_pcc - csv_base,

            "baseline_original_dice": base_orig_dice,
            "pcc_original_dice": pcc_orig_dice,
            "original_dice_gain": pcc_orig_dice - base_orig_dice,

            "baseline_repro_abs_error": base_repro_error,
            "pcc_repro_abs_error": pcc_repro_error,

            "baseline_original_iou": base_orig_iou,
            "pcc_original_iou": pcc_orig_iou,

            "baseline_core_drop": base_core_drop,
            "pcc_core_drop": pcc_core_drop,
            "pcc_minus_baseline_core_drop": pcc_core_drop - base_core_drop,

            "baseline_boundary_drop": base_boundary_drop,
            "pcc_boundary_drop": pcc_boundary_drop,
            "pcc_minus_baseline_boundary_drop": pcc_boundary_drop - base_boundary_drop,

            "baseline_peritumour_drop": base_peri_drop,
            "pcc_peritumour_drop": pcc_peri_drop,
            "pcc_minus_baseline_peritumour_drop": pcc_peri_drop - base_peri_drop,

            "baseline_control_drop_mean": base_control_drop_mean,
            "pcc_control_drop_mean": pcc_control_drop_mean,
            "baseline_control_drop_std": base_control_drop_std,
            "pcc_control_drop_std": pcc_control_drop_std,
            "pcc_minus_baseline_control_drop": pcc_control_drop_mean - base_control_drop_mean,

            "baseline_core_PRI": base_core_PRI,
            "pcc_core_PRI": pcc_core_PRI,
            "pcc_minus_baseline_core_PRI": pcc_core_PRI - base_core_PRI,

            "baseline_boundary_PRI": base_boundary_PRI,
            "pcc_boundary_PRI": pcc_boundary_PRI,
            "pcc_minus_baseline_boundary_PRI": pcc_boundary_PRI - base_boundary_PRI,

            "baseline_peritumour_PRI": base_peri_PRI,
            "pcc_peritumour_PRI": pcc_peri_PRI,
            "pcc_minus_baseline_peritumour_PRI": pcc_peri_PRI - base_peri_PRI,

            "baseline_core_relative_PRI": base_core_PRI / (base_orig_dice + eps),
            "pcc_core_relative_PRI": pcc_core_PRI / (pcc_orig_dice + eps),
            "pcc_minus_baseline_core_relative_PRI": (pcc_core_PRI / (pcc_orig_dice + eps)) - (base_core_PRI / (base_orig_dice + eps)),

            "baseline_boundary_relative_PRI": base_boundary_PRI / (base_orig_dice + eps),
            "pcc_boundary_relative_PRI": pcc_boundary_PRI / (pcc_orig_dice + eps),
            "pcc_minus_baseline_boundary_relative_PRI": (pcc_boundary_PRI / (pcc_orig_dice + eps)) - (base_boundary_PRI / (base_orig_dice + eps)),

            "baseline_peritumour_relative_PRI": base_peri_PRI / (base_orig_dice + eps),
            "pcc_peritumour_relative_PRI": pcc_peri_PRI / (pcc_orig_dice + eps),
            "pcc_minus_baseline_peritumour_relative_PRI": (pcc_peri_PRI / (pcc_orig_dice + eps)) - (base_peri_PRI / (base_orig_dice + eps)),
        }

        rows.append(case_out)

        figure_cache.append({
            "case_id": case_id,
            "score": case_out["pcc_minus_baseline_boundary_PRI"],
            "X": X,
            "Y": Y,
            "boundary": boundary,
            "control": first_control,
            "base_orig": base_orig,
            "pcc_orig": pcc_orig,
            "base_boundary": base_boundary,
            "pcc_boundary": pcc_boundary,
            "base_thr": base_thr,
            "pcc_thr": pcc_thr,
        })

        del X, Y, X_core, X_boundary, X_peri
        del base_orig, pcc_orig, base_core, pcc_core, base_boundary, pcc_boundary, base_peri, pcc_peri
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    except Exception as e:
        failed_cases.append({
            "case_id": case_id,
            "fold": fold,
            "reason": repr(e)
        })
        print("Failed case:", case_id, "reason:", repr(e))


case_df = pd.DataFrame(rows)
failed_df = pd.DataFrame(failed_cases)

assert len(case_df) > 0, "No cases evaluated."

print("\nEvaluated cases:", len(case_df))
print("Failed cases:", len(failed_df))
if len(failed_df) > 0:
    display(failed_df)


# =========================
# Summary and stats
# =========================
summary_metrics = [
    "original_dice_gain",
    "baseline_repro_abs_error",
    "pcc_repro_abs_error",

    "pcc_minus_baseline_core_drop",
    "pcc_minus_baseline_boundary_drop",
    "pcc_minus_baseline_peritumour_drop",
    "pcc_minus_baseline_control_drop",

    "pcc_minus_baseline_core_PRI",
    "pcc_minus_baseline_boundary_PRI",
    "pcc_minus_baseline_peritumour_PRI",

    "pcc_minus_baseline_core_relative_PRI",
    "pcc_minus_baseline_boundary_relative_PRI",
    "pcc_minus_baseline_peritumour_relative_PRI",
]

stat_df = pd.DataFrame([summarize_metric(case_df, m) for m in summary_metrics])

fold_rows = []
for metric in summary_metrics:
    fold_rows.extend(summarize_by_fold(case_df, metric))
fold_summary_df = pd.DataFrame(fold_rows)

overall_summary = {
    "experiment": "Layer3 FORMAL v1 40-case pathology-reliance occlusion audit",
    "cases_total_in_csv": int(len(case_metrics_all)),
    "cases_evaluated": int(len(case_df)),
    "cases_failed": int(len(failed_df)),
    "control_regions_per_case": int(N_CONTROL_REGIONS),
    "normalization": "p01_p99_brain",
    "pcc_mode": "residual_tanh3",
    "occlusion_replacement": "normalized_brain_mean",
    "boundary_radius": int(BOUNDARY_RADIUS),
    "peritumour_radius": int(PERITUMOUR_RADIUS),
    "avoid_radius": int(AVOID_RADIUS),
    "runtime_minutes": float((time.time() - start_time) / 60.0),

    "mean_csv_baseline_dice": float(case_df["csv_baseline_dice"].mean()),
    "mean_csv_pcc_dice": float(case_df["csv_pcc_dice"].mean()),
    "mean_csv_dice_gain": float(case_df["csv_dice_gain"].mean()),

    "mean_baseline_original_dice": float(case_df["baseline_original_dice"].mean()),
    "mean_pcc_original_dice": float(case_df["pcc_original_dice"].mean()),
    "mean_original_dice_gain": float(case_df["original_dice_gain"].mean()),
    "original_dice_win_rate": float((case_df["original_dice_gain"] > 0).mean()),

    "mean_baseline_repro_abs_error": float(case_df["baseline_repro_abs_error"].mean()),
    "mean_pcc_repro_abs_error": float(case_df["pcc_repro_abs_error"].mean()),
    "max_baseline_repro_abs_error": float(case_df["baseline_repro_abs_error"].max()),
    "max_pcc_repro_abs_error": float(case_df["pcc_repro_abs_error"].max()),

    "mean_baseline_core_drop": float(case_df["baseline_core_drop"].mean()),
    "mean_pcc_core_drop": float(case_df["pcc_core_drop"].mean()),
    "mean_pcc_minus_baseline_core_drop": float(case_df["pcc_minus_baseline_core_drop"].mean()),

    "mean_baseline_boundary_drop": float(case_df["baseline_boundary_drop"].mean()),
    "mean_pcc_boundary_drop": float(case_df["pcc_boundary_drop"].mean()),
    "mean_pcc_minus_baseline_boundary_drop": float(case_df["pcc_minus_baseline_boundary_drop"].mean()),

    "mean_baseline_control_drop": float(case_df["baseline_control_drop_mean"].mean()),
    "mean_pcc_control_drop": float(case_df["pcc_control_drop_mean"].mean()),
    "mean_pcc_minus_baseline_control_drop": float(case_df["pcc_minus_baseline_control_drop"].mean()),

    "mean_baseline_core_PRI": float(case_df["baseline_core_PRI"].mean()),
    "mean_pcc_core_PRI": float(case_df["pcc_core_PRI"].mean()),
    "mean_pcc_minus_baseline_core_PRI": float(case_df["pcc_minus_baseline_core_PRI"].mean()),
    "core_PRI_win_rate": float((case_df["pcc_minus_baseline_core_PRI"] > 0).mean()),

    "mean_baseline_boundary_PRI": float(case_df["baseline_boundary_PRI"].mean()),
    "mean_pcc_boundary_PRI": float(case_df["pcc_boundary_PRI"].mean()),
    "mean_pcc_minus_baseline_boundary_PRI": float(case_df["pcc_minus_baseline_boundary_PRI"].mean()),
    "boundary_PRI_win_rate": float((case_df["pcc_minus_baseline_boundary_PRI"] > 0).mean()),

    "mean_baseline_peritumour_PRI": float(case_df["baseline_peritumour_PRI"].mean()),
    "mean_pcc_peritumour_PRI": float(case_df["pcc_peritumour_PRI"].mean()),
    "mean_pcc_minus_baseline_peritumour_PRI": float(case_df["pcc_minus_baseline_peritumour_PRI"].mean()),
    "peritumour_PRI_win_rate": float((case_df["pcc_minus_baseline_peritumour_PRI"] > 0).mean()),

    "mean_pcc_minus_baseline_boundary_relative_PRI": float(case_df["pcc_minus_baseline_boundary_relative_PRI"].mean()),
    "boundary_relative_PRI_win_rate": float((case_df["pcc_minus_baseline_boundary_relative_PRI"] > 0).mean()),
}

summary_df = pd.DataFrame([overall_summary])


# =========================
# Save representative figures
# =========================
# Save top 3, middle 3, worst 3 by boundary PRI gain
figure_cache_sorted = sorted(figure_cache, key=lambda x: x["score"], reverse=True)

selected_figs = []
selected_figs.extend(figure_cache_sorted[:3])

mid_start = max(0, len(figure_cache_sorted) // 2 - 1)
selected_figs.extend(figure_cache_sorted[mid_start:mid_start + 3])

selected_figs.extend(figure_cache_sorted[-3:])

# Deduplicate
seen = set()
selected_unique = []
for item in selected_figs:
    if item["case_id"] not in seen:
        selected_unique.append(item)
        seen.add(item["case_id"])

for item in selected_unique:
    safe_case = item["case_id"]
    fig_path = FIG_DIR / f"{safe_case}_layer3_FORMAL_occlusion_audit.png"

    save_case_figure(
        case_id=item["case_id"],
        X=item["X"],
        Y=item["Y"],
        boundary=item["boundary"],
        control=item["control"],
        base_orig=item["base_orig"],
        pcc_orig=item["pcc_orig"],
        base_occ_boundary=item["base_boundary"],
        pcc_occ_boundary=item["pcc_boundary"],
        base_thr=item["base_thr"],
        pcc_thr=item["pcc_thr"],
        out_path=fig_path,
    )

print("Saved representative figures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(p)


# =========================
# Save outputs
# =========================
case_path = OUT_DIR / "Layer3_FORMAL_v1_case_metrics.csv"
summary_path = OUT_DIR / "Layer3_FORMAL_v1_overall_summary.csv"
stats_path = OUT_DIR / "Layer3_FORMAL_v1_paired_stats_bootstrap.csv"
fold_path = OUT_DIR / "Layer3_FORMAL_v1_summary_by_fold.csv"
failed_path = OUT_DIR / "Layer3_FORMAL_v1_failed_cases.csv"
run_info_path = OUT_DIR / "Layer3_FORMAL_v1_run_info.json"

case_df.to_csv(case_path, index=False)
summary_df.to_csv(summary_path, index=False)
stat_df.to_csv(stats_path, index=False)
fold_summary_df.to_csv(fold_path, index=False)
failed_df.to_csv(failed_path, index=False)

with open(run_info_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)


# Zip full output
zip_path = Path("/kaggle/working/Layer3_FORMAL_v1_40case_pathology_reliance_occlusion_RESULTS_BACKUP.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            z.write(file, arcname=str(file.relative_to(OUT_DIR.parent)))

print("\n" + "=" * 100)
print("Layer3 FORMAL v1 finished.")
print("Saved to:", OUT_DIR)
print("Backup zip:", zip_path)

print("\nOverall summary:")
display(summary_df)

print("\nPaired stats / bootstrap CI:")
display(stat_df)

print("\nFold summary:")
display(fold_summary_df)

print("\nCase metrics:")
display(case_df)

if len(failed_df) > 0:
    print("\nFailed cases:")
    display(failed_df)

print("\nSaved files:")
print(case_path)
print(summary_path)
print(stats_path)
print(fold_path)
print(failed_path)
print(run_info_path)
print("Figures:", FIG_DIR)
print("Backup zip:", zip_path)


# =========================
# Formal interpretation
# =========================
boundary_gain = overall_summary["mean_pcc_minus_baseline_boundary_PRI"]
boundary_win = overall_summary["boundary_PRI_win_rate"]
control_extra = overall_summary["mean_pcc_minus_baseline_control_drop"]
rel_boundary_gain = overall_summary["mean_pcc_minus_baseline_boundary_relative_PRI"]
rel_boundary_win = overall_summary["boundary_relative_PRI_win_rate"]
repro_base = overall_summary["mean_baseline_repro_abs_error"]
repro_pcc = overall_summary["mean_pcc_repro_abs_error"]

boundary_stats = stat_df[stat_df["metric"] == "pcc_minus_baseline_boundary_PRI"].iloc[0]
boundary_ci_low = boundary_stats["ci95_low"]
boundary_ci_high = boundary_stats["ci95_high"]
boundary_p = boundary_stats["wilcoxon_greater_p_value"]

print("\n" + "=" * 100)
print("Formal interpretation:")

print("mean_baseline_repro_abs_error:", repro_base)
print("mean_pcc_repro_abs_error:", repro_pcc)
print("mean_pcc_minus_baseline_boundary_PRI:", boundary_gain)
print("boundary_PRI_win_rate:", boundary_win)
print("boundary_PRI_95CI:", (boundary_ci_low, boundary_ci_high))
print("boundary_PRI_wilcoxon_greater_p:", boundary_p)
print("mean_pcc_minus_baseline_control_drop:", control_extra)
print("mean_pcc_minus_baseline_boundary_relative_PRI:", rel_boundary_gain)
print("boundary_relative_PRI_win_rate:", rel_boundary_win)

if repro_base > 0.01 or repro_pcc > 0.01:
    print("\nREPRODUCTION WARNING:")
    print("Original reproduction is not close enough. Do not interpret occlusion.")
elif boundary_gain > 0 and boundary_win >= 0.75 and boundary_ci_low > 0 and boundary_p < 0.01:
    print("\nFORMAL PASS / strong pathology-reliance evidence:")
    print("PCC shows stronger boundary-specific pathology reliance than baseline.")
elif boundary_gain > 0 and boundary_win >= 0.65 and boundary_p < 0.05:
    print("\nFORMAL PARTIAL PASS / moderate pathology-reliance evidence:")
    print("PCC shows positive boundary pathology-reliance evidence, but not all strong gates are met.")
else:
    print("\nFORMAL NOT PASSED:")
    print("The formal audit does not show enough evidence for stronger PCC pathology reliance.")

In [ ]:
# ============================================================
# Layer3B FORMAL v1:
# PCC Correction Localization Analysis
#
# Goal:
#   Analyze WHERE PCC changes the baseline prediction.
#
# Main question:
#   Is PCC correction concentrated around real pathological
#   structures, especially tumour core and tumour boundary?
#
# Uses:
#   Same Layer1 FORMAL_v1 5-fold checkpoints
#   Same reproduction-passed preprocessing:
#      normalization = p01_p99_brain
#      PCC mode      = residual_tanh3
#
# Outputs:
#   1. Case-level correction localization metrics
#   2. Overall summary
#   3. Bootstrap/Wilcoxon statistics
#   4. Representative correction-map figures
#   5. Backup zip
# ============================================================

from pathlib import Path
import re, json, time, gc, random, zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import nibabel as nib
import scipy.ndimage as ndi

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None


# =========================
# Config
# =========================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32

BOUNDARY_RADIUS = 5
PERITUMOUR_RADIUS = 10
AVOID_RADIUS = 8
N_CONTROL_REGIONS = 10

PCC_MAX_DELTA_LOGIT = 3.0

OUT_DIR = Path("/kaggle/working/Layer3B_FORMAL_v1_40case_correction_localization")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUT_DIR)


# =========================
# Paths
# =========================
LAYER1_DIR = Path(
    "/kaggle/input/datasets/jeechangxin/layer1-formal-v1-5fold-results-backup/"
    "pcc_independent_baseline/stage2_pcc_guided_student_learning/"
    "layer1_current_segmentation_FORMAL_v1_5fold"
)

CKPT_DIR = LAYER1_DIR / "checkpoints"
CASE_METRICS_PATH = LAYER1_DIR / "layer1_FORMAL_v1_case_metrics.csv"

DATA_ROOT = Path(
    "/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/"
    "PKG - MU-Glioma-Post/MU-Glioma-Post"
)

assert LAYER1_DIR.exists(), LAYER1_DIR
assert CKPT_DIR.exists(), CKPT_DIR
assert CASE_METRICS_PATH.exists(), CASE_METRICS_PATH
assert DATA_ROOT.exists(), DATA_ROOT

case_metrics_all = pd.read_csv(CASE_METRICS_PATH)
case_metrics_all = case_metrics_all.sort_values(["fold", "case_id"]).reset_index(drop=True)

print("Layer1 dir:", LAYER1_DIR)
print("Checkpoint dir:", CKPT_DIR)
print("Dataset root:", DATA_ROOT)
print("case_metrics:", case_metrics_all.shape)
display(case_metrics_all.head())


# =========================
# Model architecture
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()

        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))

        u2 = self.up2(b)
        if u2.shape[-2:] != e2.shape[-2:]:
            u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[-2:] != e1.shape[-2:]:
            u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        return self.out(d1)


def get_threshold_from_ckpt(ckpt, default=0.5):
    for k in ["threshold", "best_threshold", "selected_threshold", "thr"]:
        if k in ckpt:
            return float(ckpt[k])
    return float(default)


def load_fold_models(fold):
    base_path = CKPT_DIR / f"baseline_seg_fold_{fold}_FORMAL_v1.pt"
    pcc_path = CKPT_DIR / f"pcc_seg_corrector_fold_{fold}_FORMAL_v1.pt"

    assert base_path.exists(), base_path
    assert pcc_path.exists(), pcc_path

    base_ckpt = torch.load(base_path, map_location=DEVICE)
    pcc_ckpt = torch.load(pcc_path, map_location=DEVICE)

    baseline = SmallUNet2D(in_ch=1, out_ch=1, base=16).to(DEVICE)
    pcc = SmallUNet2D(in_ch=2, out_ch=1, base=16).to(DEVICE)

    baseline.load_state_dict(base_ckpt["model_state_dict"])
    pcc.load_state_dict(pcc_ckpt["model_state_dict"])

    baseline.eval()
    pcc.eval()

    base_thr = get_threshold_from_ckpt(base_ckpt, 0.5)
    pcc_thr = get_threshold_from_ckpt(pcc_ckpt, 0.5)

    print(f"Loaded fold {fold}: base_thr={base_thr}, pcc_thr={pcc_thr}")

    return baseline, pcc, base_thr, pcc_thr


# =========================
# Data loading helpers
# =========================
def parse_case_id(case_id):
    case_id = str(case_id)

    m_patient = re.search(r"(PatientID_\d+)", case_id)
    m_t = re.search(r"_T(\d+)", case_id)

    if m_patient is None or m_t is None:
        raise ValueError(f"Cannot parse case_id: {case_id}")

    return m_patient.group(1), int(m_t.group(1))


def find_patient_dir(patient_id):
    direct = DATA_ROOT / patient_id
    if direct.exists():
        return direct

    hits = list(DATA_ROOT.glob(f"**/{patient_id}"))
    hits = [h for h in hits if h.is_dir()]

    if not hits:
        raise FileNotFoundError(f"Patient folder not found: {patient_id}")

    return hits[0]


def find_timepoint_file(patient_dir, tp, kind):
    patient_dir = Path(patient_dir)

    if kind == "brain_t1c":
        patterns = ["*brain_t1c.nii", "*brain_t1c.nii.gz"]
    elif kind == "tumorMask":
        patterns = ["*tumorMask.nii", "*tumorMask.nii.gz"]
    else:
        patterns = [f"*{kind}.nii", f"*{kind}.nii.gz"]

    candidates = []
    for pat in patterns:
        candidates.extend(list(patient_dir.rglob(pat)))

    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        raise FileNotFoundError(f"No {kind} found in {patient_dir}")

    tp_tokens = [
        f"Timepoint_{tp}",
        f"Timepoint-{tp}",
        f"Timepoint {tp}",
        f"_T{tp}_",
        f"/T{tp}/",
        f"TP{tp}",
    ]

    filtered = []
    for p in candidates:
        s = str(p)
        if any(tok in s for tok in tp_tokens) or f"_Timepoint_{tp}_" in p.name:
            filtered.append(p)

    if filtered:
        return filtered[0]

    raise FileNotFoundError(
        f"Found {len(candidates)} {kind} candidates but none matched Timepoint {tp}. "
        f"First candidates: {[str(c) for c in candidates[:5]]}"
    )


def to_z_hw(arr):
    arr = np.asarray(arr)

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {arr.shape}")

    # Common NIfTI shape: H,W,Z -> Z,H,W
    if arr.shape[0] >= 128 and arr.shape[1] >= 128 and arr.shape[2] < 200:
        arr = np.moveaxis(arr, -1, 0)

    return arr.astype(np.float32)


def normalize_p01_p99_brain(vol, brain):
    vol = np.nan_to_num(vol.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    brain = brain.astype(bool)

    vals = vol[brain]
    if vals.size < 100:
        vals = vol.reshape(-1)

    lo, hi = np.percentile(vals, [1, 99])
    x = (vol - lo) / (hi - lo + 1e-6)
    x = np.clip(x, 0.0, 1.0).astype(np.float32)

    brain_mean_norm = float(x[brain].mean()) if brain.sum() > 0 else float(x.mean())

    return x, brain_mean_norm


def load_layer1_case(case_id):
    patient_id, current_tp = parse_case_id(case_id)
    pdir = find_patient_dir(patient_id)

    t1c_path = find_timepoint_file(pdir, current_tp, "brain_t1c")
    mask_path = find_timepoint_file(pdir, current_tp, "tumorMask")

    img_raw = to_z_hw(nib.load(str(t1c_path)).get_fdata()).astype(np.float32)
    mask_raw = to_z_hw(nib.load(str(mask_path)).get_fdata()).astype(np.float32)

    mask = mask_raw > 0.5
    brain = np.abs(img_raw) > 1e-6
    if brain.sum() < 100:
        brain = np.ones_like(img_raw, dtype=bool)

    img_norm, brain_mean_norm = normalize_p01_p99_brain(img_raw, brain)

    X = img_norm[:, None, :, :].astype(np.float32)
    Y = mask[:, None, :, :].astype(np.float32)

    return X, Y, brain.astype(bool), brain_mean_norm, {
        "patient_id": patient_id,
        "current_tp": current_tp,
        "t1c_path": str(t1c_path),
        "mask_path": str(mask_path),
    }


# =========================
# Prediction and metrics
# =========================
@torch.no_grad()
def predict_baseline_and_pcc(baseline_model, pcc_model, X, batch_size=32):
    baseline_model.eval()
    pcc_model.eval()

    base_probs = []
    pcc_probs = []

    for start in range(0, X.shape[0], batch_size):
        xb = torch.from_numpy(X[start:start + batch_size]).float().to(DEVICE)

        base_logits = baseline_model(xb)
        base_prob = torch.sigmoid(base_logits)

        pcc_input = torch.cat([xb, base_prob], dim=1)
        residual_raw = pcc_model(pcc_input)

        corrected_logits = base_logits + PCC_MAX_DELTA_LOGIT * torch.tanh(residual_raw)
        pcc_prob = torch.sigmoid(corrected_logits)

        base_probs.append(base_prob.detach().cpu().numpy().astype(np.float32))
        pcc_probs.append(pcc_prob.detach().cpu().numpy().astype(np.float32))

    return np.concatenate(base_probs, axis=0), np.concatenate(pcc_probs, axis=0)


def dice_iou_prob(prob, target, threshold=0.5, eps=1e-8):
    pred = prob >= threshold
    target = target.astype(bool)

    inter = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()
    pred_sum = pred.sum()
    target_sum = target.sum()

    dice = (2 * inter + eps) / (pred_sum + target_sum + eps)
    iou = (inter + eps) / (union + eps)

    return float(dice), float(iou)


# =========================
# Region helpers
# =========================
def make_regions(target_zyx, brain_zyx):
    target = target_zyx.astype(bool)
    brain = brain_zyx.astype(bool)

    core = np.logical_and(target, brain)

    dil = ndi.binary_dilation(core, iterations=BOUNDARY_RADIUS)
    ero = ndi.binary_erosion(core, iterations=BOUNDARY_RADIUS)
    boundary = np.logical_and(dil, np.logical_not(ero))
    boundary = np.logical_and(boundary, brain)

    peritumour = ndi.binary_dilation(core, iterations=PERITUMOUR_RADIUS)
    peritumour = np.logical_and(peritumour, np.logical_not(core))
    peritumour = np.logical_and(peritumour, brain)

    avoid = ndi.binary_dilation(core, iterations=AVOID_RADIUS)
    avoid = np.logical_or(avoid, np.logical_not(brain))

    non_tumour_brain = np.logical_and(brain, np.logical_not(core))

    return core, boundary, peritumour, non_tumour_brain, avoid


def make_shifted_control(region_mask, avoid_mask, brain_mask, rng, tries=100):
    region = region_mask.astype(bool)
    avoid = avoid_mask.astype(bool)
    brain = brain_mask.astype(bool)

    n = int(region.sum())
    if n <= 0:
        return np.zeros_like(region, dtype=bool)

    zdim, h, w = region.shape
    best = None
    best_count = -1

    for _ in range(tries):
        dz = rng.integers(-max(2, zdim // 3), max(3, zdim // 3))
        dy = rng.integers(-max(8, h // 3), max(9, h // 3))
        dx = rng.integers(-max(8, w // 3), max(9, w // 3))

        shifted = np.roll(region, shift=(dz, dy, dx), axis=(0, 1, 2))
        shifted = np.logical_and(shifted, brain)
        shifted = np.logical_and(shifted, np.logical_not(avoid))

        c = int(shifted.sum())

        if c > best_count:
            best = shifted
            best_count = c

        if c >= int(0.80 * n):
            return shifted

    candidates = np.logical_and(brain, np.logical_not(avoid))
    idx = np.argwhere(candidates)

    if len(idx) == 0:
        return np.zeros_like(region, dtype=bool)

    take = min(n, len(idx))
    chosen = idx[rng.choice(len(idx), size=take, replace=False)]

    control = np.zeros_like(region, dtype=bool)
    control[chosen[:, 0], chosen[:, 1], chosen[:, 2]] = True

    return control


def safe_region_sum(arr, mask, eps=1e-8):
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return 0.0
    return float(arr[mask].sum())


def safe_region_mean(arr, mask):
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return 0.0
    return float(arr[mask].mean())


# =========================
# Stats helpers
# =========================
def bootstrap_ci(vals, n_boot=5000, seed=42):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = []

    for _ in range(n_boot):
        s = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.mean(s))

    return float(np.mean(vals)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def summarize_metric(case_df, metric):
    vals = case_df[metric].values.astype(float)
    mean, low, high = bootstrap_ci(vals)

    if wilcoxon is not None:
        try:
            w_greater = wilcoxon(vals, alternative="greater")
            p_greater = float(w_greater.pvalue)
            stat_greater = float(w_greater.statistic)

            w_two = wilcoxon(vals, alternative="two-sided")
            p_two = float(w_two.pvalue)
            stat_two = float(w_two.statistic)
        except Exception:
            p_greater = np.nan
            stat_greater = np.nan
            p_two = np.nan
            stat_two = np.nan
    else:
        p_greater = np.nan
        stat_greater = np.nan
        p_two = np.nan
        stat_two = np.nan

    return {
        "metric": metric,
        "mean": mean,
        "median": float(np.nanmedian(vals)),
        "std": float(np.nanstd(vals)),
        "ci95_low": low,
        "ci95_high": high,
        "positive_cases": int(np.sum(vals > 0)),
        "negative_cases": int(np.sum(vals < 0)),
        "zero_cases": int(np.sum(vals == 0)),
        "positive_rate": float(np.mean(vals > 0)),
        "wilcoxon_greater_statistic": stat_greater,
        "wilcoxon_greater_p_value": p_greater,
        "wilcoxon_two_sided_statistic": stat_two,
        "wilcoxon_two_sided_p_value": p_two,
    }


def summarize_by_fold(case_df, metric):
    rows = []
    for fold, g in case_df.groupby("fold"):
        vals = g[metric].values.astype(float)
        rows.append({
            "fold": int(fold),
            "metric": metric,
            "n_cases": int(len(g)),
            "mean": float(np.mean(vals)),
            "median": float(np.median(vals)),
            "positive_cases": int(np.sum(vals > 0)),
            "positive_rate": float(np.mean(vals > 0)),
        })
    return rows


# =========================
# Figure helper
# =========================
def save_correction_figure(
    case_id, X, Y, base_prob, pcc_prob, delta, abs_delta,
    core, boundary, peritumour, control,
    base_thr, pcc_thr, out_path
):
    target = Y[:, 0] > 0.5

    score = abs_delta.reshape(abs_delta.shape[0], -1).sum(axis=1)
    if score.max() <= 0:
        score = target.reshape(target.shape[0], -1).sum(axis=1)

    z = int(np.argmax(score))

    img = X[z, 0]
    true_z = target[z]
    base_bin = base_prob[z, 0] >= base_thr
    pcc_bin = pcc_prob[z, 0] >= pcc_thr

    delta_z = delta[z]
    abs_delta_z = abs_delta[z]

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    axes[0, 0].imshow(img, cmap="gray")
    axes[0, 0].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 0].set_title("MRI + true mask")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(img, cmap="gray")
    axes[0, 1].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 1].contour(base_bin, levels=[0.5], colors="blue", linewidths=1)
    axes[0, 1].set_title("Baseline prediction")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(img, cmap="gray")
    axes[0, 2].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 2].contour(pcc_bin, levels=[0.5], colors="red", linewidths=1)
    axes[0, 2].set_title("PCC prediction")
    axes[0, 2].axis("off")

    axes[0, 3].imshow(img, cmap="gray")
    axes[0, 3].imshow(boundary[z], cmap="Reds", alpha=0.35)
    axes[0, 3].imshow(control[z], cmap="Blues", alpha=0.25)
    axes[0, 3].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[0, 3].set_title("Boundary red / control blue")
    axes[0, 3].axis("off")

    axes[1, 0].imshow(img, cmap="gray")
    axes[1, 0].imshow(abs_delta_z, cmap="hot", alpha=np.clip(abs_delta_z / (abs_delta_z.max() + 1e-8), 0, 0.85))
    axes[1, 0].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 0].set_title("|PCC - baseline|")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(delta_z, cmap="bwr", vmin=-np.max(np.abs(delta_z))-1e-8, vmax=np.max(np.abs(delta_z))+1e-8)
    axes[1, 1].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 1].set_title("Signed correction: red up, blue down")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(img, cmap="gray")
    axes[1, 2].imshow(core[z], cmap="Reds", alpha=0.30)
    axes[1, 2].imshow(peritumour[z], cmap="Purples", alpha=0.25)
    axes[1, 2].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 2].set_title("Core red / peritumour purple")
    axes[1, 2].axis("off")

    useful = np.zeros_like(delta_z, dtype=np.float32)
    useful[np.logical_and(true_z, delta_z > 0)] = delta_z[np.logical_and(true_z, delta_z > 0)]
    useful[np.logical_and(~true_z, delta_z < 0)] = -delta_z[np.logical_and(~true_z, delta_z < 0)]

    axes[1, 3].imshow(img, cmap="gray")
    axes[1, 3].imshow(useful, cmap="hot", alpha=np.clip(useful / (useful.max() + 1e-8), 0, 0.85))
    axes[1, 3].contour(true_z, levels=[0.5], colors="lime", linewidths=1)
    axes[1, 3].set_title("Useful correction")
    axes[1, 3].axis("off")

    fig.suptitle(f"{case_id} | slice {z}", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close(fig)


# =========================
# Main formal experiment
# =========================
print("\nFormal cases:")
display(case_metrics_all[["case_id", "fold", "baseline_dice", "pcc_dice", "dice_gain", "target_voxels"]])

start_time = time.time()
rng = np.random.default_rng(SEED)

fold_model_cache = {}
rows = []
failed_cases = []
figure_cache = []

for _, row in tqdm(case_metrics_all.iterrows(), total=len(case_metrics_all), desc="Layer3B Correction Localization"):
    case_id = row["case_id"]
    fold = int(row["fold"])

    if fold not in fold_model_cache:
        fold_model_cache[fold] = load_fold_models(fold)

    baseline_model, pcc_model, base_thr, pcc_thr = fold_model_cache[fold]

    try:
        X, Y, brain_mask, brain_mean_norm, meta = load_layer1_case(case_id)

        target = Y[:, 0] > 0.5
        core, boundary, peritumour, non_tumour_brain, avoid = make_regions(target, brain_mask)

        if core.sum() <= 0 or boundary.sum() <= 0:
            raise ValueError("Empty tumour core or boundary")

        base_prob, pcc_prob = predict_baseline_and_pcc(baseline_model, pcc_model, X, BATCH_SIZE)

        base_dice, base_iou = dice_iou_prob(base_prob, Y, base_thr)
        pcc_dice, pcc_iou = dice_iou_prob(pcc_prob, Y, pcc_thr)

        csv_base = float(row["baseline_dice"])
        csv_pcc = float(row["pcc_dice"])

        base_repro_error = abs(base_dice - csv_base)
        pcc_repro_error = abs(pcc_dice - csv_pcc)

        delta = (pcc_prob[:, 0] - base_prob[:, 0]).astype(np.float32)
        abs_delta = np.abs(delta).astype(np.float32)
        pos_delta = np.clip(delta, 0, None)
        neg_delta = np.clip(-delta, 0, None)

        total_abs = float(abs_delta[brain_mask].sum() + 1e-8)

        # Matched control regions for boundary-like comparison
        control_masks = []
        control_abs_sums = []
        control_abs_means = []

        for k in range(N_CONTROL_REGIONS):
            control = make_shifted_control(boundary, avoid, brain_mask, rng)
            control_masks.append(control)

            control_abs_sums.append(safe_region_sum(abs_delta, control))
            control_abs_means.append(safe_region_mean(abs_delta, control))

        mean_control_abs_sum = float(np.mean(control_abs_sums))
        mean_control_abs_mean = float(np.mean(control_abs_means))
        first_control = control_masks[0]

        # Region-level correction mass
        core_abs_sum = safe_region_sum(abs_delta, core)
        boundary_abs_sum = safe_region_sum(abs_delta, boundary)
        peritumour_abs_sum = safe_region_sum(abs_delta, peritumour)
        non_tumour_abs_sum = safe_region_sum(abs_delta, non_tumour_brain)

        core_abs_mean = safe_region_mean(abs_delta, core)
        boundary_abs_mean = safe_region_mean(abs_delta, boundary)
        peritumour_abs_mean = safe_region_mean(abs_delta, peritumour)
        non_tumour_abs_mean = safe_region_mean(abs_delta, non_tumour_brain)

        # Useful correction:
        #   inside target: increasing probability is useful
        #   outside target: decreasing probability is useful
        useful_inside = np.logical_and(target, delta > 0)
        harmful_inside = np.logical_and(target, delta < 0)

        useful_outside = np.logical_and(np.logical_and(~target, brain_mask), delta < 0)
        harmful_outside = np.logical_and(np.logical_and(~target, brain_mask), delta > 0)

        useful_mass_inside = float(pos_delta[useful_inside].sum())
        harmful_mass_inside = float(neg_delta[harmful_inside].sum())

        useful_mass_outside = float(neg_delta[useful_outside].sum())
        harmful_mass_outside = float(pos_delta[harmful_outside].sum())

        useful_mass_total = useful_mass_inside + useful_mass_outside
        harmful_mass_total = harmful_mass_inside + harmful_mass_outside

        # Baseline threshold errors
        base_bin = base_prob[:, 0] >= base_thr
        pcc_bin = pcc_prob[:, 0] >= pcc_thr

        baseline_fn = np.logical_and(target, ~base_bin)
        baseline_fp = np.logical_and(np.logical_and(~target, brain_mask), base_bin)

        # Correction on baseline error regions
        fn_positive_sum = safe_region_sum(pos_delta, baseline_fn)
        fn_negative_sum = safe_region_sum(neg_delta, baseline_fn)
        fp_negative_sum = safe_region_sum(neg_delta, baseline_fp)
        fp_positive_sum = safe_region_sum(pos_delta, baseline_fp)

        fn_useful_dominance = fn_positive_sum - fn_negative_sum
        fp_useful_dominance = fp_negative_sum - fp_positive_sum

        # Boundary false-negative focus
        boundary_fn = np.logical_and(boundary, baseline_fn)
        boundary_fn_positive_sum = safe_region_sum(pos_delta, boundary_fn)
        boundary_fn_negative_sum = safe_region_sum(neg_delta, boundary_fn)
        boundary_fn_useful_dominance = boundary_fn_positive_sum - boundary_fn_negative_sum

        # Ratios / enrichment
        eps = 1e-8

        out = {
            "case_id": case_id,
            "fold": fold,
            "patient_id": meta["patient_id"],
            "current_tp": meta["current_tp"],

            "target_voxels": int(target.sum()),
            "core_voxels": int(core.sum()),
            "boundary_voxels": int(boundary.sum()),
            "peritumour_voxels": int(peritumour.sum()),
            "brain_voxels": int(brain_mask.sum()),
            "mean_control_voxels": float(np.mean([c.sum() for c in control_masks])),

            "csv_baseline_dice": csv_base,
            "csv_pcc_dice": csv_pcc,
            "csv_dice_gain": csv_pcc - csv_base,

            "baseline_original_dice": base_dice,
            "pcc_original_dice": pcc_dice,
            "original_dice_gain": pcc_dice - base_dice,

            "baseline_repro_abs_error": base_repro_error,
            "pcc_repro_abs_error": pcc_repro_error,

            "baseline_original_iou": base_iou,
            "pcc_original_iou": pcc_iou,

            "total_abs_correction_mass": total_abs,

            # Region correction ratios
            "core_abs_correction_ratio": core_abs_sum / total_abs,
            "boundary_abs_correction_ratio": boundary_abs_sum / total_abs,
            "peritumour_abs_correction_ratio": peritumour_abs_sum / total_abs,
            "non_tumour_abs_correction_ratio": non_tumour_abs_sum / total_abs,

            # Region mean correction intensity
            "core_abs_correction_mean": core_abs_mean,
            "boundary_abs_correction_mean": boundary_abs_mean,
            "peritumour_abs_correction_mean": peritumour_abs_mean,
            "non_tumour_abs_correction_mean": non_tumour_abs_mean,
            "control_abs_correction_mean": mean_control_abs_mean,

            # Enrichment against matched control
            "core_vs_control_abs_mean_enrichment": core_abs_mean / (mean_control_abs_mean + eps),
            "boundary_vs_control_abs_mean_enrichment": boundary_abs_mean / (mean_control_abs_mean + eps),
            "peritumour_vs_control_abs_mean_enrichment": peritumour_abs_mean / (mean_control_abs_mean + eps),

            "boundary_minus_control_abs_mean": boundary_abs_mean - mean_control_abs_mean,
            "core_minus_control_abs_mean": core_abs_mean - mean_control_abs_mean,
            "peritumour_minus_control_abs_mean": peritumour_abs_mean - mean_control_abs_mean,

            # Useful vs harmful correction
            "useful_mass_inside_target": useful_mass_inside,
            "harmful_mass_inside_target": harmful_mass_inside,
            "useful_mass_outside_target": useful_mass_outside,
            "harmful_mass_outside_target": harmful_mass_outside,
            "useful_mass_total": useful_mass_total,
            "harmful_mass_total": harmful_mass_total,
            "useful_minus_harmful_mass": useful_mass_total - harmful_mass_total,
            "useful_correction_fraction": useful_mass_total / (useful_mass_total + harmful_mass_total + eps),

            # Baseline error correction
            "baseline_fn_voxels": int(baseline_fn.sum()),
            "baseline_fp_voxels": int(baseline_fp.sum()),
            "fn_positive_correction_sum": fn_positive_sum,
            "fn_negative_correction_sum": fn_negative_sum,
            "fp_negative_correction_sum": fp_negative_sum,
            "fp_positive_correction_sum": fp_positive_sum,
            "fn_useful_dominance": fn_useful_dominance,
            "fp_useful_dominance": fp_useful_dominance,
            "error_region_useful_dominance": fn_useful_dominance + fp_useful_dominance,

            "boundary_fn_voxels": int(boundary_fn.sum()),
            "boundary_fn_positive_correction_sum": boundary_fn_positive_sum,
            "boundary_fn_negative_correction_sum": boundary_fn_negative_sum,
            "boundary_fn_useful_dominance": boundary_fn_useful_dominance,
        }

        rows.append(out)

        figure_cache.append({
            "case_id": case_id,
            "score": out["boundary_vs_control_abs_mean_enrichment"],
            "X": X,
            "Y": Y,
            "base_prob": base_prob,
            "pcc_prob": pcc_prob,
            "delta": delta,
            "abs_delta": abs_delta,
            "core": core,
            "boundary": boundary,
            "peritumour": peritumour,
            "control": first_control,
            "base_thr": base_thr,
            "pcc_thr": pcc_thr,
        })

        del X, Y, base_prob, pcc_prob, delta, abs_delta, pos_delta, neg_delta
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    except Exception as e:
        failed_cases.append({
            "case_id": case_id,
            "fold": fold,
            "reason": repr(e),
        })
        print("Failed case:", case_id, "reason:", repr(e))


case_df = pd.DataFrame(rows)
failed_df = pd.DataFrame(failed_cases)

assert len(case_df) > 0, "No cases evaluated."

print("\nEvaluated cases:", len(case_df))
print("Failed cases:", len(failed_df))
if len(failed_df) > 0:
    display(failed_df)


# =========================
# Summary and stats
# =========================
summary_metrics = [
    "original_dice_gain",
    "baseline_repro_abs_error",
    "pcc_repro_abs_error",

    "core_abs_correction_ratio",
    "boundary_abs_correction_ratio",
    "peritumour_abs_correction_ratio",
    "non_tumour_abs_correction_ratio",

    "core_vs_control_abs_mean_enrichment",
    "boundary_vs_control_abs_mean_enrichment",
    "peritumour_vs_control_abs_mean_enrichment",

    "core_minus_control_abs_mean",
    "boundary_minus_control_abs_mean",
    "peritumour_minus_control_abs_mean",

    "useful_minus_harmful_mass",
    "useful_correction_fraction",

    "fn_useful_dominance",
    "fp_useful_dominance",
    "error_region_useful_dominance",
    "boundary_fn_useful_dominance",
]

stat_df = pd.DataFrame([summarize_metric(case_df, m) for m in summary_metrics])

fold_rows = []
for metric in summary_metrics:
    fold_rows.extend(summarize_by_fold(case_df, metric))
fold_summary_df = pd.DataFrame(fold_rows)

overall_summary = {
    "experiment": "Layer3B FORMAL v1 40-case PCC correction localization analysis",
    "cases_total_in_csv": int(len(case_metrics_all)),
    "cases_evaluated": int(len(case_df)),
    "cases_failed": int(len(failed_df)),
    "normalization": "p01_p99_brain",
    "pcc_mode": "residual_tanh3",
    "boundary_radius": int(BOUNDARY_RADIUS),
    "peritumour_radius": int(PERITUMOUR_RADIUS),
    "control_regions_per_case": int(N_CONTROL_REGIONS),
    "runtime_minutes": float((time.time() - start_time) / 60.0),

    "mean_csv_baseline_dice": float(case_df["csv_baseline_dice"].mean()),
    "mean_csv_pcc_dice": float(case_df["csv_pcc_dice"].mean()),
    "mean_csv_dice_gain": float(case_df["csv_dice_gain"].mean()),

    "mean_baseline_original_dice": float(case_df["baseline_original_dice"].mean()),
    "mean_pcc_original_dice": float(case_df["pcc_original_dice"].mean()),
    "mean_original_dice_gain": float(case_df["original_dice_gain"].mean()),

    "mean_baseline_repro_abs_error": float(case_df["baseline_repro_abs_error"].mean()),
    "mean_pcc_repro_abs_error": float(case_df["pcc_repro_abs_error"].mean()),

    "mean_core_abs_correction_ratio": float(case_df["core_abs_correction_ratio"].mean()),
    "mean_boundary_abs_correction_ratio": float(case_df["boundary_abs_correction_ratio"].mean()),
    "mean_peritumour_abs_correction_ratio": float(case_df["peritumour_abs_correction_ratio"].mean()),
    "mean_non_tumour_abs_correction_ratio": float(case_df["non_tumour_abs_correction_ratio"].mean()),

    "mean_core_vs_control_abs_mean_enrichment": float(case_df["core_vs_control_abs_mean_enrichment"].mean()),
    "mean_boundary_vs_control_abs_mean_enrichment": float(case_df["boundary_vs_control_abs_mean_enrichment"].mean()),
    "mean_peritumour_vs_control_abs_mean_enrichment": float(case_df["peritumour_vs_control_abs_mean_enrichment"].mean()),

    "boundary_enrichment_positive_rate": float((case_df["boundary_vs_control_abs_mean_enrichment"] > 1.0).mean()),
    "core_enrichment_positive_rate": float((case_df["core_vs_control_abs_mean_enrichment"] > 1.0).mean()),

    "mean_boundary_minus_control_abs_mean": float(case_df["boundary_minus_control_abs_mean"].mean()),
    "mean_core_minus_control_abs_mean": float(case_df["core_minus_control_abs_mean"].mean()),

    "mean_useful_correction_fraction": float(case_df["useful_correction_fraction"].mean()),
    "useful_fraction_above_50_rate": float((case_df["useful_correction_fraction"] > 0.5).mean()),

    "mean_useful_minus_harmful_mass": float(case_df["useful_minus_harmful_mass"].mean()),
    "useful_minus_harmful_positive_rate": float((case_df["useful_minus_harmful_mass"] > 0).mean()),

    "mean_fn_useful_dominance": float(case_df["fn_useful_dominance"].mean()),
    "fn_useful_dominance_positive_rate": float((case_df["fn_useful_dominance"] > 0).mean()),

    "mean_fp_useful_dominance": float(case_df["fp_useful_dominance"].mean()),
    "fp_useful_dominance_positive_rate": float((case_df["fp_useful_dominance"] > 0).mean()),

    "mean_error_region_useful_dominance": float(case_df["error_region_useful_dominance"].mean()),
    "error_region_useful_dominance_positive_rate": float((case_df["error_region_useful_dominance"] > 0).mean()),

    "mean_boundary_fn_useful_dominance": float(case_df["boundary_fn_useful_dominance"].mean()),
    "boundary_fn_useful_dominance_positive_rate": float((case_df["boundary_fn_useful_dominance"] > 0).mean()),
}

summary_df = pd.DataFrame([overall_summary])


# =========================
# Representative figures
# =========================
figure_cache_sorted = sorted(figure_cache, key=lambda x: x["score"], reverse=True)

selected_figs = []
selected_figs.extend(figure_cache_sorted[:3])

mid_start = max(0, len(figure_cache_sorted) // 2 - 1)
selected_figs.extend(figure_cache_sorted[mid_start:mid_start + 3])

selected_figs.extend(figure_cache_sorted[-3:])

seen = set()
selected_unique = []

for item in selected_figs:
    if item["case_id"] not in seen:
        selected_unique.append(item)
        seen.add(item["case_id"])

for item in selected_unique:
    fig_path = FIG_DIR / f"{item['case_id']}_Layer3B_correction_localization.png"

    save_correction_figure(
        case_id=item["case_id"],
        X=item["X"],
        Y=item["Y"],
        base_prob=item["base_prob"],
        pcc_prob=item["pcc_prob"],
        delta=item["delta"],
        abs_delta=item["abs_delta"],
        core=item["core"],
        boundary=item["boundary"],
        peritumour=item["peritumour"],
        control=item["control"],
        base_thr=item["base_thr"],
        pcc_thr=item["pcc_thr"],
        out_path=fig_path,
    )

print("Saved representative figures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(p)


# =========================
# Save outputs
# =========================
case_path = OUT_DIR / "Layer3B_FORMAL_v1_case_metrics.csv"
summary_path = OUT_DIR / "Layer3B_FORMAL_v1_overall_summary.csv"
stats_path = OUT_DIR / "Layer3B_FORMAL_v1_paired_stats_bootstrap.csv"
fold_path = OUT_DIR / "Layer3B_FORMAL_v1_summary_by_fold.csv"
failed_path = OUT_DIR / "Layer3B_FORMAL_v1_failed_cases.csv"
run_info_path = OUT_DIR / "Layer3B_FORMAL_v1_run_info.json"

case_df.to_csv(case_path, index=False)
summary_df.to_csv(summary_path, index=False)
stat_df.to_csv(stats_path, index=False)
fold_summary_df.to_csv(fold_path, index=False)
failed_df.to_csv(failed_path, index=False)

with open(run_info_path, "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, indent=2)


# Zip full output
zip_path = Path("/kaggle/working/Layer3B_FORMAL_v1_40case_correction_localization_RESULTS_BACKUP.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            z.write(file, arcname=str(file.relative_to(OUT_DIR.parent)))

print("\n" + "=" * 100)
print("Layer3B FORMAL v1 finished.")
print("Saved to:", OUT_DIR)
print("Backup zip:", zip_path)

print("\nOverall summary:")
display(summary_df)

print("\nPaired stats / bootstrap CI:")
display(stat_df)

print("\nFold summary:")
display(fold_summary_df)

print("\nCase metrics:")
display(case_df)

if len(failed_df) > 0:
    print("\nFailed cases:")
    display(failed_df)

print("\nSaved files:")
print(case_path)
print(summary_path)
print(stats_path)
print(fold_path)
print(failed_path)
print(run_info_path)
print("Figures:", FIG_DIR)
print("Backup zip:", zip_path)


# =========================
# Formal interpretation
# =========================
boundary_enrich = overall_summary["mean_boundary_vs_control_abs_mean_enrichment"]
boundary_enrich_rate = overall_summary["boundary_enrichment_positive_rate"]
boundary_minus_control = overall_summary["mean_boundary_minus_control_abs_mean"]

useful_fraction = overall_summary["mean_useful_correction_fraction"]
useful_rate = overall_summary["useful_fraction_above_50_rate"]

error_useful = overall_summary["mean_error_region_useful_dominance"]
error_useful_rate = overall_summary["error_region_useful_dominance_positive_rate"]

repro_base = overall_summary["mean_baseline_repro_abs_error"]
repro_pcc = overall_summary["mean_pcc_repro_abs_error"]

boundary_stat = stat_df[stat_df["metric"] == "boundary_vs_control_abs_mean_enrichment"].iloc[0]
boundary_ci_low = boundary_stat["ci95_low"]
boundary_ci_high = boundary_stat["ci95_high"]
boundary_p = boundary_stat["wilcoxon_greater_p_value"]

boundary_minus_stat = stat_df[stat_df["metric"] == "boundary_minus_control_abs_mean"].iloc[0]
boundary_minus_ci_low = boundary_minus_stat["ci95_low"]
boundary_minus_p = boundary_minus_stat["wilcoxon_greater_p_value"]

useful_stat = stat_df[stat_df["metric"] == "useful_correction_fraction"].iloc[0]
useful_ci_low = useful_stat["ci95_low"]

error_stat = stat_df[stat_df["metric"] == "error_region_useful_dominance"].iloc[0]
error_ci_low = error_stat["ci95_low"]
error_p = error_stat["wilcoxon_greater_p_value"]

print("\n" + "=" * 100)
print("Formal interpretation:")

print("mean_baseline_repro_abs_error:", repro_base)
print("mean_pcc_repro_abs_error:", repro_pcc)

print("mean_boundary_vs_control_abs_mean_enrichment:", boundary_enrich)
print("boundary_enrichment_positive_rate:", boundary_enrich_rate)
print("boundary_enrichment_95CI:", (boundary_ci_low, boundary_ci_high))
print("boundary_enrichment_wilcoxon_greater_p:", boundary_p)

print("mean_boundary_minus_control_abs_mean:", boundary_minus_control)
print("boundary_minus_control_95CI_low:", boundary_minus_ci_low)
print("boundary_minus_control_wilcoxon_greater_p:", boundary_minus_p)

print("mean_useful_correction_fraction:", useful_fraction)
print("useful_fraction_above_50_rate:", useful_rate)

print("mean_error_region_useful_dominance:", error_useful)
print("error_region_useful_dominance_positive_rate:", error_useful_rate)
print("error_region_useful_dominance_95CI_low:", error_ci_low)
print("error_region_useful_dominance_wilcoxon_greater_p:", error_p)

if repro_base > 0.01 or repro_pcc > 0.01:
    print("\nREPRODUCTION WARNING:")
    print("Original reproduction is not close enough. Do not interpret localization.")
elif boundary_enrich > 1.0 and boundary_enrich_rate >= 0.70 and boundary_minus_control > 0 and boundary_minus_ci_low > 0:
    if useful_fraction > 0.5 and error_useful > 0:
        print("\nFORMAL PASS / strong correction-localization evidence:")
        print("PCC corrections are enriched around tumour boundary and are directionally useful overall.")
    else:
        print("\nFORMAL PARTIAL PASS / localization evidence:")
        print("PCC corrections are enriched around tumour boundary, but useful-correction directionality is only moderate.")
elif boundary_enrich > 1.0 and boundary_enrich_rate >= 0.60:
    print("\nFORMAL PARTIAL PASS / moderate localization evidence:")
    print("PCC corrections show some boundary enrichment, but not all strong localization gates are met.")
else:
    print("\nFORMAL NOT PASSED:")
    print("Correction localization does not show clear pathology-region enrichment.")

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt

FIG_DIR = Path("/kaggle/working/Layer3B_FORMAL_v1_40case_correction_localization/figures")

figs = sorted(FIG_DIR.glob("*.png"))

print("Number of figures:", len(figs))
for p in figs:
    print(p.name)

for p in figs:
    img = Image.open(p)
    plt.figure(figsize=(18, 9))
    plt.imshow(img)
    plt.axis("off")
    plt.title(p.name, fontsize=12)
    plt.show()

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import FileLink, display
import math

FIG_DIR = Path("/kaggle/working/Layer3B_FORMAL_v1_40case_correction_localization/figures")
OUT_PATH = Path("/kaggle/working/Layer3B_correction_localization_contact_sheet.jpg")

figs = sorted(FIG_DIR.glob("*.png"))
assert len(figs) > 0, "No figures found."

thumb_w = 900
thumb_h = 450
label_h = 50
cols = 1
rows = len(figs)

sheet = Image.new("RGB", (thumb_w * cols, rows * (thumb_h + label_h)), "white")
draw = ImageDraw.Draw(sheet)

for i, p in enumerate(figs):
    img = Image.open(p).convert("RGB")
    img.thumbnail((thumb_w, thumb_h))
    
    canvas = Image.new("RGB", (thumb_w, thumb_h), "white")
    x = (thumb_w - img.width) // 2
    y = (thumb_h - img.height) // 2
    canvas.paste(img, (x, y))
    
    top = i * (thumb_h + label_h)
    sheet.paste(canvas, (0, top))
    draw.text((10, top + thumb_h + 10), p.name, fill=(0, 0, 0))

sheet.save(OUT_PATH, quality=95)

print("Saved:", OUT_PATH)
display(FileLink(str(OUT_PATH)))

# Layer 2E: Equal-Information-Access Correction Baseline

Purpose:
This experiment tests whether PCC improvement is merely due to access to the retrospective future-change target, or whether PCC provides additional correction value beyond target access alone.

Locked cohort:
- MU-Glioma-Post
- 40 usable longitudinal T1c pairs
- Same baseline future-change maps as Layer 2
- Same future-change target: future tumour mask AND NOT current tumour mask
- Same evaluation metrics: Dice, IoU, target focus, log10 ratio

New equal-information-access baselines:
1. EIA-linear
2. EIA-blend-0.75
3. EIA-morph

Primary comparison:
- PCC vs EIA-linear

Secondary comparisons:
- PCC vs EIA-blend-0.75
- PCC vs EIA-morph

In [ ]:
from pathlib import Path
import os, json, glob, re, math, random
import numpy as np
import pandas as pd

RUN_NAME = "Layer2E_equal_information_access_baseline_v1"

OUT_DIR = Path("/kaggle/working") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = OUT_DIR / "figures"
MAP_DIR = OUT_DIR / "maps"
TABLE_DIR = OUT_DIR / "tables"
LOG_DIR = OUT_DIR / "logs"

for d in [FIG_DIR, MAP_DIR, TABLE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)
print("Subdirectories created:")
for d in [FIG_DIR, MAP_DIR, TABLE_DIR, LOG_DIR]:
    print(" -", d)

In [ ]:
from pathlib import Path
import os

INPUT_ROOT = Path("/kaggle/input")

print("Datasets under /kaggle/input:")
for p in sorted(INPUT_ROOT.iterdir()):
    if p.is_dir():
        print(" -", p)

print("\nSearching for potentially relevant files...")

keywords = [
    "layer2",
    "pcc",
    "baseline",
    "future",
    "change",
    "case",
    "metric",
    "fixed",
    "naive",
    "target",
    "nii",
    "csv",
    "json"
]

matches = []

for root, dirs, files in os.walk(INPUT_ROOT):
    for f in files:
        fp = Path(root) / f
        name_low = f.lower()
        path_low = str(fp).lower()
        if any(k in name_low or k in path_low for k in keywords):
            matches.append(fp)

print(f"Found {len(matches)} potentially relevant files.")

for fp in matches[:300]:
    print(fp)

if len(matches) > 300:
    print(f"... truncated, {len(matches)-300} more files not shown")

In [ ]:
from pathlib import Path
import os, glob, json, re
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/input/datasets")

C2_DIR = ROOT / "jeechangxin/c2-v2-5epoch-backup"
MODELA_DIR = ROOT / "jeechangxin/model-a-backup-final"
TABLES_DIR = ROOT / "jeechangxin/pcc-results-tables-only-backup"
RAW_DIR = ROOT / "stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post"

paths = {
    "C2_DIR": C2_DIR,
    "MODELA_DIR": MODELA_DIR,
    "TABLES_DIR": TABLES_DIR,
    "RAW_DIR": RAW_DIR,
}

for name, p in paths.items():
    print(name, "exists =", p.exists(), "->", p)

In [ ]:
csv_candidates = [
    C2_DIR / "c2_v2_alpha_case_metrics.csv",
    C2_DIR / "pairwise_C2_v2_alpha_vs_Model_A.csv",
    C2_DIR / "model_A_vs_C2_v2_alpha_compact_comparison.csv",
    C2_DIR / "c2_v2_alpha_summary.csv",
    C2_DIR / "c2_v2_teacher_quality.csv",
    C2_DIR / "c2_v2_teacher_quality_summary.csv",
    MODELA_DIR / "direct_target_case_metrics.csv",
    MODELA_DIR / "direct_target_summary.csv",
    MODELA_DIR / "matched_5fold_splits_seed42.csv",
]

for fp in csv_candidates:
    print("\n" + "="*100)
    print("FILE:", fp)
    print("exists:", fp.exists())
    if fp.exists():
        try:
            df = pd.read_csv(fp)
            print("shape:", df.shape)
            print("columns:")
            print(list(df.columns))
            display(df.head(3))
        except Exception as e:
            print("READ ERROR:", repr(e))

In [ ]:
pcc_map_dir = C2_DIR / "pcc_on_modelA_oof_teacher_maps"
modela_map_dir = MODELA_DIR / "direct_target_pred_maps"

pcc_maps = sorted(pcc_map_dir.glob("*.npy")) if pcc_map_dir.exists() else []
modela_maps = sorted(modela_map_dir.glob("*.npy")) if modela_map_dir.exists() else []

print("PCC map dir:", pcc_map_dir)
print("PCC maps:", len(pcc_maps))

print("Model A map dir:", modela_map_dir)
print("Model A maps:", len(modela_maps))

def inspect_npy(fp):
    arr = np.load(fp)
    return {
        "file": fp.name,
        "shape": arr.shape,
        "dtype": str(arr.dtype),
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
        "mean": float(np.nanmean(arr)),
        "sum": float(np.nansum(arr)),
    }

print("\nSample PCC maps:")
for fp in pcc_maps[:5]:
    print(inspect_npy(fp))

print("\nSample Model A maps:")
for fp in modela_maps[:5]:
    print(inspect_npy(fp))

In [ ]:
def extract_case_id_from_pcc(name):
    # PatientID_0053_T1_to_T3_t1c_pcc_on_modelA_oof_teacher.npy
    return name.replace("_pcc_on_modelA_oof_teacher.npy", "")

def extract_case_id_from_modela(name):
    # PatientID_0035_T1_to_T2_t1c_direct_target_student_fold_3.npy
    return re.sub(r"_direct_target_student_fold_\d+\.npy$", "", name)

pcc_case_to_file = {extract_case_id_from_pcc(fp.name): fp for fp in pcc_maps}
modela_case_to_file = {extract_case_id_from_modela(fp.name): fp for fp in modela_maps}

pcc_cases = set(pcc_case_to_file.keys())
modela_cases = set(modela_case_to_file.keys())

common_cases = sorted(pcc_cases & modela_cases)
only_pcc = sorted(pcc_cases - modela_cases)
only_modela = sorted(modela_cases - pcc_cases)

print("PCC cases:", len(pcc_cases))
print("Model A cases:", len(modela_cases))
print("Common cases:", len(common_cases))
print("Only PCC:", len(only_pcc))
print("Only Model A:", len(only_modela))

print("\nFirst 10 common cases:")
for c in common_cases[:10]:
    print(c)

if only_pcc:
    print("\nOnly PCC examples:", only_pcc[:10])
if only_modela:
    print("\nOnly Model A examples:", only_modela[:10])

In [ ]:
from pathlib import Path
import os, re, json
import pandas as pd
import numpy as np

ROOT = Path("/kaggle/input/datasets")

search_terms = [
    "model_baseline_map_for_pcc",
    "fixed baseline",
    "naive self-tightening",
    "pcc correction",
    "0.542112",
    "0.670115",
    "0.513793",
    "0.788182",
    "0.381554",
    "0.400270",
    "shuffled-target",
    "difference-map",
    "diff_raw",
    "diff_excl_mask",
    "layer2",
    "future-change",
    "future_change",
]

candidate_files = []

text_exts = {".csv", ".json", ".txt", ".md", ".log"}
map_exts = {".nii", ".gz", ".npy", ".npz"}

for fp in ROOT.rglob("*"):
    if not fp.is_file():
        continue
    
    low_path = str(fp).lower()
    low_name = fp.name.lower()
    
    # file/path name hit
    if any(term.replace(" ", "_").lower() in low_path or term.lower() in low_path for term in search_terms):
        candidate_files.append(("PATH_HIT", fp))
        continue
    
    # content hit for readable files
    if fp.suffix.lower() in text_exts:
        try:
            txt = fp.read_text(errors="ignore")[:300000]
            low_txt = txt.lower()
            if any(term.lower() in low_txt for term in search_terms):
                candidate_files.append(("CONTENT_HIT", fp))
        except Exception:
            pass

print("Candidate files:", len(candidate_files))

for kind, fp in candidate_files[:300]:
    print(kind, fp)

if len(candidate_files) > 300:
    print("... truncated:", len(candidate_files) - 300)

In [ ]:
from collections import Counter, defaultdict

dir_counter = Counter()
dir_examples = defaultdict(list)

for kind, fp in candidate_files:
    parent = fp.parent
    dir_counter[parent] += 1
    if len(dir_examples[parent]) < 5:
        dir_examples[parent].append(fp.name)

print("Top candidate directories:")
for d, n in dir_counter.most_common(30):
    print("\nCOUNT:", n)
    print("DIR:", d)
    print("EXAMPLES:", dir_examples[d])

# Layer 2R: Publication-grade Rebuild of Retrospective Longitudinal PCC Correction with EIA Baselines

This experiment rebuilds the second-layer retrospective longitudinal PCC correction experiment in a fully traceable publication-grade pipeline.

Reason for rebuild:
The original Layer 2 aggregate results were valid and internally recorded, but the complete per-case output maps from that run were not preserved. Therefore, this rebuild regenerates all baseline maps, PCC maps, EIA maps, metrics, and visual outputs in one locked pipeline.

Locked cohort:
- Dataset: MU-Glioma-Post
- Modality: T1c
- Cohort: 40 usable longitudinal pairs
- Target: future tumour mask AND NOT current tumour mask
- Input to baseline model: current T1c MRI + current tumour mask
- Baseline target: future-change target

Methods:
1. Fixed baseline
2. Naive self-tightening
3. PCC correction
4. EIA-linear
5. EIA-blend-0.75
6. EIA-morph

Primary comparisons:
- PCC vs Fixed baseline
- PCC vs EIA-linear

Secondary comparisons:
- PCC vs Naive self-tightening
- PCC vs EIA-blend-0.75
- PCC vs EIA-morph

Main metrics:
- Dice
- IoU
- Target focus
- Log10 target-to-non-target ratio

All case-level maps, metrics, summaries, logs, and protocol metadata will be saved.

In [ ]:
from pathlib import Path
import os, json, glob, re, math, random, time
import numpy as np
import pandas as pd

RUN_NAME = "Layer2R_publication_rebuild_EIA_v1"

OUT_DIR = Path("/kaggle/working") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIRS = {
    "maps": OUT_DIR / "maps",
    "figures": OUT_DIR / "figures",
    "tables": OUT_DIR / "tables",
    "logs": OUT_DIR / "logs",
    "checkpoints": OUT_DIR / "checkpoints",
    "case_outputs": OUT_DIR / "case_outputs",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Run name:", RUN_NAME)
print("Output directory:", OUT_DIR)
for k, v in DIRS.items():
    print(f"{k}: {v}")

In [ ]:
from pathlib import Path
import pandas as pd
import json

ROOT = Path("/kaggle/input/datasets")
MODELA_DIR = ROOT / "jeechangxin/model-a-backup-final"
RAW_DIR = ROOT / "stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post"

case_csv = MODELA_DIR / "direct_target_case_metrics.csv"

df_cases = pd.read_csv(case_csv)
locked_cases = sorted(df_cases["case_id"].unique().tolist())

print("Locked cases:", len(locked_cases))
for c in locked_cases[:10]:
    print(c)

locked_case_df = pd.DataFrame({"case_id": locked_cases})
locked_case_df.to_csv(OUT_DIR / "tables" / "locked_40_cases.csv", index=False)

protocol = {
    "run_name": RUN_NAME,
    "dataset": "MU-Glioma-Post",
    "modality": "T1c",
    "locked_cases_n": len(locked_cases),
    "target_definition": "future tumour mask AND NOT current tumour mask",
    "methods": [
        "fixed_baseline",
        "naive_self_tightening",
        "pcc_correction",
        "eia_linear",
        "eia_blend_075",
        "eia_morph"
    ],
    "primary_comparisons": [
        "PCC vs Fixed baseline",
        "PCC vs EIA-linear"
    ],
    "metrics": [
        "Dice",
        "IoU",
        "target_focus",
        "log10_ratio"
    ]
}

with open(OUT_DIR / "protocol.json", "w") as f:
    json.dump(protocol, f, indent=2)

print("Saved locked case list and protocol.")

In [ ]:
import re
from pathlib import Path

def parse_case_id(case_id):
    """
    Example:
    PatientID_0003_T1_to_T2_t1c
    """
    m = re.match(r"(PatientID_\d+)_T(\d+)_to_T(\d+)_(\w+)", case_id)
    if not m:
        raise ValueError(f"Cannot parse case_id: {case_id}")
    patient_id = m.group(1)
    cur_tp = int(m.group(2))
    fut_tp = int(m.group(3))
    modality = m.group(4)
    return patient_id, cur_tp, fut_tp, modality

def get_raw_paths(case_id):
    patient_id, cur_tp, fut_tp, modality = parse_case_id(case_id)
    
    patient_dir = RAW_DIR / patient_id
    cur_dir = patient_dir / f"Timepoint_{cur_tp}"
    fut_dir = patient_dir / f"Timepoint_{fut_tp}"
    
    cur_img = cur_dir / f"{patient_id}_Timepoint_{cur_tp}_brain_{modality}.nii"
    fut_img = fut_dir / f"{patient_id}_Timepoint_{fut_tp}_brain_{modality}.nii"
    cur_mask = cur_dir / f"{patient_id}_Timepoint_{cur_tp}_tumorMask.nii"
    fut_mask = fut_dir / f"{patient_id}_Timepoint_{fut_tp}_tumorMask.nii"
    
    return {
        "case_id": case_id,
        "patient_id": patient_id,
        "cur_tp": cur_tp,
        "fut_tp": fut_tp,
        "modality": modality,
        "cur_img": cur_img,
        "fut_img": fut_img,
        "cur_mask": cur_mask,
        "fut_mask": fut_mask,
    }

rows = []
missing = []

for case_id in locked_cases:
    paths = get_raw_paths(case_id)
    row = {"case_id": case_id}
    ok = True
    for k in ["cur_img", "fut_img", "cur_mask", "fut_mask"]:
        exists = paths[k].exists()
        row[k] = str(paths[k])
        row[k + "_exists"] = exists
        if not exists:
            ok = False
    row["all_exists"] = ok
    rows.append(row)
    if not ok:
        missing.append(case_id)

manifest_df = pd.DataFrame(rows)
manifest_df.to_csv(OUT_DIR / "tables" / "Layer2R_raw_file_manifest.csv", index=False)

print("Total cases:", len(manifest_df))
print("All files exist cases:", manifest_df["all_exists"].sum())
print("Missing cases:", len(missing))

if missing:
    print("Missing examples:")
    for c in missing[:20]:
        print(c)

display(manifest_df.head())

In [ ]:
# ============================================================
# Layer2R Smoke Test 01:
# Load one longitudinal case, construct future-change target,
# save arrays and visualization.
# ============================================================

from pathlib import Path
import json, math, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import nibabel as nib
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "nibabel is not installed. In Kaggle, enable internet or add nibabel if needed."
    )

# -----------------------------
# Utility functions
# -----------------------------

def load_nii_raw(path):
    path = Path(path)
    img = nib.load(str(path))
    arr = img.get_fdata(dtype=np.float32)
    return arr, img.affine, img.header

def to_zhw(arr):
    """
    Convert common NIfTI shape H,W,Z into Z,H,W.
    If already Z,H,W, keep unchanged.
    This is important because previous prediction maps often used (Z, H, W).
    """
    arr = np.asarray(arr)

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D array, got shape {arr.shape}")

    # Common raw MRI shape is often (240, 240, 155).
    # Convert it to (155, 240, 240).
    if arr.shape[0] == arr.shape[1] and arr.shape[-1] < arr.shape[0]:
        arr = np.transpose(arr, (2, 0, 1))

    return arr.astype(np.float32)

def load_nii_zhw(path):
    arr, affine, header = load_nii_raw(path)
    arr_zhw = to_zhw(arr)
    return arr_zhw, affine, header, arr.shape

def robust_normalize(img, brain_mask=None, p_low=1, p_high=99, eps=1e-6):
    img = img.astype(np.float32)

    if brain_mask is None:
        brain_mask = img != 0

    vals = img[brain_mask]
    if vals.size < 10:
        return np.zeros_like(img, dtype=np.float32)

    lo, hi = np.percentile(vals, [p_low, p_high])
    if hi <= lo + eps:
        return np.zeros_like(img, dtype=np.float32)

    out = np.clip(img, lo, hi)
    out = (out - lo) / (hi - lo + eps)
    out[~brain_mask] = 0
    return out.astype(np.float32)

def dice_score(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)
    inter = np.logical_and(pred, target).sum()
    denom = pred.sum() + target.sum()
    return float((2 * inter + eps) / (denom + eps))

def iou_score(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)
    inter = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()
    return float((inter + eps) / (union + eps))

def choose_target_slice(target):
    per_slice = target.sum(axis=(1, 2))
    if per_slice.max() > 0:
        return int(np.argmax(per_slice))
    return int(target.shape[0] // 2)

def overlay_mask(ax, image, mask, title):
    ax.imshow(image, cmap="gray")
    ax.imshow(np.ma.masked_where(mask == 0, mask), alpha=0.45)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

# -----------------------------
# Choose one smoke-test case
# -----------------------------

CASE_ID = "PatientID_0003_T1_to_T2_t1c"

paths = get_raw_paths(CASE_ID)

print("Smoke-test case:", CASE_ID)
for k in ["cur_img", "fut_img", "cur_mask", "fut_mask"]:
    print(k, "->", paths[k], "| exists:", paths[k].exists())

# -----------------------------
# Load raw data
# -----------------------------

cur_img, cur_affine, cur_header, cur_raw_shape = load_nii_zhw(paths["cur_img"])
fut_img, fut_affine, fut_header, fut_raw_shape = load_nii_zhw(paths["fut_img"])
cur_mask, _, _, cur_mask_raw_shape = load_nii_zhw(paths["cur_mask"])
fut_mask, _, _, fut_mask_raw_shape = load_nii_zhw(paths["fut_mask"])

cur_mask = cur_mask > 0
fut_mask = fut_mask > 0

print("\nRaw shapes:")
print("cur_img raw:", cur_raw_shape)
print("fut_img raw:", fut_raw_shape)
print("cur_mask raw:", cur_mask_raw_shape)
print("fut_mask raw:", fut_mask_raw_shape)

print("\nZHW shapes:")
print("cur_img:", cur_img.shape)
print("fut_img:", fut_img.shape)
print("cur_mask:", cur_mask.shape)
print("fut_mask:", fut_mask.shape)

assert cur_img.shape == fut_img.shape == cur_mask.shape == fut_mask.shape, \
    "Shape mismatch after ZHW conversion."

# -----------------------------
# Construct future-change target
# -----------------------------

future_change_target = np.logical_and(fut_mask, np.logical_not(cur_mask))

brain_mask = np.logical_or(cur_img != 0, fut_img != 0)

cur_img_norm = robust_normalize(cur_img, brain_mask=brain_mask)
fut_img_norm = robust_normalize(fut_img, brain_mask=brain_mask)

z = choose_target_slice(future_change_target)

stats = {
    "case_id": CASE_ID,
    "cur_raw_shape": list(cur_raw_shape),
    "fut_raw_shape": list(fut_raw_shape),
    "zhw_shape": list(cur_img.shape),
    "cur_mask_voxels": int(cur_mask.sum()),
    "fut_mask_voxels": int(fut_mask.sum()),
    "future_change_target_voxels": int(future_change_target.sum()),
    "brain_voxels": int(brain_mask.sum()),
    "target_slice": int(z),
    "target_voxels_on_slice": int(future_change_target[z].sum()),
}

print("\nSmoke-test stats:")
for k, v in stats.items():
    print(f"{k}: {v}")

# -----------------------------
# Save case smoke-test outputs
# -----------------------------

case_out_dir = DIRS["case_outputs"] / CASE_ID
case_out_dir.mkdir(parents=True, exist_ok=True)

np.save(case_out_dir / "current_t1c_norm_zhw.npy", cur_img_norm.astype(np.float32))
np.save(case_out_dir / "future_t1c_norm_zhw.npy", fut_img_norm.astype(np.float32))
np.save(case_out_dir / "current_mask_zhw.npy", cur_mask.astype(np.uint8))
np.save(case_out_dir / "future_mask_zhw.npy", fut_mask.astype(np.uint8))
np.save(case_out_dir / "future_change_target_zhw.npy", future_change_target.astype(np.uint8))

with open(case_out_dir / "smoke_test_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

# -----------------------------
# Visualization
# -----------------------------

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(cur_img_norm[z], cmap="gray")
axes[0].set_title(f"Current T1c\nslice {z}", fontsize=10)
axes[0].axis("off")

axes[1].imshow(fut_img_norm[z], cmap="gray")
axes[1].set_title(f"Future T1c\nslice {z}", fontsize=10)
axes[1].axis("off")

overlay_mask(axes[2], cur_img_norm[z], cur_mask[z], "Current mask")
overlay_mask(axes[3], fut_img_norm[z], fut_mask[z], "Future mask")
overlay_mask(axes[4], fut_img_norm[z], future_change_target[z], "Future-change target")

plt.tight_layout()

fig_path = DIRS["figures"] / f"smoke_target_check_{CASE_ID}.png"
plt.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()

print("\nSaved smoke-test case outputs to:", case_out_dir)
print("Saved figure to:", fig_path)

# -----------------------------
# Smoke-test gate
# -----------------------------

if future_change_target.sum() <= 0:
    print("\nSMOKE TEST WARNING: future-change target is empty.")
else:
    print("\nSMOKE TEST PASS: target constructed and saved.")

In [ ]:
# ============================================================
# Layer2R Smoke Test 02:
# Train a case-specific Mini U-Net baseline for one case.
# Input: current T1c + current tumour mask
# Target: future-change target
# Output: baseline_prob_map.npy + baseline metrics + visualization
# ============================================================

from pathlib import Path
import json, random, time, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# Reproducibility
# -----------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# Case setup
# -----------------------------

CASE_ID = "PatientID_0003_T1_to_T2_t1c"
case_out_dir = DIRS["case_outputs"] / CASE_ID
case_out_dir.mkdir(parents=True, exist_ok=True)

cur_img = np.load(case_out_dir / "current_t1c_norm_zhw.npy").astype(np.float32)
cur_mask = np.load(case_out_dir / "current_mask_zhw.npy").astype(np.float32)
target = np.load(case_out_dir / "future_change_target_zhw.npy").astype(np.float32)

print("Loaded arrays:")
print("cur_img:", cur_img.shape, cur_img.dtype, cur_img.min(), cur_img.max())
print("cur_mask:", cur_mask.shape, cur_mask.dtype, cur_mask.min(), cur_mask.max())
print("target:", target.shape, target.dtype, target.min(), target.max(), "voxels:", int(target.sum()))

assert cur_img.shape == cur_mask.shape == target.shape

# -----------------------------
# Metrics
# -----------------------------

def dice_binary(pred, gt, eps=1e-6):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return float((2 * inter + eps) / (denom + eps))

def iou_binary(pred, gt, eps=1e-6):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return float((inter + eps) / (union + eps))

def target_focus(prob, gt, eps=1e-8):
    prob = prob.astype(np.float32)
    gt = gt.astype(bool)
    total = float(prob.sum())
    inside = float(prob[gt].sum())
    return inside / (total + eps)

def log10_ratio(prob, gt, eps=1e-8):
    prob = prob.astype(np.float32)
    gt = gt.astype(bool)
    inside_mean = float(prob[gt].mean()) if gt.sum() > 0 else 0.0
    outside_mean = float(prob[~gt].mean()) if (~gt).sum() > 0 else 0.0
    return float(np.log10((inside_mean + eps) / (outside_mean + eps)))

def topk_mask(prob, k):
    prob = prob.astype(np.float32)
    flat = prob.reshape(-1)
    k = int(k)
    k = max(1, min(k, flat.size))
    idx = np.argpartition(flat, -k)[-k:]
    out = np.zeros_like(flat, dtype=np.uint8)
    out[idx] = 1
    return out.reshape(prob.shape).astype(bool)

def eval_prob_map(prob, gt, threshold=0.5):
    gt_bool = gt.astype(bool)

    pred_fixed = prob >= threshold
    k = int(gt_bool.sum())
    pred_topk = topk_mask(prob, k)

    return {
        "dice_fixed05": dice_binary(pred_fixed, gt_bool),
        "iou_fixed05": iou_binary(pred_fixed, gt_bool),
        "dice_topk": dice_binary(pred_topk, gt_bool),
        "iou_topk": iou_binary(pred_topk, gt_bool),
        "target_focus": target_focus(prob, gt_bool),
        "log10_ratio": log10_ratio(prob, gt_bool),
        "pred_fixed05_voxels": int(pred_fixed.sum()),
        "pred_topk_voxels": int(pred_topk.sum()),
        "target_voxels": int(gt_bool.sum()),
    }

# -----------------------------
# Dataset
# -----------------------------

class SliceDataset(Dataset):
    def __init__(self, cur_img, cur_mask, target):
        self.cur_img = cur_img.astype(np.float32)
        self.cur_mask = cur_mask.astype(np.float32)
        self.target = target.astype(np.float32)

        # Use all slices for smoke test.
        # This is a retrospective case-specific training smoke test.
        self.indices = np.arange(cur_img.shape[0])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        z = self.indices[idx]
        x = np.stack([self.cur_img[z], self.cur_mask[z]], axis=0).astype(np.float32)
        y = self.target[z][None, :, :].astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y), int(z)

dataset = SliceDataset(cur_img, cur_mask, target)
loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)

print("Training slices:", len(dataset))

# -----------------------------
# Mini U-Net
# -----------------------------

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class MiniUNet(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)

# -----------------------------
# Loss
# -----------------------------

def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dims)
    denom = torch.sum(probs, dims) + torch.sum(targets, dims)
    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()

pos = float(target.sum())
neg = float(target.size - target.sum())
pos_weight_value = min(80.0, max(1.0, neg / max(pos, 1.0)))
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

print("Positive voxels:", int(pos))
print("Negative voxels:", int(neg))
print("BCE pos_weight:", float(pos_weight_value))

model = MiniUNet(in_ch=2, out_ch=1, base=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def combined_loss(logits, y):
    return bce_loss(logits, y) + soft_dice_loss_from_logits(logits, y)

# -----------------------------
# Prediction function
# -----------------------------

@torch.no_grad()
def predict_full_volume(model, cur_img, cur_mask):
    model.eval()
    preds = np.zeros_like(cur_img, dtype=np.float32)

    for z in range(cur_img.shape[0]):
        x = np.stack([cur_img[z], cur_mask[z]], axis=0)[None].astype(np.float32)
        x_t = torch.from_numpy(x).to(device)
        logits = model(x_t)
        prob = torch.sigmoid(logits).cpu().numpy()[0, 0]
        preds[z] = prob.astype(np.float32)

    return preds

# -----------------------------
# Train smoke baseline
# -----------------------------

SMOKE_EPOCHS = 5
best_dice_topk = -1.0
best_state = None
history = []

start_time = time.time()

for epoch in range(1, SMOKE_EPOCHS + 1):
    model.train()
    epoch_losses = []

    for x, y, z_idx in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = combined_loss(logits, y)
        loss.backward()
        optimizer.step()

        epoch_losses.append(float(loss.item()))

    # Evaluate on same case for smoke test
    prob_map = predict_full_volume(model, cur_img, cur_mask)
    metrics = eval_prob_map(prob_map, target, threshold=0.5)

    row = {
        "epoch": epoch,
        "loss": float(np.mean(epoch_losses)),
        **metrics
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"loss={row['loss']:.4f} | "
        f"dice_fixed05={row['dice_fixed05']:.4f} | "
        f"dice_topk={row['dice_topk']:.4f} | "
        f"focus={row['target_focus']:.4f}"
    )

    if row["dice_topk"] > best_dice_topk:
        best_dice_topk = row["dice_topk"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

elapsed = time.time() - start_time

# Restore best model by top-k Dice for smoke test
model.load_state_dict(best_state)
baseline_prob = predict_full_volume(model, cur_img, cur_mask)
baseline_metrics = eval_prob_map(baseline_prob, target, threshold=0.5)

# -----------------------------
# Save outputs
# -----------------------------

baseline_map_path = case_out_dir / "baseline_prob_map_smoke.npy"
baseline_fixed_path = case_out_dir / "baseline_fixed05_binary_smoke.npy"
baseline_topk_path = case_out_dir / "baseline_topk_binary_smoke.npy"
ckpt_path = DIRS["checkpoints"] / f"{CASE_ID}_baseline_smoke_best.pt"

np.save(baseline_map_path, baseline_prob.astype(np.float32))
np.save(baseline_fixed_path, (baseline_prob >= 0.5).astype(np.uint8))
np.save(baseline_topk_path, topk_mask(baseline_prob, int(target.sum())).astype(np.uint8))

torch.save(
    {
        "case_id": CASE_ID,
        "model_state_dict": model.state_dict(),
        "smoke_epochs": SMOKE_EPOCHS,
        "best_dice_topk": float(best_dice_topk),
        "baseline_metrics": baseline_metrics,
        "history": history,
    },
    ckpt_path
)

history_df = pd.DataFrame(history)
history_df.to_csv(case_out_dir / "baseline_smoke_training_history.csv", index=False)

with open(case_out_dir / "baseline_smoke_metrics.json", "w") as f:
    json.dump(
        {
            "case_id": CASE_ID,
            "elapsed_sec": elapsed,
            "smoke_epochs": SMOKE_EPOCHS,
            "best_dice_topk": float(best_dice_topk),
            "baseline_metrics": baseline_metrics,
            "baseline_map_path": str(baseline_map_path),
            "checkpoint_path": str(ckpt_path),
        },
        f,
        indent=2
    )

print("\nFinal baseline smoke metrics:")
for k, v in baseline_metrics.items():
    print(f"{k}: {v}")

print("\nSaved baseline probability map:", baseline_map_path)
print("Saved baseline checkpoint:", ckpt_path)
print("Elapsed seconds:", round(elapsed, 2))

# -----------------------------
# Visualization
# -----------------------------

z = choose_target_slice(target)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(cur_img[z], cmap="gray")
axes[0].set_title(f"Current T1c\nslice {z}")
axes[0].axis("off")

axes[1].imshow(target[z], cmap="gray")
axes[1].set_title("Future-change target")
axes[1].axis("off")

axes[2].imshow(baseline_prob[z], cmap="hot")
axes[2].set_title("Baseline probability")
axes[2].axis("off")

axes[3].imshow((baseline_prob[z] >= 0.5), cmap="gray")
axes[3].set_title("Baseline fixed 0.5")
axes[3].axis("off")

axes[4].imshow(topk_mask(baseline_prob, int(target.sum()))[z], cmap="gray")
axes[4].set_title("Baseline top-k")
axes[4].axis("off")

plt.tight_layout()
fig_path = DIRS["figures"] / f"baseline_smoke_{CASE_ID}.png"
plt.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()

print("Saved baseline smoke figure:", fig_path)

print("\nBASELINE SMOKE TEST PASS.")

In [ ]:
# ============================================================
# Layer2R Smoke Test 03:
# Run Fixed / Naive / PCC / EIA baselines on one baseline map.
# Input: baseline_prob_map_smoke.npy + future_change_target
# Output: corrected maps, metrics table, comparison figure
# ============================================================

from pathlib import Path
import json, math, os, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.ndimage import gaussian_filter, binary_fill_holes, binary_closing, label, distance_transform_edt
    SCIPY_OK = True
except Exception as e:
    SCIPY_OK = False
    print("WARNING: scipy not available:", repr(e))

# -----------------------------
# Rebuild DIRS if needed
# -----------------------------

try:
    DIRS
except NameError:
    OUT_DIR = Path("/kaggle/working/Layer2R_publication_rebuild_EIA_v1")
    DIRS = {
        "maps": OUT_DIR / "maps",
        "figures": OUT_DIR / "figures",
        "tables": OUT_DIR / "tables",
        "logs": OUT_DIR / "logs",
        "checkpoints": OUT_DIR / "checkpoints",
        "case_outputs": OUT_DIR / "case_outputs",
    }
    for d in DIRS.values():
        d.mkdir(parents=True, exist_ok=True)

CASE_ID = "PatientID_0003_T1_to_T2_t1c"
case_out_dir = DIRS["case_outputs"] / CASE_ID
case_out_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load arrays
# -----------------------------

cur_img = np.load(case_out_dir / "current_t1c_norm_zhw.npy").astype(np.float32)
target = np.load(case_out_dir / "future_change_target_zhw.npy").astype(np.uint8)
baseline = np.load(case_out_dir / "baseline_prob_map_smoke.npy").astype(np.float32)

target_bool = target.astype(bool)

print("Loaded:")
print("cur_img:", cur_img.shape, cur_img.dtype, cur_img.min(), cur_img.max())
print("target:", target.shape, target.dtype, "voxels:", int(target_bool.sum()))
print("baseline:", baseline.shape, baseline.dtype, baseline.min(), baseline.max(), baseline.mean())

assert cur_img.shape == target.shape == baseline.shape

# -----------------------------
# Metrics
# -----------------------------

def dice_binary(pred, gt, eps=1e-6):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return float((2 * inter + eps) / (denom + eps))

def iou_binary(pred, gt, eps=1e-6):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return float((inter + eps) / (union + eps))

def topk_mask(prob, k):
    prob = prob.astype(np.float32)
    flat = prob.reshape(-1)
    k = int(k)
    k = max(1, min(k, flat.size))
    idx = np.argpartition(flat, -k)[-k:]
    out = np.zeros_like(flat, dtype=np.uint8)
    out[idx] = 1
    return out.reshape(prob.shape).astype(bool)

def target_focus(prob, gt, eps=1e-8):
    prob = prob.astype(np.float32)
    gt = gt.astype(bool)
    total = float(prob.sum())
    inside = float(prob[gt].sum())
    return float(inside / (total + eps))

def log10_ratio(prob, gt, eps=1e-8):
    prob = prob.astype(np.float32)
    gt = gt.astype(bool)
    inside_mean = float(prob[gt].mean()) if gt.sum() > 0 else 0.0
    outside_mean = float(prob[~gt].mean()) if (~gt).sum() > 0 else 0.0
    return float(np.log10((inside_mean + eps) / (outside_mean + eps)))

def eval_prob_map(prob, gt, threshold=0.5, main_mode="topk"):
    prob = np.nan_to_num(prob.astype(np.float32), nan=0.0, posinf=1.0, neginf=0.0)
    prob = np.clip(prob, 0, 1)
    gt_bool = gt.astype(bool)

    pred_fixed = prob >= threshold
    k = int(gt_bool.sum())
    pred_topk = topk_mask(prob, k)

    out = {
        "dice_fixed05": dice_binary(pred_fixed, gt_bool),
        "iou_fixed05": iou_binary(pred_fixed, gt_bool),
        "dice_topk": dice_binary(pred_topk, gt_bool),
        "iou_topk": iou_binary(pred_topk, gt_bool),
        "target_focus": target_focus(prob, gt_bool),
        "log10_ratio": log10_ratio(prob, gt_bool),
        "pred_fixed05_voxels": int(pred_fixed.sum()),
        "pred_topk_voxels": int(pred_topk.sum()),
        "target_voxels": int(gt_bool.sum()),
        "prob_min": float(prob.min()),
        "prob_max": float(prob.max()),
        "prob_mean": float(prob.mean()),
        "prob_sum": float(prob.sum()),
    }

    if main_mode == "topk":
        out["dice"] = out["dice_topk"]
        out["iou"] = out["iou_topk"]
        out["main_mode"] = "topk"
    else:
        out["dice"] = out["dice_fixed05"]
        out["iou"] = out["iou_fixed05"]
        out["main_mode"] = "fixed05"

    return out

def choose_target_slice(target):
    per_slice = target.astype(bool).sum(axis=(1, 2))
    if per_slice.max() > 0:
        return int(np.argmax(per_slice))
    return int(target.shape[0] // 2)

# -----------------------------
# Helper transforms
# -----------------------------

def safe_clip_prob(x):
    return np.clip(np.nan_to_num(x.astype(np.float32), nan=0.0, posinf=1.0, neginf=0.0), 0, 1)

def safe_logit(p, eps=1e-5):
    p = np.clip(p.astype(np.float32), eps, 1 - eps)
    return np.log(p / (1 - p)).astype(np.float32)

def sigmoid(x):
    x = np.clip(x.astype(np.float32), -30, 30)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)

def normalize01(x, eps=1e-8):
    x = x.astype(np.float32)
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx <= mn + eps:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn + eps)).astype(np.float32)

def make_dilated_region(mask_bool, radius=26):
    mask_bool = mask_bool.astype(bool)
    if SCIPY_OK:
        # distance to target <= radius gives a dilation-like support.
        dist = distance_transform_edt(~mask_bool)
        return (dist <= radius)
    else:
        # fallback: no dilation
        return mask_bool.copy()

def smooth_mask(mask_bool, sigma=2.0):
    x = mask_bool.astype(np.float32)
    if SCIPY_OK:
        x = gaussian_filter(x, sigma=sigma)
    return normalize01(x)

# -----------------------------
# Parameters
# -----------------------------

DILATION_RADIUS = 26
SIGMA = 2.0

PCC_ROUNDS = 10
PCC_ETA = 0.30

EIA_ALPHA = 0.30
EIA_BETA = 0.30
EIA_BLEND_LAMBDA = 0.75

# -----------------------------
# Shared target-derived signals
# -----------------------------

R = make_dilated_region(target_bool, radius=DILATION_RADIUS)
S = smooth_mask(target_bool, sigma=SIGMA)

print("\nSupport/target signal:")
print("target voxels:", int(target_bool.sum()))
print("dilated support voxels:", int(R.sum()))
print("smoothed target min/max/mean:", float(S.min()), float(S.max()), float(S.mean()))

# -----------------------------
# Method 1: Fixed baseline
# -----------------------------

fixed_baseline = safe_clip_prob(baseline)

# -----------------------------
# Method 2: Naive self-tightening
# No target access.
# Sharpen baseline probabilities using logit scaling.
# -----------------------------

NAIVE_GAMMA = 2.5
naive = sigmoid(NAIVE_GAMMA * safe_logit(fixed_baseline))

# -----------------------------
# Method 3: EIA-linear
# Equal-information-access one-step target-guided correction.
# Same target-derived S and R, but no iterative PCC.
# -----------------------------

eia_linear = safe_clip_prob(
    fixed_baseline
    + EIA_ALPHA * S * (1.0 - fixed_baseline)
    - EIA_BETA * (~R).astype(np.float32) * fixed_baseline
)

# -----------------------------
# Method 4: EIA-blend-0.75
# Strong target-guided blending control.
# -----------------------------

eia_blend075 = safe_clip_prob(
    EIA_BLEND_LAMBDA * fixed_baseline
    + (1.0 - EIA_BLEND_LAMBDA) * S
)

# -----------------------------
# Method 5: EIA-morph
# Morphology-only target-region correction.
# -----------------------------

baseline_binary = fixed_baseline >= 0.5
morph = np.logical_and(baseline_binary, R)

if SCIPY_OK:
    morph = binary_closing(morph, iterations=1)
    morph = binary_fill_holes(morph)

    # remove tiny components
    lab, n_lab = label(morph)
    min_size = 20
    keep = np.zeros_like(morph, dtype=bool)
    for lab_id in range(1, n_lab + 1):
        comp = lab == lab_id
        if comp.sum() >= min_size:
            keep |= comp
    morph = keep

eia_morph = morph.astype(np.float32)

# -----------------------------
# Method 6: PCC iterative correction
# Iterative prediction-comparison-correction in logit space.
# Uses the same target-derived signal as EIA, but performs
# repeated correction based on residual target-vs-current-map error.
# -----------------------------

pcc = fixed_baseline.copy()
pcc_history = []

for r in range(1, PCC_ROUNDS + 1):
    p = safe_clip_prob(pcc)

    # Residual comparison signal: where target and current map disagree.
    # Within support, target-prediction drives reinforcement/suppression.
    residual = (target_bool.astype(np.float32) - p) * R.astype(np.float32)

    if SCIPY_OK:
        residual_smooth = gaussian_filter(residual, sigma=SIGMA)
    else:
        residual_smooth = residual

    # Background suppression outside target support.
    background_suppression = (~R).astype(np.float32) * p

    logits = safe_logit(p)
    logits = logits + PCC_ETA * residual_smooth - PCC_ETA * background_suppression

    pcc = safe_clip_prob(sigmoid(logits))

    pcc_metrics_r = eval_prob_map(pcc, target_bool, threshold=0.5, main_mode="topk")
    pcc_history.append({
        "round": r,
        "dice": pcc_metrics_r["dice"],
        "iou": pcc_metrics_r["iou"],
        "target_focus": pcc_metrics_r["target_focus"],
        "log10_ratio": pcc_metrics_r["log10_ratio"],
    })

pcc = safe_clip_prob(pcc)

# -----------------------------
# Evaluate all methods
# -----------------------------

method_maps = {
    "fixed_baseline": fixed_baseline,
    "naive_self_tightening": naive,
    "eia_linear": eia_linear,
    "eia_blend075": eia_blend075,
    "eia_morph": eia_morph,
    "pcc_correction": pcc,
}

rows = []

for method, prob in method_maps.items():
    metrics = eval_prob_map(prob, target_bool, threshold=0.5, main_mode="topk")
    row = {
        "case_id": CASE_ID,
        "method": method,
        **metrics
    }
    rows.append(row)

metrics_df = pd.DataFrame(rows)

# sort display order
order = {
    "fixed_baseline": 1,
    "naive_self_tightening": 2,
    "eia_linear": 3,
    "eia_blend075": 4,
    "eia_morph": 5,
    "pcc_correction": 6,
}
metrics_df["order"] = metrics_df["method"].map(order)
metrics_df = metrics_df.sort_values("order").drop(columns=["order"])

print("\nSmoke-test method metrics:")
display(metrics_df[[
    "method",
    "dice",
    "iou",
    "dice_fixed05",
    "iou_fixed05",
    "dice_topk",
    "iou_topk",
    "target_focus",
    "log10_ratio",
    "pred_fixed05_voxels",
    "target_voxels",
    "prob_mean",
]])

# -----------------------------
# Paired-style comparison for this one case
# -----------------------------

pcc_row = metrics_df[metrics_df["method"] == "pcc_correction"].iloc[0].to_dict()

comparison_rows = []
for method in ["fixed_baseline", "naive_self_tightening", "eia_linear", "eia_blend075", "eia_morph"]:
    row = metrics_df[metrics_df["method"] == method].iloc[0].to_dict()
    comparison_rows.append({
        "case_id": CASE_ID,
        "comparison": f"PCC vs {method}",
        "dice_diff": pcc_row["dice"] - row["dice"],
        "iou_diff": pcc_row["iou"] - row["iou"],
        "target_focus_diff": pcc_row["target_focus"] - row["target_focus"],
        "log10_ratio_diff": pcc_row["log10_ratio"] - row["log10_ratio"],
        "pcc_better_dice": pcc_row["dice"] > row["dice"],
        "pcc_better_iou": pcc_row["iou"] > row["iou"],
    })

comparison_df = pd.DataFrame(comparison_rows)

print("\nOne-case PCC comparisons:")
display(comparison_df)

# -----------------------------
# Save maps and tables
# -----------------------------

for method, prob in method_maps.items():
    np.save(case_out_dir / f"{method}_smoke.npy", safe_clip_prob(prob).astype(np.float32))

np.save(case_out_dir / "target_dilated_support_R_smoke.npy", R.astype(np.uint8))
np.save(case_out_dir / "target_smoothed_signal_S_smoke.npy", S.astype(np.float32))

metrics_path = case_out_dir / "Layer2R_smoke_PCC_EIA_metrics.csv"
comparison_path = case_out_dir / "Layer2R_smoke_PCC_EIA_comparisons.csv"
pcc_history_path = case_out_dir / "Layer2R_smoke_PCC_round_history.csv"

metrics_df.to_csv(metrics_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
pd.DataFrame(pcc_history).to_csv(pcc_history_path, index=False)

with open(case_out_dir / "Layer2R_smoke_PCC_EIA_protocol.json", "w") as f:
    json.dump(
        {
            "case_id": CASE_ID,
            "dilation_radius": DILATION_RADIUS,
            "sigma": SIGMA,
            "pcc_rounds": PCC_ROUNDS,
            "pcc_eta": PCC_ETA,
            "eia_alpha": EIA_ALPHA,
            "eia_beta": EIA_BETA,
            "eia_blend_lambda": EIA_BLEND_LAMBDA,
            "main_metric_mode": "topk",
            "threshold": 0.5,
            "note": "One-case smoke test only; not publication-level aggregate result."
        },
        f,
        indent=2
    )

print("\nSaved metrics:", metrics_path)
print("Saved comparisons:", comparison_path)
print("Saved PCC round history:", pcc_history_path)

# -----------------------------
# Visualization
# -----------------------------

z = choose_target_slice(target_bool)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

axes = axes.ravel()

vis_items = [
    ("Current T1c", cur_img[z], "gray"),
    ("Future-change target", target_bool[z].astype(float), "gray"),
    ("Fixed baseline", fixed_baseline[z], "hot"),
    ("Naive", naive[z], "hot"),
    ("EIA-linear", eia_linear[z], "hot"),
    ("EIA-blend-0.75", eia_blend075[z], "hot"),
    ("EIA-morph", eia_morph[z], "gray"),
    ("PCC correction", pcc[z], "hot"),
]

for ax, (title, img, cmap) in zip(axes, vis_items):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.tight_layout()
fig_path = DIRS["figures"] / f"Layer2R_smoke_PCC_EIA_{CASE_ID}.png"
plt.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()

print("Saved comparison figure:", fig_path)

# -----------------------------
# Gate
# -----------------------------

required_methods = set([
    "fixed_baseline",
    "naive_self_tightening",
    "eia_linear",
    "eia_blend075",
    "eia_morph",
    "pcc_correction",
])

if set(metrics_df["method"]) == required_methods:
    print("\nPCC + EIA SMOKE TEST PASS.")
else:
    print("\nPCC + EIA SMOKE TEST WARNING: method set incomplete.")

In [ ]:
# ============================================================
# Layer2R Pilot:
# Run 3-case baseline training + Fixed/Naive/PCC/EIA evaluation.
# This is still a pilot, not final 40-case publication run.
# ============================================================

from pathlib import Path
import json, time, random, math, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from scipy.ndimage import gaussian_filter, binary_fill_holes, binary_closing, label, distance_transform_edt
    SCIPY_OK = True
except Exception as e:
    SCIPY_OK = False
    print("WARNING: scipy not available:", repr(e))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PILOT_CASES = [
    "PatientID_0003_T1_to_T2_t1c",
    "PatientID_0005_T3_to_T4_t1c",
    "PatientID_0008_T4_to_T6_t1c",
]

PILOT_EPOCHS = 5
BATCH_SIZE = 8
LR = 1e-3

# PCC / EIA parameters
DILATION_RADIUS = 26
SIGMA = 2.0
PCC_ROUNDS = 10
PCC_ETA = 0.30
EIA_ALPHA = 0.30
EIA_BETA = 0.30
EIA_BLEND_LAMBDA = 0.75
NAIVE_GAMMA = 2.5

pilot_out_dir = OUT_DIR / "pilot_3case"
pilot_out_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Dataset class
# -----------------------------

class SliceDataset(Dataset):
    def __init__(self, cur_img, cur_mask, target):
        self.cur_img = cur_img.astype(np.float32)
        self.cur_mask = cur_mask.astype(np.float32)
        self.target = target.astype(np.float32)
        self.indices = np.arange(cur_img.shape[0])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        z = self.indices[idx]
        x = np.stack([self.cur_img[z], self.cur_mask[z]], axis=0).astype(np.float32)
        y = self.target[z][None, :, :].astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y), int(z)

# -----------------------------
# Model / loss helpers
# -----------------------------

def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dims)
    denom = torch.sum(probs, dims) + torch.sum(targets, dims)
    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()

def train_case_baseline(case_id, cur_img, cur_mask, target, epochs=PILOT_EPOCHS):
    dataset = SliceDataset(cur_img, cur_mask, target)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    pos = float(target.sum())
    neg = float(target.size - target.sum())
    pos_weight_value = min(80.0, max(1.0, neg / max(pos, 1.0)))
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

    model = MiniUNet(in_ch=2, out_ch=1, base=16).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def combined_loss(logits, y):
        return bce_loss(logits, y) + soft_dice_loss_from_logits(logits, y)

    best_dice_topk = -1.0
    best_state = None
    history = []

    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []

        for x, y, _ in loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = combined_loss(logits, y)
            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

        prob_map = predict_full_volume(model, cur_img, cur_mask)
        m = eval_prob_map(prob_map, target, threshold=0.5, main_mode="topk")

        row = {
            "case_id": case_id,
            "epoch": epoch,
            "loss": float(np.mean(losses)),
            **m
        }
        history.append(row)

        print(
            f"{case_id} | Epoch {epoch:02d} | "
            f"loss={row['loss']:.4f} | "
            f"dice_topk={row['dice_topk']:.4f} | "
            f"dice_fixed05={row['dice_fixed05']:.4f}"
        )

        if row["dice_topk"] > best_dice_topk:
            best_dice_topk = row["dice_topk"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    baseline_prob = predict_full_volume(model, cur_img, cur_mask)

    elapsed = time.time() - start

    return baseline_prob.astype(np.float32), pd.DataFrame(history), float(best_dice_topk), elapsed

# -----------------------------
# Correction methods
# -----------------------------

def run_corrections_for_case(baseline, target):
    target_bool = target.astype(bool)

    fixed_baseline = safe_clip_prob(baseline)
    naive = sigmoid(NAIVE_GAMMA * safe_logit(fixed_baseline))

    R = make_dilated_region(target_bool, radius=DILATION_RADIUS)
    S = smooth_mask(target_bool, sigma=SIGMA)

    eia_linear = safe_clip_prob(
        fixed_baseline
        + EIA_ALPHA * S * (1.0 - fixed_baseline)
        - EIA_BETA * (~R).astype(np.float32) * fixed_baseline
    )

    eia_blend075 = safe_clip_prob(
        EIA_BLEND_LAMBDA * fixed_baseline
        + (1.0 - EIA_BLEND_LAMBDA) * S
    )

    baseline_binary = fixed_baseline >= 0.5
    morph = np.logical_and(baseline_binary, R)

    if SCIPY_OK:
        morph = binary_closing(morph, iterations=1)
        morph = binary_fill_holes(morph)
        lab, n_lab = label(morph)
        min_size = 20
        keep = np.zeros_like(morph, dtype=bool)
        for lab_id in range(1, n_lab + 1):
            comp = lab == lab_id
            if comp.sum() >= min_size:
                keep |= comp
        morph = keep

    eia_morph = morph.astype(np.float32)

    pcc = fixed_baseline.copy()
    for r in range(1, PCC_ROUNDS + 1):
        p = safe_clip_prob(pcc)
        residual = (target_bool.astype(np.float32) - p) * R.astype(np.float32)

        if SCIPY_OK:
            residual_smooth = gaussian_filter(residual, sigma=SIGMA)
        else:
            residual_smooth = residual

        background_suppression = (~R).astype(np.float32) * p

        logits = safe_logit(p)
        logits = logits + PCC_ETA * residual_smooth - PCC_ETA * background_suppression
        pcc = safe_clip_prob(sigmoid(logits))

    return {
        "fixed_baseline": fixed_baseline,
        "naive_self_tightening": naive,
        "eia_linear": eia_linear,
        "eia_blend075": eia_blend075,
        "eia_morph": eia_morph,
        "pcc_correction": pcc,
        "target_support_R": R.astype(np.uint8),
        "target_signal_S": S.astype(np.float32),
    }

# -----------------------------
# Main pilot loop
# -----------------------------

all_metric_rows = []
all_comparison_rows = []
all_training_rows = []

for case_id in PILOT_CASES:
    print("\n" + "="*100)
    print("Running pilot case:", case_id)
    print("="*100)

    case_out_dir = DIRS["case_outputs"] / case_id
    case_out_dir.mkdir(parents=True, exist_ok=True)

    # Load or create case arrays
    if not (case_out_dir / "current_t1c_norm_zhw.npy").exists():
        paths = get_raw_paths(case_id)

        cur_img_raw, _, _, _ = load_nii_zhw(paths["cur_img"])
        fut_img_raw, _, _, _ = load_nii_zhw(paths["fut_img"])
        cur_mask, _, _, _ = load_nii_zhw(paths["cur_mask"])
        fut_mask, _, _, _ = load_nii_zhw(paths["fut_mask"])

        cur_mask = cur_mask > 0
        fut_mask = fut_mask > 0
        target = np.logical_and(fut_mask, np.logical_not(cur_mask))
        brain_mask = np.logical_or(cur_img_raw != 0, fut_img_raw != 0)

        cur_img = robust_normalize(cur_img_raw, brain_mask=brain_mask)

        np.save(case_out_dir / "current_t1c_norm_zhw.npy", cur_img.astype(np.float32))
        np.save(case_out_dir / "current_mask_zhw.npy", cur_mask.astype(np.uint8))
        np.save(case_out_dir / "future_change_target_zhw.npy", target.astype(np.uint8))
    else:
        cur_img = np.load(case_out_dir / "current_t1c_norm_zhw.npy").astype(np.float32)
        cur_mask = np.load(case_out_dir / "current_mask_zhw.npy").astype(np.float32)
        target = np.load(case_out_dir / "future_change_target_zhw.npy").astype(np.uint8)

    print("cur_img:", cur_img.shape, "target voxels:", int(target.sum()))

    # Train baseline
    baseline_prob, hist_df, best_dice, elapsed = train_case_baseline(
        case_id, cur_img, cur_mask, target, epochs=PILOT_EPOCHS
    )

    hist_df["elapsed_sec_total"] = elapsed
    all_training_rows.append(hist_df)

    # Save baseline
    np.save(case_out_dir / "baseline_prob_map_pilot.npy", baseline_prob.astype(np.float32))
    hist_df.to_csv(case_out_dir / "baseline_training_history_pilot.csv", index=False)

    # Run methods
    maps = run_corrections_for_case(baseline_prob, target)

    for method, prob in maps.items():
        if method in ["target_support_R", "target_signal_S"]:
            continue

        np.save(case_out_dir / f"{method}_pilot.npy", safe_clip_prob(prob).astype(np.float32))

        m = eval_prob_map(prob, target.astype(bool), threshold=0.5, main_mode="topk")
        all_metric_rows.append({
            "case_id": case_id,
            "method": method,
            "baseline_best_dice_topk": best_dice,
            "baseline_training_elapsed_sec": elapsed,
            **m
        })

    # Comparisons
    tmp = pd.DataFrame([r for r in all_metric_rows if r["case_id"] == case_id])
    pcc_row = tmp[tmp["method"] == "pcc_correction"].iloc[0].to_dict()

    for method in ["fixed_baseline", "naive_self_tightening", "eia_linear", "eia_blend075", "eia_morph"]:
        row = tmp[tmp["method"] == method].iloc[0].to_dict()
        all_comparison_rows.append({
            "case_id": case_id,
            "comparison": f"PCC vs {method}",
            "dice_diff": pcc_row["dice"] - row["dice"],
            "iou_diff": pcc_row["iou"] - row["iou"],
            "target_focus_diff": pcc_row["target_focus"] - row["target_focus"],
            "log10_ratio_diff": pcc_row["log10_ratio"] - row["log10_ratio"],
            "pcc_better_dice": pcc_row["dice"] > row["dice"],
            "pcc_better_iou": pcc_row["iou"] > row["iou"],
        })

    # Figure
    z = choose_target_slice(target)

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.ravel()

    vis_items = [
        ("Current T1c", cur_img[z], "gray"),
        ("Target", target[z].astype(float), "gray"),
        ("Fixed baseline", maps["fixed_baseline"][z], "hot"),
        ("Naive", maps["naive_self_tightening"][z], "hot"),
        ("EIA-linear", maps["eia_linear"][z], "hot"),
        ("EIA-blend-0.75", maps["eia_blend075"][z], "hot"),
        ("EIA-morph", maps["eia_morph"][z], "gray"),
        ("PCC", maps["pcc_correction"][z], "hot"),
    ]

    for ax, (title, img, cmap) in zip(axes, vis_items):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    fig_path = DIRS["figures"] / f"Layer2R_pilot_{case_id}.png"
    plt.savefig(fig_path, dpi=160, bbox_inches="tight")
    plt.show()

    print("Saved figure:", fig_path)

# -----------------------------
# Save pilot summary
# -----------------------------

pilot_metrics_df = pd.DataFrame(all_metric_rows)
pilot_comparisons_df = pd.DataFrame(all_comparison_rows)
pilot_training_df = pd.concat(all_training_rows, ignore_index=True)

pilot_metrics_path = DIRS["tables"] / "Layer2R_3case_pilot_metrics.csv"
pilot_comparisons_path = DIRS["tables"] / "Layer2R_3case_pilot_comparisons.csv"
pilot_training_path = DIRS["tables"] / "Layer2R_3case_pilot_training_history.csv"

pilot_metrics_df.to_csv(pilot_metrics_path, index=False)
pilot_comparisons_df.to_csv(pilot_comparisons_path, index=False)
pilot_training_df.to_csv(pilot_training_path, index=False)

print("\nSaved pilot metrics:", pilot_metrics_path)
print("Saved pilot comparisons:", pilot_comparisons_path)
print("Saved pilot training:", pilot_training_path)

print("\nPilot metrics:")
display(pilot_metrics_df[[
    "case_id", "method", "dice", "iou", "target_focus", "log10_ratio", "dice_fixed05", "dice_topk"
]])

print("\nPilot comparisons:")
display(pilot_comparisons_df)

print("\n3-CASE PILOT PASS.")

In [ ]:
from pathlib import Path
import json, shutil, time

OUT_DIR = Path("/kaggle/working/Layer2R_publication_rebuild_EIA_v1")

status = {
    "run_name": "Layer2R_publication_rebuild_EIA_v1",
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "current_status": "3-case pilot completed",
    "completed_steps": [
        "Created Layer2R output directories",
        "Locked 40 usable longitudinal cases",
        "Verified all raw MRI and mask files exist for 40 cases",
        "Completed 1-case target construction smoke test",
        "Completed 1-case baseline training smoke test",
        "Completed 1-case PCC + EIA smoke test",
        "Completed 3-case pilot run"
    ],
    "pilot_cases": [
        "PatientID_0003_T1_to_T2_t1c",
        "PatientID_0005_T3_to_T4_t1c",
        "PatientID_0008_T4_to_T6_t1c"
    ],
    "key_pilot_observation": {
        "PCC_vs_fixed": "PCC won 3/3",
        "PCC_vs_naive": "PCC won 3/3",
        "PCC_vs_EIA_linear": "PCC won 3/3",
        "PCC_vs_EIA_morph": "PCC won 3/3",
        "PCC_vs_EIA_blend075": "PCC lost 0/3; treat EIA-blend-0.75 as strong target-blending upper control"
    },
    "next_steps": [
        "Add EIA-blend-0.90 as conservative target-blending control",
        "Update protocol note: EIA-linear is primary EIA comparator; EIA-blend-0.75 is strong upper control",
        "Enable GPU if possible",
        "Run 40-case formal quick or publication run with resume support"
    ]
}

OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUT_DIR / "RUN_STATUS_BEFORE_CLOSE.json", "w") as f:
    json.dump(status, f, indent=2)

next_steps_text = """
# Layer2R Resume Notes

Current status:
- Layer2R publication rebuild directory created.
- 40 locked cases verified.
- 1-case target construction smoke test passed.
- 1-case baseline training smoke test passed.
- 1-case PCC + EIA smoke test passed.
- 3-case pilot passed.

Important pilot conclusion:
- PCC won against Fixed, Naive, EIA-linear, and EIA-morph in 3/3 pilot cases.
- EIA-blend-0.75 beat PCC in 3/3 cases, so it should be treated as a strong target-blending upper control, not the primary fairness comparator.
- EIA-linear remains the primary equal-information-access comparator.

Next step tomorrow:
1. Add EIA-blend-0.90.
2. Update protocol note.
3. Turn on GPU if possible.
4. Run 40-case formal run with full saving and resume support.

Do not delete:
- /kaggle/working/Layer2R_publication_rebuild_EIA_v1
"""

with open(OUT_DIR / "NEXT_STEPS_TOMORROW.md", "w") as f:
    f.write(next_steps_text)

print("Saved status files:")
print(OUT_DIR / "RUN_STATUS_BEFORE_CLOSE.json")
print(OUT_DIR / "NEXT_STEPS_TOMORROW.md")

In [ ]:
from pathlib import Path
import shutil
from IPython.display import FileLink, display

OUT_DIR = Path("/kaggle/working/Layer2R_publication_rebuild_EIA_v1")
ZIP_BASE = Path("/kaggle/working/Layer2R_publication_rebuild_EIA_v1_BACKUP_BEFORE_CLOSE")

zip_path = shutil.make_archive(str(ZIP_BASE), "zip", root_dir=str(OUT_DIR))

print("Backup zip created:")
print(zip_path)

display(FileLink(zip_path))

In [ ]:
from pathlib import Path
import shutil, os, json

OUT_DIR = Path("/kaggle/working/Layer2R_publication_rebuild_EIA_v1")
LIGHT_DIR = Path("/kaggle/working/Layer2R_LIGHT_BACKUP_BEFORE_CLOSE")

if LIGHT_DIR.exists():
    shutil.rmtree(LIGHT_DIR)

LIGHT_DIR.mkdir(parents=True, exist_ok=True)

# copy important small folders
for sub in ["tables", "figures", "logs"]:
    src = OUT_DIR / sub
    dst = LIGHT_DIR / sub
    if src.exists():
        shutil.copytree(src, dst)

# copy important root files
for fname in [
    "protocol.json",
    "RUN_STATUS_BEFORE_CLOSE.json",
    "NEXT_STEPS_TOMORROW.md"
]:
    src = OUT_DIR / fname
    if src.exists():
        shutil.copy2(src, LIGHT_DIR / fname)

# copy selected small case files only
case_src = OUT_DIR / "case_outputs"
case_dst = LIGHT_DIR / "case_outputs_selected"
case_dst.mkdir(parents=True, exist_ok=True)

if case_src.exists():
    for case_dir in case_src.iterdir():
        if case_dir.is_dir():
            dst_case = case_dst / case_dir.name
            dst_case.mkdir(parents=True, exist_ok=True)
            for fp in case_dir.iterdir():
                # keep csv/json/png only, skip large npy and pt
                if fp.suffix.lower() in [".csv", ".json", ".png", ".md", ".txt"]:
                    shutil.copy2(fp, dst_case / fp.name)

# create manifest
manifest = []
for fp in LIGHT_DIR.rglob("*"):
    if fp.is_file():
        manifest.append({
            "path": str(fp.relative_to(LIGHT_DIR)),
            "size_bytes": fp.stat().st_size
        })

with open(LIGHT_DIR / "LIGHT_BACKUP_MANIFEST.json", "w") as f:
    json.dump(manifest, f, indent=2)

zip_path = shutil.make_archive(
    "/kaggle/working/Layer2R_LIGHT_BACKUP_BEFORE_CLOSE",
    "zip",
    root_dir=str(LIGHT_DIR)
)

print("Light backup created:")
print(zip_path)
print("Light backup size MB:", round(Path(zip_path).stat().st_size / 1024 / 1024, 2))
print("Files included:", len(manifest))

In [1]:
from pathlib import Path
import json
import pandas as pd

OUT_DIR = Path("/kaggle/working/Layer2R_publication_rebuild_EIA_v1")
LIGHT_ZIP = Path("/kaggle/working/Layer2R_LIGHT_BACKUP_BEFORE_CLOSE.zip")

print("OUT_DIR exists:", OUT_DIR.exists())
print("LIGHT_ZIP exists:", LIGHT_ZIP.exists())

if OUT_DIR.exists():
    print("\nOUT_DIR contents:")
    for p in OUT_DIR.iterdir():
        print(" -", p.name)

    status_file = OUT_DIR / "RUN_STATUS_BEFORE_CLOSE.json"
    next_file = OUT_DIR / "NEXT_STEPS_TOMORROW.md"

    print("\nStatus file exists:", status_file.exists())
    print("Next steps file exists:", next_file.exists())

    if status_file.exists():
        with open(status_file, "r") as f:
            status = json.load(f)
        print("\nCurrent status:")
        print(status.get("current_status"))
        print("\nCompleted steps:")
        for s in status.get("completed_steps", []):
            print(" -", s)

    pilot_metrics = OUT_DIR / "tables" / "Layer2R_3case_pilot_metrics.csv"
    pilot_comp = OUT_DIR / "tables" / "Layer2R_3case_pilot_comparisons.csv"

    print("\nPilot metrics exists:", pilot_metrics.exists())
    print("Pilot comparisons exists:", pilot_comp.exists())

    if pilot_metrics.exists():
        df = pd.read_csv(pilot_metrics)
        print("\nPilot metrics shape:", df.shape)
        display(df.head())

    if pilot_comp.exists():
        dfc = pd.read_csv(pilot_comp)
        print("\nPilot comparisons shape:", dfc.shape)
        display(dfc.head())

OUT_DIR exists: False
LIGHT_ZIP exists: False


In [1]:
# ============================================================
# Layer2R 40-case formal run with resume support
# GPU version
# Default: run 5 new cases per batch, then manually Save Version
# ============================================================

from pathlib import Path
import os, re, json, time, random, math, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    import nibabel as nib
except ModuleNotFoundError:
    raise ModuleNotFoundError("nibabel is required but not installed.")

try:
    from scipy.ndimage import (
        gaussian_filter,
        binary_fill_holes,
        binary_closing,
        label,
        distance_transform_edt,
    )
    SCIPY_OK = True
except Exception as e:
    SCIPY_OK = False
    print("WARNING: scipy not available:", repr(e))

# -----------------------------
# Basic config
# -----------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type != "cuda":
    print("WARNING: GPU is not active. Stop this cell and enable GPU if you intended to use GPU.")

RUN_NAME = "Layer2R_publication_rebuild_EIA_v1"
OUT_DIR = Path("/kaggle/working") / RUN_NAME

DIRS = {
    "maps": OUT_DIR / "maps",
    "figures": OUT_DIR / "figures",
    "tables": OUT_DIR / "tables",
    "logs": OUT_DIR / "logs",
    "checkpoints": OUT_DIR / "checkpoints",
    "case_outputs": OUT_DIR / "case_outputs",
    "formal_results": OUT_DIR / "formal_results",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

ROOT = Path("/kaggle/input/datasets")
MODELA_DIR = ROOT / "jeechangxin/model-a-backup-final"
RAW_DIR = ROOT / "stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post"

case_csv = MODELA_DIR / "direct_target_case_metrics.csv"
assert case_csv.exists(), f"Missing case CSV: {case_csv}"

df_cases = pd.read_csv(case_csv)
locked_cases = sorted(df_cases["case_id"].unique().tolist())

print("Locked cases:", len(locked_cases))
print("First 5:", locked_cases[:5])

# -----------------------------
# Formal run settings
# -----------------------------

FORMAL_EPOCHS = 12
BATCH_SIZE = 8
LR = 1e-3
BASE_CHANNELS = 16

# Run only N new cases in this batch.
# After it finishes, Save Version, then run this same cell again tomorrow/next batch.
MAX_NEW_CASES = 5

# PCC / EIA parameters
DILATION_RADIUS = 26
SIGMA = 2.0

PCC_ROUNDS = 10
PCC_ETA = 0.30

NAIVE_GAMMA = 2.5

EIA_ALPHA = 0.30
EIA_BETA = 0.30

EIA_BLEND_LAMBDA_090 = 0.90
EIA_BLEND_LAMBDA_075 = 0.75

MAIN_MODE = "topk"
THRESHOLD = 0.5

protocol = {
    "run_name": RUN_NAME,
    "formal_epochs": FORMAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "base_channels": BASE_CHANNELS,
    "device": str(device),
    "max_new_cases_per_batch": MAX_NEW_CASES,
    "dataset": "MU-Glioma-Post",
    "modality": "T1c",
    "locked_cases_n": len(locked_cases),
    "target_definition": "future tumour mask AND NOT current tumour mask",
    "main_metric_mode": MAIN_MODE,
    "threshold": THRESHOLD,
    "methods": [
        "fixed_baseline",
        "naive_self_tightening",
        "eia_linear",
        "eia_blend090",
        "eia_blend075",
        "eia_morph",
        "pcc_correction",
    ],
    "primary_eia_comparator": "eia_linear",
    "conservative_target_blending_control": "eia_blend090",
    "strong_target_blending_upper_control": "eia_blend075",
    "pcc_params": {
        "rounds": PCC_ROUNDS,
        "eta": PCC_ETA,
        "dilation_radius": DILATION_RADIUS,
        "sigma": SIGMA,
    },
    "eia_params": {
        "alpha": EIA_ALPHA,
        "beta": EIA_BETA,
        "blend_lambda_090": EIA_BLEND_LAMBDA_090,
        "blend_lambda_075": EIA_BLEND_LAMBDA_075,
    },
}

with open(OUT_DIR / "protocol.json", "w") as f:
    json.dump(protocol, f, indent=2)

pd.DataFrame({"case_id": locked_cases}).to_csv(DIRS["tables"] / "locked_40_cases.csv", index=False)

print("Protocol saved:", OUT_DIR / "protocol.json")

# -----------------------------
# Path utilities
# -----------------------------

def parse_case_id(case_id):
    m = re.match(r"(PatientID_\d+)_T(\d+)_to_T(\d+)_(\w+)", case_id)
    if not m:
        raise ValueError(f"Cannot parse case_id: {case_id}")
    patient_id = m.group(1)
    cur_tp = int(m.group(2))
    fut_tp = int(m.group(3))
    modality = m.group(4)
    return patient_id, cur_tp, fut_tp, modality

def get_raw_paths(case_id):
    patient_id, cur_tp, fut_tp, modality = parse_case_id(case_id)

    patient_dir = RAW_DIR / patient_id
    cur_dir = patient_dir / f"Timepoint_{cur_tp}"
    fut_dir = patient_dir / f"Timepoint_{fut_tp}"

    cur_img = cur_dir / f"{patient_id}_Timepoint_{cur_tp}_brain_{modality}.nii"
    fut_img = fut_dir / f"{patient_id}_Timepoint_{fut_tp}_brain_{modality}.nii"
    cur_mask = cur_dir / f"{patient_id}_Timepoint_{cur_tp}_tumorMask.nii"
    fut_mask = fut_dir / f"{patient_id}_Timepoint_{fut_tp}_tumorMask.nii"

    return {
        "case_id": case_id,
        "patient_id": patient_id,
        "cur_tp": cur_tp,
        "fut_tp": fut_tp,
        "modality": modality,
        "cur_img": cur_img,
        "fut_img": fut_img,
        "cur_mask": cur_mask,
        "fut_mask": fut_mask,
    }

def load_nii_raw(path):
    img = nib.load(str(path))
    arr = img.get_fdata(dtype=np.float32)
    return arr, img.affine, img.header

def to_zhw(arr):
    arr = np.asarray(arr)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D array, got {arr.shape}")
    # Common raw shape: H,W,Z = 240,240,155 -> Z,H,W
    if arr.shape[0] == arr.shape[1] and arr.shape[-1] < arr.shape[0]:
        arr = np.transpose(arr, (2, 0, 1))
    return arr.astype(np.float32)

def load_nii_zhw(path):
    arr, affine, header = load_nii_raw(path)
    arr_zhw = to_zhw(arr)
    return arr_zhw, affine, header, arr.shape

def robust_normalize(img, brain_mask=None, p_low=1, p_high=99, eps=1e-6):
    img = img.astype(np.float32)
    if brain_mask is None:
        brain_mask = img != 0

    vals = img[brain_mask]
    if vals.size < 10:
        return np.zeros_like(img, dtype=np.float32)

    lo, hi = np.percentile(vals, [p_low, p_high])
    if hi <= lo + eps:
        return np.zeros_like(img, dtype=np.float32)

    out = np.clip(img, lo, hi)
    out = (out - lo) / (hi - lo + eps)
    out[~brain_mask] = 0
    return out.astype(np.float32)

# -----------------------------
# Metrics
# -----------------------------

def dice_binary(pred, gt, eps=1e-6):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return float((2 * inter + eps) / (denom + eps))

def iou_binary(pred, gt, eps=1e-6):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return float((inter + eps) / (union + eps))

def topk_mask(prob, k):
    prob = prob.astype(np.float32)
    flat = prob.reshape(-1)
    k = int(k)
    k = max(1, min(k, flat.size))
    idx = np.argpartition(flat, -k)[-k:]
    out = np.zeros_like(flat, dtype=np.uint8)
    out[idx] = 1
    return out.reshape(prob.shape).astype(bool)

def target_focus(prob, gt, eps=1e-8):
    prob = prob.astype(np.float32)
    gt = gt.astype(bool)
    total = float(prob.sum())
    inside = float(prob[gt].sum())
    return float(inside / (total + eps))

def log10_ratio(prob, gt, eps=1e-8):
    prob = prob.astype(np.float32)
    gt = gt.astype(bool)
    inside_mean = float(prob[gt].mean()) if gt.sum() > 0 else 0.0
    outside_mean = float(prob[~gt].mean()) if (~gt).sum() > 0 else 0.0
    return float(np.log10((inside_mean + eps) / (outside_mean + eps)))

def safe_clip_prob(x):
    return np.clip(
        np.nan_to_num(x.astype(np.float32), nan=0.0, posinf=1.0, neginf=0.0),
        0,
        1,
    )

def safe_logit(p, eps=1e-5):
    p = np.clip(p.astype(np.float32), eps, 1 - eps)
    return np.log(p / (1 - p)).astype(np.float32)

def sigmoid(x):
    x = np.clip(x.astype(np.float32), -30, 30)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)

def eval_prob_map(prob, gt, threshold=0.5, main_mode="topk"):
    prob = safe_clip_prob(prob)
    gt_bool = gt.astype(bool)

    pred_fixed = prob >= threshold
    k = int(gt_bool.sum())
    pred_topk = topk_mask(prob, k)

    out = {
        "dice_fixed05": dice_binary(pred_fixed, gt_bool),
        "iou_fixed05": iou_binary(pred_fixed, gt_bool),
        "dice_topk": dice_binary(pred_topk, gt_bool),
        "iou_topk": iou_binary(pred_topk, gt_bool),
        "target_focus": target_focus(prob, gt_bool),
        "log10_ratio": log10_ratio(prob, gt_bool),
        "pred_fixed05_voxels": int(pred_fixed.sum()),
        "pred_topk_voxels": int(pred_topk.sum()),
        "target_voxels": int(gt_bool.sum()),
        "prob_min": float(prob.min()),
        "prob_max": float(prob.max()),
        "prob_mean": float(prob.mean()),
        "prob_sum": float(prob.sum()),
    }

    if main_mode == "topk":
        out["dice"] = out["dice_topk"]
        out["iou"] = out["iou_topk"]
        out["main_mode"] = "topk"
    else:
        out["dice"] = out["dice_fixed05"]
        out["iou"] = out["iou_fixed05"]
        out["main_mode"] = "fixed05"

    return out

def choose_target_slice(target):
    per_slice = target.astype(bool).sum(axis=(1, 2))
    if per_slice.max() > 0:
        return int(np.argmax(per_slice))
    return int(target.shape[0] // 2)

# -----------------------------
# Target-derived supports
# -----------------------------

def normalize01(x, eps=1e-8):
    x = x.astype(np.float32)
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx <= mn + eps:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn + eps)).astype(np.float32)

def make_dilated_region(mask_bool, radius=26):
    mask_bool = mask_bool.astype(bool)
    if SCIPY_OK:
        dist = distance_transform_edt(~mask_bool)
        return dist <= radius
    return mask_bool.copy()

def smooth_mask(mask_bool, sigma=2.0):
    x = mask_bool.astype(np.float32)
    if SCIPY_OK:
        x = gaussian_filter(x, sigma=sigma)
    return normalize01(x)

# -----------------------------
# Dataset and model
# -----------------------------

class SliceDataset(Dataset):
    def __init__(self, cur_img, cur_mask, target):
        self.cur_img = cur_img.astype(np.float32)
        self.cur_mask = cur_mask.astype(np.float32)
        self.target = target.astype(np.float32)
        self.indices = np.arange(cur_img.shape[0])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        z = self.indices[idx]
        x = np.stack([self.cur_img[z], self.cur_mask[z]], axis=0).astype(np.float32)
        y = self.target[z][None, :, :].astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y), int(z)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class MiniUNet(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)

def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dims)
    denom = torch.sum(probs, dims) + torch.sum(targets, dims)
    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()

@torch.no_grad()
def predict_full_volume(model, cur_img, cur_mask):
    model.eval()
    preds = np.zeros_like(cur_img, dtype=np.float32)

    for z in range(cur_img.shape[0]):
        x = np.stack([cur_img[z], cur_mask[z]], axis=0)[None].astype(np.float32)
        x_t = torch.from_numpy(x).to(device)
        logits = model(x_t)
        prob = torch.sigmoid(logits).detach().cpu().numpy()[0, 0]
        preds[z] = prob.astype(np.float32)

    return preds

def train_case_baseline(case_id, cur_img, cur_mask, target):
    dataset = SliceDataset(cur_img, cur_mask, target)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    pos = float(target.sum())
    neg = float(target.size - target.sum())
    pos_weight_value = min(80.0, max(1.0, neg / max(pos, 1.0)))
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

    model = MiniUNet(in_ch=2, out_ch=1, base=BASE_CHANNELS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def combined_loss(logits, y):
        return bce_loss(logits, y) + soft_dice_loss_from_logits(logits, y)

    best_dice_topk = -1.0
    best_state = None
    history = []

    start = time.time()

    for epoch in range(1, FORMAL_EPOCHS + 1):
        model.train()
        losses = []

        for x, y, _ in loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = combined_loss(logits, y)
            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

        prob_map = predict_full_volume(model, cur_img, cur_mask)
        m = eval_prob_map(prob_map, target, threshold=THRESHOLD, main_mode=MAIN_MODE)

        row = {
            "case_id": case_id,
            "epoch": epoch,
            "loss": float(np.mean(losses)),
            "pos_weight": float(pos_weight_value),
            **m,
        }
        history.append(row)

        print(
            f"{case_id} | Epoch {epoch:02d}/{FORMAL_EPOCHS} | "
            f"loss={row['loss']:.4f} | "
            f"dice_topk={row['dice_topk']:.4f} | "
            f"dice_fixed05={row['dice_fixed05']:.4f} | "
            f"focus={row['target_focus']:.4f}"
        )

        if row["dice_topk"] > best_dice_topk:
            best_dice_topk = row["dice_topk"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    baseline_prob = predict_full_volume(model, cur_img, cur_mask)

    elapsed = time.time() - start

    return model, baseline_prob.astype(np.float32), pd.DataFrame(history), float(best_dice_topk), elapsed

# -----------------------------
# Correction methods
# -----------------------------

def run_corrections_for_case(baseline, target):
    target_bool = target.astype(bool)

    fixed_baseline = safe_clip_prob(baseline)
    naive = sigmoid(NAIVE_GAMMA * safe_logit(fixed_baseline))

    R = make_dilated_region(target_bool, radius=DILATION_RADIUS)
    S = smooth_mask(target_bool, sigma=SIGMA)

    eia_linear = safe_clip_prob(
        fixed_baseline
        + EIA_ALPHA * S * (1.0 - fixed_baseline)
        - EIA_BETA * (~R).astype(np.float32) * fixed_baseline
    )

    eia_blend090 = safe_clip_prob(
        EIA_BLEND_LAMBDA_090 * fixed_baseline
        + (1.0 - EIA_BLEND_LAMBDA_090) * S
    )

    eia_blend075 = safe_clip_prob(
        EIA_BLEND_LAMBDA_075 * fixed_baseline
        + (1.0 - EIA_BLEND_LAMBDA_075) * S
    )

    baseline_binary = fixed_baseline >= THRESHOLD
    morph = np.logical_and(baseline_binary, R)

    if SCIPY_OK:
        morph = binary_closing(morph, iterations=1)
        morph = binary_fill_holes(morph)
        lab, n_lab = label(morph)
        min_size = 20
        keep = np.zeros_like(morph, dtype=bool)
        for lab_id in range(1, n_lab + 1):
            comp = lab == lab_id
            if comp.sum() >= min_size:
                keep |= comp
        morph = keep

    eia_morph = morph.astype(np.float32)

    pcc = fixed_baseline.copy()
    pcc_round_rows = []

    for r in range(1, PCC_ROUNDS + 1):
        p = safe_clip_prob(pcc)

        residual = (target_bool.astype(np.float32) - p) * R.astype(np.float32)

        if SCIPY_OK:
            residual_smooth = gaussian_filter(residual, sigma=SIGMA)
        else:
            residual_smooth = residual

        background_suppression = (~R).astype(np.float32) * p

        logits = safe_logit(p)
        logits = logits + PCC_ETA * residual_smooth - PCC_ETA * background_suppression
        pcc = safe_clip_prob(sigmoid(logits))

        rm = eval_prob_map(pcc, target_bool, threshold=THRESHOLD, main_mode=MAIN_MODE)
        pcc_round_rows.append({
            "round": r,
            "dice": rm["dice"],
            "iou": rm["iou"],
            "target_focus": rm["target_focus"],
            "log10_ratio": rm["log10_ratio"],
        })

    return {
        "fixed_baseline": fixed_baseline,
        "naive_self_tightening": naive,
        "eia_linear": eia_linear,
        "eia_blend090": eia_blend090,
        "eia_blend075": eia_blend075,
        "eia_morph": eia_morph,
        "pcc_correction": pcc,
        "target_support_R": R.astype(np.uint8),
        "target_signal_S": S.astype(np.float32),
        "pcc_round_history": pd.DataFrame(pcc_round_rows),
    }

# -----------------------------
# Load/prepare case arrays
# -----------------------------

def prepare_case_arrays(case_id, case_out_dir):
    paths = get_raw_paths(case_id)

    for k in ["cur_img", "fut_img", "cur_mask", "fut_mask"]:
        if not paths[k].exists():
            raise FileNotFoundError(f"Missing {k}: {paths[k]}")

    cur_img_raw, _, _, cur_raw_shape = load_nii_zhw(paths["cur_img"])
    fut_img_raw, _, _, fut_raw_shape = load_nii_zhw(paths["fut_img"])
    cur_mask, _, _, cur_mask_raw_shape = load_nii_zhw(paths["cur_mask"])
    fut_mask, _, _, fut_mask_raw_shape = load_nii_zhw(paths["fut_mask"])

    cur_mask = cur_mask > 0
    fut_mask = fut_mask > 0

    target = np.logical_and(fut_mask, np.logical_not(cur_mask))
    brain_mask = np.logical_or(cur_img_raw != 0, fut_img_raw != 0)

    cur_img = robust_normalize(cur_img_raw, brain_mask=brain_mask)

    assert cur_img.shape == cur_mask.shape == fut_mask.shape == target.shape

    np.save(case_out_dir / "current_t1c_norm_zhw.npy", cur_img.astype(np.float32))
    np.save(case_out_dir / "current_mask_zhw.npy", cur_mask.astype(np.uint8))
    np.save(case_out_dir / "future_mask_zhw.npy", fut_mask.astype(np.uint8))
    np.save(case_out_dir / "future_change_target_zhw.npy", target.astype(np.uint8))

    meta = {
        "case_id": case_id,
        "cur_raw_shape": list(cur_raw_shape),
        "fut_raw_shape": list(fut_raw_shape),
        "cur_mask_raw_shape": list(cur_mask_raw_shape),
        "fut_mask_raw_shape": list(fut_mask_raw_shape),
        "zhw_shape": list(cur_img.shape),
        "cur_mask_voxels": int(cur_mask.sum()),
        "fut_mask_voxels": int(fut_mask.sum()),
        "target_voxels": int(target.sum()),
        "brain_voxels": int(brain_mask.sum()),
    }

    with open(case_out_dir / "case_meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    return cur_img, cur_mask.astype(np.float32), target.astype(np.uint8), meta

# -----------------------------
# Resume logic
# -----------------------------

metrics_csv = DIRS["formal_results"] / "Layer2R_formal_case_method_metrics.csv"
comparisons_csv = DIRS["formal_results"] / "Layer2R_formal_pairwise_comparisons.csv"
training_csv = DIRS["formal_results"] / "Layer2R_formal_training_history.csv"
completed_txt = DIRS["formal_results"] / "completed_cases.txt"
failed_csv = DIRS["formal_results"] / "failed_cases.csv"

completed_cases = set()

if completed_txt.exists():
    completed_cases.update([x.strip() for x in completed_txt.read_text().splitlines() if x.strip()])

if metrics_csv.exists():
    try:
        tmp_metrics = pd.read_csv(metrics_csv)
        done_from_metrics = tmp_metrics.groupby("case_id")["method"].nunique()
        completed_cases.update(done_from_metrics[done_from_metrics >= 7].index.tolist())
    except Exception:
        pass

print("Already completed cases:", len(completed_cases))

cases_to_run = [c for c in locked_cases if c not in completed_cases]
if MAX_NEW_CASES is not None:
    cases_to_run = cases_to_run[:MAX_NEW_CASES]

print("Cases to run in this batch:", len(cases_to_run))
for c in cases_to_run:
    print(" -", c)

# -----------------------------
# Append helpers
# -----------------------------

def append_df_to_csv(df, path):
    path = Path(path)
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

def update_summary():
    if not metrics_csv.exists():
        return

    df = pd.read_csv(metrics_csv)
    if df.empty:
        return

    summary = (
        df.groupby("method")
        .agg(
            n=("case_id", "nunique"),
            dice_mean=("dice", "mean"),
            dice_median=("dice", "median"),
            iou_mean=("iou", "mean"),
            iou_median=("iou", "median"),
            target_focus_mean=("target_focus", "mean"),
            target_focus_median=("target_focus", "median"),
            log10_ratio_mean=("log10_ratio", "mean"),
            log10_ratio_median=("log10_ratio", "median"),
        )
        .reset_index()
    )

    summary.to_csv(DIRS["formal_results"] / "Layer2R_formal_summary_by_method.csv", index=False)

    # Pairwise aggregate if exists
    if comparisons_csv.exists():
        comp = pd.read_csv(comparisons_csv)
        comp_summary = (
            comp.groupby("comparison")
            .agg(
                n=("case_id", "nunique"),
                dice_mean_diff=("dice_diff", "mean"),
                dice_median_diff=("dice_diff", "median"),
                dice_wins=("pcc_better_dice", "sum"),
                iou_mean_diff=("iou_diff", "mean"),
                iou_median_diff=("iou_diff", "median"),
                iou_wins=("pcc_better_iou", "sum"),
            )
            .reset_index()
        )
        comp_summary["dice_win_rate"] = comp_summary["dice_wins"] / comp_summary["n"]
        comp_summary["iou_win_rate"] = comp_summary["iou_wins"] / comp_summary["n"]
        comp_summary.to_csv(DIRS["formal_results"] / "Layer2R_formal_pairwise_summary.csv", index=False)

# -----------------------------
# Main formal batch loop
# -----------------------------

batch_start = time.time()
failed_rows = []

for case_i, case_id in enumerate(cases_to_run, start=1):
    print("\n" + "=" * 110)
    print(f"Running case {case_i}/{len(cases_to_run)} in this batch: {case_id}")
    print("=" * 110)

    case_start = time.time()
    case_out_dir = DIRS["case_outputs"] / case_id
    case_out_dir.mkdir(parents=True, exist_ok=True)

    try:
        cur_img, cur_mask, target, meta = prepare_case_arrays(case_id, case_out_dir)

        if int(target.sum()) <= 0:
            raise ValueError("Future-change target is empty.")

        print("Shape:", cur_img.shape, "| target voxels:", int(target.sum()))

        # Train baseline
        model, baseline_prob, hist_df, best_dice_topk, elapsed_train = train_case_baseline(
            case_id, cur_img, cur_mask, target
        )

        hist_df["formal_epochs"] = FORMAL_EPOCHS
        hist_df["training_elapsed_sec_total"] = elapsed_train
        append_df_to_csv(hist_df, training_csv)
        hist_df.to_csv(case_out_dir / "baseline_training_history_formal.csv", index=False)

        # Save checkpoint and baseline map
        torch.save(
            {
                "case_id": case_id,
                "formal_epochs": FORMAL_EPOCHS,
                "model_state_dict": model.state_dict(),
                "best_dice_topk": best_dice_topk,
                "protocol": protocol,
            },
            DIRS["checkpoints"] / f"{case_id}_baseline_formal_best.pt",
        )

        np.save(case_out_dir / "baseline_prob_map_formal_float16.npy", baseline_prob.astype(np.float16))

        # Run corrections
        maps = run_corrections_for_case(baseline_prob, target)

        # Save support signals
        np.save(case_out_dir / "target_support_R_formal.npy", maps["target_support_R"].astype(np.uint8))
        np.save(case_out_dir / "target_signal_S_formal_float16.npy", maps["target_signal_S"].astype(np.float16))
        maps["pcc_round_history"].to_csv(case_out_dir / "pcc_round_history_formal.csv", index=False)

        # Evaluate methods
        method_rows = []

        method_names = [
            "fixed_baseline",
            "naive_self_tightening",
            "eia_linear",
            "eia_blend090",
            "eia_blend075",
            "eia_morph",
            "pcc_correction",
        ]

        for method in method_names:
            prob = maps[method]
            m = eval_prob_map(prob, target.astype(bool), threshold=THRESHOLD, main_mode=MAIN_MODE)

            row = {
                "case_id": case_id,
                "method": method,
                "formal_epochs": FORMAL_EPOCHS,
                "baseline_best_dice_topk": best_dice_topk,
                "baseline_training_elapsed_sec": elapsed_train,
                **m,
            }
            method_rows.append(row)

            # Save maps. Float16 saves disk space.
            if method == "eia_morph":
                np.save(case_out_dir / f"{method}_formal_uint8.npy", (prob > 0.5).astype(np.uint8))
            else:
                np.save(case_out_dir / f"{method}_formal_float16.npy", safe_clip_prob(prob).astype(np.float16))

        method_df = pd.DataFrame(method_rows)
        append_df_to_csv(method_df, metrics_csv)
        method_df.to_csv(case_out_dir / "case_method_metrics_formal.csv", index=False)

        # Pairwise comparisons
        pcc_row = method_df[method_df["method"] == "pcc_correction"].iloc[0].to_dict()
        comp_rows = []

        for method in [
            "fixed_baseline",
            "naive_self_tightening",
            "eia_linear",
            "eia_blend090",
            "eia_blend075",
            "eia_morph",
        ]:
            row = method_df[method_df["method"] == method].iloc[0].to_dict()
            comp_rows.append({
                "case_id": case_id,
                "comparison": f"PCC vs {method}",
                "dice_diff": pcc_row["dice"] - row["dice"],
                "iou_diff": pcc_row["iou"] - row["iou"],
                "target_focus_diff": pcc_row["target_focus"] - row["target_focus"],
                "log10_ratio_diff": pcc_row["log10_ratio"] - row["log10_ratio"],
                "pcc_better_dice": bool(pcc_row["dice"] > row["dice"]),
                "pcc_better_iou": bool(pcc_row["iou"] > row["iou"]),
            })

        comp_df = pd.DataFrame(comp_rows)
        append_df_to_csv(comp_df, comparisons_csv)
        comp_df.to_csv(case_out_dir / "case_pairwise_comparisons_formal.csv", index=False)

        # Visualization
        z = choose_target_slice(target)

        fig, axes = plt.subplots(2, 4, figsize=(18, 8))
        axes = axes.ravel()

        vis_items = [
            ("Current T1c", cur_img[z], "gray"),
            ("Target", target[z].astype(float), "gray"),
            ("Fixed baseline", maps["fixed_baseline"][z], "hot"),
            ("EIA-linear", maps["eia_linear"][z], "hot"),
            ("EIA-blend-0.90", maps["eia_blend090"][z], "hot"),
            ("EIA-blend-0.75", maps["eia_blend075"][z], "hot"),
            ("EIA-morph", maps["eia_morph"][z], "gray"),
            ("PCC", maps["pcc_correction"][z], "hot"),
        ]

        for ax, (title, img, cmap) in zip(axes, vis_items):
            ax.imshow(img, cmap=cmap)
            ax.set_title(title, fontsize=10)
            ax.axis("off")

        plt.tight_layout()
        fig_path = DIRS["figures"] / f"Layer2R_formal_{case_id}.png"
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        # Mark completed only at the very end
        with open(completed_txt, "a") as f:
            f.write(case_id + "\n")

        completed_cases.add(case_id)

        case_elapsed = time.time() - case_start
        print(f"CASE COMPLETED: {case_id} | elapsed sec: {case_elapsed:.1f}")
        print("Method metrics:")
        display(method_df[["case_id", "method", "dice", "iou", "target_focus", "log10_ratio", "dice_fixed05", "dice_topk"]])
        print("Comparisons:")
        display(comp_df)

        update_summary()

        # Clear GPU memory
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    except Exception as e:
        fail_row = {
            "case_id": case_id,
            "error": repr(e),
            "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        failed_rows.append(fail_row)
        pd.DataFrame([fail_row]).to_csv(
            failed_csv,
            mode="a",
            header=not failed_csv.exists(),
            index=False,
        )
        print("CASE FAILED:", case_id)
        print("ERROR:", repr(e))

# -----------------------------
# Final batch status
# -----------------------------

update_summary()

batch_elapsed = time.time() - batch_start

status = {
    "run_name": RUN_NAME,
    "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "device": str(device),
    "formal_epochs": FORMAL_EPOCHS,
    "max_new_cases_this_batch": MAX_NEW_CASES,
    "new_cases_attempted_this_batch": len(cases_to_run),
    "failed_this_batch": failed_rows,
    "completed_cases_total": len(completed_cases),
    "remaining_cases_total": len([c for c in locked_cases if c not in completed_cases]),
    "batch_elapsed_sec": batch_elapsed,
    "next_instruction": "Save Version now before running the next batch.",
}

with open(DIRS["formal_results"] / "RUN_STATUS_LATEST.json", "w") as f:
    json.dump(status, f, indent=2)

print("\n" + "=" * 110)
print("BATCH FINISHED")
print("=" * 110)
print(json.dumps(status, indent=2))

if metrics_csv.exists():
    dfm = pd.read_csv(metrics_csv)
    print("\nCurrent summary by method:")
    display(pd.read_csv(DIRS["formal_results"] / "Layer2R_formal_summary_by_method.csv"))

if comparisons_csv.exists():
    print("\nCurrent pairwise summary:")
    display(pd.read_csv(DIRS["formal_results"] / "Layer2R_formal_pairwise_summary.csv"))

print("\nIMPORTANT: Save Version now before running the next batch.")

Device: cuda
Locked cases: 40
First 5: ['PatientID_0003_T1_to_T2_t1c', 'PatientID_0005_T3_to_T4_t1c', 'PatientID_0006_T2_to_T4_t1c', 'PatientID_0007_T2_to_T3_t1c', 'PatientID_0008_T4_to_T6_t1c']
Protocol saved: /kaggle/working/Layer2R_publication_rebuild_EIA_v1/protocol.json
Already completed cases: 0
Cases to run in this batch: 5
 - PatientID_0003_T1_to_T2_t1c
 - PatientID_0005_T3_to_T4_t1c
 - PatientID_0006_T2_to_T4_t1c
 - PatientID_0007_T2_to_T3_t1c
 - PatientID_0008_T4_to_T6_t1c

Running case 1/5 in this batch: PatientID_0003_T1_to_T2_t1c
Shape: (155, 240, 240) | target voxels: 24613
PatientID_0003_T1_to_T2_t1c | Epoch 01/12 | loss=1.5811 | dice_topk=0.1344 | dice_fixed05=0.0000 | focus=0.0030
PatientID_0003_T1_to_T2_t1c | Epoch 02/12 | loss=1.4688 | dice_topk=0.5868 | dice_fixed05=0.1671 | focus=0.0068
PatientID_0003_T1_to_T2_t1c | Epoch 03/12 | loss=1.4058 | dice_topk=0.6258 | dice_fixed05=0.4452 | focus=0.0082
PatientID_0003_T1_to_T2_t1c | Epoch 04/12 | loss=1.3456 | dice_topk=0

,case_id,method,dice,iou,target_focus,log10_ratio,dice_fixed05,dice_topk
0,PatientID_0003_T1_to_T2_t1c,fixed_baseline,0.674928,0.509352,0.021092,0.891765,0.329421,0.674928
1,PatientID_0003_T1_to_T2_t1c,naive_self_tightening,0.674969,0.509398,0.079521,1.494856,0.329421,0.674969
2,PatientID_0003_T1_to_T2_t1c,eia_linear,0.686913,0.523129,0.028060,1.018838,0.351489,0.686913
3,PatientID_0003_T1_to_T2_t1c,eia_blend090,0.828952,0.707872,0.022590,0.922221,0.350987,0.828952
4,PatientID_0003_T1_to_T2_t1c,eia_blend075,0.839922,0.724022,0.025562,0.977230,0.401483,0.839922
5,PatientID_0003_T1_to_T2_t1c,eia_morph,0.198838,0.110394,0.197232,1.948779,0.324181,0.198838
6,PatientID_0003_T1_to_T2_t1c,pcc_correction,0.757405,0.609534,0.029004,1.033639,0.493284,0.757405


Comparisons:


,case_id,comparison,dice_diff,iou_diff,target_focus_diff,log10_ratio_diff,pcc_better_dice,pcc_better_iou
0,PatientID_0003_T1_to_T2_t1c,PCC vs fixed_baseline,0.082477,0.100183,0.007913,0.141874,True,True
1,PatientID_0003_T1_to_T2_t1c,PCC vs naive_self_tightening,0.082436,0.100136,-0.050516,-0.461217,True,True
2,PatientID_0003_T1_to_T2_t1c,PCC vs eia_linear,0.070491,0.086406,0.000945,0.014801,True,True
3,PatientID_0003_T1_to_T2_t1c,PCC vs eia_blend090,-0.071548,-0.098338,0.006415,0.111418,False,False
4,PatientID_0003_T1_to_T2_t1c,PCC vs eia_blend075,-0.082517,-0.114488,0.003443,0.056409,False,False
5,PatientID_0003_T1_to_T2_t1c,PCC vs eia_morph,0.558567,0.499140,-0.168228,-0.915140,True,True



Running case 2/5 in this batch: PatientID_0005_T3_to_T4_t1c
Shape: (155, 240, 240) | target voxels: 18835
PatientID_0005_T3_to_T4_t1c | Epoch 01/12 | loss=1.4993 | dice_topk=0.2068 | dice_fixed05=0.0000 | focus=0.0022
PatientID_0005_T3_to_T4_t1c | Epoch 02/12 | loss=1.3703 | dice_topk=0.5414 | dice_fixed05=0.4743 | focus=0.0052
PatientID_0005_T3_to_T4_t1c | Epoch 03/12 | loss=1.3165 | dice_topk=0.5493 | dice_fixed05=0.2805 | focus=0.0078
PatientID_0005_T3_to_T4_t1c | Epoch 04/12 | loss=1.2703 | dice_topk=0.5215 | dice_fixed05=0.2905 | focus=0.0090
PatientID_0005_T3_to_T4_t1c | Epoch 05/12 | loss=1.2269 | dice_topk=0.5784 | dice_fixed05=0.2426 | focus=0.0110
PatientID_0005_T3_to_T4_t1c | Epoch 06/12 | loss=1.1884 | dice_topk=0.5388 | dice_fixed05=0.3225 | focus=0.0133
PatientID_0005_T3_to_T4_t1c | Epoch 07/12 | loss=1.1568 | dice_topk=0.6024 | dice_fixed05=0.3998 | focus=0.0146
PatientID_0005_T3_to_T4_t1c | Epoch 08/12 | loss=1.1271 | dice_topk=0.6191 | dice_fixed05=0.3815 | focus=0.01

...


BATCH FINISHED
{
  "run_name": "Layer2R_publication_rebuild_EIA_v1",
  "time": "2026-06-17 00:27:23",
  "device": "cuda",
  "formal_epochs": 12,
  "max_new_cases_this_batch": 5,
  "new_cases_attempted_this_batch": 5,
  "failed_this_batch": [],
  "completed_cases_total": 5,
  "remaining_cases_total": 35,
  "batch_elapsed_sec": 250.44420456886292,
  "next_instruction": "Save Version now before running the next batch."
}

Current summary by method:


,method,n,dice_mean,dice_median,iou_mean,iou_median,target_focus_mean,target_focus_median,log10_ratio_mean,log10_ratio_median
0,eia_blend075,5,0.784830,0.784338,0.651151,0.645194,0.017575,0.017077,0.976189,0.977230
1,eia_blend090,5,0.746829,0.760499,0.604696,0.613553,0.015781,0.014985,0.926735,0.922221
2,eia_linear,5,0.622687,0.606304,0.453714,0.435033,0.019678,0.018388,1.030617,1.018838
3,eia_morph,5,0.196223,0.198838,0.109540,0.110394,0.184776,0.197232,2.162014,2.202657
4,fixed_baseline,5,0.587020,0.572780,0.418870,0.401325,0.014877,0.013935,0.899567,0.891765
5,naive_self_tightening,5,0.586991,0.572780,0.418841,0.401325,0.094682,0.079521,1.720486,1.671332
6,pcc_correction,5,0.718072,0.709265,0.563561,0.549504,0.019114,0.019709,1.012246,1.033639



Current pairwise summary:


,comparison,n,dice_mean_diff,dice_median_diff,dice_wins,iou_mean_diff,iou_median_diff,iou_wins,dice_win_rate,iou_win_rate
0,PCC vs eia_blend075,5,-0.066758,-0.079525,1,-0.087590,-0.110737,1,0.2,0.2
1,PCC vs eia_blend090,5,-0.028757,-0.051234,1,-0.041135,-0.064048,1,0.2,0.2
2,PCC vs eia_linear,5,0.095385,0.070491,5,0.109846,0.086406,5,1.0,1.0
3,PCC vs eia_morph,5,0.521848,0.476125,5,0.454021,0.413621,5,1.0,1.0
4,PCC vs fixed_baseline,5,0.131051,0.082477,5,0.144691,0.100183,5,1.0,1.0
5,PCC vs naive_self_tightening,5,0.131081,0.082436,5,0.144719,0.100136,5,1.0,1.0



IMPORTANT: Save Version now before running the next batch.


In [2]:
# ============================================================
# Layer2R 40-case formal run with resume support
# GPU version
# Continue from completed cases and run all remaining cases
# ============================================================

from pathlib import Path
import os, re, json, time, random, math, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    import nibabel as nib
except ModuleNotFoundError:
    raise ModuleNotFoundError("nibabel is required but not installed.")

try:
    from scipy.ndimage import (
        gaussian_filter,
        binary_fill_holes,
        binary_closing,
        label,
        distance_transform_edt,
    )
    SCIPY_OK = True
except Exception as e:
    SCIPY_OK = False
    print("WARNING: scipy not available:", repr(e))

# -----------------------------
# Basic config
# -----------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type != "cuda":
    raise RuntimeError("GPU is not active. Please enable GPU before running the full formal experiment.")

RUN_NAME = "Layer2R_publication_rebuild_EIA_v1"
OUT_DIR = Path("/kaggle/working") / RUN_NAME

DIRS = {
    "maps": OUT_DIR / "maps",
    "figures": OUT_DIR / "figures",
    "tables": OUT_DIR / "tables",
    "logs": OUT_DIR / "logs",
    "checkpoints": OUT_DIR / "checkpoints",
    "case_outputs": OUT_DIR / "case_outputs",
    "formal_results": OUT_DIR / "formal_results",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

ROOT = Path("/kaggle/input/datasets")
MODELA_DIR = ROOT / "jeechangxin/model-a-backup-final"
RAW_DIR = ROOT / "stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post"

case_csv = MODELA_DIR / "direct_target_case_metrics.csv"
assert case_csv.exists(), f"Missing case CSV: {case_csv}"

df_cases = pd.read_csv(case_csv)
locked_cases = sorted(df_cases["case_id"].unique().tolist())

print("Locked cases:", len(locked_cases))
print("First 5:", locked_cases[:5])

# -----------------------------
# Formal run settings
# -----------------------------

FORMAL_EPOCHS = 12
BATCH_SIZE = 8
LR = 1e-3
BASE_CHANNELS = 16

# Run all remaining cases.
# None means no artificial batch limit.
MAX_NEW_CASES = None

# Safety check:
# Since you already completed 5 cases, this prevents accidental restart from 0.
REQUIRE_EXISTING_PROGRESS = True
EXPECTED_MIN_COMPLETED_CASES = 5

# PCC / EIA parameters
DILATION_RADIUS = 26
SIGMA = 2.0

PCC_ROUNDS = 10
PCC_ETA = 0.30

NAIVE_GAMMA = 2.5

EIA_ALPHA = 0.30
EIA_BETA = 0.30

EIA_BLEND_LAMBDA_090 = 0.90
EIA_BLEND_LAMBDA_075 = 0.75

MAIN_MODE = "topk"
THRESHOLD = 0.5

protocol = {
    "run_name": RUN_NAME,
    "formal_epochs": FORMAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "base_channels": BASE_CHANNELS,
    "device": str(device),
    "max_new_cases_per_batch": MAX_NEW_CASES,
    "dataset": "MU-Glioma-Post",
    "modality": "T1c",
    "locked_cases_n": len(locked_cases),
    "target_definition": "future tumour mask AND NOT current tumour mask",
    "main_metric_mode": MAIN_MODE,
    "threshold": THRESHOLD,
    "methods": [
        "fixed_baseline",
        "naive_self_tightening",
        "eia_linear",
        "eia_blend090",
        "eia_blend075",
        "eia_morph",
        "pcc_correction",
    ],
    "primary_eia_comparator": "eia_linear",
    "conservative_target_blending_control": "eia_blend090",
    "strong_target_blending_upper_control": "eia_blend075",
    "pcc_params": {
        "rounds": PCC_ROUNDS,
        "eta": PCC_ETA,
        "dilation_radius": DILATION_RADIUS,
        "sigma": SIGMA,
    },
    "eia_params": {
        "alpha": EIA_ALPHA,
        "beta": EIA_BETA,
        "blend_lambda_090": EIA_BLEND_LAMBDA_090,
        "blend_lambda_075": EIA_BLEND_LAMBDA_075,
    },
}

with open(OUT_DIR / "protocol.json", "w") as f:
    json.dump(protocol, f, indent=2)

pd.DataFrame({"case_id": locked_cases}).to_csv(DIRS["tables"] / "locked_40_cases.csv", index=False)

print("Protocol saved:", OUT_DIR / "protocol.json")

# -----------------------------
# Path utilities
# -----------------------------

def parse_case_id(case_id):
    m = re.match(r"(PatientID_\d+)_T(\d+)_to_T(\d+)_(\w+)", case_id)
    if not m:
        raise ValueError(f"Cannot parse case_id: {case_id}")
    patient_id = m.group(1)
    cur_tp = int(m.group(2))
    fut_tp = int(m.group(3))
    modality = m.group(4)
    return patient_id, cur_tp, fut_tp, modality

def get_raw_paths(case_id):
    patient_id, cur_tp, fut_tp, modality = parse_case_id(case_id)

    patient_dir = RAW_DIR / patient_id
    cur_dir = patient_dir / f"Timepoint_{cur_tp}"
    fut_dir = patient_dir / f"Timepoint_{fut_tp}"

    cur_img = cur_dir / f"{patient_id}_Timepoint_{cur_tp}_brain_{modality}.nii"
    fut_img = fut_dir / f"{patient_id}_Timepoint_{fut_tp}_brain_{modality}.nii"
    cur_mask = cur_dir / f"{patient_id}_Timepoint_{cur_tp}_tumorMask.nii"
    fut_mask = fut_dir / f"{patient_id}_Timepoint_{fut_tp}_tumorMask.nii"

    return {
        "case_id": case_id,
        "patient_id": patient_id,
        "cur_tp": cur_tp,
        "fut_tp": fut_tp,
        "modality": modality,
        "cur_img": cur_img,
        "fut_img": fut_img,
        "cur_mask": cur_mask,
        "fut_mask": fut_mask,
    }

def load_nii_raw(path):
    img = nib.load(str(path))
    arr = img.get_fdata(dtype=np.float32)
    return arr, img.affine, img.header

def to_zhw(arr):
    arr = np.asarray(arr)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D array, got {arr.shape}")
    if arr.shape[0] == arr.shape[1] and arr.shape[-1] < arr.shape[0]:
        arr = np.transpose(arr, (2, 0, 1))
    return arr.astype(np.float32)

def load_nii_zhw(path):
    arr, affine, header = load_nii_raw(path)
    arr_zhw = to_zhw(arr)
    return arr_zhw, affine, header, arr.shape

def robust_normalize(img, brain_mask=None, p_low=1, p_high=99, eps=1e-6):
    img = img.astype(np.float32)
    if brain_mask is None:
        brain_mask = img != 0

    vals = img[brain_mask]
    if vals.size < 10:
        return np.zeros_like(img, dtype=np.float32)

    lo, hi = np.percentile(vals, [p_low, p_high])
    if hi <= lo + eps:
        return np.zeros_like(img, dtype=np.float32)

    out = np.clip(img, lo, hi)
    out = (out - lo) / (hi - lo + eps)
    out[~brain_mask] = 0
    return out.astype(np.float32)

# -----------------------------
# Metrics
# -----------------------------

def dice_binary(pred, gt, eps=1e-6):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return float((2 * inter + eps) / (denom + eps))

def iou_binary(pred, gt, eps=1e-6):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return float((inter + eps) / (union + eps))

def topk_mask(prob, k):
    prob = prob.astype(np.float32)
    flat = prob.reshape(-1)
    k = int(k)
    k = max(1, min(k, flat.size))
    idx = np.argpartition(flat, -k)[-k:]
    out = np.zeros_like(flat, dtype=np.uint8)
    out[idx] = 1
    return out.reshape(prob.shape).astype(bool)

def target_focus(prob, gt, eps=1e-8):
    prob = prob.astype(np.float32)
    gt = gt.astype(bool)
    total = float(prob.sum())
    inside = float(prob[gt].sum())
    return float(inside / (total + eps))

def log10_ratio(prob, gt, eps=1e-8):
    prob = prob.astype(np.float32)
    gt = gt.astype(bool)
    inside_mean = float(prob[gt].mean()) if gt.sum() > 0 else 0.0
    outside_mean = float(prob[~gt].mean()) if (~gt).sum() > 0 else 0.0
    return float(np.log10((inside_mean + eps) / (outside_mean + eps)))

def safe_clip_prob(x):
    return np.clip(
        np.nan_to_num(x.astype(np.float32), nan=0.0, posinf=1.0, neginf=0.0),
        0,
        1,
    )

def safe_logit(p, eps=1e-5):
    p = np.clip(p.astype(np.float32), eps, 1 - eps)
    return np.log(p / (1 - p)).astype(np.float32)

def sigmoid(x):
    x = np.clip(x.astype(np.float32), -30, 30)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)

def eval_prob_map(prob, gt, threshold=0.5, main_mode="topk"):
    prob = safe_clip_prob(prob)
    gt_bool = gt.astype(bool)

    pred_fixed = prob >= threshold
    k = int(gt_bool.sum())
    pred_topk = topk_mask(prob, k)

    out = {
        "dice_fixed05": dice_binary(pred_fixed, gt_bool),
        "iou_fixed05": iou_binary(pred_fixed, gt_bool),
        "dice_topk": dice_binary(pred_topk, gt_bool),
        "iou_topk": iou_binary(pred_topk, gt_bool),
        "target_focus": target_focus(prob, gt_bool),
        "log10_ratio": log10_ratio(prob, gt_bool),
        "pred_fixed05_voxels": int(pred_fixed.sum()),
        "pred_topk_voxels": int(pred_topk.sum()),
        "target_voxels": int(gt_bool.sum()),
        "prob_min": float(prob.min()),
        "prob_max": float(prob.max()),
        "prob_mean": float(prob.mean()),
        "prob_sum": float(prob.sum()),
    }

    if main_mode == "topk":
        out["dice"] = out["dice_topk"]
        out["iou"] = out["iou_topk"]
        out["main_mode"] = "topk"
    else:
        out["dice"] = out["dice_fixed05"]
        out["iou"] = out["iou_fixed05"]
        out["main_mode"] = "fixed05"

    return out

def choose_target_slice(target):
    per_slice = target.astype(bool).sum(axis=(1, 2))
    if per_slice.max() > 0:
        return int(np.argmax(per_slice))
    return int(target.shape[0] // 2)

# -----------------------------
# Target-derived supports
# -----------------------------

def normalize01(x, eps=1e-8):
    x = x.astype(np.float32)
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx <= mn + eps:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn + eps)).astype(np.float32)

def make_dilated_region(mask_bool, radius=26):
    mask_bool = mask_bool.astype(bool)
    if SCIPY_OK:
        dist = distance_transform_edt(~mask_bool)
        return dist <= radius
    return mask_bool.copy()

def smooth_mask(mask_bool, sigma=2.0):
    x = mask_bool.astype(np.float32)
    if SCIPY_OK:
        x = gaussian_filter(x, sigma=sigma)
    return normalize01(x)

# -----------------------------
# Dataset and model
# -----------------------------

class SliceDataset(Dataset):
    def __init__(self, cur_img, cur_mask, target):
        self.cur_img = cur_img.astype(np.float32)
        self.cur_mask = cur_mask.astype(np.float32)
        self.target = target.astype(np.float32)
        self.indices = np.arange(cur_img.shape[0])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        z = self.indices[idx]
        x = np.stack([self.cur_img[z], self.cur_mask[z]], axis=0).astype(np.float32)
        y = self.target[z][None, :, :].astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y), int(z)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class MiniUNet(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)

def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dims)
    denom = torch.sum(probs, dims) + torch.sum(targets, dims)
    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()

@torch.no_grad()
def predict_full_volume(model, cur_img, cur_mask):
    model.eval()
    preds = np.zeros_like(cur_img, dtype=np.float32)

    for z in range(cur_img.shape[0]):
        x = np.stack([cur_img[z], cur_mask[z]], axis=0)[None].astype(np.float32)
        x_t = torch.from_numpy(x).to(device)
        logits = model(x_t)
        prob = torch.sigmoid(logits).detach().cpu().numpy()[0, 0]
        preds[z] = prob.astype(np.float32)

    return preds

def train_case_baseline(case_id, cur_img, cur_mask, target):
    dataset = SliceDataset(cur_img, cur_mask, target)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    pos = float(target.sum())
    neg = float(target.size - target.sum())
    pos_weight_value = min(80.0, max(1.0, neg / max(pos, 1.0)))
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

    model = MiniUNet(in_ch=2, out_ch=1, base=BASE_CHANNELS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def combined_loss(logits, y):
        return bce_loss(logits, y) + soft_dice_loss_from_logits(logits, y)

    best_dice_topk = -1.0
    best_state = None
    history = []

    start = time.time()

    for epoch in range(1, FORMAL_EPOCHS + 1):
        model.train()
        losses = []

        for x, y, _ in loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = combined_loss(logits, y)
            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

        prob_map = predict_full_volume(model, cur_img, cur_mask)
        m = eval_prob_map(prob_map, target, threshold=THRESHOLD, main_mode=MAIN_MODE)

        row = {
            "case_id": case_id,
            "epoch": epoch,
            "loss": float(np.mean(losses)),
            "pos_weight": float(pos_weight_value),
            **m,
        }
        history.append(row)

        print(
            f"{case_id} | Epoch {epoch:02d}/{FORMAL_EPOCHS} | "
            f"loss={row['loss']:.4f} | "
            f"dice_topk={row['dice_topk']:.4f} | "
            f"dice_fixed05={row['dice_fixed05']:.4f} | "
            f"focus={row['target_focus']:.4f}"
        )

        if row["dice_topk"] > best_dice_topk:
            best_dice_topk = row["dice_topk"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    baseline_prob = predict_full_volume(model, cur_img, cur_mask)

    elapsed = time.time() - start

    return model, baseline_prob.astype(np.float32), pd.DataFrame(history), float(best_dice_topk), elapsed

# -----------------------------
# Correction methods
# -----------------------------

def run_corrections_for_case(baseline, target):
    target_bool = target.astype(bool)

    fixed_baseline = safe_clip_prob(baseline)
    naive = sigmoid(NAIVE_GAMMA * safe_logit(fixed_baseline))

    R = make_dilated_region(target_bool, radius=DILATION_RADIUS)
    S = smooth_mask(target_bool, sigma=SIGMA)

    eia_linear = safe_clip_prob(
        fixed_baseline
        + EIA_ALPHA * S * (1.0 - fixed_baseline)
        - EIA_BETA * (~R).astype(np.float32) * fixed_baseline
    )

    eia_blend090 = safe_clip_prob(
        EIA_BLEND_LAMBDA_090 * fixed_baseline
        + (1.0 - EIA_BLEND_LAMBDA_090) * S
    )

    eia_blend075 = safe_clip_prob(
        EIA_BLEND_LAMBDA_075 * fixed_baseline
        + (1.0 - EIA_BLEND_LAMBDA_075) * S
    )

    baseline_binary = fixed_baseline >= THRESHOLD
    morph = np.logical_and(baseline_binary, R)

    if SCIPY_OK:
        morph = binary_closing(morph, iterations=1)
        morph = binary_fill_holes(morph)
        lab, n_lab = label(morph)
        min_size = 20
        keep = np.zeros_like(morph, dtype=bool)
        for lab_id in range(1, n_lab + 1):
            comp = lab == lab_id
            if comp.sum() >= min_size:
                keep |= comp
        morph = keep

    eia_morph = morph.astype(np.float32)

    pcc = fixed_baseline.copy()
    pcc_round_rows = []

    for r in range(1, PCC_ROUNDS + 1):
        p = safe_clip_prob(pcc)

        residual = (target_bool.astype(np.float32) - p) * R.astype(np.float32)

        if SCIPY_OK:
            residual_smooth = gaussian_filter(residual, sigma=SIGMA)
        else:
            residual_smooth = residual

        background_suppression = (~R).astype(np.float32) * p

        logits = safe_logit(p)
        logits = logits + PCC_ETA * residual_smooth - PCC_ETA * background_suppression
        pcc = safe_clip_prob(sigmoid(logits))

        rm = eval_prob_map(pcc, target_bool, threshold=THRESHOLD, main_mode=MAIN_MODE)
        pcc_round_rows.append({
            "round": r,
            "dice": rm["dice"],
            "iou": rm["iou"],
            "target_focus": rm["target_focus"],
            "log10_ratio": rm["log10_ratio"],
        })

    return {
        "fixed_baseline": fixed_baseline,
        "naive_self_tightening": naive,
        "eia_linear": eia_linear,
        "eia_blend090": eia_blend090,
        "eia_blend075": eia_blend075,
        "eia_morph": eia_morph,
        "pcc_correction": pcc,
        "target_support_R": R.astype(np.uint8),
        "target_signal_S": S.astype(np.float32),
        "pcc_round_history": pd.DataFrame(pcc_round_rows),
    }

# -----------------------------
# Load/prepare case arrays
# -----------------------------

def prepare_case_arrays(case_id, case_out_dir):
    paths = get_raw_paths(case_id)

    for k in ["cur_img", "fut_img", "cur_mask", "fut_mask"]:
        if not paths[k].exists():
            raise FileNotFoundError(f"Missing {k}: {paths[k]}")

    cur_img_raw, _, _, cur_raw_shape = load_nii_zhw(paths["cur_img"])
    fut_img_raw, _, _, fut_raw_shape = load_nii_zhw(paths["fut_img"])
    cur_mask, _, _, cur_mask_raw_shape = load_nii_zhw(paths["cur_mask"])
    fut_mask, _, _, fut_mask_raw_shape = load_nii_zhw(paths["fut_mask"])

    cur_mask = cur_mask > 0
    fut_mask = fut_mask > 0

    target = np.logical_and(fut_mask, np.logical_not(cur_mask))
    brain_mask = np.logical_or(cur_img_raw != 0, fut_img_raw != 0)

    cur_img = robust_normalize(cur_img_raw, brain_mask=brain_mask)

    assert cur_img.shape == cur_mask.shape == fut_mask.shape == target.shape

    np.save(case_out_dir / "current_t1c_norm_zhw.npy", cur_img.astype(np.float32))
    np.save(case_out_dir / "current_mask_zhw.npy", cur_mask.astype(np.uint8))
    np.save(case_out_dir / "future_mask_zhw.npy", fut_mask.astype(np.uint8))
    np.save(case_out_dir / "future_change_target_zhw.npy", target.astype(np.uint8))

    meta = {
        "case_id": case_id,
        "cur_raw_shape": list(cur_raw_shape),
        "fut_raw_shape": list(fut_raw_shape),
        "cur_mask_raw_shape": list(cur_mask_raw_shape),
        "fut_mask_raw_shape": list(fut_mask_raw_shape),
        "zhw_shape": list(cur_img.shape),
        "cur_mask_voxels": int(cur_mask.sum()),
        "fut_mask_voxels": int(fut_mask.sum()),
        "target_voxels": int(target.sum()),
        "brain_voxels": int(brain_mask.sum()),
    }

    with open(case_out_dir / "case_meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    return cur_img, cur_mask.astype(np.float32), target.astype(np.uint8), meta

# -----------------------------
# Resume logic
# -----------------------------

metrics_csv = DIRS["formal_results"] / "Layer2R_formal_case_method_metrics.csv"
comparisons_csv = DIRS["formal_results"] / "Layer2R_formal_pairwise_comparisons.csv"
training_csv = DIRS["formal_results"] / "Layer2R_formal_training_history.csv"
completed_txt = DIRS["formal_results"] / "completed_cases.txt"
failed_csv = DIRS["formal_results"] / "failed_cases.csv"

completed_cases = set()

if completed_txt.exists():
    completed_cases.update([x.strip() for x in completed_txt.read_text().splitlines() if x.strip()])

if metrics_csv.exists():
    try:
        tmp_metrics = pd.read_csv(metrics_csv)
        done_from_metrics = tmp_metrics.groupby("case_id")["method"].nunique()
        completed_cases.update(done_from_metrics[done_from_metrics >= 7].index.tolist())
    except Exception:
        pass

print("Already completed cases:", len(completed_cases))

if REQUIRE_EXISTING_PROGRESS and len(completed_cases) < EXPECTED_MIN_COMPLETED_CASES:
    raise RuntimeError(
        f"Safety stop: expected at least {EXPECTED_MIN_COMPLETED_CASES} completed cases, "
        f"but found {len(completed_cases)}. Do not continue, otherwise the experiment may restart from 0."
    )

cases_to_run = [c for c in locked_cases if c not in completed_cases]
if MAX_NEW_CASES is not None:
    cases_to_run = cases_to_run[:MAX_NEW_CASES]

print("Cases to run in this batch:", len(cases_to_run))
for c in cases_to_run:
    print(" -", c)

# -----------------------------
# Append helpers
# -----------------------------

def append_df_to_csv(df, path):
    path = Path(path)
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

def update_summary():
    if not metrics_csv.exists():
        return

    df = pd.read_csv(metrics_csv)
    if df.empty:
        return

    summary = (
        df.groupby("method")
        .agg(
            n=("case_id", "nunique"),
            dice_mean=("dice", "mean"),
            dice_median=("dice", "median"),
            iou_mean=("iou", "mean"),
            iou_median=("iou", "median"),
            target_focus_mean=("target_focus", "mean"),
            target_focus_median=("target_focus", "median"),
            log10_ratio_mean=("log10_ratio", "mean"),
            log10_ratio_median=("log10_ratio", "median"),
        )
        .reset_index()
    )

    summary.to_csv(DIRS["formal_results"] / "Layer2R_formal_summary_by_method.csv", index=False)

    if comparisons_csv.exists():
        comp = pd.read_csv(comparisons_csv)
        comp_summary = (
            comp.groupby("comparison")
            .agg(
                n=("case_id", "nunique"),
                dice_mean_diff=("dice_diff", "mean"),
                dice_median_diff=("dice_diff", "median"),
                dice_wins=("pcc_better_dice", "sum"),
                iou_mean_diff=("iou_diff", "mean"),
                iou_median_diff=("iou_diff", "median"),
                iou_wins=("pcc_better_iou", "sum"),
            )
            .reset_index()
        )
        comp_summary["dice_win_rate"] = comp_summary["dice_wins"] / comp_summary["n"]
        comp_summary["iou_win_rate"] = comp_summary["iou_wins"] / comp_summary["n"]
        comp_summary.to_csv(DIRS["formal_results"] / "Layer2R_formal_pairwise_summary.csv", index=False)

# -----------------------------
# Main formal batch loop
# -----------------------------

batch_start = time.time()
failed_rows = []

for case_i, case_id in enumerate(cases_to_run, start=1):
    print("\n" + "=" * 110)
    print(f"Running case {case_i}/{len(cases_to_run)} in this run: {case_id}")
    print("=" * 110)

    case_start = time.time()
    case_out_dir = DIRS["case_outputs"] / case_id
    case_out_dir.mkdir(parents=True, exist_ok=True)

    try:
        cur_img, cur_mask, target, meta = prepare_case_arrays(case_id, case_out_dir)

        if int(target.sum()) <= 0:
            raise ValueError("Future-change target is empty.")

        print("Shape:", cur_img.shape, "| target voxels:", int(target.sum()))

        model, baseline_prob, hist_df, best_dice_topk, elapsed_train = train_case_baseline(
            case_id, cur_img, cur_mask, target
        )

        hist_df["formal_epochs"] = FORMAL_EPOCHS
        hist_df["training_elapsed_sec_total"] = elapsed_train
        append_df_to_csv(hist_df, training_csv)
        hist_df.to_csv(case_out_dir / "baseline_training_history_formal.csv", index=False)

        torch.save(
            {
                "case_id": case_id,
                "formal_epochs": FORMAL_EPOCHS,
                "model_state_dict": model.state_dict(),
                "best_dice_topk": best_dice_topk,
                "protocol": protocol,
            },
            DIRS["checkpoints"] / f"{case_id}_baseline_formal_best.pt",
        )

        np.save(case_out_dir / "baseline_prob_map_formal_float16.npy", baseline_prob.astype(np.float16))

        maps = run_corrections_for_case(baseline_prob, target)

        np.save(case_out_dir / "target_support_R_formal.npy", maps["target_support_R"].astype(np.uint8))
        np.save(case_out_dir / "target_signal_S_formal_float16.npy", maps["target_signal_S"].astype(np.float16))
        maps["pcc_round_history"].to_csv(case_out_dir / "pcc_round_history_formal.csv", index=False)

        method_rows = []

        method_names = [
            "fixed_baseline",
            "naive_self_tightening",
            "eia_linear",
            "eia_blend090",
            "eia_blend075",
            "eia_morph",
            "pcc_correction",
        ]

        for method in method_names:
            prob = maps[method]
            m = eval_prob_map(prob, target.astype(bool), threshold=THRESHOLD, main_mode=MAIN_MODE)

            row = {
                "case_id": case_id,
                "method": method,
                "formal_epochs": FORMAL_EPOCHS,
                "baseline_best_dice_topk": best_dice_topk,
                "baseline_training_elapsed_sec": elapsed_train,
                **m,
            }
            method_rows.append(row)

            if method == "eia_morph":
                np.save(case_out_dir / f"{method}_formal_uint8.npy", (prob > 0.5).astype(np.uint8))
            else:
                np.save(case_out_dir / f"{method}_formal_float16.npy", safe_clip_prob(prob).astype(np.float16))

        method_df = pd.DataFrame(method_rows)
        append_df_to_csv(method_df, metrics_csv)
        method_df.to_csv(case_out_dir / "case_method_metrics_formal.csv", index=False)

        pcc_row = method_df[method_df["method"] == "pcc_correction"].iloc[0].to_dict()
        comp_rows = []

        for method in [
            "fixed_baseline",
            "naive_self_tightening",
            "eia_linear",
            "eia_blend090",
            "eia_blend075",
            "eia_morph",
        ]:
            row = method_df[method_df["method"] == method].iloc[0].to_dict()
            comp_rows.append({
                "case_id": case_id,
                "comparison": f"PCC vs {method}",
                "dice_diff": pcc_row["dice"] - row["dice"],
                "iou_diff": pcc_row["iou"] - row["iou"],
                "target_focus_diff": pcc_row["target_focus"] - row["target_focus"],
                "log10_ratio_diff": pcc_row["log10_ratio"] - row["log10_ratio"],
                "pcc_better_dice": bool(pcc_row["dice"] > row["dice"]),
                "pcc_better_iou": bool(pcc_row["iou"] > row["iou"]),
            })

        comp_df = pd.DataFrame(comp_rows)
        append_df_to_csv(comp_df, comparisons_csv)
        comp_df.to_csv(case_out_dir / "case_pairwise_comparisons_formal.csv", index=False)

        z = choose_target_slice(target)

        fig, axes = plt.subplots(2, 4, figsize=(18, 8))
        axes = axes.ravel()

        vis_items = [
            ("Current T1c", cur_img[z], "gray"),
            ("Target", target[z].astype(float), "gray"),
            ("Fixed baseline", maps["fixed_baseline"][z], "hot"),
            ("EIA-linear", maps["eia_linear"][z], "hot"),
            ("EIA-blend-0.90", maps["eia_blend090"][z], "hot"),
            ("EIA-blend-0.75", maps["eia_blend075"][z], "hot"),
            ("EIA-morph", maps["eia_morph"][z], "gray"),
            ("PCC", maps["pcc_correction"][z], "hot"),
        ]

        for ax, (title, img, cmap) in zip(axes, vis_items):
            ax.imshow(img, cmap=cmap)
            ax.set_title(title, fontsize=10)
            ax.axis("off")

        plt.tight_layout()
        fig_path = DIRS["figures"] / f"Layer2R_formal_{case_id}.png"
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        with open(completed_txt, "a") as f:
            f.write(case_id + "\n")

        completed_cases.add(case_id)

        case_elapsed = time.time() - case_start
        print(f"CASE COMPLETED: {case_id} | elapsed sec: {case_elapsed:.1f}")
        print("Method metrics:")
        display(method_df[["case_id", "method", "dice", "iou", "target_focus", "log10_ratio", "dice_fixed05", "dice_topk"]])
        print("Comparisons:")
        display(comp_df)

        update_summary()

        del model
        del baseline_prob
        del maps
        if device.type == "cuda":
            torch.cuda.empty_cache()

    except Exception as e:
        fail_row = {
            "case_id": case_id,
            "error": repr(e),
            "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        failed_rows.append(fail_row)
        pd.DataFrame([fail_row]).to_csv(
            failed_csv,
            mode="a",
            header=not failed_csv.exists(),
            index=False,
        )
        print("CASE FAILED:", case_id)
        print("ERROR:", repr(e))

# -----------------------------
# Final status
# -----------------------------

update_summary()

batch_elapsed = time.time() - batch_start

remaining_cases = [c for c in locked_cases if c not in completed_cases]

status = {
    "run_name": RUN_NAME,
    "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "device": str(device),
    "formal_epochs": FORMAL_EPOCHS,
    "max_new_cases_this_run": MAX_NEW_CASES,
    "new_cases_attempted_this_run": len(cases_to_run),
    "failed_this_run": failed_rows,
    "completed_cases_total": len(completed_cases),
    "remaining_cases_total": len(remaining_cases),
    "batch_elapsed_sec": batch_elapsed,
    "next_instruction": "If remaining_cases_total is 0, Save Version immediately and then run final statistical analysis.",
}

with open(DIRS["formal_results"] / "RUN_STATUS_LATEST.json", "w") as f:
    json.dump(status, f, indent=2)

print("\n" + "=" * 110)
print("FULL REMAINING RUN FINISHED")
print("=" * 110)
print(json.dumps(status, indent=2))

if metrics_csv.exists():
    print("\nCurrent summary by method:")
    display(pd.read_csv(DIRS["formal_results"] / "Layer2R_formal_summary_by_method.csv"))

if comparisons_csv.exists():
    print("\nCurrent pairwise summary:")
    display(pd.read_csv(DIRS["formal_results"] / "Layer2R_formal_pairwise_summary.csv"))

if len(remaining_cases) > 0:
    print("\nRemaining cases:")
    for c in remaining_cases:
        print(" -", c)
else:
    print("\nALL 40 CASES COMPLETED.")

print("\nIMPORTANT: Save Version now.")

Device: cuda
Locked cases: 40
First 5: ['PatientID_0003_T1_to_T2_t1c', 'PatientID_0005_T3_to_T4_t1c', 'PatientID_0006_T2_to_T4_t1c', 'PatientID_0007_T2_to_T3_t1c', 'PatientID_0008_T4_to_T6_t1c']
Protocol saved: /kaggle/working/Layer2R_publication_rebuild_EIA_v1/protocol.json
Already completed cases: 5
Cases to run in this batch: 35
 - PatientID_0010_T1_to_T4_t1c
 - PatientID_0011_T1_to_T2_t1c
 - PatientID_0012_T2_to_T3_t1c
 - PatientID_0013_T1_to_T2_t1c
 - PatientID_0014_T1_to_T2_t1c
 - PatientID_0018_T1_to_T2_t1c
 - PatientID_0019_T4_to_T5_t1c
 - PatientID_0020_T1_to_T2_t1c
 - PatientID_0021_T2_to_T3_t1c
 - PatientID_0022_T1_to_T2_t1c
 - PatientID_0024_T2_to_T3_t1c
 - PatientID_0025_T1_to_T2_t1c
 - PatientID_0026_T1_to_T2_t1c
 - PatientID_0029_T1_to_T3_t1c
 - PatientID_0030_T1_to_T3_t1c
 - PatientID_0031_T2_to_T3_t1c
 - PatientID_0032_T1_to_T2_t1c
 - PatientID_0033_T1_to_T2_t1c
 - PatientID_0034_T1_to_T2_t1c
 - PatientID_0035_T1_to_T2_t1c
 - PatientID_0036_T1_to_T2_t1c
 - PatientID_00

,case_id,method,dice,iou,target_focus,log10_ratio,dice_fixed05,dice_topk
0,PatientID_0010_T1_to_T4_t1c,fixed_baseline,0.467236,0.304833,0.019878,0.724101,0.256565,0.467236
1,PatientID_0010_T1_to_T4_t1c,naive_self_tightening,0.467236,0.304833,0.097663,1.451377,0.256565,0.467236
2,PatientID_0010_T1_to_T4_t1c,eia_linear,0.599054,0.427607,0.027579,0.869746,0.334576,0.599054
3,PatientID_0010_T1_to_T4_t1c,eia_blend090,0.645226,0.476261,0.021786,0.764766,0.288272,0.645226
4,PatientID_0010_T1_to_T4_t1c,eia_blend075,0.791788,0.655338,0.025573,0.836042,0.376274,0.791788
5,PatientID_0010_T1_to_T4_t1c,eia_morph,0.224601,0.126508,0.239306,1.914759,0.384939,0.224601
6,PatientID_0010_T1_to_T4_t1c,pcc_correction,0.842042,0.727178,0.027810,0.873461,0.745473,0.842042


Comparisons:


,case_id,comparison,dice_diff,iou_diff,target_focus_diff,log10_ratio_diff,pcc_better_dice,pcc_better_iou
0,PatientID_0010_T1_to_T4_t1c,PCC vs fixed_baseline,0.374805,0.422345,0.007932,0.149361,True,True
1,PatientID_0010_T1_to_T4_t1c,PCC vs naive_self_tightening,0.374805,0.422345,-0.069853,-0.577915,True,True
2,PatientID_0010_T1_to_T4_t1c,PCC vs eia_linear,0.242988,0.299571,0.000230,0.003715,True,True
3,PatientID_0010_T1_to_T4_t1c,PCC vs eia_blend090,0.196816,0.250918,0.006023,0.108696,True,True
4,PatientID_0010_T1_to_T4_t1c,PCC vs eia_blend075,0.050254,0.071840,0.002237,0.037419,True,True
5,PatientID_0010_T1_to_T4_t1c,PCC vs eia_morph,0.617441,0.600671,-0.211496,-1.041298,True,True



Running case 2/35 in this run: PatientID_0011_T1_to_T2_t1c
Shape: (155, 240, 240) | target voxels: 62041
PatientID_0011_T1_to_T2_t1c | Epoch 01/12 | loss=1.6679 | dice_topk=0.0412 | dice_fixed05=0.0000 | focus=0.0079
PatientID_0011_T1_to_T2_t1c | Epoch 02/12 | loss=1.4491 | dice_topk=0.5476 | dice_fixed05=0.5478 | focus=0.0126
PatientID_0011_T1_to_T2_t1c | Epoch 03/12 | loss=1.3931 | dice_topk=0.5668 | dice_fixed05=0.4189 | focus=0.0203
PatientID_0011_T1_to_T2_t1c | Epoch 04/12 | loss=1.3430 | dice_topk=0.6104 | dice_fixed05=0.3432 | focus=0.0247
PatientID_0011_T1_to_T2_t1c | Epoch 05/12 | loss=1.3056 | dice_topk=0.5955 | dice_fixed05=0.2796 | focus=0.0274
PatientID_0011_T1_to_T2_t1c | Epoch 06/12 | loss=1.2689 | dice_topk=0.6840 | dice_fixed05=0.4487 | focus=0.0303
PatientID_0011_T1_to_T2_t1c | Epoch 07/12 | loss=1.2400 | dice_topk=0.6613 | dice_fixed05=0.4604 | focus=0.0323
PatientID_0011_T1_to_T2_t1c | Epoch 08/12 | loss=1.2101 | dice_topk=0.6550 | dice_fixed05=0.5279 | focus=0.035

...


FULL REMAINING RUN FINISHED
{
  "run_name": "Layer2R_publication_rebuild_EIA_v1",
  "time": "2026-06-17 01:03:05",
  "device": "cuda",
  "formal_epochs": 12,
  "max_new_cases_this_run": null,
  "new_cases_attempted_this_run": 35,
  "failed_this_run": [],
  "completed_cases_total": 40,
  "remaining_cases_total": 0,
  "batch_elapsed_sec": 1755.120477437973,
  "next_instruction": "If remaining_cases_total is 0, Save Version immediately and then run final statistical analysis."
}

Current summary by method:


,method,n,dice_mean,dice_median,iou_mean,iou_median,target_focus_mean,target_focus_median,log10_ratio_mean,log10_ratio_median
0,eia_blend075,40,0.791867,0.821818,0.664159,0.697538,0.027731,0.025567,0.948130,0.911711
1,eia_blend090,40,0.736545,0.761790,0.593486,0.615237,0.024389,0.022428,0.892356,0.851543
2,eia_linear,40,0.631146,0.648795,0.469234,0.480174,0.029895,0.027998,0.996773,0.963368
3,eia_morph,40,0.260892,0.234779,0.158462,0.133030,0.262536,0.232991,2.223423,2.171178
4,fixed_baseline,40,0.597941,0.616808,0.435498,0.445931,0.022694,0.020878,0.861310,0.823465
5,naive_self_tightening,40,0.597039,0.598088,0.434580,0.426886,0.125069,0.107804,1.724365,1.705671
6,pcc_correction,40,0.736352,0.762534,0.592723,0.616233,0.030465,0.028407,0.986294,0.942458



Current pairwise summary:


,comparison,n,dice_mean_diff,dice_median_diff,dice_wins,iou_mean_diff,iou_median_diff,iou_wins,dice_win_rate,iou_win_rate
0,PCC vs eia_blend075,40,-0.055514,-0.062986,5,-0.071436,-0.086657,5,0.125,0.125
1,PCC vs eia_blend090,40,-0.000193,-0.025704,16,-0.000762,-0.036454,16,0.400,0.400
2,PCC vs eia_linear,40,0.105206,0.096093,40,0.123489,0.115100,40,1.000,1.000
3,PCC vs eia_morph,40,0.475460,0.478948,40,0.434261,0.441107,40,1.000,1.000
4,PCC vs fixed_baseline,40,0.138411,0.122785,40,0.157225,0.138351,40,1.000,1.000
5,PCC vs naive_self_tightening,40,0.139313,0.125530,40,0.158143,0.138933,40,1.000,1.000



ALL 40 CASES COMPLETED.

IMPORTANT: Save Version now.
